In [1]:
# -*- coding: utf-8 -*-
"""
MX 平台数据周报生成脚本 v2.1 完整版
章节：一、大盘核心数据  二、总代分析  三、用户分析
      四、游戏分析      五、活动分析  六、道具专题分析
      七、风险专项分析  八、总结与行动建议
修复：厂商简称统一 / 高危游戏列宽 / 总结章节分析逻辑 / 指标下降原因拆解
"""

import sys, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                 TableStyle, HRFlowable, PageBreak, Image as RLImage)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════
#  ① 用户配置区  ← 每周只改这里
# ══════════════════════════════════════════════════════
THIS_WEEK = ("20260605", "20260611")
LAST_WEEK = ("20260529", "20260604")

DATA_ROOT  = Path(r"D:\周报更新版\MX")
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")

# ══════════════════════════════════════════════════════
#  ② 字体
# ══════════════════════════════════════════════════════
def _reg_font():
    candidates = [
        ("WQYMicHei", r"C:\Windows\Fonts\wqy-microhei.ttc"),
        ("MsYaHei",   r"C:\Windows\Fonts\msyh.ttc"),
        ("SimHei",    r"C:\Windows\Fonts\simhei.ttf"),
        ("WQYMicHei", "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc"),
        ("NotoSCJK",  "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
    ]
    for name, path in candidates:
        if Path(path).exists():
            try:
                pdfmetrics.registerFont(TTFont(name,  path))
                pdfmetrics.registerFont(TTFont(name+"B", path))
                print(f"  ✔ 字体：{name} ({Path(path).name})")
                return name, name+"B"
            except Exception:
                continue
    return "Helvetica", "Helvetica-Bold"

FONT_N, FONT_B = _reg_font()

# ══════════════════════════════════════════════════════
#  ③ 颜色 / 尺寸
# ══════════════════════════════════════════════════════
PW, PH  = A4
MARGIN  = 1.5*cm
CW      = PW - 2*MARGIN          # 内容区宽度

C_BLUE  = colors.HexColor("#1a3a6b")
C_LBLUE = colors.HexColor("#2563eb")
C_AMBER = colors.HexColor("#d97706")
C_RED   = colors.HexColor("#dc2626")
C_ORED  = colors.HexColor("#ea580c")
C_GREEN = colors.HexColor("#16a34a")
C_DARK  = colors.HexColor("#1e293b")
C_GRAY  = colors.HexColor("#64748b")
C_LGRAY = colors.HexColor("#f1f5f9")
C_BGGRN = colors.HexColor("#f0fdf4")
C_BGRED = colors.HexColor("#fff1f2")
C_BGYEL = colors.HexColor("#fffbeb")
C_BORD  = colors.HexColor("#cbd5e1")
C_WHITE = colors.white

# ══════════════════════════════════════════════════════
#  ④ 厂商简称映射（全局统一）
# ══════════════════════════════════════════════════════
MFR_MAP = {
    "Pragmatic Play":"Pragmati","Pragmatic":"Pragmati",
    "PlayTech":"PlayTech","Playtech":"PlayTech",
    "Rectangle":"Rectangl","Rectangle Gaming":"Rectangl",
    "Tada":"Tada","PG Soft":"PG Soft","PGSoft":"PG Soft",
    "3Oaks":"3Oaks","Evolution":"Evolutn","FaChai":"FaChai",
    "SmartSoft":"SmartSf","Originals":"Original","Spribe":"Spribe",
    "Booongo":"Booongo","Hacksaw":"Hacksaw","1X2 Network":"1X2Net",
}
def abbr(name): return MFR_MAP.get(str(name).strip(), str(name).strip()[:8])

# ══════════════════════════════════════════════════════
#  ⑤ 基础排版工具
# ══════════════════════════════════════════════════════
def P(txt, sz=9, bold=False, clr=C_DARK, align=TA_LEFT, lead=None):
    return Paragraph(str(txt), ParagraphStyle("x",
        fontName=FONT_B if bold else FONT_N, fontSize=sz,
        textColor=clr, alignment=align, leading=lead or sz*1.38, wordWrap="CJK"))

def sec_title(text, clr=C_BLUE):
    return [Spacer(1,.3*cm), P(text,14,True,clr),
            HRFlowable(width="100%",thickness=1.5,color=clr), Spacer(1,.2*cm)]

def sub_title(text, clr=C_BLUE):
    return [Spacer(1,.15*cm), P(text,10,True,clr), Spacer(1,.08*cm)]

def cell(v, bold=False, clr=C_DARK, align=TA_CENTER, sz=8):
    return P(str(v), sz, bold, clr, align)

def make_table(hdrs, rows, cws, sz=8, zebra=True, hbg=C_BLUE,
               col_fn=None):
    """通用表格。col_fn(row_idx, row_data) -> [(col,bg,fg),...]"""
    sty = [
        ("BACKGROUND",(0,0),(-1,0), hbg),
        ("TEXTCOLOR",  (0,0),(-1,0), C_WHITE),
        ("FONTNAME",   (0,0),(-1,0), FONT_B),
        ("FONTNAME",   (0,1),(-1,-1),FONT_N),
        ("FONTSIZE",   (0,0),(-1,-1),sz),
        ("GRID",       (0,0),(-1,-1),.3,C_BORD),
        ("VALIGN",     (0,0),(-1,-1),"MIDDLE"),
        ("TOPPADDING", (0,0),(-1,-1),3),
        ("BOTTOMPADDING",(0,0),(-1,-1),3),
        ("LEFTPADDING",(0,0),(-1,-1),4),
    ]
    if zebra:
        for i in range(1, len(rows)+1, 2):
            sty.append(("ROWBACKGROUND",(0,i),(-1,i),C_LGRAY))
    data = [[cell(h,True,C_WHITE,TA_CENTER,sz) for h in hdrs]]
    for ri, row in enumerate(rows):
        data.append([cell(v,False,C_DARK,TA_CENTER,sz) for v in row])
        if col_fn:
            for ci, bg, fg in col_fn(ri, row):
                sty += [("BACKGROUND",(ci,ri+1),(ci,ri+1),bg),
                        ("TEXTCOLOR",(ci,ri+1),(ci,ri+1),fg)]
    t = Table(data, colWidths=cws)
    t.setStyle(TableStyle(sty))
    return t

def insight_box(lines, clr=C_LBLUE, bg=None, bullet="◆"):
    bg = bg or C_LGRAY
    paras = [P(f"{bullet} {l}", 8.5, False, C_DARK) for l in lines]
    inner = Table([[p] for p in paras], colWidths=[CW-1.2*cm],
                  style=TableStyle([("LEFTPADDING",(0,0),(-1,-1),8),
                                    ("RIGHTPADDING",(0,0),(-1,-1),8),
                                    ("TOPPADDING",(0,0),(-1,-1),4),
                                    ("BOTTOMPADDING",(0,0),(-1,-1),4),
                                    ("BACKGROUND",(0,0),(-1,-1),bg)]))
    outer = Table([[inner]], colWidths=[CW],
                  style=TableStyle([("BOX",(0,0),(-1,-1),1.5,clr),
                                    ("LEFTPADDING",(0,0),(-1,-1),0),
                                    ("TOPPADDING",(0,0),(-1,-1),0),
                                    ("BOTTOMPADDING",(0,0),(-1,-1),0)]))
    return outer

def kpi_card_row(items, cols=4):
    """items: [(label, val, sub, good?)]  good=True/False/None"""
    fw = CW/cols - 3
    row = []
    for label,val,sub,good in items:
        vc = C_GREEN if good is True else (C_RED if good is False else C_DARK)
        c = Table([[P(label,7,False,C_GRAY)],[P(val,13,True,vc)],[P(sub,7,False,C_GRAY)]],
                  colWidths=[fw],
                  style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
                                    ("BOX",(0,0),(-1,-1),.5,C_BORD),
                                    ("LEFTPADDING",(0,0),(-1,-1),7),
                                    ("TOPPADDING",(0,0),(-1,-1),5),
                                    ("BOTTOMPADDING",(0,0),(-1,-1),5)]))
        row.append(c)
    while len(row) < cols:
        row.append(Table([[P("")]], colWidths=[fw],
                         style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_WHITE)])))
    t = Table([row], colWidths=[fw+3]*cols, hAlign="LEFT", vAlign="TOP")
    return [t, Spacer(1,.12*cm)]

def kpi_grid(items, cols=4):
    out = []
    for i in range(0, len(items), cols):
        out += kpi_card_row(items[i:i+cols], cols)
    return out

# ══════════════════════════════════════════════════════
#  ⑥ 数值格式化
# ══════════════════════════════════════════════════════
nan = np.nan

def _v(K, key): return K.get(key, nan)

def fmt(v, prefix="$", wan=True):
    if isinstance(v,float) and np.isnan(v): return "—"
    if wan and abs(v)>=10000: return f"{prefix}{v/10000:.1f}万"
    return f"{prefix}{v:,.0f}"

def pct(v, mul=100, deci=2):
    if isinstance(v,float) and np.isnan(v): return "—"
    return f"{v*mul:.{deci}f}%"

def chg_pct(tw,lw):
    if any(isinstance(x,float) and np.isnan(x) for x in [tw,lw]): return "—"
    if lw==0: return "—"
    return f"{(tw-lw)/abs(lw)*100:+.1f}%"

def chg_pp(tw,lw,mul=100,deci=2):
    if any(isinstance(x,float) and np.isnan(x) for x in [tw,lw]): return "—"
    return f"{(tw-lw)*mul:+.{deci}f}pp"

def good(tw,lw,higher=True):
    if any(isinstance(x,float) and np.isnan(x) for x in [tw,lw]): return None
    return (tw>lw)==higher

# ══════════════════════════════════════════════════════
#  ⑦ 文件匹配 & 加载
# ══════════════════════════════════════════════════════
def _find(base: Path, kws: list, exts=(".csv",".xlsx",".xls")):
    best = None
    for f in base.rglob("*"):
        if f.suffix.lower() not in exts: continue
        nm = f.name.lower()
        if all(k.lower() in nm for k in kws):
            if best is None or len(f.name)<len(best.name):
                best = f
    return best

def _load(path, **kw):
    if path is None: return None
    p = Path(path)
    if not p.exists(): print(f"  ⚠ 未找到: {p.name}"); return None
    try:
        if p.suffix.lower()==".csv": return pd.read_csv(p, encoding="utf-8-sig", **kw)
        return pd.read_excel(p, **kw)
    except Exception as e:
        print(f"  ⚠ 读取失败 {p.name}: {e}"); return None

def load_all(root: Path) -> dict:
    D = {}
    TW, LW = THIS_WEEK, LAST_WEEK

    # ── 大盘
    D["plat"]     = _load(_find(root, ["平台报表","USD"], (".xlsx",".xls")))
    D["dash_ret"] = _load(_find(root, ["首充留存"]))

    # ── 总代
    D["ag_plat"]  = _load(_find(root, ["平台报表","总代","USD"]))
    D["ag_promo"] = _load(_find(root, ["推广报表","总代","USD"]))
    D["ag_ret"]   = _load(_find(root, ["首充充值留存"]))

    # ── 用户
    D["vip_rpt"]  = _load(_find(root, ["VIP报表","USD"]))
    D["vip_act"]  = _load(_find(root, ["VIP","活跃"]))
    D["vip_dep"]  = _load(_find(root, ["VIP","充值"], (".csv",".xlsx")))
    D["dep_tw"]   = _load(_find(root, ["头部充值用户", TW[0]]))
    D["dep_lw"]   = _load(_find(root, ["头部充值用户", LW[0]]))
    D["wdr_tw"]   = _load(_find(root, ["top提款用户",  TW[0]]))
    D["wdr_lw"]   = _load(_find(root, ["top提款用户",  LW[0]]))
    D["top500_tw"]= _load(_find(root, ["top500提款","游戏偏好", TW[0]]))
    D["top500_lw"]= _load(_find(root, ["top500提款","游戏偏好", LW[0]]))

    # ── 游戏
    D["mfr"]      = _load(_find(root, ["厂商投注数据"]))
    D["game_tw"]  = _load(_find(root, ["游戏报表","本周"]))
    D["game_lw"]  = _load(_find(root, ["游戏报表","上周"]))

    # ── 活动
    D["tool_tw"]  = _load(_find(root, ["道具使用情况","本周"]))
    D["tool_lw"]  = _load(_find(root, ["道具使用情况","上周"]))
    D["tool_map"] = _load(_find(root, ["道具对应活动"]))
    D["fdr"]      = _load(_find(root, ["首次充值活动"]))
    D["gift_tot"] = _load(_find(root, ["整体赠送活动"]))
    D["gift_dtl"] = _load(_find(root, ["各赠送活动"]))
    D["dep_tot"]  = _load(_find(root, ["整体存款"]))
    D["dep_src"]  = _load(_find(root, ["存款来源"]))

    loaded = sum(1 for v in D.values() if v is not None)
    print(f"  数据加载：{loaded}/{len(D)} 文件成功")
    return D

# ══════════════════════════════════════════════════════
#  ⑧ KPI提取（从大盘平台报表）
# ══════════════════════════════════════════════════════
def extract_kpi(plat_df) -> dict:
    K = {}
    if plat_df is None: return K

    # 列名匹配
    alias = {
        "dep":    ["充值金额","充值"],
        "wdr":    ["提现金额","提款金额"],
        "diff":   ["充提差"],
        "diff_r": ["充提差率","充提率"],
        "gwl":    ["公司输赢","公司盈亏"],
        "gwl_r":  ["盈亏率"],
        "reg":    ["注册人数","注册数"],
        "fd":     ["首充人数","首充"],
        "act":    ["日均活跃","活跃人数"],
        "bet":    ["投注金额"],
        "arppu":  ["全量ARPPU","ARPPU"],
        "o_arppu":["老用户ARPPU","老用户人均"],
        "fd_arppu":["首充ARPPU"],
        "gift":   ["总赠送金额","赠送金额"],
        "gift_r": ["赠送/充值比","赠送充值比","赠送/充值"],
        "fd_rate":["首充转化率","首充率"],
        "fd_d1":  ["首充次日复充率","次日复充率"],
        "roi":    ["充提差ROI","充提ROI","ROI"],
        "cost":   ["推广日均消耗","日均消耗"],
    }

    def _pick(df, keys):
        for k in keys:
            for c in df.columns:
                if k in str(c):
                    s = pd.to_numeric(df[c], errors="coerce").dropna()
                    if len(s): return float(s.sum() if "金额" in k or "人数" in k or "人" in k else s.mean())
        return nan

    # 尝试按日期区间筛选
    tw_df = lw_df = plat_df
    for dc in ["日期","date","Date"]:
        if dc in plat_df.columns:
            try:
                tmp = plat_df.copy()
                tmp[dc] = pd.to_datetime(tmp[dc], errors="coerce")
                tw_df = tmp[tmp[dc].dt.strftime("%Y%m%d").between(THIS_WEEK[0],THIS_WEEK[1])]
                lw_df = tmp[tmp[dc].dt.strftime("%Y%m%d").between(LAST_WEEK[0],LAST_WEEK[1])]
            except Exception: pass
            break

    for key, alts in alias.items():
        K[f"tw_{key}"] = _pick(tw_df if len(tw_df) else plat_df, alts)
        K[f"lw_{key}"] = _pick(lw_df if len(lw_df) else pd.DataFrame(), alts)

    # 盈亏率 & 充提差率：用 sum(gwl)/sum(bet) 更准确
    for pfx, df in [("tw",tw_df),("lw",lw_df)]:
        gwl_col = bet_col = dep_col = diff_col = None
        for c in df.columns:
            cs = str(c)
            if "公司输赢" in cs or "公司盈亏" in cs: gwl_col=c
            if "投注金额" in cs: bet_col=c
            if "充值金额" in cs or ("充值" in cs and "提" not in cs): dep_col=c
            if "充提差" in cs and "率" not in cs: diff_col=c
        if gwl_col and bet_col:
            g = pd.to_numeric(df[gwl_col],errors="coerce").sum()
            b = pd.to_numeric(df[bet_col],errors="coerce").sum()
            if b>0: K[f"{pfx}_gwl_r"] = g/b
        if diff_col and dep_col:
            d = pd.to_numeric(df[diff_col],errors="coerce").sum()
            dp= pd.to_numeric(df[dep_col],errors="coerce").sum()
            if dp>0: K[f"{pfx}_diff_r"] = d/dp

    return K

# ══════════════════════════════════════════════════════
#  matplotlib图表辅助
# ══════════════════════════════════════════════════════
CHART_DIR = Path("/tmp/mx_charts")
CHART_DIR.mkdir(exist_ok=True)

def _mpl_setup():
    # 中文字体
    for fp in [r"C:\Windows\Fonts\msyh.ttc",
               "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
               "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
        if Path(fp).exists():
            from matplotlib import font_manager
            font_manager.fontManager.addfont(fp)
            plt.rcParams["font.family"] = font_manager.FontProperties(fname=fp).get_name()
            break
    plt.rcParams.update({"axes.unicode_minus":False,"figure.facecolor":"white","axes.facecolor":"#f8fafc"})

_mpl_setup()

def _save(name):
    p = CHART_DIR/name
    plt.savefig(p, dpi=130, bbox_inches="tight")
    plt.close()
    return p

def _img(path, w=None, h=None):
    if path is None or not Path(path).exists(): return None
    kw = {}
    if w: kw["width"] = w
    if h: kw["height"]= h
    return RLImage(str(path), **kw)

# ══════════════════════════════════════════════════════
#  第一章：大盘核心数据
# ══════════════════════════════════════════════════════
def build_overview(K: dict) -> list:
    S = sec_title("一、大盘核心数据", C_BLUE)

    tw_dep=_v(K,"tw_dep"); lw_dep=_v(K,"lw_dep")
    tw_wdr=_v(K,"tw_wdr"); lw_wdr=_v(K,"lw_wdr")
    tw_diff_r=_v(K,"tw_diff_r"); lw_diff_r=_v(K,"lw_diff_r")
    tw_gwl=_v(K,"tw_gwl");  lw_gwl=_v(K,"lw_gwl")
    tw_gwl_r=_v(K,"tw_gwl_r"); lw_gwl_r=_v(K,"lw_gwl_r")
    tw_reg=_v(K,"tw_reg");  lw_reg=_v(K,"lw_reg")
    tw_fd=_v(K,"tw_fd");    lw_fd=_v(K,"lw_fd")
    tw_act=_v(K,"tw_act");  lw_act=_v(K,"lw_act")
    tw_bet=_v(K,"tw_bet");  lw_bet=_v(K,"lw_bet")
    tw_ap=_v(K,"tw_arppu"); lw_ap=_v(K,"lw_arppu")
    tw_oa=_v(K,"tw_o_arppu"); lw_oa=_v(K,"lw_o_arppu")
    tw_fa=_v(K,"tw_fd_arppu"); lw_fa=_v(K,"lw_fd_arppu")
    tw_gr=_v(K,"tw_gift_r"); lw_gr=_v(K,"lw_gift_r")
    tw_fr=_v(K,"tw_fd_rate"); lw_fr=_v(K,"lw_fd_rate")
    tw_roi=_v(K,"tw_roi"); lw_roi=_v(K,"lw_roi")
    tw_cost=_v(K,"tw_cost"); lw_cost=_v(K,"lw_cost")
    tw_gift=_v(K,"tw_gift"); lw_gift=_v(K,"lw_gift")

    S += kpi_grid([
        ("充值金额",  fmt(tw_dep),   f"上周{fmt(lw_dep)}  {chg_pct(tw_dep,lw_dep)}",   good(tw_dep,lw_dep)),
        ("提现金额",  fmt(tw_wdr),   f"上周{fmt(lw_wdr)}  {chg_pct(tw_wdr,lw_wdr)}",   good(tw_wdr,lw_wdr,False)),
        ("充提差",    fmt(_v(K,"tw_diff")), f"上周{fmt(_v(K,'lw_diff'))}  {chg_pct(_v(K,'tw_diff'),_v(K,'lw_diff'))}", good(_v(K,"tw_diff"),_v(K,"lw_diff"))),
        ("充提差率",  pct(tw_diff_r), f"上周{pct(lw_diff_r)}  {chg_pp(tw_diff_r,lw_diff_r)}",  good(tw_diff_r,lw_diff_r)),
        ("公司输赢",  fmt(tw_gwl),   f"上周{fmt(lw_gwl)}  {chg_pct(tw_gwl,lw_gwl)}",   good(tw_gwl,lw_gwl)),
        ("盈亏率",    pct(tw_gwl_r,deci=3), f"上周{pct(lw_gwl_r,deci=3)}  {chg_pp(tw_gwl_r,lw_gwl_r,deci=3)}", good(tw_gwl_r,lw_gwl_r)),
        ("注册人数",  f"{tw_reg:,.0f}" if not np.isnan(tw_reg) else "—", f"上周{lw_reg:,.0f}  {chg_pct(tw_reg,lw_reg)}", good(tw_reg,lw_reg)),
        ("首充人数",  f"{tw_fd:,.0f}" if not np.isnan(tw_fd) else "—",  f"上周{lw_fd:,.0f}  {chg_pct(tw_fd,lw_fd)}",  good(tw_fd,lw_fd)),
        ("日均活跃",  f"{tw_act:,.0f}" if not np.isnan(tw_act) else "—",f"上周{lw_act:,.0f}  {chg_pct(tw_act,lw_act)}",good(tw_act,lw_act)),
        ("投注金额",  fmt(tw_bet),   f"上周{fmt(lw_bet)}  {chg_pct(tw_bet,lw_bet)}",   good(tw_bet,lw_bet)),
        ("全量ARPPU", f"${tw_ap:.2f}" if not np.isnan(tw_ap) else "—",  f"上周${lw_ap:.2f}  {chg_pct(tw_ap,lw_ap)}",  good(tw_ap,lw_ap)),
        ("老用户ARPPU",f"${tw_oa:.2f}"if not np.isnan(tw_oa) else "—",  f"上周${lw_oa:.2f}  {chg_pct(tw_oa,lw_oa)}",  good(tw_oa,lw_oa)),
        ("首充ARPPU", f"${tw_fa:.2f}"if not np.isnan(tw_fa) else "—",   f"上周${lw_fa:.2f}  {chg_pct(tw_fa,lw_fa)}",  good(tw_fa,lw_fa)),
        ("总赠送金额",fmt(tw_gift),  f"上周{fmt(lw_gift)}  {chg_pct(tw_gift,lw_gift)}", good(tw_gift,lw_gift,False)),
        ("赠送/充值比",pct(tw_gr),   f"上周{pct(lw_gr)}  {chg_pp(tw_gr,lw_gr)}",        good(tw_gr,lw_gr,False)),
        ("首充转化率", pct(tw_fr,deci=1),  f"上周{pct(lw_fr,deci=1)}  {chg_pp(tw_fr,lw_fr,deci=1)}", good(tw_fr,lw_fr)),
        ("推广日均消耗",fmt(tw_cost), f"上周{fmt(lw_cost)}  {chg_pct(tw_cost,lw_cost)}", good(tw_cost,lw_cost,False)),
        ("充提差ROI", f"{tw_roi:.2f}x"if not np.isnan(tw_roi) else "—", f"上周{lw_roi:.2f}x  {chg_pct(tw_roi,lw_roi)}", good(tw_roi,lw_roi)),
    ], cols=4)

    S.append(insight_box(_overview_insights(K), C_LBLUE, C_LGRAY))
    return S

def _overview_insights(K):
    ins=[]
    tw_dep=_v(K,"tw_dep"); lw_dep=_v(K,"lw_dep")
    tw_fd=_v(K,"tw_fd");   lw_fd=_v(K,"lw_fd")
    tw_reg=_v(K,"tw_reg"); lw_reg=_v(K,"lw_reg")
    tw_fr=_v(K,"tw_fd_rate"); lw_fr=_v(K,"lw_fd_rate")
    tw_diff_r=_v(K,"tw_diff_r"); lw_diff_r=_v(K,"lw_diff_r")
    tw_gwl_r=_v(K,"tw_gwl_r"); lw_gwl_r=_v(K,"lw_gwl_r")
    tw_roi=_v(K,"tw_roi"); lw_roi=_v(K,"lw_roi")
    tw_cost=_v(K,"tw_cost"); lw_cost=_v(K,"lw_cost")
    tw_gr=_v(K,"tw_gift_r"); lw_gr=_v(K,"lw_gift_r")

    dep_chg = (tw_dep-lw_dep)/lw_dep*100 if not any(np.isnan(x) for x in [tw_dep,lw_dep]) and lw_dep>0 else nan
    fd_chg  = (tw_fd-lw_fd)/lw_fd*100    if not any(np.isnan(x) for x in [tw_fd,lw_fd])   and lw_fd>0  else nan
    reg_chg = (tw_reg-lw_reg)/lw_reg*100  if not any(np.isnan(x) for x in [tw_reg,lw_reg]) and lw_reg>0 else nan
    cost_chg= (tw_cost-lw_cost)/lw_cost*100 if not any(np.isnan(x) for x in [tw_cost,lw_cost]) and lw_cost>0 else nan

    # 充值健康
    if not np.isnan(dep_chg):
        if dep_chg>=3:
            ins.append(f"充值{fmt(tw_dep)}（{dep_chg:+.1f}%），规模增长；充提差率{pct(tw_diff_r)}（{chg_pp(tw_diff_r,lw_diff_r)}），资金循环改善。")
        elif dep_chg>=-3:
            ins.append(f"充值{fmt(tw_dep)}（{dep_chg:+.1f}%），与上周持平；充提差率{pct(tw_diff_r)}（{chg_pp(tw_diff_r,lw_diff_r)}）。")
        else:
            ins.append(f"充值{fmt(tw_dep)}（{dep_chg:+.1f}%），规模收缩，需排查推广缩量或活动力度减弱。")

    # 注册/首充下降拆解
    if not np.isnan(reg_chg) and (reg_chg<-5 or (not np.isnan(fd_chg) and fd_chg<-10)):
        fr_pp = (tw_fr-lw_fr)*100 if not any(np.isnan(x) for x in [tw_fr,lw_fr]) else nan
        parts=[]
        if not np.isnan(cost_chg) and cost_chg<-5:
            parts.append(f"推广消耗{cost_chg:+.1f}%（投放缩量）")
        if not np.isnan(fr_pp) and fr_pp<-2:
            parts.append(f"首充转化率{tw_fr*100:.1f}%（{fr_pp:+.1f}pp，漏斗效率恶化）")
        if not np.isnan(fd_chg) and not np.isnan(reg_chg) and fd_chg<reg_chg:
            parts.append("首充跌幅大于注册跌幅→转化率同步恶化，非纯流量问题")
        cause = "；".join(parts) if parts else "建议核查各总代本周投放量变化"
        ins.append(f"⚠ 注册{tw_reg:,.0f}（{reg_chg:+.1f}%），首充{tw_fd:,.0f}（{fd_chg:+.1f}%）双指标下滑：{cause}。")

    # 盈亏
    if not any(np.isnan(x) for x in [tw_gwl_r,lw_gwl_r]):
        pp=(tw_gwl_r-lw_gwl_r)*100
        if abs(pp)>=0.1:
            note="，需关注高赔游戏拖累" if pp<-0.2 else ""
            ins.append(f"盈亏率{pct(tw_gwl_r,deci=3)}（{pp:+.3f}pp）{note}。")

    # ROI
    if not any(np.isnan(x) for x in [tw_roi,lw_roi]):
        roi_chg=(tw_roi-lw_roi)/lw_roi*100
        if roi_chg>10:  ins.append(f"充提差ROI {tw_roi:.2f}x（{roi_chg:+.1f}%），推广效率显著提升。")
        elif roi_chg<-10: ins.append(f"充提差ROI {tw_roi:.2f}x（{roi_chg:+.1f}%），推广效率下降，建议复核渠道ROI拆分。")

    # 赠送偏高
    if not any(np.isnan(x) for x in [tw_gr,lw_gr]) and tw_gr>0.12:
        ins.append(f"赠送/充值比{pct(tw_gr)}（上周{pct(lw_gr)}），促销成本偏高，建议评估各活动ROI。")

    if not ins: ins.append("本周大盘各核心指标基本平稳，无重大异常。")
    return ins

# ══════════════════════════════════════════════════════
#  第二章：总代分析
# ══════════════════════════════════════════════════════
def build_agents(D: dict) -> list:
    S = sec_title("二、总代分析", C_BLUE)
    ag_plat  = D.get("ag_plat")
    ag_promo = D.get("ag_promo")
    ag_ret   = D.get("ag_ret")

    if ag_plat is None:
        S.append(P("⚠ 总代平台报表未加载，跳过本章。",9,False,C_AMBER))
        return S

    # 列名检测
    id_col = name_col = dep_col = gwl_col = reg_col = fd_col = arppu_col = None
    for c in ag_plat.columns:
        cs=str(c)
        if "总代" in cs and "ID" in cs.upper(): id_col=c
        elif "总代" in cs and ("名" in cs or "代理" in cs): name_col=c
        elif "充值金额" in cs: dep_col=c
        elif "公司输赢" in cs or "盈利" in cs: gwl_col=c
        elif "注册" in cs: reg_col=c
        elif "首充人数" in cs or "首充" in cs: fd_col=c
        elif "ARPPU" in cs or "人均" in cs: arppu_col=c

    if dep_col is None:
        S.append(P("⚠ 未找到充值金额列，请检查总代报表结构。",9,False,C_RED))
        return S

    # 按周期筛选（若有日期列）
    tw_ag = lw_ag = ag_plat
    for dc in ["日期","date"]:
        if dc in ag_plat.columns:
            try:
                tmp=ag_plat.copy()
                tmp[dc]=pd.to_datetime(tmp[dc],errors="coerce")
                tw_ag=tmp[tmp[dc].dt.strftime("%Y%m%d").between(THIS_WEEK[0],THIS_WEEK[1])]
                lw_ag=tmp[tmp[dc].dt.strftime("%Y%m%d").between(LAST_WEEK[0],LAST_WEEK[1])]
            except: pass
            break

    grp_col = name_col or id_col
    if grp_col is None:
        S.append(P("⚠ 未找到总代名称列。",9,False,C_RED))
        return S

    def _agg(df):
        cols={dep_col:"sum"}
        if gwl_col: cols[gwl_col]="sum"
        if reg_col: cols[reg_col]="sum"
        if fd_col:  cols[fd_col]="sum"
        if arppu_col: cols[arppu_col]="mean"
        for c in cols:
            if c in df.columns:
                df[c]=pd.to_numeric(df[c],errors="coerce")
        return df.groupby(grp_col).agg({k:v for k,v in cols.items() if k in df.columns}).reset_index()

    tw_sum = _agg(tw_ag)
    lw_sum = _agg(lw_ag)

    merged = tw_sum.merge(lw_sum[[grp_col,dep_col]], on=grp_col, how="outer",
                          suffixes=("_tw","_lw")).fillna(0)
    merged = merged.sort_values(f"{dep_col}_tw" if f"{dep_col}_tw" in merged.columns else dep_col, ascending=False)

    # 推广留存
    ret_d1_map = {}
    if ag_ret is not None:
        for c in ag_ret.columns:
            if "总代" in str(c) and "名" in str(c): rnm_col=c; break
            else: rnm_col=None
        for c in ag_ret.columns:
            if "1日" in str(c) or "第1日" in str(c) or "次日" in str(c): rd1_col=c; break
            else: rd1_col=None
        if rnm_col and rd1_col:
            for _,r in ag_ret.iterrows():
                nm=str(r[rnm_col])
                v=pd.to_numeric(r[rd1_col],errors="coerce")
                if not np.isnan(v):
                    ret_d1_map[nm]=v

    # 推广ROI
    roi_map={}
    if ag_promo is not None:
        for pc in ag_promo.columns:
            pcs=str(pc)
            if "总代" in pcs and ("名" in pcs or "代理" in pcs): prm_nm=pc; break
            else: prm_nm=None
        for pc in ag_promo.columns:
            if "消耗" in str(pc) or "花费" in str(pc): cost_c=pc; break
            else: cost_c=None
        if prm_nm and cost_c and dep_col:
            pr_grp=ag_promo.groupby(prm_nm).agg({cost_c:"sum"}).reset_index()
            dep_tw=tw_sum.set_index(grp_col)[dep_col].to_dict()
            for _,r in pr_grp.iterrows():
                nm=str(r[prm_nm])
                cost=pd.to_numeric(r[cost_c],errors="coerce")
                dep=dep_tw.get(nm,0)
                if cost>0 and dep>0:
                    roi_map[nm]=dep/cost*100

    S += sub_title("2.1  总代充值表现（本周 vs 上周）")

    dep_tw_col = f"{dep_col}_tw" if f"{dep_col}_tw" in merged.columns else dep_col
    dep_lw_col = f"{dep_col}_lw" if f"{dep_col}_lw" in merged.columns else dep_col

    rows=[]
    for _,r in merged.head(20).iterrows():
        nm   = str(r[grp_col])
        tw_d = float(r.get(dep_tw_col,0))
        lw_d = float(r.get(dep_lw_col,0))
        chg  = f"{(tw_d-lw_d)/lw_d*100:+.1f}%" if lw_d>0 else "新增"
        gwl_v= r.get(f"{gwl_col}_tw" if gwl_col and f"{gwl_col}_tw" in r.index else gwl_col, nan)
        gwl_s= f"${float(gwl_v):,.0f}" if not (isinstance(gwl_v,float) and np.isnan(gwl_v)) else "—"
        rp_r = fmt(tw_d)
        d1   = ret_d1_map.get(nm,nan)
        d1s  = f"{d1:.1f}%" if not np.isnan(d1) else "—"
        roi_v= roi_map.get(nm,nan)
        roi_s= f"{roi_v:.0f}%" if not np.isnan(roi_v) else "—"
        rows.append([nm, rp_r, fmt(lw_d), chg, gwl_s, d1s, roi_s])

    def ag_color(i,row):
        patches=[]
        try:
            chg_str=row[3]
            if chg_str.startswith("+") and chg_str!="新增": patches.append((3,C_BGGRN,C_GREEN))
            elif chg_str.startswith("-"): patches.append((3,C_BGRED,C_RED))
        except: pass
        return patches

    t=make_table(["总代名称","本周充值","上周充值","变化","公司输赢","首充次日留存","ROI"],
                 rows,
                 [CW*.28,CW*.11,CW*.11,CW*.10,CW*.10,CW*.14,CW*.10],
                 sz=8, col_fn=ag_color)
    S.append(t)
    S.append(Spacer(1,.15*cm))

    # 自动洞察
    if len(rows)>0:
        top=rows[0]; bot=sorted(rows, key=lambda r: _parse_num(r[3]))
        ins=[f"本周共{len(rows)}个总代活跃，头部总代「{top[0]}」充值{top[1]}（{top[3]}）。"]
        neg=[r for r in rows if r[3].startswith("-")]
        if neg: ins.append(f"其中{len(neg)}个总代充值环比下滑，需排查投放缩量或用户质量问题。")
        S.append(insight_box(ins,C_LBLUE,C_LGRAY))
    return S

def _parse_num(s):
    try:
        return float(str(s).replace("%","").replace("+","").replace(",",""))
    except: return 0

# ══════════════════════════════════════════════════════
#  第三章：用户分析
# ══════════════════════════════════════════════════════
def build_users(D: dict, K: dict) -> list:
    S = sec_title("三、用户分析", C_BLUE)

    # 3.1 VIP留存热图
    S += sub_title("3.1  VIP充值留存（近28天）")
    vip_dep = D.get("vip_dep")
    vip_act = D.get("vip_act")

    if vip_dep is not None:
        # 找VIP等级列和留存列
        vip_col=None
        for c in vip_dep.columns:
            if "VIP" in str(c) or "等级" in str(c): vip_col=c; break
        ret_cols=[c for c in vip_dep.columns if any(x in str(c) for x in ["1日","2日","3日","7日","14日","30日","次日"])]

        if vip_col and ret_cols:
            rows=[]
            for _,r in vip_dep.iterrows():
                row=[str(r[vip_col])]
                for rc in ret_cols[:6]:
                    v=pd.to_numeric(r[rc],errors="coerce")
                    row.append(f"{v:.1f}%" if not np.isnan(v) else "—")
                rows.append(row)
            t=make_table(["VIP等级"]+[str(c) for c in ret_cols[:6]],rows,
                         [CW*.18]+[CW*.13]*(min(6,len(ret_cols))),sz=8)
            S.append(t)
        else:
            S.append(P("VIP留存数据列名未匹配，请检查文件结构。",9,False,C_AMBER))
    else:
        S.append(P("VIP充值留存文件未加载。",9,False,C_GRAY))

    S.append(Spacer(1,.2*cm))

    # 3.2 头部充值用户
    S += sub_title("3.2  头部充值用户对比（Top20）")
    dep_tw = D.get("dep_tw")
    dep_lw = D.get("dep_lw")

    for label, df in [("本周",dep_tw),("上周",dep_lw)]:
        if df is None: continue
        # 找列
        uid_c=dep_c=gwl_c=None
        for c in df.columns:
            cs=str(c)
            if "账户" in cs or "用户ID" in cs: uid_c=c
            elif "充值金额" in cs: dep_c=c
            elif "公司输赢" in cs: gwl_c=c
        if not dep_c: continue
        df=df.copy()
        df[dep_c]=pd.to_numeric(df[dep_c],errors="coerce")
        top=df.nlargest(20,dep_c)

        rows=[]
        for i,(_,r) in enumerate(top.iterrows()):
            uid=str(r.get(uid_c,"—")) if uid_c else f"#{i+1}"
            dp=r.get(dep_c,0)
            gw=r.get(gwl_c,nan) if gwl_c else nan
            rows.append([uid, f"${float(dp):,.0f}",
                         f"${float(gw):,.0f}" if not np.isnan(gw) else "—"])

        S.append(P(f"◎ {label} Top20充值用户",8.5,True,C_BLUE))
        t=make_table(["账户ID","充值金额","公司输赢"],rows,
                     [CW*.5,CW*.25,CW*.25],sz=8)
        S.append(t)
        S.append(Spacer(1,.12*cm))

    # 3.3 Top提款用户对比
    S += sub_title("3.3  Top提款用户对比")
    wdr_tw=D.get("wdr_tw"); wdr_lw=D.get("wdr_lw")

    for label, df in [("本周",wdr_tw),("上周",wdr_lw)]:
        if df is None: continue
        uid_c=wdr_c=gwl_c=dep_c=None
        for c in df.columns:
            cs=str(c)
            if "账户" in cs or "用户ID" in cs: uid_c=c
            elif "提款金额" in cs or "提现" in cs: wdr_c=c
            elif "充值金额" in cs: dep_c=c
            elif "公司输赢" in cs: gwl_c=c
        if not wdr_c: continue
        df=df.copy()
        df[wdr_c]=pd.to_numeric(df[wdr_c],errors="coerce")
        top=df.nlargest(15,wdr_c)

        rows=[]
        for i,(_,r) in enumerate(top.iterrows()):
            uid=str(r.get(uid_c,"—")) if uid_c else f"#{i+1}"
            wd=r.get(wdr_c,0)
            dp=r.get(dep_c,nan) if dep_c else nan
            gw=r.get(gwl_c,nan) if gwl_c else nan
            diff=float(dp)-float(wd) if not any(np.isnan(x) for x in [float(dp) if dp else nan,float(wd)]) else nan
            rows.append([uid,f"${float(wd):,.0f}",
                         f"${float(dp):,.0f}" if not np.isnan(dp) else "—",
                         f"${diff:,.0f}" if not np.isnan(diff) else "—",
                         f"${float(gw):,.0f}" if not np.isnan(gw) else "—"])

        S.append(P(f"◎ {label} Top15提款用户",8.5,True,C_BLUE))
        t=make_table(["账户ID","提款金额","充值金额","充提差","公司输赢"],rows,
                     [CW*.35,CW*.16,CW*.16,CW*.16,CW*.17],sz=8)
        S.append(t)
        S.append(Spacer(1,.12*cm))

    # 自动洞察
    ins=[]
    tw_arppu=_v(K,"tw_arppu"); lw_arppu=_v(K,"lw_arppu")
    tw_oa=_v(K,"tw_o_arppu"); lw_oa=_v(K,"lw_o_arppu")
    if not any(np.isnan(x) for x in [tw_arppu,lw_arppu]):
        ins.append(f"全量ARPPU ${tw_arppu:.2f}（{chg_pct(tw_arppu,lw_arppu)}），"
                   f"老用户ARPPU ${tw_oa:.2f}（{chg_pct(tw_oa,lw_oa)}），付费用户质量{"改善" if tw_oa>lw_oa else "下降"}。")
    if dep_tw is not None:
        dep_c_=[c for c in dep_tw.columns if "充值金额" in str(c)]
        if dep_c_:
            top10_dep = pd.to_numeric(dep_tw[dep_c_[0]],errors="coerce").nlargest(10).sum()
            tot_dep   = pd.to_numeric(dep_tw[dep_c_[0]],errors="coerce").sum()
            if tot_dep>0:
                ins.append(f"Top10充值用户贡献 ${top10_dep:,.0f}，占总充值{top10_dep/tot_dep*100:.1f}%，大R集中度{"偏高，需维护" if top10_dep/tot_dep>0.3 else "正常"}。")
    if not ins: ins.append("本周用户结构基本稳定，详见VIP及头部用户明细。")
    S.append(insight_box(ins,C_LBLUE,C_LGRAY))
    return S

# ══════════════════════════════════════════════════════
#  第四章：游戏分析
# ══════════════════════════════════════════════════════
def build_games(D: dict) -> list:
    S = sec_title("四、游戏分析", C_BLUE)
    mfr_df = D.get("mfr")
    game_tw = D.get("game_tw")
    game_lw = D.get("game_lw")

    # 4.1 厂商投注份额
    S += sub_title("4.1  厂商投注份额（本周 vs 上周）")
    if mfr_df is not None:
        mfr_c=bet_c=gwl_c=None
        for c in mfr_df.columns:
            cs=str(c)
            if "厂商" in cs: mfr_c=c
            elif "投注金额" in cs: bet_c=c
            elif "公司输赢" in cs: gwl_c=c

        if mfr_c and bet_c:
            # 按周期筛
            tw_mfr=lw_mfr=mfr_df
            for dc in ["日期","date"]:
                if dc in mfr_df.columns:
                    try:
                        tmp=mfr_df.copy()
                        tmp[dc]=pd.to_datetime(tmp[dc],errors="coerce")
                        tw_mfr=tmp[tmp[dc].dt.strftime("%Y%m%d").between(THIS_WEEK[0],THIS_WEEK[1])]
                        lw_mfr=tmp[tmp[dc].dt.strftime("%Y%m%d").between(LAST_WEEK[0],LAST_WEEK[1])]
                    except: pass
                    break

            def _mfr_agg(df):
                df=df.copy()
                df[bet_c]=pd.to_numeric(df[bet_c],errors="coerce")
                if gwl_c: df[gwl_c]=pd.to_numeric(df[gwl_c],errors="coerce")
                agg={bet_c:"sum"}
                if gwl_c: agg[gwl_c]="sum"
                return df.groupby(mfr_c).agg(agg).reset_index()

            tw_m=_mfr_agg(tw_mfr); lw_m=_mfr_agg(lw_mfr)
            merged=tw_m.merge(lw_m[[mfr_c,bet_c]],on=mfr_c,how="outer",suffixes=("_tw","_lw")).fillna(0)
            bet_tw_col=f"{bet_c}_tw"; bet_lw_col=f"{bet_c}_lw"
            merged=merged.sort_values(bet_tw_col,ascending=False)
            tot_tw=merged[bet_tw_col].sum()
            tot_lw=merged[bet_lw_col].sum()

            rows=[]
            for _,r in merged.head(12).iterrows():
                nm=abbr(r[mfr_c])
                bt=float(r.get(bet_tw_col,0))
                bl=float(r.get(bet_lw_col,0))
                share_tw=bt/tot_tw*100 if tot_tw>0 else 0
                share_lw=bl/tot_lw*100 if tot_lw>0 else 0
                gwl_v=r.get(f"{gwl_c}_tw" if gwl_c and f"{gwl_c}_tw" in r.index else gwl_c, nan)
                gwl_r_v=float(gwl_v)/bt if gwl_c and bt>0 and not np.isnan(gwl_v) else nan
                rows.append([nm, fmt(bt,wan=True),
                             f"{share_tw:.1f}%",f"{share_lw:.1f}%",
                             f"{share_tw-share_lw:+.1f}pp",
                             pct(gwl_r_v,deci=2) if not np.isnan(gwl_r_v) else "—"])

            def mfr_color(i,row):
                pp=row[4]
                if pp.startswith("+"): return [(4,C_BGGRN,C_GREEN)]
                if pp.startswith("-"): return [(4,C_BGRED,C_RED)]
                return []

            t=make_table(["厂商","本周投注","本周份额","上周份额","份额变化","盈亏率"],
                         rows,[CW*.14,CW*.16,CW*.12,CW*.12,CW*.12,CW*.12],sz=8,col_fn=mfr_color)
            S.append(t)
            S.append(Spacer(1,.15*cm))

            # 饼图
            try:
                top8=merged.head(8)
                others_bet=merged.iloc[8:][bet_tw_col].sum() if len(merged)>8 else 0
                labels=[abbr(r[mfr_c]) for _,r in top8.iterrows()]+["其他"]
                sizes=[float(r.get(bet_tw_col,0)) for _,r in top8.iterrows()]+[others_bet]
                fig,ax=plt.subplots(figsize=(7,4))
                ax.pie(sizes,labels=labels,autopct="%1.1f%%",startangle=90,
                       textprops={"fontsize":8})
                ax.set_title("厂商投注份额分布（本周）",fontsize=10)
                cp=_save("mfr_pie.png")
                img=_img(cp,w=CW*.65)
                if img:
                    S.append(Table([[img]],colWidths=[CW],
                                   style=TableStyle([("ALIGN",(0,0),(-1,-1),"CENTER")])))
            except Exception as e:
                print(f"  ⚠ 厂商饼图生成失败: {e}")
    else:
        S.append(P("厂商投注数据未加载。",9,False,C_GRAY))

    # 4.2 Top30游戏
    S += sub_title("4.2  Top30游戏排名（投注金额）")
    if game_tw is not None:
        gn_c=mf_c=bt_c=us_c=gwl_c=gwl_r_c=wr_c=None
        for c in game_tw.columns:
            cs=str(c)
            if "游戏名" in cs: gn_c=c
            elif "厂商" in cs: mf_c=c
            elif "投注金额" in cs: bt_c=c
            elif "玩家" in cs or "人数" in cs: us_c=c
            elif "公司输赢" in cs: gwl_c=c
            elif "盈亏率" in cs: gwl_r_c=c
            elif "赢家率" in cs or "赢家" in cs: wr_c=c

        if bt_c:
            game_tw_cp=game_tw.copy()
            game_tw_cp[bt_c]=pd.to_numeric(game_tw_cp[bt_c],errors="coerce")
            top30=game_tw_cp.nlargest(30,bt_c)

            # 上周排名对比
            rank_lw={}
            if game_lw is not None and gn_c and bt_c in game_lw.columns:
                lw_cp=game_lw.copy()
                lw_cp[bt_c]=pd.to_numeric(lw_cp[bt_c],errors="coerce")
                for i,(_,r) in enumerate(lw_cp.nlargest(30,bt_c).iterrows()):
                    rank_lw[str(r.get(gn_c,""))]=i+1

            rows=[]
            for rank_tw,(_, r) in enumerate(top30.iterrows(),1):
                gn=str(r.get(gn_c,"—")) if gn_c else f"Game{rank_tw}"
                mf=abbr(r.get(mf_c,"—")) if mf_c else "—"
                bt=float(r.get(bt_c,0))
                us=int(r.get(us_c,0)) if us_c else 0
                gwl=r.get(gwl_c,nan) if gwl_c else nan
                gwl_r=r.get(gwl_r_c,nan) if gwl_r_c else nan
                lw_rk=rank_lw.get(gn,nan)
                rk_chg=f"{lw_rk-rank_tw:+.0f}" if not np.isnan(lw_rk) else "新入"
                rows.append([rank_tw, gn, mf, fmt(bt,wan=True),
                             f"{us:,}",
                             f"${float(gwl):,.0f}" if not np.isnan(gwl) else "—",
                             pct(float(gwl_r)) if not np.isnan(gwl_r) else "—",
                             rk_chg])

            def game_color(i,row):
                patches=[]
                try:
                    gwl_str=str(row[5])
                    if gwl_str.startswith("$-"): patches.append((5,C_BGRED,C_RED))
                except: pass
                try:
                    rk=row[7]
                    if str(rk).startswith("+"): patches.append((7,C_BGGRN,C_GREEN))
                    elif str(rk).startswith("-"): patches.append((7,C_BGRED,C_RED))
                except: pass
                return patches

            t=make_table(["排名","游戏名称","厂商","投注金额","玩家数","公司输赢","盈亏率","排名变化"],
                         rows,
                         [CW*.06,CW*.25,CW*.10,CW*.12,CW*.09,CW*.12,CW*.10,CW*.10],
                         sz=7.5,col_fn=game_color)
            S.append(t)
        else:
            S.append(P("游戏报表列名未匹配。",9,False,C_AMBER))
    else:
        S.append(P("本周游戏报表未加载。",9,False,C_GRAY))

    # 洞察
    ins=["游戏分析：Tada系列占据投注份额主导地位，Fortune系列持续为主力游戏。"]
    if mfr_df is not None:
        ins.append("建议关注高份额厂商的盈亏率变化趋势，对盈亏率持续偏低的游戏进行RTP参数复核。")
    S.append(insight_box(ins,C_LBLUE,C_LGRAY))
    return S

# ══════════════════════════════════════════════════════
#  第五章：活动分析
# ══════════════════════════════════════════════════════
def build_activities(D: dict) -> list:
    S = sec_title("五、活动分析", C_BLUE)

    # 5.1 整体赠送对比
    S += sub_title("5.1  整体赠送活动（本周 vs 上周）")
    gift_tot = D.get("gift_tot")
    if gift_tot is not None:
        # 寻找关键列
        rows=[]
        nm_c=tw_c=lw_c=None
        for c in gift_tot.columns:
            cs=str(c)
            if "活动" in cs or "名称" in cs: nm_c=c
            elif any(d in cs for d in [THIS_WEEK[0][:6],THIS_WEEK[1][:6],"本周"]): tw_c=c
            elif any(d in cs for d in [LAST_WEEK[0][:6],LAST_WEEK[1][:6],"上周"]): lw_c=c

        # 若没有明确本周/上周列，取数值列
        num_cols=[c for c in gift_tot.columns if pd.api.types.is_numeric_dtype(gift_tot[c]) or
                  gift_tot[c].apply(lambda x: pd.to_numeric(str(x).replace(",","").replace("$",""),errors="coerce")).notna().mean()>0.5]

        for _,r in gift_tot.iterrows():
            nm=str(r.get(nm_c,"—")) if nm_c else "—"
            tw_v=r.get(tw_c,nan) if tw_c else (r[num_cols[0]] if num_cols else nan)
            lw_v=r.get(lw_c,nan) if lw_c else (r[num_cols[1]] if len(num_cols)>1 else nan)
            tw_n=pd.to_numeric(str(tw_v).replace(",","").replace("$",""),errors="coerce")
            lw_n=pd.to_numeric(str(lw_v).replace(",","").replace("$",""),errors="coerce")
            chg_s=f"{(tw_n-lw_n)/abs(lw_n)*100:+.1f}%" if not any(np.isnan(x) for x in [tw_n,lw_n]) and lw_n!=0 else "—"
            rows.append([nm, f"${tw_n:,.0f}" if not np.isnan(tw_n) else str(tw_v),
                         f"${lw_n:,.0f}" if not np.isnan(lw_n) else str(lw_v), chg_s])

        if rows:
            t=make_table(["活动名称","本周赠送","上周赠送","变化"],rows,
                         [CW*.40,CW*.20,CW*.20,CW*.20],sz=8)
            S.append(t)
            S.append(Spacer(1,.12*cm))
    else:
        S.append(P("整体赠送活动数据未加载。",9,False,C_GRAY))

    # 5.2 各赠送活动明细
    S += sub_title("5.2  各赠送活动明细")
    gift_dtl = D.get("gift_dtl")
    if gift_dtl is not None:
        # 取前10行展示
        rows=[]
        for _,r in gift_dtl.head(15).iterrows():
            row=[str(v)[:25] for v in r.values[:6]]
            rows.append(row)
        if rows:
            hdrs=[str(c)[:12] for c in gift_dtl.columns[:6]]
            cw_eq=CW/min(6,len(hdrs))
            t=make_table(hdrs,rows,[cw_eq]*min(6,len(hdrs)),sz=7.5)
            S.append(t)
            S.append(Spacer(1,.12*cm))
    else:
        S.append(P("各赠送活动明细未加载。",9,False,C_GRAY))

    # 5.3 存款来源
    S += sub_title("5.3  存款来源分析")
    dep_src = D.get("dep_src")
    if dep_src is not None:
        rows=[]
        for _,r in dep_src.head(12).iterrows():
            row=[str(v)[:20] for v in r.values[:5]]
            rows.append(row)
        if rows:
            hdrs=[str(c)[:15] for c in dep_src.columns[:5]]
            cw_eq=CW/min(5,len(hdrs))
            t=make_table(hdrs,rows,[cw_eq]*min(5,len(hdrs)),sz=8)
            S.append(t)
    else:
        S.append(P("存款来源数据未加载。",9,False,C_GRAY))

    tw_gr=_v({},""+" "); lw_gr=_v({},"")
    ins=["活动分析：赠送活动是用户充值的重要激励因素，需持续监控赠送金额与充值转化的关系。",
         "建议：对ROI<1x的赠送活动压缩预算；对首充激活效果好的活动增加曝光和触达频次。"]
    S.append(insight_box(ins,C_AMBER,C_BGYEL))
    return S

# ══════════════════════════════════════════════════════
#  第六章：道具专题分析
# ══════════════════════════════════════════════════════
def build_tools(D: dict) -> list:
    S = sec_title("六、道具专题分析", C_BLUE)

    tool_tw = D.get("tool_tw")
    tool_lw = D.get("tool_lw")
    tool_map= D.get("tool_map")
    fdr     = D.get("fdr")

    # 6.1 道具使用对比
    S += sub_title("6.1  道具使用量（本周 vs 上周）")
    if tool_tw is not None or tool_lw is not None:
        # 合并本周/上周
        def _agg_tool(df, label):
            if df is None: return None
            nm_c=cnt_c=usd_c=None
            for c in df.columns:
                cs=str(c)
                if "道具" in cs or "名称" in cs: nm_c=c
                elif "使用次数" in cs or "次数" in cs: cnt_c=c
                elif "金额" in cs or "USD" in cs: usd_c=c
            if nm_c is None: return None
            agg={} 
            if cnt_c: agg[cnt_c]="sum"
            if usd_c: agg[usd_c]="sum"
            if not agg: return None
            for c in agg: df[c]=pd.to_numeric(df[c],errors="coerce")
            r=df.groupby(nm_c).agg(agg).reset_index()
            r.columns=[nm_c]+[f"{col}_{label}" for col in list(agg.keys())]
            return r

        tw_agg=_agg_tool(tool_tw,"tw")
        lw_agg=_agg_tool(tool_lw,"lw")

        if tw_agg is not None:
            nm_c=tw_agg.columns[0]
            if lw_agg is not None:
                merged=tw_agg.merge(lw_agg,on=nm_c,how="outer").fillna(0)
            else:
                merged=tw_agg
            merged=merged.sort_values(merged.columns[1],ascending=False)

            rows=[]
            for _,r in merged.head(15).iterrows():
                nm=str(r[nm_c])
                vals=[str(v)[:12] for v in r.values[1:]]
                rows.append([nm]+vals)
            if rows:
                hdrs=["道具名称"]+[c for c in merged.columns[1:]]
                cw_eq=CW/len(hdrs)
                t=make_table(hdrs,rows,[cw_eq]*len(hdrs),sz=7.5)
                S.append(t)
                S.append(Spacer(1,.12*cm))
        else:
            S.append(P("道具使用数据列名未匹配。",9,False,C_AMBER))
    else:
        S.append(P("道具使用数据未加载。",9,False,C_GRAY))

    # 6.2 道具对应活动
    S += sub_title("6.2  道具归属活动")
    if tool_map is not None:
        rows=[[str(v)[:25] for v in r.values[:4]] for _,r in tool_map.head(20).iterrows()]
        if rows:
            hdrs=[str(c)[:15] for c in tool_map.columns[:4]]
            cw_eq=CW/min(4,len(hdrs))
            t=make_table(hdrs,rows,[cw_eq]*min(4,len(hdrs)),sz=8)
            S.append(t)
            S.append(Spacer(1,.12*cm))
    else:
        S.append(P("道具对应活动文件未加载。",9,False,C_GRAY))

    # 6.3 首次充值活动用户留存
    S += sub_title("6.3  首次充值活动用户留存")
    if fdr is not None:
        ret_cols=[c for c in fdr.columns if any(x in str(c) for x in ["1日","2日","3日","7日","次日","留存"])]
        nm_c=None
        for c in fdr.columns:
            if "活动" in str(c) or "名称" in str(c) or "总代" in str(c): nm_c=c; break

        if nm_c and ret_cols:
            rows=[]
            for _,r in fdr.iterrows():
                row=[str(r[nm_c])]
                for rc in ret_cols[:7]:
                    v=pd.to_numeric(r[rc],errors="coerce")
                    row.append(f"{v:.1f}%" if not np.isnan(v) else "—")
                rows.append(row)
            hdrs=["活动/总代"]+[str(c)[:6] for c in ret_cols[:7]]
            cws=[CW*.25]+[CW*.75/min(7,len(ret_cols))]*min(7,len(ret_cols))
            t=make_table(hdrs,rows,cws,sz=8)
            S.append(t)
        else:
            # 直接展示前5行前8列
            rows=[[str(v)[:15] for v in r.values[:8]] for _,r in fdr.head(10).iterrows()]
            hdrs=[str(c)[:10] for c in fdr.columns[:8]]
            if rows:
                cw_eq=CW/len(hdrs)
                t=make_table(hdrs,rows,[cw_eq]*len(hdrs),sz=7.5)
                S.append(t)
    else:
        S.append(P("首充活动用户留存数据未加载。",9,False,C_GRAY))

    ins=["道具活动对次日充值有显著正向影响，24小时内触达效果最佳。",
         "建议对D1/D2留存用户设置阶梯式再充值道具激励；对D3后流失用户做专项召回。"]
    S.append(insight_box(ins,C_LBLUE,C_LGRAY))
    return S

# ══════════════════════════════════════════════════════
#  第七章：风险专项分析（修复版）
# ══════════════════════════════════════════════════════
def build_risk(D: dict, K: dict) -> list:
    S = sec_title("七、用户游戏风险专项分析", colors.HexColor("#7c2d12"))

    top500 = D.get("top500_tw")
    game_tw= D.get("game_tw")

    # 7.1 Top500提款用户总览
    S += sub_title("7.1  Top500提款用户总览")
    if top500 is not None and len(top500):
        dep_c=wdr_c=gwl_c=uid_c=bet_c=None
        for c in top500.columns:
            cs=str(c)
            if "账户" in cs or "用户ID" in cs: uid_c=c
            elif "充值金额" in cs: dep_c=c
            elif "提款金额" in cs or "提现" in cs: wdr_c=c
            elif "公司输赢" in cs: gwl_c=c
            elif "投注金额" in cs: bet_c=c

        if wdr_c: top500[wdr_c]=pd.to_numeric(top500[wdr_c],errors="coerce")
        if gwl_c: top500[gwl_c]=pd.to_numeric(top500[gwl_c],errors="coerce")

        tot_tx=top500[wdr_c].sum() if wdr_c else 0
        tot_win=top500[gwl_c].sum() if gwl_c else 0
        wns=(top500[gwl_c]<0).sum() if gwl_c else 0
        hr=top500[top500[gwl_c]<-10000] if gwl_c else pd.DataFrame()

        S += kpi_grid([
            ("Top500提款总额", fmt(tot_tx),     "本周合计",                      None),
            ("平台净赔付",     fmt(abs(tot_win)),"平台向该群体净赔",              tot_win>0),
            ("赢家比例",       f"{wns/len(top500)*100:.1f}%", f"{wns}赢/{len(top500)-wns}输", None),
            ("高风险用户",     f"{len(hr)}人",   "公司净输>$10,000",              len(hr)>3),
        ],cols=4)

        # 高风险用户明细
        if len(hr):
            S += sub_title("高风险用户明细（公司净输>$10,000）")
            hr_s=hr.sort_values(gwl_c).head(10)
            rows=[]
            for _,r in hr_s.iterrows():
                uid=str(r.get(uid_c,"—")) if uid_c else "—"
                dp=r.get(dep_c,0) if dep_c else 0
                wd=r.get(wdr_c,0) if wdr_c else 0
                gw=r.get(gwl_c,0)
                bt=r.get(bet_c,nan) if bet_c else nan
                rows.append([uid,f"${float(dp):,.0f}",f"${float(wd):,.0f}",
                             f"${float(gw):,.0f}",
                             f"${float(bt):,.0f}" if not np.isnan(bt) else "—",
                             "🔴即时审核"])
            def hr_c(i,row): return [(5,C_RED,C_WHITE)]
            t=make_table(["账户ID","充值","提款","公司输赢","投注金额","建议"],rows,
                         [CW*.24,CW*.13,CW*.13,CW*.13,CW*.13,CW*.18],sz=8,col_fn=hr_c)
            S.append(t)
            S.append(Spacer(1,.15*cm))
    else:
        S.append(P("Top500提款用户数据未加载。",9,False,C_GRAY))

    # 7.2 高危游戏专项（修复：简称统一+列宽加宽）
    S += sub_title("7.2  高危游戏专项（盈亏率<-5% 或 赢家率>60%）")
    if game_tw is not None and len(game_tw):
        gn_c=mf_c=us_c=bt_c=gwl_c=gwl_r_c=wr_c=ab_c=None
        for c in game_tw.columns:
            cs=str(c)
            if "游戏名" in cs:    gn_c=c
            elif "厂商" in cs:    mf_c=c
            elif "玩家" in cs or "人数" in cs: us_c=c
            elif "投注金额" in cs: bt_c=c
            elif "公司输赢" in cs: gwl_c=c
            elif "盈亏率" in cs:   gwl_r_c=c
            elif "赢家率" in cs or "赢家" in cs: wr_c=c
            elif "人均投注" in cs: ab_c=c

        risk_df=game_tw.copy()
        cond=pd.Series(False,index=risk_df.index)
        if gwl_r_c:
            r=pd.to_numeric(risk_df[gwl_r_c],errors="coerce").fillna(0)
            cond|=(r<-0.05)
        if wr_c:
            wr=pd.to_numeric(risk_df[wr_c],errors="coerce").fillna(0)
            cond|=(wr>0.60)
        risk_df=risk_df[cond]

        if len(risk_df):
            if gwl_c: risk_df=risk_df.sort_values(gwl_c)
            rows=[]
            for _,r in risk_df.head(15).iterrows():
                gn=str(r.get(gn_c,"—")) if gn_c else "—"
                mf=abbr(r.get(mf_c,"—")) if mf_c else "—"   # ← 统一简称
                us=r.get(us_c,0) if us_c else 0
                bt=r.get(bt_c,0) if bt_c else 0
                gw=r.get(gwl_c,nan) if gwl_c else nan
                gr=r.get(gwl_r_c,nan) if gwl_r_c else nan
                wr_v=r.get(wr_c,nan) if wr_c else nan
                ab=r.get(ab_c,nan) if ab_c else nan

                gr_f=float(gr) if not np.isnan(gr) else 0
                wr_f=float(wr_v) if not np.isnan(wr_v) else 0
                if gr_f<-0.1 or wr_f>0.80: lv="极高"
                elif gr_f<-0.05 or wr_f>0.65: lv="高"
                else: lv="关注"

                rows.append([gn, mf,
                             f"{int(float(us)):,}" if us else "—",
                             fmt(float(bt),wan=True) if bt else "—",
                             f"${float(gw):,.0f}" if not np.isnan(gw) else "—",
                             f"{gr_f*100:.1f}%" if not np.isnan(gr) else "—",
                             f"{wr_f*100:.0f}%" if not np.isnan(wr_v) else "—",
                             f"${float(ab):,.0f}" if not np.isnan(ab) else "—",
                             lv])

            def risk_c(i,row):
                lv=row[-1]
                bg={" 极高":C_RED,"高":C_ORED,"关注":colors.HexColor("#fef3c7")}.get(lv,C_LGRAY)
                fg=C_WHITE if lv in ("极高","高") else C_DARK
                patches=[(8,bg,fg)]
                if row[4].startswith("$-"): patches.append((4,C_BGRED,C_RED))
                return patches

            # 游戏名称列宽24%，不截断
            t=make_table(
                ["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"],
                rows,
                [CW*.24,CW*.09,CW*.07,CW*.09,CW*.10,CW*.08,CW*.08,CW*.09,CW*.08],
                sz=7.5,col_fn=risk_c)
            S.append(t)
            S.append(Spacer(1,.12*cm))

            # 高危洞察
            risk_ins=[]
            if gwl_c:
                tot_loss=risk_df[gwl_c].sum()
                if tot_loss<0:
                    risk_ins.append(f"高危游戏合计平台损失 ${abs(tot_loss):,.0f}，需重点监控。")
            if wr_c:
                extreme=risk_df[pd.to_numeric(risk_df[wr_c],errors="coerce")>0.75]
                if len(extreme) and gn_c:
                    gn_ex=str(extreme.iloc[0][gn_c])
                    wr_ex=pd.to_numeric(extreme.iloc[0][wr_c],errors="coerce")
                    risk_ins.append(f"「{gn_ex}」赢家率{wr_ex*100:.0f}%，疑似集群套利，建议立即实施单账户赢额上限并审查近7天爆奖记录。")
            if mf_c and gwl_c:
                mfr_loss=risk_df.groupby(mf_c)[gwl_c].sum()
                worst=mfr_loss.idxmin()
                if mfr_loss.min()<0:
                    risk_ins.append(f"厂商「{abbr(worst)}」旗下高危游戏亏损最严重，建议优先向厂商提交RTP复核申请。")
            risk_ins.append("行动：对盈亏率<-10%游戏暂停高倍率奖励触发；赢家率>60%游戏实施$5,000/日赢额上限。")
            S.append(insight_box(risk_ins,C_RED,C_BGRED))
        else:
            S.append(P("✓ 本周无明显高危游戏（盈亏率均>-5%且赢家率<60%）",9,False,C_GREEN))
    else:
        S.append(P("游戏数据未加载。",9,False,C_GRAY))

    return S

# ══════════════════════════════════════════════════════
#  第八章：总结与行动建议（全面优化）
# ══════════════════════════════════════════════════════
def build_conclusion(K: dict, D: dict) -> list:
    S = sec_title("八、总结与行动建议", C_BLUE)

    tw_dep=_v(K,"tw_dep"); lw_dep=_v(K,"lw_dep")
    tw_fd=_v(K,"tw_fd");   lw_fd=_v(K,"lw_fd")
    tw_reg=_v(K,"tw_reg"); lw_reg=_v(K,"lw_reg")
    tw_diff_r=_v(K,"tw_diff_r"); lw_diff_r=_v(K,"lw_diff_r")
    tw_fr=_v(K,"tw_fd_rate"); lw_fr=_v(K,"lw_fd_rate")
    tw_arppu=_v(K,"tw_arppu"); lw_arppu=_v(K,"lw_arppu")
    tw_gwl_r=_v(K,"tw_gwl_r"); lw_gwl_r=_v(K,"lw_gwl_r")
    tw_roi=_v(K,"tw_roi"); lw_roi=_v(K,"lw_roi")
    tw_cost=_v(K,"tw_cost"); lw_cost=_v(K,"lw_cost")
    tw_gr=_v(K,"tw_gift_r")

    def _nan(*vs): return any(isinstance(x,float) and np.isnan(x) for x in vs)

    dep_chg = (tw_dep-lw_dep)/lw_dep*100 if not _nan(tw_dep,lw_dep) and lw_dep>0 else nan
    fd_chg  = (tw_fd-lw_fd)/lw_fd*100    if not _nan(tw_fd,lw_fd)   and lw_fd>0  else nan
    reg_chg = (tw_reg-lw_reg)/lw_reg*100  if not _nan(tw_reg,lw_reg) and lw_reg>0 else nan
    cost_chg= (tw_cost-lw_cost)/lw_cost*100 if not _nan(tw_cost,lw_cost) and lw_cost>0 else nan
    arppu_chg=(tw_arppu-lw_arppu)/lw_arppu*100 if not _nan(tw_arppu,lw_arppu) and lw_arppu>0 else nan

    # ── 8.1 本周亮点（严格正向才放）
    hl=[]
    if not _nan(tw_diff_r,lw_diff_r) and tw_diff_r>lw_diff_r:
        hl.append(f"充提差率{pct(tw_diff_r)}（{chg_pp(tw_diff_r,lw_diff_r)}），"
                  f"充提差{fmt(_v(K,'tw_diff'))}，资金循环效率改善。")
    if not _nan(dep_chg) and dep_chg>=2:
        hl.append(f"充值{fmt(tw_dep)}（{dep_chg:+.1f}%），规模环比增长，整体营收稳健。")
    if not _nan(arppu_chg) and arppu_chg>=5:
        hl.append(f"全量ARPPU ${tw_arppu:.2f}（{arppu_chg:+.1f}%），付费用户单均价值提升。")
    if not _nan(tw_roi,lw_roi) and tw_roi>lw_roi and tw_roi>=1.4:
        hl.append(f"充提差ROI {tw_roi:.2f}x（{chg_pct(tw_roi,lw_roi)}），推广效率改善。")
    if not _nan(tw_arppu) and not _nan(_v(K,"tw_fd_arppu"),_v(K,"lw_fd_arppu")):
        fa_chg=((_v(K,"tw_fd_arppu")-_v(K,"lw_fd_arppu"))/_v(K,"lw_fd_arppu")*100)
        if fa_chg>=10:
            hl.append(f"首充ARPPU ${_v(K,'tw_fd_arppu'):.2f}（{fa_chg:+.1f}%），首充用户付费质量提升。")
    if not hl:
        hl.append(f"本周大盘整体{"平稳" if _nan(dep_chg) or abs(dep_chg)<3 else "承压"}，充提差率维持在{pct(tw_diff_r)}。")

    # ── 8.2 核心问题（有因果分析）
    pb=[]
    # 注册/首充双降
    if not _nan(reg_chg) and reg_chg<-5:
        fr_pp=(tw_fr-lw_fr)*100 if not _nan(tw_fr,lw_fr) else nan
        parts=[]
        if not _nan(cost_chg) and cost_chg<-5:
            parts.append(f"推广消耗{cost_chg:+.1f}%（投放缩量为主因）")
        if not _nan(fr_pp) and fr_pp<-2:
            parts.append(f"首充转化率{tw_fr*100:.1f}%（{fr_pp:+.1f}pp，漏斗效率恶化）")
        sym = "注册跌幅与首充跌幅同步，属纯流量缩量" if (_nan(fd_chg) or abs(fd_chg-reg_chg)<3) else f"首充跌幅（{fd_chg:+.1f}%）远大于注册跌幅（{reg_chg:+.1f}%），转化率同步恶化，非单纯流量问题"
        cause="；".join(parts) if parts else "原因待查"
        pb.append(f"【流量漏斗收缩】注册{tw_reg:,.0f}（{reg_chg:+.1f}%），首充{tw_fd:,.0f}（{fd_chg:+.1f}%）。"
                  f" 驱动因素：{cause}。"
                  f" 结构判断：{sym}。"
                  f" 建议：按总代拆分本周注册量，定位缩量来源；同步检查首充活动力度。")
    elif not _nan(fd_chg) and fd_chg<-10:
        pb.append(f"【转化率恶化】注册相对稳定，但首充{tw_fd:,.0f}（{fd_chg:+.1f}%），"
                  f"首充转化率{pct(tw_fr,deci=1)}（{chg_pp(tw_fr,lw_fr,deci=1)}）。"
                  f"建议优化注册后24h内触达策略（首充礼包/短信推送）。")
    # 盈亏率
    if not _nan(tw_gwl_r,lw_gwl_r):
        pp=(tw_gwl_r-lw_gwl_r)*100
        if pp<-0.2:
            pb.append(f"【盈亏率下滑】{pp:.3f}pp，高赔率/高赢家率游戏是主要拖累，"
                      f"建议优先对盈亏率<-10%游戏做RTP复核，并实施赢额上限。")
    # 活动成本
    if not _nan(tw_gr) and tw_gr>0.12:
        pb.append(f"【促销成本偏高】赠送/充值比{pct(tw_gr)}，建议评估各活动ROI，压缩低效预算。")

    # ── 亮点展示
    S.append(P("▌ 本周亮点",11,True,C_GREEN))
    S.append(Spacer(1,.08*cm))
    hl_t=Table([[P(f"• {h}",9,False,C_DARK)] for h in hl],colWidths=[CW-1*cm],
               style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_BGGRN),
                                 ("BOX",(0,0),(-1,-1),1,C_GREEN),
                                 ("LEFTPADDING",(0,0),(-1,-1),10),
                                 ("TOPPADDING",(0,0),(-1,-1),4),
                                 ("BOTTOMPADDING",(0,0),(-1,-1),4)]))
    S.append(hl_t)
    S.append(Spacer(1,.18*cm))

    if pb:
        S.append(P("▌ 核心问题与风险",11,True,C_RED))
        S.append(Spacer(1,.08*cm))
        pb_t=Table([[P(f"• {p}",9,False,C_DARK)] for p in pb],colWidths=[CW-1*cm],
                   style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_BGRED),
                                     ("BOX",(0,0),(-1,-1),1,C_RED),
                                     ("LEFTPADDING",(0,0),(-1,-1),10),
                                     ("TOPPADDING",(0,0),(-1,-1),4),
                                     ("BOTTOMPADDING",(0,0),(-1,-1),4)]))
        S.append(pb_t)
        S.append(Spacer(1,.18*cm))

    # ── 8.3 行动建议
    S.append(P("▌ 行动建议",11,True,C_BLUE))
    S.append(Spacer(1,.08*cm))

    acts=_gen_actions(K,D)
    act_rows=[[a["p"],a["item"],a["detail"]] for a in acts]
    def act_c(i,row):
        bg={"[紧急]":C_RED,"[本周]":C_AMBER,"[次周]":C_LBLUE}.get(row[0],C_LGRAY)
        return [(0,bg,C_WHITE)]
    t=make_table(["优先级","建议事项","执行说明与数据依据"],act_rows,
                 [CW*.10,CW*.22,CW*.68],sz=8.5,col_fn=act_c)
    S.append(t)
    return S

def _gen_actions(K,D):
    acts=[]
    tw_reg=_v(K,"tw_reg"); lw_reg=_v(K,"lw_reg")
    tw_fd=_v(K,"tw_fd");   lw_fd=_v(K,"lw_fd")
    tw_fr=_v(K,"tw_fd_rate"); lw_fr=_v(K,"lw_fd_rate")
    tw_gwl_r=_v(K,"tw_gwl_r"); lw_gwl_r=_v(K,"lw_gwl_r")
    tw_gr=_v(K,"tw_gift_r")
    tw_cost=_v(K,"tw_cost"); lw_cost=_v(K,"lw_cost")

    def _nan(*vs): return any(isinstance(x,float) and np.isnan(x) for x in vs)

    # 高风险用户
    top500=D.get("top500_tw")
    if top500 is not None:
        gwl_c=next((c for c in top500.columns if "公司输赢" in str(c)),None)
        if gwl_c:
            top500[gwl_c]=pd.to_numeric(top500[gwl_c],errors="coerce")
            hr=top500[top500[gwl_c]<-10000]
            if len(hr):
                uid_c=next((c for c in top500.columns if "账户" in str(c) or "用户ID" in str(c)),None)
                top_uid=str(hr.sort_values(gwl_c).iloc[0].get(uid_c,"—")) if uid_c else "—"
                acts.append({"p":"[紧急]","item":f"审查{len(hr)}名高风险用户",
                             "detail":f"账户{top_uid}等{len(hr)}名用户公司净输均>$10,000，"
                                      f"建议人工复核提款申请，核查IP/设备指纹及奖励来源。"})

    # 高危游戏
    game_tw=D.get("game_tw")
    if game_tw is not None:
        wr_c=next((c for c in game_tw.columns if "赢家率" in str(c) or "赢家" in str(c)),None)
        gwl_r_c=next((c for c in game_tw.columns if "盈亏率" in str(c)),None)
        gn_c=next((c for c in game_tw.columns if "游戏名" in str(c)),None)
        if wr_c:
            wr=pd.to_numeric(game_tw[wr_c],errors="coerce")
            ex=game_tw[wr>0.75]
            if len(ex) and gn_c:
                gn=str(ex.iloc[0][gn_c])
                acts.append({"p":"[紧急]","item":"高赢家率游戏套利防控",
                             "detail":f"「{gn}」等{len(ex)}款游戏赢家率>75%，立即实施单账户赢额上限$3,000/日，"
                                      f"并向厂商提交RTP复核申请。"})
        if gwl_r_c:
            gr=pd.to_numeric(game_tw[gwl_r_c],errors="coerce")
            bad=game_tw[gr<-0.10]
            if len(bad):
                acts.append({"p":"[本周]","item":"高亏损游戏RTP复核",
                             "detail":f"{len(bad)}款游戏盈亏率<-10%，建议暂停高倍率奖励触发，同步提交RTP验证申请。"})

    # 注册/首充下降
    reg_chg=(tw_reg-lw_reg)/lw_reg*100 if not _nan(tw_reg,lw_reg) and lw_reg>0 else nan
    fd_chg=(tw_fd-lw_fd)/lw_fd*100     if not _nan(tw_fd,lw_fd)   and lw_fd>0  else nan
    cost_chg=(tw_cost-lw_cost)/lw_cost*100 if not _nan(tw_cost,lw_cost) and lw_cost>0 else nan
    fr_pp=(tw_fr-lw_fr)*100 if not _nan(tw_fr,lw_fr) else nan

    if not np.isnan(reg_chg) and reg_chg<-5:
        if not np.isnan(cost_chg) and cost_chg<-5:
            acts.append({"p":"[本周]","item":"推广缩量原因排查",
                         "detail":f"推广消耗{cost_chg:+.1f}%，注册量随之{reg_chg:+.1f}%，"
                                  f"需按总代拆分投放量，确认是主动削减还是渠道效率问题。"})
        if not np.isnan(fr_pp) and fr_pp<-2:
            acts.append({"p":"[本周]","item":"首充转化率提升",
                         "detail":f"首充转化率{tw_fr*100:.1f}%（{fr_pp:+.1f}pp），"
                                  f"建议A/B测试首充礼包面值，优化注册后24h内触达（短信/站内信）。"})

    # 盈亏率
    gwl_pp=(tw_gwl_r-lw_gwl_r)*100 if not _nan(tw_gwl_r,lw_gwl_r) else nan
    if not np.isnan(gwl_pp) and gwl_pp<-0.2:
        acts.append({"p":"[本周]","item":"盈亏率改善",
                     "detail":f"盈亏率{gwl_pp:.3f}pp，对高危游戏实施赢额限制，"
                              f"同步优化游戏组合策略，提升整体盈利率。"})

    # 活动ROI
    if not np.isnan(tw_gr) and tw_gr>0.12:
        acts.append({"p":"[次周]","item":"活动ROI评审",
                     "detail":f"赠送/充值比{pct(tw_gr)}，拉取各活动参与率×留存率交叉分析，"
                              f"压缩ROI<1x活动预算。"})

    if not acts:
        acts.append({"p":"[本周]","item":"常规监控",
                     "detail":"本周无重大风险，维持日常监控节奏，持续关注高赢家率游戏和头部用户充提行为。"})
    return acts

# ══════════════════════════════════════════════════════
#  页眉页脚
# ══════════════════════════════════════════════════════
def header_footer(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(C_BLUE)
    canvas.rect(0,PH-1.2*cm,PW,1.2*cm,fill=1,stroke=0)
    canvas.setFont(FONT_B,10); canvas.setFillColor(C_WHITE)
    tw_str=(f"{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — "
            f"{THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    lw_str=(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — "
            f"{LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    canvas.drawString(MARGIN,PH-.85*cm,f"MX 平台数据周报   {tw_str}")
    canvas.setFont(FONT_N,8)
    canvas.drawRightString(PW-MARGIN,PH-.85*cm,lw_str)
    canvas.setFillColor(C_GRAY); canvas.setFont(FONT_N,7.5)
    canvas.drawRightString(PW-MARGIN,.8*cm,f"第 {doc.page} 页")
    canvas.restoreState()

# ══════════════════════════════════════════════════════
#  主函数
# ══════════════════════════════════════════════════════
def main():
    print("="*60)
    print(f"MX 平台数据周报 v2.1 完整版")
    print(f"本周：{THIS_WEEK[0]}-{THIS_WEEK[1]}")
    print(f"对比：{LAST_WEEK[0]}-{LAST_WEEK[1]}")
    print("="*60)

    if not DATA_ROOT.exists():
        raise RuntimeError(
            f"\n❌ 数据目录不存在：{DATA_ROOT}\n"
            f"请修改脚本顶部 DATA_ROOT 为实际路径。")

    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

    print("\n▶ 1/4  加载数据...")
    D=load_all(DATA_ROOT)

    print("▶ 2/4  提取KPI...")
    K=extract_kpi(D.get("plat"))

    out_name=f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    out_path=OUTPUT_DIR/out_name

    print("▶ 3/4  构建PDF章节...")
    doc=SimpleDocTemplate(str(out_path),pagesize=A4,
                          leftMargin=MARGIN,rightMargin=MARGIN,
                          topMargin=1.8*cm,bottomMargin=1.5*cm)

    story=[]
    # 封面
    story+=[Spacer(1,3*cm),
            P("MX  平台数据周报",28,True,C_BLUE,TA_CENTER),
            Spacer(1,.5*cm),
            P(f"{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",
              16,False,C_GRAY,TA_CENTER),
            Spacer(1,.3*cm),
            P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",
              12,False,C_GRAY,TA_CENTER),
            Spacer(1,.5*cm),
            HRFlowable(width="50%",thickness=2,color=C_AMBER,hAlign="CENTER"),
            PageBreak()]

    print("   章节 1：大盘核心数据");  story+=build_overview(K);         story.append(PageBreak())
    print("   章节 2：总代分析");       story+=build_agents(D);            story.append(PageBreak())
    print("   章节 3：用户分析");       story+=build_users(D,K);           story.append(PageBreak())
    print("   章节 4：游戏分析");       story+=build_games(D);             story.append(PageBreak())
    print("   章节 5：活动分析");       story+=build_activities(D);        story.append(PageBreak())
    print("   章节 6：道具专题");       story+=build_tools(D);             story.append(PageBreak())
    print("   章节 7：风险专项");       story+=build_risk(D,K);            story.append(PageBreak())
    print("   章节 8：总结建议");       story+=build_conclusion(K,D)

    print("▶ 4/4  渲染PDF（可能需要1-3分钟）...")
    doc.build(story,onFirstPage=header_footer,onLaterPages=header_footer)

    sz=out_path.stat().st_size/1024
    print(f"\n✅ 完成！{out_path}  ({sz:.0f} KB)")
    return str(out_path)

if __name__=="__main__":
    main()

  ✔ 字体：MsYaHei (msyh.ttc)


FileNotFoundError: [WinError 3] 系统找不到指定的路径。: '\\tmp\\mx_charts'

In [11]:
"""
MX 平台数据周报 v7.5.3 — 本周: 20260529-20260604
修改说明：
  - 适配本次上传文件路径（DATA_ROOT 指向上传目录）
  - 文件匹配键名已按实际文件名更新
  - THIS_WEEK / LAST_WEEK 已更新为本周周期
  - 总代平台报表为空时做容错处理（推广报表有数据即可）
"""
from pathlib import Path
import sys, os, io, warnings
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec, matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image, PageBreak, HRFlowable, KeepTogether)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# ── 路径配置 ─────────────────────────────────────────────
DATA_ROOT  = Path(r"D:\周报更新版\MX")   # ← 修改为你的数据根目录
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")   # ← 修改为输出目录

THIS_WEEK  = ("20260529", "20260604")
LAST_WEEK  = ("20260522", "20260528")
REPORT_END = THIS_WEEK[1]

RET_TARGETS = {"次留": 21.0, "3留": 15.0, "7留": 11.0, "14留": 8.0, "30留": 6.0}

FN, FNB = "WQY", "WQYB"
FILES: dict = {}

C_BLUE   = colors.HexColor("#1d4ed8");  C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669");  C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706");  C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b");  C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white;                C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1");  C_ROW    = colors.HexColor("#f8fafc")
C_TEAL   = colors.HexColor("#0f766e");  C_TARGET = colors.HexColor("#0369a1")

PW, PH = A4
MARGIN  = 1.6 * cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
CR_HIGH, CR_LOW = 17, 5
MFR_SHORT = {"Rectangle":"RG","Pragmatic Play":"PP","PG Soft":"PG","PlayTech":"PT",
             "Originals":"自研","Fat Panda":"FP","Tada":"Tada"}

TRUNCATE = {"首充次日复充率":1,"首充次日复投率":1,"首充当日复充率":0,
            "首充7日复充率":6,"首充30日复充率":999,"首充2日复充率":1,"首充3日复充率":2}

# ── 工具函数 ──────────────────────────────────────────────
def shorten_mfr(n):
    if not isinstance(n, str): return str(n)
    for k, v in MFR_SHORT.items():
        if k in n: return v
    return n[:8]

def ret_end(week_start: str, lag: int):
    e = (datetime.strptime(REPORT_END, "%Y%m%d") - timedelta(days=lag)).strftime("%Y%m%d")
    return e if e >= week_start else None

def fmt_lbl(d):
    return f"{d[4:6]}/{d[6:]}" if d else "-"

def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c not in ("日期","总代.名称","name_总代","总代.ID")]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o):   return (n-o)/abs(o)*100 if o and o != 0 else 0.0
def pct_vec(ns, os):
    return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop else s
    nz = v[v > 0]; return nz.mean() if len(nz) else v.mean()

def wavg_series(vals, weights):
    ok = vals.notna() & (weights > 0)
    if not ok.any(): return np.nan
    return float(np.average(vals[ok], weights=weights[ok]))

def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

def setup():
    candidates = [
        # Windows
        r"C:\Windows\Fonts\msyh.ttc",
        r"C:\Windows\Fonts\msyhbd.ttc",
        r"C:\Windows\Fonts\simhei.ttf",
        r"C:\Windows\Fonts\simsun.ttc",
        # Linux
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        # macOS
        "/System/Library/Fonts/PingFang.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
    ]
    font_path = next((p for p in candidates if Path(p).exists()), None)
    if not font_path:
        from matplotlib import font_manager as fm
        for f in fm.fontManager.ttflist:
            if any(k in f.name for k in ["YaHei","Hei","SimSun","WenQuanYi","Noto Sans CJK","PingFang"]):
                if Path(f.fname).exists():
                    font_path = f.fname; break
    if not font_path:
        raise FileNotFoundError(
            "未找到中文字体！请确认系统已安装微软雅黑(msyh.ttc)或黑体(simhei.ttf)。"
        )
    print(f"  ▶ 字体：{font_path}")
    ttc = font_path.lower().endswith(".ttc")
    pdfmetrics.registerFont(TTFont(FN,  font_path, subfontIndex=0) if ttc else TTFont(FN,  font_path))
    try:    pdfmetrics.registerFont(TTFont(FNB, font_path, subfontIndex=1) if ttc else TTFont(FNB, font_path))
    except: pdfmetrics.registerFont(TTFont(FNB, font_path, subfontIndex=0) if ttc else TTFont(FNB, font_path))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(font_path)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == font_path]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

def resolve_files(root):
    global FILES
    tw0, tw1 = THIS_WEEK
    lw0, lw1 = LAST_WEEK
    all_files = []
    try:
        for h in root.rglob("*"):
            try:
                if h.is_file() and not h.name.startswith("~$"):
                    all_files.append(h)
            except (PermissionError, OSError):
                pass
    except Exception as e:
        print(f"  ⚠️  扫描目录出错: {e}")

    def find(kws):
        kws = kws if isinstance(kws, list) else [kws]
        hits = [h for h in all_files if all(k in h.name for k in kws)]
        return max(hits, key=lambda h: h.stat().st_mtime) if hits else None

    km = {
        "platform":      ["平台报表_USD"],
        "daily":         ["日报-大盘日报_USD"],
        "retention":     ["整体", "首充留存"],
        "agent_plat":    ["平台报表-总代_USD"],
        "agent_promo":   ["推广报表-总代_USD"],
        "agent_ret":     ["首充充值留存_20260508"],
        "vip":           ["VIP报表_USD"],
        "dt_tw":         [f"top提款用户_全量数据_{tw0}"],
        "dt_lw":         [f"top提款用户_全量数据_{lw0}"],
        "dc_tw":         [f"头部充值用户_全量数据_{tw0}"],
        "dc_lw":         [f"头部充值用户_全量数据_{lw0}"],
        "pref_tw":       [f"本周top500提款用户游戏偏好_全量数据_{tw0}"],
        "pref_lw":       [f"上周top500提款用户游戏偏好_全量数据_{lw0}"],
        "mfr":           ["厂商投注数据_全量数据"],
        "game_tw":       ["游戏报表-详情_USD_本周"],
        "game_lw":       ["游戏报表-详情_USD_上周"],
        "gift":          ["各赠送活动_全量数据"],
        "tool_map":      ["道具对应活动"],
        "vip_ret_chg":   ["VIP充值-充值_近28天"],
        "vip_ret_act":   ["VIP充值-活跃_近28天"],
        "tool_tw":       ["本周道具使用情况"],
        "tool_lw":       ["上周道具使用情况"],
        "first_dep_ret": ["首次充值活动用户充值留存情况"],
    }
    fail = 0
    for key, kws in km.items():
        p = find(kws)
        if p:
            FILES[key] = p
            print(f"     ✅ [{key:15s}] {p.name}")
        else:
            print(f"     ❌ [{key:15s}] 找不到含{kws}的文件")
            fail += 1
    print(f"  ▶ 文件匹配完成（{len(km)-fail}/{len(km)}）\n")

# ── PDF 样式工具 ─────────────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.4,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5), HRFlowable(width="100%",thickness=1.5,color=C_BLUE2),
                         Spacer(1,3), P(f"■  {text}",10,True,C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}",8.5,False,C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8, extra_style=None):
    full = PW - 2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),       ("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2),      ("BOTTOMPADDING",(0,0),(-1,-1),2),
        ("LEFTPADDING",(0,0),(-1,-1),2),     ("RIGHTPADDING",(0,0),(-1,-1),2),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    if extra_style:
        for cmd in extra_style: st.add(*cmd)
    hrow = [P(h,fsize,True,C_WHITE,TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]),fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c,(list,tuple)) else P(str(c),fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t,bold=False,clr=colors.black,align=TA_LEFT): return (t,bold,clr,align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup=good_up
    c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{'+' if v>=0 else ''}{v:.{d}f}%",False,c,TA_RIGHT)
def gclr(v,t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)
def fret(v): return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

def kpi_card4(items, cols=4):
    fw = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,_ in items:
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([[P(label,7.5,False,C_GRAY)],[P(str(tv),14,True,C_DARK)],
                       [P(f"上周：{lv}",7.5,False,C_GRAY)],
                       [P(f"{'+' if chg>=0 else ''}{chg:.1f}%",8,True,pclr)]],
                      colWidths=[fw],
                      style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
                                        ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
                                        ("LEFTPADDING",(0,0),(-1,-1),8),
                                        ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(fw,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════════════════════════
def load_platform():
    df = pd.read_excel(FILES["platform"]); df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]; lw_d = dd[dd["日期"].between(*LAST_WEEK)]
    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["首充转化率","首充当日复充率","首充次日复充率",
              "首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])

    # ★ 充提差率 = sum(充提差)/sum(充值金额)
    K["tw_充提差比"] = tw["充提差"].sum() / tw["充值金额"].sum() * 100 if tw["充值金额"].sum() > 0 else np.nan
    K["lw_充提差比"] = lw["充提差"].sum() / lw["充值金额"].sum() * 100 if lw["充值金额"].sum() > 0 else np.nan
    K["pct_充提差比"] = pct(K["tw_充提差比"], K["lw_充提差比"])
    # ★ 盈亏率 = sum(公司输赢)/sum(投注金额)
    K["tw_盈亏率"] = tw["公司输赢"].sum() / tw["投注金额"].sum() * 100 if tw["投注金额"].sum() > 0 else np.nan
    K["lw_盈亏率"] = lw["公司输赢"].sum() / lw["投注金额"].sum() * 100 if lw["投注金额"].sum() > 0 else np.nan
    K["pct_盈亏率"] = pct(K["tw_盈亏率"], K["lw_盈亏率"])

    # 推广消耗：固定剔除本周最后一天
    tw_d_asc = tw_d.sort_values("日期")
    tw_cost_valid  = tw_d_asc["真实消耗"].iloc[:-1]
    lw_cost_days   = len(lw_d)
    tw_cost_days   = len(tw_cost_valid)
    K["tw_真实消耗_日均"]   = tw_cost_valid.sum() / tw_cost_days  if tw_cost_days > 0 else 0
    K["lw_真实消耗_日均"]   = lw_d["真实消耗"].sum() / lw_cost_days if lw_cost_days > 0 else 0
    K["tw_真实消耗"]         = tw_cost_valid.sum()
    K["lw_真实消耗"]         = lw_d["真实消耗"].sum()
    K["tw_真实消耗_有效天"]  = tw_cost_days
    K["pct_真实消耗"]        = pct(K["tw_真实消耗_日均"], K["lw_真实消耗_日均"])

    tw_sorted = tw.sort_values("日期")
    tw_cd_valid = tw_sorted["充提差"].iloc[:tw_cost_days]
    K["tw_充提差_日均_roi"]  = tw_cd_valid.sum() / tw_cost_days if tw_cost_days > 0 else 0
    lw_sorted = lw.sort_values("日期")
    K["lw_充提差_日均_roi"]  = lw_sorted["充提差"].sum() / lw_cost_days if lw_cost_days > 0 else 0
    K["tw_充提差ROI"] = (K["tw_充提差_日均_roi"] / K["tw_真实消耗_日均"]
                         if K["tw_真实消耗_日均"] > 0 else 0)
    K["lw_充提差ROI"] = (K["lw_充提差_日均_roi"] / K["lw_真实消耗_日均"]
                         if K["lw_真实消耗_日均"] > 0 else 0)
    K["pct_充提差ROI"] = pct(K["tw_充提差ROI"], K["lw_充提差ROI"])

    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])

    last14 = df.tail(14)
    trend = {"dates": [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
             "充值": last14["充值金额"].tolist(), "提现": last14["提现金额"].tolist(),
             "充提差比": last14["充提差比"].tolist(), "公司输赢": last14["公司输赢"].tolist(),
             "首充": last14["首充人数"].tolist(), "注册": last14["注册人数"].tolist()}
    return K, trend


def load_dashboard_retention():
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str)
    specs = [("nd","首充2日复充率",1),("td","首充3日复充率",2),
             ("sd","首充7日复充率",6),("30d","首充30日复充率",29)]
    for _,col,_ in specs:
        if col in dd.columns:
            dd[col] = pd.to_numeric(dd[col].astype(str).str.replace("%","").str.strip(), errors="coerce")
    R = {}
    for key, col, lag in specs:
        for ws, we, prefix in [(THIS_WEEK[0],THIS_WEEK[1],"tw_"), (LAST_WEEK[0],LAST_WEEK[1],"lw_")]:
            ec = ret_end(ws, lag)
            sub = dd[(dd["日期"]>=ws) & (dd["日期"]<=we)]
            if ec: sub = sub[sub["日期"]<=ec]
            R[f"{prefix}{key}"] = float(sub[col].mean()) if len(sub) and col in sub else np.nan
        R[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        R[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)
    return R


def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
    daily["ds"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
    dr = daily[daily["指标"]=="留存率"].copy()
    du = daily[daily["指标"]=="留存人数"].copy()
    for c in ["第1日","第2日","第3日","第6日","第7日"]:
        if c in dr.columns:
            dr[c] = pd.to_numeric(dr[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
    def _d(s): return datetime.strptime(s,"%Y%m%d")
    def _f(d): return d.strftime("%Y-%m-%d")
    def _l(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s,tw_e = _d(THIS_WEEK[0]),_d(THIS_WEEK[1])
    lw_s,lw_e = _d(LAST_WEEK[0]),_d(LAST_WEEK[1])
    w2e = lw_s-timedelta(days=1); w2s = w2e-timedelta(days=6)
    w1e = w2s-timedelta(days=1); w1s = w1e-timedelta(days=6)
    weeks = [(f"第1周\n{_l(w1s,w1e)}",_f(w1s),_f(w1e)),
             (f"第2周\n{_l(w2s,w2e)}",_f(w2s),_f(w2e)),
             (f"上周\n{_l(lw_s,lw_e)}",_f(lw_s),_f(lw_e)),
             (f"本周\n{_l(tw_s,tw_e)}",_f(tw_s),_f(tw_e))]
    result = []
    for wk,s,e in weeks:
        mr = dr[(dr["ds"]>=s)&(dr["ds"]<=e)]; mu = du[(du["ds"]>=s)&(du["ds"]<=e)]
        row = {"week": wk, "users": mu["充值成功事件用户数"].sum() if "充值成功事件用户数" in mu.columns else 0}
        for col in ["第1日","第2日","第3日","第6日","第7日"]:
            rs = mr[col].values if col in mr.columns else np.array([])
            us_col = "充值成功事件用户数" if "充值成功事件用户数" in mu.columns else None
            us = mu[us_col].values if us_col else np.ones(len(rs))
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = float(np.average(rs[v],weights=us[v])) if v.sum()>0 else np.nan
            else: row[col] = float(np.nanmean(rs)) if len(rs)>0 else np.nan
        result.append(row)
    return result


def load_agents():
    # 推广报表（有数据）
    dr = pd.read_excel(FILES["agent_promo"]); dr["日期"]=dr["日期"].astype(str)
    dr=to_num(dr)
    tw_r=dr[dr["日期"].between(*THIS_WEEK)]; lw_r=dr[dr["日期"].between(*LAST_WEEK)]

    # 总代平台报表（本次可能为空，用推广报表的充值金额替代）
    dp = pd.read_excel(FILES["agent_plat"]); dp["日期"]=dp["日期"].astype(str)
    dp=to_num(dp)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]; lw_p=dp[dp["日期"].between(*LAST_WEEK)]

    sa=["充值金额","提现金额","充提差","首充金额","首充人数","注册人数","充值人数","投注金额","公司输赢","总赠送金额"]

    def agg_p(d):
        g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sa if c in d.columns}).reset_index()
        for c in ["充提差比","首充次日充值留存"]:
            if c in d.columns: g=g.merge(d.groupby("总代.ID")[c].mean().rename(c),on="总代.ID",how="left")
        g["充提差率"]=g["充提差"]/g["充值金额"]*100; return g
    tw_pa=agg_p(tw_p); lw_pa=agg_p(lw_p)

    sr=["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        avail=[c for c in sr if c in d.columns]
        g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in avail}).reset_index()
        if "总消耗" in g.columns and "一级首充人数" in g.columns:
            g["一级首充成本"]=g["总消耗"]/g["一级首充人数"].replace(0,np.nan)
        else:
            g["一级首充成本"]=np.nan
        return g

    tw_ra=agg_r(tw_r); lw_ra=agg_r(lw_r)
    lw_ra["lw_fc_cost"]=lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan) if "总消耗" in lw_ra.columns else np.nan

    merge_key = "总代.ID" if "总代.ID" in tw_pa.columns else "总代.名称"
    lw_merge_cols = [merge_key]
    for c in ["充值金额","注册人数","充提差率"]:
        if c in lw_pa.columns: lw_merge_cols.append(c)
    lw_sub = lw_pa[lw_merge_cols].rename(columns={
        "充值金额":"lw_充值","注册人数":"lw_注册","充提差率":"lw_充提差率"})

    m=tw_pa.merge(lw_sub, on=merge_key, how="left")
    m=m.merge(tw_ra[[merge_key,"总消耗","一级首充成本","一级首充人数"]] if "总消耗" in tw_ra.columns
              else tw_ra[[merge_key,"一级首充人数"]], on=merge_key, how="left")
    m=m.merge(lw_ra[[merge_key,"lw_fc_cost"]] if "lw_fc_cost" in lw_ra.columns
              else lw_ra[[merge_key]], on=merge_key, how="left")
    if "注册人数" in m.columns and "lw_注册" in m.columns:
        m["注册环比"]=pct_vec(m["注册人数"],m["lw_注册"])
    else:
        m["注册环比"]=0.0
    if "充值金额" not in m.columns: m["充值金额"]=0
    return m.sort_values("充值金额",ascending=False)


def load_agent_ret():
    df = pd.read_csv(FILES["agent_ret"])
    # 首充充值留存文件：列名为"初始事件的发生时间"，"1日"/"2日"等
    time_col = "初始事件的发生时间" if "初始事件的发生时间" in df.columns else "初始事件发生时间"
    df["_d"] = df[time_col].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"] = df["_d"].str.replace("-","").fillna("")

    # 列名检测：可能是"第1日"或"1日"
    col_1 = "第1日" if "第1日" in df.columns else "1日"
    col_2 = "第2日" if "第2日" in df.columns else "2日"
    col_6 = "第7日" if "第7日" in df.columns else ("第6日" if "第6日" in df.columns else "7日")

    for c in [col_1, col_2, col_6]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(), errors="coerce")

    user_col = "首充用户数" if "首充用户数" in df.columns else "充值成功事件用户数"
    if user_col in df.columns:
        df[user_col] = pd.to_numeric(df[user_col], errors="coerce").fillna(0)
    else:
        df[user_col] = 1

    name_col = "name_总代" if "name_总代" in df.columns else "总代.名称"
    ind_col  = "总代"      if "总代" in df.columns else "总代.ID"

    daily = df[df["_yyyymmdd"].str.match(r"^\d{8}$",na=False)].copy()
    daily_ret = daily[daily["指标"]=="留存率"].reset_index(drop=True) if "指标" in daily.columns else daily

    ends = {}
    for key, lag in [("nd",1),("3d",2),("7d",6)]:
        ends[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        ends[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)

    def _slice(ws, we, ec):
        sub = daily_ret[daily_ret["_yyyymmdd"].between(ws,we)]
        if ec: sub = sub[sub["_yyyymmdd"] <= min(ec,we)]
        return sub

    slices = {
        "tw_nd": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_nd_end"]),
        "tw_3d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_3d_end"]),
        "tw_7d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_7d_end"]),
        "lw_nd": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_nd_end"]),
        "lw_3d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_3d_end"]),
        "lw_7d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_7d_end"]),
    }
    col_map = {"tw_nd":col_1,"tw_3d":col_2,"tw_7d":col_6,
               "lw_nd":col_1,"lw_3d":col_2,"lw_7d":col_6}

    def _wavg_by_agent(df_sub, col):
        res = {}
        if name_col not in df_sub.columns or col not in df_sub.columns: return res
        for nm, grp in df_sub.groupby(name_col):
            v = wavg_series(grp[col], grp[user_col])
            if not np.isnan(v): res[nm] = v
        return res

    per_agent = {k: _wavg_by_agent(slices[k], col_map[k]) for k in slices}

    id_map = {}
    if name_col in daily_ret.columns and ind_col in daily_ret.columns:
        for nm, grp in daily_ret.groupby(name_col):
            vals = pd.to_numeric(grp[ind_col], errors="coerce").dropna().values
            if len(vals): id_map[nm] = int(vals[0])

    all_names = set().union(*[set(v) for v in per_agent.values()])
    ret = {}
    for nm in all_names:
        ret[nm] = {k: per_agent[k].get(nm, np.nan) for k in per_agent}
        ret[nm]["agent_id"] = id_map.get(nm)
    return ret, ends


def load_vip():
    df=pd.read_excel(FILES["vip"]); df["日期"]=df["日期"].astype(str)
    num=["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df=to_num(df,num)
    tw=df[df["日期"].between(*THIS_WEEK)]; lw=df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()


def load_vip_retention():
    res={}
    for fk,rt in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df=pd.read_csv(FILES[fk])
        time_col2 = "初始事件发生时间" if "初始事件发生时间" in df.columns else "初始事件的发生时间"
        df["ds"]=df[time_col2].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
        df["yyyymmdd"]=df["ds"].str.replace("-","").fillna("")
        for c in ["1日","2日","3日","7日"]:
            if c in df.columns:
                df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
        n_col = "充值成功事件用户数" if "充值成功事件用户数" in df.columns else (df.columns[2] if len(df.columns)>2 else None)
        df["n"]=pd.to_numeric(df[n_col],errors="coerce") if n_col else 1
        df["vip"]=pd.to_numeric(df["vip_level"],errors="coerce") if "vip_level" in df.columns else np.nan
        ind_col2 = "指标" if "指标" in df.columns else None
        if ind_col2:
            daily=df[(df[ind_col2]=="留存率")&df["yyyymmdd"].notna()&df["vip"].notna()]
        else:
            daily=df[df["yyyymmdd"].notna()&df["vip"].notna()]
        tw_d=daily[daily["yyyymmdd"].between(*THIS_WEEK)]; lw_d=daily[daily["yyyymmdd"].between(*LAST_WEEK)]
        def wavg(d,col):
            r={}
            if col not in d.columns: return r
            for v,g in d.groupby("vip"):
                s=g[g[col].notna()]
                if len(s): r[int(v)]=float(np.average(s[col].values,weights=s["n"].values))
            return r
        cols_avail = [c for c in ["1日","2日","3日","7日"] if c in daily.columns]
        res[rt]={"tw":{c:wavg(tw_d,c) for c in cols_avail},
                 "lw":{c:wavg(lw_d,c) for c in cols_avail}}
    return res


def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])

    # 列名统一化：本周文件有"对比时段 充值金额"等，上周数据取对比时段列
    def _std_dc(df, is_lw=False):
        if is_lw:
            # 上周文件：用"对比时段"列作为主列（即再上周）——实际用本期列
            pass
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c].astype(str).str.replace(",",""), errors="coerce")
        return df

    dc_tw = _std_dc(dc_tw); dc_lw = _std_dc(dc_lw)
    dt_tw = _std_dc(dt_tw); dt_lw = _std_dc(dt_lw)

    # 上周比较数据：从上周文件取本期主列
    lw_dc_chg = dc_lw["充值金额"] if "充值金额" in dc_lw.columns else pd.Series([0]*len(dc_lw))
    lw_dt_tx  = dt_lw["提款金额"] if "提款金额" in dt_lw.columns else pd.Series([0]*len(dt_lw))

    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]
    total_c=tw_p["充值金额"].sum(); total_t=tw_p["提现金额"].sum()
    actual_cr=(total_c-total_t)/total_c*100 if total_c>0 else 0
    tot_c=dc_tw["充值金额"].sum() if "充值金额" in dc_tw.columns else 1
    tot_t=dt_tw["提款金额"].sum() if "提款金额" in dt_tw.columns else 1

    dep_t=[]
    for t in [1,10,50,100,200,500]:
        tw=dc_tw.head(t); lw=dc_lw.head(t)
        tc=tw["充值金额"].sum() if "充值金额" in tw.columns else 0
        tt=tw["提款金额"].sum() if "提款金额" in tw.columns else 0
        lc=lw["充值金额"].sum() if "充值金额" in lw.columns else 0
        lt=lw["提款金额"].sum() if "提款金额" in lw.columns else 0
        win=tw["公司输赢"].sum() if "公司输赢" in tw.columns else 0
        dep_t.append({"tier":f"Top{t}","tw_chg":tc,"lw_chg":lc,"tw_avg":tc/t,"lw_avg":lc/t,
            "tw_cr":(tc-tt)/tc*100 if tc>0 else 0,"lw_cr":(lc-lt)/lc*100 if lc>0 else 0,
            "tw_win":win,"占全量":tc/tot_c*100 if tot_c>0 else 0})

    wdr_t=[]
    for t in [1,10,50,100,200,500]:
        tw2=dt_tw.head(t); lw2=dt_lw.head(t)
        tt2=tw2["提款金额"].sum() if "提款金额" in tw2.columns else 0
        tc2=tw2["充值金额"].sum() if "充值金额" in tw2.columns else 0
        lt2=lw2["提款金额"].sum() if "提款金额" in lw2.columns else 0
        lc2=lw2["充值金额"].sum() if "充值金额" in lw2.columns else 0
        wns=(tw2["公司输赢"]<0).sum() if "公司输赢" in tw2.columns else 0
        act_sum=tw2["活动奖励"].sum() if "活动奖励" in tw2.columns else 0
        act_pct=act_sum/(tc2+act_sum)*100 if (tc2+act_sum)>0 else 0
        excl_c=total_c-tc2; excl_t=total_t-tt2
        excl_cr=(excl_c-excl_t)/excl_c*100 if excl_c>0 else 0
        wdr_t.append({"tier":f"Top{t}","tw_tx":tt2,"lw_tx":lt2,"tw_avg":tt2/t,"lw_avg":lt2/t,
            "tw_cr":(tc2-tt2)/tc2*100 if tc2>0 else 0,"lw_cr":(lc2-lt2)/lc2*100 if lc2>0 else 0,
            "赢家":wns,"总数":t,"赢家率":wns/t*100,"活动占比":act_pct,
            "占全量":tt2/tot_t*100 if tot_t>0 else 0,"大盘影响":excl_cr-actual_cr})
    return dep_t,wdr_t,dc_tw.head(200),dt_tw.head(20)


def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); dl=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); dl["阶段汇总"]=dl["阶段汇总"].apply(clean)
    mfr_col = "show_name_厂商标签id" if "show_name_厂商标签id" in df.columns else df.columns[1]
    game_col = "游戏名称" if "游戏名称" in df.columns else df.columns[2]
    ind_col2 = "分析指标" if "分析指标" in df.columns else df.columns[3]
    bet=df[df[ind_col2]=="投注金额"]; win=df[df[ind_col2]=="公司输赢"]; bl=dl[dl[ind_col2]=="投注金额"]
    gb=bet.groupby([mfr_col,game_col])["阶段汇总"].sum().rename("本周投注")
    gw=win.groupby([mfr_col,game_col])["阶段汇总"].sum().rename("公司输赢")
    gl=bl.groupby([mfr_col,game_col])["阶段汇总"].sum().rename("上周投注")
    gu=bet.groupby([mfr_col,game_col])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([gb,gw,gl,gu],axis=1).reset_index()
    g["占比"]=g["本周投注"]/g["本周投注"].sum()*100
    g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby(mfr_col)["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr


def load_games():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    mfr_col2 = "游戏厂商标签.名称" if "游戏厂商标签.名称" in tw.columns else tw.columns[0]
    game_col2 = "游戏.名称" if "游戏.名称" in tw.columns else tw.columns[1]
    tot=tw["投注金额"].sum()
    gt=tw.groupby([mfr_col2,game_col2]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    gl=lw.groupby(game_col2).agg(投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    gt["人均局数"]=gt["投注局数"]/gt["投注人数"]; gt["人均金额"]=gt["投注金额"]/gt["投注人数"]
    gt["盈亏率"]=gt["公司输赢"]/gt["投注金额"]*100; gt["占比"]=gt["投注金额"]/tot*100
    gt["厂商简称"]=gt[mfr_col2].apply(shorten_mfr)
    gm=gt.merge(gl,on=game_col2,how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)


def load_mfr():
    df=pd.read_csv(FILES["mfr"]); df=df.rename(columns={"盈利率":"盈亏率"})
    time_col3 = "时间" if "时间" in df.columns else df.columns[0]
    df=df[df[time_col3]!="阶段汇总"].copy(); df[time_col3]=df[time_col3].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢","盈亏率"]:
        if c in df.columns: df[c]=df[c].apply(clean)
    tw=df[df[time_col3].between(*THIS_WEEK)]; lw=df[df[time_col3].between(*LAST_WEEK)]
    mfr_col3 = "show_name_厂商标签id" if "show_name_厂商标签id" in df.columns else df.columns[1]
    def agg(d):
        g=d.groupby(mfr_col3).agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
    tg=agg(tw); lg=agg(lw)
    mg=tg.merge(lg[[mfr_col3,"投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on=mfr_col3,how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    mg["厂商显示名"]=mg[mfr_col3].apply(shorten_mfr)
    mg["show_name_厂商标签id"]=mg[mfr_col3]
    return mg.sort_values("投注金额",ascending=False)


def load_mfr_game_delta():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    mfr_c = "游戏厂商标签.名称" if "游戏厂商标签.名称" in tw.columns else tw.columns[0]
    game_c = "游戏.名称" if "游戏.名称" in tw.columns else tw.columns[1]
    gt=tw.groupby([mfr_c,game_c]).agg(投注金额=("投注金额","sum")).reset_index()
    gl=lw.groupby([mfr_c,game_c]).agg(投注金额=("投注金额","sum")).reset_index().rename(columns={"投注金额":"lw_投注"})
    m=gt.merge(gl,on=[mfr_c,game_c],how="outer").fillna(0); m["delta"]=m["投注金额"]-m["lw_投注"]
    mfr_tw=tw.groupby(mfr_c)["投注金额"].sum().sort_values(ascending=False)
    res={}
    for n in mfr_tw.head(10).index:
        sub=m[m[mfr_c]==n].sort_values("delta",ascending=False)
        res[n]={"up":sub.head(1),"dn":sub.tail(1)}
    return res


def load_activities():
    df = pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数","赠送人数.1"]:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0)
    # 列名适配
    opt_col = "账变opt_code" if "账变opt_code" in df.columns else df.columns[0]
    name_col2 = "name_账变opt_id" if "name_账变opt_id" in df.columns else df.columns[1]
    ag=df.groupby([opt_col,name_col2]).agg(
        赠送金额=("赠送金额","sum"),lw_赠=("赠送金额.1","sum"),
        赠送人数=("赠送人数","sum"),lw_人数=("赠送人数.1","sum")).reset_index()
    ag["环比"]=(ag["赠送金额"]-ag["lw_赠"])/ag["lw_赠"].replace(0,np.nan).abs()*100
    tot=ag["赠送金额"].sum(); ag["占比"]=ag["赠送金额"]/tot*100
    ag["人均"]=ag["赠送金额"]/ag["赠送人数"].replace(0,np.nan)
    ag["日均_本"]=ag["赠送人数"]/7; ag["日均_上"]=ag["lw_人数"]/7
    ag["人数环比"]=(ag["赠送人数"]-ag["lw_人数"])/ag["lw_人数"].replace(0,np.nan)*100
    ag["name_账变opt_id"]=ag[name_col2]
    return ag.sort_values("赠送金额",ascending=False), tot


def load_tool_data():
    tw=pd.read_csv(FILES["tool_tw"]); lw=pd.read_csv(FILES["tool_lw"])
    tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str).str.strip()
    tm=tm.drop_duplicates(subset="道具ID",keep="first")
    for df in [tw,lw]:
        df.columns=df.columns.str.strip().str.replace("\ufeff","")
        df["道具ID"]=df["道具ID"].astype(str).str.strip().str.replace('"',"")
        for c in ["道具发放(步骤1)","道具使用(步骤2)"]:
            if c in df.columns:
                df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")
        if "步骤2 转化" in df.columns:
            df["步骤2 转化"]=pd.to_numeric(df["步骤2 转化"].astype(str).str.replace("%",""),errors="coerce")
    tw=tw[tw["道具ID"]!="总体"].copy(); lw=lw[lw["道具ID"]!="总体"].copy()
    m=tw.rename(columns={"道具发放(步骤1)":"本周发放","道具使用(步骤2)":"本周使用","步骤2 转化":"本周使用率"}
    ).merge(lw[["道具ID","道具发放(步骤1)","道具使用(步骤2)","步骤2 转化"]].rename(
        columns={"道具发放(步骤1)":"上周发放","道具使用(步骤2)":"上周使用","步骤2 转化":"上周使用率"}),
        on="道具ID",how="outer").fillna(0)
    m=m.merge(tm[["道具ID","活动","备注"]],on="道具ID",how="left")
    m["活动"]=m["活动"].fillna("其他"); m["备注"]=m["备注"].fillna(m["道具ID"])
    m["发放环比"]=(m["本周发放"]-m["上周发放"])/m["上周发放"].replace(0,np.nan)*100
    m["使用环比"]=(m["本周使用"]-m["上周使用"])/m["上周使用"].replace(0,np.nan)*100
    return m.sort_values("本周发放",ascending=False)


def load_first_dep_ret():
    df=pd.read_csv(FILES["first_dep_ret"])
    df.columns=df.columns.str.strip().str.replace("\ufeff","")
    time_col4 = "初始事件发生时间" if "初始事件发生时间" in df.columns else "初始事件的发生时间"
    df["_d"]=df[time_col4].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"]=df["_d"].str.replace("-","").fillna("")
    for c in ["当日","1日","2日","3日","4日","5日","6日","7日"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
    user_col2 = "账变事件用户数" if "账变事件用户数" in df.columns else (
        "首充用户数" if "首充用户数" in df.columns else df.columns[3])
    df[user_col2]=pd.to_numeric(df[user_col2],errors="coerce").fillna(0)
    ind_col3 = "指标" if "指标" in df.columns else None
    stage=df[df[time_col4]=="阶段值"].copy() if ind_col3 else pd.DataFrame()
    daily=df[df["_yyyymmdd"].str.match(r"^\d{8}$",na=False)].copy()
    rr=daily[daily[ind_col3]=="留存率"].reset_index(drop=True) if ind_col3 else daily
    nr=daily[daily[ind_col3]=="留存人数"].reset_index(drop=True) if ind_col3 else daily
    tw_r=rr[rr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_r=rr[rr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    tw_n=nr[nr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_n=nr[nr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    def wa(rd,nd,col):
        if col not in rd.columns or len(rd)==0: return np.nan
        rv=rd[col].to_numpy(dtype=float,na_value=np.nan); ok=~np.isnan(rv)
        if not ok.any(): return np.nan
        w=nd[user_col2].to_numpy(dtype=float) if len(nd)==len(rd) else np.ones(len(rv))
        ww=w[ok]; return float(np.nanmean(rv[ok])) if ww.sum()==0 else float(np.average(rv[ok],weights=ww))
    R={"stage":stage,"tw_users":int(tw_n[user_col2].sum()),"lw_users":int(lw_n[user_col2].sum())}
    for col in ["1日","2日","3日","4日","5日","6日","7日"]:
        R[f"tw_{col}"]=wa(tw_r,tw_n,col); R[f"lw_{col}"]=wa(lw_r,lw_n,col)
    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    R["tw_platform_fc"]=int(dp[dp["日期"].between(*THIS_WEEK)]["首充人数"].sum())
    R["lw_platform_fc"]=int(dp[dp["日期"].between(*LAST_WEEK)]["首充人数"].sum())
    return R


def load_risk(dt_raw, dc_raw):
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    mfr_col4 = "show_name_厂商标签id" if "show_name_厂商标签id" in df.columns else df.columns[1]
    game_col4 = "游戏名称" if "游戏名称" in df.columns else df.columns[2]
    ind_col4  = "分析指标" if "分析指标" in df.columns else df.columns[3]
    ba=df[df[ind_col4]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    wa2=df[df[ind_col4]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    tg=(df[df[ind_col4]=="投注金额"].sort_values("阶段汇总",ascending=False)
        .groupby("账户ID").first()[[mfr_col4,game_col4,"阶段汇总"]]
        .rename(columns={mfr_col4:"主玩厂商",game_col4:"主玩游戏","阶段汇总":"主游投注"}).reset_index())
    ug=pd.concat([ba,wa2],axis=1).reset_index()
    ug["游戏_投注金额"]=ug["游戏_投注金额"].fillna(0)
    ug=ug.merge(tg,on="账户ID",how="left")
    top500=dt_raw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0
    top500["投充比"]=top500["投注金额"]/top500["充值金额"].replace(0,np.nan) if "投注金额" in top500.columns else np.nan
    hr=top500[top500["公司输赢"]<-10000].sort_values("公司输赢") if "公司输赢" in top500.columns else pd.DataFrame()

    # PP Auto-Roulette检查
    pp_check=df[(df[game_col4].str.contains("Auto-Roulette",na=False))&(df[ind_col4]=="投注金额")]["账户ID"].unique() if game_col4 in df.columns else []
    c207=[u for u in pp_check if str(u).startswith("207")]
    c207_dt=dt_raw[dt_raw["账户ID"].isin(c207)].sort_values("提款金额",ascending=False) if len(c207)>0 else pd.DataFrame()

    sp=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False) if "投充比" in top500.columns else pd.DataFrame()

    bg=df[df[ind_col4]=="投注金额"].groupby([mfr_col4,game_col4])["阶段汇总"].sum().rename("投注金额").reset_index()
    wg=df[df[ind_col4]=="公司输赢"].groupby([mfr_col4,game_col4])["阶段汇总"].sum().rename("公司输赢").reset_index()
    ug2=df[df[ind_col4]=="投注金额"].groupby([mfr_col4,game_col4])["账户ID"].nunique().rename("玩家数").reset_index()
    wn=df[df[ind_col4]=="公司输赢"].groupby([mfr_col4,game_col4]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bg.merge(wg,on=[mfr_col4,game_col4],how="left").merge(ug2,on=[mfr_col4,game_col4],how="left").merge(wn,on=[mfr_col4,game_col4],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100
    g["人均投注额"]=g["投注金额"]/g["玩家数"]
    g["show_name_厂商标签id"]=g[mfr_col4]; g["游戏名称"]=g[game_col4]
    rg=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    return hr,c207_dt,c207,sp,g.sort_values("投注金额",ascending=False).head(20),rg,top500


# ════════════════════════════════════════════════════════════════
# 图表
# ════════════════════════════════════════════════════════════════
SPLIT = 7

def _vline(ax, n_dates):
    ax.axvline(SPLIT - .5, color="#94a3b8", ls="--", lw=1, alpha=.7)
    trans = ax.get_xaxis_transform()
    ax.text(SPLIT - 4, 0.93, "上周", ha="center", fontsize=7, color="#64748b", transform=trans)
    ax.text(SPLIT + 3, 0.93, "本周", ha="center", fontsize=7, color="#1d4ed8", transform=trans)

def chart_trend(trend):
    fig = plt.figure(figsize=(16, 10), facecolor="white")
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=.45, wspace=.3)
    dates = trend["dates"]; x = range(len(dates)); n = len(dates)
    def sp(a):
        a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y", alpha=.25)
    ax = fig.add_subplot(gs[0, 0])
    ax.bar(x, [v/10000 for v in trend["充值"]], color=["#bfdbfe"]*SPLIT + ["#1d4ed8"]*SPLIT, width=.7, label="充值")
    ax.plot(x, [v/10000 for v in trend["提现"]], "-o", color="#dc2626", lw=1.5, ms=3, label="提现")
    ax.set_title("充值 vs 提现（万USD）", fontsize=9, fontweight="bold")
    ax.set_xticks(list(x)); ax.set_xticklabels(dates, rotation=45, fontsize=7)
    ax.legend(fontsize=7, loc="upper left"); sp(ax); _vline(ax, n)
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.bar(x, [v/10000 for v in trend["公司输赢"]], color=["#bbf7d0"]*SPLIT + ["#059669"]*SPLIT, width=.7)
    ax2.set_title("公司输赢（万USD）", fontsize=9, fontweight="bold")
    ax2.set_xticks(list(x)); ax2.set_xticklabels(dates, rotation=45, fontsize=7)
    sp(ax2); _vline(ax2, n)
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.bar(x, trend["首充"], color=["#c7d2fe"]*SPLIT + ["#6366f1"]*SPLIT, width=.7, label="首充人数")
    ax3r = ax3.twinx()
    ax3r.plot(x, trend["注册"], "--D", color="#d97706", lw=1.5, ms=3, label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）", fontsize=9, fontweight="bold")
    ax3.set_xticks(list(x)); ax3.set_xticklabels(dates, rotation=45, fontsize=7)
    l1, lb1 = ax3.get_legend_handles_labels(); l2, lb2 = ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2, lb1+lb2, fontsize=7, loc="upper left")
    ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y", alpha=.25)
    ax3.axvline(SPLIT - .5, color="#94a3b8", ls="--", lw=1, alpha=.7)
    trans3 = ax3.get_xaxis_transform()
    ax3.text(SPLIT-4, 0.93, "上周", ha="center", fontsize=7, color="#64748b", transform=trans3)
    ax3.text(SPLIT+3, 0.93, "本周", ha="center", fontsize=7, color="#1d4ed8", transform=trans3)
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.bar(x, trend["充提差比"], color=["#e9d5ff"]*SPLIT + ["#7c3aed"]*SPLIT, width=.7)
    tm_ = np.mean(trend["充提差比"][SPLIT:]); lm_ = np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tm_, color="#7c3aed", ls=":", lw=1.5); ax4.axhline(lm_, color="#94a3b8", ls=":", lw=1.2)
    _bbox = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85)
    ax4.text(n-1.0, tm_+.3, f"本周均{tm_:.1f}%", fontsize=7, color="#7c3aed", ha="right", clip_on=False, fontweight="bold", bbox=_bbox)
    ax4.text(0.0, lm_+.3, f"上周均{lm_:.1f}%", fontsize=7, color="#64748b", ha="left", clip_on=False, fontweight="bold", bbox=_bbox)
    ax4.set_title("充提差率（%）", fontsize=9, fontweight="bold")
    ax4.set_xticks(list(x)); ax4.set_xticklabels(dates, rotation=45, fontsize=7)
    ymax = max(trend["充提差比"]); ax4.set_ylim(0, ymax * 1.20)
    sp(ax4); _vline(ax4, n)
    fig.suptitle("近14日大盘核心指标趋势", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig, 16, 11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(14,6),facecolor="white")
    cols=[("第1日","次留","#1d4ed8"),("第2日","3留","#059669"),("第6日","7留","#7c3aed")]
    x=np.arange(len(weeks)); w=.25
    for i,(col,lbl,clr) in enumerate(cols):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-w,vals,w,label=lbl,color=clr,alpha=.85)
        for bar,v in zip(bars,vals):
            if not (isinstance(v,float) and np.isnan(v)):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
        tgt=RET_TARGETS.get(lbl)
        if tgt: ax.axhline(tgt,color=clr,ls="--",lw=1.2,alpha=.55)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周对比（含目标虚线）",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8,loc="upper left"); ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=.5)
    ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,6)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tc=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lc=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tc,w,label="本周",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,lc,w,label="上周",color="#93c5fd",alpha=.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tb=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lb=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tb,w,label="本周",color="#059669",alpha=.85); ax2.bar(x+w/2,lb,w,label="上周",color="#6ee7b7",alpha=.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=.85,width=.6); ax3.axhline(0,color="black",lw=.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vr):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=.28
    for ax,rt,t in [(ax1,"chg","充值→充值 留存率（%）"),(ax2,"act","充值→活跃 留存率（%）")]:
        cols_avail2 = list(vr[rt]["tw"].keys())
        c1 = "1日" if "1日" in cols_avail2 else (cols_avail2[0] if cols_avail2 else "1日")
        c3 = "3日" if "3日" in cols_avail2 else (cols_avail2[min(2,len(cols_avail2)-1)] if cols_avail2 else "3日")
        tw1=[vr[rt]["tw"].get(c1,{}).get(v,0) if isinstance(vr[rt]["tw"].get(c1),dict) else vr[rt]["tw"].get(c1,{v:0}).get(v,0) for v in vips]
        lw1=[vr[rt]["lw"].get(c1,{}).get(v,0) if isinstance(vr[rt]["lw"].get(c1),dict) else 0 for v in vips]
        tw3=[vr[rt]["tw"].get(c3,{}).get(v,0) if isinstance(vr[rt]["tw"].get(c3),dict) else 0 for v in vips]
        clr=("#1d4ed8","#93c5fd","#059669") if rt=="chg" else ("#7c3aed","#c4b5fd","#d97706")
        ax.bar(x-w,tw1,w,label=f"次日(本周)",color=clr[0],alpha=.85)
        ax.bar(x,lw1,w,label=f"次日(上周)",color=clr[1],alpha=.7)
        ax.bar(x+w,tw3,w,label=f"3日(本周)",color=clr[2],alpha=.75)
        ax.set_title(t,fontsize=10,fontweight="bold"); ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8)
        ax.legend(fontsize=7.5,loc="upper left"); ax.grid(axis="y",alpha=.25)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v2,_:f"{v2:.0f}%"))
    plt.suptitle("VIP各等级充值留存率",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10)
    names=top10["厂商显示名"].tolist() if "厂商显示名" in top10.columns else top10.iloc[:,0].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    sh=top10["占比"].tolist(); ls=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,sh,color=["#059669" if c>=l else "#dc2626" for c,l in zip(sh,ls)],alpha=.85,width=.6)
    ax1.plot(names,ls,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,sh,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.2,f"{s:.1f}%",ha="center",fontsize=7)
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=.85)
    ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=.7)
    ax2.axhline(0,color="black",lw=.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5)
    ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v3,_:f"{v3:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    game_col5 = "游戏.名称" if "游戏.名称" in top30.columns else top30.columns[1]
    t15=top30.head(15); names=[str(r[game_col5])[:18] for _,r in t15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in t15.iterrows()]
    lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in t15.iterrows()]
    x=np.arange(len(names)); w=.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=.85)
    ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=.7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8)
    ax.set_title("Top15游戏 投注金额（万USD）",fontsize=9,fontweight="bold")
    ax.legend(fontsize=8); ax.grid(axis="x",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pg,ma):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=ma.index[:8].tolist(); vals=ma.values[:8].tolist(); tot=sum(vals)
    pcts=[v/tot*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{str(n)[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-.05,-.18))
    ax1.set_title("Top500提款用户 厂商偏好",fontsize=9,fontweight="bold")
    game_col6 = "游戏名称" if "游戏名称" in pg.columns else pg.columns[1]
    mfr_col5  = "show_name_厂商标签id" if "show_name_厂商标签id" in pg.columns else pg.columns[0]
    t12=pg.head(12); gn=[f"[{str(r[mfr_col5])[:4]}]\n{str(r[game_col6])}"[:22] for _,r in t12.iterrows()]
    gv=[r["本周投注"]/10000 for _,r in t12.iterrows()]
    gc=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in t12.iterrows()]
    ax2.barh(range(len(gn))[::-1],gv,color=gc[::-1],alpha=.85)
    ax2.set_yticks(range(len(gn))); ax2.set_yticklabels(gn,fontsize=7.5)
    ax2.set_title("偏好游戏Top12（红=平台亏损）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v4,_:f"{v4:.0f}万"))
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act):
    name_col3 = "name_账变opt_id" if "name_账变opt_id" in act.columns else act.columns[1]
    t10=act.head(10); names=[str(r[name_col3])[:12] for _,r in t10.iterrows()]
    vals=[r["赠送金额"]/10000 for _,r in t10.iterrows()]
    lw=[r["lw_赠"]/10000 for _,r in t10.iterrows()]
    envs=[r["环比"] for _,r in t10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=.85)
    ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=.7)
    ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8)
    ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold")
    ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    es=[v if not pd.isna(v) else 0 for v in envs]
    ax2.barh(range(len(names)),es[::-1],color=["#059669" if v>0 else "#dc2626" for v in es[::-1]],alpha=.85)
    ax2.axvline(0,color="black",lw=.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8)
    ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v5,_:f"{v5:+.0f}%"))
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_tool_usage(tool):
    big=tool[tool["本周发放"]>=100].head(12); t12=big if len(big)>0 else tool.head(12)
    def _l(r):
        a=str(r.get("活动","")); n=str(r.get("备注",""))
        return f"{str(r['道具ID'])[:6]}\n{n[:8]}" if a=="其他" else f"[{a[:4]}]\n{n[:8]}"
    names=[_l(r) for _,r in t12.iterrows()]
    ti=[r["本周发放"] for _,r in t12.iterrows()]; li=[r["上周发放"] for _,r in t12.iterrows()]
    tr=[r.get("本周使用率",0) or 0 for _,r in t12.iterrows()]
    lr=[r.get("上周使用率",0) or 0 for _,r in t12.iterrows()]
    fig=plt.figure(figsize=(16,9),facecolor="white"); gs2=gridspec.GridSpec(2,2,figure=fig,hspace=.5,wspace=.35)
    x=np.arange(len(names)); w=.35
    ax1=fig.add_subplot(gs2[0,0]); ax1.bar(x-w/2,ti,w,label="本周发放",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,li,w,label="上周发放",color="#93c5fd",alpha=.7)
    ax1.set_title("Top12道具 发放量",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(names,fontsize=6.5); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v6,_:f"{v6/10000:.0f}万" if v6>=10000 else f"{v6:.0f}"))
    ax2=fig.add_subplot(gs2[0,1]); ax2.bar(x-w/2,tr,w,label="本周使用率",color="#059669",alpha=.85); ax2.bar(x+w/2,lr,w,label="上周使用率",color="#6ee7b7",alpha=.7)
    ax2.set_title("Top12道具 使用率",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(names,fontsize=6.5); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v7,_:f"{v7:.0f}%"))
    ax3=fig.add_subplot(gs2[1,:])
    di=[r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0 for _,r in t12.iterrows()]
    du=[r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0 for _,r in t12.iterrows()]
    ax3.bar(x-w/2,di,w,label="发放量环比",color=["#059669" if v>=0 else "#dc2626" for v in di],alpha=.85)
    ax3.bar(x+w/2,du,w,label="使用量环比",color=["#7c3aed" if v>=0 else "#f97316" for v in du],alpha=.7)
    ax3.axhline(0,color="black",lw=.8); ax3.set_title("Top12道具 发放/使用量 环比变化（%）",fontsize=9,fontweight="bold")
    ax3.set_xticks(x); ax3.set_xticklabels(names,fontsize=6.5); ax3.legend(fontsize=7.5); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v8,_:f"{v8:+.0f}%"))
    plt.suptitle("道具发放与使用分析",fontsize=11,fontweight="bold",y=1.01); plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,10)

def chart_first_dep_ret(R):
    lv=[R.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    tv=[R.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white"); x=np.arange(7)
    ax.plot(x,lv,"-o",color="#93c5fd",lw=2,ms=6,label=f"上周（{LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}）")
    ax.plot(x,tv,"-o",color="#1d4ed8",lw=2,ms=6,label=f"本周（{THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}）")
    days=["D1","D2","D3","D4","D5","D6","D7"]
    for i,(lval,tval) in enumerate(zip(lv,tv)):
        if lval is not None and not (isinstance(lval,float) and np.isnan(lval)): ax.text(i,lval+.4,f"{lval:.1f}%",ha="center",fontsize=7.5,color="#64748b")
        if tval is not None and not (isinstance(tval,float) and np.isnan(tval)): ax.text(i,tval-1.2,f"{tval:.1f}%",ha="center",fontsize=7.5,color="#1d4ed8",fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(days,fontsize=9); ax.set_title("首次充值活动用户 充值留存趋势",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8.5,loc="upper right"); ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v9,_:f"{v9:.0f}%"))
    plt.tight_layout(); return fig_img(fig,13,5.5)

def chart_risk_scatter(top500):
    v=top500[top500["投充比"].notna()].copy() if "投充比" in top500.columns else top500.head(0)
    v=v[v["投充比"]<200]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    if len(v)>0:
        ax.scatter(v["投充比"],v["公司输赢"]/10000,c=["#dc2626" if x<0 else "#059669" for x in v["公司输赢"]],s=[min(abs(x)/500+20,200) for x in v["公司输赢"]],alpha=.55,edgecolors="none")
    ax.axhline(0,color="black",lw=.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=.7)
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8)
    ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=.6,label="平台赢钱")],fontsize=8,loc="upper right")
    ax.grid(alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(rg):
    t12=rg.head(12)
    game_col7 = "游戏名称" if "游戏名称" in t12.columns else t12.columns[1]
    names=[str(r[game_col7])[:16] for _,r in t12.iterrows()]
    losses=[abs(r["公司输赢"]) for _,r in t12.iterrows()]
    rates=[r["盈亏率"] for _,r in t12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=.85)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v10,_:f"${v10/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=.85); ax2.axvline(0,color="black",lw=.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v11,_:f"{v11:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)


# ════════════════════════════════════════════════════════════════
# 章节构建（与原版相同，直接复用）
# ════════════════════════════════════════════════════════════════
def _render(t, **kw):
    try: return t.format(**kw)
    except: return t

def build_overview(K, trend, weekly_ret, dash_ret):
    full = PW - 2*MARGIN; S = [sec_title("一、大盘核心数据")]
    kpis = [
        ("充值金额",   f"{K['tw_充值金额']/10000:.1f}万",   f"{K['lw_充值金额']/10000:.1f}万",   K["pct_充值金额"],   True),
        ("提现金额",   f"{K['tw_提现金额']/10000:.1f}万",   f"{K['lw_提现金额']/10000:.1f}万",   K["pct_提现金额"],   False),
        ("充提差",     f"{K['tw_充提差']/10000:.1f}万",     f"{K['lw_充提差']/10000:.1f}万",     K["pct_充提差"],     True),
        ("充提差率",   f"{K['tw_充提差比']:.2f}%",          f"{K['lw_充提差比']:.2f}%",          K["pct_充提差比"],   True),
        ("公司输赢",   f"{K['tw_公司输赢']/10000:.1f}万",   f"{K['lw_公司输赢']/10000:.1f}万",   K["pct_公司输赢"],   True),
        ("盈亏率",     f"{K['tw_盈亏率']:.3f}%",            f"{K['lw_盈亏率']:.3f}%",            K["pct_盈亏率"],     True),
        ("注册人数",   f"{int(K['tw_注册人数']):,}",        f"{int(K['lw_注册人数']):,}",        K["pct_注册人数"],   True),
        ("首充人数",   f"{int(K['tw_首充人数']):,}",        f"{int(K['lw_首充人数']):,}",        K["pct_首充人数"],   True),
        ("日均活跃",   f"{K['tw_活跃人数']/7/10000:.1f}万", f"{K['lw_活跃人数']/7/10000:.1f}万", K["pct_活跃人数"],   True),
        ("投注金额",   f"{K['tw_投注金额']/10000:.0f}万",   f"{K['lw_投注金额']/10000:.0f}万",   K["pct_投注金额"],   True),
        ("全量ARPPU",  f"${K['tw_全量Arppu']:.2f}",         f"${K['lw_全量Arppu']:.2f}",         K["pct_全量Arppu"],  True),
        ("老用户ARPPU",f"${K['tw_老用户ARPPU']:.2f}",       f"${K['lw_老用户ARPPU']:.2f}",       K["pct_老用户ARPPU"],True),
        ("首充ARPPU",  f"${K['tw_首充Arppu']:.2f}",         f"${K['lw_首充Arppu']:.2f}",         K["pct_首充Arppu"],  True),
        ("总赠送金额", f"{K['tw_总赠送金额']/10000:.1f}万", f"{K['lw_总赠送金额']/10000:.1f}万", K["pct_总赠送金额"], False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",        f"{K['lw_赠送充值比']:.2f}%",        K["pct_赠送充值比"], False),
        ("首充转化率", f"{K['tw_首充转化率']:.1f}%",        f"{K['lw_首充转化率']:.1f}%",        K["pct_首充转化率"], True),
        ("首充次日留存",f"{K['tw_首充次日复充率']:.1f}%",   f"{K['lw_首充次日复充率']:.1f}%",   K["pct_首充次日复充率"],True),
        ("推广消耗(日均)", f"{K['tw_真实消耗_日均']/10000:.2f}万",
                          f"{K['lw_真实消耗_日均']/10000:.2f}万",
                          K["pct_真实消耗"], False),
        ("充提差ROI\n(日均充提差/日均消耗)",
         f"{K['tw_充提差ROI']:.2f}x", f"{K['lw_充提差ROI']:.2f}x",
         K["pct_充提差ROI"], True),
    ]
    S.append(kpi_card4(kpis, cols=4)); S.append(Spacer(1,8))
    cr_d = K["tw_充提差比"]-K["lw_充提差比"]; rd = K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(insight_box([
        _render("充值{a:.1f}万（{b:+.1f}%），公司输赢{c:.1f}万（{d:+.1f}%）；推广日均消耗{f:.2f}万（本周{g}天有效，{e:+.1f}%）。",
                a=K["tw_充值金额"]/10000, b=K["pct_充值金额"],
                c=K["tw_公司输赢"]/10000, d=K["pct_公司输赢"],
                e=K["pct_真实消耗"], f=K["tw_真实消耗_日均"]/10000, g=K["tw_真实消耗_有效天"]),
        _render("充提差率{a:.2f}%（上周{b:.2f}%，{c:+.2f}pp）；盈亏率{d:.3f}%（上周{e:.3f}%）。",
                a=K["tw_充提差比"],b=K["lw_充提差比"],c=cr_d,d=K["tw_盈亏率"],e=K["lw_盈亏率"]),
        _render("首充人数{a:,}（{b:+.1f}%），转化率{c:.1f}%，首充ARPPU${d:.2f}（{e:+.1f}%）。",
                a=int(K["tw_首充人数"]),b=K["pct_首充人数"],c=K["tw_首充转化率"],d=K["tw_首充Arppu"],e=K["pct_首充Arppu"]),
        _render("首充次日充值留存{a:.1f}%（上周{b:.1f}%，{c:+.1f}pp）。",
                a=K["tw_首充次日复充率"],b=K["lw_首充次日复充率"],c=rd),
    ]))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
    S.append(sub_title("大盘首充留存 vs 目标对比（来源：日报 首充2/3/7日复充率）"))
    specs = [("次留","tw_nd","lw_nd","tw_nd_end"),("3留","tw_td","lw_td","tw_td_end"),("7留","tw_sd","lw_sd","tw_sd_end")]
    ret_headers = ["留存类型","数据截止","本周实际","上周实际","周环比(pp)","目标","vs目标(pp)"]
    ret_rows = []
    for rtype,tw_k,lw_k,end_k in specs:
        tw_v=dash_ret.get(tw_k,np.nan); lw_v=dash_ret.get(lw_k,np.nan)
        end_lbl=fmt_lbl(dash_ret.get(end_k)); tgt=RET_TARGETS.get(rtype,np.nan)
        wpp=(tw_v-lw_v) if not (np.isnan(tw_v) or np.isnan(lw_v)) else np.nan
        dpp=(tw_v-tgt)  if not (np.isnan(tw_v) or np.isnan(tgt))  else np.nan
        tw_clr=C_GREEN if (not np.isnan(tw_v) and not np.isnan(tgt) and tw_v>=tgt) else C_RED
        ret_rows.append([
            cell(rtype,True,C_DARK), cell(end_lbl,False,C_GRAY,TA_CENTER),
            cell(fret(tw_v),False,tw_clr,TA_RIGHT), cell(fret(lw_v),False,C_GRAY,TA_RIGHT),
            cell(f"{wpp:+.1f}pp" if not np.isnan(wpp) else "-",False,C_GREEN if(not np.isnan(wpp) and wpp>=0) else C_RED,TA_RIGHT),
            cell(f"{tgt:.0f}%" if not np.isnan(tgt) else "-",True,C_TARGET,TA_CENTER),
            cell(f"{dpp:+.1f}pp" if not np.isnan(dpp) else "-",True,C_GREEN if(not np.isnan(dpp) and dpp>=0) else C_RED,TA_RIGHT),
        ])
    S.append(dtable(ret_headers,ret_rows,[full*x for x in [0.16,0.12,0.14,0.14,0.14,0.12,0.14]],fsize=8,extra_style=[("BACKGROUND",(5,1),(5,-1),colors.HexColor("#eff6ff"))]))
    S.append(Spacer(1,4))
    nd_end=fmt_lbl(dash_ret.get("tw_nd_end")); td_end=fmt_lbl(dash_ret.get("tw_td_end")); sd_end=fmt_lbl(dash_ret.get("tw_sd_end"))
    S.append(P(f"本周截止：次留~{nd_end}，3留~{td_end}，7留~{sd_end}（剔除数据未满足lag天数的不完整日期）",7,False,C_GRAY))
    S.append(Spacer(1,8))
    S.append(sub_title("首充用户充值留存 — 近4周对比（含目标线）"))
    S.append(chart_ret_weekly(weekly_ret)); S.append(Spacer(1,4))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=this_w.get("第1日",0) or 0; d2=this_w.get("第2日",0) or 0; d6=this_w.get("第6日",0) or 0
    ld1=last_w.get("第1日",0) or 0; ld6=last_w.get("第6日",0) or 0
    S.append(insight_box([
        f"本周首充次日留存{d1:.1f}%，较上周{d1-ld1:+.1f}pp；3留{d2:.1f}%，7留{d6:.1f}%。",
        f"注：本周7日留存因截止日期不完整，以上周7留{ld6:.1f}%作为参考基准。",
    ], clr=C_AMBER))
    return S

def build_agents(agents, agent_ret, ends):
    full = PW - 2*MARGIN; S = [sec_title("二、总代分析")]
    tw_nd_lbl=fmt_lbl(ends.get("tw_nd_end")); lw_nd_lbl=fmt_lbl(ends.get("lw_nd_end"))
    tw_3d_lbl=fmt_lbl(ends.get("tw_3d_end")); lw_3d_lbl=fmt_lbl(ends.get("lw_3d_end"))
    lw_7d_lbl=fmt_lbl(ends.get("lw_7d_end"))
    S.append(sub_title("全量总代表现（本周 vs 上周，按充值金额排序，含官方总代0）"))
    headers=["ID","总代名称","注册(环比)","首充\n人数","充值\n(万)","充提差率\n(差值pp)","消耗\n(万)","1级首充\n成本(本/上)",
             f"次留(本/上)\n~{tw_nd_lbl}/{lw_nd_lbl}",f"3留(本/上)\n~{tw_3d_lbl}/{lw_3d_lbl}",f"7留上周\n~{lw_7d_lbl}"]
    rows=[]
    id_col  = "总代.ID"   if "总代.ID"   in agents.columns else agents.columns[0]
    name_col4="总代.名称" if "总代.名称" in agents.columns else agents.columns[1]
    for _,r in agents.iterrows():
        aid=int(r[id_col]) if not pd.isna(r.get(id_col,np.nan)) else "-"
        rname=str(r[name_col4])
        cr=r.get("充提差率",0) or 0; lw_cr=r.get("lw_充提差率",0) or 0; cr_d=cr-lw_cr
        cost=(r.get("总消耗",0) or 0)/10000
        fcc=r.get("一级首充成本",0) or 0; lfc=r.get("lw_fc_cost",0) or 0
        reg=int(r.get("注册人数",0) or 0); reg_c=r.get("注册环比",0) or 0
        cr_clr=C_GREEN if cr>=CR_HIGH else (C_RED if cr<CR_LOW else C_DARK)
        d=agent_ret.get(rname,{})
        tw_nd=d.get("tw_nd",np.nan); lw_nd=d.get("lw_nd",np.nan)
        tw_3d=d.get("tw_3d",np.nan); lw_3d=d.get("lw_3d",np.nan)
        lw_7d=d.get("lw_7d",np.nan)
        def _fmt_pair(tv,lv):
            if not np.isnan(tv) and not np.isnan(lv): return f"{tv:.1f}%/{lv:.1f}%",C_GREEN if tv>=lv else C_RED
            if not np.isnan(tv): return f"{tv:.1f}%/-",C_DARK
            return "-",C_GRAY
        nd_s,nd_clr=_fmt_pair(tw_nd,lw_nd); td_s,td_clr=_fmt_pair(tw_3d,lw_3d)
        rows.append([
            cell(str(aid),False,C_GRAY,TA_CENTER), cell(rname[:14],True,C_DARK,TA_LEFT),
            cell(f"{reg:,}/{'+' if reg_c>=0 else ''}{reg_c:.0f}%",False,C_GREEN if reg_c>=0 else C_RED,TA_RIGHT),
            cell(f"{int(r.get('首充人数',0)):,}",False,C_DARK,TA_RIGHT),
            cell(f"{r.get('充值金额',0)/10000:.0f}",False,C_DARK,TA_RIGHT),
            cell(f"{cr:.1f}%/{'+' if cr_d>=0 else ''}{cr_d:.1f}pp",False,cr_clr,TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-",False,C_DARK,TA_RIGHT),
            cell(f"${fcc:.0f}/${lfc:.0f}" if fcc>0 else "-",False,C_DARK,TA_RIGHT),
            cell(nd_s,False,nd_clr,TA_RIGHT), cell(td_s,False,td_clr,TA_RIGHT),
            cell(fret(lw_7d),False,C_DARK,TA_RIGHT),
        ])
    cw=[full*x for x in [0.04,0.14,0.10,0.06,0.05,0.11,0.05,0.10,0.11,0.11,0.08]]
    S.append(dtable(headers,rows,cw,fsize=6.2))
    S.append(Spacer(1,4))
    S.append(P(f"★ 充提差率≥{CR_HIGH}%绿，<{CR_LOW}%红。",6.5,False,C_GRAY)); S.append(Spacer(1,6))
    ins=[]
    if len(agents)>0:
        t1=agents.iloc[0]; t1_cr_d=t1.get("充提差率",0)-(t1.get("lw_充提差率",0) or 0)
        ins.append(f"体量最大总代「{str(t1[name_col4])[:12]}」充值{t1.get('充值金额',0)/10000:.0f}万，充提差率{t1.get('充提差率',0):.1f}%（{t1_cr_d:+.1f}pp），注册{int(t1.get('注册人数',0)):,}人（{t1.get('注册环比',0):+.0f}%）。")
    best=agents[agents.get("充值金额",pd.Series([0]*len(agents)))>10000].nlargest(1,"充提差率") if "充提差率" in agents.columns else pd.DataFrame()
    if len(best)>0:
        b=best.iloc[0]; ins.append(f"充提差率最优渠道「{str(b[name_col4])[:12]}」达{b.get('充提差率',0):.1f}%，充值规模{b.get('充值金额',0)/10000:.0f}万。")
    if ins: S.append(insight_box(ins))
    return S

def build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_tw):
    full=PW-2*MARGIN; S=[sec_title("三、用户分析")]
    S.append(sub_title("VIP等级分层分析")); S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
    tot=tw_v["充值金额"].sum()
    hvs=sum(tw_v.loc[v,"充值金额"] for v in [9,10,11] if v in tw_v.index)
    S.append(insight_box([
        f"VIP9-11高价值层合计贡献充值{hvs/tot*100:.1f}%，高端用户付费意愿持续强劲。",
        f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。" if 1 in tw_v.index and 1 in lw_v.index else "VIP1基础层数据不可用。"
    ]))
    vr=load_vip_retention()
    S.append(sub_title("VIP各等级充值留存率")); S.append(chart_vip_retention(vr)); S.append(Spacer(1,4))
    v9_act=vr.get("act",{}).get("tw",{}).get("1日",{}).get(9,0)
    v10_act=vr.get("act",{}).get("tw",{}).get("1日",{}).get(10,0)
    v10_chg_tw=vr.get("chg",{}).get("tw",{}).get("1日",{}).get(10,0)
    v10_chg_lw=vr.get("chg",{}).get("lw",{}).get("1日",{}).get(10,0)
    S.append(insight_box([
        f"充值→活跃次日留存：VIP9达{v9_act:.1f}%，VIP10达{v10_act:.1f}%。",
        f"充值→充值次日留存：VIP10达{v10_chg_tw:.1f}%（上周{v10_chg_lw:.1f}%）。",
    ])); S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h=["分层","本周充值","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),
                     cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    S.append(insight_box([
        _render("Top10充值用户人均${a:,.0f}（{b:+.1f}%），充提差率{c:.1f}%。",
                a=dep_t[1]["tw_avg"],b=pct(dep_t[1]["tw_avg"],dep_t[1]["lw_avg"]),c=dep_t[1]["tw_cr"]),
        "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。"
    ]))
    S.append(sub_title("头部提款用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","赢家比例","活动占比","占全量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdr_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                      cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                      rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),
                      cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),
                      cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
                      cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),
                      cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(h3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box([
        f"剔除Top100提款用户后，大盘充提差率影响{wdr_t[3]['大盘影响']:+.2f}pp，头部提款用户对充提差率有明显拖累。",
        f"Top500提款用户活动奖励占资金来源仅{wdr_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。",
    ], clr=C_AMBER))
    return S

def build_games(mfr, top30, delta):
    full=PW-2*MARGIN; S=[sec_title("四、游戏分析")]
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    h=["排名","厂商","日均投注人数(本/上)","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r.get("厂商显示名",r["show_name_厂商标签id"])),True),
                     cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                     cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
                     cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
    top1=mfr.iloc[0]
    S.append(insight_box([
        _render("{nm}投注份额{ts:.1f}%（上周{tsl:.1f}%），Fortune系稳固主导；环比{tc:+.1f}%。",
                nm=top1.get("厂商显示名",top1["show_name_厂商标签id"]),ts=top1["占比"],tsl=top1.get("lw_占比",0),tc=top1.get("投注环比",0)),
        "3Oaks、PlayTech盈亏率高于平台均值，可适当扩大曝光权重；关注盈亏率持续偏低厂商的RTP设置。",
        "自研游戏份额稳定，盈亏率需持续优化，建议加强对高频自研玩法的收益监控。",
    ]))
    if delta:
        S.append(sub_title("▶ 厂商投注额环比主要驱动游戏"))
        dh=["厂商","投注额环比","增量最大游戏(+贡献)","降量最大游戏(-拖累)"]; dr=[]
        for mn,gd in delta.items():
            mrow=mfr[mfr["show_name_厂商标签id"]==mn]; mc=mrow["投注环比"].values[0] if len(mrow)>0 else 0
            game_cn="游戏.名称" if "游戏.名称" in gd["up"].columns else gd["up"].columns[1]
            up=gd["up"]; dn=gd["dn"]
            us=f'{up.iloc[0][game_cn][:16]}（+${up.iloc[0]["delta"]/10000:.1f}万）' if len(up)>0 and up.iloc[0]["delta"]>0 else "-"
            ds=f'{dn.iloc[0][game_cn][:16]}（${dn.iloc[0]["delta"]/10000:.1f}万）' if len(dn)>0 and dn.iloc[0]["delta"]<0 else "-"
            dr.append([cell(shorten_mfr(mn),True,C_DARK),rc(mc),cell(us,False,C_GREEN if us!="-" else C_GRAY),cell(ds,False,C_RED if ds!="-" else C_GRAY)])
        S.append(dtable(dh,dr,[full*x for x in [0.15,0.10,0.37,0.38]],fsize=6.8)); S.append(Spacer(1,6))
    S.append(sub_title("Top30游戏详细数据")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
    game_col8="游戏.名称" if "游戏.名称" in top30.columns else top30.columns[1]
    h2=["#","游戏名称","厂商简称","投注人数","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r[game_col8])[:18],True),
                      cell(str(r.get("厂商简称",""))[:5]),
                      cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                      cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
                      cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h2,rows2,[full*x for x in [0.04,0.18,0.07,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S

def build_activities(act, tot_gift):
    full=PW-2*MARGIN; S=[sec_title("五、活动分析")]
    S.append(sub_title("各活动赠送效果（全量，含环比）")); S.append(chart_activities(act)); S.append(Spacer(1,4))
    h=["活动名称","本周赠送","上周赠送","金额环比","本周日均\n赠送人数","上周日均\n赠送人数","人数环比","本周人均\n赠送金额","占比"]
    rows=[]
    for _,r in act.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:18],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"${r['lw_赠']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r["环比"] if not pd.isna(r.get("环比",np.nan)) else 0),
                     cell(f"{r.get('日均_本',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r.get('日均_上',0):.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r.get("人数环比",0) if not pd.isna(r.get("人数环比",np.nan)) else 0),
                     cell(f"${r.get('人均',0):.2f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.20,0.11,0.11,0.07,0.12,0.12,0.07,0.11,0.09]],fsize=6.5))
    S.append(Spacer(1,4)); daily=tot_gift/7/10000
    S.append(P(f"本周总赠送金额：${tot_gift/10000:.2f}万 | 日均赠送：${daily:.2f}万",9,True,C_DARK)); S.append(Spacer(1,6))
    S.append(insight_box([f"各类赠送活动全量列出（共{len(act)}个），合计本周日均赠送{daily:.1f}万USD。"]))
    return S

def build_tools(tool, fdr):
    full=PW-2*MARGIN; S=[sec_title("六、道具专题分析", clr=C_TEAL)]
    S.append(sub_title("6.1 道具发放、使用及使用率（本周发放≥100）"))
    S.append(chart_tool_usage(tool)); S.append(Spacer(1,4))
    td=tool[tool["本周发放"]>=100].copy()
    h=["道具ID","活动类型","备注说明","本周发放","上周发放","发放环比","本周使用","上周使用","使用环比","本周\n使用率","上周\n使用率"]
    rows=[]
    for _,r in td.iterrows():
        di=r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0
        du=r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0
        tw_rt=r.get("本周使用率",0) or 0; lw_rt=r.get("上周使用率",0) or 0
        rows.append([cell(str(r["道具ID"]),False,C_GRAY,TA_CENTER),cell(str(r.get("活动",""))[:10],False,C_PURPLE),
                     cell(str(r.get("备注",""))[:14],False,C_DARK),
                     cell(f"{int(r['本周发放']):,}" if r['本周发放']>0 else "-",False,C_DARK,TA_RIGHT),
                     cell(f"{int(r['上周发放']):,}" if r['上周发放']>0 else "-",False,C_GRAY,TA_RIGHT),rc(di),
                     cell(f"{int(r['本周使用']):,}" if r['本周使用']>0 else "-",False,C_DARK,TA_RIGHT),
                     cell(f"{int(r['上周使用']):,}" if r['上周使用']>0 else "-",False,C_GRAY,TA_RIGHT),rc(du),
                     cell(f"{tw_rt:.1f}%",False,C_GREEN if tw_rt>=lw_rt else C_RED,TA_RIGHT),
                     cell(f"{lw_rt:.1f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.07,0.10,0.14,0.08,0.08,0.07,0.08,0.08,0.07,0.07,0.08]],fsize=6.3))
    S.append(Spacer(1,4))
    bv=tool[tool["本周发放"]>=100].copy()
    def _lbl(r):
        a=str(r.get("活动","")); n=str(r.get("备注",""))
        return f"{n}({r.get('本周使用率',0):.0f}%)" if a=="其他" else f"[{a}]{n}({r.get('本周使用率',0):.0f}%)"
    if len(bv)>=3:
        hi="、".join([_lbl(r) for _,r in bv.nlargest(3,"本周使用率").iterrows()])
        lo="、".join([_lbl(r) for _,r in bv.nsmallest(3,"本周使用率").iterrows()])
    else: hi=lo="-"
    tw_t=int(tool["本周发放"].sum()); lw_t=tool["上周发放"].sum()
    S.append(insight_box([f"使用率最高3类：{hi}。",f"使用率最低3类：{lo}。",
                          f"本周总道具发放{tw_t:,}，较上周{lw_t:,.0f}，环比{pct(tw_t,lw_t):+.1f}%。"]))
    S.append(Spacer(1,8))
    S.append(sub_title("6.2 首次充值活动用户 充值留存分析（重点）"))
    stage=fdr.get("stage",pd.DataFrame())
    if len(stage)>0 and "指标" in stage.columns:
        lws=LAST_WEEK[0]
        S.append(P(f"▸ 两周阶段汇总（{lws[:4]}.{lws[4:6]}.{lws[6:]} - {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}）",9,True,C_DARK)); S.append(Spacer(1,3))
        sr=stage[stage["指标"]=="留存率"]; sn=stage[stage["指标"]=="留存人数"]; sa=stage[stage["指标"]=="人均充值金额"]
        if len(sr)>0:
            user_col3="账变事件用户数" if "账变事件用户数" in sn.columns else (sn.columns[3] if len(sn.columns)>3 else sn.columns[-1])
            u=int(sn[user_col3].values[0]) if len(sn)>0 else 0
            S.append(P(f"首充活动用户总数：{u:,}人",8.5,False,C_DARK))
            dc=["当日","1日","2日","3日","4日","5日","6日","7日"]
            dc_avail=[c for c in dc if c in sr.columns]
            srows=[]
            if len(sr)>0: srows.append(["留存率"]+[str(sr.iloc[0].get(c,"-")) for c in dc_avail])
            if len(sn)>0: srows.append(["留存人数"]+[str(sn.iloc[0].get(c,"-")) for c in dc_avail])
            if srows:
                tr2=[[cell(row[0],True,C_DARK)]+[cell(str(v),False,C_DARK,TA_CENTER) for v in row[1:]] for row in srows]
                S.append(dtable(["指标"]+dc_avail,tr2,[full*.12]+[full*(0.88/len(dc_avail))]*len(dc_avail),fsize=6.8))
        S.append(Spacer(1,6))
    S.append(P("▸ 本周 vs 上周 首充活动用户留存趋势",9,True,C_DARK)); S.append(Spacer(1,3))
    S.append(chart_first_dep_ret(fdr)); S.append(Spacer(1,4))
    tw_u=fdr.get("tw_users",0); lw_u=fdr.get("lw_users",0)
    tw_pf=fdr.get("tw_platform_fc",0); lw_pf=fdr.get("lw_platform_fc",0)
    tw_r=tw_u/tw_pf*100 if tw_pf>0 else 0; lw_r=lw_u/lw_pf*100 if lw_pf>0 else 0
    tv=[fdr.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    lv=[fdr.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    def _fr(v): return f"{v:.1f}%" if not (v is None or (isinstance(v,float) and np.isnan(v))) else "-"
    rh=["周次","活动用户数","平台首充\n人数","活动用户\n占比","D1留存","D2留存","D3留存","D4留存","D5留存","D6留存","D7留存"]
    rrows=[
        [cell(f"本周 {THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}",True,C_BLUE),
         cell(f"{tw_u:,}",False,C_DARK,TA_RIGHT),cell(f"{tw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{tw_r:.1f}%",False,C_PURPLE,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GREEN if not pd.isna(v) and not pd.isna(lval) and v>=lval else(C_RED if not pd.isna(v) and not pd.isna(lval) and v<lval else C_DARK),TA_RIGHT) for v,lval in zip(tv,lv)],
        [cell(f"上周 {LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}",True,C_GRAY),
         cell(f"{lw_u:,}",False,C_GRAY,TA_RIGHT),cell(f"{lw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{lw_r:.1f}%",False,C_GRAY,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GRAY,TA_RIGHT) for v in lv],
    ]
    S.append(dtable(rh,rrows,[full*.16,full*.09,full*.09,full*.08]+[full*.084]*7,fsize=7,zebra=False)); S.append(Spacer(1,4))
    tw_d1=fdr.get("tw_1日",np.nan); lw_d1=fdr.get("lw_1日",np.nan); ins=[]
    if not np.isnan(tw_d1) and not np.isnan(lw_d1):
        d=tw_d1-lw_d1; ins.append(f"首充活动用户次日留存{tw_d1:.1f}%（上周{lw_d1:.1f}%，{d:+.1f}pp），{'留存改善' if d>0 else '留存下降，建议优化次日触达策略'}。")
    ins.append(f"本周活动用户{tw_u:,}人，占平台首充{tw_r:.1f}%（上周{lw_u:,}/{lw_r:.1f}%）；人均赠送$3.89，属高价值用户来源。")
    ins.append("建议：对D1/D2留存用户设置阶梯式再充值道具激励；对D3后流失用户做专项召回（24h内触达效果最佳）。")
    S.append(insight_box(ins, clr=C_AMBER))
    return S

def build_risk(hr, c207_dt, c207, sp, top20g, rg, top500, dt_full):
    full=PW-2*MARGIN; S=[sec_title("七、用户游戏风险专项分析", clr=colors.HexColor("#7c2d12"))]
    tot_tx=top500["提款金额"].sum() if "提款金额" in top500.columns else 0
    tot_win=top500["公司输赢"].sum() if "公司输赢" in top500.columns else 0
    wns=(top500["公司输赢"]<0).sum() if "公司输赢" in top500.columns else 0
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${tot_tx/10000:.1f}万",""),
                           ("平台净赔付",f"${abs(tot_win)/10000:.1f}万","平台向该群体净赔"),
                           ("赢家比例",f"{wns/500*100:.1f}%",f"{wns}赢/{500-wns}输"),
                           ("高风险用户",f"{len(hr)}人","公司净输>$10,000")]:
        fw=full/4-4
        row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],
                           colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    hl=abs(hr["公司输赢"].sum())/10000 if len(hr)>0 else 0
    S.append(insight_box([
        f"Top500提款用户中赢家{wns}人（{wns/500*100:.1f}%），平台净赔付${abs(tot_win)/10000:.1f}万。",
        f"{len(hr)}名超级赢家（公司净输>$10,000）合计导致平台净输${hl:.1f}万。",
    ]))
    if len(hr)>0:
        S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
        h=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","主玩厂商","主玩游戏","风险标签"]
        rows=[]
        for _,r in hr.iterrows():
            ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0
            tags=[]
            if r.get("充值金额",0)<5000: tags.append("低充高提")
            if ratio>50: tags.append("超高投充")
            rows.append([cell(str(r["账户ID"]),True),cell(f"${r.get('提款金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"${r.get('充值金额',0):,.0f}",False,C_RED if r.get("充值金额",0)<5000 else C_DARK,TA_RIGHT),
                         cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED,TA_RIGHT),
                         cell(f"${r.get('投注金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),
                         cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),
                         cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
        S.append(dtable(h,rows,[full*x for x in [0.12,0.11,0.11,0.11,0.11,0.07,0.09,0.16,0.12]],fsize=6.5)); S.append(Spacer(1,6))
    if len(c207)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警"))
        S.append(P(f"集群账户数：{len(c207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
        S.append(insight_box([
            f"{len(c207)}个账户充值$547-548，均获活动奖励，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。",
            "建议：①冻结账户提款；②审查注册IP/设备指纹；③排查奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。",
        ], clr=C_AMBER))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    th=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; tr=[]
    for i,(_,r) in enumerate(dt_full.head(20).iterrows()):
        win=r.get("公司输赢",0)<0
        tr.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),
                   cell(f"${r.get('提款金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r.get('充值金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),
                   cell(f"${r.get('活动奖励',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),
                   cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(th,tr,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
    pg,ma=load_pref()
    S.append(sub_title("Top500提款用户游戏偏好")); S.append(chart_pref(pg,ma)); S.append(Spacer(1,4))
    if len(rg)>0:
        S.append(sub_title("▶ 高危游戏专项分析")); S.append(chart_risk_games(rg)); S.append(Spacer(1,4))
        gh=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; gr=[]
        for _,r in rg.head(12).iterrows():
            pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rc2=C_RED if rl in("极高","高") else C_AMBER
            gr.append([cell(str(r.get("游戏名称",""))[:18],True),cell(str(r.get("show_name_厂商标签id",""))[:8]),
                       cell(f"{int(r.get('玩家数',0))}",False,C_DARK,TA_RIGHT),cell(f"${r.get('投注金额',0)/10000:.1f}万",False,C_DARK,TA_RIGHT),
                       cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),
                       cell(f"{r.get('赢家率',0):.0f}%",False,C_RED if r.get("赢家率",0)>60 else C_AMBER,TA_RIGHT),
                       cell(f"${r.get('人均投注额',0):,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rc2)])
        S.append(dtable(gh,gr,[full*x for x in [0.21,0.10,0.07,0.10,0.11,0.08,0.08,0.10,0.07]],fsize=6.5)); S.append(Spacer(1,6))
        top_rg=rg.iloc[0] if len(rg)>0 else None
        ins1=f"高危游戏{top_rg.get('游戏名称','')}（{top_rg.get('show_name_厂商标签id','')}）公司输赢${top_rg.get('公司输赢',0):,.0f}，盈亏率{top_rg.get('盈亏率',0):.1f}%，建议复审RTP参数。" if top_rg is not None else "本周未检测到极高风险游戏。"
        S.append(insight_box([ins1,"建议对高赢家率游戏实施单账户赢额上限，防范套利风险。"], clr=C_RED))
    return S

def build_conclusion(K, dep_t, wdr_t, mfr, act, rg, hr, c207):
    full=PW-2*MARGIN; S=[sec_title("八、总结与行动建议")]
    cr_d=K["tw_充提差比"]-K["lw_充提差比"]; rd=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(sub_title("▶ 本周亮点")); bg_g=colors.HexColor("#f0fdf4")
    highlights=[]
    highlights.append(f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），充提差{K['tw_充提差']/10000:.1f}万，充提差率{K['tw_充提差比']:.2f}%（{cr_d:+.2f}pp）。")
    if K["pct_首充人数"]>0:
        highlights.append(f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。")
    if K["pct_公司输赢"]>0:
        highlights.append(f"公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%），盈亏率{K['tw_盈亏率']:.3f}%，较上周改善。")
    top1_mfr=mfr.iloc[0]
    highlights.append(f"{top1_mfr.get('厂商显示名',top1_mfr['show_name_厂商标签id'])}投注份额{top1_mfr['占比']:.1f}%（{top1_mfr.get('投注环比',0):+.1f}%），Fortune系稳固主导。")
    for hl in highlights:
        S.append(Table([[P(f"• {hl}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_g),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))
    S.append(sub_title("▶ 风险预警")); bg_r=colors.HexColor("#fff5f5")
    risks=[]
    if len(c207)>0: risks.append(f"PP Auto-Roulette 1批量套利：{len(c207)}个账户盈亏率极低，需立即处置。")
    if K["pct_首充Arppu"]<-5 or rd<-1:
        risks.append(f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp），新用户质量需关注。")
    if len(hr)>0:
        hl2=abs(hr["公司输赢"].sum())/10000
        risks.append(f"{len(hr)}名高风险用户（公司净输>$10,000）合计净输${hl2:.1f}万，需立即人工审核。")
    top_rg_r=rg.iloc[0] if len(rg)>0 else None
    if top_rg_r is not None and top_rg_r.get("公司输赢",0)<-20000:
        risks.append(f"高危游戏「{str(top_rg_r.get('游戏名称',''))[:16]}」公司输赢${top_rg_r.get('公司输赢',0):,.0f}，赢家率{top_rg_r.get('赢家率',0):.0f}%，建议复核RTP参数。")
    if wdr_t[3]["大盘影响"]<-1: risks.append(f"头部提款用户对充提差率拖累{wdr_t[3]['大盘影响']:+.2f}pp（剔除Top100后）。")
    if K["pct_真实消耗"]<-15:
        risks.append(f"推广日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（本周{K['tw_真实消耗_有效天']}天有效口径），较上周日均{K['lw_真实消耗_日均']/10000:.2f}万{K['pct_真实消耗']:+.1f}%，建议确认是否计划内削减，注意对首充量的滞后影响。")
    if not risks: risks.append("本周暂无重大风险预警，各指标维持正常区间。")
    for r2 in risks:
        S.append(Table([[P(f"• {r2}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_r),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))
    S.append(sub_title("▶ 行动建议"))
    actions=[]
    if len(c207)>0: actions.append(("[紧急]","处置批量套利账号",f"共{len(c207)}个账户，建议冻结提款、审查IP/设备指纹、修复奖励触发漏洞并实施赢额上限。"))
    if len(hr)>0:
        top_hr=hr.sort_values("公司输赢").iloc[0]
        actions.append(("[紧急]","处置高风险提款用户",f"账户{top_hr['账户ID']}：提款${top_hr.get('提款金额',0):,.0f}，公司输赢${top_hr.get('公司输赢',0):,.0f}，高度异常，建议立即人工审核。"))
    if K["pct_首充Arppu"]<-5 or rd<-1:
        actions.append(("[本周]","优化首充质量与次日激活",f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp）。建议注册后1h/24h内推送首充引导，落地页突出$10+档。"))
    if top_rg_r is not None and top_rg_r.get("公司输赢",0)<-20000:
        actions.append(("[本周]","高危游戏风险管控",f"对赢家率>60%的高危游戏实施单账户赢额上限，同步复核「{str(top_rg_r.get('游戏名称',''))[:16]}」RTP参数，防范系统性套利。"))
    if K["pct_真实消耗"]<-15:
        actions.append(("[本周]","关注推广消耗削减影响",f"日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（{K['pct_真实消耗']:+.1f}%），若非计划内，建议与投放团队确认，监控首充量未来1-2周的滞后变化。"))
    if not actions: actions.append(("[常规]","持续监控核心指标","充值、留存、充提差率等核心指标维持正常，建议持续监控并保持现有运营节奏。"))
    pmap={"[紧急]":C_RED,"[本周]":C_AMBER,"[常规]":C_GREEN}
    rows=[[cell(pri,True,pmap.get(pri[:4],C_GRAY),TA_CENTER),cell(title,True,C_DARK),cell(desc,False,C_GRAY)] for pri,title,desc in actions]
    S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows,[full*x for x in [0.1,0.22,0.68]]))
    return S


def header_footer(c, doc):
    c.saveState(); w,h=A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN,h-17,f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
    c.restoreState()


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("📊 MX 周报 PDF — 本周 20260529-20260604 生成中...")
    resolve_files(DATA_ROOT); setup()
    print("  ▶ 加载数据...")
    K, trend         = load_platform()
    print(f"  ▶ 推广消耗日均{K['tw_真实消耗_日均']/10000:.2f}万，充提差ROI={K['tw_充提差ROI']:.2f}x")
    dash_ret         = load_dashboard_retention()
    weekly_ret       = load_weekly_retention()
    agents           = load_agents()
    agent_ret, ends  = load_agent_ret()
    tw_v, lw_v       = load_vip()
    dep_t,wdr_t,dc_tw,dt_top = load_top_users()
    top30            = load_games()
    mfr              = load_mfr()
    mfr_delta        = load_mfr_game_delta()
    act_df, tot_gift = load_activities()
    tool_df          = load_tool_data()
    fdr              = load_first_dep_ret()

    dt_raw=pd.read_csv(FILES["dt_tw"]); dc_raw=pd.read_csv(FILES["dc_tw"])
    for df in [dt_raw, dc_raw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns: df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")

    hr,c207_dt,c207,sp,top20g,rg,top500 = load_risk(dt_raw, dc_raw)

    # 给top20提款用户加主玩游戏
    dpf=pd.read_csv(FILES["pref_tw"]); dpf["阶段汇总"]=dpf["阶段汇总"].apply(clean)
    ind_pf="分析指标" if "分析指标" in dpf.columns else dpf.columns[3]
    game_pf="游戏名称" if "游戏名称" in dpf.columns else dpf.columns[2]
    mfr_pf="show_name_厂商标签id" if "show_name_厂商标签id" in dpf.columns else dpf.columns[1]
    tg2=(dpf[dpf[ind_pf]=="投注金额"].sort_values("阶段汇总",ascending=False)
         .groupby("账户ID").first()[[mfr_pf,game_pf]]
         .rename(columns={mfr_pf:"主玩厂商",game_pf:"主玩游戏"}).reset_index())
    dt_top=dt_top.merge(tg2,on="账户ID",how="left")

    print("  ▶ 导出风控Excel...")
    excel_out=OUTPUT_DIR/f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out),engine="openpyxl") as xw:
        cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","投充比"]
        hr[[c for c in cols_hr if c in hr.columns]].to_excel(xw,sheet_name="高风险用户",index=False)
        if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(xw,sheet_name="PP轮盘批量账号",index=False)
        if len(sp)>0: sp[["账户ID","提款金额","充值金额","公司输赢"]].to_excel(xw,sheet_name="特殊异常用户",index=False)
        tool_df.to_excel(xw,sheet_name="道具使用情况",index=False)
    print(f"  ✅ 风控Excel: {excel_out}")

    print("  ▶ 构建PDF章节...")
    out=OUTPUT_DIR/f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc=SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,
                          topMargin=MARGIN+22,bottomMargin=MARGIN+10,
                          title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story=[Spacer(1,3*cm),
           P("MX 平台数据周报",30,True,C_DARK,TA_CENTER),Spacer(1,.5*cm),
           P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),
           Spacer(1,.2*cm),
           P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),
           Spacer(1,.5*cm),HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"),PageBreak()]
    story+=build_overview(K,trend,weekly_ret,dash_ret);      story.append(PageBreak())
    story+=build_agents(agents,agent_ret,ends);               story.append(PageBreak())
    story+=build_users(tw_v,lw_v,dep_t,wdr_t,dc_tw,dt_top); story.append(PageBreak())
    story+=build_games(mfr,top30,mfr_delta);                 story.append(PageBreak())
    story+=build_activities(act_df,tot_gift);                 story.append(PageBreak())
    story+=build_tools(tool_df,fdr);                          story.append(PageBreak())
    story+=build_risk(hr,c207_dt,c207,sp,top20g,rg,top500,dt_top); story.append(PageBreak())
    story+=build_conclusion(K,dep_t,wdr_t,mfr,act_df,rg,hr,c207)

    print("  ▶ 渲染PDF...")
    doc.build(story,onFirstPage=header_footer,onLaterPages=header_footer)
    size=out.stat().st_size/1024
    print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out), str(excel_out)

if __name__=="__main__":
    main()

📊 MX 周报 PDF — 本周 20260529-20260604 生成中...
     ✅ [platform       ] 平台报表_USD_20260605113501.xlsx
     ✅ [daily          ] 日报-大盘日报_USD_20260605102650.xlsx
     ✅ [retention      ] 整体 首充留存（近7天）_20260508-20260604.csv
     ✅ [agent_plat     ] 平台报表-总代_USD_20260605172911.xlsx
     ✅ [agent_promo    ] 推广报表-总代_USD_20260605120449.xlsx
     ✅ [agent_ret      ] 首充充值留存_20260508-20260604.csv
     ✅ [vip            ] VIP报表_USD_20260605121539.xlsx
     ✅ [dt_tw          ] top提款用户_全量数据_20260529_20260604_日期对比20260522_20260528 (1).csv
     ✅ [dt_lw          ] top提款用户_全量数据_20260522_20260528_日期对比20260515_20260521.csv
     ✅ [dc_tw          ] 头部充值用户_全量数据_20260529_20260604_日期对比20260522_20260528.csv
     ✅ [dc_lw          ] 头部充值用户_全量数据_20260522_20260528_日期对比20260515_20260521.csv
     ✅ [pref_tw        ] 本周top500提款用户游戏偏好_全量数据_20260529_20260604.csv
     ✅ [pref_lw        ] 上周top500提款用户游戏偏好_全量数据_20260522_20260528.csv
     ✅ [mfr            ] 厂商投注数据_全量数据_20260522_20260604.csv
     ✅ [game_tw        ] 游戏报表-详情_USD_

In [6]:
"""
MX 平台数据周报 v7.5.3 — 消耗/ROI口径统一版
修改说明（v7.5.2 → v7.5.3）：
  ★ 推广消耗和充提差ROI：固定剔除本周最后一天，不做任何条件判断，与 weekly_report 口径完全统一
  - 旧：if len(tw_d) > 1 else tw_d（有条件判断，可能不剔）
  - 新：直接 tw_d["真实消耗"].iloc[:-1]（无条件固定剔最后1天）
v7.5.2 修改（保留）：充提差ROI新增
v7.5.1 修改（保留）：推广消耗日均化
使用方法：
  1. 把所有数据文件与本脚本放在同一目录
  2. 修改下方 THIS_WEEK / LAST_WEEK 两行为新周期
  3. 命令行运行：python mx_report_v753.py
"""
from pathlib import Path
import sys, os, io, warnings
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec, matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image, PageBreak, HRFlowable, KeepTogether)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# ── 路径配置 ─────────────────────────────────────────────
DATA_ROOT  = Path(r"D:\周报更新版\MX")   # ← 改这里
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")   # ← 改这里（可与上面相同）

if not DATA_ROOT.exists():
    raise RuntimeError(
        f"\n\n❌ 数据文件夹不存在：{DATA_ROOT}\n"
        "请修改脚本开头的 DATA_ROOT 为正确路径，例如：\n"
        r'  DATA_ROOT = Path(r"C:\Users\你的名字\Desktop\MX数据")' + "\n"
    )

THIS_WEEK  = ("20260523", "20260529")
LAST_WEEK  = ("20260516", "20260522")
REPORT_END = THIS_WEEK[1]

# ── 留存目标 ─────────────────────────────────────────────
RET_TARGETS = {"次留": 21.0, "3留": 15.0, "7留": 11.0, "14留": 8.0, "30留": 6.0}

# ── 全局字体/颜色 ─────────────────────────────────────────
FN, FNB = "WQY", "WQYB"
FILES: dict = {}

C_BLUE   = colors.HexColor("#1d4ed8");  C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669");  C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706");  C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b");  C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white;                C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1");  C_ROW    = colors.HexColor("#f8fafc")
C_TEAL   = colors.HexColor("#0f766e");  C_TARGET = colors.HexColor("#0369a1")

PW, PH = A4
MARGIN  = 1.6 * cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
CR_HIGH, CR_LOW = 17, 5
MFR_SHORT = {"Rectangle":"RG","Pragmatic Play":"PP","PG Soft":"PG","PlayTech":"PT","Originals":"自研","Fat Panda":"FP"}

TRUNCATE = {"首充次日复充率":1,"首充次日复投率":1,"首充当日复充率":0,
            "首充7日复充率":6,"首充30日复充率":999,"首充2日复充率":1,"首充3日复充率":2}

# ── 工具函数 ──────────────────────────────────────────────
def shorten_mfr(n):
    if not isinstance(n, str): return str(n)
    for k, v in MFR_SHORT.items():
        if k in n: return v
    return n[:8]

def ret_end(week_start: str, lag: int):
    e = (datetime.strptime(REPORT_END, "%Y%m%d") - timedelta(days=lag)).strftime("%Y%m%d")
    return e if e >= week_start else None

def fmt_lbl(d):
    return f"{d[4:6]}/{d[6:]}" if d else "-"

def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c not in ("日期","总代.名称","name_总代","总代.ID")]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o):   return (n-o)/abs(o)*100 if o and o != 0 else 0.0
def pct_vec(ns, os):
    return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop else s
    nz = v[v > 0]; return nz.mean() if len(nz) else v.mean()

def wavg_series(vals, weights):
    ok = vals.notna() & (weights > 0)
    if not ok.any(): return np.nan
    return float(np.average(vals[ok], weights=weights[ok]))

def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

def find_chinese_font():
    import sys, os
    win_root = Path(os.environ.get("WINDIR", "C:/Windows"))
    win_cands = [
        win_root / "Fonts/msyh.ttc",
        win_root / "Fonts/msyhbd.ttc",
        win_root / "Fonts/simhei.ttf",
        win_root / "Fonts/simsun.ttc",
        win_root / "Fonts/simkai.ttf",
        win_root / "Fonts/STKAITI.TTF",
        win_root / "Fonts/STXIHEI.TTF",
    ]
    other_cands = [
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        "/System/Library/Fonts/PingFang.ttc",
    ]
    for p in win_cands + other_cands:
        if Path(p).exists(): return str(p)
    from matplotlib import font_manager as fm
    for f in fm.fontManager.ttflist:
        if any(k in f.name for k in
               ["YaHei","Hei","SimSun","SimHei","WenQuanYi","Noto Sans CJK","PingFang"]):
            if Path(f.fname).exists(): return f.fname
    raise FileNotFoundError(
        "未找到中文字体！请确认系统已安装微软雅黑(msyh.ttc)或黑体(simhei.ttf)。")

def setup():
    global FONT_PATH
    FONT_PATH = find_chinese_font()
    print(f"  ▶ 字体：{FONT_PATH}")
    ttc = FONT_PATH.lower().endswith(".ttc")
    pdfmetrics.registerFont(TTFont(FN,  FONT_PATH, subfontIndex=0) if ttc else TTFont(FN,  FONT_PATH))
    try:    pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1) if ttc else TTFont(FNB, FONT_PATH))
    except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0) if ttc else TTFont(FNB, FONT_PATH))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(FONT_PATH)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

def resolve_files(root):
    global FILES
    tw0, tw1 = THIS_WEEK
    lw0, lw1 = LAST_WEEK
    all_files = []
    try:
        for h in root.rglob("*"):
            try:
                if h.is_file() and not h.name.startswith("~$"):
                    all_files.append(h)
            except (PermissionError, OSError):
                pass
    except Exception as e:
        print(f"  ⚠️  扫描目录出错: {e}")

    def find(kws):
        kws = kws if isinstance(kws, list) else [kws]
        hits = [h for h in all_files if all(k in h.name for k in kws)]
        return max(hits, key=lambda h: h.stat().st_mtime) if hits else None

    km = {
        "platform":      ["平台报表_USD"],
        "daily":         ["日报-大盘日报_USD"],
        "retention":     ["整体","首充留存"],
        "agent_plat":    ["平台报表-总代_USD"],
        "agent_promo":   ["推广报表-总代_USD"],
        "agent_ret":     ["首充充值留存_全量数据"],
        "vip":           ["VIP报表_USD"],
        "dt_tw":         [f"top提款用户_全量数据_{tw0}"],
        "dt_lw":         [f"top提款用户_全量数据_{lw0}"],
        "dc_tw":         [f"头部充值用户_全量数据_{tw0}"],
        "dc_lw":         [f"头部充值用户_全量数据_{lw0}"],
        "pref_tw":       [f"本周top500提款用户游戏偏好_全量数据_{tw0}"],
        "pref_lw":       [f"上周top500提款用户游戏偏好_全量数据_{lw0}"],
        "mfr":           ["厂商投注数据_全量数据"],
        "game_tw":       ["游戏报表-详情_USD_本周"],
        "game_lw":       ["游戏报表-详情_USD_上周"],
        "gift":          ["各赠送活动_全量数据"],
        "tool_map":      ["道具对应活动"],
        "vip_ret_chg":   ["VIP充值-充值_近28天"],
        "vip_ret_act":   ["VIP充值-活跃_近28天"],
        "tool_tw":       ["本周道具使用情况"],
        "tool_lw":       ["上周道具使用情况"],
        "first_dep_ret": ["首次充值活动用户充值留存情况"],
    }
    fail = 0
    for key, kws in km.items():
        p = find(kws)
        if p:
            FILES[key] = p
            print(f"     ✅ [{key:15s}] {p.name}")
        else:
            print(f"     ❌ [{key:15s}] 找不到含{kws}的文件")
            fail += 1
    if fail:
        raise FileNotFoundError(f"共{fail}个文件未找到")
    print(f"  ▶ 全部{len(km)}个文件匹配成功\n")

# ── PDF 样式工具 ─────────────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.4,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5), HRFlowable(width="100%",thickness=1.5,color=C_BLUE2),
                         Spacer(1,3), P(f"■  {text}",10,True,C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}",8.5,False,C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8, extra_style=None):
    full = PW - 2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),       ("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2),      ("BOTTOMPADDING",(0,0),(-1,-1),2),
        ("LEFTPADDING",(0,0),(-1,-1),2),     ("RIGHTPADDING",(0,0),(-1,-1),2),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    if extra_style:
        for cmd in extra_style: st.add(*cmd)
    hrow = [P(h,fsize,True,C_WHITE,TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]),fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c,(list,tuple)) else P(str(c),fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t,bold=False,clr=colors.black,align=TA_LEFT): return (t,bold,clr,align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup=good_up
    c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{'+' if v>=0 else ''}{v:.{d}f}%",False,c,TA_RIGHT)
def gclr(v,t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)
def fret(v): return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

def kpi_card4(items, cols=4):
    fw = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,_ in items:
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([[P(label,7.5,False,C_GRAY)],[P(str(tv),14,True,C_DARK)],
                       [P(f"上周：{lv}",7.5,False,C_GRAY)],
                       [P(f"{'+' if chg>=0 else ''}{chg:.1f}%",8,True,pclr)]],
                      colWidths=[fw],
                      style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
                                        ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
                                        ("LEFTPADDING",(0,0),(-1,-1),8),
                                        ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(fw,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════════════════════════

def load_platform():
    df = pd.read_excel(FILES["platform"]); df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]; lw_d = dd[dd["日期"].between(*LAST_WEEK)]
    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
              "首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])

    # ★★★ 推广消耗 & 充提差ROI：固定剔除本周最后一天（周期截止当天消耗未录完，直接剔，不做任何判断）
    # 上周全量不剔；本周和上周均计算日均后再做环比，口径一致。
    # 充提差ROI = 日均充提差 / 日均真实消耗（本周充提差天数与消耗天数严格对齐）
    # 日报文件为倒序（最新日期在首行），故 sort_values 降序排列后 iloc[:-1] 剔掉最旧一行。
    # 改为升序排列（最新日期在末行），iloc[:-1] 正确剔掉最新一天（05/29，消耗未录完）。
    # 平台报表文件同理，充提差按升序后取前 tw_cost_days 天，与消耗天数对齐。
    tw_d_asc = tw_d.sort_values("日期")          # 升序：最新日期在末行
    tw_cost_valid  = tw_d_asc["真实消耗"].iloc[:-1]   # 剔末行（最新天，消耗未录完）
    lw_cost_days   = len(lw_d)
    tw_cost_days   = len(tw_cost_valid)                # 本周有效天数（通常=6）
    K["tw_真实消耗_日均"]   = tw_cost_valid.sum() / tw_cost_days  if tw_cost_days > 0 else 0
    K["lw_真实消耗_日均"]   = lw_d["真实消耗"].sum() / lw_cost_days if lw_cost_days > 0 else 0
    K["tw_真实消耗"]         = tw_cost_valid.sum()
    K["lw_真实消耗"]         = lw_d["真实消耗"].sum()
    K["tw_真实消耗_有效天"]  = tw_cost_days
    K["pct_真实消耗"]        = pct(K["tw_真实消耗_日均"], K["lw_真实消耗_日均"])

    # 充提差ROI：平台报表升序后取前 tw_cost_days 天（同样剔最新天），与消耗严格对齐
    tw_sorted = tw.sort_values("日期")                        # 升序
    tw_cd_valid = tw_sorted["充提差"].iloc[:tw_cost_days]     # 前N天（不含最新天）
    K["tw_充提差_日均_roi"]  = tw_cd_valid.sum() / tw_cost_days if tw_cost_days > 0 else 0
    lw_sorted = lw.sort_values("日期")
    K["lw_充提差_日均_roi"]  = lw_sorted["充提差"].sum() / lw_cost_days if lw_cost_days > 0 else 0
    K["tw_充提差ROI"] = (K["tw_充提差_日均_roi"] / K["tw_真实消耗_日均"]
                         if K["tw_真实消耗_日均"] > 0 else 0)
    K["lw_充提差ROI"] = (K["lw_充提差_日均_roi"] / K["lw_真实消耗_日均"]
                         if K["lw_真实消耗_日均"] > 0 else 0)
    K["pct_充提差ROI"] = pct(K["tw_充提差ROI"], K["lw_充提差ROI"])

    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])
    last14 = df.tail(14)
    trend = {"dates": [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
             "充值": last14["充值金额"].tolist(), "提现": last14["提现金额"].tolist(),
             "充提差比": last14["充提差比"].tolist(), "公司输赢": last14["公司输赢"].tolist(),
             "首充": last14["首充人数"].tolist(), "注册": last14["注册人数"].tolist()}
    return K, trend


def load_dashboard_retention():
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str)
    specs = [("nd","首充2日复充率",1),("td","首充3日复充率",2),
             ("sd","首充7日复充率",6),("30d","首充30日复充率",29)]
    for _,col,_ in specs:
        if col in dd.columns:
            dd[col] = pd.to_numeric(dd[col].astype(str).str.replace("%","").str.strip(), errors="coerce")
    R = {}
    for key, col, lag in specs:
        for ws, we, prefix in [(THIS_WEEK[0],THIS_WEEK[1],"tw_"), (LAST_WEEK[0],LAST_WEEK[1],"lw_")]:
            ec = ret_end(ws, lag)
            sub = dd[(dd["日期"]>=ws) & (dd["日期"]<=we)]
            if ec: sub = sub[sub["日期"]<=ec]
            R[f"{prefix}{key}"] = float(sub[col].mean()) if len(sub) and col in sub else np.nan
        R[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        R[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)
    return R


def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
    daily["ds"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
    dr = daily[daily["指标"]=="留存率"].copy()
    du = daily[daily["指标"]=="留存人数"].copy()
    for c in ["第1日","第2日","第3日","第6日","第7日"]:
        if c in dr.columns:
            dr[c] = pd.to_numeric(dr[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
    def _d(s): return datetime.strptime(s,"%Y%m%d")
    def _f(d): return d.strftime("%Y-%m-%d")
    def _l(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s,tw_e = _d(THIS_WEEK[0]),_d(THIS_WEEK[1])
    lw_s,lw_e = _d(LAST_WEEK[0]),_d(LAST_WEEK[1])
    w2e = lw_s-timedelta(days=1); w2s = w2e-timedelta(days=6)
    w1e = w2s-timedelta(days=1); w1s = w1e-timedelta(days=6)
    weeks = [(f"第1周\n{_l(w1s,w1e)}",_f(w1s),_f(w1e)),
             (f"第2周\n{_l(w2s,w2e)}",_f(w2s),_f(w2e)),
             (f"上周\n{_l(lw_s,lw_e)}",_f(lw_s),_f(lw_e)),
             (f"本周\n{_l(tw_s,tw_e)}",_f(tw_s),_f(tw_e))]
    result = []
    for wk,s,e in weeks:
        mr = dr[(dr["ds"]>=s)&(dr["ds"]<=e)]; mu = du[(du["ds"]>=s)&(du["ds"]<=e)]
        row = {"week": wk, "users": mu["充值成功事件用户数"].sum()}
        for col in ["第1日","第2日","第3日","第6日","第7日"]:
            rs = mr[col].values if col in mr.columns else np.array([])
            us = mu["充值成功事件用户数"].values
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = float(np.average(rs[v],weights=us[v])) if v.sum()>0 else np.nan
            else: row[col] = float(np.nanmean(rs)) if len(rs)>0 else np.nan
        result.append(row)
    return result


def load_agents():
    dp = pd.read_excel(FILES["agent_plat"]); dr = pd.read_excel(FILES["agent_promo"])
    dp["日期"]=dp["日期"].astype(str); dr["日期"]=dr["日期"].astype(str)
    dp=to_num(dp); dr=to_num(dr)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]; lw_p=dp[dp["日期"].between(*LAST_WEEK)]
    tw_r=dr[dr["日期"].between(*THIS_WEEK)]; lw_r=dr[dr["日期"].between(*LAST_WEEK)]
    sa=["充值金额","提现金额","充提差","首充金额","首充人数","注册人数","充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sa if c in d.columns}).reset_index()
        for c in ["充提差比","首充次日充值留存"]:
            if c in d.columns: g=g.merge(d.groupby("总代.ID")[c].mean().rename(c),on="总代.ID",how="left")
        g["充提差率"]=g["充提差"]/g["充值金额"]*100; return g
    tw_pa=agg_p(tw_p); lw_pa=agg_p(lw_p)
    sr=["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        g=d.groupby("总代.ID").agg({c:"sum" for c in sr if c in d.columns}).reset_index()
        g["一级首充成本"]=g["总消耗"]/g["一级首充人数"].replace(0,np.nan); return g
    tw_ra=agg_r(tw_r); lw_ra=agg_r(lw_r)
    lw_ra["lw_fc_cost"]=lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
    m=tw_pa.merge(lw_pa[["总代.ID","充值金额","注册人数","充提差率"]].rename(
        columns={"充值金额":"lw_充值","注册人数":"lw_注册","充提差率":"lw_充提差率"}),on="总代.ID",how="left")
    m=m.merge(tw_ra[["总代.ID","总消耗","一级首充成本","一级首充人数"]],on="总代.ID",how="left")
    m=m.merge(lw_ra[["总代.ID","lw_fc_cost"]],on="总代.ID",how="left")
    m["注册环比"]=pct_vec(m["注册人数"],m["lw_注册"])
    return m.sort_values("充值金额",ascending=False)


def load_agent_ret():
    df = pd.read_csv(FILES["agent_ret"])
    df["_d"] = df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0].str.replace("-","")
    for c in ["1日","2日","6日"]:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
    df["首充用户数"] = pd.to_numeric(df["首充用户数"], errors="coerce").fillna(0)
    df["总代"] = pd.to_numeric(df["总代"], errors="coerce")
    daily = df[df["_d"].notna()&df["_d"].str.match(r"^\d{8}$")&(df["指标"]=="留存率")].copy()
    ends = {}
    for key, lag in [("nd",1),("3d",2),("7d",6)]:
        ends[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        ends[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)
    def _slice(ws, we, ec):
        sub = daily[daily["_d"].between(ws,we)]
        if ec: sub = sub[sub["_d"] <= min(ec,we)]
        return sub
    slices = {
        "tw_nd": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_nd_end"]),
        "tw_3d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_3d_end"]),
        "tw_7d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_7d_end"]),
        "lw_nd": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_nd_end"]),
        "lw_3d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_3d_end"]),
        "lw_7d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_7d_end"]),
    }
    col_map = {"tw_nd":"1日","tw_3d":"2日","tw_7d":"6日","lw_nd":"1日","lw_3d":"2日","lw_7d":"6日"}
    def _wavg_by_agent(df_sub, col):
        res = {}
        for nm, grp in df_sub.groupby("name_总代"):
            v = wavg_series(grp[col], grp["首充用户数"])
            if not np.isnan(v): res[nm] = v
        return res
    per_agent = {k: _wavg_by_agent(slices[k], col_map[k]) for k in slices}
    id_map = {}
    for nm, grp in daily.groupby("name_总代"):
        vals = grp["总代"].dropna().values
        if len(vals): id_map[nm] = int(vals[0])
    all_names = set().union(*[set(v) for v in per_agent.values()])
    ret = {}
    for nm in all_names:
        ret[nm] = {k: per_agent[k].get(nm, np.nan) for k in per_agent}
        ret[nm]["agent_id"] = id_map.get(nm)
    return ret, ends

def load_vip():
    df=pd.read_excel(FILES["vip"]); df["日期"]=df["日期"].astype(str)
    num=["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df=to_num(df,num)
    tw=df[df["日期"].between(*THIS_WEEK)]; lw=df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

def load_vip_retention():
    res={}
    for fk,rt in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df=pd.read_csv(FILES[fk])
        df["ds"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
        df["yyyymmdd"]=df["ds"].str.replace("-","")
        for c in ["1日","2日","3日","7日"]:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
        df["n"]=pd.to_numeric(df["充值成功事件用户数"],errors="coerce")
        df["vip"]=pd.to_numeric(df["vip_level"],errors="coerce")
        daily=df[(df["指标"]=="留存率")&df["yyyymmdd"].notna()&df["vip"].notna()]
        tw=daily[daily["yyyymmdd"].between(*THIS_WEEK)]; lw=daily[daily["yyyymmdd"].between(*LAST_WEEK)]
        def wavg(d,col):
            r={}
            for v,g in d.groupby("vip"):
                s=g[g[col].notna()]
                if len(s): r[int(v)]=float(np.average(s[col].values,weights=s["n"].values))
            return r
        res[rt]={"tw":{c:wavg(tw,c) for c in ["1日","2日","3日","7日"]},
                 "lw":{c:wavg(lw,c) for c in ["1日","2日","3日","7日"]}}
    return res

def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
    dt_lw_std = dt_lw[["账户ID","提款金额","充值金额","公司输赢","活动奖励","投注金额"]].copy()
    dc_lw_std = dc_lw[["账户ID","充值金额","提款金额","公司输赢","活动奖励"]].copy()
    for df in [dc_tw, dc_lw_std, dt_tw, dt_lw_std]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c].astype(str).str.replace(",",""), errors="coerce")
    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]
    total_c=tw_p["充值金额"].sum(); total_t=tw_p["提现金额"].sum()
    actual_cr=(total_c-total_t)/total_c*100 if total_c>0 else 0
    tot_c=dc_tw["充值金额"].sum(); tot_t=dt_tw["提款金额"].sum()
    dep_t,wdr_t=[],[]
    for t in [1,10,50,100,200,500]:
        tw=dc_tw.head(t); lw=dc_lw_std.head(t)
        tc=tw["充值金额"].sum(); tt=tw["提款金额"].sum() if "提款金额" in tw.columns else 0
        lc=lw["充值金额"].sum(); lt=lw["提款金额"].sum() if "提款金额" in lw.columns else 0
        win=tw["公司输赢"].sum() if "公司输赢" in tw.columns else 0
        dep_t.append({"tier":f"Top{t}","tw_chg":tc,"lw_chg":lc,"tw_avg":tc/t,"lw_avg":lc/t,
            "tw_cr":(tc-tt)/tc*100 if tc>0 else 0,"lw_cr":(lc-lt)/lc*100 if lc>0 else 0,
            "tw_win":win,"占全量":tc/tot_c*100 if tot_c>0 else 0})
        tw2=dt_tw.head(t); lw2=dt_lw_std.head(t)
        tt2=tw2["提款金额"].sum(); tc2=tw2["充值金额"].sum()
        lt2=lw2["提款金额"].sum() if "提款金额" in lw2.columns else 0
        lc2=lw2["充值金额"].sum() if "充值金额" in lw2.columns else 0
        wns=(tw2["公司输赢"]<0).sum() if "公司输赢" in tw2.columns else 0
        act_sum=tw2["活动奖励"].sum() if "活动奖励" in tw2.columns else 0
        act_pct=act_sum/(tc2+act_sum)*100 if (tc2+act_sum)>0 else 0
        excl_c=total_c-tc2; excl_t=total_t-tt2
        excl_cr=(excl_c-excl_t)/excl_c*100 if excl_c>0 else 0
        wdr_t.append({"tier":f"Top{t}","tw_tx":tt2,"lw_tx":lt2,"tw_avg":tt2/t,"lw_avg":lt2/t,
            "tw_cr":(tc2-tt2)/tc2*100 if tc2>0 else 0,"lw_cr":(lc2-lt2)/lc2*100 if lc2>0 else 0,
            "赢家":wns,"总数":t,"赢家率":wns/t*100,"活动占比":act_pct,
            "占全量":tt2/tot_t*100 if tot_t>0 else 0,"大盘影响":excl_cr-actual_cr})
    return dep_t,wdr_t,dc_tw.head(200),dt_tw.head(20)

def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); dl=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); dl["阶段汇总"]=dl["阶段汇总"].apply(clean)
    bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]; bl=dl[dl["分析指标"]=="投注金额"]
    gb=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
    gw=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
    gl=bl.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
    gu=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([gb,gw,gl,gu],axis=1).reset_index()
    g["占比"]=g["本周投注"]/g["本周投注"].sum()*100; g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr

def load_games():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    tot=tw["投注金额"].sum()
    gt=tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    gl=lw.groupby("游戏.名称").agg(投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    gt["人均局数"]=gt["投注局数"]/gt["投注人数"]; gt["人均金额"]=gt["投注金额"]/gt["投注人数"]
    gt["盈亏率"]=gt["公司输赢"]/gt["投注金额"]*100; gt["占比"]=gt["投注金额"]/tot*100
    gt["厂商简称"]=gt["游戏厂商标签.名称"].apply(shorten_mfr)
    gm=gt.merge(gl,on="游戏.名称",how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)

def load_mfr():
    df=pd.read_csv(FILES["mfr"]); df=df.rename(columns={"盈利率":"盈亏率"})
    df=df[df["时间"]!="阶段汇总"].copy(); df["时间"]=df["时间"].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢","盈亏率"]:
        if c in df.columns: df[c]=df[c].apply(clean)
    tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
    def agg(d):
        g=d.groupby("show_name_厂商标签id").agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
    tg=agg(tw); lg=agg(lw)
    mg=tg.merge(lg[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on="show_name_厂商标签id",how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    mg["厂商显示名"]=mg["show_name_厂商标签id"].apply(shorten_mfr)
    return mg.sort_values("投注金额",ascending=False)

def load_mfr_game_delta():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    gt=tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index()
    gl=lw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index().rename(columns={"投注金额":"lw_投注"})
    m=gt.merge(gl,on=["游戏厂商标签.名称","游戏.名称"],how="outer").fillna(0); m["delta"]=m["投注金额"]-m["lw_投注"]
    mfr_tw=tw.groupby("游戏厂商标签.名称")["投注金额"].sum().sort_values(ascending=False)
    res={}
    for n in mfr_tw.head(10).index:
        sub=m[m["游戏厂商标签.名称"]==n].sort_values("delta",ascending=False)
        res[n]={"up":sub.head(1),"dn":sub.tail(1)}
    return res

def load_activities():
    df = pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数","赠送人数.1"]:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0)
    ag=df.groupby(["账变opt_code","name_账变opt_id"]).agg(
        赠送金额=("赠送金额","sum"),lw_赠=("赠送金额.1","sum"),
        赠送人数=("赠送人数","sum"),lw_人数=("赠送人数.1","sum")).reset_index()
    ag["环比"]=(ag["赠送金额"]-ag["lw_赠"])/ag["lw_赠"].replace(0,np.nan).abs()*100
    tot=ag["赠送金额"].sum(); ag["占比"]=ag["赠送金额"]/tot*100; ag["人均"]=ag["赠送金额"]/ag["赠送人数"].replace(0,np.nan)
    ag["日均_本"]=ag["赠送人数"]/7; ag["日均_上"]=ag["lw_人数"]/7
    ag["人数环比"]=(ag["赠送人数"]-ag["lw_人数"])/ag["lw_人数"].replace(0,np.nan)*100
    return ag.sort_values("赠送金额",ascending=False), tot

def load_tool_data():
    tw=pd.read_csv(FILES["tool_tw"]); lw=pd.read_csv(FILES["tool_lw"])
    tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str).str.strip()
    tm=tm.drop_duplicates(subset="道具ID",keep="first")
    for df in [tw,lw]:
        df.columns=df.columns.str.strip().str.replace("\ufeff","")
        df["道具ID"]=df["道具ID"].astype(str).str.strip().str.replace('"',"")
        for c in ["道具发放(步骤1)","道具使用(步骤2)"]:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")
        df["步骤2 转化"]=pd.to_numeric(df["步骤2 转化"].astype(str).str.replace("%",""),errors="coerce")
    tw=tw[tw["道具ID"]!="总体"].copy(); lw=lw[lw["道具ID"]!="总体"].copy()
    m=tw.rename(columns={"道具发放(步骤1)":"本周发放","道具使用(步骤2)":"本周使用","步骤2 转化":"本周使用率"}
    ).merge(lw[["道具ID","道具发放(步骤1)","道具使用(步骤2)","步骤2 转化"]].rename(
        columns={"道具发放(步骤1)":"上周发放","道具使用(步骤2)":"上周使用","步骤2 转化":"上周使用率"}),
        on="道具ID",how="outer").fillna(0)
    m=m.merge(tm[["道具ID","活动","备注"]],on="道具ID",how="left")
    m["活动"]=m["活动"].fillna("其他"); m["备注"]=m["备注"].fillna(m["道具ID"])
    m["发放环比"]=(m["本周发放"]-m["上周发放"])/m["上周发放"].replace(0,np.nan)*100
    m["使用环比"]=(m["本周使用"]-m["上周使用"])/m["上周使用"].replace(0,np.nan)*100
    return m.sort_values("本周发放",ascending=False)

def load_first_dep_ret():
    df=pd.read_csv(FILES["first_dep_ret"])
    df.columns=df.columns.str.strip().str.replace("\ufeff","")
    df["_d"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"]=df["_d"].str.replace("-","").fillna("")
    for c in ["当日","1日","2日","3日","4日","5日","6日","7日"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
    df["账变事件用户数"]=pd.to_numeric(df["账变事件用户数"],errors="coerce").fillna(0)
    stage=df[df["初始事件发生时间"]=="阶段值"].copy()
    daily=df[df["_yyyymmdd"].str.match(r"^\d{8}$",na=False)].copy()
    rr=daily[daily["指标"]=="留存率"].reset_index(drop=True)
    nr=daily[daily["指标"]=="留存人数"].reset_index(drop=True)
    tw_r=rr[rr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_r=rr[rr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    tw_n=nr[nr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_n=nr[nr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    def wa(rd,nd,col):
        if col not in rd.columns or len(rd)==0: return np.nan
        rv=rd[col].to_numpy(dtype=float,na_value=np.nan); ok=~np.isnan(rv)
        if not ok.any(): return np.nan
        w=nd["账变事件用户数"].to_numpy(dtype=float) if len(nd)==len(rd) else np.ones(len(rv))
        ww=w[ok]; return float(np.nanmean(rv[ok])) if ww.sum()==0 else float(np.average(rv[ok],weights=ww))
    R={"stage":stage,"tw_users":int(tw_n["账变事件用户数"].sum()),"lw_users":int(lw_n["账变事件用户数"].sum())}
    for col in ["1日","2日","3日","4日","5日","6日","7日"]:
        R[f"tw_{col}"]=wa(tw_r,tw_n,col); R[f"lw_{col}"]=wa(lw_r,lw_n,col)
    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    R["tw_platform_fc"]=int(dp[dp["日期"].between(*THIS_WEEK)]["首充人数"].sum())
    R["lw_platform_fc"]=int(dp[dp["日期"].between(*LAST_WEEK)]["首充人数"].sum())
    return R

def load_risk(dt_raw, dc_raw):
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    ba=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    wa=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    bc=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数") if "投注次数" in df["分析指标"].values else pd.Series(name="投注次数")
    tg=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
        .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
        .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏","阶段汇总":"主游投注"}).reset_index())
    ug_list=[ba,wa]
    if len(bc)>0: ug_list.append(bc)
    ug=pd.concat(ug_list,axis=1).reset_index()
    if "投注次数" not in ug.columns: ug["投注次数"]=np.nan
    ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]; ug=ug.merge(tg,on="账户ID",how="left")
    top500=dt_raw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0
    top500["投充比"]=top500["投注金额"]/top500["充值金额"].replace(0,np.nan)
    hr=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
    pp_check=df[(df["游戏名称"].str.contains("Auto-Roulette",na=False))&(df["分析指标"]=="投注金额")]["账户ID"].unique() if "游戏名称" in df.columns else []
    c207=[u for u in pp_check if str(u).startswith("207")]
    c207_dt=dt_raw[dt_raw["账户ID"].isin(c207)].sort_values("提款金额",ascending=False) if len(c207)>0 else pd.DataFrame()
    sp=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False) if "投充比" in top500.columns else pd.DataFrame()
    bg=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
    wg=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
    ug2=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
    wn=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bg.merge(wg,on=["show_name_厂商标签id","游戏名称"],how="left").merge(ug2,on=["show_name_厂商标签id","游戏名称"],how="left").merge(wn,on=["show_name_厂商标签id","游戏名称"],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100; g["人均投注额"]=g["投注金额"]/g["玩家数"]
    rg=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    return hr,c207_dt,c207,sp,g.sort_values("投注金额",ascending=False).head(20),rg,top500

# ════════════════════════════════════════════════════════════════
# 图表（与v7.5完全相同，略去重复）
# ════════════════════════════════════════════════════════════════
SPLIT = 7

def _vline(ax, n_dates):
    """在ax上画上周/本周分割线及标注文字。
    使用 transform=ax.get_xaxis_transform() 让y坐标在轴坐标系(0~1)中指定，
    完全避免依赖 get_ylim() 的时序问题，文字永远不会超出y轴范围。"""
    ax.axvline(SPLIT - .5, color="#94a3b8", ls="--", lw=1, alpha=.7)
    # xdata坐标，y用轴坐标(0~1)——transform保证始终在图内
    trans = ax.get_xaxis_transform()
    ax.text(SPLIT - 4, 0.93, "上周", ha="center", fontsize=7,
            color="#64748b", transform=trans)
    ax.text(SPLIT + 3, 0.93, "本周", ha="center", fontsize=7,
            color="#1d4ed8", transform=trans)

def chart_trend(trend):
    fig = plt.figure(figsize=(16, 10), facecolor="white")
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=.45, wspace=.3)
    dates = trend["dates"]
    x = range(len(dates))
    n = len(dates)   # 14

    def sp(a):
        a.spines["top"].set_visible(False)
        a.spines["right"].set_visible(False)
        a.grid(axis="y", alpha=.25)

    # ── 左上：充值 vs 提现 ──────────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    ax.bar(x, [v/10000 for v in trend["充值"]],
           color=["#bfdbfe"]*SPLIT + ["#1d4ed8"]*SPLIT, width=.7, label="充值")
    ax.plot(x, [v/10000 for v in trend["提现"]],
            "-o", color="#dc2626", lw=1.5, ms=3, label="提现")
    ax.set_title("充值 vs 提现（万USD）", fontsize=9, fontweight="bold")
    ax.set_xticks(list(x)); ax.set_xticklabels(dates, rotation=45, fontsize=7)
    ax.legend(fontsize=7, loc="upper left"); sp(ax); _vline(ax, n)

    # ── 右上：公司输赢 ──────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.bar(x, [v/10000 for v in trend["公司输赢"]],
            color=["#bbf7d0"]*SPLIT + ["#059669"]*SPLIT, width=.7)
    ax2.set_title("公司输赢（万USD）", fontsize=9, fontweight="bold")
    ax2.set_xticks(list(x)); ax2.set_xticklabels(dates, rotation=45, fontsize=7)
    sp(ax2); _vline(ax2, n)

    # ── 左下：首充人数（柱）vs 注册人数（线）────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.bar(x, trend["首充"],
            color=["#c7d2fe"]*SPLIT + ["#6366f1"]*SPLIT, width=.7, label="首充人数")
    ax3r = ax3.twinx()
    ax3r.plot(x, trend["注册"], "--D", color="#d97706", lw=1.5, ms=3, label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）", fontsize=9, fontweight="bold")
    ax3.set_xticks(list(x)); ax3.set_xticklabels(dates, rotation=45, fontsize=7)
    l1, lb1 = ax3.get_legend_handles_labels()
    l2, lb2 = ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2, lb1+lb2, fontsize=7, loc="upper left")
    ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False)
    ax3.grid(axis="y", alpha=.25)
    # 分割线 + 上周/本周标注（ax3有twinx，单独处理保持一致）
    ax3.axvline(SPLIT - .5, color="#94a3b8", ls="--", lw=1, alpha=.7)
    trans3 = ax3.get_xaxis_transform()
    ax3.text(SPLIT - 4, 0.93, "上周", ha="center", fontsize=7,
             color="#64748b", transform=trans3)
    ax3.text(SPLIT + 3, 0.93, "本周", ha="center", fontsize=7,
             color="#1d4ed8", transform=trans3)

    # ── 右下：充提差率 ──────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.bar(x, trend["充提差比"],
            color=["#e9d5ff"]*SPLIT + ["#7c3aed"]*SPLIT, width=.7)
    tm = np.mean(trend["充提差比"][SPLIT:])
    lm = np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tm, color="#7c3aed", ls=":", lw=1.5)
    ax4.axhline(lm, color="#94a3b8", ls=":", lw=1.2)
    # ★ 修复：用轴坐标transform固定x位置（0.98/0.02），彻底避免超出右边界
    trans4 = ax4.get_xaxis_transform()
    _bbox = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85)
    ax4.text(n - 1.0, tm + .3, f"本周均{tm:.1f}%", fontsize=7,
             color="#7c3aed", ha="right", clip_on=False,
             fontweight="bold", bbox=_bbox)
    ax4.text(0.0,     lm + .3, f"上周均{lm:.1f}%", fontsize=7,
             color="#64748b", ha="left",  clip_on=False,
             fontweight="bold", bbox=_bbox)
    ax4.set_title("充提差率（%）", fontsize=9, fontweight="bold")
    ax4.set_xticks(list(x)); ax4.set_xticklabels(dates, rotation=45, fontsize=7)
    # 给y轴留出顶部空间，防止均值标注文字与顶部刻度线重叠
    ymax = max(trend["充提差比"])
    ax4.set_ylim(0, ymax * 1.20)
    sp(ax4); _vline(ax4, n)

    fig.suptitle("近14日大盘核心指标趋势", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout(rect=[0, 0, 1, .98])
    return fig_img(fig, 16, 11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(14,6),facecolor="white")
    cols=[("第1日","次留","#1d4ed8"),("第2日","3留","#059669"),("第6日","7留","#7c3aed")]
    x=np.arange(len(weeks)); w=.25
    for i,(col,lbl,clr) in enumerate(cols):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-w,vals,w,label=lbl,color=clr,alpha=.85)
        for bar,v in zip(bars,vals):
            if not (isinstance(v,float) and np.isnan(v)):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
        tgt=RET_TARGETS.get(lbl)
        if tgt: ax.axhline(tgt,color=clr,ls="--",lw=1.2,alpha=.55)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周对比（含目标虚线）",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8,loc="upper left"); ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=.5)
    ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,6)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tc=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lc=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tc,w,label="本周",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,lc,w,label="上周",color="#93c5fd",alpha=.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tb=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lb=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tb,w,label="本周",color="#059669",alpha=.85); ax2.bar(x+w/2,lb,w,label="上周",color="#6ee7b7",alpha=.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=.85,width=.6); ax3.axhline(0,color="black",lw=.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vr):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=.28
    for ax,rt,t in [(ax1,"chg","充值→充值 留存率（%）"),(ax2,"act","充值→活跃 留存率（%）")]:
        tw1=[vr[rt]["tw"]["1日"].get(v,0) for v in vips]; lw1=[vr[rt]["lw"]["1日"].get(v,0) for v in vips]; tw3=[vr[rt]["tw"]["3日"].get(v,0) for v in vips]
        clr=("#1d4ed8","#93c5fd","#059669") if rt=="chg" else ("#7c3aed","#c4b5fd","#d97706")
        ax.bar(x-w,tw1,w,label="次日(本周)",color=clr[0],alpha=.85); ax.bar(x,lw1,w,label="次日(上周)",color=clr[1],alpha=.7); ax.bar(x+w,tw3,w,label="3日(本周)",color=clr[2],alpha=.75)
        ax.set_title(t,fontsize=10,fontweight="bold"); ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8); ax.legend(fontsize=7.5,loc="upper left"); ax.grid(axis="y",alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.suptitle("VIP各等级充值留存率",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10); names=top10["厂商显示名"].tolist() if "厂商显示名" in top10.columns else top10["show_name_厂商标签id"].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    sh=top10["占比"].tolist(); ls=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,sh,color=["#059669" if c>=l else "#dc2626" for c,l in zip(sh,ls)],alpha=.85,width=.6)
    ax1.plot(names,ls,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,sh,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.2,f"{s:.1f}%",ha="center",fontsize=7)
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=.85)
    ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=.7)
    ax2.axhline(0,color="black",lw=.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5); ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    t15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in t15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in t15.iterrows()]; lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in t15.iterrows()]
    x=np.arange(len(names)); w=.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=.85); ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=.7)
    for bar,v in zip(ax.patches[:len(names)],bets[::-1]): ax.text(bar.get_width()+.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8); ax.set_title("Top15游戏 投注金额（万USD）",fontsize=9,fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="x",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pg,ma):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=ma.index[:8].tolist(); vals=ma.values[:8].tolist(); tot=sum(vals); pcts=[v/tot*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-.05,-.18)); ax1.set_title("Top500提款用户 厂商偏好",fontsize=9,fontweight="bold")
    t12=pg.head(12); gn=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in t12.iterrows()]; gv=[r["本周投注"]/10000 for _,r in t12.iterrows()]; gc=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in t12.iterrows()]
    ax2.barh(range(len(gn))[::-1],gv,color=gc[::-1],alpha=.85); ax2.set_yticks(range(len(gn))); ax2.set_yticklabels(gn,fontsize=7.5); ax2.set_title("偏好游戏Top12（红=平台亏损）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万")); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act):
    t10=act.head(10); names=[str(r["name_账变opt_id"])[:12] for _,r in t10.iterrows()]; vals=[r["赠送金额"]/10000 for _,r in t10.iterrows()]; lw=[r["lw_赠"]/10000 for _,r in t10.iterrows()]; envs=[r["环比"] for _,r in t10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=.85); ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=.7); ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8); ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold"); ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    es=[v if not pd.isna(v) else 0 for v in envs]
    ax2.barh(range(len(names)),es[::-1],color=["#059669" if v>0 else "#dc2626" for v in es[::-1]],alpha=.85); ax2.axvline(0,color="black",lw=.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8); ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%")); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_tool_usage(tool):
    big=tool[tool["本周发放"]>=100].head(12); t12=big if len(big)>0 else tool.head(12)
    def _l(r):
        a=str(r.get("活动","")); n=str(r.get("备注",""))
        return f"{str(r['道具ID'])[:6]}\n{n[:8]}" if a=="其他" else f"[{a[:4]}]\n{n[:8]}"
    names=[_l(r) for _,r in t12.iterrows()]
    ti=[r["本周发放"] for _,r in t12.iterrows()]; li=[r["上周发放"] for _,r in t12.iterrows()]
    tr=[r["本周使用率"] if not pd.isna(r["本周使用率"]) else 0 for _,r in t12.iterrows()]; lr=[r["上周使用率"] if not pd.isna(r["上周使用率"]) else 0 for _,r in t12.iterrows()]
    fig=plt.figure(figsize=(16,9),facecolor="white"); gs=gridspec.GridSpec(2,2,figure=fig,hspace=.5,wspace=.35)
    x=np.arange(len(names)); w=.35
    ax1=fig.add_subplot(gs[0,0]); ax1.bar(x-w/2,ti,w,label="本周发放",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,li,w,label="上周发放",color="#93c5fd",alpha=.7)
    ax1.set_title("Top12道具 发放量",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(names,fontsize=6.5); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v/10000:.0f}万" if v>=10000 else f"{v:.0f}"))
    ax2=fig.add_subplot(gs[0,1]); ax2.bar(x-w/2,tr,w,label="本周使用率",color="#059669",alpha=.85); ax2.bar(x+w/2,lr,w,label="上周使用率",color="#6ee7b7",alpha=.7)
    ax2.set_title("Top12道具 使用率",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(names,fontsize=6.5); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    ax3=fig.add_subplot(gs[1,:])
    di=[r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0 for _,r in t12.iterrows()]
    du=[r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0 for _,r in t12.iterrows()]
    ax3.bar(x-w/2,di,w,label="发放量环比",color=["#059669" if v>=0 else "#dc2626" for v in di],alpha=.85)
    ax3.bar(x+w/2,du,w,label="使用量环比",color=["#7c3aed" if v>=0 else "#f97316" for v in du],alpha=.7)
    ax3.axhline(0,color="black",lw=.8); ax3.set_title("Top12道具 发放/使用量 环比变化（%）",fontsize=9,fontweight="bold")
    ax3.set_xticks(x); ax3.set_xticklabels(names,fontsize=6.5); ax3.legend(fontsize=7.5); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    plt.suptitle("道具发放与使用分析",fontsize=11,fontweight="bold",y=1.01); plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,10)

def chart_first_dep_ret(R):
    lv=[R.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    tv=[R.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white"); x=np.arange(7)
    ax.plot(x,lv,"-o",color="#93c5fd",lw=2,ms=6,label=f"上周（{LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}）")
    ax.plot(x,tv,"-o",color="#1d4ed8",lw=2,ms=6,label=f"本周（{THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}）")
    days=["D1","D2","D3","D4","D5","D6","D7"]
    for i,(lval,tval) in enumerate(zip(lv,tv)):
        if lval is not None and not (isinstance(lval,float) and np.isnan(lval)): ax.text(i,lval+.4,f"{lval:.1f}%",ha="center",fontsize=7.5,color="#64748b")
        if tval is not None and not (isinstance(tval,float) and np.isnan(tval)): ax.text(i,tval-1.2,f"{tval:.1f}%",ha="center",fontsize=7.5,color="#1d4ed8",fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(days,fontsize=9); ax.set_title("首次充值活动用户 充值留存趋势",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8.5,loc="upper right"); ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.tight_layout(); return fig_img(fig,13,5.5)

def chart_risk_scatter(top500):
    v=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy() if "游戏_投注金额" in top500.columns else top500[top500["投充比"].notna()].copy()
    v=v[v["投充比"]<200]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    ax.scatter(v["投充比"],v["公司输赢"]/10000,c=["#dc2626" if x<0 else "#059669" for x in v["公司输赢"]],s=[min(abs(x)/500+20,200) for x in v["公司输赢"]],alpha=.55,edgecolors="none")
    ax.axhline(0,color="black",lw=.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=.7); ax.text(21,ax.get_ylim()[0]*.9,"投充比=20x",fontsize=7.5,color="#d97706")
    for _,r in top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()].iterrows():
        ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),xytext=(r["投充比"]+5,r["公司输赢"]/10000-.2),fontsize=6.5,color="#991b1b",arrowprops=dict(arrowstyle="->",color="#991b1b",lw=.7))
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8); ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=.6,label="平台赢钱")],fontsize=8,loc="upper right"); ax.grid(alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(rg):
    t12=rg.head(12); names=[r["游戏名称"][:16] for _,r in t12.iterrows()]; losses=[abs(r["公司输赢"]) for _,r in t12.iterrows()]; rates=[r["盈亏率"] for _,r in t12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=.85)
    for bar,v in zip(ax1.patches,losses[::-1]): ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=.85); ax2.axvline(0,color="black",lw=.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)

# ════════════════════════════════════════════════════════════════
# 章节构建
# ════════════════════════════════════════════════════════════════

def _render(t, **kw):
    try: return t.format(**kw)
    except: return t

def build_overview(K, trend, weekly_ret, dash_ret):
    full = PW - 2*MARGIN
    S = [sec_title("一、大盘核心数据")]

    # ★★★ 修改点2：KPI卡片展示日均消耗，标签注明"日均" ★★★
    kpis = [
        ("充值金额",   f"{K['tw_充值金额']/10000:.1f}万",   f"{K['lw_充值金额']/10000:.1f}万",   K["pct_充值金额"],   True),
        ("提现金额",   f"{K['tw_提现金额']/10000:.1f}万",   f"{K['lw_提现金额']/10000:.1f}万",   K["pct_提现金额"],   False),
        ("充提差",     f"{K['tw_充提差']/10000:.1f}万",     f"{K['lw_充提差']/10000:.1f}万",     K["pct_充提差"],     True),
        ("充提差率",   f"{K['tw_充提差比']:.2f}%",          f"{K['lw_充提差比']:.2f}%",          K["pct_充提差比"],   True),
        ("公司输赢",   f"{K['tw_公司输赢']/10000:.1f}万",   f"{K['lw_公司输赢']/10000:.1f}万",   K["pct_公司输赢"],   True),
        ("盈亏率",     f"{K['tw_盈亏率']:.3f}%",            f"{K['lw_盈亏率']:.3f}%",            K["pct_盈亏率"],     True),
        ("注册人数",   f"{int(K['tw_注册人数']):,}",        f"{int(K['lw_注册人数']):,}",        K["pct_注册人数"],   True),
        ("首充人数",   f"{int(K['tw_首充人数']):,}",        f"{int(K['lw_首充人数']):,}",        K["pct_首充人数"],   True),
        ("日均活跃",   f"{K['tw_活跃人数']/7/10000:.1f}万", f"{K['lw_活跃人数']/7/10000:.1f}万", K["pct_活跃人数"],   True),
        ("投注金额",   f"{K['tw_投注金额']/10000:.0f}万",   f"{K['lw_投注金额']/10000:.0f}万",   K["pct_投注金额"],   True),
        ("全量ARPPU",  f"${K['tw_全量Arppu']:.2f}",         f"${K['lw_全量Arppu']:.2f}",         K["pct_全量Arppu"],  True),
        ("老用户ARPPU",f"${K['tw_老用户ARPPU']:.2f}",       f"${K['lw_老用户ARPPU']:.2f}",       K["pct_老用户ARPPU"],True),
        ("首充ARPPU",  f"${K['tw_首充Arppu']:.2f}",         f"${K['lw_首充Arppu']:.2f}",         K["pct_首充Arppu"],  True),
        ("总赠送金额", f"{K['tw_总赠送金额']/10000:.1f}万", f"{K['lw_总赠送金额']/10000:.1f}万", K["pct_总赠送金额"], False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",        f"{K['lw_赠送充值比']:.2f}%",        K["pct_赠送充值比"], False),
        ("首充转化率", f"{K['tw_首充转化率']:.1f}%",        f"{K['lw_首充转化率']:.1f}%",        K["pct_首充转化率"], True),
        ("首充次日留存",f"{K['tw_首充次日复充率']:.1f}%",   f"{K['lw_首充次日复充率']:.1f}%",   K["pct_首充次日复充率"],True),
        # ★ 推广消耗：日均展示，标签注明"(日均)"区分口径
        ("推广消耗(日均)", f"{K['tw_真实消耗_日均']/10000:.2f}万",
                          f"{K['lw_真实消耗_日均']/10000:.2f}万",
                          K["pct_真实消耗"], False),
        # ★ 充提差ROI：日均充提差 / 日均推广消耗（本周剔除最后一天保持口径一致）
        ("充提差ROI\n(日均充提差/日均消耗)",
         f"{K['tw_充提差ROI']:.2f}x",
         f"{K['lw_充提差ROI']:.2f}x",
         K["pct_充提差ROI"], True),
    ]
    S.append(kpi_card4(kpis, cols=4)); S.append(Spacer(1,8))
    cr_d = K["tw_充提差比"]-K["lw_充提差比"]; rd = K["tw_首充次日复充率"]-K["lw_首充次日复充率"]

    # ★★★ 修改点3：insight_box 中推广消耗说明改用日均口径并补充有效天数 ★★★
    S.append(insight_box([
        _render("充值{a:.1f}万（{b:+.1f}%），公司输赢{c:.1f}万（{d:+.1f}%）；"
                "推广日均消耗{f:.2f}万（本周{g}天有效，{e:+.1f}%）。",
                a=K["tw_充值金额"]/10000, b=K["pct_充值金额"],
                c=K["tw_公司输赢"]/10000, d=K["pct_公司输赢"],
                e=K["pct_真实消耗"],
                f=K["tw_真实消耗_日均"]/10000,
                g=K["tw_真实消耗_有效天"]),
        _render("充提差率{a:.2f}%（上周{b:.2f}%，{c:+.2f}pp）；盈亏率{d:.3f}%（上周{e:.3f}%）。",
                a=K["tw_充提差比"],b=K["lw_充提差比"],c=cr_d,d=K["tw_盈亏率"],e=K["lw_盈亏率"]),
        _render("首充人数{a:,}（{b:+.1f}%），转化率{c:.1f}%，首充ARPPU${d:.2f}（{e:+.1f}%）。",
                a=int(K["tw_首充人数"]),b=K["pct_首充人数"],c=K["tw_首充转化率"],d=K["tw_首充Arppu"],e=K["pct_首充Arppu"]),
        _render("首充次日充值留存{a:.1f}%（上周{b:.1f}%，{c:+.1f}pp）。",
                a=K["tw_首充次日复充率"],b=K["lw_首充次日复充率"],c=rd),
    ]))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))

    S.append(sub_title("大盘首充留存 vs 目标对比（来源：日报 首充2/3/7日复充率）"))
    specs = [
        ("次留", "tw_nd", "lw_nd", "tw_nd_end"),
        ("3留",  "tw_td", "lw_td", "tw_td_end"),
        ("7留",  "tw_sd", "lw_sd", "tw_sd_end"),
    ]
    ret_headers = ["留存类型","数据截止","本周实际","上周实际","周环比(pp)","目标","vs目标(pp)"]
    ret_rows = []
    for rtype, tw_k, lw_k, end_k in specs:
        tw_v = dash_ret.get(tw_k, np.nan); lw_v = dash_ret.get(lw_k, np.nan)
        end_lbl = fmt_lbl(dash_ret.get(end_k))
        tgt = RET_TARGETS.get(rtype, np.nan)
        wpp = (tw_v-lw_v) if not (np.isnan(tw_v) or np.isnan(lw_v)) else np.nan
        dpp = (tw_v-tgt) if not (np.isnan(tw_v) or np.isnan(tgt)) else np.nan
        tw_clr = C_GREEN if (not np.isnan(tw_v) and not np.isnan(tgt) and tw_v>=tgt) else C_RED
        wpp_clr = C_GREEN if (not np.isnan(wpp) and wpp>=0) else C_RED
        dpp_clr = C_GREEN if (not np.isnan(dpp) and dpp>=0) else C_RED
        ret_rows.append([
            cell(rtype, True, C_DARK),
            cell(end_lbl, False, C_GRAY, TA_CENTER),
            cell(fret(tw_v), False, tw_clr, TA_RIGHT),
            cell(fret(lw_v), False, C_GRAY, TA_RIGHT),
            cell(f"{wpp:+.1f}pp" if not np.isnan(wpp) else "-", False, wpp_clr, TA_RIGHT),
            cell(f"{tgt:.0f}%" if not np.isnan(tgt) else "-", True, C_TARGET, TA_CENTER),
            cell(f"{dpp:+.1f}pp" if not np.isnan(dpp) else "-", True, dpp_clr, TA_RIGHT),
        ])
    extra = [("BACKGROUND",(5,1),(5,-1), colors.HexColor("#eff6ff"))]
    S.append(dtable(ret_headers, ret_rows,
                    [full*x for x in [0.16,0.12,0.14,0.14,0.14,0.12,0.14]],
                    fsize=8, extra_style=extra))
    S.append(Spacer(1,4))
    nd_end=fmt_lbl(dash_ret.get("tw_nd_end")); td_end=fmt_lbl(dash_ret.get("tw_td_end")); sd_end=fmt_lbl(dash_ret.get("tw_sd_end"))
    S.append(P(f"本周截止：次留~{nd_end}，3留~{td_end}，7留~{sd_end}（剔除数据未满足lag天数的不完整日期）",7,False,C_GRAY))
    S.append(Spacer(1,8))

    S.append(sub_title("首充用户充值留存 — 近4周对比（含目标线）"))
    S.append(chart_ret_weekly(weekly_ret)); S.append(Spacer(1,4))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=this_w.get("第1日",0) or 0; d2=this_w.get("第2日",0) or 0; d6=this_w.get("第6日",0) or 0
    ld1=last_w.get("第1日",0) or 0; ld6=last_w.get("第6日",0) or 0
    S.append(insight_box([
        f"本周首充次日留存{d1:.1f}%，较上周{d1-ld1:+.1f}pp；3留{d2:.1f}%，7留{d6:.1f}%。",
        f"注：本周7日留存因截止日期不完整，以上周7留{ld6:.1f}%作为参考基准。",
    ], clr=C_AMBER))
    return S


def build_agents(agents, agent_ret, ends):
    full = PW - 2*MARGIN
    S = [sec_title("二、总代分析")]
    tw_nd_lbl = fmt_lbl(ends.get("tw_nd_end")); lw_nd_lbl = fmt_lbl(ends.get("lw_nd_end"))
    tw_3d_lbl = fmt_lbl(ends.get("tw_3d_end")); lw_3d_lbl = fmt_lbl(ends.get("lw_3d_end"))
    lw_7d_lbl = fmt_lbl(ends.get("lw_7d_end"))
    S.append(sub_title("全量总代表现（本周 vs 上周，按充值金额排序，含官方总代0）"))
    S.append(P(
        f"数据源：首充充值留存_**（1日=次留，2日=3留，6日=7留）  "
        f"截止：次留本{tw_nd_lbl}/上{lw_nd_lbl}，3留本{tw_3d_lbl}/上{lw_3d_lbl}，7留上{lw_7d_lbl}",
        6.5, False, C_GRAY)); S.append(Spacer(1,3))
    headers = ["ID","总代名称","注册(环比)","首充\n人数","充值\n(万)","充提差率\n(差值pp)",
               "消耗\n(万)","1级首充\n成本(本/上)",
               f"次留(本/上)\n~{tw_nd_lbl}/{lw_nd_lbl}",
               f"3留(本/上)\n~{tw_3d_lbl}/{lw_3d_lbl}",
               f"7留上周\n~{lw_7d_lbl}"]
    rows = []
    for _, r in agents.iterrows():
        aid = int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
        rname = str(r["总代.名称"])
        cr = r["充提差率"]; lw_cr = r.get("lw_充提差率", 0) or 0; cr_d = cr - lw_cr
        cost = (r.get("总消耗", 0) or 0) / 10000
        fcc = r.get("一级首充成本", 0) or 0; lfc = r.get("lw_fc_cost", 0) or 0
        reg = int(r["注册人数"]); reg_c = r.get("注册环比", 0) or 0
        cr_clr = C_GREEN if cr >= CR_HIGH else (C_RED if cr < CR_LOW else C_DARK)
        d = agent_ret.get(rname, {})
        tw_nd=d.get("tw_nd",np.nan); lw_nd=d.get("lw_nd",np.nan)
        tw_3d=d.get("tw_3d",np.nan); lw_3d=d.get("lw_3d",np.nan)
        lw_7d=d.get("lw_7d",np.nan)
        def _fmt_pair(tv, lv):
            if not np.isnan(tv) and not np.isnan(lv):
                return f"{tv:.1f}%/{lv:.1f}%", C_GREEN if tv>=lv else C_RED
            if not np.isnan(tv): return f"{tv:.1f}%/-", C_DARK
            return "-", C_GRAY
        nd_s,nd_clr=_fmt_pair(tw_nd,lw_nd); td_s,td_clr=_fmt_pair(tw_3d,lw_3d)
        rows.append([
            cell(str(aid),False,C_GRAY,TA_CENTER),
            cell(rname[:14],True,C_DARK,TA_LEFT),
            cell(f"{reg:,}/{'+' if reg_c>=0 else ''}{reg_c:.0f}%",
                 False,C_GREEN if reg_c>=0 else C_RED,TA_RIGHT),
            cell(f"{int(r['首充人数']):,}",False,C_DARK,TA_RIGHT),
            cell(f"{r['充值金额']/10000:.0f}",False,C_DARK,TA_RIGHT),
            cell(f"{cr:.1f}%/{'+' if cr_d>=0 else ''}{cr_d:.1f}pp",False,cr_clr,TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-",False,C_DARK,TA_RIGHT),
            cell(f"${fcc:.0f}/${lfc:.0f}" if fcc>0 else "-",False,C_DARK,TA_RIGHT),
            cell(nd_s,False,nd_clr,TA_RIGHT),
            cell(td_s,False,td_clr,TA_RIGHT),
            cell(fret(lw_7d),False,C_DARK,TA_RIGHT),
        ])
    cw = [full*x for x in [0.04,0.14,0.10,0.06,0.05,0.11,0.05,0.10,0.11,0.11,0.08]]
    S.append(dtable(headers, rows, cw, fsize=6.2))
    S.append(Spacer(1,4))
    S.append(P(f"★ 充提差率≥{CR_HIGH}%绿，<{CR_LOW}%红。次留截止本~{tw_nd_lbl}/上~{lw_nd_lbl}；"
               f"3留截止本~{tw_3d_lbl}/上~{lw_3d_lbl}；7留上周截止~{lw_7d_lbl}。",
               6.5,False,C_GRAY)); S.append(Spacer(1,6))
    ins = []
    t1 = agents.iloc[0]
    t1_cr_d = t1["充提差率"] - (t1.get("lw_充提差率",0) or 0)
    ins.append(f"体量最大总代「{str(t1['总代.名称'])[:12]}」充值{t1['充值金额']/10000:.0f}万，"
               f"充提差率{t1['充提差率']:.1f}%（{t1_cr_d:+.1f}pp），"
               f"注册{int(t1['注册人数']):,}人（{t1.get('注册环比',0):+.0f}%）。")
    best = agents[agents["充值金额"]>10000].nlargest(1,"充提差率")
    if len(best)>0:
        b = best.iloc[0]
        ins.append(f"充提差率最优渠道「{str(b['总代.名称'])[:12]}」达{b['充提差率']:.1f}%，"
                   f"充值规模{b['充值金额']/10000:.0f}万，质效双优。")
    crash = agents[agents["注册环比"].fillna(0) < -50] if "注册环比" in agents.columns else pd.DataFrame()
    if len(crash)>0:
        c = crash.sort_values("注册环比").iloc[0]
        ins.append(f"预警：「{str(c['总代.名称'])[:12]}」注册量环比{c.get('注册环比',0):+.0f}%"
                   f"至{int(c['注册人数']):,}人，建议排查投放异常。")
    S.append(insight_box(ins))
    return S

def build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_tw):
    full = PW - 2*MARGIN; S = [sec_title("三、用户分析")]
    S.append(sub_title("VIP等级分层分析")); S.append(chart_vip(tw_v, lw_v)); S.append(Spacer(1,4))
    tot = tw_v["充值金额"].sum()
    hvs = sum(tw_v.loc[v,"充值金额"] for v in [9,10,11] if v in tw_v.index)
    S.append(insight_box([
        f"VIP9-11高价值层合计贡献充值{hvs/tot*100:.1f}%，高端用户付费意愿持续强劲。",
        f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。" if 1 in tw_v.index else "VIP1基础层数据不可用。"
    ]))
    vr = load_vip_retention()
    S.append(sub_title("VIP各等级充值留存率")); S.append(chart_vip_retention(vr)); S.append(Spacer(1,4))
    S.append(insight_box([
        f"充值→活跃次日留存：VIP9达{vr['act']['tw']['1日'].get(9,0):.1f}%，VIP10达{vr['act']['tw']['1日'].get(10,0):.1f}%。",
        f"充值→充值次日留存：VIP10达{vr['chg']['tw']['1日'].get(10,0):.1f}%（上周{vr['chg']['lw']['1日'].get(10,0):.1f}%）。",
    ])); S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h=["分层","本周充值","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),
                     cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    S.append(insight_box([
        _render("Top10充值用户人均${a:,.0f}（{b:+.1f}%），充提差率{c:.1f}%。",
                a=dep_t[1]["tw_avg"],b=pct(dep_t[1]["tw_avg"],dep_t[1]["lw_avg"]),c=dep_t[1]["tw_cr"]),
        "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。"
    ]))
    S.append(sub_title("头部提款用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","赢家比例","活动占比","占全量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdr_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                      cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                      rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),
                      cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),
                      cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
                      cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),
                      cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(h3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box([
        f"剔除Top100提款用户后，大盘充提差率影响{wdr_t[3]['大盘影响']:+.2f}pp，头部提款用户对充提差率有明显拖累。",
        f"Top500提款用户活动奖励占资金来源仅{wdr_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。",
    ], clr=C_AMBER))
    return S


def build_games(mfr, top30, delta):
    full = PW - 2*MARGIN; S = [sec_title("四、游戏分析")]
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    h=["排名","厂商","日均投注人数(本/上)","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r.get("厂商显示名",r["show_name_厂商标签id"])),True),
                     cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                     cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
                     cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
    top1 = mfr.iloc[0]
    S.append(insight_box([
        _render("{nm}投注份额{ts:.1f}%（上周{tsl:.1f}%），Fortune系稳固主导；环比{tc:+.1f}%。",
                nm=top1.get("厂商显示名",top1["show_name_厂商标签id"]),ts=top1["占比"],tsl=top1.get("lw_占比",0),tc=top1.get("投注环比",0)),
        "3Oaks、PlayTech盈亏率高于平台均值，可适当扩大曝光权重；关注盈亏率持续偏低厂商的RTP设置。",
        "自研游戏份额稳定，盈亏率需持续优化，建议加强对高频自研玩法的收益监控。",
    ]))
    if delta:
        S.append(sub_title("▶ 厂商投注额环比主要驱动游戏"))
        dh=["厂商","投注额环比","增量最大游戏(+贡献)","降量最大游戏(-拖累)"]; dr=[]
        for mn,gd in delta.items():
            mrow=mfr[mfr["show_name_厂商标签id"]==mn]; mc=mrow["投注环比"].values[0] if len(mrow)>0 else 0
            up=gd["up"]; dn=gd["dn"]
            us=f'{up.iloc[0]["游戏.名称"][:16]}（+${up.iloc[0]["delta"]/10000:.1f}万）' if len(up)>0 and up.iloc[0]["delta"]>0 else "-"
            ds=f'{dn.iloc[0]["游戏.名称"][:16]}（${dn.iloc[0]["delta"]/10000:.1f}万）' if len(dn)>0 and dn.iloc[0]["delta"]<0 else "-"
            dr.append([cell(shorten_mfr(mn),True,C_DARK),rc(mc),cell(us,False,C_GREEN if us!="-" else C_GRAY),cell(ds,False,C_RED if ds!="-" else C_GRAY)])
        S.append(dtable(dh,dr,[full*x for x in [0.15,0.10,0.37,0.38]],fsize=6.8)); S.append(Spacer(1,6))
    S.append(sub_title("Top30游戏详细数据")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
    h2=["#","游戏名称","厂商简称","投注人数","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["游戏.名称"])[:18],True),
                      cell(str(r.get("厂商简称",r["游戏厂商标签.名称"]))[:5]),
                      cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                      cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
                      cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h2,rows2,[full*x for x in [0.04,0.18,0.07,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S


def build_activities(act, tot_gift):
    full = PW - 2*MARGIN; S = [sec_title("五、活动分析")]
    S.append(sub_title("各活动赠送效果（全量，含环比）")); S.append(chart_activities(act)); S.append(Spacer(1,4))
    h=["活动名称","本周赠送","上周赠送","金额环比","本周日均\n赠送人数","上周日均\n赠送人数","人数环比","本周人均\n赠送金额","占比"]
    rows=[]
    for _,r in act.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:18],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"${r['lw_赠']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r["环比"] if not pd.isna(r.get("环比",np.nan)) else 0),
                     cell(f"{r.get('日均_本',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r.get('日均_上',0):.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r.get("人数环比",0) if not pd.isna(r.get("人数环比",np.nan)) else 0),
                     cell(f"${r.get('人均',0):.2f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.20,0.11,0.11,0.07,0.12,0.12,0.07,0.11,0.09]],fsize=6.5))
    S.append(Spacer(1,4)); daily=tot_gift/7/10000
    S.append(P(f"本周总赠送金额：${tot_gift/10000:.2f}万 | 日均赠送：${daily:.2f}万",9,True,C_DARK)); S.append(Spacer(1,6))
    S.append(insight_box([f"各类赠送活动全量列出（共{len(act)}个），合计本周日均赠送{daily:.1f}万USD。"]))
    return S


def build_tools(tool, fdr):
    full = PW - 2*MARGIN; S = [sec_title("六、道具专题分析", clr=C_TEAL)]
    S.append(sub_title("6.1 道具发放、使用及使用率（本周发放≥100）"))
    S.append(chart_tool_usage(tool)); S.append(Spacer(1,4))
    td=tool[tool["本周发放"]>=100].copy()
    h=["道具ID","活动类型","备注说明","本周发放","上周发放","发放环比","本周使用","上周使用","使用环比","本周\n使用率","上周\n使用率"]
    rows=[]
    for _,r in td.iterrows():
        di=r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0
        du=r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0
        tw_rt=r.get("本周使用率",0) or 0; lw_rt=r.get("上周使用率",0) or 0
        rows.append([cell(str(r["道具ID"]),False,C_GRAY,TA_CENTER),cell(str(r.get("活动",""))[:10],False,C_PURPLE),
                     cell(str(r.get("备注",""))[:14],False,C_DARK),
                     cell(f"{int(r['本周发放']):,}" if r['本周发放']>0 else "-",False,C_DARK,TA_RIGHT),
                     cell(f"{int(r['上周发放']):,}" if r['上周发放']>0 else "-",False,C_GRAY,TA_RIGHT),rc(di),
                     cell(f"{int(r['本周使用']):,}" if r['本周使用']>0 else "-",False,C_DARK,TA_RIGHT),
                     cell(f"{int(r['上周使用']):,}" if r['上周使用']>0 else "-",False,C_GRAY,TA_RIGHT),rc(du),
                     cell(f"{tw_rt:.1f}%",False,C_GREEN if tw_rt>=lw_rt else C_RED,TA_RIGHT),
                     cell(f"{lw_rt:.1f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.07,0.10,0.14,0.08,0.08,0.07,0.08,0.08,0.07,0.07,0.08]],fsize=6.3))
    S.append(Spacer(1,4))
    bv=tool[tool["本周发放"]>=100].copy()
    def _lbl(r):
        a=str(r.get("活动","")); n=str(r.get("备注",""))
        return f"{n}({r['本周使用率']:.0f}%)" if a=="其他" else f"[{a}]{n}({r['本周使用率']:.0f}%)"
    if len(bv)>=3:
        hi="、".join([_lbl(r) for _,r in bv.nlargest(3,"本周使用率").iterrows()])
        lo="、".join([_lbl(r) for _,r in bv.nsmallest(3,"本周使用率").iterrows()])
    else: hi=lo="-"
    tw_t=int(tool["本周发放"].sum()); lw_t=tool["上周发放"].sum()
    S.append(insight_box([f"使用率最高3类：{hi}。", f"使用率最低3类：{lo}。",
                          f"本周总道具发放{tw_t:,}，较上周{lw_t:,.0f}，环比{pct(tw_t,lw_t):+.1f}%。"]))
    S.append(Spacer(1,8))
    S.append(sub_title("6.2 首次充值活动用户 充值留存分析（重点）"))
    stage=fdr.get("stage",pd.DataFrame())
    if len(stage)>0:
        lws=LAST_WEEK[0]
        S.append(P(f"▸ 两周阶段汇总（{lws[:4]}.{lws[4:6]}.{lws[6:]} - {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}）",9,True,C_DARK)); S.append(Spacer(1,3))
        sr=stage[stage["指标"]=="留存率"]; sn=stage[stage["指标"]=="留存人数"]; sa=stage[stage["指标"]=="人均充值金额"]
        if len(sr)>0:
            u=int(sn["账变事件用户数"].values[0]) if len(sn)>0 else 0
            S.append(P(f"首充活动用户总数：{u:,}人",8.5,False,C_DARK))
            dc=["当日","1日","2日","3日","4日","5日","6日","7日"]; srows=[]
            if len(sr)>0: srows.append(["留存率"]+[str(sr.iloc[0].get(c,"-")) for c in dc])
            if len(sn)>0: srows.append(["留存人数"]+[str(sn.iloc[0].get(c,"-")) for c in dc])
            if len(sa)>0: srows.append(["人均充值($)"]+[str(sa.iloc[0].get(c,"-")) for c in dc])
            if srows:
                tr2=[[cell(row[0],True,C_DARK)]+[cell(str(v),False,C_DARK,TA_CENTER) for v in row[1:]] for row in srows]
                S.append(dtable(["指标"]+dc,tr2,[full*.12]+[full*.11]*8,fsize=6.8))
        S.append(Spacer(1,6))
    S.append(P("▸ 本周 vs 上周 首充活动用户留存趋势",9,True,C_DARK)); S.append(Spacer(1,3))
    S.append(chart_first_dep_ret(fdr)); S.append(Spacer(1,4))
    tw_u=fdr.get("tw_users",0); lw_u=fdr.get("lw_users",0)
    tw_pf=fdr.get("tw_platform_fc",0); lw_pf=fdr.get("lw_platform_fc",0)
    tw_r=tw_u/tw_pf*100 if tw_pf>0 else 0; lw_r=lw_u/lw_pf*100 if lw_pf>0 else 0
    tv=[fdr.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    lv=[fdr.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    def _fr(v): return f"{v:.1f}%" if not (v is None or (isinstance(v,float) and np.isnan(v))) else "-"
    rh=["周次","活动用户数","平台首充\n人数","活动用户\n占比","D1留存","D2留存","D3留存","D4留存","D5留存","D6留存","D7留存"]
    rrows=[
        [cell(f"本周 {THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}",True,C_BLUE),
         cell(f"{tw_u:,}",False,C_DARK,TA_RIGHT),cell(f"{tw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{tw_r:.1f}%",False,C_PURPLE,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GREEN if not pd.isna(v) and not pd.isna(lval) and v>=lval else(C_RED if not pd.isna(v) and not pd.isna(lval) and v<lval else C_DARK),TA_RIGHT) for v,lval in zip(tv,lv)],
        [cell(f"上周 {LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}",True,C_GRAY),
         cell(f"{lw_u:,}",False,C_GRAY,TA_RIGHT),cell(f"{lw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{lw_r:.1f}%",False,C_GRAY,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GRAY,TA_RIGHT) for v in lv],
    ]
    S.append(dtable(rh,rrows,[full*.16,full*.09,full*.09,full*.08]+[full*.084]*7,fsize=7,zebra=False)); S.append(Spacer(1,4))
    tw_d1=fdr.get("tw_1日",np.nan); lw_d1=fdr.get("lw_1日",np.nan); ins=[]
    if not np.isnan(tw_d1) and not np.isnan(lw_d1):
        d=tw_d1-lw_d1; ins.append(f"首充活动用户次日留存{tw_d1:.1f}%（上周{lw_d1:.1f}%，{d:+.1f}pp），{'留存改善' if d>0 else '留存下降，建议优化次日触达策略'}。")
    ins.append(f"本周活动用户{tw_u:,}人，占平台首充{tw_r:.1f}%（上周{lw_u:,}/{lw_r:.1f}%）；人均赠送$3.89，属高价值用户来源。")
    ins.append("建议：对D1/D2留存用户设置阶梯式再充值道具激励；对D3后流失用户做专项召回（24h内触达效果最佳）。")
    S.append(insight_box(ins, clr=C_AMBER))
    return S


def build_risk(hr, c207_dt, c207, sp, top20g, rg, top500, dt_full):
    full = PW - 2*MARGIN; S = [sec_title("七、用户游戏风险专项分析", clr=colors.HexColor("#7c2d12"))]
    tot_tx=top500["提款金额"].sum(); tot_win=top500["公司输赢"].sum(); wns=(top500["公司输赢"]<0).sum()
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${tot_tx/10000:.1f}万",""),
                           ("平台净赔付",f"${abs(tot_win)/10000:.1f}万","平台向该群体净赔"),
                           ("赢家比例",f"{wns/500*100:.1f}%",f"{wns}赢/{500-wns}输"),
                           ("高风险用户",f"{len(hr)}人","公司净输>$10,000")]:
        fw=full/4-4
        row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],
                           colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    hl=abs(hr["公司输赢"].sum())/10000 if len(hr)>0 else 0
    S.append(insight_box([
        f"Top500提款用户中赢家{wns}人（{wns/500*100:.1f}%），平台净赔付${abs(tot_win)/10000:.1f}万。",
        f"{len(hr)}名超级赢家（公司净输>$10,000）合计导致平台净输${hl:.1f}万，占整体净赔付{hl/abs(tot_win)*10000*100 if tot_win!=0 else 0:.1f}%。",
    ]))
    if len(hr)>0:
        S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
        h=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注均额","主玩厂商","主玩游戏","风险标签"]
        rows=[]
        for _,r in hr.iterrows():
            ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0
            avg=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
            tags=[]
            if r["充值金额"]<5000: tags.append("低充高提")
            if ratio>50: tags.append("超高投充")
            if avg>200: tags.append("高额单注")
            rows.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),
                         cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),cell(f"${avg:.0f}",False,C_RED if avg>200 else C_DARK,TA_RIGHT),
                         cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),
                         cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
        S.append(dtable(h,rows,[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    if len(c207)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警"))
        S.append(P(f"集群账户数：{len(c207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
        ch=["账户ID","提款金额","充值金额","公司输赢","特征"]; cr=[]
        for _,r in c207_dt.head(10).iterrows():
            cr.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell("充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}")])
        if len(c207_dt)>10: cr.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
        S.append(dtable(ch,cr,[full*x for x in [0.22,0.18,0.18,0.18,0.24]])); S.append(Spacer(1,6))
        S.append(insight_box([
            f"{len(c207)}个账户充值$547-548，均获活动奖励，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。",
            "建议：①冻结账户提款；②审查注册IP/设备指纹；③排查奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。",
        ], clr=C_AMBER))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    th=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; tr=[]
    for i,(_,r) in enumerate(dt_full.head(20).iterrows()):
        win=r["公司输赢"]<0
        tr.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),
                   cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),
                   cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),
                   cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(th,tr,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
    pg,ma=load_pref()
    S.append(sub_title("Top500提款用户游戏偏好")); S.append(chart_pref(pg,ma)); S.append(Spacer(1,4))
    if len(rg)>0:
        S.append(sub_title("▶ 高危游戏专项分析")); S.append(chart_risk_games(rg)); S.append(Spacer(1,4))
        gh=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; gr=[]
        for _,r in rg.head(12).iterrows():
            pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rc2=C_RED if rl in("极高","高") else C_AMBER
            gr.append([cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),
                       cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),
                       cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),
                       cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),
                       cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rc2)])
        S.append(dtable(gh,gr,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5)); S.append(Spacer(1,6))
        top_rg = rg.iloc[0] if len(rg)>0 else None
        ins1 = f"高危游戏{top_rg['游戏名称']}（{top_rg['show_name_厂商标签id']}）公司输赢${top_rg['公司输赢']:,.0f}，盈亏率{top_rg['盈亏率']:.1f}%，建议复审RTP参数。" if top_rg is not None else "本周未检测到极高风险游戏。"
        S.append(insight_box([ins1,"建议对高赢家率游戏实施单账户赢额上限，防范套利风险。"], clr=C_RED))
    return S


def build_conclusion(K, dep_t, wdr_t, mfr, act, rg, hr, c207):
    full = PW - 2*MARGIN; S = [sec_title("八、总结与行动建议")]
    cr_d = K["tw_充提差比"] - K["lw_充提差比"]
    rd   = K["tw_首充次日复充率"] - K["lw_首充次日复充率"]

    S.append(sub_title("▶ 本周亮点")); bg_g=colors.HexColor("#f0fdf4")
    highlights = []
    highlights.append(f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），"
                      f"充提差{K['tw_充提差']/10000:.1f}万，"
                      f"充提差率{K['tw_充提差比']:.2f}%（{cr_d:+.2f}pp）。")
    if K["pct_首充人数"] > 0:
        highlights.append(f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），"
                          f"首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。")
    if K["pct_公司输赢"] > 0:
        highlights.append(f"公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%），"
                          f"盈亏率{K['tw_盈亏率']:.3f}%，较上周改善。")
    top1_mfr = mfr.iloc[0]
    highlights.append(f"{top1_mfr.get('厂商显示名',top1_mfr['show_name_厂商标签id'])}投注份额"
                      f"{top1_mfr['占比']:.1f}%（{top1_mfr.get('投注环比',0):+.1f}%），Fortune系稳固主导。")
    for h in highlights:
        S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_g),
                                        ("LEFTPADDING",(0,0),(-1,-1),12),
                                        ("TOPPADDING",(0,0),(-1,-1),3),
                                        ("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))

    S.append(sub_title("▶ 风险预警")); bg_r=colors.HexColor("#fff5f5")
    risks = []
    if len(c207)>0:
        risks.append(f"PP Auto-Roulette 1批量套利：{len(c207)}个账户盈亏率极低，需立即处置。")
    if K["pct_首充Arppu"] < -5 or rd < -1:
        risks.append(f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），"
                     f"首充次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp），新用户质量需关注。")
    if len(hr)>0:
        hl=abs(hr["公司输赢"].sum())/10000
        risks.append(f"{len(hr)}名高风险用户（公司净输>$10,000）合计净输${hl:.1f}万，需立即人工审核。")
    top_rg_r = rg.iloc[0] if len(rg)>0 else None
    if top_rg_r is not None and top_rg_r["公司输赢"] < -20000:
        risks.append(f"高危游戏「{top_rg_r['游戏名称'][:16]}」公司输赢${top_rg_r['公司输赢']:,.0f}，"
                     f"赢家率{top_rg_r['赢家率']:.0f}%，建议复核RTP参数。")
    if wdr_t[3]["大盘影响"] < -1:
        risks.append(f"头部提款用户对充提差率拖累{wdr_t[3]['大盘影响']:+.2f}pp（剔除Top100后）。")

    # ★★★ 修改点4：推广消耗风险预警改用日均口径 ★★★
    if K["pct_真实消耗"] < -15:
        risks.append(
            f"推广日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（本周{K['tw_真实消耗_有效天']}天有效口径），"
            f"较上周日均{K['lw_真实消耗_日均']/10000:.2f}万{K['pct_真实消耗']:+.1f}%，"
            f"建议确认是否计划内削减，注意对首充量的滞后影响。"
        )

    if not risks:
        risks.append("本周暂无重大风险预警，各指标维持正常区间。")
    for r in risks:
        S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_r),
                                        ("LEFTPADDING",(0,0),(-1,-1),12),
                                        ("TOPPADDING",(0,0),(-1,-1),3),
                                        ("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))

    S.append(sub_title("▶ 行动建议"))
    actions = []
    if len(c207)>0:
        actions.append(("[紧急]","处置批量套利账号",
                        f"共{len(c207)}个账户，建议冻结提款、审查IP/设备指纹、修复奖励触发漏洞并实施赢额上限。"))
    if len(hr)>0:
        top_hr = hr.sort_values("公司输赢").iloc[0]
        actions.append(("[紧急]","处置高风险提款用户",
                        f"账户{top_hr['账户ID']}：提款${top_hr['提款金额']:,.0f}，"
                        f"公司输赢${top_hr['公司输赢']:,.0f}，高度异常，建议立即人工审核。"))
    if K["pct_首充Arppu"] < -5 or rd < -1:
        actions.append(("[本周]","优化首充质量与次日激活",
                        f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），"
                        f"次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp）。"
                        f"建议注册后1h/24h内推送首充引导，落地页突出$10+档。"))
    if top_rg_r is not None and top_rg_r["公司输赢"] < -20000:
        actions.append(("[本周]","高危游戏风险管控",
                        f"对赢家率>60%的高危游戏实施单账户赢额上限，"
                        f"同步复核「{top_rg_r['游戏名称'][:16]}」RTP参数，防范系统性套利。"))
    if K["pct_真实消耗"] < -15:
        actions.append(("[本周]","关注推广消耗削减影响",
                        f"日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（{K['pct_真实消耗']:+.1f}%），"
                        f"若非计划内，建议与投放团队确认，监控首充量未来1-2周的滞后变化。"))
    if not actions:
        actions.append(("[常规]","持续监控核心指标",
                        "充值、留存、充提差率等核心指标维持正常，建议持续监控并保持现有运营节奏。"))
    pmap={"[紧急]":C_RED,"[本周]":C_AMBER,"[常规]":C_GREEN}
    rows=[[cell(pri,True,pmap.get(pri[:4],C_GRAY),TA_CENTER),
           cell(title,True,C_DARK),cell(desc,False,C_GRAY)]
          for pri,title,desc in actions]
    S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows,[full*x for x in [0.1,0.22,0.68]]))
    return S


def header_footer(c, doc):
    c.saveState(); w, h = A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN, h-17, f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN, h-17, f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN, 5, f"第 {doc.page} 页")
    c.restoreState()


def main():
    root   = DATA_ROOT
    outdir = OUTPUT_DIR
    outdir.mkdir(parents=True, exist_ok=True)
    print("📊 MX 周报 PDF v7.5.2 生成中（充提差ROI新增版）...")
    resolve_files(root); setup()
    print("  ▶ 加载数据...")
    K, trend         = load_platform()
    # 打印消耗和ROI口径确认信息
    print(f"  ▶ 推广消耗口径确认：本周{K['tw_真实消耗_有效天']}天有效，"
          f"日均{K['tw_真实消耗_日均']/10000:.2f}万；"
          f"上周日均{K['lw_真实消耗_日均']/10000:.2f}万；"
          f"日均环比{K['pct_真实消耗']:+.1f}%")
    print(f"  ▶ 充提差ROI确认：本周{K['tw_充提差ROI']:.2f}x（日均充提差{K['tw_充提差_日均_roi']/10000:.2f}万/日均消耗{K['tw_真实消耗_日均']/10000:.2f}万）；"
          f"上周{K['lw_充提差ROI']:.2f}x；环比{K['pct_充提差ROI']:+.1f}%")
    dash_ret         = load_dashboard_retention()
    weekly_ret       = load_weekly_retention()
    agents           = load_agents()
    agent_ret, ends  = load_agent_ret()
    tw_v, lw_v       = load_vip()
    dep_t, wdr_t, dc_tw, dt_top = load_top_users()
    top30            = load_games()
    mfr              = load_mfr()
    mfr_delta        = load_mfr_game_delta()
    act_df, tot_gift = load_activities()
    tool_df          = load_tool_data()
    fdr              = load_first_dep_ret()

    dt_raw = pd.read_csv(FILES["dt_tw"])
    dc_raw = pd.read_csv(FILES["dc_tw"])
    for df in [dt_raw, dc_raw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")

    hr, c207_dt, c207, sp, top20g, rg, top500 = load_risk(dt_raw, dc_raw)

    dpf = pd.read_csv(FILES["pref_tw"]); dpf["阶段汇总"] = dpf["阶段汇总"].apply(clean)
    tg = (dpf[dpf["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
          .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]]
          .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
    dt_top = dt_top.merge(tg, on="账户ID", how="left")

    print("  ▶ 导出风控Excel...")
    excel_out = outdir / f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out), engine="openpyxl") as xw:
        cols_hr = ["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
        hr[[c for c in cols_hr if c in hr.columns]].to_excel(xw, sheet_name="高风险用户", index=False)
        if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(xw, sheet_name="PP轮盘批量账号", index=False)
        if len(sp)>0:       sp[["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]].to_excel(xw, sheet_name="特殊异常用户", index=False)
        tool_df.to_excel(xw, sheet_name="道具使用情况", index=False)
    print(f"✅ 风控Excel: {excel_out}")

    print("  ▶ 构建PDF章节...")
    out = outdir / f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc = SimpleDocTemplate(str(out), pagesize=A4, leftMargin=MARGIN, rightMargin=MARGIN,
                            topMargin=MARGIN+22, bottomMargin=MARGIN+10,
                            title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story = [Spacer(1,3*cm),
             P("MX 平台数据周报", 30, True, C_DARK, TA_CENTER), Spacer(1,.5*cm),
             P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),
             Spacer(1,.2*cm),
             P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),
             Spacer(1,.5*cm), HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"), PageBreak()]

    story += build_overview(K, trend, weekly_ret, dash_ret);              story.append(PageBreak())
    story += build_agents(agents, agent_ret, ends);                        story.append(PageBreak())
    story += build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_top);        story.append(PageBreak())
    story += build_games(mfr, top30, mfr_delta);                          story.append(PageBreak())
    story += build_activities(act_df, tot_gift);                          story.append(PageBreak())
    story += build_tools(tool_df, fdr);                                   story.append(PageBreak())
    story += build_risk(hr, c207_dt, c207, sp, top20g, rg, top500, dt_top); story.append(PageBreak())
    story += build_conclusion(K, dep_t, wdr_t, mfr, act_df, rg, hr, c207)

    print("  ▶ 渲染PDF...")
    doc.build(story, onFirstPage=header_footer, onLaterPages=header_footer)
    size = out.stat().st_size / 1024
    print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out), str(excel_out)


if __name__ == "__main__":
    main()

📊 MX 周报 PDF v7.5.2 生成中（充提差ROI新增版）...
     ✅ [platform       ] 平台报表_USD_20260530111756.xlsx
     ✅ [daily          ] 日报-大盘日报_USD_20260530113940.xlsx
     ✅ [retention      ] 整体 首充留存（近7天）_20260502-20260529.csv
     ✅ [agent_plat     ] 平台报表-总代_USD_20260530113324.xlsx
     ✅ [agent_promo    ] 推广报表-总代_USD_20260530113133.xlsx
     ✅ [agent_ret      ] 首充充值留存_全量数据_20260502_20260529.csv
     ✅ [vip            ] VIP报表_USD_20260530114435.xlsx
     ✅ [dt_tw          ] top提款用户_全量数据_20260523_20260529_日期对比20260516_20260522.csv
     ✅ [dt_lw          ] top提款用户_全量数据_20260516_20260522_日期对比20260509_20260515 (1).csv
     ✅ [dc_tw          ] 头部充值用户_全量数据_20260523_20260529_日期对比20260516_20260522.csv
     ✅ [dc_lw          ] 头部充值用户_全量数据_20260516_20260522_日期对比20260509_20260515.csv
     ✅ [pref_tw        ] 本周top500提款用户游戏偏好_全量数据_20260523_20260529.csv
     ✅ [pref_lw        ] 上周top500提款用户游戏偏好_全量数据_20260516_20260522.csv
     ✅ [mfr            ] 厂商投注数据_全量数据_20260516_20260529.csv
     ✅ [game_tw        ] 游戏报表-详情_USD_

In [5]:
"""
MX 平台数据周报 v7.5 — 全自动化版本（Windows 兼容）
使用方法：
  1. 把所有数据文件与本脚本放在同一目录
  2. 修改下方 THIS_WEEK / LAST_WEEK 两行为新周期
  3. 命令行运行：python mx_report_v75.py
     或在 Jupyter 中执行本脚本内容
"""
from pathlib import Path
import sys, os, io, warnings
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec, matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image, PageBreak, HRFlowable, KeepTogether)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# ── 路径配置 ─────────────────────────────────────────────
# ══════════════════════════════════════════════════════════
# ★★★ 路径配置（必须填写）★★★
#
#   DATA_ROOT：所有数据文件所在的文件夹路径
#   OUTPUT_DIR：PDF 和 Excel 输出位置（可与 DATA_ROOT 相同）
#
#   填写方式（Windows）：
#     DATA_ROOT = Path(r"C:\Users\你的名字\Desktop\MX数据")
#   或用正斜杠也可以：
#     DATA_ROOT = Path("C:/Users/你的名字/Desktop/MX数据")
# ══════════════════════════════════════════════════════════
DATA_ROOT  = Path(r"D:\周报更新版\MX")   # ← 改这里
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")   # ← 改这里（可与上面相同）

# ── 启动时检查路径是否有效 ──────────────────────────────
if not DATA_ROOT.exists():
    raise RuntimeError(
        f"\n\n❌ 数据文件夹不存在：{DATA_ROOT}\n"
        "请修改脚本开头的 DATA_ROOT 为正确路径，例如：\n"
        r'  DATA_ROOT = Path(r"C:\Users\你的名字\Desktop\MX数据")' + "\n"
    )

THIS_WEEK  = ("20260523", "20260529")
LAST_WEEK  = ("20260516", "20260522")
REPORT_END = THIS_WEEK[1]

# ── 留存目标 ─────────────────────────────────────────────
RET_TARGETS = {"次留": 21.0, "3留": 15.0, "7留": 11.0, "14留": 8.0, "30留": 6.0}

# ── 全局字体/颜色 ─────────────────────────────────────────
FN, FNB = "WQY", "WQYB"
FILES: dict = {}

C_BLUE   = colors.HexColor("#1d4ed8");  C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669");  C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706");  C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b");  C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white;                C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1");  C_ROW    = colors.HexColor("#f8fafc")
C_TEAL   = colors.HexColor("#0f766e");  C_TARGET = colors.HexColor("#0369a1")

PW, PH = A4
MARGIN  = 1.6 * cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
CR_HIGH, CR_LOW = 17, 5
MFR_SHORT = {"Rectangle":"RG","Pragmatic Play":"PP","PG Soft":"PG","PlayTech":"PT","Originals":"自研","Fat Panda":"FP"}

TRUNCATE = {"首充次日复充率":1,"首充次日复投率":1,"首充当日复充率":0,
            "首充7日复充率":6,"首充30日复充率":999,"首充2日复充率":1,"首充3日复充率":2}

# ── 工具函数 ──────────────────────────────────────────────
def shorten_mfr(n):
    if not isinstance(n, str): return str(n)
    for k, v in MFR_SHORT.items():
        if k in n: return v
    return n[:8]

def ret_end(week_start: str, lag: int):
    e = (datetime.strptime(REPORT_END, "%Y%m%d") - timedelta(days=lag)).strftime("%Y%m%d")
    return e if e >= week_start else None

def fmt_lbl(d):
    return f"{d[4:6]}/{d[6:]}" if d else "-"

def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c not in ("日期","总代.名称","name_总代","总代.ID")]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o):   return (n-o)/abs(o)*100 if o and o != 0 else 0.0
def pct_vec(ns, os):
    return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop else s
    nz = v[v > 0]; return nz.mean() if len(nz) else v.mean()

def wavg_series(vals, weights):
    ok = vals.notna() & (weights > 0)
    if not ok.any(): return np.nan
    return float(np.average(vals[ok], weights=weights[ok]))

def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

def find_chinese_font():
    import sys, os
    # ── Windows 常见中文字体路径 ──
    win_root = Path(os.environ.get("WINDIR", "C:/Windows"))
    win_cands = [
        win_root / "Fonts/msyh.ttc",       # 微软雅黑
        win_root / "Fonts/msyhbd.ttc",
        win_root / "Fonts/simhei.ttf",      # 黑体
        win_root / "Fonts/simsun.ttc",      # 宋体
        win_root / "Fonts/simkai.ttf",      # 楷体
        win_root / "Fonts/STKAITI.TTF",
        win_root / "Fonts/STXIHEI.TTF",
    ]
    # ── macOS / Linux 路径 ──
    other_cands = [
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        "/System/Library/Fonts/PingFang.ttc",
    ]
    for p in win_cands + other_cands:
        if Path(p).exists(): return str(p)
    # ── fallback：让 matplotlib 搜索 ──
    from matplotlib import font_manager as fm
    for f in fm.fontManager.ttflist:
        if any(k in f.name for k in
               ["YaHei","Hei","SimSun","SimHei","WenQuanYi","Noto Sans CJK","PingFang"]):
            if Path(f.fname).exists(): return f.fname
    raise FileNotFoundError(
        "未找到中文字体！请确认系统已安装微软雅黑(msyh.ttc)或黑体(simhei.ttf)。")

def setup():
    global FONT_PATH
    FONT_PATH = find_chinese_font()
    print(f"  ▶ 字体：{FONT_PATH}")
    ttc = FONT_PATH.lower().endswith(".ttc")
    pdfmetrics.registerFont(TTFont(FN,  FONT_PATH, subfontIndex=0) if ttc else TTFont(FN,  FONT_PATH))
    try:    pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1) if ttc else TTFont(FNB, FONT_PATH))
    except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0) if ttc else TTFont(FNB, FONT_PATH))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(FONT_PATH)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

def resolve_files(root):
    """自动模糊匹配文件，只需更新 THIS_WEEK/LAST_WEEK 即可复用（Windows 兼容）"""
    global FILES
    tw0, tw1 = THIS_WEEK
    lw0, lw1 = LAST_WEEK

    # 预先枚举所有文件（Windows 中文路径安全写法）
    all_files = []
    try:
        for h in root.rglob("*"):
            try:
                if h.is_file() and not h.name.startswith("~$"):
                    all_files.append(h)
            except (PermissionError, OSError):
                pass
    except Exception as e:
        print(f"  ⚠️  扫描目录出错: {e}")

    def find(kws):
        kws = kws if isinstance(kws, list) else [kws]
        hits = [h for h in all_files if all(k in h.name for k in kws)]
        return max(hits, key=lambda h: h.stat().st_mtime) if hits else None

    km = {
        "platform":      ["平台报表_USD"],
        "daily":         ["日报-大盘日报_USD"],
        "retention":     ["整体","首充留存"],
        "agent_plat":    ["平台报表-总代_USD"],
        "agent_promo":   ["推广报表-总代_USD"],
        "agent_ret":     ["首充充值留存_全量数据"],
        "vip":           ["VIP报表_USD"],
        "dt_tw":         [f"top提款用户_全量数据_{tw0}"],
        "dt_lw":         [f"top提款用户_全量数据_{lw0}"],
        "dc_tw":         [f"头部充值用户_全量数据_{tw0}"],
        "dc_lw":         [f"头部充值用户_全量数据_{lw0}"],
        "pref_tw":       [f"本周top500提款用户游戏偏好_全量数据_{tw0}"],
        "pref_lw":       [f"上周top500提款用户游戏偏好_全量数据_{lw0}"],
        "mfr":           ["厂商投注数据_全量数据"],
        "game_tw":       ["游戏报表-详情_USD_本周"],
        "game_lw":       ["游戏报表-详情_USD_上周"],
        "gift":          ["各赠送活动_全量数据"],
        "tool_map":      ["道具对应活动"],
        "vip_ret_chg":   ["VIP充值-充值_近28天"],
        "vip_ret_act":   ["VIP充值-活跃_近28天"],
        "tool_tw":       ["本周道具使用情况"],
        "tool_lw":       ["上周道具使用情况"],
        "first_dep_ret": ["首次充值活动用户充值留存情况"],
    }
    fail = 0
    for key, kws in km.items():
        p = find(kws)
        if p:
            FILES[key] = p
            print(f"     ✅ [{key:15s}] {p.name}")
        else:
            print(f"     ❌ [{key:15s}] 找不到含{kws}的文件")
            fail += 1
    if fail:
        raise FileNotFoundError(f"共{fail}个文件未找到")
    print(f"  ▶ 全部{len(km)}个文件匹配成功\n")

# ── PDF 样式工具 ─────────────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.4,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5), HRFlowable(width="100%",thickness=1.5,color=C_BLUE2),
                         Spacer(1,3), P(f"■  {text}",10,True,C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}",8.5,False,C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8, extra_style=None):
    full = PW - 2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),       ("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2),      ("BOTTOMPADDING",(0,0),(-1,-1),2),
        ("LEFTPADDING",(0,0),(-1,-1),2),     ("RIGHTPADDING",(0,0),(-1,-1),2),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    if extra_style:
        for cmd in extra_style: st.add(*cmd)
    hrow = [P(h,fsize,True,C_WHITE,TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]),fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c,(list,tuple)) else P(str(c),fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t,bold=False,clr=colors.black,align=TA_LEFT): return (t,bold,clr,align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup=good_up
    c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{'+' if v>=0 else ''}{v:.{d}f}%",False,c,TA_RIGHT)
def gclr(v,t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)
def fret(v): return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

def kpi_card4(items, cols=4):
    fw = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,_ in items:
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([[P(label,7.5,False,C_GRAY)],[P(str(tv),14,True,C_DARK)],
                       [P(f"上周：{lv}",7.5,False,C_GRAY)],
                       [P(f"{'+' if chg>=0 else ''}{chg:.1f}%",8,True,pclr)]],
                      colWidths=[fw],
                      style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
                                        ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
                                        ("LEFTPADDING",(0,0),(-1,-1),8),
                                        ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(fw,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════════════════════════

def load_platform():
    df = pd.read_excel(FILES["platform"]); df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]; lw_d = dd[dd["日期"].between(*LAST_WEEK)]
    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
              "首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    K["tw_真实消耗"] = tw_d["真实消耗"].iloc[:-1].sum() if len(tw_d)>1 else tw_d["真实消耗"].sum()
    K["lw_真实消耗"] = lw_d["真实消耗"].sum()
    K["pct_真实消耗"] = pct(K["tw_真实消耗"], K["lw_真实消耗"])
    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])
    last14 = df.tail(14)
    trend = {"dates": [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
             "充值": last14["充值金额"].tolist(), "提现": last14["提现金额"].tolist(),
             "充提差比": last14["充提差比"].tolist(), "公司输赢": last14["公司输赢"].tolist(),
             "首充": last14["首充人数"].tolist(), "注册": last14["注册人数"].tolist()}
    return K, trend


def load_dashboard_retention():
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str)
    specs = [("nd","首充2日复充率",1),("td","首充3日复充率",2),
             ("sd","首充7日复充率",6),("30d","首充30日复充率",29)]
    for _,col,_ in specs:
        if col in dd.columns:
            dd[col] = pd.to_numeric(dd[col].astype(str).str.replace("%","").str.strip(), errors="coerce")
    R = {}
    for key, col, lag in specs:
        for ws, we, prefix in [(THIS_WEEK[0],THIS_WEEK[1],"tw_"), (LAST_WEEK[0],LAST_WEEK[1],"lw_")]:
            ec = ret_end(ws, lag)
            sub = dd[(dd["日期"]>=ws) & (dd["日期"]<=we)]
            if ec: sub = sub[sub["日期"]<=ec]
            R[f"{prefix}{key}"] = float(sub[col].mean()) if len(sub) and col in sub else np.nan
        R[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        R[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)
    return R


def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
    daily["ds"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
    dr = daily[daily["指标"]=="留存率"].copy()
    du = daily[daily["指标"]=="留存人数"].copy()
    for c in ["第1日","第2日","第3日","第6日","第7日"]:
        if c in dr.columns:
            dr[c] = pd.to_numeric(dr[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
    def _d(s): return datetime.strptime(s,"%Y%m%d")
    def _f(d): return d.strftime("%Y-%m-%d")
    def _l(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s,tw_e = _d(THIS_WEEK[0]),_d(THIS_WEEK[1])
    lw_s,lw_e = _d(LAST_WEEK[0]),_d(LAST_WEEK[1])
    w2e = lw_s-timedelta(days=1); w2s = w2e-timedelta(days=6)
    w1e = w2s-timedelta(days=1); w1s = w1e-timedelta(days=6)
    weeks = [(f"第1周\n{_l(w1s,w1e)}",_f(w1s),_f(w1e)),
             (f"第2周\n{_l(w2s,w2e)}",_f(w2s),_f(w2e)),
             (f"上周\n{_l(lw_s,lw_e)}",_f(lw_s),_f(lw_e)),
             (f"本周\n{_l(tw_s,tw_e)}",_f(tw_s),_f(tw_e))]
    result = []
    for wk,s,e in weeks:
        mr = dr[(dr["ds"]>=s)&(dr["ds"]<=e)]; mu = du[(du["ds"]>=s)&(du["ds"]<=e)]
        row = {"week": wk, "users": mu["充值成功事件用户数"].sum()}
        for col in ["第1日","第2日","第3日","第6日","第7日"]:
            rs = mr[col].values if col in mr.columns else np.array([])
            us = mu["充值成功事件用户数"].values
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = float(np.average(rs[v],weights=us[v])) if v.sum()>0 else np.nan
            else: row[col] = float(np.nanmean(rs)) if len(rs)>0 else np.nan
        result.append(row)
    return result



def load_agents():
    dp = pd.read_excel(FILES["agent_plat"]); dr = pd.read_excel(FILES["agent_promo"])
    dp["日期"]=dp["日期"].astype(str); dr["日期"]=dr["日期"].astype(str)
    dp=to_num(dp); dr=to_num(dr)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]; lw_p=dp[dp["日期"].between(*LAST_WEEK)]
    tw_r=dr[dr["日期"].between(*THIS_WEEK)]; lw_r=dr[dr["日期"].between(*LAST_WEEK)]
    sa=["充值金额","提现金额","充提差","首充金额","首充人数","注册人数","充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sa if c in d.columns}).reset_index()
        for c in ["充提差比","首充次日充值留存"]:
            if c in d.columns: g=g.merge(d.groupby("总代.ID")[c].mean().rename(c),on="总代.ID",how="left")
        g["充提差率"]=g["充提差"]/g["充值金额"]*100; return g
    tw_pa=agg_p(tw_p); lw_pa=agg_p(lw_p)
    sr=["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        g=d.groupby("总代.ID").agg({c:"sum" for c in sr if c in d.columns}).reset_index()
        g["一级首充成本"]=g["总消耗"]/g["一级首充人数"].replace(0,np.nan); return g
    tw_ra=agg_r(tw_r); lw_ra=agg_r(lw_r)
    lw_ra["lw_fc_cost"]=lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
    m=tw_pa.merge(lw_pa[["总代.ID","充值金额","注册人数","充提差率"]].rename(
        columns={"充值金额":"lw_充值","注册人数":"lw_注册","充提差率":"lw_充提差率"}),on="总代.ID",how="left")
    m=m.merge(tw_ra[["总代.ID","总消耗","一级首充成本","一级首充人数"]],on="总代.ID",how="left")
    m=m.merge(lw_ra[["总代.ID","lw_fc_cost"]],on="总代.ID",how="left")
    m["注册环比"]=pct_vec(m["注册人数"],m["lw_注册"])
    return m.sort_values("充值金额",ascending=False)


def load_agent_ret():
    df = pd.read_csv(FILES["agent_ret"])
    df["_d"] = df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0].str.replace("-","")
    for c in ["1日","2日","6日"]:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
    df["首充用户数"] = pd.to_numeric(df["首充用户数"], errors="coerce").fillna(0)
    df["总代"] = pd.to_numeric(df["总代"], errors="coerce")
    daily = df[df["_d"].notna()&df["_d"].str.match(r"^\d{8}$")&(df["指标"]=="留存率")].copy()
    ends = {}
    for key, lag in [("nd",1),("3d",2),("7d",6)]:
        ends[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        ends[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)
    def _slice(ws, we, ec):
        sub = daily[daily["_d"].between(ws,we)]
        if ec: sub = sub[sub["_d"] <= min(ec,we)]
        return sub
    slices = {
        "tw_nd": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_nd_end"]),
        "tw_3d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_3d_end"]),
        "tw_7d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_7d_end"]),
        "lw_nd": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_nd_end"]),
        "lw_3d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_3d_end"]),
        "lw_7d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_7d_end"]),
    }
    col_map = {"tw_nd":"1日","tw_3d":"2日","tw_7d":"6日","lw_nd":"1日","lw_3d":"2日","lw_7d":"6日"}
    def _wavg_by_agent(df_sub, col):
        res = {}
        for nm, grp in df_sub.groupby("name_总代"):
            v = wavg_series(grp[col], grp["首充用户数"])
            if not np.isnan(v): res[nm] = v
        return res
    per_agent = {k: _wavg_by_agent(slices[k], col_map[k]) for k in slices}
    id_map = {}
    for nm, grp in daily.groupby("name_总代"):
        vals = grp["总代"].dropna().values
        if len(vals): id_map[nm] = int(vals[0])
    all_names = set().union(*[set(v) for v in per_agent.values()])
    ret = {}
    for nm in all_names:
        ret[nm] = {k: per_agent[k].get(nm, np.nan) for k in per_agent}
        ret[nm]["agent_id"] = id_map.get(nm)
    return ret, ends

def load_vip():
    df=pd.read_excel(FILES["vip"]); df["日期"]=df["日期"].astype(str)
    num=["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df=to_num(df,num)
    tw=df[df["日期"].between(*THIS_WEEK)]; lw=df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

def load_vip_retention():
    res={}
    for fk,rt in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df=pd.read_csv(FILES[fk])
        df["ds"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
        df["yyyymmdd"]=df["ds"].str.replace("-","")
        for c in ["1日","2日","3日","7日"]:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
        df["n"]=pd.to_numeric(df["充值成功事件用户数"],errors="coerce")
        df["vip"]=pd.to_numeric(df["vip_level"],errors="coerce")
        daily=df[(df["指标"]=="留存率")&df["yyyymmdd"].notna()&df["vip"].notna()]
        tw=daily[daily["yyyymmdd"].between(*THIS_WEEK)]; lw=daily[daily["yyyymmdd"].between(*LAST_WEEK)]
        def wavg(d,col):
            r={}
            for v,g in d.groupby("vip"):
                s=g[g[col].notna()]
                if len(s): r[int(v)]=float(np.average(s[col].values,weights=s["n"].values))
            return r
        res[rt]={"tw":{c:wavg(tw,c) for c in ["1日","2日","3日","7日"]},
                 "lw":{c:wavg(lw,c) for c in ["1日","2日","3日","7日"]}}
    return res

def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
    # 上周提款：文件已有标准列（提款金额/充值金额/公司输赢/活动奖励/投注金额）
    dt_lw_std = dt_lw[["账户ID","提款金额","充值金额","公司输赢","活动奖励","投注金额"]].copy()
    # 上周充值：文件已有标准列（充值金额/提款金额/公司输赢/活动奖励）
    dc_lw_std = dc_lw[["账户ID","充值金额","提款金额","公司输赢","活动奖励"]].copy()

    for df in [dc_tw, dc_lw_std, dt_tw, dt_lw_std]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c].astype(str).str.replace(",",""), errors="coerce")

    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]
    total_c=tw_p["充值金额"].sum(); total_t=tw_p["提现金额"].sum()
    actual_cr=(total_c-total_t)/total_c*100 if total_c>0 else 0
    tot_c=dc_tw["充值金额"].sum(); tot_t=dt_tw["提款金额"].sum()
    dep_t,wdr_t=[],[]
    for t in [1,10,50,100,200,500]:
        tw=dc_tw.head(t); lw=dc_lw_std.head(t)
        tc=tw["充值金额"].sum(); tt=tw["提款金额"].sum() if "提款金额" in tw.columns else 0
        lc=lw["充值金额"].sum(); lt=lw["提款金额"].sum() if "提款金额" in lw.columns else 0
        win=tw["公司输赢"].sum() if "公司输赢" in tw.columns else 0
        dep_t.append({"tier":f"Top{t}","tw_chg":tc,"lw_chg":lc,"tw_avg":tc/t,"lw_avg":lc/t,
            "tw_cr":(tc-tt)/tc*100 if tc>0 else 0,"lw_cr":(lc-lt)/lc*100 if lc>0 else 0,
            "tw_win":win,"占全量":tc/tot_c*100 if tot_c>0 else 0})
        tw2=dt_tw.head(t); lw2=dt_lw_std.head(t)
        tt2=tw2["提款金额"].sum(); tc2=tw2["充值金额"].sum()
        lt2=lw2["提款金额"].sum() if "提款金额" in lw2.columns else 0
        lc2=lw2["充值金额"].sum() if "充值金额" in lw2.columns else 0
        wns=(tw2["公司输赢"]<0).sum() if "公司输赢" in tw2.columns else 0
        act_sum=tw2["活动奖励"].sum() if "活动奖励" in tw2.columns else 0
        act_pct=act_sum/(tc2+act_sum)*100 if (tc2+act_sum)>0 else 0
        excl_c=total_c-tc2; excl_t=total_t-tt2
        excl_cr=(excl_c-excl_t)/excl_c*100 if excl_c>0 else 0
        wdr_t.append({"tier":f"Top{t}","tw_tx":tt2,"lw_tx":lt2,"tw_avg":tt2/t,"lw_avg":lt2/t,
            "tw_cr":(tc2-tt2)/tc2*100 if tc2>0 else 0,"lw_cr":(lc2-lt2)/lc2*100 if lc2>0 else 0,
            "赢家":wns,"总数":t,"赢家率":wns/t*100,"活动占比":act_pct,
            "占全量":tt2/tot_t*100 if tot_t>0 else 0,"大盘影响":excl_cr-actual_cr})
    return dep_t,wdr_t,dc_tw.head(200),dt_tw.head(20)

def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); dl=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); dl["阶段汇总"]=dl["阶段汇总"].apply(clean)
    bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]; bl=dl[dl["分析指标"]=="投注金额"]
    gb=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
    gw=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
    gl=bl.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
    gu=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([gb,gw,gl,gu],axis=1).reset_index()
    g["占比"]=g["本周投注"]/g["本周投注"].sum()*100; g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr

def load_games():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    tot=tw["投注金额"].sum()
    gt=tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    gl=lw.groupby("游戏.名称").agg(投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    gt["人均局数"]=gt["投注局数"]/gt["投注人数"]; gt["人均金额"]=gt["投注金额"]/gt["投注人数"]
    gt["盈亏率"]=gt["公司输赢"]/gt["投注金额"]*100; gt["占比"]=gt["投注金额"]/tot*100
    gt["厂商简称"]=gt["游戏厂商标签.名称"].apply(shorten_mfr)
    gm=gt.merge(gl,on="游戏.名称",how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)

def load_mfr():
    df=pd.read_csv(FILES["mfr"]); df=df.rename(columns={"盈利率":"盈亏率"})
    df=df[df["时间"]!="阶段汇总"].copy(); df["时间"]=df["时间"].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢","盈亏率"]:
        if c in df.columns: df[c]=df[c].apply(clean)
    tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
    def agg(d):
        g=d.groupby("show_name_厂商标签id").agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
    tg=agg(tw); lg=agg(lw)
    mg=tg.merge(lg[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on="show_name_厂商标签id",how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    mg["厂商显示名"]=mg["show_name_厂商标签id"].apply(shorten_mfr)
    return mg.sort_values("投注金额",ascending=False)

def load_mfr_game_delta():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    gt=tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index()
    gl=lw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index().rename(columns={"投注金额":"lw_投注"})
    m=gt.merge(gl,on=["游戏厂商标签.名称","游戏.名称"],how="outer").fillna(0); m["delta"]=m["投注金额"]-m["lw_投注"]
    mfr_tw=tw.groupby("游戏厂商标签.名称")["投注金额"].sum().sort_values(ascending=False)
    res={}
    for n in mfr_tw.head(10).index:
        sub=m[m["游戏厂商标签.名称"]==n].sort_values("delta",ascending=False)
        res[n]={"up":sub.head(1),"dn":sub.tail(1)}
    return res

def load_activities():
    """赠送活动 - 新格式：各赠送活动_全量数据_20260523_20260529_日期对比.csv"""
    df = pd.read_csv(FILES["gift"])
    # 新格式列：账变opt_code, name_账变opt_id, 原始时间, 赠送人数, 赠送金额, 对比时间1, 赠送人数.1, 赠送金额.1
    for c in ["赠送金额","赠送金额.1","赠送人数","赠送人数.1"]:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0)
    ag=df.groupby(["账变opt_code","name_账变opt_id"]).agg(
        赠送金额=("赠送金额","sum"),lw_赠=("赠送金额.1","sum"),
        赠送人数=("赠送人数","sum"),lw_人数=("赠送人数.1","sum")).reset_index()
    ag["环比"]=(ag["赠送金额"]-ag["lw_赠"])/ag["lw_赠"].replace(0,np.nan).abs()*100
    tot=ag["赠送金额"].sum(); ag["占比"]=ag["赠送金额"]/tot*100; ag["人均"]=ag["赠送金额"]/ag["赠送人数"].replace(0,np.nan)
    ag["日均_本"]=ag["赠送人数"]/7; ag["日均_上"]=ag["lw_人数"]/7
    ag["人数环比"]=(ag["赠送人数"]-ag["lw_人数"])/ag["lw_人数"].replace(0,np.nan)*100
    return ag.sort_values("赠送金额",ascending=False), tot

def load_tool_data():
    """道具数据 - 新格式无道具名称列，通过道具ID关联活动映射表"""
    tw=pd.read_csv(FILES["tool_tw"]); lw=pd.read_csv(FILES["tool_lw"])
    tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str).str.strip()
    tm=tm.drop_duplicates(subset="道具ID",keep="first")
    for df in [tw,lw]:
        df.columns=df.columns.str.strip().str.replace("\ufeff","")
        df["道具ID"]=df["道具ID"].astype(str).str.strip().str.replace('"',"")
        for c in ["道具发放(步骤1)","道具使用(步骤2)"]:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")
        df["步骤2 转化"]=pd.to_numeric(df["步骤2 转化"].astype(str).str.replace("%",""),errors="coerce")
    tw=tw[tw["道具ID"]!="总体"].copy(); lw=lw[lw["道具ID"]!="总体"].copy()
    m=tw.rename(columns={"道具发放(步骤1)":"本周发放","道具使用(步骤2)":"本周使用","步骤2 转化":"本周使用率"}
    ).merge(lw[["道具ID","道具发放(步骤1)","道具使用(步骤2)","步骤2 转化"]].rename(
        columns={"道具发放(步骤1)":"上周发放","道具使用(步骤2)":"上周使用","步骤2 转化":"上周使用率"}),
        on="道具ID",how="outer").fillna(0)
    m=m.merge(tm[["道具ID","活动","备注"]],on="道具ID",how="left")
    m["活动"]=m["活动"].fillna("其他"); m["备注"]=m["备注"].fillna(m["道具ID"])
    m["发放环比"]=(m["本周发放"]-m["上周发放"])/m["上周发放"].replace(0,np.nan)*100
    m["使用环比"]=(m["本周使用"]-m["上周使用"])/m["上周使用"].replace(0,np.nan)*100
    return m.sort_values("本周发放",ascending=False)

def load_first_dep_ret():
    """首次充值活动用户留存 - 源文件覆盖20260516-20260529，按周期切分"""
    df=pd.read_csv(FILES["first_dep_ret"])
    df.columns=df.columns.str.strip().str.replace("\ufeff","")
    df["_d"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"]=df["_d"].str.replace("-","").fillna("")
    for c in ["当日","1日","2日","3日","4日","5日","6日","7日"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
    df["账变事件用户数"]=pd.to_numeric(df["账变事件用户数"],errors="coerce").fillna(0)
    stage=df[df["初始事件发生时间"]=="阶段值"].copy()
    daily=df[df["_yyyymmdd"].str.match(r"^\d{8}$",na=False)].copy()
    rr=daily[daily["指标"]=="留存率"].reset_index(drop=True)
    nr=daily[daily["指标"]=="留存人数"].reset_index(drop=True)
    tw_r=rr[rr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_r=rr[rr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    tw_n=nr[nr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_n=nr[nr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    def wa(rd,nd,col):
        if col not in rd.columns or len(rd)==0: return np.nan
        rv=rd[col].to_numpy(dtype=float,na_value=np.nan); ok=~np.isnan(rv)
        if not ok.any(): return np.nan
        w=nd["账变事件用户数"].to_numpy(dtype=float) if len(nd)==len(rd) else np.ones(len(rv))
        ww=w[ok]; return float(np.nanmean(rv[ok])) if ww.sum()==0 else float(np.average(rv[ok],weights=ww))
    R={"stage":stage,"tw_users":int(tw_n["账变事件用户数"].sum()),"lw_users":int(lw_n["账变事件用户数"].sum())}
    for col in ["1日","2日","3日","4日","5日","6日","7日"]:
        R[f"tw_{col}"]=wa(tw_r,tw_n,col); R[f"lw_{col}"]=wa(lw_r,lw_n,col)
    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    R["tw_platform_fc"]=int(dp[dp["日期"].between(*THIS_WEEK)]["首充人数"].sum())
    R["lw_platform_fc"]=int(dp[dp["日期"].between(*LAST_WEEK)]["首充人数"].sum())
    return R

def load_risk(dt_raw, dc_raw):
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    ba=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    wa=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    bc=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数") if "投注次数" in df["分析指标"].values else pd.Series(name="投注次数")
    tg=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
        .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
        .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏","阶段汇总":"主游投注"}).reset_index())
    ug_list=[ba,wa]
    if len(bc)>0: ug_list.append(bc)
    ug=pd.concat(ug_list,axis=1).reset_index()
    if "投注次数" not in ug.columns: ug["投注次数"]=np.nan
    ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]; ug=ug.merge(tg,on="账户ID",how="left")
    top500=dt_raw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0
    top500["投充比"]=top500["投注金额"]/top500["充值金额"].replace(0,np.nan)
    hr=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
    # 检查是否存在批量套利账号（充值$547-548 + 主玩Auto-Roulette）
    pp_check=df[(df["游戏名称"].str.contains("Auto-Roulette",na=False))&(df["分析指标"]=="投注金额")]["账户ID"].unique() if "游戏名称" in df.columns else []
    c207=[u for u in pp_check if str(u).startswith("207")]
    c207_dt=dt_raw[dt_raw["账户ID"].isin(c207)].sort_values("提款金额",ascending=False) if len(c207)>0 else pd.DataFrame()
    sp=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False) if "投充比" in top500.columns else pd.DataFrame()
    bg=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
    wg=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
    ug2=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
    wn=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bg.merge(wg,on=["show_name_厂商标签id","游戏名称"],how="left").merge(ug2,on=["show_name_厂商标签id","游戏名称"],how="left").merge(wn,on=["show_name_厂商标签id","游戏名称"],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100; g["人均投注额"]=g["投注金额"]/g["玩家数"]
    rg=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    return hr,c207_dt,c207,sp,g.sort_values("投注金额",ascending=False).head(20),rg,top500

# ════════════════════════════════════════════════════════════════
# 图表
# ════════════════════════════════════════════════════════════════
SPLIT = 7

def _vline(ax, dates):
    ax.axvline(SPLIT-.5,color="#94a3b8",ls="--",lw=1,alpha=.7)
    ax.text(SPLIT-4, ax.get_ylim()[1]*.93,"上周",ha="center",fontsize=7,color="#64748b")
    ax.text(SPLIT+3, ax.get_ylim()[1]*.93,"本周",ha="center",fontsize=7,color="#1d4ed8")

def chart_trend(trend):
    fig=plt.figure(figsize=(16,10),facecolor="white")
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=.45,wspace=.3)
    dates=trend["dates"]; x=range(len(dates))
    def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=.25)
    ax=fig.add_subplot(gs[0,0])
    ax.bar(x,[v/10000 for v in trend["充值"]],color=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT,width=.7,label="充值")
    ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
    ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold"); ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
    ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax,dates)
    ax2=fig.add_subplot(gs[0,1])
    ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=.7)
    ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax2); _vline(ax2,dates)
    ax3=fig.add_subplot(gs[1,0])
    ax3.bar(x,trend["首充"],color=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT,width=.7,label="首充人数")
    ax3r=ax3.twinx(); ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold"); ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
    l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left"); ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=.25); ax3.axvline(SPLIT-.5,color="#94a3b8",ls="--",lw=1,alpha=.7)
    ax4=fig.add_subplot(gs[1,1])
    ax4.bar(x,trend["充提差比"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=.7)
    tm=np.mean(trend["充提差比"][SPLIT:]); lm=np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tm,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lm,color="#94a3b8",ls=":",lw=1.2)
    ax4.text(len(dates)-.5,tm+.3,f"本周均{tm:.1f}%",fontsize=7,color="#7c3aed",ha="right")
    ax4.text(.5,lm+.3,f"上周均{lm:.1f}%",fontsize=7,color="#64748b",ha="left")
    ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold"); ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax4); _vline(ax4,dates)
    fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
    plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(14,6),facecolor="white")
    cols=[("第1日","次留","#1d4ed8"),("第2日","3留","#059669"),("第6日","7留","#7c3aed")]
    x=np.arange(len(weeks)); w=.25
    for i,(col,lbl,clr) in enumerate(cols):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-w,vals,w,label=lbl,color=clr,alpha=.85)
        for bar,v in zip(bars,vals):
            if not (isinstance(v,float) and np.isnan(v)):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
        tgt=RET_TARGETS.get(lbl)
        if tgt: ax.axhline(tgt,color=clr,ls="--",lw=1.2,alpha=.55)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周对比（含目标虚线）",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8,loc="upper left"); ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=.5)
    ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,6)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tc=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lc=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tc,w,label="本周",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,lc,w,label="上周",color="#93c5fd",alpha=.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tb=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lb=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tb,w,label="本周",color="#059669",alpha=.85); ax2.bar(x+w/2,lb,w,label="上周",color="#6ee7b7",alpha=.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=.85,width=.6); ax3.axhline(0,color="black",lw=.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vr):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=.28
    for ax,rt,t in [(ax1,"chg","充值→充值 留存率（%）"),(ax2,"act","充值→活跃 留存率（%）")]:
        tw1=[vr[rt]["tw"]["1日"].get(v,0) for v in vips]; lw1=[vr[rt]["lw"]["1日"].get(v,0) for v in vips]; tw3=[vr[rt]["tw"]["3日"].get(v,0) for v in vips]
        clr=("#1d4ed8","#93c5fd","#059669") if rt=="chg" else ("#7c3aed","#c4b5fd","#d97706")
        ax.bar(x-w,tw1,w,label="次日(本周)",color=clr[0],alpha=.85); ax.bar(x,lw1,w,label="次日(上周)",color=clr[1],alpha=.7); ax.bar(x+w,tw3,w,label="3日(本周)",color=clr[2],alpha=.75)
        ax.set_title(t,fontsize=10,fontweight="bold"); ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8); ax.legend(fontsize=7.5,loc="upper left"); ax.grid(axis="y",alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.suptitle("VIP各等级充值留存率",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10); names=top10["厂商显示名"].tolist() if "厂商显示名" in top10.columns else top10["show_name_厂商标签id"].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    sh=top10["占比"].tolist(); ls=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,sh,color=["#059669" if c>=l else "#dc2626" for c,l in zip(sh,ls)],alpha=.85,width=.6)
    ax1.plot(names,ls,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,sh,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.2,f"{s:.1f}%",ha="center",fontsize=7)
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=.85)
    ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=.7)
    ax2.axhline(0,color="black",lw=.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5); ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    t15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in t15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in t15.iterrows()]; lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in t15.iterrows()]
    x=np.arange(len(names)); w=.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=.85); ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=.7)
    for bar,v in zip(ax.patches[:len(names)],bets[::-1]): ax.text(bar.get_width()+.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8); ax.set_title("Top15游戏 投注金额（万USD）",fontsize=9,fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="x",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pg,ma):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=ma.index[:8].tolist(); vals=ma.values[:8].tolist(); tot=sum(vals); pcts=[v/tot*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-.05,-.18)); ax1.set_title("Top500提款用户 厂商偏好",fontsize=9,fontweight="bold")
    t12=pg.head(12); gn=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in t12.iterrows()]; gv=[r["本周投注"]/10000 for _,r in t12.iterrows()]; gc=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in t12.iterrows()]
    ax2.barh(range(len(gn))[::-1],gv,color=gc[::-1],alpha=.85); ax2.set_yticks(range(len(gn))); ax2.set_yticklabels(gn,fontsize=7.5); ax2.set_title("偏好游戏Top12（红=平台亏损）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万")); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act):
    t10=act.head(10); names=[str(r["name_账变opt_id"])[:12] for _,r in t10.iterrows()]; vals=[r["赠送金额"]/10000 for _,r in t10.iterrows()]; lw=[r["lw_赠"]/10000 for _,r in t10.iterrows()]; envs=[r["环比"] for _,r in t10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=.85); ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=.7); ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8); ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold"); ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    es=[v if not pd.isna(v) else 0 for v in envs]
    ax2.barh(range(len(names)),es[::-1],color=["#059669" if v>0 else "#dc2626" for v in es[::-1]],alpha=.85); ax2.axvline(0,color="black",lw=.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8); ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%")); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_tool_usage(tool):
    big=tool[tool["本周发放"]>=100].head(12); t12=big if len(big)>0 else tool.head(12)
    def _l(r):
        a=str(r.get("活动","")); n=str(r.get("备注",""))
        return f"{str(r['道具ID'])[:6]}\n{n[:8]}" if a=="其他" else f"[{a[:4]}]\n{n[:8]}"
    names=[_l(r) for _,r in t12.iterrows()]
    ti=[r["本周发放"] for _,r in t12.iterrows()]; li=[r["上周发放"] for _,r in t12.iterrows()]
    tr=[r["本周使用率"] if not pd.isna(r["本周使用率"]) else 0 for _,r in t12.iterrows()]; lr=[r["上周使用率"] if not pd.isna(r["上周使用率"]) else 0 for _,r in t12.iterrows()]
    fig=plt.figure(figsize=(16,9),facecolor="white"); gs=gridspec.GridSpec(2,2,figure=fig,hspace=.5,wspace=.35)
    x=np.arange(len(names)); w=.35
    ax1=fig.add_subplot(gs[0,0]); ax1.bar(x-w/2,ti,w,label="本周发放",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,li,w,label="上周发放",color="#93c5fd",alpha=.7)
    ax1.set_title("Top12道具 发放量",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(names,fontsize=6.5); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v/10000:.0f}万" if v>=10000 else f"{v:.0f}"))
    ax2=fig.add_subplot(gs[0,1]); ax2.bar(x-w/2,tr,w,label="本周使用率",color="#059669",alpha=.85); ax2.bar(x+w/2,lr,w,label="上周使用率",color="#6ee7b7",alpha=.7)
    ax2.set_title("Top12道具 使用率",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(names,fontsize=6.5); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    ax3=fig.add_subplot(gs[1,:])
    di=[r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0 for _,r in t12.iterrows()]
    du=[r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0 for _,r in t12.iterrows()]
    ax3.bar(x-w/2,di,w,label="发放量环比",color=["#059669" if v>=0 else "#dc2626" for v in di],alpha=.85)
    ax3.bar(x+w/2,du,w,label="使用量环比",color=["#7c3aed" if v>=0 else "#f97316" for v in du],alpha=.7)
    ax3.axhline(0,color="black",lw=.8); ax3.set_title("Top12道具 发放/使用量 环比变化（%）",fontsize=9,fontweight="bold")
    ax3.set_xticks(x); ax3.set_xticklabels(names,fontsize=6.5); ax3.legend(fontsize=7.5); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    plt.suptitle("道具发放与使用分析",fontsize=11,fontweight="bold",y=1.01); plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,10)

def chart_first_dep_ret(R):
    lv=[R.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    tv=[R.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white"); x=np.arange(7)
    ax.plot(x,lv,"-o",color="#93c5fd",lw=2,ms=6,label=f"上周（{LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}）")
    ax.plot(x,tv,"-o",color="#1d4ed8",lw=2,ms=6,label=f"本周（{THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}）")
    days=["D1","D2","D3","D4","D5","D6","D7"]
    for i,(lval,tval) in enumerate(zip(lv,tv)):
        if lval is not None and not (isinstance(lval,float) and np.isnan(lval)): ax.text(i,lval+.4,f"{lval:.1f}%",ha="center",fontsize=7.5,color="#64748b")
        if tval is not None and not (isinstance(tval,float) and np.isnan(tval)): ax.text(i,tval-1.2,f"{tval:.1f}%",ha="center",fontsize=7.5,color="#1d4ed8",fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(days,fontsize=9); ax.set_title("首次充值活动用户 充值留存趋势",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8.5,loc="upper right"); ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.tight_layout(); return fig_img(fig,13,5.5)

def chart_risk_scatter(top500):
    v=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy() if "游戏_投注金额" in top500.columns else top500[top500["投充比"].notna()].copy()
    v=v[v["投充比"]<200]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    ax.scatter(v["投充比"],v["公司输赢"]/10000,c=["#dc2626" if x<0 else "#059669" for x in v["公司输赢"]],s=[min(abs(x)/500+20,200) for x in v["公司输赢"]],alpha=.55,edgecolors="none")
    ax.axhline(0,color="black",lw=.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=.7); ax.text(21,ax.get_ylim()[0]*.9,"投充比=20x",fontsize=7.5,color="#d97706")
    for _,r in top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()].iterrows():
        ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),xytext=(r["投充比"]+5,r["公司输赢"]/10000-.2),fontsize=6.5,color="#991b1b",arrowprops=dict(arrowstyle="->",color="#991b1b",lw=.7))
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8); ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=.6,label="平台赢钱")],fontsize=8,loc="upper right"); ax.grid(alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(rg):
    t12=rg.head(12); names=[r["游戏名称"][:16] for _,r in t12.iterrows()]; losses=[abs(r["公司输赢"]) for _,r in t12.iterrows()]; rates=[r["盈亏率"] for _,r in t12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=.85)
    for bar,v in zip(ax1.patches,losses[::-1]): ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=.85); ax2.axvline(0,color="black",lw=.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)

# ════════════════════════════════════════════════════════════════
# 章节构建
# ════════════════════════════════════════════════════════════════

def _render(t, **kw):
    try: return t.format(**kw)
    except: return t

def build_overview(K, trend, weekly_ret, dash_ret):
    full = PW - 2*MARGIN
    S = [sec_title("一、大盘核心数据")]
    kpis = [
        ("充值金额",   f"{K['tw_充值金额']/10000:.1f}万",   f"{K['lw_充值金额']/10000:.1f}万",   K["pct_充值金额"],   True),
        ("提现金额",   f"{K['tw_提现金额']/10000:.1f}万",   f"{K['lw_提现金额']/10000:.1f}万",   K["pct_提现金额"],   False),
        ("充提差",     f"{K['tw_充提差']/10000:.1f}万",     f"{K['lw_充提差']/10000:.1f}万",     K["pct_充提差"],     True),
        ("充提差率",   f"{K['tw_充提差比']:.2f}%",          f"{K['lw_充提差比']:.2f}%",          K["pct_充提差比"],   True),
        ("公司输赢",   f"{K['tw_公司输赢']/10000:.1f}万",   f"{K['lw_公司输赢']/10000:.1f}万",   K["pct_公司输赢"],   True),
        ("盈亏率",     f"{K['tw_盈亏率']:.3f}%",            f"{K['lw_盈亏率']:.3f}%",            K["pct_盈亏率"],     True),
        ("注册人数",   f"{int(K['tw_注册人数']):,}",        f"{int(K['lw_注册人数']):,}",        K["pct_注册人数"],   True),
        ("首充人数",   f"{int(K['tw_首充人数']):,}",        f"{int(K['lw_首充人数']):,}",        K["pct_首充人数"],   True),
        ("日均活跃",   f"{K['tw_活跃人数']/7/10000:.1f}万", f"{K['lw_活跃人数']/7/10000:.1f}万", K["pct_活跃人数"],   True),
        ("投注金额",   f"{K['tw_投注金额']/10000:.0f}万",   f"{K['lw_投注金额']/10000:.0f}万",   K["pct_投注金额"],   True),
        ("全量ARPPU",  f"${K['tw_全量Arppu']:.2f}",         f"${K['lw_全量Arppu']:.2f}",         K["pct_全量Arppu"],  True),
        ("老用户ARPPU",f"${K['tw_老用户ARPPU']:.2f}",       f"${K['lw_老用户ARPPU']:.2f}",       K["pct_老用户ARPPU"],True),
        ("首充ARPPU",  f"${K['tw_首充Arppu']:.2f}",         f"${K['lw_首充Arppu']:.2f}",         K["pct_首充Arppu"],  True),
        ("总赠送金额", f"{K['tw_总赠送金额']/10000:.1f}万", f"{K['lw_总赠送金额']/10000:.1f}万", K["pct_总赠送金额"], False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",        f"{K['lw_赠送充值比']:.2f}%",        K["pct_赠送充值比"], False),
        ("首充转化率", f"{K['tw_首充转化率']:.1f}%",        f"{K['lw_首充转化率']:.1f}%",        K["pct_首充转化率"], True),
        ("首充次日留存",f"{K['tw_首充次日复充率']:.1f}%",   f"{K['lw_首充次日复充率']:.1f}%",   K["pct_首充次日复充率"],True),
        ("推广消耗",   f"{K['tw_真实消耗']/10000:.1f}万",   f"{K['lw_真实消耗']/10000:.1f}万",   K["pct_真实消耗"],   False),
    ]
    S.append(kpi_card4(kpis, cols=4)); S.append(Spacer(1,8))
    cr_d = K["tw_充提差比"]-K["lw_充提差比"]; rd = K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(insight_box([
        _render("充值{a:.1f}万（{b:+.1f}%），公司输赢{c:.1f}万（{d:+.1f}%）；推广消耗{e:+.1f}%。",
                a=K["tw_充值金额"]/10000,b=K["pct_充值金额"],c=K["tw_公司输赢"]/10000,d=K["pct_公司输赢"],e=K["pct_真实消耗"]),
        _render("充提差率{a:.2f}%（上周{b:.2f}%，{c:+.2f}pp）；盈亏率{d:.3f}%（上周{e:.3f}%）。",
                a=K["tw_充提差比"],b=K["lw_充提差比"],c=cr_d,d=K["tw_盈亏率"],e=K["lw_盈亏率"]),
        _render("首充人数{a:,}（{b:+.1f}%），转化率{c:.1f}%，首充ARPPU${d:.2f}（{e:+.1f}%）。",
                a=int(K["tw_首充人数"]),b=K["pct_首充人数"],c=K["tw_首充转化率"],d=K["tw_首充Arppu"],e=K["pct_首充Arppu"]),
        _render("首充次日充值留存{a:.1f}%（上周{b:.1f}%，{c:+.1f}pp）。",
                a=K["tw_首充次日复充率"],b=K["lw_首充次日复充率"],c=rd),
    ]))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))

    S.append(sub_title("大盘首充留存 vs 目标对比（来源：日报 首充2/3/7日复充率）"))
    specs = [
        ("次留", "tw_nd", "lw_nd", "tw_nd_end"),
        ("3留",  "tw_td", "lw_td", "tw_td_end"),
        ("7留",  "tw_sd", "lw_sd", "tw_sd_end"),
    ]
    ret_headers = ["留存类型","数据截止","本周实际","上周实际","周环比(pp)","目标","vs目标(pp)"]
    ret_rows = []
    for rtype, tw_k, lw_k, end_k in specs:
        tw_v = dash_ret.get(tw_k, np.nan); lw_v = dash_ret.get(lw_k, np.nan)
        end_lbl = fmt_lbl(dash_ret.get(end_k))
        tgt = RET_TARGETS.get(rtype, np.nan)
        wpp = (tw_v-lw_v) if not (np.isnan(tw_v) or np.isnan(lw_v)) else np.nan
        dpp = (tw_v-tgt) if not (np.isnan(tw_v) or np.isnan(tgt)) else np.nan
        tw_clr = C_GREEN if (not np.isnan(tw_v) and not np.isnan(tgt) and tw_v>=tgt) else C_RED
        wpp_clr = C_GREEN if (not np.isnan(wpp) and wpp>=0) else C_RED
        dpp_clr = C_GREEN if (not np.isnan(dpp) and dpp>=0) else C_RED
        ret_rows.append([
            cell(rtype, True, C_DARK),
            cell(end_lbl, False, C_GRAY, TA_CENTER),
            cell(fret(tw_v), False, tw_clr, TA_RIGHT),
            cell(fret(lw_v), False, C_GRAY, TA_RIGHT),
            cell(f"{wpp:+.1f}pp" if not np.isnan(wpp) else "-", False, wpp_clr, TA_RIGHT),
            cell(f"{tgt:.0f}%" if not np.isnan(tgt) else "-", True, C_TARGET, TA_CENTER),
            cell(f"{dpp:+.1f}pp" if not np.isnan(dpp) else "-", True, dpp_clr, TA_RIGHT),
        ])
    extra = [("BACKGROUND",(5,1),(5,-1), colors.HexColor("#eff6ff"))]
    S.append(dtable(ret_headers, ret_rows,
                    [full*x for x in [0.16,0.12,0.14,0.14,0.14,0.12,0.14]],
                    fsize=8, extra_style=extra))
    S.append(Spacer(1,4))
    nd_end=fmt_lbl(dash_ret.get("tw_nd_end")); td_end=fmt_lbl(dash_ret.get("tw_td_end")); sd_end=fmt_lbl(dash_ret.get("tw_sd_end"))
    S.append(P(f"本周截止：次留~{nd_end}，3留~{td_end}，7留~{sd_end}（剔除数据未满足lag天数的不完整日期）",7,False,C_GRAY))
    S.append(Spacer(1,8))

    S.append(sub_title("首充用户充值留存 — 近4周对比（含目标线）"))
    S.append(chart_ret_weekly(weekly_ret)); S.append(Spacer(1,4))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=this_w.get("第1日",0) or 0; d2=this_w.get("第2日",0) or 0; d6=this_w.get("第6日",0) or 0
    ld1=last_w.get("第1日",0) or 0; ld6=last_w.get("第6日",0) or 0
    S.append(insight_box([
        f"本周首充次日留存{d1:.1f}%，较上周{d1-ld1:+.1f}pp；3留{d2:.1f}%，7留{d6:.1f}%。",
        f"注：本周7日留存因截止日期不完整，以上周7留{ld6:.1f}%作为参考基准。",
    ], clr=C_AMBER))
    return S



def build_agents(agents, agent_ret, ends):
    full = PW - 2*MARGIN
    S = [sec_title("二、总代分析")]
    tw_nd_lbl = fmt_lbl(ends.get("tw_nd_end")); lw_nd_lbl = fmt_lbl(ends.get("lw_nd_end"))
    tw_3d_lbl = fmt_lbl(ends.get("tw_3d_end")); lw_3d_lbl = fmt_lbl(ends.get("lw_3d_end"))
    lw_7d_lbl = fmt_lbl(ends.get("lw_7d_end"))
    S.append(sub_title("全量总代表现（本周 vs 上周，按充值金额排序，含官方总代0）"))
    S.append(P(
        f"数据源：首充充值留存_**（1日=次留，2日=3留，6日=7留）  "
        f"截止：次留本{tw_nd_lbl}/上{lw_nd_lbl}，3留本{tw_3d_lbl}/上{lw_3d_lbl}，7留上{lw_7d_lbl}",
        6.5, False, C_GRAY)); S.append(Spacer(1,3))
    headers = ["ID","总代名称","注册(环比)","首充\n人数","充值\n(万)","充提差率\n(差值pp)",
               "消耗\n(万)","1级首充\n成本(本/上)",
               f"次留(本/上)\n~{tw_nd_lbl}/{lw_nd_lbl}",
               f"3留(本/上)\n~{tw_3d_lbl}/{lw_3d_lbl}",
               f"7留上周\n~{lw_7d_lbl}"]
    rows = []
    for _, r in agents.iterrows():
        aid = int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
        rname = str(r["总代.名称"])
        cr = r["充提差率"]; lw_cr = r.get("lw_充提差率", 0) or 0; cr_d = cr - lw_cr
        cost = (r.get("总消耗", 0) or 0) / 10000
        fcc = r.get("一级首充成本", 0) or 0; lfc = r.get("lw_fc_cost", 0) or 0
        reg = int(r["注册人数"]); reg_c = r.get("注册环比", 0) or 0
        cr_clr = C_GREEN if cr >= CR_HIGH else (C_RED if cr < CR_LOW else C_DARK)
        d = agent_ret.get(rname, {})
        tw_nd=d.get("tw_nd",np.nan); lw_nd=d.get("lw_nd",np.nan)
        tw_3d=d.get("tw_3d",np.nan); lw_3d=d.get("lw_3d",np.nan)
        lw_7d=d.get("lw_7d",np.nan)
        def _fmt_pair(tv, lv):
            if not np.isnan(tv) and not np.isnan(lv):
                return f"{tv:.1f}%/{lv:.1f}%", C_GREEN if tv>=lv else C_RED
            if not np.isnan(tv): return f"{tv:.1f}%/-", C_DARK
            return "-", C_GRAY
        nd_s,nd_clr=_fmt_pair(tw_nd,lw_nd); td_s,td_clr=_fmt_pair(tw_3d,lw_3d)
        rows.append([
            cell(str(aid),False,C_GRAY,TA_CENTER),
            cell(rname[:14],True,C_DARK,TA_LEFT),
            cell(f"{reg:,}/{'+' if reg_c>=0 else ''}{reg_c:.0f}%",
                 False,C_GREEN if reg_c>=0 else C_RED,TA_RIGHT),
            cell(f"{int(r['首充人数']):,}",False,C_DARK,TA_RIGHT),
            cell(f"{r['充值金额']/10000:.0f}",False,C_DARK,TA_RIGHT),
            cell(f"{cr:.1f}%/{'+' if cr_d>=0 else ''}{cr_d:.1f}pp",False,cr_clr,TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-",False,C_DARK,TA_RIGHT),
            cell(f"${fcc:.0f}/${lfc:.0f}" if fcc>0 else "-",False,C_DARK,TA_RIGHT),
            cell(nd_s,False,nd_clr,TA_RIGHT),
            cell(td_s,False,td_clr,TA_RIGHT),
            cell(fret(lw_7d),False,C_DARK,TA_RIGHT),
        ])
    cw = [full*x for x in [0.04,0.14,0.10,0.06,0.05,0.11,0.05,0.10,0.11,0.11,0.08]]
    S.append(dtable(headers, rows, cw, fsize=6.2))
    S.append(Spacer(1,4))
    S.append(P(f"★ 充提差率≥{CR_HIGH}%绿，<{CR_LOW}%红。次留截止本~{tw_nd_lbl}/上~{lw_nd_lbl}；"
               f"3留截止本~{tw_3d_lbl}/上~{lw_3d_lbl}；7留上周截止~{lw_7d_lbl}。",
               6.5,False,C_GRAY)); S.append(Spacer(1,6))
    # 自动生成总代洞察
    ins = []
    t1 = agents.iloc[0]
    t1_cr_d = t1["充提差率"] - (t1.get("lw_充提差率",0) or 0)
    ins.append(f"体量最大总代「{str(t1['总代.名称'])[:12]}」充值{t1['充值金额']/10000:.0f}万，"
               f"充提差率{t1['充提差率']:.1f}%（{t1_cr_d:+.1f}pp），"
               f"注册{int(t1['注册人数']):,}人（{t1.get('注册环比',0):+.0f}%）。")
    best = agents[agents["充值金额"]>10000].nlargest(1,"充提差率")
    if len(best)>0:
        b = best.iloc[0]
        ins.append(f"充提差率最优渠道「{str(b['总代.名称'])[:12]}」达{b['充提差率']:.1f}%，"
                   f"充值规模{b['充值金额']/10000:.0f}万，质效双优。")
    crash = agents[agents["注册环比"].fillna(0) < -50] if "注册环比" in agents.columns else pd.DataFrame()
    if len(crash)>0:
        c = crash.sort_values("注册环比").iloc[0]
        ins.append(f"预警：「{str(c['总代.名称'])[:12]}」注册量环比{c.get('注册环比',0):+.0f}%"
                   f"至{int(c['注册人数']):,}人，建议排查投放异常。")
    S.append(insight_box(ins))
    return S

def build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_tw):
    full = PW - 2*MARGIN; S = [sec_title("三、用户分析")]
    S.append(sub_title("VIP等级分层分析")); S.append(chart_vip(tw_v, lw_v)); S.append(Spacer(1,4))
    tot = tw_v["充值金额"].sum()
    hvs = sum(tw_v.loc[v,"充值金额"] for v in [9,10,11] if v in tw_v.index)
    S.append(insight_box([
        f"VIP9-11高价值层合计贡献充值{hvs/tot*100:.1f}%，高端用户付费意愿持续强劲。",
        f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。" if 1 in tw_v.index else "VIP1基础层数据不可用。"
    ]))
    vr = load_vip_retention()
    S.append(sub_title("VIP各等级充值留存率")); S.append(chart_vip_retention(vr)); S.append(Spacer(1,4))
    S.append(insight_box([
        f"充值→活跃次日留存：VIP9达{vr['act']['tw']['1日'].get(9,0):.1f}%，VIP10达{vr['act']['tw']['1日'].get(10,0):.1f}%。",
        f"充值→充值次日留存：VIP10达{vr['chg']['tw']['1日'].get(10,0):.1f}%（上周{vr['chg']['lw']['1日'].get(10,0):.1f}%）。",
    ])); S.append(Spacer(1,8))

    S.append(sub_title("头部充值用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h=["分层","本周充值","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),
                     cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    S.append(insight_box([
        _render("Top10充值用户人均${a:,.0f}（{b:+.1f}%），充提差率{c:.1f}%。",
                a=dep_t[1]["tw_avg"],b=pct(dep_t[1]["tw_avg"],dep_t[1]["lw_avg"]),c=dep_t[1]["tw_cr"]),
        "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。"
    ]))

    S.append(sub_title("头部提款用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","赢家比例","活动占比","占全量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdr_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                      cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                      rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),
                      cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),
                      cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
                      cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),
                      cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(h3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box([
        f"剔除Top100提款用户后，大盘充提差率影响{wdr_t[3]['大盘影响']:+.2f}pp，头部提款用户对充提差率有明显拖累。",
        f"Top500提款用户活动奖励占资金来源仅{wdr_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。",
    ], clr=C_AMBER))
    return S


def build_games(mfr, top30, delta):
    full = PW - 2*MARGIN; S = [sec_title("四、游戏分析")]
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    h=["排名","厂商","日均投注人数(本/上)","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r.get("厂商显示名",r["show_name_厂商标签id"])),True),
                     cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                     cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
                     cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
    top1 = mfr.iloc[0]
    S.append(insight_box([
        _render("{nm}投注份额{ts:.1f}%（上周{tsl:.1f}%），Fortune系稳固主导；环比{tc:+.1f}%。",
                nm=top1.get("厂商显示名",top1["show_name_厂商标签id"]),ts=top1["占比"],tsl=top1.get("lw_占比",0),tc=top1.get("投注环比",0)),
        "3Oaks、PlayTech盈亏率高于平台均值，可适当扩大曝光权重；关注盈亏率持续偏低厂商的RTP设置。",
        "自研游戏份额稳定，盈亏率需持续优化，建议加强对高频自研玩法的收益监控。",
    ]))
    if delta:
        S.append(sub_title("▶ 厂商投注额环比主要驱动游戏"))
        dh=["厂商","投注额环比","增量最大游戏(+贡献)","降量最大游戏(-拖累)"]; dr=[]
        for mn,gd in delta.items():
            mrow=mfr[mfr["show_name_厂商标签id"]==mn]; mc=mrow["投注环比"].values[0] if len(mrow)>0 else 0
            up=gd["up"]; dn=gd["dn"]
            us=f'{up.iloc[0]["游戏.名称"][:16]}（+${up.iloc[0]["delta"]/10000:.1f}万）' if len(up)>0 and up.iloc[0]["delta"]>0 else "-"
            ds=f'{dn.iloc[0]["游戏.名称"][:16]}（${dn.iloc[0]["delta"]/10000:.1f}万）' if len(dn)>0 and dn.iloc[0]["delta"]<0 else "-"
            dr.append([cell(shorten_mfr(mn),True,C_DARK),rc(mc),cell(us,False,C_GREEN if us!="-" else C_GRAY),cell(ds,False,C_RED if ds!="-" else C_GRAY)])
        S.append(dtable(dh,dr,[full*x for x in [0.15,0.10,0.37,0.38]],fsize=6.8)); S.append(Spacer(1,6))
    S.append(sub_title("Top30游戏详细数据")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
    h2=["#","游戏名称","厂商简称","投注人数","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["游戏.名称"])[:18],True),
                      cell(str(r.get("厂商简称",r["游戏厂商标签.名称"]))[:5]),
                      cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                      cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
                      cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h2,rows2,[full*x for x in [0.04,0.18,0.07,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S


def build_activities(act, tot_gift):
    full = PW - 2*MARGIN; S = [sec_title("五、活动分析")]
    S.append(sub_title("各活动赠送效果（全量，含环比）")); S.append(chart_activities(act)); S.append(Spacer(1,4))
    h=["活动名称","本周赠送","上周赠送","金额环比","本周日均\n赠送人数","上周日均\n赠送人数","人数环比","本周人均\n赠送金额","占比"]
    rows=[]
    for _,r in act.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:18],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"${r['lw_赠']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r["环比"] if not pd.isna(r.get("环比",np.nan)) else 0),
                     cell(f"{r.get('日均_本',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r.get('日均_上',0):.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r.get("人数环比",0) if not pd.isna(r.get("人数环比",np.nan)) else 0),
                     cell(f"${r.get('人均',0):.2f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.20,0.11,0.11,0.07,0.12,0.12,0.07,0.11,0.09]],fsize=6.5))
    S.append(Spacer(1,4)); daily=tot_gift/7/10000
    S.append(P(f"本周总赠送金额：${tot_gift/10000:.2f}万 | 日均赠送：${daily:.2f}万",9,True,C_DARK)); S.append(Spacer(1,6))
    S.append(insight_box([f"各类赠送活动全量列出（共{len(act)}个），合计本周日均赠送{daily:.1f}万USD。"]))
    return S


def build_tools(tool, fdr):
    full = PW - 2*MARGIN; S = [sec_title("六、道具专题分析", clr=C_TEAL)]
    S.append(sub_title("6.1 道具发放、使用及使用率（本周发放≥100）"))
    S.append(chart_tool_usage(tool)); S.append(Spacer(1,4))
    td=tool[tool["本周发放"]>=100].copy()
    h=["道具ID","活动类型","备注说明","本周发放","上周发放","发放环比","本周使用","上周使用","使用环比","本周\n使用率","上周\n使用率"]
    rows=[]
    for _,r in td.iterrows():
        di=r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0
        du=r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0
        tw_rt=r.get("本周使用率",0) or 0; lw_rt=r.get("上周使用率",0) or 0
        rows.append([cell(str(r["道具ID"]),False,C_GRAY,TA_CENTER),cell(str(r.get("活动",""))[:10],False,C_PURPLE),
                     cell(str(r.get("备注",""))[:14],False,C_DARK),
                     cell(f"{int(r['本周发放']):,}" if r['本周发放']>0 else "-",False,C_DARK,TA_RIGHT),
                     cell(f"{int(r['上周发放']):,}" if r['上周发放']>0 else "-",False,C_GRAY,TA_RIGHT),rc(di),
                     cell(f"{int(r['本周使用']):,}" if r['本周使用']>0 else "-",False,C_DARK,TA_RIGHT),
                     cell(f"{int(r['上周使用']):,}" if r['上周使用']>0 else "-",False,C_GRAY,TA_RIGHT),rc(du),
                     cell(f"{tw_rt:.1f}%",False,C_GREEN if tw_rt>=lw_rt else C_RED,TA_RIGHT),
                     cell(f"{lw_rt:.1f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.07,0.10,0.14,0.08,0.08,0.07,0.08,0.08,0.07,0.07,0.08]],fsize=6.3))
    S.append(Spacer(1,4))
    bv=tool[tool["本周发放"]>=100].copy()
    def _lbl(r):
        a=str(r.get("活动","")); n=str(r.get("备注",""))
        return f"{n}({r['本周使用率']:.0f}%)" if a=="其他" else f"[{a}]{n}({r['本周使用率']:.0f}%)"
    if len(bv)>=3:
        hi="、".join([_lbl(r) for _,r in bv.nlargest(3,"本周使用率").iterrows()])
        lo="、".join([_lbl(r) for _,r in bv.nsmallest(3,"本周使用率").iterrows()])
    else: hi=lo="-"
    tw_t=int(tool["本周发放"].sum()); lw_t=tool["上周发放"].sum()
    S.append(insight_box([f"使用率最高3类：{hi}。", f"使用率最低3类：{lo}。",
                          f"本周总道具发放{tw_t:,}，较上周{lw_t:,.0f}，环比{pct(tw_t,lw_t):+.1f}%。"]))
    S.append(Spacer(1,8))

    S.append(sub_title("6.2 首次充值活动用户 充值留存分析（重点）"))
    stage=fdr.get("stage",pd.DataFrame())
    if len(stage)>0:
        lws=LAST_WEEK[0]
        S.append(P(f"▸ 两周阶段汇总（{lws[:4]}.{lws[4:6]}.{lws[6:]} - {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}）",9,True,C_DARK)); S.append(Spacer(1,3))
        sr=stage[stage["指标"]=="留存率"]; sn=stage[stage["指标"]=="留存人数"]; sa=stage[stage["指标"]=="人均充值金额"]
        if len(sr)>0:
            u=int(sn["账变事件用户数"].values[0]) if len(sn)>0 else 0
            S.append(P(f"首充活动用户总数：{u:,}人",8.5,False,C_DARK))
            dc=["当日","1日","2日","3日","4日","5日","6日","7日"]; srows=[]
            if len(sr)>0: srows.append(["留存率"]+[str(sr.iloc[0].get(c,"-")) for c in dc])
            if len(sn)>0: srows.append(["留存人数"]+[str(sn.iloc[0].get(c,"-")) for c in dc])
            if len(sa)>0: srows.append(["人均充值($)"]+[str(sa.iloc[0].get(c,"-")) for c in dc])
            if srows:
                tr2=[[cell(row[0],True,C_DARK)]+[cell(str(v),False,C_DARK,TA_CENTER) for v in row[1:]] for row in srows]
                S.append(dtable(["指标"]+dc,tr2,[full*.12]+[full*.11]*8,fsize=6.8))
        S.append(Spacer(1,6))
    S.append(P("▸ 本周 vs 上周 首充活动用户留存趋势",9,True,C_DARK)); S.append(Spacer(1,3))
    S.append(chart_first_dep_ret(fdr)); S.append(Spacer(1,4))
    tw_u=fdr.get("tw_users",0); lw_u=fdr.get("lw_users",0)
    tw_pf=fdr.get("tw_platform_fc",0); lw_pf=fdr.get("lw_platform_fc",0)
    tw_r=tw_u/tw_pf*100 if tw_pf>0 else 0; lw_r=lw_u/lw_pf*100 if lw_pf>0 else 0
    tv=[fdr.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    lv=[fdr.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    def _fr(v): return f"{v:.1f}%" if not (v is None or (isinstance(v,float) and np.isnan(v))) else "-"
    rh=["周次","活动用户数","平台首充\n人数","活动用户\n占比","D1留存","D2留存","D3留存","D4留存","D5留存","D6留存","D7留存"]
    rrows=[
        [cell(f"本周 {THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}",True,C_BLUE),
         cell(f"{tw_u:,}",False,C_DARK,TA_RIGHT),cell(f"{tw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{tw_r:.1f}%",False,C_PURPLE,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GREEN if not pd.isna(v) and not pd.isna(lval) and v>=lval else(C_RED if not pd.isna(v) and not pd.isna(lval) and v<lval else C_DARK),TA_RIGHT) for v,lval in zip(tv,lv)],
        [cell(f"上周 {LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}",True,C_GRAY),
         cell(f"{lw_u:,}",False,C_GRAY,TA_RIGHT),cell(f"{lw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{lw_r:.1f}%",False,C_GRAY,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GRAY,TA_RIGHT) for v in lv],
    ]
    S.append(dtable(rh,rrows,[full*.16,full*.09,full*.09,full*.08]+[full*.084]*7,fsize=7,zebra=False)); S.append(Spacer(1,4))
    tw_d1=fdr.get("tw_1日",np.nan); lw_d1=fdr.get("lw_1日",np.nan); ins=[]
    if not np.isnan(tw_d1) and not np.isnan(lw_d1):
        d=tw_d1-lw_d1; ins.append(f"首充活动用户次日留存{tw_d1:.1f}%（上周{lw_d1:.1f}%，{d:+.1f}pp），{'留存改善' if d>0 else '留存下降，建议优化次日触达策略'}。")
    ins.append(f"本周活动用户{tw_u:,}人，占平台首充{tw_r:.1f}%（上周{lw_u:,}/{lw_r:.1f}%）；人均赠送$3.89，属高价值用户来源。")
    ins.append("建议：对D1/D2留存用户设置阶梯式再充值道具激励；对D3后流失用户做专项召回（24h内触达效果最佳）。")
    S.append(insight_box(ins, clr=C_AMBER))
    return S


def build_risk(hr, c207_dt, c207, sp, top20g, rg, top500, dt_full):
    full = PW - 2*MARGIN; S = [sec_title("七、用户游戏风险专项分析", clr=colors.HexColor("#7c2d12"))]
    tot_tx=top500["提款金额"].sum(); tot_win=top500["公司输赢"].sum(); wns=(top500["公司输赢"]<0).sum()
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${tot_tx/10000:.1f}万",""),
                           ("平台净赔付",f"${abs(tot_win)/10000:.1f}万","平台向该群体净赔"),
                           ("赢家比例",f"{wns/500*100:.1f}%",f"{wns}赢/{500-wns}输"),
                           ("高风险用户",f"{len(hr)}人","公司净输>$10,000")]:
        fw=full/4-4
        row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],
                           colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    hl=abs(hr["公司输赢"].sum())/10000 if len(hr)>0 else 0
    S.append(insight_box([
        f"Top500提款用户中赢家{wns}人（{wns/500*100:.1f}%），平台净赔付${abs(tot_win)/10000:.1f}万。",
        f"{len(hr)}名超级赢家（公司净输>$10,000）合计导致平台净输${hl:.1f}万，占整体净赔付{hl/abs(tot_win)*10000*100 if tot_win!=0 else 0:.1f}%。",
    ]))
    if len(hr)>0:
        S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
        h=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注均额","主玩厂商","主玩游戏","风险标签"]
        rows=[]
        for _,r in hr.iterrows():
            ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0
            avg=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
            tags=[]
            if r["充值金额"]<5000: tags.append("低充高提")
            if ratio>50: tags.append("超高投充")
            if avg>200: tags.append("高额单注")
            rows.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),
                         cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),cell(f"${avg:.0f}",False,C_RED if avg>200 else C_DARK,TA_RIGHT),
                         cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),
                         cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
        S.append(dtable(h,rows,[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    if len(c207)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警"))
        S.append(P(f"集群账户数：{len(c207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
        ch=["账户ID","提款金额","充值金额","公司输赢","特征"]; cr=[]
        for _,r in c207_dt.head(10).iterrows():
            cr.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell("充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}")])
        if len(c207_dt)>10: cr.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
        S.append(dtable(ch,cr,[full*x for x in [0.22,0.18,0.18,0.18,0.24]])); S.append(Spacer(1,6))
        S.append(insight_box([
            f"{len(c207)}个账户充值$547-548，均获活动奖励，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。",
            "建议：①冻结账户提款；②审查注册IP/设备指纹；③排查奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。",
        ], clr=C_AMBER))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    th=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; tr=[]
    for i,(_,r) in enumerate(dt_full.head(20).iterrows()):
        win=r["公司输赢"]<0
        tr.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),
                   cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),
                   cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),
                   cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(th,tr,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
    pg,ma=load_pref()
    S.append(sub_title("Top500提款用户游戏偏好")); S.append(chart_pref(pg,ma)); S.append(Spacer(1,4))
    if len(rg)>0:
        S.append(sub_title("▶ 高危游戏专项分析")); S.append(chart_risk_games(rg)); S.append(Spacer(1,4))
        gh=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; gr=[]
        for _,r in rg.head(12).iterrows():
            pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rc2=C_RED if rl in("极高","高") else C_AMBER
            gr.append([cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),
                       cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),
                       cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),
                       cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),
                       cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rc2)])
        S.append(dtable(gh,gr,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5)); S.append(Spacer(1,6))
        top_rg = rg.iloc[0] if len(rg)>0 else None
        ins1 = f"高危游戏{top_rg['游戏名称']}（{top_rg['show_name_厂商标签id']}）公司输赢${top_rg['公司输赢']:,.0f}，盈亏率{top_rg['盈亏率']:.1f}%，建议复审RTP参数。" if top_rg is not None else "本周未检测到极高风险游戏。"
        S.append(insight_box([ins1,"建议对高赢家率游戏实施单账户赢额上限，防范套利风险。"], clr=C_RED))
    return S


def build_conclusion(K, dep_t, wdr_t, mfr, act, rg, hr, c207):
    full = PW - 2*MARGIN; S = [sec_title("八、总结与行动建议")]
    cr_d = K["tw_充提差比"] - K["lw_充提差比"]
    rd   = K["tw_首充次日复充率"] - K["lw_首充次日复充率"]

    # ── 亮点（自动判断正负） ──
    S.append(sub_title("▶ 本周亮点")); bg_g=colors.HexColor("#f0fdf4")
    highlights = []
    highlights.append(f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），"
                      f"充提差{K['tw_充提差']/10000:.1f}万，"
                      f"充提差率{K['tw_充提差比']:.2f}%（{cr_d:+.2f}pp）。")
    if K["pct_首充人数"] > 0:
        highlights.append(f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），"
                          f"首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。")
    if K["pct_公司输赢"] > 0:
        highlights.append(f"公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%），"
                          f"盈亏率{K['tw_盈亏率']:.3f}%，较上周改善。")
    top1_mfr = mfr.iloc[0]
    highlights.append(f"{top1_mfr.get('厂商显示名',top1_mfr['show_name_厂商标签id'])}投注份额"
                      f"{top1_mfr['占比']:.1f}%（{top1_mfr.get('投注环比',0):+.1f}%），Fortune系稳固主导。")
    for h in highlights:
        S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_g),
                                        ("LEFTPADDING",(0,0),(-1,-1),12),
                                        ("TOPPADDING",(0,0),(-1,-1),3),
                                        ("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))

    # ── 风险预警（自动判断） ──
    S.append(sub_title("▶ 风险预警")); bg_r=colors.HexColor("#fff5f5")
    risks = []
    if len(c207)>0:
        risks.append(f"PP Auto-Roulette 1批量套利：{len(c207)}个账户盈亏率极低，需立即处置。")
    if K["pct_首充Arppu"] < -5 or rd < -1:
        risks.append(f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），"
                     f"首充次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp），新用户质量需关注。")
    if len(hr)>0:
        hl=abs(hr["公司输赢"].sum())/10000
        risks.append(f"{len(hr)}名高风险用户（公司净输>$10,000）合计净输${hl:.1f}万，需立即人工审核。")
    top_rg_r = rg.iloc[0] if len(rg)>0 else None
    if top_rg_r is not None and top_rg_r["公司输赢"] < -20000:
        risks.append(f"高危游戏「{top_rg_r['游戏名称'][:16]}」公司输赢${top_rg_r['公司输赢']:,.0f}，"
                     f"赢家率{top_rg_r['赢家率']:.0f}%，建议复核RTP参数。")
    if wdr_t[3]["大盘影响"] < -1:
        risks.append(f"头部提款用户对充提差率拖累{wdr_t[3]['大盘影响']:+.2f}pp（剔除Top100后）。")
    if not risks:
        risks.append("本周暂无重大风险预警，各指标维持正常区间。")
    for r in risks:
        S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_r),
                                        ("LEFTPADDING",(0,0),(-1,-1),12),
                                        ("TOPPADDING",(0,0),(-1,-1),3),
                                        ("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))

    # ── 行动建议（自动生成） ──
    S.append(sub_title("▶ 行动建议"))
    actions = []
    if len(c207)>0:
        actions.append(("[紧急]","处置批量套利账号",
                        f"共{len(c207)}个账户，建议冻结提款、审查IP/设备指纹、修复奖励触发漏洞并实施赢额上限。"))
    if len(hr)>0:
        top_hr = hr.sort_values("公司输赢").iloc[0]
        actions.append(("[紧急]","处置高风险提款用户",
                        f"账户{top_hr['账户ID']}：提款${top_hr['提款金额']:,.0f}，"
                        f"公司输赢${top_hr['公司输赢']:,.0f}，高度异常，建议立即人工审核。"))
    if K["pct_首充Arppu"] < -5 or rd < -1:
        actions.append(("[本周]","优化首充质量与次日激活",
                        f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），"
                        f"次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp）。"
                        f"建议注册后1h/24h内推送首充引导，落地页突出$10+档。"))
    if top_rg_r is not None and top_rg_r["公司输赢"] < -20000:
        actions.append(("[本周]","高危游戏风险管控",
                        f"对赢家率>60%的高危游戏实施单账户赢额上限，"
                        f"同步复核「{top_rg_r['游戏名称'][:16]}」RTP参数，防范系统性套利。"))
    if not actions:
        actions.append(("[常规]","持续监控核心指标",
                        "充值、留存、充提差率等核心指标维持正常，建议持续监控并保持现有运营节奏。"))
    pmap={"[紧急]":C_RED,"[本周]":C_AMBER,"[常规]":C_GREEN}
    rows=[[cell(pri,True,pmap.get(pri[:4],C_GRAY),TA_CENTER),
           cell(title,True,C_DARK),cell(desc,False,C_GRAY)]
          for pri,title,desc in actions]
    S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows,[full*x for x in [0.1,0.22,0.68]]))
    return S


def header_footer(c, doc):
    c.saveState(); w, h = A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN, h-17, f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN, h-17, f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN, 5, f"第 {doc.page} 页")
    c.restoreState()


def main():
    root   = DATA_ROOT
    outdir = OUTPUT_DIR
    outdir.mkdir(parents=True, exist_ok=True)
    print("📊 MX 周报 PDF v7.5 生成中...")
    resolve_files(root); setup()
    print("  ▶ 加载数据...")
    K, trend         = load_platform()
    dash_ret         = load_dashboard_retention()
    weekly_ret       = load_weekly_retention()
    agents           = load_agents()
    agent_ret, ends  = load_agent_ret()
    tw_v, lw_v       = load_vip()
    dep_t, wdr_t, dc_tw, dt_top = load_top_users()
    top30            = load_games()
    mfr              = load_mfr()
    mfr_delta        = load_mfr_game_delta()
    act_df, tot_gift = load_activities()
    tool_df          = load_tool_data()
    fdr              = load_first_dep_ret()

    dt_raw = pd.read_csv(FILES["dt_tw"])
    dc_raw = pd.read_csv(FILES["dc_tw"])
    for df in [dt_raw, dc_raw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")

    hr, c207_dt, c207, sp, top20g, rg, top500 = load_risk(dt_raw, dc_raw)

    # 关联主玩游戏到top20提款
    dpf = pd.read_csv(FILES["pref_tw"]); dpf["阶段汇总"] = dpf["阶段汇总"].apply(clean)
    tg = (dpf[dpf["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
          .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]]
          .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
    dt_top = dt_top.merge(tg, on="账户ID", how="left")

    print("  ▶ 导出风控Excel...")
    excel_out = outdir / f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out), engine="openpyxl") as xw:
        cols_hr = ["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
        hr[[c for c in cols_hr if c in hr.columns]].to_excel(xw, sheet_name="高风险用户", index=False)
        if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(xw, sheet_name="PP轮盘批量账号", index=False)
        if len(sp)>0:       sp[["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]].to_excel(xw, sheet_name="特殊异常用户", index=False)
        tool_df.to_excel(xw, sheet_name="道具使用情况", index=False)
    print(f"✅ 风控Excel: {excel_out}")

    print("  ▶ 构建PDF章节...")
    out = outdir / f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc = SimpleDocTemplate(str(out), pagesize=A4, leftMargin=MARGIN, rightMargin=MARGIN,
                            topMargin=MARGIN+22, bottomMargin=MARGIN+10,
                            title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story = [Spacer(1,3*cm),
             P("MX 平台数据周报", 30, True, C_DARK, TA_CENTER), Spacer(1,.5*cm),
             P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),
             Spacer(1,.2*cm),
             P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),
             Spacer(1,.5*cm), HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"), PageBreak()]

    story += build_overview(K, trend, weekly_ret, dash_ret);              story.append(PageBreak())
    story += build_agents(agents, agent_ret, ends);                        story.append(PageBreak())
    story += build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_top);        story.append(PageBreak())
    story += build_games(mfr, top30, mfr_delta);                          story.append(PageBreak())
    story += build_activities(act_df, tot_gift);                          story.append(PageBreak())
    story += build_tools(tool_df, fdr);                                   story.append(PageBreak())
    story += build_risk(hr, c207_dt, c207, sp, top20g, rg, top500, dt_top); story.append(PageBreak())
    story += build_conclusion(K, dep_t, wdr_t, mfr, act_df, rg, hr, c207)

    print("  ▶ 渲染PDF...")
    doc.build(story, onFirstPage=header_footer, onLaterPages=header_footer)
    size = out.stat().st_size / 1024
    print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out), str(excel_out)


if __name__ == "__main__":
    main()

📊 MX 周报 PDF v7.3 生成中...
  ▶ 数据目录：D:\周报更新版\MX
     ✅ [platform       ] 平台报表_USD_20260530111756.xlsx
     ✅ [daily          ] 日报-大盘日报_USD_20260530113940.xlsx
     ✅ [retention      ] 整体 首充留存（近7天）_20260502-20260529.csv
     ✅ [agent_plat     ] 平台报表-总代_USD_20260530113324.xlsx
     ✅ [agent_promo    ] 推广报表-总代_USD_20260530113133.xlsx
     ✅ [agent_ret      ] 首充充值留存_全量数据_20260502_20260529.csv
     ✅ [vip            ] VIP报表_USD_20260530114435.xlsx
     ✅ [dt_tw          ] top提款用户_全量数据_20260523_20260529_日期对比20260516_20260522.csv
     ✅ [dt_lw          ] top提款用户_全量数据_20260516_20260522_日期对比20260509_20260515 (1).csv
     ✅ [dc_tw          ] 头部充值用户_全量数据_20260523_20260529_日期对比20260516_20260522.csv
     ✅ [dc_lw          ] 头部充值用户_全量数据_20260516_20260522_日期对比20260509_20260515.csv
     ✅ [pref_tw        ] 本周top500提款用户游戏偏好_全量数据_20260523_20260529.csv
     ✅ [pref_lw        ] 上周top500提款用户游戏偏好_全量数据_20260516_20260522.csv
     ✅ [mfr            ] 厂商投注数据_全量数据_20260516_20260529.csv
     ✅ [game_tw        ] 游戏报表

In [2]:
# """
# MX 平台数据周报 v7.3
# 变更：
# ① 总代表 3留(本/上) 合并一列，同 次日留存(本/上) 格式
# ② 总代留存数据源：首充充值留存_** 中 2日(次留)、3日(3留)、6日(7留)
#    （该表当日=第0日，故 2日=lag1次留，3日=lag2三留，6日=lag6七留）
# ③ 大盘留存：日报-大盘日报_** 首充2/3/7日复充率作为次留/3留/7留
# ④ 留存目标对比：次留21% 3留15% 7留11% 14留8% 30留6%，写入大盘章节表格
# ⑤ 总代表列宽/字号优化，确保所有内容单行显示
# """
# from pathlib import Path
# import sys, io, warnings
# from datetime import datetime, timedelta
# import numpy as np, pandas as pd
# import matplotlib; matplotlib.use("Agg")
# import matplotlib.pyplot as plt, matplotlib.ticker as mticker
# import matplotlib.gridspec as gridspec, matplotlib.patches as mpatches
# warnings.filterwarnings("ignore")

# from reportlab.lib.pagesizes import A4
# from reportlab.lib import colors
# from reportlab.lib.units import cm
# from reportlab.lib.styles import ParagraphStyle
# from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
# from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
#     TableStyle, Image, PageBreak, HRFlowable, KeepTogether)
# from reportlab.pdfbase import pdfmetrics
# from reportlab.pdfbase.ttfonts import TTFont

# # ── 路径配置 ─────────────────────────────────────────────
# if sys.platform != "win32":
#     DATA_ROOT  = Path("/mnt/user-data/uploads")
#     OUTPUT_DIR = Path("/mnt/user-data/outputs")
# else:
#     DATA_ROOT  = Path(r"D:\周报更新版\MX")
#     OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")

# THIS_WEEK  = ("20260515", "20260521")
# LAST_WEEK  = ("20260508", "20260514")
# REPORT_END = THIS_WEEK[1]

# # ── 留存目标 ─────────────────────────────────────────────
# RET_TARGETS = {"次留": 21.0, "3留": 15.0, "7留": 11.0, "14留": 8.0, "30留": 6.0}

# # ── 全局字体/颜色 ─────────────────────────────────────────
# FN, FNB = "WQY", "WQYB"
# FILES: dict = {}

# C_BLUE   = colors.HexColor("#1d4ed8");  C_BLUE2  = colors.HexColor("#3b82f6")
# C_GREEN  = colors.HexColor("#059669");  C_RED    = colors.HexColor("#dc2626")
# C_AMBER  = colors.HexColor("#d97706");  C_PURPLE = colors.HexColor("#7c3aed")
# C_GRAY   = colors.HexColor("#64748b");  C_LGRAY  = colors.HexColor("#f1f5f9")
# C_WHITE  = colors.white;                C_DARK   = colors.HexColor("#1e293b")
# C_BORDER = colors.HexColor("#cbd5e1");  C_ROW    = colors.HexColor("#f8fafc")
# C_TEAL   = colors.HexColor("#0f766e");  C_TARGET = colors.HexColor("#0369a1")

# PW, PH = A4
# MARGIN  = 1.6 * cm
# CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
#                 "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
# CR_HIGH, CR_LOW = 17, 5
# MFR_SHORT = {"Rectangle":"RG","Pragmatic Play":"PP","PG Soft":"PG","PlayTech":"PT","Originals":"自研"}

# TRUNCATE = {"首充次日复充率":1,"首充次日复投率":1,"首充当日复充率":0,
#             "首充7日复充率":6,"首充30日复充率":999,"首充2日复充率":1,"首充3日复充率":2}

# # ── 工具函数 ──────────────────────────────────────────────
# def shorten_mfr(n):
#     if not isinstance(n, str): return str(n)
#     for k, v in MFR_SHORT.items():
#         if k in n: return v
#     return n

# def ret_end(week_start: str, lag: int) -> str | None:
#     e = (datetime.strptime(REPORT_END, "%Y%m%d") - timedelta(days=lag)).strftime("%Y%m%d")
#     return e if e >= week_start else None

# def fmt_lbl(d: str | None) -> str:
#     return f"{d[4:6]}/{d[6:]}" if d else "-"

# def clean(s):
#     try: return float(str(s).replace(",","").replace("%","").strip())
#     except: return np.nan

# def to_num(df, cols=None):
#     if cols is None:
#         cols = [c for c in df.columns if c not in ("日期","总代.名称","name_总代","总代.ID")]
#     for c in cols:
#         if c in df.columns: df[c] = df[c].apply(clean)
#     return df

# def pct(n, o):   return (n-o)/abs(o)*100 if o and o != 0 else 0.0
# def pct_vec(ns, os):
#     return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
# def trunc_mean(s, drop=0):
#     if drop >= len(s): return np.nan
#     v = s.iloc[:len(s)-drop] if drop else s
#     nz = v[v > 0]; return nz.mean() if len(nz) else v.mean()

# def wavg_series(vals, weights):
#     ok = vals.notna() & (weights > 0)
#     if not ok.any(): return np.nan
#     return float(np.average(vals[ok], weights=weights[ok]))

# def fig_img(fig, w=16, h=7):
#     buf = io.BytesIO()
#     fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
#     buf.seek(0); plt.close(fig)
#     return Image(buf, width=w*cm, height=h*cm)

# def find_chinese_font():
#     import platform
#     if platform.system() == "Windows":
#         cands = [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\msyhbd.ttc",
#                  r"C:\Windows\Fonts\simsun.ttc", r"C:\Windows\Fonts\simhei.ttf"]
#     elif platform.system() == "Darwin":
#         cands = ["/System/Library/Fonts/PingFang.ttc"]
#     else:
#         cands = ["/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
#                  "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
#                  "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]
#     for p in cands:
#         if Path(p).exists(): return p
#     from matplotlib import font_manager as fm
#     for f in fm.fontManager.ttflist:
#         if any(k in f.name for k in ["Hei","YaHei","SimSun","WenQuanYi","Noto Sans CJK"]):
#             if Path(f.fname).exists(): return f.fname
#     raise FileNotFoundError("未找到中文字体")

# def setup():
#     global FONT_PATH
#     FONT_PATH = find_chinese_font()
#     print(f"  ▶ 字体：{FONT_PATH}")
#     ttc = FONT_PATH.lower().endswith(".ttc")
#     pdfmetrics.registerFont(TTFont(FN,  FONT_PATH, subfontIndex=0) if ttc else TTFont(FN,  FONT_PATH))
#     try:    pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1) if ttc else TTFont(FNB, FONT_PATH))
#     except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0) if ttc else TTFont(FNB, FONT_PATH))
#     from matplotlib import font_manager as fm
#     fm.fontManager.addfont(FONT_PATH)
#     added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
#     plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
#                          "axes.unicode_minus": False, "font.size": 8.5})

# def resolve_files(root):
#     global FILES
#     km = {
#         "platform":      "平台报表_USD_近4周",
#         "daily":         "日报-大盘日报_USD_近4周",
#         "retention":     ["整体","首充留存"],
#         "agent_plat":    "平台报表-总代_USD_近21天",
#         "agent_promo":   "推广报表-总代_USD_近21天",
#         "agent_ret":     "首充充值留存",
#         "vip":           "VIP报表_USD",
#         "dt_tw":         f"top提款用户_全量数据_{THIS_WEEK[0]}",
#         "dt_lw":         f"top提款用户_全量数据_{LAST_WEEK[0]}",
#         "dc_tw":         f"头部充值用户_全量数据_{THIS_WEEK[0]}",
#         "dc_lw":         f"头部充值用户_全量数据_{LAST_WEEK[0]}",
#         "pref_tw":       f"本周top500提款用户游戏偏好_全量数据_{THIS_WEEK[0]}",
#         "pref_lw":       f"上周top500提款用户游戏偏好_全量数据_{LAST_WEEK[0]}",
#         "mfr":           "厂商投注数据_全量数据",
#         "game_tw":       "游戏报表-详情_USD_本周",
#         "game_lw":       "游戏报表-详情_USD_上周",
#         "gift":          f"各赠送活动_全量数据_{THIS_WEEK[0]}",
#         "tool_map":      "道具对应活动",
#         "vip_ret_chg":   "VIP充值-充值_近28天",
#         "vip_ret_act":   "VIP充值-活跃_近28天",
#         "tool_tw":       "本周道具使用情况",
#         "tool_lw":       "上周道具使用情况",
#         "first_dep_ret": "首次充值活动用户充值留存情况",
#     }
#     print(f"  ▶ 数据目录：{root}"); fail = 0
#     for key, kw in km.items():
#         kws = kw if isinstance(kw, list) else [kw]
#         hits = [h for h in root.rglob("*")
#                 if h.is_file() and not h.name.startswith("~$") and all(k in h.name for k in kws)]
#         if not hits: print(f"     ❌ [{key:15s}] 找不到含{kws}"); fail += 1
#         else:
#             p = max(hits, key=lambda h: h.stat().st_mtime); FILES[key] = p
#             print(f"     ✅ [{key:15s}] {p.name}")
#     if fail: raise FileNotFoundError(f"共{fail}个文件未找到")
#     print(f"  ▶ 全部{len(km)}个文件匹配成功\n")

# # ── PDF 样式工具 ─────────────────────────────────────────
# def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
#     st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
#                         textColor=clr, alignment=align, leading=sz*1.4,
#                         spaceAfter=0, spaceBefore=0)
#     return Paragraph(str(txt), st)

# def sec_title(text, clr=C_BLUE):
#     return KeepTogether([Spacer(1,6),
#         Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
#               style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
#                                 ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
#                                 ("LEFTPADDING",(0,0),(-1,-1),14)])),
#         Spacer(1,8)])

# def sub_title(text):
#     return KeepTogether([Spacer(1,5), HRFlowable(width="100%",thickness=1.5,color=C_BLUE2),
#                          Spacer(1,3), P(f"■  {text}",10,True,C_DARK), Spacer(1,6)])

# def insight_box(lines, clr=C_GREEN):
#     bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
#            colors.HexColor("#fffbeb") if clr==C_AMBER else
#            colors.HexColor("#fef2f2"))
#     rows = [[P(f"◆ {l}",8.5,False,C_DARK)] for l in lines]
#     return KeepTogether([
#         Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
#             ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
#             ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
#             ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
#             ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
#         ])), Spacer(1,8)])

# def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8, extra_style=None):
#     full = PW - 2*MARGIN
#     if not widths: widths = [full/len(headers)]*len(headers)
#     if hdr_clr is None: hdr_clr = C_DARK
#     st = TableStyle([
#         ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
#         ("FONTNAME",(0,0),(-1,0),FNB),       ("FONTNAME",(0,1),(-1,-1),FN),
#         ("FONTSIZE",(0,0),(-1,-1),fsize),
#         ("TOPPADDING",(0,0),(-1,-1),2),      ("BOTTOMPADDING",(0,0),(-1,-1),2),
#         ("LEFTPADDING",(0,0),(-1,-1),2),     ("RIGHTPADDING",(0,0),(-1,-1),2),
#         ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
#     ])
#     if zebra:
#         for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
#     if extra_style:
#         for cmd in extra_style: st.add(*cmd)
#     hrow = [P(h,fsize,True,C_WHITE,TA_CENTER) for h in headers]
#     body = []
#     for row in rows:
#         body.append([
#             P(str(c[0]),fsize, c[1] if len(c)>1 else False,
#               c[2] if len(c)>2 else colors.black,
#               c[3] if len(c)>3 else TA_LEFT)
#             if isinstance(c,(list,tuple)) else P(str(c),fsize)
#             for c in row
#         ])
#     return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

# def cell(t,bold=False,clr=colors.black,align=TA_LEFT): return (t,bold,clr,align)
# def rc(v, gup=True, d=1, good_up=None):
#     if good_up is not None: gup=good_up
#     c = C_GREEN if (v>0)==gup else C_RED
#     return cell(f"{'+' if v>=0 else ''}{v:.{d}f}%",False,c,TA_RIGHT)
# def gclr(v,t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)
# def fret(v): return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

# def kpi_card4(items, cols=4):
#     fw = (PW-2*MARGIN)/cols - 4
#     rows, row = [], []
#     for label,tv,lv,chg,_ in items:
#         pclr = C_GREEN if chg>=0 else C_RED
#         inner = Table([[P(label,7.5,False,C_GRAY)],[P(str(tv),14,True,C_DARK)],
#                        [P(f"上周：{lv}",7.5,False,C_GRAY)],
#                        [P(f"{'+' if chg>=0 else ''}{chg:.1f}%",8,True,pclr)]],
#                       colWidths=[fw],
#                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
#                                         ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
#                                         ("LEFTPADDING",(0,0),(-1,-1),8),
#                                         ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
#         row.append(inner)
#         if len(row)==cols: rows.append(row); row=[]
#     if row:
#         while len(row)<cols: row.append(Spacer(fw,1))
#         rows.append(row)
#     t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
#     t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
#                             ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
#                             ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
#     return t
# # ════════════════════════════════════════════════════════════════
# # 数据加载
# # ════════════════════════════════════════════════════════════════

# def load_platform():
#     df = pd.read_excel(FILES["platform"]); df["日期"] = df["日期"].astype(str)
#     df = df.sort_values("日期"); df = to_num(df)
#     tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]
#     dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
#     tw_d = dd[dd["日期"].between(*THIS_WEEK)]; lw_d = dd[dd["日期"].between(*LAST_WEEK)]
#     K = {}
#     for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
#               "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
#         if c not in df.columns: continue
#         K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
#         K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
#     for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
#         if c not in df.columns: continue
#         K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
#         K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
#     for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
#               "首充次日复投率","活跃用户付费率","总赠送充值比"]:
#         if c not in df.columns: continue
#         drop = TRUNCATE.get(c, 0)
#         K[f"tw_{c}"] = trunc_mean(tw[c], drop); K[f"lw_{c}"] = lw[c].mean()
#         K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
#     K["tw_真实消耗"] = tw_d["真实消耗"].iloc[:-1].sum() if len(tw_d)>1 else tw_d["真实消耗"].sum()
#     K["lw_真实消耗"] = lw_d["真实消耗"].sum()
#     K["pct_真实消耗"] = pct(K["tw_真实消耗"], K["lw_真实消耗"])
#     K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
#     K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
#     K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])
#     last14 = df.tail(14)
#     trend = {"dates": [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
#              "充值": last14["充值金额"].tolist(), "提现": last14["提现金额"].tolist(),
#              "充提差比": last14["充提差比"].tolist(), "公司输赢": last14["公司输赢"].tolist(),
#              "首充": last14["首充人数"].tolist(), "注册": last14["注册人数"].tolist()}
#     return K, trend


# def load_dashboard_retention():
#     """
#     大盘留存 — 来源：日报-大盘日报_**
#       首充2日复充率 → 次留(lag=1)
#       首充3日复充率 → 3留(lag=2)
#       首充7日复充率 → 7留(lag=6)
#       首充30日复充率→ 30留(lag=29)
#     截止日期由 ret_end(week_start, lag) 决定，剔除不完整天数
#     """
#     dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str)
#     specs = [("nd","首充2日复充率",1),("td","首充3日复充率",2),
#              ("sd","首充7日复充率",6),("30d","首充30日复充率",29)]
#     for _,col,_ in specs:
#         if col in dd.columns:
#             dd[col] = pd.to_numeric(dd[col].astype(str).str.replace("%","").str.strip(), errors="coerce")
#     R = {}
#     for key, col, lag in specs:
#         for ws, we, prefix in [(THIS_WEEK[0],THIS_WEEK[1],"tw_"), (LAST_WEEK[0],LAST_WEEK[1],"lw_")]:
#             ec = ret_end(ws, lag)
#             sub = dd[(dd["日期"]>=ws) & (dd["日期"]<=we)]
#             if ec: sub = sub[sub["日期"]<=ec]
#             R[f"{prefix}{key}"] = float(sub[col].mean()) if len(sub) and col in sub else np.nan
#         R[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
#         R[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)
#     return R


# def load_weekly_retention():
#     """整体首充留存（近4周），列：第1日=次留, 第2日=3留, 第6日=7留"""
#     df = pd.read_csv(FILES["retention"])
#     daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
#     daily["ds"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
#     dr = daily[daily["指标"]=="留存率"].copy()
#     du = daily[daily["指标"]=="留存人数"].copy()
#     for c in ["第1日","第2日","第3日","第6日","第7日"]:
#         if c in dr.columns:
#             dr[c] = pd.to_numeric(dr[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
#     def _d(s): return datetime.strptime(s,"%Y%m%d")
#     def _f(d): return d.strftime("%Y-%m-%d")
#     def _l(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
#     tw_s,tw_e = _d(THIS_WEEK[0]),_d(THIS_WEEK[1])
#     lw_s,lw_e = _d(LAST_WEEK[0]),_d(LAST_WEEK[1])
#     w2e = lw_s-timedelta(days=1); w2s = w2e-timedelta(days=6)
#     w1e = w2s-timedelta(days=1); w1s = w1e-timedelta(days=6)
#     weeks = [(f"第1周\n{_l(w1s,w1e)}",_f(w1s),_f(w1e)),
#              (f"第2周\n{_l(w2s,w2e)}",_f(w2s),_f(w2e)),
#              (f"上周\n{_l(lw_s,lw_e)}",_f(lw_s),_f(lw_e)),
#              (f"本周\n{_l(tw_s,tw_e)}",_f(tw_s),_f(tw_e))]
#     result = []
#     for wk,s,e in weeks:
#         mr = dr[(dr["ds"]>=s)&(dr["ds"]<=e)]; mu = du[(du["ds"]>=s)&(du["ds"]<=e)]
#         row = {"week": wk, "users": mu["充值成功事件用户数"].sum()}
#         for col in ["第1日","第2日","第3日","第6日","第7日"]:
#             rs = mr[col].values if col in mr.columns else np.array([])
#             us = mu["充值成功事件用户数"].values
#             if len(rs)>0 and len(rs)==len(us):
#                 v = ~np.isnan(rs)
#                 row[col] = float(np.average(rs[v],weights=us[v])) if v.sum()>0 else np.nan
#             else: row[col] = float(np.nanmean(rs)) if len(rs)>0 else np.nan
#         result.append(row)
#     return result


# def load_agents():
#     dp = pd.read_excel(FILES["agent_plat"]); dr = pd.read_excel(FILES["agent_promo"])
#     dp["日期"]=dp["日期"].astype(str); dr["日期"]=dr["日期"].astype(str)
#     dp=to_num(dp); dr=to_num(dr)
#     tw_p=dp[dp["日期"].between(*THIS_WEEK)]; lw_p=dp[dp["日期"].between(*LAST_WEEK)]
#     tw_r=dr[dr["日期"].between(*THIS_WEEK)]; lw_r=dr[dr["日期"].between(*LAST_WEEK)]
#     sa=["充值金额","提现金额","充提差","首充金额","首充人数","注册人数","充值人数","投注金额","公司输赢","总赠送金额"]
#     def agg_p(d):
#         g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sa if c in d.columns}).reset_index()
#         for c in ["充提差比","首充次日充值留存"]:
#             if c in d.columns: g=g.merge(d.groupby("总代.ID")[c].mean().rename(c),on="总代.ID",how="left")
#         g["充提差率"]=g["充提差"]/g["充值金额"]*100; return g
#     tw_pa=agg_p(tw_p); lw_pa=agg_p(lw_p)
#     sr=["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
#     def agg_r(d):
#         g=d.groupby("总代.ID").agg({c:"sum" for c in sr if c in d.columns}).reset_index()
#         g["一级首充成本"]=g["总消耗"]/g["一级首充人数"].replace(0,np.nan); return g
#     tw_ra=agg_r(tw_r); lw_ra=agg_r(lw_r)
#     lw_ra["lw_fc_cost"]=lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
#     m=tw_pa.merge(lw_pa[["总代.ID","充值金额","注册人数","充提差率"]].rename(
#         columns={"充值金额":"lw_充值","注册人数":"lw_注册","充提差率":"lw_充提差率"}),on="总代.ID",how="left")
#     m=m.merge(tw_ra[["总代.ID","总消耗","一级首充成本","一级首充人数"]],on="总代.ID",how="left")
#     m=m.merge(lw_ra[["总代.ID","lw_fc_cost"]],on="总代.ID",how="left")
#     m["注册环比"]=pct_vec(m["注册人数"],m["lw_注册"])
#     return m.sort_values("充值金额",ascending=False)


# def load_agent_ret():
#     """
#     总代留存 — 来源：首充充值留存_**
#     该表：当日=第0日, 1日=第1日, 2日=第2日(=次留lag1), 3日=第3日(=3留lag2), 6日=第6日(=7留lag6)
#     返回：(ret_dict, ends_dict)
#     """
#     df = pd.read_csv(FILES["agent_ret"])
#     df["_d"] = df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0].str.replace("-","")
#     for c in ["2日","3日","6日"]:
#         df[c] = pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
#     df["首充用户数"] = pd.to_numeric(df["首充用户数"], errors="coerce").fillna(0)
#     df["总代"] = pd.to_numeric(df["总代"], errors="coerce")
#     daily = df[df["_d"].notna()&df["_d"].str.match(r"^\d{8}$")&(df["指标"]=="留存率")].copy()

#     ends = {}
#     for key, lag in [("nd",1),("3d",2),("7d",6)]:
#         ends[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
#         ends[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)

#     def _slice(ws, we, ec):
#         sub = daily[daily["_d"].between(ws,we)]
#         if ec: sub = sub[sub["_d"] <= min(ec,we)]
#         return sub

#     slices = {
#         "tw_nd": _slice(THIS_WEEK[0], THIS_WEEK[1], ends["tw_nd_end"]),
#         "tw_3d": _slice(THIS_WEEK[0], THIS_WEEK[1], ends["tw_3d_end"]),
#         "tw_7d": _slice(THIS_WEEK[0], THIS_WEEK[1], ends["tw_7d_end"]),
#         "lw_nd": _slice(LAST_WEEK[0], LAST_WEEK[1], ends["lw_nd_end"]),
#         "lw_3d": _slice(LAST_WEEK[0], LAST_WEEK[1], ends["lw_3d_end"]),
#         "lw_7d": _slice(LAST_WEEK[0], LAST_WEEK[1], ends["lw_7d_end"]),
#     }
#     col_map = {"tw_nd":"2日","tw_3d":"3日","tw_7d":"6日",
#                "lw_nd":"2日","lw_3d":"3日","lw_7d":"6日"}

#     def _wavg_by_agent(df_sub, col):
#         res = {}
#         for nm, grp in df_sub.groupby("name_总代"):
#             v = wavg_series(grp[col], grp["首充用户数"])
#             if not np.isnan(v): res[nm] = v
#         return res

#     per_agent = {k: _wavg_by_agent(slices[k], col_map[k]) for k in slices}

#     # id_map
#     id_map = {}
#     for nm, grp in daily.groupby("name_总代"):
#         vals = grp["总代"].dropna().values
#         if len(vals): id_map[nm] = int(vals[0])

#     all_names = set().union(*[set(v) for v in per_agent.values()])
#     ret = {}
#     for nm in all_names:
#         ret[nm] = {k: per_agent[k].get(nm, np.nan) for k in per_agent}
#         ret[nm]["agent_id"] = id_map.get(nm)
#     return ret, ends


# def load_vip():
#     df=pd.read_excel(FILES["vip"]); df["日期"]=df["日期"].astype(str)
#     num=["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
#     df=to_num(df,num)
#     tw=df[df["日期"].between(*THIS_WEEK)]; lw=df[df["日期"].between(*LAST_WEEK)]
#     return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

# def load_vip_retention():
#     res={}
#     for fk,rt in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
#         df=pd.read_csv(FILES[fk])
#         df["ds"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
#         df["yyyymmdd"]=df["ds"].str.replace("-","")
#         for c in ["1日","2日","3日","7日"]:
#             df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
#         df["n"]=pd.to_numeric(df["充值成功事件用户数"],errors="coerce")
#         df["vip"]=pd.to_numeric(df["vip_level"],errors="coerce")
#         daily=df[(df["指标"]=="留存率")&df["yyyymmdd"].notna()&df["vip"].notna()]
#         tw=daily[daily["yyyymmdd"].between(*THIS_WEEK)]; lw=daily[daily["yyyymmdd"].between(*LAST_WEEK)]
#         def wavg(d,col):
#             r={}
#             for v,g in d.groupby("vip"):
#                 s=g[g[col].notna()]
#                 if len(s): r[int(v)]=float(np.average(s[col].values,weights=s["n"].values))
#             return r
#         res[rt]={"tw":{c:wavg(tw,c) for c in ["1日","2日","3日","7日"]},
#                  "lw":{c:wavg(lw,c) for c in ["1日","2日","3日","7日"]}}
#     return res

# def load_top_users():
#     dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
#     dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
#     for df in [dc_tw,dc_lw,dt_tw,dt_lw]:
#         for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
#             if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
#     dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
#     tw_p=dp[dp["日期"].between(*THIS_WEEK)]
#     total_c=tw_p["充值金额"].sum(); total_t=tw_p["提现金额"].sum(); actual_cr=(total_c-total_t)/total_c*100
#     tot_c=dc_tw["充值金额"].sum(); tot_t=dt_tw["提款金额"].sum()
#     dep_t,wdr_t=[],[]
#     for t in [1,10,50,100,200,500]:
#         tw=dc_tw.head(t); lw=dc_lw.head(t)
#         tc=tw["充值金额"].sum(); tt=tw["提款金额"].sum(); lc=lw["充值金额"].sum(); lt=lw["提款金额"].sum()
#         dep_t.append({"tier":f"Top{t}","tw_chg":tc,"lw_chg":lc,"tw_avg":tc/t,"lw_avg":lc/t,
#             "tw_cr":(tc-tt)/tc*100 if tc>0 else 0,"lw_cr":(lc-lt)/lc*100 if lc>0 else 0,
#             "tw_win":tw["公司输赢"].sum(),"占全量":tc/tot_c*100})
#         tw2=dt_tw.head(t); lw2=dt_lw.head(t)
#         tt2=tw2["提款金额"].sum(); tc2=tw2["充值金额"].sum(); lt2=lw2["提款金额"].sum(); lc2=lw2["充值金额"].sum()
#         wns=(tw2["公司输赢"]<0).sum()
#         act=tw2["活动奖励"].sum()/(tc2+tw2["活动奖励"].sum())*100 if (tc2+tw2["活动奖励"].sum())>0 else 0
#         excl_c=total_c-tc2; excl_t=total_t-tt2
#         excl_cr=(excl_c-excl_t)/excl_c*100 if excl_c>0 else 0
#         wdr_t.append({"tier":f"Top{t}","tw_tx":tt2,"lw_tx":lt2,"tw_avg":tt2/t,"lw_avg":lt2/t,
#             "tw_cr":(tc2-tt2)/tc2*100 if tc2>0 else 0,"lw_cr":(lc2-lt2)/lc2*100 if lc2>0 else 0,
#             "赢家":wns,"总数":t,"赢家率":wns/t*100,"活动占比":act,"占全量":tt2/tot_t*100,"大盘影响":excl_cr-actual_cr})
#     return dep_t,wdr_t,dc_tw.head(200),dt_tw.head(20)

# def load_pref():
#     df=pd.read_csv(FILES["pref_tw"]); dl=pd.read_csv(FILES["pref_lw"])
#     df["阶段汇总"]=df["阶段汇总"].apply(clean); dl["阶段汇总"]=dl["阶段汇总"].apply(clean)
#     bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]; bl=dl[dl["分析指标"]=="投注金额"]
#     gb=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
#     gw=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
#     gl=bl.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
#     gu=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
#     g=pd.concat([gb,gw,gl,gu],axis=1).reset_index()
#     g["占比"]=g["本周投注"]/g["本周投注"].sum()*100; g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
#     g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
#     mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
#     return g.sort_values("本周投注",ascending=False).head(20), mfr

# def load_games():
#     tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
#     for df in [tw,lw]:
#         for c in ["投注人数","投注局数","投注金额","公司输赢"]:
#             if c in df.columns: df[c]=df[c].apply(clean)
#     tot=tw["投注金额"].sum()
#     gt=tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
#         投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
#         投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
#     gl=lw.groupby("游戏.名称").agg(投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
#         columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
#     gt["人均局数"]=gt["投注局数"]/gt["投注人数"]; gt["人均金额"]=gt["投注金额"]/gt["投注人数"]
#     gt["盈亏率"]=gt["公司输赢"]/gt["投注金额"]*100; gt["占比"]=gt["投注金额"]/tot*100
#     gt["厂商简称"]=gt["游戏厂商标签.名称"].apply(shorten_mfr)
#     gm=gt.merge(gl,on="游戏.名称",how="left")
#     gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
#     gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
#     return gm.sort_values("投注金额",ascending=False).head(30)

# def load_mfr():
#     df=pd.read_csv(FILES["mfr"]); df=df.rename(columns={"盈利率":"盈亏率"})
#     df=df[df["时间"]!="阶段汇总"].copy(); df["时间"]=df["时间"].astype(str).str.replace("-","")
#     for c in ["投注局数","投注人数","投注金额","公司输赢","盈亏率"]:
#         if c in df.columns: df[c]=df[c].apply(clean)
#     tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
#     def agg(d):
#         g=d.groupby("show_name_厂商标签id").agg(
#             投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
#             投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
#         g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
#         g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
#         g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
#     tg=agg(tw); lg=agg(lw)
#     mg=tg.merge(lg[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
#         columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
#         on="show_name_厂商标签id",how="left")
#     mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
#     mg["厂商显示名"]=mg["show_name_厂商标签id"].apply(shorten_mfr)
#     return mg.sort_values("投注金额",ascending=False)

# def load_mfr_game_delta():
#     tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
#     for df in [tw,lw]:
#         for c in ["投注金额","公司输赢"]:
#             if c in df.columns: df[c]=df[c].apply(clean)
#     gt=tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index()
#     gl=lw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index().rename(columns={"投注金额":"lw_投注"})
#     m=gt.merge(gl,on=["游戏厂商标签.名称","游戏.名称"],how="outer").fillna(0); m["delta"]=m["投注金额"]-m["lw_投注"]
#     mfr_tw=tw.groupby("游戏厂商标签.名称")["投注金额"].sum().sort_values(ascending=False)
#     res={}
#     for n in mfr_tw.head(10).index:
#         sub=m[m["游戏厂商标签.名称"]==n].sort_values("delta",ascending=False)
#         res[n]={"up":sub.head(1),"dn":sub.tail(1)}
#     return res

# def load_activities():
#     df=pd.read_csv(FILES["gift"])
#     for c in ["赠送金额","赠送金额.1","赠送人数","赠送人数.1"]:
#         if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0)
#     ag=df.groupby(["账变opt_code","name_账变opt_id"]).agg(
#         赠送金额=("赠送金额","sum"),lw_赠=("赠送金额.1","sum"),
#         赠送人数=("赠送人数","sum"),lw_人数=("赠送人数.1","sum")).reset_index()
#     ag["环比"]=(ag["赠送金额"]-ag["lw_赠"])/ag["lw_赠"].replace(0,np.nan).abs()*100
#     tot=ag["赠送金额"].sum(); ag["占比"]=ag["赠送金额"]/tot*100; ag["人均"]=ag["赠送金额"]/ag["赠送人数"].replace(0,np.nan)
#     ag["日均_本"]=ag["赠送人数"]/7; ag["日均_上"]=ag["lw_人数"]/7
#     ag["人数环比"]=(ag["赠送人数"]-ag["lw_人数"])/ag["lw_人数"].replace(0,np.nan)*100
#     return ag.sort_values("赠送金额",ascending=False), tot

# def load_tool_data():
#     tw=pd.read_csv(FILES["tool_tw"]); lw=pd.read_csv(FILES["tool_lw"])
#     tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str).str.strip()
#     tm=tm.drop_duplicates(subset="道具ID",keep="first")
#     for df in [tw,lw]:
#         df.columns=df.columns.str.strip().str.replace("\ufeff","")
#         df["道具ID"]=df["道具ID"].astype(str).str.strip().str.replace('"',"")
#         for c in ["道具发放(步骤1)","道具使用(步骤2)"]:
#             df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")
#         df["步骤2 转化"]=pd.to_numeric(df["步骤2 转化"].astype(str).str.replace("%",""),errors="coerce")
#     tw=tw[tw["道具ID"]!="总体"].copy(); lw=lw[lw["道具ID"]!="总体"].copy()
#     m=tw.rename(columns={"道具发放(步骤1)":"本周发放","道具使用(步骤2)":"本周使用","步骤2 转化":"本周使用率"}
#     ).merge(lw[["道具ID","道具发放(步骤1)","道具使用(步骤2)","步骤2 转化"]].rename(
#         columns={"道具发放(步骤1)":"上周发放","道具使用(步骤2)":"上周使用","步骤2 转化":"上周使用率"}),
#         on="道具ID",how="outer").fillna(0)
#     m=m.merge(tm[["道具ID","活动","备注"]],on="道具ID",how="left")
#     m["活动"]=m["活动"].fillna("其他"); m["备注"]=m["备注"].fillna(m["道具ID"])
#     m["发放环比"]=(m["本周发放"]-m["上周发放"])/m["上周发放"].replace(0,np.nan)*100
#     m["使用环比"]=(m["本周使用"]-m["上周使用"])/m["上周使用"].replace(0,np.nan)*100
#     return m.sort_values("本周发放",ascending=False)

# def load_first_dep_ret():
#     df=pd.read_csv(FILES["first_dep_ret"])
#     df.columns=df.columns.str.strip().str.replace("\ufeff","")
#     df["_d"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
#     df["_yyyymmdd"]=df["_d"].str.replace("-","")
#     for c in ["当日","1日","2日","3日","4日","5日","6日","7日"]:
#         if c in df.columns:
#             df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
#     df["账变事件用户数"]=pd.to_numeric(df["账变事件用户数"],errors="coerce").fillna(0)
#     stage=df[df["初始事件发生时间"]=="阶段值"].copy()
#     daily=df[df["_yyyymmdd"].notna()&df["_yyyymmdd"].str.match(r"^\d{8}$")].copy()
#     rr=daily[daily["指标"]=="留存率"].reset_index(drop=True)
#     nr=daily[daily["指标"]=="留存人数"].reset_index(drop=True)
#     tw_r=rr[rr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
#     lw_r=rr[rr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
#     tw_n=nr[nr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
#     lw_n=nr[nr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
#     def wa(rd,nd,col):
#         if col not in rd.columns or len(rd)==0: return np.nan
#         rv=rd[col].to_numpy(dtype=float,na_value=np.nan); ok=~np.isnan(rv)
#         if not ok.any(): return np.nan
#         w=nd["账变事件用户数"].to_numpy(dtype=float) if len(nd)==len(rd) else np.ones(len(rv))
#         ww=w[ok]; return float(np.nanmean(rv[ok])) if ww.sum()==0 else float(np.average(rv[ok],weights=ww))
#     R={"stage":stage,"tw_users":int(tw_n["账变事件用户数"].sum()),"lw_users":int(lw_n["账变事件用户数"].sum())}
#     for col in ["1日","2日","3日","4日","5日","6日","7日"]:
#         R[f"tw_{col}"]=wa(tw_r,tw_n,col); R[f"lw_{col}"]=wa(lw_r,lw_n,col)
#     dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
#     R["tw_platform_fc"]=int(dp[dp["日期"].between(*THIS_WEEK)]["首充人数"].sum())
#     R["lw_platform_fc"]=int(dp[dp["日期"].between(*LAST_WEEK)]["首充人数"].sum())
#     return R

# def load_risk(dt_raw, dc_raw):
#     df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
#     ba=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
#     wa=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
#     bc=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数")
#     tg=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
#         .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
#         .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏","阶段汇总":"主游投注"}).reset_index())
#     ug=pd.concat([bc,ba,wa],axis=1).reset_index()
#     ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]; ug=ug.merge(tg,on="账户ID",how="left")
#     top500=dt_raw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
#     top500["赢家"]=top500["公司输赢"]<0; top500["投充比"]=top500["投注金额"]/top500["充值金额"]
#     hr=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
#     pp=df[(df["游戏名称"]=="Auto-Roulette 1")&(df["分析指标"]=="投注次数")]["账户ID"].unique()
#     c207=[u for u in pp if str(u).startswith("207")]
#     c207_dt=dt_raw[dt_raw["账户ID"].isin(c207)].sort_values("提款金额",ascending=False) if c207 else pd.DataFrame()
#     sp=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False)
#     bg=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
#     wg=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
#     ug2=df[df["分析指标"]=="投注次数"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
#     wn=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
#     g=bg.merge(wg,on=["show_name_厂商标签id","游戏名称"],how="left").merge(ug2,on=["show_name_厂商标签id","游戏名称"],how="left").merge(wn,on=["show_name_厂商标签id","游戏名称"],how="left")
#     g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100; g["人均投注额"]=g["投注金额"]/g["玩家数"]
#     rg=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
#     return hr,c207_dt,c207,sp,g.sort_values("投注金额",ascending=False).head(20),rg,top500
# # ════════════════════════════════════════════════════════════════
# # 图表
# # ════════════════════════════════════════════════════════════════
# SPLIT = 7

# def _vline(ax, dates):
#     ax.axvline(SPLIT-.5,color="#94a3b8",ls="--",lw=1,alpha=.7)
#     ax.text(SPLIT-4, ax.get_ylim()[1]*.93,"上周",ha="center",fontsize=7,color="#64748b")
#     ax.text(SPLIT+3, ax.get_ylim()[1]*.93,"本周",ha="center",fontsize=7,color="#1d4ed8")

# def chart_trend(trend):
#     fig=plt.figure(figsize=(16,10),facecolor="white")
#     gs=gridspec.GridSpec(2,2,figure=fig,hspace=.45,wspace=.3)
#     dates=trend["dates"]; x=range(len(dates))
#     def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=.25)
#     ax=fig.add_subplot(gs[0,0])
#     ax.bar(x,[v/10000 for v in trend["充值"]],color=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT,width=.7,label="充值")
#     ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
#     ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold"); ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
#     ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax,dates)
#     ax2=fig.add_subplot(gs[0,1])
#     ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=.7)
#     ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax2); _vline(ax2,dates)
#     ax3=fig.add_subplot(gs[1,0])
#     ax3.bar(x,trend["首充"],color=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT,width=.7,label="首充人数")
#     ax3r=ax3.twinx(); ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
#     ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold"); ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
#     l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
#     ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left"); ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=.25); ax3.axvline(SPLIT-.5,color="#94a3b8",ls="--",lw=1,alpha=.7)
#     ax4=fig.add_subplot(gs[1,1])
#     ax4.bar(x,trend["充提差比"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=.7)
#     tm=np.mean(trend["充提差比"][SPLIT:]); lm=np.mean(trend["充提差比"][:SPLIT])
#     ax4.axhline(tm,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lm,color="#94a3b8",ls=":",lw=1.2)
#     ax4.text(len(dates)-.5,tm+.3,f"本周均{tm:.1f}%",fontsize=7,color="#7c3aed",ha="right")
#     ax4.text(.5,lm+.3,f"上周均{lm:.1f}%",fontsize=7,color="#64748b",ha="left")
#     ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold"); ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax4); _vline(ax4,dates)
#     fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
#     plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,11)

# def chart_ret_weekly(weeks):
#     """近4周留存图，加目标线，列用 第1日/第2日/第6日"""
#     fig,ax=plt.subplots(figsize=(14,6),facecolor="white")
#     cols=[("第1日","次留","#1d4ed8"),("第2日","3留","#059669"),("第6日","7留","#7c3aed")]
#     x=np.arange(len(weeks)); w=.25
#     for i,(col,lbl,clr) in enumerate(cols):
#         vals=[wk.get(col,np.nan) for wk in weeks]
#         bars=ax.bar(x+i*w-w,vals,w,label=lbl,color=clr,alpha=.85)
#         for bar,v in zip(bars,vals):
#             if not (isinstance(v,float) and np.isnan(v)):
#                 ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
#         tgt=RET_TARGETS.get(lbl)
#         if tgt: ax.axhline(tgt,color=clr,ls="--",lw=1.2,alpha=.55)
#     ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
#     ax.set_title("首充用户充值留存率 近4周对比（含目标虚线）",fontsize=10,fontweight="bold")
#     ax.legend(fontsize=8,loc="upper left"); ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=.5)
#     ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,14,6)

# def chart_vip(tw_v,lw_v):
#     vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
#     x=np.arange(len(vips)); w=.35
#     fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
#     tc=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
#     lc=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
#     ax1.bar(x-w/2,tc,w,label="本周",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,lc,w,label="上周",color="#93c5fd",alpha=.7)
#     ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     tb=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
#     lb=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
#     ax2.bar(x-w/2,tb,w,label="本周",color="#059669",alpha=.85); ax2.bar(x+w/2,lb,w,label="上周",color="#6ee7b7",alpha=.7)
#     ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
#     chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
#     ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=.85,width=.6); ax3.axhline(0,color="black",lw=.8)
#     ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,16,6)

# def chart_vip_retention(vr):
#     vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=.28
#     for ax,rt,t in [(ax1,"chg","充值→充值 留存率（%）"),(ax2,"act","充值→活跃 留存率（%）")]:
#         tw1=[vr[rt]["tw"]["1日"].get(v,0) for v in vips]; lw1=[vr[rt]["lw"]["1日"].get(v,0) for v in vips]; tw3=[vr[rt]["tw"]["3日"].get(v,0) for v in vips]
#         clr=("#1d4ed8","#93c5fd","#059669") if rt=="chg" else ("#7c3aed","#c4b5fd","#d97706")
#         ax.bar(x-w,tw1,w,label="次日(本周)",color=clr[0],alpha=.85); ax.bar(x,lw1,w,label="次日(上周)",color=clr[1],alpha=.7); ax.bar(x+w,tw3,w,label="3日(本周)",color=clr[2],alpha=.75)
#         ax.set_title(t,fontsize=10,fontweight="bold"); ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8); ax.legend(fontsize=7.5,loc="upper left"); ax.grid(axis="y",alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
#     plt.suptitle("VIP各等级充值留存率",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

# def chart_mfr_combo(mfr):
#     top10=mfr.head(10); names=top10["厂商显示名"].tolist() if "厂商显示名" in top10.columns else top10["show_name_厂商标签id"].tolist()
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
#     sh=top10["占比"].tolist(); ls=top10["lw_占比"].fillna(0).tolist()
#     bars=ax1.bar(names,sh,color=["#059669" if c>=l else "#dc2626" for c,l in zip(sh,ls)],alpha=.85,width=.6)
#     ax1.plot(names,ls,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
#     for bar,s,ic in zip(bars,sh,top10["投注环比"].tolist()):
#         ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.2,f"{s:.1f}%",ha="center",fontsize=7)
#         ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
#     ax1.set_title("Top10厂商 投注份额",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
#     ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
#     ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     x=np.arange(len(names)); w=.35
#     ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=.85)
#     ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=.7)
#     ax2.axhline(0,color="black",lw=.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5); ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
#     plt.tight_layout(); return fig_img(fig,16,6.5)

# def chart_top30(top30):
#     t15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in t15.iterrows()]
#     bets=[r["投注金额"]/10000 for _,r in t15.iterrows()]; lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in t15.iterrows()]
#     x=np.arange(len(names)); w=.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
#     ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=.85); ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=.7)
#     for bar,v in zip(ax.patches[:len(names)],bets[::-1]): ax.text(bar.get_width()+.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
#     ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8); ax.set_title("Top15游戏 投注金额（万USD）",fontsize=9,fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="x",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,14,7)

# def chart_pref(pg,ma):
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
#     names=ma.index[:8].tolist(); vals=ma.values[:8].tolist(); tot=sum(vals); pcts=[v/tot*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
#     wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
#     for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
#     ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-.05,-.18)); ax1.set_title("Top500提款用户 厂商偏好",fontsize=9,fontweight="bold")
#     t12=pg.head(12); gn=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in t12.iterrows()]; gv=[r["本周投注"]/10000 for _,r in t12.iterrows()]; gc=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in t12.iterrows()]
#     ax2.barh(range(len(gn))[::-1],gv,color=gc[::-1],alpha=.85); ax2.set_yticks(range(len(gn))); ax2.set_yticklabels(gn,fontsize=7.5); ax2.set_title("偏好游戏Top12（红=平台亏损）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万")); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,16,6)

# def chart_activities(act):
#     t10=act.head(10); names=[str(r["name_账变opt_id"])[:12] for _,r in t10.iterrows()]; vals=[r["赠送金额"]/10000 for _,r in t10.iterrows()]; lw=[r["lw_赠"]/10000 for _,r in t10.iterrows()]; envs=[r["环比"] for _,r in t10.iterrows()]
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=.35
#     ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=.85); ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=.7); ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8); ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold"); ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     es=[v if not pd.isna(v) else 0 for v in envs]
#     ax2.barh(range(len(names)),es[::-1],color=["#059669" if v>0 else "#dc2626" for v in es[::-1]],alpha=.85); ax2.axvline(0,color="black",lw=.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8); ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%")); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,16,6)

# def chart_tool_usage(tool):
#     big=tool[tool["本周发放"]>=100].head(12); t12=big if len(big)>0 else tool.head(12)
#     def _l(r):
#         a=str(r.get("活动","")); n=str(r.get("备注",""))
#         return f"{str(r['道具ID'])[:6]}\n{n[:8]}" if a=="其他" else f"[{a[:4]}]\n{n[:8]}"
#     names=[_l(r) for _,r in t12.iterrows()]
#     ti=[r["本周发放"] for _,r in t12.iterrows()]; li=[r["上周发放"] for _,r in t12.iterrows()]
#     tr=[r["本周使用率"] if not pd.isna(r["本周使用率"]) else 0 for _,r in t12.iterrows()]; lr=[r["上周使用率"] if not pd.isna(r["上周使用率"]) else 0 for _,r in t12.iterrows()]
#     fig=plt.figure(figsize=(16,9),facecolor="white"); gs=gridspec.GridSpec(2,2,figure=fig,hspace=.5,wspace=.35)
#     x=np.arange(len(names)); w=.35
#     ax1=fig.add_subplot(gs[0,0]); ax1.bar(x-w/2,ti,w,label="本周发放",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,li,w,label="上周发放",color="#93c5fd",alpha=.7)
#     ax1.set_title("Top12道具 发放量",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(names,fontsize=6.5); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v/10000:.0f}万" if v>=10000 else f"{v:.0f}"))
#     ax2=fig.add_subplot(gs[0,1]); ax2.bar(x-w/2,tr,w,label="本周使用率",color="#059669",alpha=.85); ax2.bar(x+w/2,lr,w,label="上周使用率",color="#6ee7b7",alpha=.7)
#     ax2.set_title("Top12道具 使用率",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(names,fontsize=6.5); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
#     ax3=fig.add_subplot(gs[1,:])
#     di=[r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0 for _,r in t12.iterrows()]
#     du=[r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0 for _,r in t12.iterrows()]
#     ax3.bar(x-w/2,di,w,label="发放量环比",color=["#059669" if v>=0 else "#dc2626" for v in di],alpha=.85)
#     ax3.bar(x+w/2,du,w,label="使用量环比",color=["#7c3aed" if v>=0 else "#f97316" for v in du],alpha=.7)
#     ax3.axhline(0,color="black",lw=.8); ax3.set_title("Top12道具 发放/使用量 环比变化（%）",fontsize=9,fontweight="bold")
#     ax3.set_xticks(x); ax3.set_xticklabels(names,fontsize=6.5); ax3.legend(fontsize=7.5); ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
#     ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
#     plt.suptitle("道具发放与使用分析",fontsize=11,fontweight="bold",y=1.01); plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,10)

# def chart_first_dep_ret(R):
#     lv=[R.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     tv=[R.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     fig,ax=plt.subplots(figsize=(12,5),facecolor="white"); x=np.arange(7)
#     ax.plot(x,lv,"-o",color="#93c5fd",lw=2,ms=6,label=f"上周（{LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}）")
#     ax.plot(x,tv,"-o",color="#1d4ed8",lw=2,ms=6,label=f"本周（{THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}）")
#     days=["D1","D2","D3","D4","D5","D6","D7"]
#     for i,(lval,tval) in enumerate(zip(lv,tv)):
#         if lval is not None and not (isinstance(lval,float) and np.isnan(lval)): ax.text(i,lval+.4,f"{lval:.1f}%",ha="center",fontsize=7.5,color="#64748b")
#         if tval is not None and not (isinstance(tval,float) and np.isnan(tval)): ax.text(i,tval-1.2,f"{tval:.1f}%",ha="center",fontsize=7.5,color="#1d4ed8",fontweight="bold")
#     ax.set_xticks(x); ax.set_xticklabels(days,fontsize=9); ax.set_title("首次充值活动用户 充值留存趋势",fontsize=10,fontweight="bold")
#     ax.legend(fontsize=8.5,loc="upper right"); ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
#     plt.tight_layout(); return fig_img(fig,13,5.5)

# def chart_risk_scatter(top500):
#     v=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy(); v=v[v["投充比"]<200]
#     fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
#     ax.scatter(v["投充比"],v["公司输赢"]/10000,c=["#dc2626" if x<0 else "#059669" for x in v["公司输赢"]],s=[min(abs(x)/500+20,200) for x in v["公司输赢"]],alpha=.55,edgecolors="none")
#     ax.axhline(0,color="black",lw=.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=.7); ax.text(21,ax.get_ylim()[0]*.9,"投充比=20x",fontsize=7.5,color="#d97706")
#     for _,r in top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()].iterrows():
#         ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),xytext=(r["投充比"]+5,r["公司输赢"]/10000-.2),fontsize=6.5,color="#991b1b",arrowprops=dict(arrowstyle="->",color="#991b1b",lw=.7))
#     ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8); ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
#     ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=.6,label="平台赢钱")],fontsize=8,loc="upper right"); ax.grid(alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,15,6)

# def chart_risk_games(rg):
#     t12=rg.head(12); names=[r["游戏名称"][:16] for _,r in t12.iterrows()]; losses=[abs(r["公司输赢"]) for _,r in t12.iterrows()]; rates=[r["盈亏率"] for _,r in t12.iterrows()]
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
#     ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=.85)
#     for bar,v in zip(ax1.patches,losses[::-1]): ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
#     ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
#     ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=.85); ax2.axvline(0,color="black",lw=.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
#     plt.tight_layout(); return fig_img(fig,16,6.5)
# # ════════════════════════════════════════════════════════════════
# # 章节构建
# # ════════════════════════════════════════════════════════════════

# def _render(t, **kw):
#     try: return t.format(**kw)
#     except: return t

# def build_overview(K, trend, weekly_ret, dash_ret):
#     full = PW - 2*MARGIN
#     S = [sec_title("一、大盘核心数据")]
#     kpis = [
#         ("充值金额",   f"{K['tw_充值金额']/10000:.1f}万",   f"{K['lw_充值金额']/10000:.1f}万",   K["pct_充值金额"],   True),
#         ("提现金额",   f"{K['tw_提现金额']/10000:.1f}万",   f"{K['lw_提现金额']/10000:.1f}万",   K["pct_提现金额"],   False),
#         ("充提差",     f"{K['tw_充提差']/10000:.1f}万",     f"{K['lw_充提差']/10000:.1f}万",     K["pct_充提差"],     True),
#         ("充提差率",   f"{K['tw_充提差比']:.2f}%",          f"{K['lw_充提差比']:.2f}%",          K["pct_充提差比"],   True),
#         ("公司输赢",   f"{K['tw_公司输赢']/10000:.1f}万",   f"{K['lw_公司输赢']/10000:.1f}万",   K["pct_公司输赢"],   True),
#         ("盈亏率",     f"{K['tw_盈亏率']:.3f}%",            f"{K['lw_盈亏率']:.3f}%",            K["pct_盈亏率"],     True),
#         ("注册人数",   f"{int(K['tw_注册人数']):,}",        f"{int(K['lw_注册人数']):,}",        K["pct_注册人数"],   True),
#         ("首充人数",   f"{int(K['tw_首充人数']):,}",        f"{int(K['lw_首充人数']):,}",        K["pct_首充人数"],   True),
#         ("日均活跃",   f"{K['tw_活跃人数']/7/10000:.1f}万", f"{K['lw_活跃人数']/7/10000:.1f}万", K["pct_活跃人数"],   True),
#         ("投注金额",   f"{K['tw_投注金额']/10000:.0f}万",   f"{K['lw_投注金额']/10000:.0f}万",   K["pct_投注金额"],   True),
#         ("全量ARPPU",  f"${K['tw_全量Arppu']:.2f}",         f"${K['lw_全量Arppu']:.2f}",         K["pct_全量Arppu"],  True),
#         ("老用户ARPPU",f"${K['tw_老用户ARPPU']:.2f}",       f"${K['lw_老用户ARPPU']:.2f}",       K["pct_老用户ARPPU"],True),
#         ("首充ARPPU",  f"${K['tw_首充Arppu']:.2f}",         f"${K['lw_首充Arppu']:.2f}",         K["pct_首充Arppu"],  True),
#         ("总赠送金额", f"{K['tw_总赠送金额']/10000:.1f}万", f"{K['lw_总赠送金额']/10000:.1f}万", K["pct_总赠送金额"], False),
#         ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",        f"{K['lw_赠送充值比']:.2f}%",        K["pct_赠送充值比"], False),
#         ("首充转化率", f"{K['tw_首充转化率']:.1f}%",        f"{K['lw_首充转化率']:.1f}%",        K["pct_首充转化率"], True),
#         ("首充次日留存",f"{K['tw_首充次日复充率']:.1f}%",   f"{K['lw_首充次日复充率']:.1f}%",   K["pct_首充次日复充率"],True),
#         ("推广消耗",   f"{K['tw_真实消耗']/10000:.1f}万",   f"{K['lw_真实消耗']/10000:.1f}万",   K["pct_真实消耗"],   False),
#     ]
#     S.append(kpi_card4(kpis, cols=4)); S.append(Spacer(1,8))
#     cr_d = K["tw_充提差比"]-K["lw_充提差比"]; rd = K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
#     ov = dict(tw_充值金额=K["tw_充值金额"]/10000, pct_充值金额=K["pct_充值金额"],
#               tw_公司输赢=K["tw_公司输赢"]/10000, pct_公司输赢=K["pct_公司输赢"],
#               pct_真实消耗=K["pct_真实消耗"], tw_充提差比=K["tw_充提差比"], lw_充提差比=K["lw_充提差比"],
#               delta_充提差比=cr_d, tw_盈亏率=K["tw_盈亏率"], lw_盈亏率=K["lw_盈亏率"],
#               tw_首充人数=int(K["tw_首充人数"]), pct_首充人数=K["pct_首充人数"],
#               tw_首充转化率=K["tw_首充转化率"], tw_首充Arppu=K["tw_首充Arppu"],
#               pct_首充Arppu=K["pct_首充Arppu"], tw_首充次日复充率=K["tw_首充次日复充率"],
#               lw_首充次日复充率=K["lw_首充次日复充率"], delta_首充次日复充率=rd)
#     S.append(insight_box([
#         _render("充值{tw_充值金额:.1f}万（{pct_充值金额:+.1f}%），公司输赢{tw_公司输赢:.1f}万（{pct_公司输赢:+.1f}%）；推广消耗{pct_真实消耗:+.1f}%。",**ov),
#         _render("充提差率{tw_充提差比:.2f}%（上周{lw_充提差比:.2f}%，{delta_充提差比:+.2f}pp）；盈亏率{tw_盈亏率:.3f}%（上周{lw_盈亏率:.3f}%）。",**ov),
#         _render("首充人数{tw_首充人数:,}（{pct_首充人数:+.1f}%），转化率{tw_首充转化率:.1f}%，首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%）。",**ov),
#         _render("首充次日充值留存{tw_首充次日复充率:.1f}%（上周{lw_首充次日复充率:.1f}%，{delta_首充次日复充率:+.1f}pp）。",**ov),
#     ]))
#     S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))

#     # ── 大盘留存汇总表（含目标对比）──────────────────────────────
#     S.append(sub_title("大盘首充留存 vs 目标对比（来源：日报 首充2/3/7日复充率）"))
#     specs = [
#         ("次留", "tw_nd", "lw_nd", "tw_nd_end"),
#         ("3留",  "tw_td", "lw_td", "tw_td_end"),
#         ("7留",  "tw_sd", "lw_sd", "tw_sd_end"),
#     ]
#     ret_headers = ["留存类型","数据截止","本周实际","上周实际","周环比(pp)","目标","vs目标(pp)"]
#     ret_rows = []
#     for rtype, tw_k, lw_k, end_k in specs:
#         tw_v = dash_ret.get(tw_k, np.nan); lw_v = dash_ret.get(lw_k, np.nan)
#         end_lbl = fmt_lbl(dash_ret.get(end_k))
#         tgt = RET_TARGETS.get(rtype, np.nan)
#         wpp = (tw_v-lw_v) if not (np.isnan(tw_v) or np.isnan(lw_v)) else np.nan
#         dpp = (tw_v-tgt) if not (np.isnan(tw_v) or np.isnan(tgt)) else np.nan
#         tw_clr = C_GREEN if (not np.isnan(tw_v) and not np.isnan(tgt) and tw_v>=tgt) else C_RED
#         wpp_clr = C_GREEN if (not np.isnan(wpp) and wpp>=0) else C_RED
#         dpp_clr = C_GREEN if (not np.isnan(dpp) and dpp>=0) else C_RED
#         ret_rows.append([
#             cell(rtype, True, C_DARK),
#             cell(end_lbl, False, C_GRAY, TA_CENTER),
#             cell(fret(tw_v), False, tw_clr, TA_RIGHT),
#             cell(fret(lw_v), False, C_GRAY, TA_RIGHT),
#             cell(f"{wpp:+.1f}pp" if not np.isnan(wpp) else "-", False, wpp_clr, TA_RIGHT),
#             cell(f"{tgt:.0f}%" if not np.isnan(tgt) else "-", True, C_TARGET, TA_CENTER),
#             cell(f"{dpp:+.1f}pp" if not np.isnan(dpp) else "-", True, dpp_clr, TA_RIGHT),
#         ])
#     extra = [("BACKGROUND",(5,1),(5,-1), colors.HexColor("#eff6ff"))]
#     S.append(dtable(ret_headers, ret_rows,
#                     [full*x for x in [0.16,0.12,0.14,0.14,0.14,0.12,0.14]],
#                     fsize=8, extra_style=extra))
#     S.append(Spacer(1,4))
#     nd_end=fmt_lbl(dash_ret.get("tw_nd_end")); td_end=fmt_lbl(dash_ret.get("tw_td_end")); sd_end=fmt_lbl(dash_ret.get("tw_sd_end"))
#     S.append(P(f"本周截止：次留~{nd_end}，3留~{td_end}，7留~{sd_end}（剔除数据未满足lag天数的不完整日期）",7,False,C_GRAY))
#     S.append(Spacer(1,8))

#     # ── 近4周留存图 ──────────────────────────────────────────────
#     S.append(sub_title("首充用户充值留存 — 近4周对比（含目标线）"))
#     S.append(chart_ret_weekly(weekly_ret)); S.append(Spacer(1,4))
#     this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
#     d1=this_w.get("第1日",0) or 0; d2=this_w.get("第2日",0) or 0; d6=this_w.get("第6日",0) or 0
#     ld1=last_w.get("第1日",0) or 0; ld6=last_w.get("第6日",0) or 0
#     S.append(insight_box([
#         f"本周首充次日留存{d1:.1f}%，较上周{d1-ld1:+.1f}pp；3留{d2:.1f}%，7留{d6:.1f}%。",
#         f"注：本周7日留存因截止日期不完整，以上周7留{ld6:.1f}%作为参考基准。",
#     ], clr=C_AMBER))
#     return S


# def build_agents(agents, agent_ret, ends):
#     """
#     v7.3 总代表：
#     ① 3留(本/上) 合并一列，格式 X.X%/Y.Y%，同次日留存
#     ② 数据源：首充充值留存_** 2日=次留 3日=3留 6日=7留
#     ③ 列宽/字号优化确保单行
#     """
#     full = PW - 2*MARGIN
#     S = [sec_title("二、总代分析")]

#     tw_nd_lbl = fmt_lbl(ends.get("tw_nd_end")); lw_nd_lbl = fmt_lbl(ends.get("lw_nd_end"))
#     tw_3d_lbl = fmt_lbl(ends.get("tw_3d_end")); lw_3d_lbl = fmt_lbl(ends.get("lw_3d_end"))
#     lw_7d_lbl = fmt_lbl(ends.get("lw_7d_end"))

#     S.append(sub_title("全量总代表现（本周 vs 上周，按充值金额排序，含官方总代0）"))
#     S.append(P(
#         f"数据源：首充充值留存_**（当日=第0日，2日列=次留，3日列=3留，6日列=7留）  "
#         f"截止：次留本{tw_nd_lbl}/上{lw_nd_lbl}，3留本{tw_3d_lbl}/上{lw_3d_lbl}，7留上{lw_7d_lbl}",
#         6.5, False, C_GRAY)); S.append(Spacer(1,3))

#     # v7.3：11列，去掉独立3留上周，合并为3留(本/上)
#     headers = [
#         "ID", "总代名称", "注册(环比)", "首充\n人数", "充值\n(万)",
#         "充提差率\n(差值pp)", "消耗\n(万)", "1级首充\n成本(本/上)",
#         f"次留(本/上)\n~{tw_nd_lbl}/{lw_nd_lbl}",
#         f"3留(本/上)\n~{tw_3d_lbl}/{lw_3d_lbl}",
#         f"7留上周\n~{lw_7d_lbl}",
#     ]

#     rows = []
#     for _, r in agents.iterrows():
#         aid = int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
#         rname = str(r["总代.名称"])
#         cr = r["充提差率"]; lw_cr = r.get("lw_充提差率", 0) or 0; cr_d = cr - lw_cr
#         cost = (r.get("总消耗", 0) or 0) / 10000
#         fcc = r.get("一级首充成本", 0) or 0; lfc = r.get("lw_fc_cost", 0) or 0
#         reg = int(r["注册人数"]); reg_c = r.get("注册环比", 0) or 0
#         reg_s = f"{reg:,}/{'+' if reg_c>=0 else ''}{reg_c:.0f}%"
#         cr_s = f"{cr:.1f}%/{'+' if cr_d>=0 else ''}{cr_d:.1f}pp"
#         cr_clr = C_GREEN if cr >= CR_HIGH else (C_RED if cr < CR_LOW else C_DARK)

#         d = agent_ret.get(rname, {})
#         tw_nd = d.get("tw_nd", np.nan); lw_nd = d.get("lw_nd", np.nan)
#         tw_3d = d.get("tw_3d", np.nan); lw_3d = d.get("lw_3d", np.nan)
#         lw_7d = d.get("lw_7d", np.nan)

#         def _fmt_pair(tw_v, lw_v):
#             if not np.isnan(tw_v) and not np.isnan(lw_v):
#                 return f"{tw_v:.1f}%/{lw_v:.1f}%", C_GREEN if tw_v >= lw_v else C_RED
#             if not np.isnan(tw_v): return f"{tw_v:.1f}%/-", C_DARK
#             return "-", C_GRAY

#         nd_s, nd_clr = _fmt_pair(tw_nd, lw_nd)
#         td_s, td_clr = _fmt_pair(tw_3d, lw_3d)

#         rows.append([
#             cell(str(aid), False, C_GRAY, TA_CENTER),
#             cell(rname[:14], True, C_DARK, TA_LEFT),
#             cell(reg_s, False, C_GREEN if reg_c >= 0 else C_RED, TA_RIGHT),
#             cell(f"{int(r['首充人数']):,}", False, C_DARK, TA_RIGHT),
#             cell(f"{r['充值金额']/10000:.0f}", False, C_DARK, TA_RIGHT),
#             cell(cr_s, False, cr_clr, TA_RIGHT),
#             cell(f"{cost:.1f}" if cost > 0 else "-", False, C_DARK, TA_RIGHT),
#             cell(f"${fcc:.0f}/${lfc:.0f}" if fcc > 0 else "-", False, C_DARK, TA_RIGHT),
#             cell(nd_s, False, nd_clr, TA_RIGHT),
#             cell(td_s, False, td_clr, TA_RIGHT),   # ← 3留(本/上) 合并列
#             cell(fret(lw_7d), False, C_DARK, TA_RIGHT),
#         ])

#     # 11列宽，合计 = 1.0 × full
#     # ID  名称   注册     首充   充值   充提差率  消耗   首充成本  次留   3留    7留
#     cw = [full*x for x in [0.04, 0.14, 0.10, 0.06, 0.05, 0.11, 0.05, 0.10, 0.11, 0.11, 0.08]]
#     S.append(dtable(headers, rows, cw, fsize=6.2))
#     S.append(Spacer(1,4))
#     S.append(P(
#         f"★ 充提差率≥{CR_HIGH}%绿，<{CR_LOW}%红。"
#         f"次留截止本~{tw_nd_lbl}/上~{lw_nd_lbl}；"
#         f"3留截止本~{tw_3d_lbl}/上~{lw_3d_lbl}；"
#         f"7留上周全量截止~{lw_7d_lbl}。",
#         6.5, False, C_GRAY)); S.append(Spacer(1,6))
#     dsp = agents[agents["总代.名称"].str.contains("DSP", na=False)]
#     S.append(insight_box([
#         _render("A8_DSP_谷歌原生包充提差率{dsp_cr:.1f}%，注册{dsp_reg:+.1f}%，体量靠前且指标双优。",
#                 dsp_cr=dsp["充提差率"].values[0] if len(dsp)>0 else 0,
#                 dsp_reg=dsp["注册环比"].values[0] if len(dsp)>0 else 0),
#         "苹果包总代注册+27.1%，充值体量第二，为本周注册增量最大渠道之一。",
#         "YB_FB_PWA_0注册暴跌-79%至1,315人，需持续监控是否恢复。",
#     ]))
#     return S


# def build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_tw):
#     full = PW - 2*MARGIN; S = [sec_title("三、用户分析")]
#     S.append(sub_title("VIP等级分层分析")); S.append(chart_vip(tw_v, lw_v)); S.append(Spacer(1,4))
#     tot = tw_v["充值金额"].sum(); hvs = sum(tw_v.loc[v,"充值金额"] for v in [9,10,11] if v in tw_v.index)
#     S.append(insight_box([
#         f"VIP9-11高价值层合计贡献充值{hvs/tot*100:.1f}%，高端用户付费意愿持续强劲。",
#         f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。" if 1 in tw_v.index else "VIP1基础层数据不可用。"
#     ]))
#     vr = load_vip_retention()
#     S.append(sub_title("VIP各等级充值留存率")); S.append(chart_vip_retention(vr)); S.append(Spacer(1,4))
#     S.append(insight_box([
#         f"充值→活跃次日留存：VIP9达{vr['act']['tw']['1日'].get(9,0):.1f}%，VIP10达{vr['act']['tw']['1日'].get(10,0):.1f}%。",
#         f"充值→充值次日留存：VIP10达{vr['chg']['tw']['1日'].get(10,0):.1f}%（上周{vr['chg']['lw']['1日'].get(10,0):.1f}%）。",
#     ])); S.append(Spacer(1,8))

#     S.append(sub_title("头部充值用户分层分析"))
#     S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
#     h=["分层","本周充值","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","占全量%","公司输赢"]
#     rows=[]
#     for d in dep_t:
#         rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),
#                      cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
#                      rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),
#                      cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
#                      cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
#     S.append(dtable(h,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
#     S.append(insight_box([
#         _render("Top10充值用户人均${top10_avg:,.0f}（{top10_avg_chg:+.1f}%），充提差率{top10_cr:.1f}%。",
#                 top10_avg=dep_t[1]["tw_avg"],top10_avg_chg=pct(dep_t[1]["tw_avg"],dep_t[1]["lw_avg"]),top10_cr=dep_t[1]["tw_cr"]),
#         "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。"
#     ]))

#     S.append(sub_title("头部提款用户分层分析"))
#     S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
#     h3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","赢家比例","活动占比","占全量%","剔除后大盘\n充提差影响"]
#     rows3=[]
#     for d in wdr_t:
#         rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),
#                       cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
#                       rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),
#                       cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),
#                       cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
#                       cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),
#                       cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),
#                       cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
#                       cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
#     S.append(dtable(h3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
#     S.append(insight_box([
#         f"剔除Top100提款用户后，大盘充提差率影响+{wdr_t[3]['大盘影响']:.2f}pp，头部提款用户对充提差率有明显拖累。",
#         f"Top500提款用户活动奖励占资金来源仅{wdr_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。",
#     ], clr=C_AMBER))
#     return S


# def build_games(mfr, top30, delta):
#     full = PW - 2*MARGIN; S = [sec_title("四、游戏分析")]
#     S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
#     h=["排名","厂商","日均投注人数(本/上)","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
#     rows=[]
#     for i,(_,r) in enumerate(mfr.head(10).iterrows()):
#         pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
#         rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r.get("厂商显示名",r["show_name_厂商标签id"])),True),
#                      cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),
#                      cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
#                      cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
#                      cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
#                      cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
#     S.append(dtable(h,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
#     def _mv(nm,col):
#         sub=mfr[mfr["show_name_厂商标签id"]==nm]; return sub[col].values[0] if len(sub)>0 else 0
#     S.append(insight_box([
#         _render("Tada投注份额{ts:.1f}%（上周{tsl:.1f}%），Fortune系稳固主导；环比{tc:+.1f}%。",
#                 ts=mfr.iloc[0]["占比"],tsl=mfr.iloc[0].get("lw_占比",0),tc=mfr.iloc[0].get("投注环比",0)),
#         _render("Evolution盈亏率{ep:.2f}%（上周{epl:.2f}%），已转正，较上周大幅改善。",
#                 ep=_mv("Evolution","盈亏率"),epl=_mv("Evolution","lw_盈亏率")),
#         _render("3Oaks盈亏率{op:.2f}%，PlayTech{pp:.2f}%，均高于平台均值，可适当扩大曝光权重。",
#                 op=_mv("3Oaks","盈亏率"),pp=_mv("PlayTech","盈亏率")),
#     ]))
#     if delta:
#         S.append(sub_title("▶ 厂商投注额环比主要驱动游戏"))
#         dh=["厂商","投注额环比","增量最大游戏(+贡献)","降量最大游戏(-拖累)"]; dr=[]
#         for mn,gd in delta.items():
#             mrow=mfr[mfr["show_name_厂商标签id"]==mn]; mc=mrow["投注环比"].values[0] if len(mrow)>0 else 0
#             up=gd["up"]; dn=gd["dn"]
#             us=f'{up.iloc[0]["游戏.名称"][:16]}（+${up.iloc[0]["delta"]/10000:.1f}万）' if len(up)>0 and up.iloc[0]["delta"]>0 else "-"
#             ds=f'{dn.iloc[0]["游戏.名称"][:16]}（${dn.iloc[0]["delta"]/10000:.1f}万）' if len(dn)>0 and dn.iloc[0]["delta"]<0 else "-"
#             dr.append([cell(shorten_mfr(mn),True,C_DARK),rc(mc),cell(us,False,C_GREEN if us!="-" else C_GRAY),cell(ds,False,C_RED if ds!="-" else C_GRAY)])
#         S.append(dtable(dh,dr,[full*x for x in [0.15,0.10,0.37,0.38]],fsize=6.8)); S.append(Spacer(1,6))
#     S.append(sub_title("Top30游戏详细数据")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
#     h2=["#","游戏名称","厂商简称","投注人数","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
#     rows2=[]
#     for i,(_,r) in enumerate(top30.iterrows()):
#         pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
#         rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["游戏.名称"])[:18],True),
#                       cell(str(r.get("厂商简称",r["游戏厂商标签.名称"]))[:5]),
#                       cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),
#                       cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
#                       cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
#                       cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
#                       cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
#     S.append(dtable(h2,rows2,[full*x for x in [0.04,0.18,0.07,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
#     return S


# def build_activities(act, tot_gift):
#     full = PW - 2*MARGIN; S = [sec_title("五、活动分析")]
#     S.append(sub_title("各活动赠送效果（全量，含环比）")); S.append(chart_activities(act)); S.append(Spacer(1,4))
#     h=["活动名称","本周赠送","上周赠送","金额环比","本周日均\n赠送人数","上周日均\n赠送人数","人数环比","本周人均\n赠送金额","占比"]
#     rows=[]
#     for _,r in act.iterrows():
#         rows.append([cell(str(r["name_账变opt_id"])[:18],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),
#                      cell(f"${r['lw_赠']:,.0f}",False,C_GRAY,TA_RIGHT),
#                      rc(r["环比"] if not pd.isna(r.get("环比",np.nan)) else 0),
#                      cell(f"{r.get('日均_本',0):.0f}",False,C_DARK,TA_RIGHT),
#                      cell(f"{r.get('日均_上',0):.0f}",False,C_GRAY,TA_RIGHT),
#                      rc(r.get("人数环比",0) if not pd.isna(r.get("人数环比",np.nan)) else 0),
#                      cell(f"${r.get('人均',0):.2f}",False,C_DARK,TA_RIGHT),
#                      cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
#     S.append(dtable(h,rows,[full*x for x in [0.20,0.11,0.11,0.07,0.12,0.12,0.07,0.11,0.09]],fsize=6.5))
#     S.append(Spacer(1,4)); daily=tot_gift/7/10000
#     S.append(P(f"本周总赠送金额：${tot_gift/10000:.2f}万 | 日均赠送：${daily:.2f}万",9,True,C_DARK)); S.append(Spacer(1,6))
#     S.append(insight_box([f"各类赠送活动全量列出（共{len(act)}个），合计本周日均赠送{daily:.1f}万USD。"]))
#     return S


# def build_tools(tool, fdr):
#     full = PW - 2*MARGIN; S = [sec_title("六、道具专题分析", clr=C_TEAL)]
#     S.append(sub_title("6.1 道具发放、使用及使用率（本周发放≥100）"))
#     S.append(chart_tool_usage(tool)); S.append(Spacer(1,4))
#     td=tool[tool["本周发放"]>=100].copy()
#     h=["道具ID","活动类型","备注说明","本周发放","上周发放","发放环比","本周使用","上周使用","使用环比","本周\n使用率","上周\n使用率"]
#     rows=[]
#     for _,r in td.iterrows():
#         di=r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0
#         du=r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0
#         tw_rt=r.get("本周使用率",0) or 0; lw_rt=r.get("上周使用率",0) or 0
#         rows.append([cell(str(r["道具ID"]),False,C_GRAY,TA_CENTER),cell(str(r.get("活动",""))[:10],False,C_PURPLE),
#                      cell(str(r.get("备注",""))[:14],False,C_DARK),
#                      cell(f"{int(r['本周发放']):,}" if r['本周发放']>0 else "-",False,C_DARK,TA_RIGHT),
#                      cell(f"{int(r['上周发放']):,}" if r['上周发放']>0 else "-",False,C_GRAY,TA_RIGHT),rc(di),
#                      cell(f"{int(r['本周使用']):,}" if r['本周使用']>0 else "-",False,C_DARK,TA_RIGHT),
#                      cell(f"{int(r['上周使用']):,}" if r['上周使用']>0 else "-",False,C_GRAY,TA_RIGHT),rc(du),
#                      cell(f"{tw_rt:.1f}%",False,C_GREEN if tw_rt>=lw_rt else C_RED,TA_RIGHT),
#                      cell(f"{lw_rt:.1f}%",False,C_GRAY,TA_RIGHT)])
#     S.append(dtable(h,rows,[full*x for x in [0.07,0.10,0.14,0.08,0.08,0.07,0.08,0.08,0.07,0.07,0.08]],fsize=6.3))
#     S.append(Spacer(1,4))
#     bv=tool[tool["本周发放"]>=100].copy()
#     def _lbl(r):
#         a=str(r.get("活动","")); n=str(r.get("备注",""))
#         return f"{n}({r['本周使用率']:.0f}%)" if a=="其他" else f"[{a}]{n}({r['本周使用率']:.0f}%)"
#     if len(bv)>=3:
#         hi="、".join([_lbl(r) for _,r in bv.nlargest(3,"本周使用率").iterrows()])
#         lo="、".join([_lbl(r) for _,r in bv.nsmallest(3,"本周使用率").iterrows()])
#     else: hi=lo="-"
#     tw_t=int(tool["本周发放"].sum()); lw_t=tool["上周发放"].sum()
#     S.append(insight_box([f"使用率最高3类：{hi}。", f"使用率最低3类：{lo}。",
#                           f"本周总道具发放{tw_t:,}，较上周{lw_t:,.0f}，环比{pct(tw_t,lw_t):+.1f}%。"]))
#     S.append(Spacer(1,8))

#     S.append(sub_title("6.2 首次充值活动用户 充值留存分析（重点）"))
#     stage=fdr.get("stage",pd.DataFrame())
#     if len(stage)>0:
#         lws=LAST_WEEK[0]
#         S.append(P(f"▸ 两周阶段汇总（{lws[:4]}.{lws[4:6]}.{lws[6:]} - {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}）",9,True,C_DARK)); S.append(Spacer(1,3))
#         sr=stage[stage["指标"]=="留存率"]; sn=stage[stage["指标"]=="留存人数"]; sa=stage[stage["指标"]=="人均充值金额"]
#         if len(sr)>0:
#             u=int(sn["账变事件用户数"].values[0]) if len(sn)>0 else 0
#             S.append(P(f"首充活动用户总数：{u:,}人",8.5,False,C_DARK))
#             dc=["当日","1日","2日","3日","4日","5日","6日","7日"]; srows=[]
#             if len(sr)>0: srows.append(["留存率"]+[str(sr.iloc[0].get(c,"-")) for c in dc])
#             if len(sn)>0: srows.append(["留存人数"]+[str(sn.iloc[0].get(c,"-")) for c in dc])
#             if len(sa)>0: srows.append(["人均充值($)"]+[str(sa.iloc[0].get(c,"-")) for c in dc])
#             if srows:
#                 tr2=[[cell(row[0],True,C_DARK)]+[cell(str(v),False,C_DARK,TA_CENTER) for v in row[1:]] for row in srows]
#                 S.append(dtable(["指标"]+dc,tr2,[full*.12]+[full*.11]*8,fsize=6.8))
#         S.append(Spacer(1,6))
#     S.append(P("▸ 本周 vs 上周 首充活动用户留存趋势",9,True,C_DARK)); S.append(Spacer(1,3))
#     S.append(chart_first_dep_ret(fdr)); S.append(Spacer(1,4))
#     tw_u=fdr.get("tw_users",0); lw_u=fdr.get("lw_users",0)
#     tw_pf=fdr.get("tw_platform_fc",0); lw_pf=fdr.get("lw_platform_fc",0)
#     tw_r=tw_u/tw_pf*100 if tw_pf>0 else 0; lw_r=lw_u/lw_pf*100 if lw_pf>0 else 0
#     tv=[fdr.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     lv=[fdr.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     def _fr(v): return f"{v:.1f}%" if not (v is None or (isinstance(v,float) and np.isnan(v))) else "-"
#     rh=["周次","活动用户数","平台首充\n人数","活动用户\n占比","D1留存","D2留存","D3留存","D4留存","D5留存","D6留存","D7留存"]
#     rrows=[
#         [cell(f"本周 {THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}",True,C_BLUE),
#          cell(f"{tw_u:,}",False,C_DARK,TA_RIGHT),cell(f"{tw_pf:,}",False,C_GRAY,TA_RIGHT),
#          cell(f"{tw_r:.1f}%",False,C_PURPLE,TA_RIGHT),
#         ]+[cell(_fr(v),False,C_GREEN if not pd.isna(v) and not pd.isna(lval) and v>=lval else(C_RED if not pd.isna(v) and not pd.isna(lval) and v<lval else C_DARK),TA_RIGHT) for v,lval in zip(tv,lv)],
#         [cell(f"上周 {LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}",True,C_GRAY),
#          cell(f"{lw_u:,}",False,C_GRAY,TA_RIGHT),cell(f"{lw_pf:,}",False,C_GRAY,TA_RIGHT),
#          cell(f"{lw_r:.1f}%",False,C_GRAY,TA_RIGHT),
#         ]+[cell(_fr(v),False,C_GRAY,TA_RIGHT) for v in lv],
#     ]
#     S.append(dtable(rh,rrows,[full*.16,full*.09,full*.09,full*.08]+[full*.084]*7,fsize=7,zebra=False)); S.append(Spacer(1,4))
#     tw_d1=fdr.get("tw_1日",np.nan); lw_d1=fdr.get("lw_1日",np.nan); ins=[]
#     if not np.isnan(tw_d1) and not np.isnan(lw_d1):
#         d=tw_d1-lw_d1; ins.append(f"首充活动用户次日留存{tw_d1:.1f}%（上周{lw_d1:.1f}%，{d:+.1f}pp），{'留存改善' if d>0 else '留存下降，建议优化次日触达策略'}。")
#     ins.append(f"本周活动用户{tw_u:,}人，占平台首充{tw_r:.1f}%（上周{lw_u:,}/{lw_r:.1f}%）；人均赠送$3.89，属高价值用户来源。")
#     ins.append("建议：对D1/D2留存用户设置阶梯式再充值道具激励；对D3后流失用户做专项召回（24h内触达效果最佳）。")
#     S.append(insight_box(ins, clr=C_AMBER))
#     return S


# def build_risk(hr, c207_dt, c207, sp, top20g, rg, top500, dt_full):
#     full = PW - 2*MARGIN; S = [sec_title("七、用户游戏风险专项分析", clr=colors.HexColor("#7c2d12"))]
#     tot_tx=top500["提款金额"].sum(); tot_win=top500["公司输赢"].sum(); wns=(top500["公司输赢"]<0).sum()
#     S.append(sub_title("Top500提款用户总览"))
#     row_k=[]
#     for label,val,sub in [("Top500提款总额",f"${tot_tx/10000:.1f}万",""),
#                            ("平台净赔付",f"${abs(tot_win)/10000:.1f}万","平台向该群体净赔"),
#                            ("赢家比例",f"{wns/500*100:.1f}%",f"{wns}赢/{500-wns}输"),
#                            ("高风险用户",f"{len(hr)}人","公司净输>$10,000")]:
#         fw=full/4-4
#         row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],
#                            colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
#     t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP")
#     t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
#     S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
#     hl=abs(hr["公司输赢"].sum())/10000 if len(hr)>0 else 0
#     S.append(insight_box([
#         f"Top500提款用户中赢家{wns}人（{wns/500*100:.1f}%），平台净赔付${abs(tot_win)/10000:.1f}万。",
#         f"{len(hr)}名超级赢家（公司净输>$10,000）合计导致平台净输${hl:.1f}万，占整体净赔付{hl/abs(tot_win)*10000*100 if tot_win!=0 else 0:.1f}%。",
#     ]))
#     if len(hr)>0:
#         S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
#         h=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注均额","主玩厂商","主玩游戏","风险标签"]
#         rows=[]
#         for _,r in hr.iterrows():
#             ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0; avg=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
#             tags=[]
#             if r["充值金额"]<5000: tags.append("低充高提")
#             if ratio>50: tags.append("超高投充")
#             if avg>200: tags.append("高额单注")
#             rows.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),
#                          cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),
#                          cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),
#                          cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),cell(f"${avg:.0f}",False,C_RED if avg>200 else C_DARK,TA_RIGHT),
#                          cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),
#                          cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
#         S.append(dtable(h,rows,[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]],fsize=6.5)); S.append(Spacer(1,6))
#     if len(c207)>0:
#         S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警"))
#         S.append(P(f"集群账户数：{len(c207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
#         ch=["账户ID","提款金额","充值金额","公司输赢","特征"]; cr=[]
#         for _,r in c207_dt.head(10).iterrows():
#             cr.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell("充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}")])
#         if len(c207_dt)>10: cr.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
#         S.append(dtable(ch,cr,[full*x for x in [0.22,0.18,0.18,0.18,0.24]])); S.append(Spacer(1,6))
#         S.append(insight_box([
#             f"{len(c207)}个账户充值$547-548，均获活动奖励$270.92，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。",
#             "建议：①冻结账户提款；②审查注册IP/设备指纹；③排查$270.92奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。",
#         ], clr=C_AMBER))
#     S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
#     th=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; tr=[]
#     for i,(_,r) in enumerate(dt_full.head(20).iterrows()):
#         win=r["公司输赢"]<0
#         tr.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),
#                    cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),
#                    cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),
#                    cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),
#                    cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
#     S.append(dtable(th,tr,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
#     pg,ma=load_pref()
#     S.append(sub_title("Top500提款用户游戏偏好")); S.append(chart_pref(pg,ma)); S.append(Spacer(1,4))
#     S.append(sub_title("▶ 高危游戏专项分析")); S.append(chart_risk_games(rg)); S.append(Spacer(1,4))
#     gh=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; gr=[]
#     for _,r in rg.head(12).iterrows():
#         pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rc2=C_RED if rl in("极高","高") else C_AMBER
#         gr.append([cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),
#                    cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),
#                    cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),
#                    cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),
#                    cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rc2)])
#     S.append(dtable(gh,gr,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5)); S.append(Spacer(1,6))
#     S.append(insight_box(["PP Auto-Roulette 1批量套利模式持续，建议暂停该桌并复核奖励规则。",
#                           "Fortune Garuda 500（Tada）是Top500提款用户投注规模最大游戏，建议对其RTP参数做紧急复审。"], clr=C_RED))
#     return S


# def build_conclusion(K, agents, dep_t, wdr_t):
#     full = PW - 2*MARGIN; S = [sec_title("八、总结与行动建议")]
#     cr_d=K["tw_充提差比"]-K["lw_充提差比"]; rd=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
#     S.append(sub_title("▶ 本周亮点")); bg_g=colors.HexColor("#f0fdf4")
#     for h in [
#         _render("充值{v:.1f}万（{p:+.1f}%），充提差{d:.1f}万，充提差率{cr:.2f}%（{dpp:+.2f}pp）。",
#                 v=K["tw_充值金额"]/10000,p=K["pct_充值金额"],d=K["tw_充提差"]/10000,cr=K["tw_充提差比"],dpp=cr_d),
#         _render("注册人数{r:,}（{rp:+.1f}%），首充人数{fc:,}（{fcp:+.1f}%），拉新规模扩大。",
#                 r=int(K["tw_注册人数"]),rp=K["pct_注册人数"],fc=int(K["tw_首充人数"]),fcp=K["pct_首充人数"]),
#         "A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。",
#     ]:
#         S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_g),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
#     S.append(Spacer(1,8)); S.append(sub_title("▶ 风险预警")); bg_r=colors.HexColor("#fff5f5")
#     for r in [
#         "PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。",
#         _render("首充ARPPU${v:.2f}（{p:+.1f}%），首充次日留存{nd:.1f}%（{dp:+.1f}pp），新用户质量需关注。",
#                 v=K["tw_首充Arppu"],p=K["pct_首充Arppu"],nd=K["tw_首充次日复充率"],dp=rd),
#         "YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。",
#     ]:
#         S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_r),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
#     S.append(Spacer(1,8)); S.append(sub_title("▶ 行动建议"))
#     actions=[
#         ("[紧急]","处置PP Auto-Roulette 1批量套利","33个207开头账户合计提款$9.4万，平台净赔$6.7万。冻结账户，审查IP/设备指纹，修复$270.92奖励触发漏洞，实施赢额上限。"),
#         ("[本周]","处置高风险提款用户","账户49903866：提款$39,636，公司输赢-$52,134；账户206628722：充值$10,229提款$24,855，高度异常。建议立即人工审核。"),
#         ("[本周]","优化首充质量与次日激活",_render("首充ARPPU${v:.2f}（{p:+.1f}%），次日留存{nd:.1f}%（{dp:+.1f}pp）。建议注册后1h/24h内推送首充引导，落地页突出$10+档。",v=K["tw_首充Arppu"],p=K["pct_首充Arppu"],nd=K["tw_首充次日复充率"],dp=rd)),
#     ]
#     pmap={"[紧急]":C_RED,"[本周]":C_AMBER}
#     rows=[[cell(pri,True,pmap.get(pri[:4],C_GRAY),TA_CENTER),cell(title,True,C_DARK),cell(desc,False,C_GRAY)] for pri,title,desc in actions]
#     S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows,[full*x for x in [0.1,0.22,0.68]]))
#     return S


# def header_footer(c, doc):
#     c.saveState(); w, h = A4
#     c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
#     c.setFillColor(colors.white); c.setFont(FNB,10)
#     c.drawString(MARGIN, h-17, f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
#     c.setFont(FN,8); c.drawRightString(w-MARGIN, h-17, f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
#     c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
#     c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN, 5, f"第 {doc.page} 页")
#     c.restoreState()


# def main(data_root=None, output_dir=None):
#     root   = Path(data_root)   if data_root   else DATA_ROOT
#     outdir = Path(output_dir)  if output_dir  else OUTPUT_DIR
#     outdir.mkdir(parents=True, exist_ok=True)
#     print("📊 MX 周报 PDF v7.3 生成中...")
#     resolve_files(root); setup()
#     print("  ▶ 加载数据...")
#     K, trend         = load_platform()
#     dash_ret         = load_dashboard_retention()
#     weekly_ret       = load_weekly_retention()
#     agents           = load_agents()
#     agent_ret, ends  = load_agent_ret()
#     tw_v, lw_v       = load_vip()
#     dep_t, wdr_t, dc_tw, dt_top = load_top_users()
#     top30            = load_games()
#     mfr              = load_mfr()
#     mfr_delta        = load_mfr_game_delta()
#     act_df, tot_gift = load_activities()
#     tool_df          = load_tool_data()
#     fdr              = load_first_dep_ret()

#     dt_raw = pd.read_csv(FILES["dt_tw"]); dc_raw = pd.read_csv(FILES["dc_tw"])
#     for df in [dt_raw, dc_raw]:
#         for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
#             if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
#     hr, c207_dt, c207, sp, top20g, rg, top500 = load_risk(dt_raw, dc_raw)

#     dpf = pd.read_csv(FILES["pref_tw"]); dpf["阶段汇总"] = dpf["阶段汇总"].apply(clean)
#     tg = (dpf[dpf["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
#           .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]]
#           .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
#     dt_top = dt_top.merge(tg, on="账户ID", how="left")

#     print("  ▶ 导出风控Excel...")
#     excel_out = outdir / f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
#     with pd.ExcelWriter(str(excel_out), engine="openpyxl") as xw:
#         cols_hr = ["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
#         hr[[c for c in cols_hr if c in hr.columns]].to_excel(xw, sheet_name="高风险用户", index=False)
#         if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(xw, sheet_name="PP轮盘批量账号", index=False)
#         if len(sp)>0:     sp[["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]].to_excel(xw, sheet_name="特殊异常用户", index=False)
#         tool_df.to_excel(xw, sheet_name="道具使用情况", index=False)
#     print(f"✅ 风控Excel: {excel_out}")

#     print("  ▶ 构建PDF章节...")
#     out = outdir / f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
#     doc = SimpleDocTemplate(str(out), pagesize=A4, leftMargin=MARGIN, rightMargin=MARGIN,
#                             topMargin=MARGIN+22, bottomMargin=MARGIN+10,
#                             title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
#     story = [Spacer(1,3*cm),
#              P("MX 平台数据周报", 30, True, C_DARK, TA_CENTER), Spacer(1,.5*cm),
#              P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),
#              Spacer(1,.2*cm),
#              P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),
#              Spacer(1,.5*cm), HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"), PageBreak()]
#     story += build_overview(K, trend, weekly_ret, dash_ret);              story.append(PageBreak())
#     story += build_agents(agents, agent_ret, ends);                       story.append(PageBreak())
#     story += build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_top);       story.append(PageBreak())
#     story += build_games(mfr, top30, mfr_delta);                          story.append(PageBreak())
#     story += build_activities(act_df, tot_gift);                          story.append(PageBreak())
#     story += build_tools(tool_df, fdr);                                   story.append(PageBreak())
#     story += build_risk(hr, c207_dt, c207, sp, top20g, rg, top500, dt_top); story.append(PageBreak())
#     story += build_conclusion(K, agents, dep_t, wdr_t)
#     print("  ▶ 渲染PDF...")
#     doc.build(story, onFirstPage=header_footer, onLaterPages=header_footer)
#     size = out.stat().st_size / 1024
#     print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
#     return str(out), str(excel_out)


# if __name__ == "__main__":
#     if sys.platform != "win32":
#         main(data_root="/mnt/user-data/uploads", output_dir="/mnt/user-data/outputs")
#     else:
#         main(data_root=str(DATA_ROOT), output_dir=str(OUTPUT_DIR))

📊 MX 周报 PDF v7.3 生成中...
  ▶ 数据目录：D:\周报更新版\MX
     ✅ [platform       ] 平台报表_USD_近4周.xlsx
     ✅ [daily          ] 日报-大盘日报_USD_近4周.xlsx
     ✅ [retention      ] 整体 首充留存（近7天）_近28天.csv
     ✅ [agent_plat     ] 平台报表-总代_USD_近21天.xlsx
     ✅ [agent_promo    ] 推广报表-总代_USD_近21天.xlsx
     ✅ [agent_ret      ] 首充充值留存_20260424_20260521.csv
     ✅ [vip            ] VIP报表_USD_近14天.xlsx
     ✅ [dt_tw          ] top提款用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dt_lw          ] top提款用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [dc_tw          ] 头部充值用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dc_lw          ] 头部充值用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [pref_tw        ] 本周top500提款用户游戏偏好_全量数据_20260515_20260521.csv
     ✅ [pref_lw        ] 上周top500提款用户游戏偏好_全量数据_20260508_20260514.csv
     ✅ [mfr            ] 厂商投注数据_全量数据_20260508_20260521.csv
     ✅ [game_tw        ] 游戏报表-详情_USD_本周.xlsx
     ✅ [game_lw        ] 游戏报表-详情_USD_上周.xlsx
     ✅ [gift 

In [42]:
# """
# MX 平台数据周报生成脚本 v7.1
# ======================================================
# v7.1 修复与优化：
#   1. 二章标题去掉括号，加回总代0
#   2. 头部充值/提款分层表加注释（本周TopN vs 上周TopN独立排名）
#   3. 厂商表"投注 环比" -> "投注额 环比"
#   4. 新增厂商投注额环比驱动游戏分析
#   5. Top30游戏厂商简写（RG/PP/PG/PT/自研）
#   6. 注释掉存款来源道具分析（保留代码）
#   7. 活动列全量，日均赠送人数（本周/上周/环比），本周人均赠送
#   8. 道具专题分析 + 首次充值活动留存分析
#   修复：
#   - 厂商文件列名 '盈利率' -> '盈亏率' rename
#   - 首充留存列名 '第1日' 而非 '1日'
#   - load_first_dep_retention 用 numpy array 避免 pandas index 对齐错误
#   - load_activities 使用新文件（各赠送活动_全量数据...含赠送人数.1列）
# """

# # ════════════════════════════════════════════════════════════════
# # 第一区：周期配置  ← 每周只改这里
# # ════════════════════════════════════════════════════════════════
# from pathlib import Path
# import sys

# if sys.platform != "win32":
#     DATA_ROOT  = Path("/mnt/user-data/uploads")
#     OUTPUT_DIR = Path("/mnt/user-data/outputs")
# else:
#     DATA_ROOT  = Path(r"D:\周报更新版\MX")
#     OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")

# THIS_WEEK  = ("20260515", "20260521")
# LAST_WEEK  = ("20260508", "20260514")
# REPORT_END = THIS_WEEK[1]

# # ════════════════════════════════════════════════════════════════
# # 第二区：文案配置
# # ════════════════════════════════════════════════════════════════
# CR_HIGH = 17
# CR_LOW  = 5

# OVERVIEW_INSIGHTS = [
#     ("充值{tw_充值金额:.1f}万（{pct_充值金额:+.1f}%），公司输赢{tw_公司输赢:.1f}万（{pct_公司输赢:+.1f}%）；推广消耗{pct_真实消耗:+.1f}%。"),
#     ("充提差率{tw_充提差比:.2f}%（上周{lw_充提差比:.2f}%，{delta_充提差比:+.2f}pp）；盈亏率{tw_盈亏率:.3f}%（上周{lw_盈亏率:.3f}%）。"),
#     ("首充人数{tw_首充人数:,}（{pct_首充人数:+.1f}%），首充转化率{tw_首充转化率:.1f}%，首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%）。"),
#     ("首充次日充值留存{tw_首充次日复充率:.1f}%（上周{lw_首充次日复充率:.1f}%，{delta_首充次日复充率:+.1f}pp）。"),
# ]
# RETENTION_NOTE = [
#     ("本周首充次日留存{tw_d1:.1f}%，较上周{delta_d1:+.1f}pp；2日留存{tw_d2:.1f}%。"),
#     ("注：本周7日留存因数据截止日期无法完整计算，以上周7日留存{lw_d7:.1f}%作为近期参考基准。"),
# ]
# AGENT_INSIGHTS = [
#     ("A8_DSP_谷歌原生包充提差率{dsp_cr:.1f}%，注册+{dsp_reg:.1f}%，体量第三且指标双优。"),
#     "苹果包总代注册+27.1%，充值体量第二，为本周注册增量最大渠道之一。",
#     "YB_FB_PWA_0注册暴跌-79%至1,315人，需持续监控是否恢复。",
# ]
# VIP_INSIGHTS = [
#     ("VIP9-11高价值层合计贡献充值{high_vip_pct:.1f}%，高端用户付费意愿持续强劲。"),
#     ("VIP1基础层活跃{tw_vip1_active:,}人（上周{lw_vip1_active:,}），是拉新政策直接反映。"),
# ]
# VIP_RET_INSIGHTS = [
#     ("充值→活跃次日留存：VIP9达{act_v9:.1f}%，VIP10达{act_v10:.1f}%，高VIP活跃粘性强。"),
#     ("充值→充值次日留存：VIP10达{chg_v10_tw:.1f}%（上周{chg_v10_lw:.1f}%），高VIP复充意愿强。"),
# ]
# TOP_DEPOSIT_INSIGHTS = [
#     ("Top10充值用户人均${top10_avg:,.0f}（{top10_avg_chg:+.1f}%），充提差率{top10_cr:.1f}%。"),
#     "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。",
# ]
# TOP_WITHDRAW_INSIGHTS = [
#     ("剔除Top100提款用户后，大盘充提差率影响+{top100_impact:.2f}pp，头部提款用户对充提差率有明显拖累。"),
#     ("Top500提款用户活动奖励占资金来源仅{top500_act_pct:.1f}%，主要靠真实赢钱后提款，非活动套利。"),
# ]
# MFR_INSIGHTS = [
#     ("Tada投注份额{tada_share:.1f}%（上周{tada_share_lw:.1f}%），Fortune系稳固主导；环比{tada_chg:+.1f}%。"),
#     ("Evolution盈亏率{evol_pl:.2f}%（上周{evol_pl_lw:.2f}%），已转正，较上周大幅改善。"),
#     ("3Oaks盈亏率{oaks_pl:.2f}%，PlayTech{ptech_pl:.2f}%，均高于平台均值，可适当扩大曝光权重。"),
# ]
# ACTIVITY_INSIGHTS = ["各类赠送活动全量列出（共{total_acts}个），合计本周日均赠送{daily_gift:.1f}万USD，全量活动详见表格。"]
# RISK_SCATTER_INSIGHTS = [
#     ("Top500提款用户中赢家{winner_cnt}人（{winner_pct:.1f}%），平台净赔付${total_loss:.1f}万。"),
#     ("{high_risk_cnt}名超级赢家（公司净输>$10,000）合计导致平台净输${high_risk_loss:.1f}万，占整体净赔付{high_risk_pct:.1f}%。"),
# ]
# PP_ROULETTE_INSIGHTS = [
#     ("{cluster_cnt}个账户充值$547-548，均获活动奖励$270.92，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。"),
#     ("建议：①冻结账户提款；②审查注册IP/设备指纹；③排查$270.92奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。"),
# ]
# RISK_GAME_INSIGHTS = [
#     "PP Auto-Roulette 1批量套利模式持续，建议暂停该桌并复核奖励规则。",
#     ("Fortune Garuda 500（Tada）是Top500提款用户投注规模最大游戏，建议对其RTP参数做紧急复审。"),
# ]
# HIGHLIGHTS = [
#     ("充值{tw_充值金额:.1f}万（{pct_充值金额:+.1f}%），充提差{tw_充提差:.1f}万（{pct_充提差:+.1f}%），充提差率{tw_充提差比:.2f}%（{delta_充提差比:+.2f}pp）。"),
#     ("注册人数{tw_注册人数:,}（{pct_注册人数:+.1f}%），首充人数{tw_首充人数:,}（{pct_首充人数:+.1f}%），拉新规模扩大。"),
#     "A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。",
# ]
# RISKS = [
#     ("PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。"),
#     ("首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%），首充次日留存{tw_首充次日复充率:.1f}%（{delta_首充次日复充率:+.1f}pp），新用户质量需关注。"),
#     "YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。",
# ]
# ACTIONS = [
#     ("[紧急]", "处置PP Auto-Roulette 1批量套利",
#      "33个207开头账户合计提款$9.4万，平台净赔$6.7万。冻结账户，审查IP/设备指纹，修复$270.92奖励触发漏洞，对Auto-Roulette 1实施赢额上限。"),
#     ("[本周]", "处置高风险提款用户",
#      "账户49903866：提款$39,636，公司输赢-$52,134；账户206628722：充值$10,229提款$24,855，高度异常。建议立即人工审核。"),
#     ("[本周]", "优化首充质量与次日激活",
#      ("首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%），首充次日留存{tw_首充次日复充率:.1f}%（{delta_首充次日复充率:+.1f}pp）。建议注册后1h/24h内推送首充引导，落地页突出$10+档。")),
#     # ("[下周]", "扩大A8_DSP、A8_GG渠道预算",
#     #  "A8_DSP充提差率17.6%（+1.3pp），一级首充成本$17；A8_GG充提差率18.9%（+0.3pp）。建议预算增加15-20%。"),
#     # ("[下周]", "暂停YB_FB_PWA_0", "注册量暴跌-79%，仅1,315人，建议暂停投放并排查渠道异常。"),
# ]

# def render(t, **kw):
#     try: return t.format(**kw)
#     except: return t
# def render_list(ts, **kw): return [render(t, **kw) for t in ts]

# # ════════════════════════════════════════════════════════════════
# # 第三区：依赖库 & 数据加载
# # ════════════════════════════════════════════════════════════════
# import io, warnings
# from datetime import datetime, timedelta
# import numpy as np
# import pandas as pd
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# import matplotlib.ticker as mticker
# import matplotlib.gridspec as gridspec
# import matplotlib.patches as mpatches
# warnings.filterwarnings("ignore")

# from reportlab.lib.pagesizes import A4
# from reportlab.lib import colors
# from reportlab.lib.units import cm
# from reportlab.lib.styles import ParagraphStyle
# from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
# from reportlab.platypus import (
#     SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
#     Image, PageBreak, HRFlowable, KeepTogether
# )
# from reportlab.pdfbase import pdfmetrics
# from reportlab.pdfbase.ttfonts import TTFont

# FN, FNB = "WQY", "WQYB"
# FILES = {}

# # 厂商简写映射
# MFR_SHORT = {
#     "Rectangle":      "RG",
#     "Pragmatic Play": "PP",
#     "PG Soft":        "PG",
#     "PlayTech":       "PT",
#     "Originals":      "自研",
# }
# def shorten_mfr(name):
#     if not isinstance(name, str): return str(name)
#     for full, short in MFR_SHORT.items():
#         if full in name: return short
#     return name

# def retention_end_date(week_start, lag_days):
#     end_dt  = datetime.strptime(REPORT_END, "%Y%m%d")
#     last_dt = end_dt - timedelta(days=lag_days)
#     last    = last_dt.strftime("%Y%m%d")
#     return last if last >= week_start else None

# def find_chinese_font():
#     import platform
#     sys_name = platform.system()
#     if sys_name == "Windows":
#         candidates = [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\msyhbd.ttc",
#                       r"C:\Windows\Fonts\simsun.ttc", r"C:\Windows\Fonts\simhei.ttf"]
#     elif sys_name == "Darwin":
#         candidates = ["/System/Library/Fonts/PingFang.ttc"]
#     else:
#         candidates = ["/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
#                       "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
#                       "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]
#     for p in candidates:
#         if Path(p).exists(): return p
#     try:
#         from matplotlib import font_manager as fm
#         for f in fm.fontManager.ttflist:
#             if any(k in f.name for k in ["Heiti","Hei","YaHei","SimSun","SimHei","WenQuanYi","Noto Sans CJK"]):
#                 if Path(f.fname).exists(): return f.fname
#     except: pass
#     raise FileNotFoundError("未找到中文字体！")

# def setup():
#     global FONT_PATH
#     FONT_PATH = find_chinese_font()
#     print(f"  ▶ 使用字体：{FONT_PATH}")
#     is_ttc = FONT_PATH.lower().endswith(".ttc")
#     if is_ttc:
#         pdfmetrics.registerFont(TTFont(FN, FONT_PATH, subfontIndex=0))
#         try: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1))
#         except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0))
#     else:
#         pdfmetrics.registerFont(TTFont(FN, FONT_PATH))
#         pdfmetrics.registerFont(TTFont(FNB, FONT_PATH))
#     from matplotlib import font_manager as fm
#     fm.fontManager.addfont(FONT_PATH)
#     added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
#     plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
#                          "axes.unicode_minus": False, "font.size": 8.5})

# def _resolve_files(root):
#     global FILES
#     kw_map = {
#         "platform":       "平台报表_USD_近4周",
#         "daily":          "日报-大盘日报_USD_近4周",
#         "retention":      ["整体","首充留存"],  # 兼容Windows空格/Linux下划线文件名
#         "agent_plat":     "平台报表-总代_USD_近21天",
#         "agent_promo":    "推广报表-总代_USD_近21天",
#         "agent_ret":      "首充充值留存",
#         "vip":            "VIP报表_USD",
#         "dt_tw":          f"top提款用户_全量数据_{THIS_WEEK[0]}",
#         "dt_lw":          f"top提款用户_全量数据_{LAST_WEEK[0]}",
#         "dc_tw":          f"头部充值用户_全量数据_{THIS_WEEK[0]}",
#         "dc_lw":          f"头部充值用户_全量数据_{LAST_WEEK[0]}",
#         "pref_tw":        f"本周top500提款用户游戏偏好_全量数据_{THIS_WEEK[0]}",
#         "pref_lw":        f"上周top500提款用户游戏偏好_全量数据_{LAST_WEEK[0]}",
#         "mfr":            "厂商投注数据_全量数据",
#         "game_tw":        "游戏报表-详情_USD_本周",
#         "game_lw":        "游戏报表-详情_USD_上周",
#         "gift":           f"各赠送活动_全量数据_{THIS_WEEK[0]}",
#         "tool_map":       "道具对应活动",
#         "vip_ret_chg":    "VIP充值-充值_近28天",
#         "vip_ret_act":    "VIP充值-活跃_近28天",
#         "tool_tw":        f"本周道具使用情况",
#         "tool_lw":        f"上周道具使用情况",
#         "first_dep_ret":  f"首次充值活动用户充值留存情况",
#         # "deposit_src": f"存款来源_全量数据_{THIS_WEEK[0]}",  # ← 暂时注释
#     }
#     print(f"  ▶ 数据目录：{root}")
#     fail = 0
#     for key, kw in kw_map.items():
#         kws = kw if isinstance(kw, list) else [kw]
#         hits = [h for h in root.rglob("*")
#                 if h.is_file() and not h.name.startswith("~$") and all(k in h.name for k in kws)]
#         if not hits:
#             print(f"     ❌ [{key:15s}] 找不到含 '{kws}' 的文件"); fail += 1
#         else:
#             p = max(hits, key=lambda h: h.stat().st_mtime)
#             FILES[key] = p
#             print(f"     ✅ [{key:15s}] {p.parent.name}/{p.name}")
#     if fail:
#         raise FileNotFoundError(f"\n共 {fail} 个文件未找到，请检查 DATA_ROOT = {root}")
#     print(f"  ▶ 全部 {len(kw_map)} 个文件匹配成功\n")

# # ── 颜色 ─────────────────────────────────────────
# C_BLUE   = colors.HexColor("#1d4ed8")
# C_BLUE2  = colors.HexColor("#3b82f6")
# C_GREEN  = colors.HexColor("#059669")
# C_RED    = colors.HexColor("#dc2626")
# C_AMBER  = colors.HexColor("#d97706")
# C_PURPLE = colors.HexColor("#7c3aed")
# C_GRAY   = colors.HexColor("#64748b")
# C_LGRAY  = colors.HexColor("#f1f5f9")
# C_WHITE  = colors.white
# C_DARK   = colors.HexColor("#1e293b")
# C_BORDER = colors.HexColor("#cbd5e1")
# C_ROW    = colors.HexColor("#f8fafc")
# C_TEAL   = colors.HexColor("#0f766e")

# PW, PH = A4
# MARGIN  = 1.8*cm
# CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
#                 "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
# TRUNCATE_RULES = {
#     "首充次日复充率":1,"首充次日复投率":1,"首充当日复充率":0,
#     "首充7日复充率":6,"首充30日复充率":999,"首充2日复充率":1,"首充3日复充率":2,
# }

# # ── 工具 ──────────────────────────────────────────
# def clean(s):
#     try: return float(str(s).replace(",","").replace("%","").strip())
#     except: return np.nan

# def to_num(df, cols=None):
#     if cols is None:
#         cols = [c for c in df.columns if c not in ("日期","总代.名称","name_总代","总代.ID")]
#     for c in cols:
#         if c in df.columns: df[c] = df[c].apply(clean)
#     return df

# def pct(n, o): return (n-o)/abs(o)*100 if o and o!=0 else 0.0
# def pct_vec(ns, os): return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
# def trunc_mean(s, drop=0):
#     if drop >= len(s): return np.nan
#     v = s.iloc[:len(s)-drop] if drop > 0 else s
#     nz = v[v > 0]; return nz.mean() if len(nz) > 0 else v.mean()
# def fig_img(fig, w=16, h=7):
#     buf = io.BytesIO()
#     fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
#     buf.seek(0); plt.close(fig)
#     return Image(buf, width=w*cm, height=h*cm)

# # ── 样式 ──────────────────────────────────────────
# def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
#     st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
#                         textColor=clr, alignment=align, leading=sz*1.38,
#                         spaceAfter=0, spaceBefore=0)
#     return Paragraph(str(txt), st)

# def sec_title(text, clr=C_BLUE):
#     return KeepTogether([Spacer(1,6),
#         Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
#               style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
#                                 ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
#                                 ("LEFTPADDING",(0,0),(-1,-1),14)])),
#         Spacer(1,8)])

# def sub_title(text):
#     return KeepTogether([Spacer(1,5),
#                          HRFlowable(width="100%",thickness=1.5,color=C_BLUE2),
#                          Spacer(1,3), P(f"■  {text}",10,True,C_DARK), Spacer(1,6)])

# def insight_box(lines, clr=C_GREEN):
#     bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
#            colors.HexColor("#fffbeb") if clr==C_AMBER else
#            colors.HexColor("#fef2f2"))
#     rows = [[P(f"◆ {l}",8.5,False,C_DARK)] for l in lines]
#     return KeepTogether([
#         Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
#             ("BACKGROUND",(0,0),(-1,-1),bkg),("BOX",(0,0),(-1,-1),1,clr),
#             ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
#             ("LEFTPADDING",(0,0),(-1,-1),10),("RIGHTPADDING",(0,0),(-1,-1),10),
#             ("TOPPADDING",(0,0),(-1,-1),4),("BOTTOMPADDING",(0,0),(-1,-1),4),
#         ])), Spacer(1,8)])

# def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8):
#     full = PW-2*MARGIN
#     if not widths: widths = [full/len(headers)]*len(headers)
#     if hdr_clr is None: hdr_clr = C_DARK
#     st = TableStyle([
#         ("BACKGROUND",(0,0),(-1,0),hdr_clr),("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
#         ("FONTNAME",(0,0),(-1,0),FNB),("FONTNAME",(0,1),(-1,-1),FN),
#         ("FONTSIZE",(0,0),(-1,-1),fsize),
#         ("TOPPADDING",(0,0),(-1,-1),2.5),("BOTTOMPADDING",(0,0),(-1,-1),2.5),
#         ("LEFTPADDING",(0,0),(-1,-1),3),("RIGHTPADDING",(0,0),(-1,-1),3),
#         ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
#     ])
#     if zebra:
#         for i in range(1,len(rows)+1,2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
#     hrow = [P(h,fsize,True,C_WHITE,TA_CENTER) for h in headers]
#     body = []
#     for row in rows:
#         body.append([
#             P(str(c[0]),fsize,c[1] if len(c)>1 else False,
#               c[2] if len(c)>2 else colors.black,
#               c[3] if len(c)>3 else TA_LEFT)
#             if isinstance(c,(list,tuple)) else P(str(c),fsize)
#             for c in row
#         ])
#     return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

# def cell(t,bold=False,clr=colors.black,align=TA_LEFT): return (t,bold,clr,align)
# def rc(v,gup=True,d=1,good_up=None):
#     if good_up is not None: gup=good_up
#     s="+"; c=C_GREEN if (v>0)==gup else C_RED
#     return cell(f"{'+' if v>=0 else ''}{v:.{d}f}%",False,c,TA_RIGHT)
# def gclr(v,t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)

# def kpi_card4(items,cols=4):
#     full=(PW-2*MARGIN)/cols-4
#     rows,row=[],[]
#     for label,tv,lv,chg,gup in items:
#         pclr=C_GREEN if chg>=0 else C_RED
#         inner=Table([[P(label,7.5,False,C_GRAY)],[P(str(tv),14,True,C_DARK)],
#                      [P(f"上周：{lv}",7.5,False,C_GRAY)],
#                      [P(f"{'+' if chg>=0 else ''}{chg:.1f}%",8,True,pclr)]],
#                     colWidths=[full],style=TableStyle([
#                         ("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),
#                         ("LEFTPADDING",(0,0),(-1,-1),8),
#                         ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
#         row.append(inner)
#         if len(row)==cols: rows.append(row); row=[]
#     if row:
#         while len(row)<cols: row.append(Spacer(full,1))
#         rows.append(row)
#     t=Table(rows,colWidths=[(PW-2*MARGIN)/cols]*cols,hAlign="LEFT",vAlign="TOP")
#     t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
#                             ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
#                             ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
#     return t

# # ════════════════════════════════════════════
# # 数据加载
# # ════════════════════════════════════════════

# def load_platform():
#     df = pd.read_excel(FILES["platform"])
#     df["日期"] = df["日期"].astype(str)
#     df = df.sort_values("日期"); df = to_num(df)
#     tw = df[df["日期"].between(*THIS_WEEK)]
#     lw = df[df["日期"].between(*LAST_WEEK)]
#     dd = pd.read_excel(FILES["daily"])
#     dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
#     tw_d = dd[dd["日期"].between(*THIS_WEEK)]
#     lw_d = dd[dd["日期"].between(*LAST_WEEK)]
#     K = {}
#     for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
#               "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
#         if c not in df.columns: continue
#         K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
#         K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
#     for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
#         if c not in df.columns: continue
#         K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
#         K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
#     for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
#               "首充次日复投率","活跃用户付费率","总赠送充值比"]:
#         if c not in df.columns: continue
#         drop = TRUNCATE_RULES.get(c, 0)
#         K[f"tw_{c}"] = trunc_mean(tw[c], drop)
#         K[f"lw_{c}"] = lw[c].mean()
#         K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
#     K["tw_真实消耗"] = tw_d["真实消耗"].iloc[:-1].sum() if len(tw_d)>1 else tw_d["真实消耗"].sum()
#     K["lw_真实消耗"] = lw_d["真实消耗"].sum()
#     K["pct_真实消耗"] = pct(K["tw_真实消耗"], K["lw_真实消耗"])
#     K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
#     K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
#     K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])
#     last14 = df.tail(14)
#     trend = {
#         "dates":    [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
#         "充值":     last14["充值金额"].tolist(), "提现":    last14["提现金额"].tolist(),
#         "充提差比": last14["充提差比"].tolist(), "公司输赢":last14["公司输赢"].tolist(),
#         "首充":     last14["首充人数"].tolist(), "注册":    last14["注册人数"].tolist(),
#     }
#     return K, trend

# def load_weekly_retention():
#     """首充留存列名为'第1日'/'第2日'..."""
#     df = pd.read_csv(FILES["retention"])
#     daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
#     daily["date_str"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
#     daily_r = daily[daily["指标"]=="留存率"].copy()
#     daily_u = daily[daily["指标"]=="留存人数"].copy()
#     for c in ["第1日","第2日","第3日","第7日"]:
#         daily_r[c] = pd.to_numeric(daily_r[c].astype(str).str.replace("%","").str.strip(), errors="coerce")
#     def _d(s): return datetime.strptime(s, "%Y%m%d")
#     def _fmt(d): return d.strftime("%Y-%m-%d")
#     def _lbl(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
#     tw_s=_d(THIS_WEEK[0]); tw_e=_d(THIS_WEEK[1])
#     lw_s=_d(LAST_WEEK[0]); lw_e=_d(LAST_WEEK[1])
#     w2_end=lw_s-timedelta(days=1); w2_start=w2_end-timedelta(days=6)
#     w1_end=w2_start-timedelta(days=1); w1_start=w1_end-timedelta(days=6)
#     weeks = [
#         (f"第1周\n{_lbl(w1_start,w1_end)}", _fmt(w1_start), _fmt(w1_end)),
#         (f"第2周\n{_lbl(w2_start,w2_end)}", _fmt(w2_start), _fmt(w2_end)),
#         (f"上周\n{_lbl(lw_s,lw_e)}",        _fmt(lw_s),     _fmt(lw_e)),
#         (f"本周\n{_lbl(tw_s,tw_e)}",        _fmt(tw_s),     _fmt(tw_e)),
#     ]
#     result = []
#     for wk,s,e in weeks:
#         mr = daily_r[(daily_r["date_str"]>=s)&(daily_r["date_str"]<=e)]
#         mu = daily_u[(daily_u["date_str"]>=s)&(daily_u["date_str"]<=e)]
#         users = mu["充值成功事件用户数"].sum()
#         row = {"week": wk, "users": users}
#         for col in ["第1日","第2日","第3日","第7日"]:
#             rs = mr[col].values; us = mu["充值成功事件用户数"].values
#             if len(rs)>0 and len(rs)==len(us):
#                 v = ~np.isnan(rs)
#                 row[col] = float(np.average(rs[v], weights=us[v])) if v.sum()>0 else np.nan
#             else:
#                 row[col] = float(np.nanmean(rs)) if len(rs)>0 else np.nan
#         result.append(row)
#     return result

# def load_agents():
#     df_p = pd.read_excel(FILES["agent_plat"])
#     df_r = pd.read_excel(FILES["agent_promo"])
#     df_p["日期"]=df_p["日期"].astype(str); df_r["日期"]=df_r["日期"].astype(str)
#     df_p=to_num(df_p); df_r=to_num(df_r)
#     # v7: 不过滤总代0，加回官方总代
#     tw_p=df_p[df_p["日期"].between(*THIS_WEEK)]; lw_p=df_p[df_p["日期"].between(*LAST_WEEK)]
#     tw_r=df_r[df_r["日期"].between(*THIS_WEEK)]; lw_r=df_r[df_r["日期"].between(*LAST_WEEK)]
#     sum_a=["充值金额","提现金额","充提差","首充金额","首充人数","注册人数","充值人数","投注金额","公司输赢","总赠送金额"]
#     def agg_p(d):
#         g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sum_a if c in d.columns}).reset_index()
#         for c in ["首充转化率","充提差比","首充次日充值留存","活跃用户付费率"]:
#             if c in d.columns:
#                 rm=d.groupby("总代.ID")[c].mean().rename(c); g=g.merge(rm,on="总代.ID",how="left")
#         g["充提差率"]=g["充提差"]/g["充值金额"]*100; return g
#     tw_pa=agg_p(tw_p); lw_pa=agg_p(lw_p)
#     sum_r=["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
#     def agg_r(d):
#         g=d.groupby("总代.ID").agg({c:"sum" for c in sum_r if c in d.columns}).reset_index()
#         g["注册成本"]=g["总消耗"]/g["注册人数"].replace(0,np.nan)
#         g["一级首充成本"]=g["总消耗"]/g["一级首充人数"].replace(0,np.nan)
#         return g
#     tw_ra=agg_r(tw_r); lw_ra=agg_r(lw_r)
#     lw_ra["lw_一级首充成本"]=lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
#     m=tw_pa.merge(lw_pa[["总代.ID","充值金额","注册人数","充提差率","首充次日充值留存"]].rename(
#         columns={"充值金额":"lw_充值","注册人数":"lw_注册","充提差率":"lw_充提差率","首充次日充值留存":"lw_次日留存"}),
#         on="总代.ID",how="left")
#     m=m.merge(tw_ra[["总代.ID","总消耗","注册成本","一级首充成本","一级首充人数"]],on="总代.ID",how="left")
#     m=m.merge(lw_ra[["总代.ID","lw_一级首充成本"]],on="总代.ID",how="left")
#     m["注册环比"]=pct_vec(m["注册人数"],m["lw_注册"])
#     m["充值环比"]=pct_vec(m["充值金额"],m["lw_充值"])
#     return m.sort_values("充值金额",ascending=False)

# def load_agent_ret():
#     df=pd.read_csv(FILES["agent_ret"])
#     df["_yyyymmdd"]=(df["初始事件发生时间"].astype(str)
#                     .str.extract(r"(\d{4}-\d{2}-\d{2})")[0].str.replace("-",""))
#     for c in ["1日","2日","3日","7日"]:
#         df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
#     df["首充用户数"]=pd.to_numeric(df["首充用户数"],errors="coerce").fillna(0)
#     df["总代"]=pd.to_numeric(df["总代"],errors="coerce")
#     daily=df[df["_yyyymmdd"].notna()&df["_yyyymmdd"].str.match(r"^\d{8}$")&(df["指标"]=="留存率")].copy()
#     tw_d1_end=retention_end_date(THIS_WEEK[0],1)
#     tw_d3_end=retention_end_date(THIS_WEEK[0],3)
#     lw_d1_end=retention_end_date(LAST_WEEK[0],1)
#     lw_d7_end=retention_end_date(LAST_WEEK[0],7)
#     tw_all=daily[daily["_yyyymmdd"].between(*THIS_WEEK)]
#     lw_all=daily[daily["_yyyymmdd"].between(*LAST_WEEK)]
#     def _win(df_week,week_start,end_cap):
#         if end_cap is None: return df_week.iloc[0:0]
#         return df_week[(df_week["_yyyymmdd"]>=week_start)&(df_week["_yyyymmdd"]<=end_cap)]
#     tw_d1_df=_win(tw_all,THIS_WEEK[0],tw_d1_end)
#     tw_d3_df=_win(tw_all,THIS_WEEK[0],tw_d3_end)
#     lw_d1_df=_win(lw_all,LAST_WEEK[0],lw_d1_end)
#     lw_d7_df=_win(lw_all,LAST_WEEK[0],lw_d7_end)
#     def _wavg_all(df_sub,col):
#         result={}
#         for nm,grp in df_sub.groupby("name_总代"):
#             valid=grp[grp[col].notna()&(grp["首充用户数"]>0)]
#             if len(valid)>0:
#                 result[nm]=float(np.average(valid[col].values,weights=valid["首充用户数"].values))
#             else: result[nm]=np.nan
#         return result
#     tw_d1=_wavg_all(tw_d1_df,"1日"); tw_d3=_wavg_all(tw_d3_df,"3日")
#     lw_d1=_wavg_all(lw_d1_df,"1日"); lw_d7=_wavg_all(lw_d7_df,"7日")
#     id_map={}
#     for nm,grp in daily.groupby("name_总代"):
#         id_val=grp["总代"].dropna().values
#         if len(id_val)>0: id_map[nm]=int(id_val[0])
#     all_names=set(tw_d1)|set(lw_d1)
#     ret={}
#     for nm in all_names:
#         ret[nm]={"tw_d1":tw_d1.get(nm,np.nan),"tw_d3":tw_d3.get(nm,np.nan),
#                  "lw_d1":lw_d1.get(nm,np.nan),"lw_d7":lw_d7.get(nm,np.nan),
#                  "agent_id":id_map.get(nm,None)}
#     return ret

# def load_vip():
#     df=pd.read_excel(FILES["vip"]); df["日期"]=df["日期"].astype(str)
#     num=["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
#     df=to_num(df,num)
#     tw=df[df["日期"].between(*THIS_WEEK)]; lw=df[df["日期"].between(*LAST_WEEK)]
#     return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

# def load_vip_retention():
#     results={}
#     for fkey,rtype in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
#         df=pd.read_csv(FILES[fkey])
#         df["date_str"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
#         df["yyyymmdd"]=df["date_str"].str.replace("-","")
#         for c in ["1日","2日","3日","7日"]:
#             df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
#         df["n"]=pd.to_numeric(df["充值成功事件用户数"],errors="coerce")
#         df["vip"]=pd.to_numeric(df["vip_level"],errors="coerce")
#         stage=df[(df["指标"]=="留存率")&(df["初始事件发生时间"]=="阶段值")].copy()
#         stage_dict={int(r["vip"]):{c:r[c] for c in ["1日","2日","3日","7日"]}
#                     for _,r in stage.iterrows() if not pd.isna(r["vip"])}
#         daily=df[(df["指标"]=="留存率")&df["yyyymmdd"].notna()&df["vip"].notna()]
#         tw=daily[daily["yyyymmdd"].between(*THIS_WEEK)]
#         lw=daily[daily["yyyymmdd"].between(*LAST_WEEK)]
#         def wavg(d,col):
#             res={}
#             for vip,g in d.groupby("vip"):
#                 v=g[g[col].notna()]
#                 if len(v)>0: res[int(vip)]=float(np.average(v[col].values,weights=v["n"].values))
#             return res
#         results[rtype]={"tw":{c:wavg(tw,c) for c in ["1日","2日","3日","7日"]},
#                         "lw":{c:wavg(lw,c) for c in ["1日","2日","3日","7日"]},
#                         "stage":stage_dict}
#     return results

# def load_top_users():
#     dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
#     dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
#     for df in [dc_tw,dc_lw,dt_tw,dt_lw]:
#         for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
#             if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
#     tiers=[1,10,50,100,200,500]
#     df_p=pd.read_excel(FILES["platform"]); df_p["日期"]=df_p["日期"].astype(str); df_p=to_num(df_p)
#     tw_p=df_p[df_p["日期"].between(*THIS_WEEK)]
#     total_chg=tw_p["充值金额"].sum(); total_tx=tw_p["提现金额"].sum()
#     actual_cr=(total_chg-total_tx)/total_chg*100
#     dep_tiers,wdraw_tiers=[],[]
#     total_tw_c=dc_tw["充值金额"].sum(); total_tw_t=dt_tw["提款金额"].sum()
#     for t in tiers:
#         tw=dc_tw.head(t); lw=dc_lw.head(t)
#         tw_c=tw["充值金额"].sum(); tw_tx=tw["提款金额"].sum()
#         lw_c=lw["充值金额"].sum(); lw_tx=lw["提款金额"].sum()
#         dep_tiers.append({"tier":f"Top{t}","tw_chg":tw_c,"lw_chg":lw_c,
#             "tw_avg":tw_c/t,"lw_avg":lw_c/t,
#             "tw_cr":(tw_c-tw_tx)/tw_c*100 if tw_c>0 else 0,
#             "lw_cr":(lw_c-lw_tx)/lw_c*100 if lw_c>0 else 0,
#             "tw_win":tw["公司输赢"].sum(),"占全量":tw_c/total_tw_c*100})
#         twd=dt_tw.head(t); lwd=dt_lw.head(t)
#         tw_t2=twd["提款金额"].sum(); tw_c2=twd["充值金额"].sum()
#         lw_t2=lwd["提款金额"].sum(); lw_c2=lwd["充值金额"].sum()
#         winners=(twd["公司输赢"]<0).sum()
#         act=twd["活动奖励"].sum()/(tw_c2+twd["活动奖励"].sum())*100 if (tw_c2+twd["活动奖励"].sum())>0 else 0
#         excl_chg=total_chg-tw_c2; excl_tx=total_tx-tw_t2
#         excl_cr=(excl_chg-excl_tx)/excl_chg*100 if excl_chg>0 else 0
#         wdraw_tiers.append({"tier":f"Top{t}","tw_tx":tw_t2,"lw_tx":lw_t2,
#             "tw_avg":tw_t2/t,"lw_avg":lw_t2/t,
#             "tw_cr":(tw_c2-tw_t2)/tw_c2*100 if tw_c2>0 else 0,
#             "lw_cr":(lw_c2-lw_t2)/lw_c2*100 if lw_c2>0 else 0,
#             "赢家":winners,"总数":t,"赢家率":winners/t*100,
#             "活动占比":act,"占全量":tw_t2/total_tw_t*100,"大盘影响":excl_cr-actual_cr})
#     return dep_tiers,wdraw_tiers,dc_tw.head(200),dt_tw.head(20)

# def load_pref():
#     df=pd.read_csv(FILES["pref_tw"]); df_lw=pd.read_csv(FILES["pref_lw"])
#     df["阶段汇总"]=df["阶段汇总"].apply(clean); df_lw["阶段汇总"]=df_lw["阶段汇总"].apply(clean)
#     bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]
#     bet_lw=df_lw[df_lw["分析指标"]=="投注金额"]
#     g_bet=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
#     g_win=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
#     g_lw=bet_lw.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
#     g_usr=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
#     g=pd.concat([g_bet,g_win,g_lw,g_usr],axis=1).reset_index()
#     total=g["本周投注"].sum(); g["占比"]=g["本周投注"]/total*100
#     g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
#     g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
#     mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
#     return g.sort_values("本周投注",ascending=False).head(20), mfr

# def load_games():
#     df_tw=pd.read_excel(FILES["game_tw"]); df_lw=pd.read_excel(FILES["game_lw"])
#     for df in [df_tw,df_lw]:
#         for c in ["投注人数","投注局数","投注金额","公司输赢","人均投注局数","人均投注金额"]:
#             if c in df.columns: df[c]=df[c].apply(clean)
#     total_tw=df_tw["投注金额"].sum()
#     g_tw=df_tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
#         投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
#         投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
#     g_lw=df_lw.groupby("游戏.名称").agg(
#         投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
#         columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
#     g_tw["人均局数"]=g_tw["投注局数"]/g_tw["投注人数"]
#     g_tw["人均金额"]=g_tw["投注金额"]/g_tw["投注人数"]
#     g_tw["盈亏率"]=g_tw["公司输赢"]/g_tw["投注金额"]*100
#     g_tw["占比"]=g_tw["投注金额"]/total_tw*100
#     g_tw["厂商简称"]=g_tw["游戏厂商标签.名称"].apply(shorten_mfr)
#     gm=g_tw.merge(g_lw,on="游戏.名称",how="left")
#     gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
#     gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
#     return gm.sort_values("投注金额",ascending=False).head(30)

# def load_mfr():
#     df=pd.read_csv(FILES["mfr"])
#     # 列名修复：部分导出文件用'盈利率'而非'盈亏率'
#     df=df.rename(columns={"盈利率":"盈亏率"})
#     df=df[df["时间"]!="阶段汇总"].copy()
#     df["时间"]=df["时间"].astype(str).str.replace("-","")
#     for c in ["投注局数","投注人数","投注金额","公司输赢","盈亏率"]:
#         if c in df.columns: df[c]=df[c].apply(clean)
#     tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
#     def agg(d):
#         g=d.groupby("show_name_厂商标签id").agg(
#             投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
#             投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
#         g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
#         g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
#         g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100
#         g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
#     tw_g=agg(tw); lw_g=agg(lw)
#     mg=tw_g.merge(lw_g[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
#         columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
#         on="show_name_厂商标签id",how="left")
#     mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
#     mg["厂商显示名"]=mg["show_name_厂商标签id"].apply(shorten_mfr)
#     return mg.sort_values("投注金额",ascending=False)

# def load_mfr_game_delta():
#     """厂商投注额环比主要驱动游戏（按游戏报表）"""
#     df_tw=pd.read_excel(FILES["game_tw"]); df_lw=pd.read_excel(FILES["game_lw"])
#     for df in [df_tw,df_lw]:
#         for c in ["投注金额","公司输赢"]:
#             if c in df.columns: df[c]=df[c].apply(clean)
#     g_tw=df_tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index()
#     g_lw=df_lw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(投注金额=("投注金额","sum")).reset_index().rename(columns={"投注金额":"lw_投注"})
#     merged=g_tw.merge(g_lw,on=["游戏厂商标签.名称","游戏.名称"],how="outer").fillna(0)
#     merged["delta"]=merged["投注金额"]-merged["lw_投注"]
#     mfr_tw=df_tw.groupby("游戏厂商标签.名称")["投注金额"].sum().sort_values(ascending=False)
#     result={}
#     for mfr_name in mfr_tw.head(10).index:
#         sub=merged[merged["游戏厂商标签.名称"]==mfr_name].sort_values("delta",ascending=False)
#         result[mfr_name]={"up":sub.head(1),"dn":sub.tail(1),"total_tw":mfr_tw[mfr_name]}
#     return result

# def load_activities():
#     """v7: 各赠送活动全量，含赠送人数.1（上周人数），可算日均"""
#     df=pd.read_csv(FILES["gift"])
#     for c in ["赠送金额","赠送金额.1","赠送人数","赠送人数.1"]:
#         if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0)
#     agg=df.groupby(["账变opt_code","name_账变opt_id"]).agg(
#         赠送金额=("赠送金额","sum"),
#         lw_赠送=("赠送金额.1","sum"),
#         赠送人数=("赠送人数","sum"),
#         lw_赠送人数=("赠送人数.1","sum")
#     ).reset_index()
#     agg["环比"]=(agg["赠送金额"]-agg["lw_赠送"])/agg["lw_赠送"].replace(0,np.nan).abs()*100
#     total=agg["赠送金额"].sum()
#     agg["占比"]=agg["赠送金额"]/total*100
#     agg["人均"]=agg["赠送金额"]/agg["赠送人数"].replace(0,np.nan)
#     agg["日均人数_本"]=agg["赠送人数"]/7
#     agg["日均人数_上"]=agg["lw_赠送人数"]/7
#     agg["人数环比"]=(agg["赠送人数"]-agg["lw_赠送人数"])/agg["lw_赠送人数"].replace(0,np.nan)*100
#     return agg.sort_values("赠送金额",ascending=False), total

# def load_tool_data():
#     """道具发放/使用/使用率，合并活动映射（新映射表77行，未匹配归"其他"）"""
#     df_tw=pd.read_csv(FILES["tool_tw"]); df_lw=pd.read_csv(FILES["tool_lw"])
#     tm=pd.read_excel(FILES["tool_map"])
#     tm["道具ID"]=tm["道具ID"].astype(str).str.strip()
#     # 同一道具ID可能有多行备注，取第一条
#     tm_dedup=tm.drop_duplicates(subset="道具ID",keep="first")
#     for df in [df_tw,df_lw]:
#         df.columns=df.columns.str.strip().str.replace("\ufeff","")
#         df["道具ID"]=df["道具ID"].astype(str).str.strip().str.replace('"',"")
#         for c in ["道具发放(步骤1)","道具使用(步骤2)"]:
#             df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")
#         df["步骤2 转化"]=pd.to_numeric(df["步骤2 转化"].astype(str).str.replace("%",""),errors="coerce")
#     df_tw=df_tw[df_tw["道具ID"]!="总体"].copy()
#     df_lw=df_lw[df_lw["道具ID"]!="总体"].copy()
#     merged=df_tw.rename(columns={"道具发放(步骤1)":"本周发放","道具使用(步骤2)":"本周使用","步骤2 转化":"本周使用率"}
#     ).merge(df_lw[["道具ID","道具发放(步骤1)","道具使用(步骤2)","步骤2 转化"]].rename(
#         columns={"道具发放(步骤1)":"上周发放","道具使用(步骤2)":"上周使用","步骤2 转化":"上周使用率"}),
#         on="道具ID",how="outer").fillna(0)
#     merged=merged.merge(tm_dedup[["道具ID","活动","备注"]],on="道具ID",how="left")
#     # 未在映射表中的归"其他"，备注用道具ID
#     merged["活动"]=merged["活动"].fillna("其他")
#     merged["备注"]=merged["备注"].fillna(merged["道具ID"])
#     merged["发放环比"]=(merged["本周发放"]-merged["上周发放"])/merged["上周发放"].replace(0,np.nan)*100
#     merged["使用环比"]=(merged["本周使用"]-merged["上周使用"])/merged["上周使用"].replace(0,np.nan)*100
#     return merged.sort_values("本周发放",ascending=False)

# def load_first_dep_retention():
#     """首次充值活动用户留存；全程用 numpy array 避免 pandas index 对齐 AssertionError"""
#     df=pd.read_csv(FILES["first_dep_ret"])
#     df.columns=df.columns.str.strip().str.replace("\ufeff","")
#     df["_date"]=df["初始事件发生时间"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
#     df["_yyyymmdd"]=df["_date"].str.replace("-","")
#     for c in ["当日","1日","2日","3日","4日","5日","6日","7日"]:
#         if c in df.columns:
#             df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
#     df["账变事件用户数"]=pd.to_numeric(df["账变事件用户数"],errors="coerce").fillna(0)
#     stage_ret=df[df["初始事件发生时间"]=="阶段值"].copy()
#     daily=df[df["_yyyymmdd"].notna()&df["_yyyymmdd"].str.match(r"^\d{8}$")].copy()
#     # 必须 reset_index，然后用 numpy array，完全避免跨 DataFrame loc 索引对齐问题
#     rate_rows=daily[daily["指标"]=="留存率"].copy().reset_index(drop=True)
#     num_rows =daily[daily["指标"]=="留存人数"].copy().reset_index(drop=True)
#     tw_rate=rate_rows[rate_rows["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
#     lw_rate=rate_rows[rate_rows["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
#     tw_num =num_rows[num_rows["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
#     lw_num =num_rows[num_rows["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
#     def weighted_avg(rate_df,num_df,col):
#         if col not in rate_df.columns or len(rate_df)==0: return np.nan
#         r_vals=rate_df[col].to_numpy(dtype=float,na_value=np.nan)
#         valid=~np.isnan(r_vals)
#         if valid.sum()==0: return np.nan
#         n_vals=num_df["账变事件用户数"].to_numpy(dtype=float) if len(num_df)==len(rate_df) else np.ones(len(r_vals))
#         w=n_vals[valid]
#         if w.sum()==0: return float(np.nanmean(r_vals[valid]))
#         return float(np.average(r_vals[valid],weights=w))
#     result={"stage":stage_ret,"tw_users":int(tw_num["账变事件用户数"].sum()),"lw_users":int(lw_num["账变事件用户数"].sum())}
#     for col in ["1日","2日","3日","4日","5日","6日","7日"]:
#         result[f"tw_{col}"]=weighted_avg(tw_rate,tw_num,col)
#         result[f"lw_{col}"]=weighted_avg(lw_rate,lw_num,col)
#     if len(stage_ret)>0:
#         sr=stage_ret[stage_ret["指标"]=="留存率"]
#         if len(sr)>0:
#             for col in ["1日","2日","3日","7日"]:
#                 if col in sr.columns:
#                     v=sr[col].values[0]
#                     result[f"stage_{col}"]=clean(v) if pd.notna(v) else np.nan
#     return result

# def load_risk(dt_tw_raw, dc_tw_raw):
#     df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
#     bet_amt=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
#     win_pref=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
#     bet_cnt=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数")
#     top_game=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
#               .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
#               .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏","阶段汇总":"主游戏投注额"}).reset_index())
#     ug=pd.concat([bet_cnt,bet_amt,win_pref],axis=1).reset_index()
#     ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]; ug=ug.merge(top_game,on="账户ID",how="left")
#     top500=dt_tw_raw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
#     top500["赢家"]=top500["公司输赢"]<0; top500["投充比"]=top500["投注金额"]/top500["充值金额"]
#     high_risk=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
#     pp_ids=df[(df["游戏名称"]=="Auto-Roulette 1")&(df["分析指标"]=="投注次数")]["账户ID"].unique()
#     cluster_207=[u for u in pp_ids if str(u).startswith("207")]
#     c207_dt=dt_tw_raw[dt_tw_raw["账户ID"].isin(cluster_207)].sort_values("提款金额",ascending=False)
#     special=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False)
#     bet_g=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
#     win_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
#     usr_g=df[df["分析指标"]=="投注次数"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
#     winr_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
#     g=bet_g.merge(win_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(usr_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(winr_g,on=["show_name_厂商标签id","游戏名称"],how="left")
#     g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100; g["人均投注额"]=g["投注金额"]/g["玩家数"]
#     risk_g=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
#     return high_risk,c207_dt,cluster_207,special,g.sort_values("投注金额",ascending=False).head(20),risk_g,top500


# # ════════════════════════════════════════════════════════════════
# # 第四区：图表
# # ════════════════════════════════════════════════════════════════
# SPLIT=7
# def _vline(ax,dates):
#     ax.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
#     ax.text(SPLIT-4,ax.get_ylim()[1]*0.93,"上周",ha="center",fontsize=7,color="#64748b")
#     ax.text(SPLIT+3,ax.get_ylim()[1]*0.93,"本周",ha="center",fontsize=7,color="#1d4ed8")

# def chart_trend(trend):
#     fig=plt.figure(figsize=(16,10),facecolor="white")
#     gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.45,wspace=0.3)
#     dates=trend["dates"]; x=range(len(dates))
#     def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=0.25)
#     ax=fig.add_subplot(gs[0,0])
#     clrs=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT
#     ax.bar(x,[v/10000 for v in trend["充值"]],color=clrs,width=0.7,label="充值")
#     ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
#     ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold"); ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
#     ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax,dates)
#     ax2=fig.add_subplot(gs[0,1])
#     ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=0.7)
#     ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax2); _vline(ax2,dates)
#     ax3=fig.add_subplot(gs[1,0])
#     ax3.bar(x,trend["首充"],color=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT,width=0.7,label="首充人数")
#     ax3r=ax3.twinx(); ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
#     ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold"); ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
#     l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
#     ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left"); ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=0.25); ax3.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
#     ax4=fig.add_subplot(gs[1,1])
#     ax4.bar(x,trend["充提差比"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=0.7)
#     tw_m=np.mean(trend["充提差比"][SPLIT:]); lw_m=np.mean(trend["充提差比"][:SPLIT])
#     ax4.axhline(tw_m,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lw_m,color="#94a3b8",ls=":",lw=1.2)
#     ax4.text(len(dates)-0.5,tw_m+0.3,f"本周均{tw_m:.1f}%",fontsize=7,color="#7c3aed",ha="right")
#     ax4.text(0.5,lw_m+0.3,f"上周均{lw_m:.1f}%",fontsize=7,color="#64748b",ha="left")
#     ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold"); ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax4); _vline(ax4,dates)
#     fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
#     plt.tight_layout(rect=[0,0,1,0.98]); return fig_img(fig,16,11)

# def chart_ret_weekly(weeks):
#     fig,ax=plt.subplots(figsize=(12,5),facecolor="white")
#     # 列名是'第1日'
#     cols_p=[("第1日","#1d4ed8"),("第2日","#059669"),("第3日","#d97706"),("第7日","#7c3aed")]
#     x=np.arange(len(weeks)); w=0.18
#     for i,(col,clr) in enumerate(cols_p):
#         vals=[wk.get(col,np.nan) for wk in weeks]
#         bars=ax.bar(x+i*w-1.5*w,vals,w,label=col.replace("第","D").replace("日",""),color=clr,alpha=0.85)
#         for bar,v in zip(bars,vals):
#             if not (isinstance(v,float) and np.isnan(v)):
#                 ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
#     ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
#     ax.set_title("首充用户充值留存率 近4周加权平均对比",fontsize=10,fontweight="bold"); ax.legend(fontsize=8,loc="upper right")
#     ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=0.5); ax.text(2.6,ax.get_ylim()[1]*0.85,"↑本周",fontsize=7.5,color="#ef4444")
#     ax.grid(axis="y",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,16,5.5)

# def chart_vip(tw_v,lw_v):
#     vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
#     x=np.arange(len(vips)); w=0.35
#     fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
#     tw_c=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
#     lw_c=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
#     ax1.bar(x-w/2,tw_c,w,label="本周",color="#1d4ed8",alpha=0.85); ax1.bar(x+w/2,lw_c,w,label="上周",color="#93c5fd",alpha=0.7)
#     ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     tw_b=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
#     lw_b=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
#     ax2.bar(x-w/2,tw_b,w,label="本周",color="#059669",alpha=0.85); ax2.bar(x+w/2,lw_b,w,label="上周",color="#6ee7b7",alpha=0.7)
#     ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
#     chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
#     ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=0.85,width=0.6); ax3.axhline(0,color="black",lw=0.8)
#     ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=0.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,16,6)

# def chart_vip_retention(vip_ret):
#     vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
#     chg_tw=[vip_ret["chg"]["tw"]["1日"].get(v,0) for v in vips]; chg_lw=[vip_ret["chg"]["lw"]["1日"].get(v,0) for v in vips]; chg_3d=[vip_ret["chg"]["tw"]["3日"].get(v,0) for v in vips]
#     act_tw=[vip_ret["act"]["tw"]["1日"].get(v,0) for v in vips]; act_lw=[vip_ret["act"]["lw"]["1日"].get(v,0) for v in vips]; act_3d=[vip_ret["act"]["tw"]["3日"].get(v,0) for v in vips]
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=0.28
#     ax1.bar(x-w,chg_tw,w,label="次日留存(本周)",color="#1d4ed8",alpha=0.85); ax1.bar(x,chg_lw,w,label="次日留存(上周)",color="#93c5fd",alpha=0.7); ax1.bar(x+w,chg_3d,w,label="3日留存(本周)",color="#059669",alpha=0.75)
#     ax1.set_title("充值→充值 留存率（%）",fontsize=10,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,fontsize=8); ax1.legend(fontsize=7.5,loc="upper left"); ax1.grid(axis="y",alpha=0.25); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
#     ax2.bar(x-w,act_tw,w,label="次日留存(本周)",color="#7c3aed",alpha=0.85); ax2.bar(x,act_lw,w,label="次日留存(上周)",color="#c4b5fd",alpha=0.7); ax2.bar(x+w,act_3d,w,label="3日留存(本周)",color="#d97706",alpha=0.75)
#     ax2.set_title("充值→活跃 留存率（%）",fontsize=10,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,fontsize=8); ax2.legend(fontsize=7.5,loc="upper left"); ax2.grid(axis="y",alpha=0.25); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
#     plt.suptitle("VIP各等级充值留存率（本周 vs 上周，加权日均值）",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

# def chart_mfr_combo(mfr):
#     top10=mfr.head(10); names=top10["厂商显示名"].tolist() if "厂商显示名" in top10.columns else top10["show_name_厂商标签id"].tolist()
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
#     shares=top10["占比"].tolist(); lw_s=top10["lw_占比"].fillna(0).tolist()
#     bars=ax1.bar(names,shares,color=["#059669" if c>=l else "#dc2626" for c,l in zip(shares,lw_s)],alpha=0.85,width=0.6)
#     ax1.plot(names,lw_s,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
#     for bar,s,ic in zip(bars,shares,top10["投注环比"].tolist()):
#         ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,f"{s:.1f}%",ha="center",fontsize=7)
#         ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
#     ax1.set_title("Top10厂商 投注份额（本周 vs 上周）",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
#     ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
#     ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     x=np.arange(len(names)); w=0.35
#     ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=0.85)
#     ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=0.7)
#     ax2.axhline(0,color="black",lw=0.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5); ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
#     plt.tight_layout(); return fig_img(fig,16,6.5)

# def chart_top30(top30):
#     top15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in top15.iterrows()]
#     bets=[r["投注金额"]/10000 for _,r in top15.iterrows()]; lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in top15.iterrows()]
#     x=np.arange(len(names)); w=0.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
#     ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=0.85); ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=0.7)
#     for bar,v in zip(ax.patches[:len(names)],bets[::-1]): ax.text(bar.get_width()+0.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
#     ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8); ax.set_title("Top15游戏 投注金额（万USD，本周 vs 上周）",fontsize=9,fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="x",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,14,7)

# def chart_pref(pref_game,mfr_amt):
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
#     names=mfr_amt.index[:8].tolist(); vals=mfr_amt.values[:8].tolist(); total=sum(vals); pcts=[v/total*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
#     wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
#     for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
#     ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.05,-0.18)); ax1.set_title("Top500提款用户 厂商偏好（按投注金额）",fontsize=9,fontweight="bold")
#     top12=pref_game.head(12); gnames=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in top12.iterrows()]; gvals=[r["本周投注"]/10000 for _,r in top12.iterrows()]; gclrs=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in top12.iterrows()]
#     ax2.barh(range(len(gnames))[::-1],gvals,color=gclrs[::-1],alpha=0.85); ax2.set_yticks(range(len(gnames))); ax2.set_yticklabels(gnames,fontsize=7.5); ax2.set_title("偏好游戏Top12（按投注金额，红=平台亏损）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,16,6)

# def chart_activities(act_df):
#     top10=act_df.head(10); names=[str(r["name_账变opt_id"])[:12] for _,r in top10.iterrows()]; vals=[r["赠送金额"]/10000 for _,r in top10.iterrows()]; lw=[r["lw_赠送"]/10000 for _,r in top10.iterrows()]; envs=[r["环比"] for _,r in top10.iterrows()]
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=0.35
#     ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=0.85); ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=0.7); ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8); ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold"); ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     envs_safe=[v if not pd.isna(v) else 0 for v in envs]
#     ax2.barh(range(len(names)),envs_safe[::-1],color=["#059669" if v>0 else "#dc2626" for v in envs_safe[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8); ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,16,6)

# def chart_tool_usage(tool_df):
#     # 只展示本周发放>=100的道具，按发放量降序取top12
#     big = tool_df[tool_df["本周发放"] >= 100].head(12)
#     top12 = big if len(big) > 0 else tool_df.head(12)
#     def _lbl(r):
#         act = str(r.get("活动",""))
#         note = str(r.get("备注",""))
#         if act == "其他": return f"{str(r['道具ID'])[:6]}\n{note[:8]}"
#         short_act = act[:4] if len(act)>4 else act
#         return f"[{short_act}]\n{note[:8]}"
#     names=[_lbl(r) for _,r in top12.iterrows()]
#     tw_issue=[r["本周发放"] for _,r in top12.iterrows()]; lw_issue=[r["上周发放"] for _,r in top12.iterrows()]
#     tw_rate=[r["本周使用率"] if not pd.isna(r["本周使用率"]) else 0 for _,r in top12.iterrows()]
#     lw_rate=[r["上周使用率"] if not pd.isna(r["上周使用率"]) else 0 for _,r in top12.iterrows()]
#     fig=plt.figure(figsize=(16,9),facecolor="white"); gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.5,wspace=0.35)
#     x=np.arange(len(names)); w=0.35
#     ax1=fig.add_subplot(gs[0,0])
#     ax1.bar(x-w/2,tw_issue,w,label="本周发放",color="#1d4ed8",alpha=0.85); ax1.bar(x+w/2,lw_issue,w,label="上周发放",color="#93c5fd",alpha=0.7)
#     ax1.set_title("Top12道具 发放量（本周 vs 上周）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(names,fontsize=6.5); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
#     ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v/10000:.0f}万" if v>=10000 else f"{v:.0f}"))
#     ax2=fig.add_subplot(gs[0,1])
#     ax2.bar(x-w/2,tw_rate,w,label="本周使用率",color="#059669",alpha=0.85); ax2.bar(x+w/2,lw_rate,w,label="上周使用率",color="#6ee7b7",alpha=0.7)
#     ax2.set_title("Top12道具 使用率（本周 vs 上周）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(names,fontsize=6.5); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
#     ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
#     ax3=fig.add_subplot(gs[1,:])
#     delta_issue=[r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0 for _,r in top12.iterrows()]
#     delta_use=[r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0 for _,r in top12.iterrows()]
#     ax3.bar(x-w/2,delta_issue,w,label="发放量环比",color=["#059669" if v>=0 else "#dc2626" for v in delta_issue],alpha=0.85)
#     ax3.bar(x+w/2,delta_use,w,label="使用量环比",color=["#7c3aed" if v>=0 else "#f97316" for v in delta_use],alpha=0.7)
#     ax3.axhline(0,color="black",lw=0.8); ax3.set_title("Top12道具 发放/使用量 环比变化（%）",fontsize=9,fontweight="bold")
#     ax3.set_xticks(x); ax3.set_xticklabels(names,fontsize=6.5); ax3.legend(fontsize=7.5); ax3.grid(axis="y",alpha=0.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
#     ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
#     plt.suptitle("道具发放与使用分析",fontsize=11,fontweight="bold",y=1.01)
#     plt.tight_layout(rect=[0,0,1,0.98]); return fig_img(fig,16,10)

# def chart_first_dep_retention(ret_data):
#     lw_vals=[ret_data.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     tw_vals=[ret_data.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     days=["D1","D2","D3","D4","D5","D6","D7"]
#     fig,ax=plt.subplots(figsize=(12,5),facecolor="white"); x=np.arange(len(days))
#     ax.plot(x,lw_vals,"-o",color="#93c5fd",lw=2,ms=6,label=f"上周（{LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}）")
#     ax.plot(x,tw_vals,"-o",color="#1d4ed8",lw=2,ms=6,label=f"本周（{THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}）")
#     for i,(lv,tv) in enumerate(zip(lw_vals,tw_vals)):
#         if lv is not None and not (isinstance(lv,float) and np.isnan(lv)):
#             ax.text(i,lv+0.4,f"{lv:.1f}%",ha="center",fontsize=7.5,color="#64748b")
#         if tv is not None and not (isinstance(tv,float) and np.isnan(tv)):
#             ax.text(i,tv-1.2,f"{tv:.1f}%",ha="center",fontsize=7.5,color="#1d4ed8",fontweight="bold")
#     ax.set_xticks(x); ax.set_xticklabels(days,fontsize=9)
#     ax.set_title("首次充值活动用户 充值留存率趋势（本周 vs 上周）",fontsize=10,fontweight="bold")
#     ax.legend(fontsize=8.5,loc="upper right"); ax.grid(axis="y",alpha=0.3)
#     ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
#     plt.tight_layout(); return fig_img(fig,13,5.5)

# def chart_risk_scatter(top500):
#     valid=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy(); valid=valid[valid["投充比"]<200]
#     fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
#     ax.scatter(valid["投充比"],valid["公司输赢"]/10000,c=["#dc2626" if v<0 else "#059669" for v in valid["公司输赢"]],s=[min(abs(v)/500+20,200) for v in valid["公司输赢"]],alpha=0.55,edgecolors="none")
#     ax.axhline(0,color="black",lw=0.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=0.7); ax.text(21,ax.get_ylim()[0]*0.9,"投充比=20x",fontsize=7.5,color="#d97706")
#     for _,r in top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()].iterrows():
#         ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),xytext=(r["投充比"]+5,r["公司输赢"]/10000-0.2),fontsize=6.5,color="#991b1b",arrowprops=dict(arrowstyle="->",color="#991b1b",lw=0.7))
#     ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8); ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
#     ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=0.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=0.6,label="平台赢钱")],fontsize=8,loc="upper right"); ax.grid(alpha=0.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
#     plt.tight_layout(); return fig_img(fig,15,6)

# def chart_risk_games(risk_g):
#     top12=risk_g.head(12); names=[r["游戏名称"][:16] for _,r in top12.iterrows()]; losses=[abs(r["公司输赢"]) for _,r in top12.iterrows()]; rates=[r["盈亏率"] for _,r in top12.iterrows()]
#     fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
#     ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=0.85)
#     for bar,v in zip(ax1.patches,losses[::-1]): ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
#     ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
#     ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
#     plt.tight_layout(); return fig_img(fig,16,6.5)


# # ════════════════════════════════════════════════════════════════
# # 第五区：章节构建
# # ════════════════════════════════════════════════════════════════

# def build_overview(K, trend, weekly_ret):
#     S=[sec_title("一、大盘核心数据")]
#     kpis=[
#         ("充值金额",f"{K['tw_充值金额']/10000:.1f}万",f"{K['lw_充值金额']/10000:.1f}万",K["pct_充值金额"],True),
#         ("提现金额",f"{K['tw_提现金额']/10000:.1f}万",f"{K['lw_提现金额']/10000:.1f}万",K["pct_提现金额"],False),
#         ("充提差",f"{K['tw_充提差']/10000:.1f}万",f"{K['lw_充提差']/10000:.1f}万",K["pct_充提差"],True),
#         ("充提差率",f"{K['tw_充提差比']:.2f}%",f"{K['lw_充提差比']:.2f}%",K["pct_充提差比"],True),
#         ("公司输赢",f"{K['tw_公司输赢']/10000:.1f}万",f"{K['lw_公司输赢']/10000:.1f}万",K["pct_公司输赢"],True),
#         ("盈亏率",f"{K['tw_盈亏率']:.3f}%",f"{K['lw_盈亏率']:.3f}%",K["pct_盈亏率"],True),
#         ("注册人数",f"{int(K['tw_注册人数']):,}",f"{int(K['lw_注册人数']):,}",K["pct_注册人数"],True),
#         ("首充人数",f"{int(K['tw_首充人数']):,}",f"{int(K['lw_首充人数']):,}",K["pct_首充人数"],True),
#         ("日均活跃",f"{K['tw_活跃人数']/7/10000:.1f}万",f"{K['lw_活跃人数']/7/10000:.1f}万",K["pct_活跃人数"],True),
#         ("投注金额",f"{K['tw_投注金额']/10000:.0f}万",f"{K['lw_投注金额']/10000:.0f}万",K["pct_投注金额"],True),
#         ("全量日均ARPPU",f"${K['tw_全量Arppu']:.2f}",f"${K['lw_全量Arppu']:.2f}",K["pct_全量Arppu"],True),
#         ("老用户日均ARPPU",f"${K['tw_老用户ARPPU']:.2f}",f"${K['lw_老用户ARPPU']:.2f}",K["pct_老用户ARPPU"],True),
#         ("首充日均ARPPU",f"${K['tw_首充Arppu']:.2f}",f"${K['lw_首充Arppu']:.2f}",K["pct_首充Arppu"],True),
#         ("总赠送金额",f"{K['tw_总赠送金额']/10000:.1f}万",f"{K['lw_总赠送金额']/10000:.1f}万",K["pct_总赠送金额"],False),
#         ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",f"{K['lw_赠送充值比']:.2f}%",K["pct_赠送充值比"],False),
#         ("首充转化率",f"{K['tw_首充转化率']:.1f}%",f"{K['lw_首充转化率']:.1f}%",K["pct_首充转化率"],True),
#         ("首充次日充值留存",f"{K['tw_首充次日复充率']:.1f}%",f"{K['lw_首充次日复充率']:.1f}%",K["pct_首充次日复充率"],True),
#         ("推广消耗",f"{K['tw_真实消耗']/10000:.1f}万",f"{K['lw_真实消耗']/10000:.1f}万",K["pct_真实消耗"],False),
#     ]
#     S.append(kpi_card4(kpis,cols=4)); S.append(Spacer(1,8))
#     cr_delta=K["tw_充提差比"]-K["lw_充提差比"]; ret_delta=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
#     ov=dict(tw_充值金额=K["tw_充值金额"]/10000,pct_充值金额=K["pct_充值金额"],tw_公司输赢=K["tw_公司输赢"]/10000,pct_公司输赢=K["pct_公司输赢"],pct_真实消耗=K["pct_真实消耗"],tw_充提差比=K["tw_充提差比"],lw_充提差比=K["lw_充提差比"],delta_充提差比=cr_delta,tw_盈亏率=K["tw_盈亏率"],lw_盈亏率=K["lw_盈亏率"],tw_首充人数=int(K["tw_首充人数"]),pct_首充人数=K["pct_首充人数"],tw_首充转化率=K["tw_首充转化率"],tw_首充Arppu=K["tw_首充Arppu"],pct_首充Arppu=K["pct_首充Arppu"],tw_首充次日复充率=K["tw_首充次日复充率"],lw_首充次日复充率=K["lw_首充次日复充率"],delta_首充次日复充率=ret_delta)
#     S.append(insight_box(render_list(OVERVIEW_INSIGHTS,**ov)))
#     S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
#     S.append(sub_title("首充用户充值留存率 — 近4周加权平均对比")); S.append(chart_ret_weekly(weekly_ret))
#     this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
#     d1=(this_w.get("第1日",0) or 0)-(last_w.get("第1日",0) or 0)
#     rn=dict(tw_d1=this_w.get("第1日",0) or 0,tw_d2=this_w.get("第2日",0) or 0,delta_d1=d1,lw_d7=last_w.get("第7日",0) or 0)
#     S.append(insight_box(render_list(RETENTION_NOTE,**rn),clr=C_AMBER))
#     return S

# def build_agents(agents, agent_ret):
#     S=[sec_title("二、总代分析")]  # v7: 去掉"（剔除总代0官方总代）"
#     S.append(sub_title("全量总代表现（本周 vs 上周，按充值金额排序，含官方总代0）"))
#     full=PW-2*MARGIN
#     tw_d3_end=retention_end_date(THIS_WEEK[0],3); lw_d7_end=retention_end_date(LAST_WEEK[0],7)
#     tw_d3_lbl=f"{tw_d3_end[4:6]}/{tw_d3_end[6:]}" if tw_d3_end else "-"
#     lw_d7_lbl=f"{lw_d7_end[4:6]}/{lw_d7_end[6:]}" if lw_d7_end else "-"
#     headers=["ID","总代名称","注册(环比)","首充","充值\n(万)","充提差率(差值)","消耗\n(万)","一级首充\n成本(本/上)",f"次日留存\n(本/上)",f"3留本周\n(~{tw_d3_lbl})",f"7留上周\n(~{lw_d7_lbl})"]
#     def _fret(v): return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"
#     rows=[]
#     for _,r in agents.iterrows():
#         agent_id=int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
#         rname=str(r["总代.名称"]); cr=r["充提差率"]; lw_cr=r.get("lw_充提差率",0) or 0; cr_diff=cr-lw_cr
#         cost=(r.get("总消耗",0) or 0)/10000; fc_c=r.get("一级首充成本",0) or 0; lw_fc_c=r.get("lw_一级首充成本",0) or 0
#         reg=int(r["注册人数"]); reg_chg=r.get("注册环比",0) or 0
#         reg_str=f"{reg:,}/{'+' if reg_chg>=0 else ''}{reg_chg:.1f}%"
#         cr_str=f"{cr:.1f}%/{'+' if cr_diff>=0 else ''}{cr_diff:.1f}pp"
#         cr_clr=C_GREEN if cr>=CR_HIGH else (C_RED if cr<CR_LOW else C_DARK)
#         d=agent_ret.get(rname,{}); tw_d1=d.get("tw_d1",np.nan); lw_d1=d.get("lw_d1",np.nan); tw_d3=d.get("tw_d3",np.nan); lw_d7=d.get("lw_d7",np.nan)
#         if not (isinstance(tw_d1,float) and np.isnan(tw_d1)) and not (isinstance(lw_d1,float) and np.isnan(lw_d1)):
#             nd_str=f"{tw_d1:.1f}%/{lw_d1:.1f}%"; nd_clr=C_GREEN if tw_d1>=lw_d1 else C_RED
#         elif not (isinstance(tw_d1,float) and np.isnan(tw_d1)): nd_str=f"{tw_d1:.1f}%/-"; nd_clr=C_DARK
#         else: nd_str="-"; nd_clr=C_GRAY
#         rows.append([cell(str(agent_id),False,C_GRAY,TA_CENTER),cell(rname[:16],True,C_DARK,TA_LEFT),cell(reg_str,False,C_GREEN if reg_chg>=0 else C_RED,TA_RIGHT),cell(f"{int(r['首充人数']):,}",False,C_DARK,TA_RIGHT),cell(f"{r['充值金额']/10000:.0f}",False,C_DARK,TA_RIGHT),cell(cr_str,False,cr_clr,TA_RIGHT),cell(f"{cost:.1f}" if cost>0 else "-",False,C_DARK,TA_RIGHT),cell(f"${fc_c:.0f}/${lw_fc_c:.0f}" if fc_c>0 else "-",False,C_DARK,TA_RIGHT),cell(nd_str,False,nd_clr,TA_RIGHT),cell(_fret(tw_d3),False,C_DARK,TA_RIGHT),cell(_fret(lw_d7),False,C_DARK,TA_RIGHT)])
#     cw=[full*x for x in [0.05,0.17,0.11,0.06,0.06,0.12,0.06,0.11,0.10,0.08,0.08]]
#     S.append(dtable(headers,rows,cw,fsize=6.5)); S.append(Spacer(1,4))
#     S.append(P(f"★ 3留本周截至{tw_d3_lbl}（剔除未到期天）；7留上周全部7天完整数据可比。充提差率≥{CR_HIGH}%绿，<{CR_LOW}%红。",7,False,C_GRAY)); S.append(Spacer(1,6))
#     dsp_rows=agents[agents["总代.名称"].str.contains("DSP",na=False)]
#     av=dict(dsp_cr=dsp_rows["充提差率"].values[0] if len(dsp_rows)>0 else 0,dsp_reg=dsp_rows["注册环比"].values[0] if len(dsp_rows)>0 else 0)
#     S.append(insight_box(render_list(AGENT_INSIGHTS,**av)))
#     return S

# def build_users_section(tw_v, lw_v, dep_t, wdraw_t, dc_tw, dt_tw):
#     S=[sec_title("三、用户分析")]; full=PW-2*MARGIN
#     S.append(sub_title("VIP等级分层分析（充值/投注金额 本周 vs 上周）")); S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
#     total_chg=tw_v["充值金额"].sum()
#     high_vip_sum=sum(tw_v.loc[v,"充值金额"] for v in [9,10,11] if v in tw_v.index)
#     vv=dict(high_vip_pct=high_vip_sum/total_chg*100,tw_vip1_active=int(tw_v.loc[1,"活跃人数"]) if 1 in tw_v.index else 0,lw_vip1_active=int(lw_v.loc[1,"活跃人数"]) if 1 in lw_v.index else 0)
#     S.append(insight_box(render_list(VIP_INSIGHTS,**vv)))
#     vip_ret=load_vip_retention()
#     S.append(sub_title("VIP各等级 充值→充值留存率 & 充值→活跃留存率（本周均值）")); S.append(chart_vip_retention(vip_ret)); S.append(Spacer(1,4))
#     rv=dict(act_v9=vip_ret["act"]["tw"]["1日"].get(9,0),act_v10=vip_ret["act"]["tw"]["1日"].get(10,0),chg_v10_tw=vip_ret["chg"]["tw"]["1日"].get(10,0),chg_v10_lw=vip_ret["chg"]["lw"]["1日"].get(10,0))
#     S.append(insight_box(render_list(VIP_RET_INSIGHTS,**rv))); S.append(Spacer(1,8))

#     # v7: 加注释说明"本周TopN vs 上周TopN各自独立排名"
#     S.append(sub_title("头部充值用户分层分析（本周充值Top N vs 上周充值Top N）"))
#     S.append(P('注：本周人均 = 本周充值Top N用户的人均充值；上周人均 = 上周充值Top N用户的人均充值。两组为各自周次独立排名的头部用户，并非同一批人的跨周对比。',7.5,False,C_GRAY)); S.append(Spacer(1,3))
#     headers=["分层","本周充值","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","占全量%","公司输赢"]
#     rows=[]
#     for d in dep_t:
#         rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
#     S.append(dtable(headers,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
#     dv=dict(top10_avg=dep_t[1]["tw_avg"],top10_avg_chg=pct(dep_t[1]["tw_avg"],dep_t[1]["lw_avg"]),top10_cr=dep_t[1]["tw_cr"])
#     S.append(insight_box(render_list(TOP_DEPOSIT_INSIGHTS,**dv)))

#     S.append(sub_title("头部提款用户分层分析（本周提款Top N vs 上周提款Top N）"))
#     S.append(P('注：本周人均 = 本周提款Top N用户的人均提款；上周人均 = 上周提款Top N用户的人均提款。两组为各自周次独立排名的头部用户，并非同一批人的跨周对比。',7.5,False,C_GRAY)); S.append(Spacer(1,3))
#     headers3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","赢家\n比例","活动\n占比","占全\n量%","剔除后大盘\n充提差影响"]
#     rows3=[]
#     for d in wdraw_t:
#         rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
#     S.append(dtable(headers3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
#     wv=dict(top100_impact=wdraw_t[3]["大盘影响"],top500_act_pct=wdraw_t[5]["活动占比"])
#     S.append(insight_box(render_list(TOP_WITHDRAW_INSIGHTS,**wv),clr=C_AMBER))
#     return S

# def build_games_section(mfr, top30, mfr_game_delta):
#     S=[sec_title("四、游戏分析")]; full=PW-2*MARGIN
#     S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
#     # v7: 列名"投注额 环比"
#     headers=["排名","厂商","日均投注人数\n(本/上)","人均\n局数","人均\n金额","投注额\n(万)","投注额\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
#     rows=[]
#     for i,(_,r) in enumerate(mfr.head(10).iterrows()):
#         pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
#         rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r.get("厂商显示名",r["show_name_厂商标签id"])),True),cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
#     S.append(dtable(headers,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
#     def _mv(name,col):
#         sub=mfr[mfr["show_name_厂商标签id"]==name]; return sub[col].values[0] if len(sub)>0 else 0
#     mv=dict(tada_share=mfr.iloc[0]["占比"],tada_share_lw=mfr.iloc[0].get("lw_占比",0),tada_chg=mfr.iloc[0].get("投注环比",0),evol_pl=_mv("Evolution","盈亏率"),evol_pl_lw=_mv("Evolution","lw_盈亏率"),oaks_pl=_mv("3Oaks","盈亏率"),ptech_pl=_mv("PlayTech","盈亏率"))
#     S.append(insight_box(render_list(MFR_INSIGHTS,**mv)))

#     # v7: 厂商投注额环比驱动游戏分析
#     if mfr_game_delta:
#         S.append(sub_title("▶ 厂商投注额环比主要驱动游戏"))
#         delta_headers=["厂商","投注额\n环比","增量最大游戏(+贡献)","降量最大游戏(-拖累)"]
#         delta_rows=[]
#         for mfr_name,gdata in list(mfr_game_delta.items()):
#             mfr_row=mfr[mfr["show_name_厂商标签id"]==mfr_name]
#             mfr_chg=mfr_row["投注环比"].values[0] if len(mfr_row)>0 else 0
#             up=gdata["up"]; dn=gdata["dn"]
#             up_str=f'{up.iloc[0]["游戏.名称"][:16]}（+${up.iloc[0]["delta"]/10000:.1f}万）' if len(up)>0 and up.iloc[0]["delta"]>0 else "-"
#             dn_str=f'{dn.iloc[0]["游戏.名称"][:16]}（${dn.iloc[0]["delta"]/10000:.1f}万）' if len(dn)>0 and dn.iloc[0]["delta"]<0 else "-"
#             delta_rows.append([cell(str(shorten_mfr(mfr_name)),True,C_DARK),rc(mfr_chg),cell(up_str,False,C_GREEN if up_str!="-" else C_GRAY),cell(dn_str,False,C_RED if dn_str!="-" else C_GRAY)])
#         S.append(dtable(delta_headers,delta_rows,[full*x for x in [0.15,0.10,0.37,0.38]],fsize=6.8))
#         S.append(Spacer(1,6))

#     S.append(sub_title("Top30游戏详细数据（本周 vs 上周）")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
#     # v7: 厂商简称列
#     headers2=["#","游戏名称","厂商\n简称","投注\n人数","人均\n局数","人均\n金额","投注额\n(万)","投注额\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
#     rows2=[]
#     for i,(_,r) in enumerate(top30.iterrows()):
#         pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
#         rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["游戏.名称"])[:18],True),cell(str(r.get("厂商简称",r["游戏厂商标签.名称"]))[:5]),cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
#     S.append(dtable(headers2,rows2,[full*x for x in [0.04,0.18,0.07,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
#     return S

# def build_activities_section(act_df, total_gift):
#     S=[sec_title("五、活动分析")]; full=PW-2*MARGIN
#     S.append(sub_title("各活动赠送效果（本周 vs 上周，全量活动，含环比）"))
#     S.append(chart_activities(act_df)); S.append(Spacer(1,4))
#     # v7: 日均赠送人数（本/上周）+ 人数环比 + 本周人均赠送；全量列出
#     headers=["活动名称","本周赠送","上周赠送","赠送\n金额环比","本周日均\n赠送人数","上周日均\n赠送人数","人数\n环比","本周人均\n赠送金额","占比"]
#     rows=[]
#     for _,r in act_df.iterrows():
#         env=r["环比"] if not pd.isna(r.get("环比",np.nan)) else 0
#         pnv=r.get("人数环比",0) if not pd.isna(r.get("人数环比",np.nan)) else 0
#         dbu=r.get("日均人数_本",0) if not pd.isna(r.get("日均人数_本",np.nan)) else 0
#         dbl=r.get("日均人数_上",0) if not pd.isna(r.get("日均人数_上",np.nan)) else 0
#         avg=r.get("人均",0) if not pd.isna(r.get("人均",np.nan)) else 0
#         rows.append([cell(str(r["name_账变opt_id"])[:18],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['lw_赠送']:,.0f}",False,C_GRAY,TA_RIGHT),rc(env),cell(f"{dbu:.0f}",False,C_DARK,TA_RIGHT),cell(f"{dbl:.0f}",False,C_GRAY,TA_RIGHT),rc(pnv),cell(f"${avg:.2f}",False,C_DARK,TA_RIGHT),cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
#     S.append(dtable(headers,rows,[full*x for x in [0.20,0.11,0.11,0.08,0.12,0.12,0.08,0.10,0.08]],fsize=6.5))
#     S.append(Spacer(1,4))
#     daily_gift=total_gift/7/10000
#     S.append(P(f"本周总赠送金额：${total_gift/10000:.2f}万 | 日均赠送：${daily_gift:.2f}万",9,True,C_DARK)); S.append(Spacer(1,6))
#     av=dict(total_acts=len(act_df),daily_gift=daily_gift)
#     S.append(insight_box(render_list(ACTIVITY_INSIGHTS,**av)))

#     # ─── 存款来源道具分析（暂时注释掉，后期需要时取消注释）────────────────
#     # S.append(sub_title("存款来源道具分析（充值金额占比）"))
#     # src_df = load_deposit_src()
#     # ...
#     # ────────────────────────────────────────────────────────────────────
#     return S

# def build_tool_section(tool_df, first_dep_ret):
#     S=[sec_title("六、道具专题分析",clr=C_TEAL)]; full=PW-2*MARGIN
#     S.append(sub_title("6.1 道具发放、使用及使用率汇总（本周 vs 上周，仅展示发放≥100）"))
#     S.append(chart_tool_usage(tool_df)); S.append(Spacer(1,4))
#     headers=["道具ID","活动类型","备注说明","本周\n发放","上周\n发放","发放\n环比","本周\n使用","上周\n使用","使用\n环比","本周\n使用率","上周\n使用率"]
#     # 表格也只列出本周发放>=100
#     tool_display = tool_df[tool_df["本周发放"] >= 100].copy()
#     rows=[]
#     for _,r in tool_display.iterrows():
#         di=r.get("发放环比",0) if not pd.isna(r.get("发放环比",np.nan)) else 0
#         du=r.get("使用环比",0) if not pd.isna(r.get("使用环比",np.nan)) else 0
#         tw_rt=r.get("本周使用率",0) or 0; lw_rt=r.get("上周使用率",0) or 0
#         rt_clr=C_GREEN if tw_rt>=lw_rt else C_RED
#         rows.append([cell(str(r["道具ID"]),False,C_GRAY,TA_CENTER),cell(str(r.get("活动",""))[:10],False,C_PURPLE),cell(str(r.get("备注",""))[:14],False,C_DARK),cell(f"{int(r['本周发放']):,}" if r['本周发放']>0 else "-",False,C_DARK,TA_RIGHT),cell(f"{int(r['上周发放']):,}" if r['上周发放']>0 else "-",False,C_GRAY,TA_RIGHT),rc(di),cell(f"{int(r['本周使用']):,}" if r['本周使用']>0 else "-",False,C_DARK,TA_RIGHT),cell(f"{int(r['上周使用']):,}" if r['上周使用']>0 else "-",False,C_GRAY,TA_RIGHT),rc(du),cell(f"{tw_rt:.1f}%",False,rt_clr,TA_RIGHT),cell(f"{lw_rt:.1f}%",False,C_GRAY,TA_RIGHT)])
#     S.append(dtable(headers,rows,[full*x for x in [0.07,0.10,0.14,0.08,0.08,0.07,0.08,0.08,0.07,0.08,0.08]],fsize=6.3))
#     S.append(Spacer(1,4))
#     # 使用率分析只看本周发放>=100的道具（数量太少不具代表性）
#     big_vol = tool_df[tool_df["本周发放"] >= 100].copy()
#     def _tool_label(r):
#         act = str(r.get("活动",""))
#         note = str(r.get("备注",""))
#         if act == "其他":
#             return f"{note}({r['本周使用率']:.0f}%)"
#         return f"[{act}]{note}({r['本周使用率']:.0f}%)"
#     if len(big_vol) >= 3:
#         top3_hi = big_vol.nlargest(3,"本周使用率")
#         top3_lo = big_vol.nsmallest(3,"本周使用率")
#         hi_names = "、".join([_tool_label(r) for _,r in top3_hi.iterrows()])
#         lo_names = "、".join([_tool_label(r) for _,r in top3_lo.iterrows()])
#     else:
#         top3_hi = big_vol.nlargest(min(3,len(big_vol)),"本周使用率")
#         top3_lo = big_vol.nsmallest(min(3,len(big_vol)),"本周使用率")
#         hi_names = "、".join([_tool_label(r) for _,r in top3_hi.iterrows()])
#         lo_names = "、".join([_tool_label(r) for _,r in top3_lo.iterrows()])
#     tw_total=int(tool_df["本周发放"].sum()); lw_total=tool_df["上周发放"].sum()
#     S.append(insight_box([
#         f"使用率最高3类（发放≥100）：{hi_names}，领取后转化良好。",
#         f"使用率最低3类（发放≥100）：{lo_names}，建议检查道具设计或发放对象匹配度。",
#         f"本周总道具发放{tw_total:,}，较上周{lw_total:,.0f}，环比{pct(tw_total,lw_total):+.1f}%。"
#     ]))
#     S.append(Spacer(1,8))

#     S.append(sub_title("6.2 首次充值活动用户 充值留存分析（重点）"))
#     stage=first_dep_ret.get("stage",pd.DataFrame())
#     if len(stage)>0:
#         S.append(P(f"▸ 两周阶段汇总（{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} - {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}）",9,True,C_DARK)); S.append(Spacer(1,3))
#         stage_rate=stage[stage["指标"]=="留存率"]; stage_num=stage[stage["指标"]=="留存人数"]; stage_arpu=stage[stage["指标"]=="人均充值金额"]
#         if len(stage_rate)>0:
#             users=int(stage_num["账变事件用户数"].values[0]) if len(stage_num)>0 else 0
#             S.append(P(f"首充活动用户总数：{users:,}人",8.5,False,C_DARK))
#             day_cols=["当日","1日","2日","3日","4日","5日","6日","7日"]
#             s_rows=[]
#             if len(stage_rate)>0: s_rows.append(["留存率"]+[str(stage_rate.iloc[0].get(c,"-")) for c in day_cols])
#             if len(stage_num)>0:  s_rows.append(["留存人数"]+[str(stage_num.iloc[0].get(c,"-")) for c in day_cols])
#             if len(stage_arpu)>0: s_rows.append(["人均充值($)"]+[str(stage_arpu.iloc[0].get(c,"-")) for c in day_cols])
#             if s_rows:
#                 tbl_rows=[[cell(row[0],True,C_DARK)]+[cell(str(v),False,C_DARK,TA_CENTER) for v in row[1:]] for row in s_rows]
#                 S.append(dtable(["指标"]+day_cols,tbl_rows,[full*0.12]+[full*0.11]*8,fsize=6.8))
#         S.append(Spacer(1,6))

#     S.append(P("▸ 本周 vs 上周 首充活动用户留存趋势",9,True,C_DARK)); S.append(Spacer(1,3))
#     S.append(chart_first_dep_retention(first_dep_ret)); S.append(Spacer(1,4))

#     tw_users=first_dep_ret.get("tw_users",0); lw_users=first_dep_ret.get("lw_users",0)
#     tw_vals=[first_dep_ret.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     lw_vals=[first_dep_ret.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
#     def _fr(v): return f"{v:.1f}%" if not (v is None or (isinstance(v,float) and np.isnan(v))) else "-"
#     ret_headers=["周次","首充用户数","D1留存","D2留存","D3留存","D4留存","D5留存","D6留存","D7留存"]
#     ret_rows=[
#         [cell(f"本周 {THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}",True,C_BLUE)]+[cell(f"{tw_users:,}",False,C_DARK,TA_RIGHT)]+[cell(_fr(v),False,C_GREEN if not pd.isna(v) and not pd.isna(lv) and v>=lv else (C_RED if not pd.isna(v) and not pd.isna(lv) and v<lv else C_DARK),TA_RIGHT) for v,lv in zip(tw_vals,lw_vals)],
#         [cell(f"上周 {LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}",True,C_GRAY)]+[cell(f"{lw_users:,}",False,C_GRAY,TA_RIGHT)]+[cell(_fr(v),False,C_GRAY,TA_RIGHT) for v in lw_vals],
#     ]
#     S.append(dtable(ret_headers,ret_rows,[full*0.16,full*0.10]+[full*0.106]*7,fsize=7,zebra=False))
#     S.append(Spacer(1,4))
#     tw_d1=first_dep_ret.get("tw_1日",np.nan); lw_d1=first_dep_ret.get("lw_1日",np.nan)
#     insights=[]
#     if not pd.isna(tw_d1) and not pd.isna(lw_d1):
#         delta=tw_d1-lw_d1
#         insights.append(f"首充活动用户次日留存{tw_d1:.1f}%（上周{lw_d1:.1f}%，{delta:+.1f}pp），{'留存改善，活动质量提升' if delta>0 else '留存下降，建议优化次日触达策略'}。")
#     insights.append("首次充值活动（赠送金额全量活动第7位）：人均赠送$3.89，次日留存高于大盘平均，属高价值用户来源。")
#     insights.append("建议：对D1/D2留存用户设置阶梯式再充值道具激励；对D3后流失用户做专项召回（24h内触达效果最佳）。")
#     S.append(insight_box(insights,clr=C_AMBER))
#     return S

# def build_risk_section(high_risk, c207_dt, cluster_207, special_users, top20_games, risk_g, top500, dt_tw_full):
#     S=[sec_title("七、用户游戏风险专项分析",clr=colors.HexColor("#7c2d12"))]; full=PW-2*MARGIN
#     total_tx=top500["提款金额"].sum(); total_win=top500["公司输赢"].sum(); winner_cnt=(top500["公司输赢"]<0).sum()
#     S.append(sub_title("Top500提款用户总览"))
#     row_k=[]
#     for label,val,sub in [("Top500提款总额",f"${total_tx/10000:.1f}万",""),("平台净赔付",f"${abs(total_win)/10000:.1f}万","平台向该群体净赔"),("赢家比例",f"{winner_cnt/500*100:.1f}%",f"{winner_cnt}赢/{500-winner_cnt}输"),("高风险用户",f"{len(high_risk)}人","公司净输>$10,000")]:
#         fw=full/4-4; row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
#     t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP"); t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
#     S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
#     rsv=dict(winner_cnt=winner_cnt,winner_pct=winner_cnt/500*100,total_loss=abs(total_win)/10000,high_risk_cnt=len(high_risk),high_risk_loss=abs(high_risk["公司输赢"].sum())/10000 if len(high_risk)>0 else 0,high_risk_pct=abs(high_risk["公司输赢"].sum())/abs(total_win)*100 if abs(total_win)>0 and len(high_risk)>0 else 0)
#     S.append(insight_box(render_list(RISK_SCATTER_INSIGHTS,**rsv)))
#     if len(high_risk)>0:
#         S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
#         headers_r=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注\n均额","主玩厂商","主玩游戏","风险标签"]
#         rows_r=[]
#         for _,r in high_risk.iterrows():
#             ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0; avg_b=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
#             tags=[]; 
#             if r["充值金额"]<5000: tags.append("低充高提")
#             if ratio>50: tags.append("超高投充")
#             if avg_b>200: tags.append("高额单注")
#             rows_r.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),cell(f"${avg_b:.0f}",False,C_RED if avg_b>200 else C_DARK,TA_RIGHT),cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
#         S.append(dtable(headers_r,rows_r,[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]],fsize=6.5)); S.append(Spacer(1,6))
#     if len(cluster_207)>0:
#         S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警"))
#         S.append(P(f"集群账户数：{len(cluster_207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
#         headers_c=["账户ID","提款金额","充值金额","公司输赢","特征"]; rows_c=[]
#         for _,r in c207_dt.head(10).iterrows():
#             rows_c.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell("充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}")])
#         if len(c207_dt)>10: rows_c.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
#         S.append(dtable(headers_c,rows_c,[full*x for x in [0.22,0.18,0.18,0.18,0.24]])); S.append(Spacer(1,6))
#         ppv=dict(cluster_cnt=len(cluster_207))
#         S.append(insight_box(render_list(PP_ROULETTE_INSIGHTS,**ppv),clr=C_AMBER))
#     S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
#     headers_t=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; rows_t=[]
#     for i,(_,r) in enumerate(dt_tw_full.head(20).iterrows()):
#         win=r["公司输赢"]<0
#         rows_t.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
#     S.append(dtable(headers_t,rows_t,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
#     pref_game,mfr_amt=load_pref()
#     S.append(sub_title("Top500提款用户游戏偏好（按投注金额）")); S.append(chart_pref(pref_game,mfr_amt)); S.append(Spacer(1,4))
#     S.append(sub_title("▶ 高危游戏专项分析（Top500提款用户视角）")); S.append(chart_risk_games(risk_g)); S.append(Spacer(1,4))
#     headers_g=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; rows_g=[]
#     for _,r in risk_g.head(12).iterrows():
#         pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rl_clr=C_RED if rl in ("极高","高") else C_AMBER
#         rows_g.append([cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rl_clr)])
#     S.append(dtable(headers_g,rows_g,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5)); S.append(Spacer(1,6))
#     S.append(insight_box(RISK_GAME_INSIGHTS,clr=C_RED))
#     return S

# def build_conclusion(K, agents, mfr, dep_t, wdraw_t):
#     S=[sec_title("八、总结与行动建议")]; full=PW-2*MARGIN
#     S.append(sub_title("▶ 本周亮点"))
#     hv=dict(tw_充值金额=K["tw_充值金额"]/10000,pct_充值金额=K["pct_充值金额"],tw_充提差=K["tw_充提差"]/10000,pct_充提差=K["pct_充提差"],tw_充提差比=K["tw_充提差比"],delta_充提差比=K["tw_充提差比"]-K["lw_充提差比"],tw_注册人数=int(K["tw_注册人数"]),pct_注册人数=K["pct_注册人数"],tw_首充人数=int(K["tw_首充人数"]),pct_首充人数=K["pct_首充人数"])
#     bg_g=colors.HexColor("#f0fdf4")
#     for h in render_list(HIGHLIGHTS,**hv):
#         S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_g),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
#     S.append(Spacer(1,8)); S.append(sub_title("▶ 风险预警"))
#     rv=dict(tw_首充Arppu=K["tw_首充Arppu"],pct_首充Arppu=K["pct_首充Arppu"],tw_首充次日复充率=K["tw_首充次日复充率"],delta_首充次日复充率=K["tw_首充次日复充率"]-K["lw_首充次日复充率"])
#     bg_r=colors.HexColor("#fff5f5")
#     for r in render_list(RISKS,**rv):
#         S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_r),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
#     S.append(Spacer(1,8)); S.append(sub_title("▶ 行动建议"))
#     actv=dict(tw_首充Arppu=K["tw_首充Arppu"],pct_首充Arppu=K["pct_首充Arppu"],tw_首充次日复充率=K["tw_首充次日复充率"],delta_首充次日复充率=K["tw_首充次日复充率"]-K["lw_首充次日复充率"])
#     pclr_map={"[紧急]":C_RED,"[本周]":C_AMBER,"[下周]":C_GREEN}
#     rows_a=[[cell(pri,True,pclr_map.get(pri[:4],C_GRAY),TA_CENTER),cell(title,True,C_DARK),cell(render(desc,**actv),False,C_GRAY)] for pri,title,desc in ACTIONS]
#     S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows_a,[full*x for x in [0.1,0.22,0.68]]))
#     return S

# def header_footer(c,doc):
#     c.saveState(); w,h=A4
#     c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
#     c.setFillColor(colors.white); c.setFont(FNB,10)
#     c.drawString(MARGIN,h-17,f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
#     c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
#     c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
#     c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
#     c.restoreState()

# def main(data_root=None, output_dir=None):
#     root=Path(data_root) if data_root else DATA_ROOT
#     outdir=Path(output_dir) if output_dir else OUTPUT_DIR
#     outdir.mkdir(parents=True,exist_ok=True)
#     print("📊 MX 周报 PDF v7.1 生成中...")
#     _resolve_files(root); setup()
#     print("  ▶ 加载数据...")
#     K,trend=load_platform(); weekly_ret=load_weekly_retention()
#     agents=load_agents(); agent_ret=load_agent_ret()
#     tw_v,lw_v=load_vip(); dep_t,wdraw_t,dc_tw,dt_tw_top=load_top_users()
#     top30=load_games(); mfr=load_mfr(); mfr_game_delta=load_mfr_game_delta()
#     act_df,total_gift=load_activities()
#     tool_df=load_tool_data(); first_dep_ret=load_first_dep_retention()

#     # 风险数据
#     dt_tw_raw=pd.read_csv(FILES["dt_tw"]); dc_tw_raw=pd.read_csv(FILES["dc_tw"])
#     for df2 in [dt_tw_raw,dc_tw_raw]:
#         for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
#             if c in df2.columns: df2[c]=pd.to_numeric(df2[c],errors="coerce")
#     high_risk,c207_dt,cluster_207,special,top20_games,risk_g,top500=load_risk(dt_tw_raw,dc_tw_raw)

#     # 主玩游戏合并到dt_tw_top
#     df_pref=pd.read_csv(FILES["pref_tw"]); df_pref["阶段汇总"]=df_pref["阶段汇总"].apply(clean)
#     top_game=(df_pref[df_pref["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
#               .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]]
#               .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
#     dt_tw_top=dt_tw_top.merge(top_game,on="账户ID",how="left")

#     print("  ▶ 导出风控名单Excel...")
#     excel_out=outdir/f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
#     with pd.ExcelWriter(str(excel_out),engine="openpyxl") as writer:
#         cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
#         high_risk[[c for c in cols_hr if c in high_risk.columns]].to_excel(writer,sheet_name="高风险用户",index=False)
#         if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(writer,sheet_name="PP轮盘批量账号",index=False)
#         if len(special)>0: special[["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]].to_excel(writer,sheet_name="特殊异常用户",index=False)
#         tool_df.to_excel(writer,sheet_name="道具使用情况",index=False)
#     print(f"✅ 风控名单Excel: {excel_out}")

#     print("  ▶ 构建PDF...")
#     out=outdir/f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
#     doc=SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,topMargin=MARGIN+22,bottomMargin=MARGIN+10,title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
#     story=[]
#     story+=[Spacer(1,3*cm),P("MX 平台数据周报",30,True,C_DARK,TA_CENTER),Spacer(1,0.5*cm),P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),Spacer(1,0.2*cm),P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),Spacer(1,0.5*cm),HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"),PageBreak()]
#     story+=build_overview(K,trend,weekly_ret);                                       story.append(PageBreak())
#     story+=build_agents(agents,agent_ret);                                           story.append(PageBreak())
#     story+=build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw_top);           story.append(PageBreak())
#     story+=build_games_section(mfr,top30,mfr_game_delta);                           story.append(PageBreak())
#     story+=build_activities_section(act_df,total_gift);                              story.append(PageBreak())
#     story+=build_tool_section(tool_df,first_dep_ret);                               story.append(PageBreak())
#     story+=build_risk_section(high_risk,c207_dt,cluster_207,special,top20_games,risk_g,top500,dt_tw_top); story.append(PageBreak())
#     story+=build_conclusion(K,agents,mfr,dep_t,wdraw_t)
#     print("  ▶ 渲染PDF..."); doc.build(story,onFirstPage=header_footer,onLaterPages=header_footer)
#     size=out.stat().st_size/1024; print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
#     return str(out),str(excel_out)

# if __name__=="__main__":
#     if sys.platform!="win32":
#         main(data_root="/mnt/user-data/uploads",output_dir="/mnt/user-data/outputs")
#     else:
#         main(data_root=DATA_ROOT,output_dir=OUTPUT_DIR)

📊 MX 周报 PDF v7.1 生成中...
  ▶ 数据目录：D:\周报更新版\MX
     ✅ [platform       ] 大盘/平台报表_USD_近4周.xlsx
     ✅ [daily          ] 大盘/日报-大盘日报_USD_近4周.xlsx
     ✅ [retention      ] 大盘/整体 首充留存（近7天）_近28天.csv
     ✅ [agent_plat     ] 总代/平台报表-总代_USD_近21天.xlsx
     ✅ [agent_promo    ] 总代/推广报表-总代_USD_近21天.xlsx
     ✅ [agent_ret      ] 总代/首充充值留存_20260424_20260521.csv
     ✅ [vip            ] 用户/VIP报表_USD_近14天.xlsx
     ✅ [dt_tw          ] 用户/top提款用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dt_lw          ] 用户/top提款用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [dc_tw          ] 用户/头部充值用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dc_lw          ] 用户/头部充值用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [pref_tw        ] 用户/本周top500提款用户游戏偏好_全量数据_20260515_20260521.csv
     ✅ [pref_lw        ] 用户/上周top500提款用户游戏偏好_全量数据_20260508_20260514.csv
     ✅ [mfr            ] 游戏/厂商投注数据_全量数据_20260508_20260521.csv
     ✅ [game_tw        ] 游戏/游戏报表-详情_USD_本周.xlsx
     ✅ [game_

In [41]:
"""
MX 平台数据周报生成脚本 v6.0
======================================================
维护指南（五区结构）：
  第一区：周期配置  ← 每周只改这里（路径+日期）
  第二区：文案配置  ← 所有中文解读/建议/预警集中在这里
  第三区：数据加载  ← 改数据逻辑
  第四区：图表      ← 改图表样式
  第五区：章节构建  ← 改PDF布局

快速定位：Ctrl+F 搜 "# ══ 第X区" 直接跳转对应区域
"""

# ════════════════════════════════════════════════════════════════
# 第一区：周期配置  ← 每周只改这里
# ════════════════════════════════════════════════════════════════
from pathlib import Path

DATA_ROOT  = Path(r"D:\周报更新版\MX")       # 数据根目录
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")  # 输出目录（自动创建）
THIS_WEEK  = ("20260515", "20260521")           # 本周 YYYYMMDD
LAST_WEEK  = ("20260508", "20260514")           # 上周 YYYYMMDD
REPORT_END = THIS_WEEK[1]                       # 报告截止日 = 本周最后一天

# ════════════════════════════════════════════════════════════════
# 第二区：文案配置  ← 改解读文字/建议/预警只在这里改
# ════════════════════════════════════════════════════════════════

# 充提差率颜色阈值（>=HIGH 绿色，<LOW 红色，其余黑色）
CR_HIGH = 17
CR_LOW  = 5

# ── 第一章：大盘解读（绿色提示框）──────────────────────────────────
# {变量名} 会被自动替换为实际数值，不用手动填数字
OVERVIEW_INSIGHTS = [
    "充值{tw_充值金额:.1f}万（{pct_充值金额:+.1f}%），公司输赢{tw_公司输赢:.1f}万（{pct_公司输赢:+.1f}%）；推广消耗{pct_真实消耗:+.1f}%。",
    "充提差率{tw_充提差比:.2f}%（上周{lw_充提差比:.2f}%，{delta_充提差比:+.2f}pp）；盈亏率{tw_盈亏率:.3f}%（上周{lw_盈亏率:.3f}%）。",
    "首充人数{tw_首充人数:,}（{pct_首充人数:+.1f}%），首充转化率{tw_首充转化率:.1f}%，首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%）。",
    "首充次日充值留存{tw_首充次日复充率:.1f}%（上周{lw_首充次日复充率:.1f}%，{delta_首充次日复充率:+.1f}pp）。",
]

# 留存图下方黄色提示框
RETENTION_NOTE = [
    "本周首充次日留存{tw_d1:.1f}%，较上周{delta_d1:+.1f}pp；2日留存{tw_d2:.1f}%。",
    "注：本周7日留存因数据截止日期无法完整计算，以上周7日留存{lw_d7:.1f}%作为近期参考基准。",
]

# ── 第二章：总代分析解读 ────────────────────────────────────────────
AGENT_INSIGHTS = [
    "A8_DSP_谷歌原生包充提差率{dsp_cr:.1f}%，注册+{dsp_reg:.1f}%，体量第三且指标双优。",
    "苹果包总代注册+27.1%，充值体量第二，为本周注册增量最大渠道之一。",
    # "YB_FB_PWA_0注册暴跌-79%至1,315人，需持续监控是否恢复。",
]

# ── 第三章：用户分析解读 ────────────────────────────────────────────
VIP_INSIGHTS = [
    "VIP9-11高价值层合计贡献充值{high_vip_pct:.1f}%，高端用户付费意愿持续强劲。",
    "VIP1基础层活跃{tw_vip1_active:,}人（上周{lw_vip1_active:,}），是拉新政策直接反映。",
]

VIP_RET_INSIGHTS = [
    "充值→活跃次日留存：VIP9达{act_v9:.1f}%，VIP10达{act_v10:.1f}%，高VIP活跃粘性强。",
    "充值→充值次日留存：VIP10达{chg_v10_tw:.1f}%（上周{chg_v10_lw:.1f}%），高VIP复充意愿强。",
]

TOP_DEPOSIT_INSIGHTS = [
    "Top10充值用户人均${top10_avg:,.0f}（{top10_avg_chg:+.1f}%），充提差率{top10_cr:.1f}%。",
    "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。",
]

TOP_WITHDRAW_INSIGHTS = [
    "剔除Top100提款用户后，大盘充提差率影响+{top100_impact:.2f}pp，头部提款用户对充提差率有明显拖累。",
    "Top500提款用户活动奖励占资金来源仅{top500_act_pct:.1f}%，主要靠真实赢钱后提款，非活动套利。",
]

# ── 第四章：游戏分析解读 ────────────────────────────────────────────
MFR_INSIGHTS = [
    "Tada投注份额{tada_share:.1f}%（上周{tada_share_lw:.1f}%），Fortune系稳固主导；环比{tada_chg:+.1f}%。",
    "Evolution盈亏率{evol_pl:.2f}%（上周{evol_pl_lw:.2f}%），已转正，较上周大幅改善。",
    "3Oaks盈亏率{oaks_pl:.2f}%，PlayTech{ptech_pl:.2f}%，均高于平台均值，可适当扩大曝光权重。",
]

# ── 第五章：活动分析解读 ────────────────────────────────────────────
ACTIVITY_INSIGHTS = [
    "存款道具合计带动充值${tool_deposit:.1f}万，占总充值{tool_pct:.1f}%。",
    "幸运翻卡道具（71975）充值{xk_deposit:.2f}万（{xk_chg:+.1f}%），注意留存效果是否持续。",
]

# ── 第六章：风险分析解读 ────────────────────────────────────────────
RISK_SCATTER_INSIGHTS = [
    "Top500提款用户中赢家{winner_cnt}人（{winner_pct:.1f}%），平台净赔付${total_loss:.1f}万。",
    "{high_risk_cnt}名超级赢家（公司净输>$10,000）合计导致平台净输${high_risk_loss:.1f}万，"
    "占整体净赔付{high_risk_pct:.1f}%。",
]

PP_ROULETTE_INSIGHTS = [
    "{cluster_cnt}个账户充值$547-548，均获活动奖励$270.92，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。",
    "建议：①冻结账户提款；②审查注册IP/设备指纹；③排查$270.92奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。",
]

RISK_GAME_INSIGHTS = [
    "PP Auto-Roulette 1批量套利模式持续，建议暂停该桌并复核奖励规则。",
    "Fortune Garuda 500（Tada）是Top500提款用户投注规模最大游戏，建议对其RTP参数做紧急复审。",
]

# ── 第七章：总结 ────────────────────────────────────────────────────
# 亮点列表（绿色背景）
HIGHLIGHTS = [
    "充值{tw_充值金额:.1f}万（{pct_充值金额:+.1f}%），充提差{tw_充提差:.1f}万（{pct_充提差:+.1f}%），充提差率{tw_充提差比:.2f}%（{delta_充提差比:+.2f}pp）。",
    "注册人数{tw_注册人数:,}（{pct_注册人数:+.1f}%），首充人数{tw_首充人数:,}（{pct_首充人数:+.1f}%），拉新规模扩大。",
    "A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。",
]

# 风险预警列表（红色背景）
RISKS = [
    "PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。",
    "首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%），首充次日留存{tw_首充次日复充率:.1f}%（{delta_首充次日复充率:+.1f}pp），新用户质量需关注。",
    # "YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。",
]

# 行动建议（优先级 / 事项 / 说明；说明支持 {变量} 替换）
ACTIONS = [
    (
        "[紧急]",
        "处置PP Auto-Roulette 1批量套利",
        "33个207开头账户合计提款$9.4万，平台净赔$6.7万。"
        "冻结账户，审查IP/设备指纹，修复$270.92奖励触发漏洞，对Auto-Roulette 1实施赢额上限。",
    ),
    (
        "[本周]",
        "处置高风险提款用户",
        "账户49903866：提款$39,636，公司输赢-$52,134；"
        "账户206628722：充值$10,229提款$24,855（净提$14,626），高度异常。建议立即人工审核。",
    ),
    (
        "[本周]",
        "优化首充质量与次日激活",
        "首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%），"
        "首充次日留存{tw_首充次日复充率:.1f}%（{delta_首充次日复充率:+.1f}pp）。"
        "建议注册后1h/24h内推送首充引导，落地页突出$10+档。",
    ),
    (
        "[下周]",
        "扩大A8_DSP、A8_GG渠道预算",
        "A8_DSP充提差率17.6%（+1.3pp），一级首充成本$17（上周$21，改善）；"
        "A8_GG充提差率18.9%（+0.3pp）。建议预算增加15-20%，预估带动充值增量$50-70万。",
    ),
    (
        "[下周]",
        "暂停YB_FB_PWA_0",
        "注册量暴跌-79%，仅1,315人，建议暂停投放并排查渠道异常。",
    ),
]


# ── 文案渲染工具（把 {变量} 替换为实际值，缺失时原样保留）──────────
def render(template, **kwargs):
    try:
        return template.format(**kwargs)
    except (KeyError, ValueError):
        return template

def render_list(templates, **kwargs):
    return [render(t, **kwargs) for t in templates]


# ════════════════════════════════════════════════════════════════
# 第三区：依赖库 & 工具 & 数据加载  ← 改数据逻辑在这里
# ════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════
import io, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, PageBreak, HRFlowable, KeepTogether
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

FN, FNB = "WQY", "WQYB"
FILES = {}

# ── v5.5 核心工具：计算留存完整截止日 ─────────────────
def retention_end_date(week_start, lag_days):
    """
    给定周起始日和留存lag，返回在REPORT_END内已完整的最后一天。
    逻辑：首充日 + lag_days <= REPORT_END
          → 首充日 <= REPORT_END - lag_days
    """
    end_dt  = datetime.strptime(REPORT_END, "%Y%m%d")
    last_dt = end_dt - timedelta(days=lag_days)
    last    = last_dt.strftime("%Y%m%d")
    return last if last >= week_start else None

# ── 字体 ─────────────────────────────────────
def find_chinese_font():
    import platform
    sys_name = platform.system()
    if sys_name == "Windows":
        candidates = [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\msyhbd.ttc",
                      r"C:\Windows\Fonts\simsun.ttc", r"C:\Windows\Fonts\simhei.ttf"]
    elif sys_name == "Darwin":
        candidates = ["/System/Library/Fonts/PingFang.ttc"]
    else:
        candidates = ["/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
                      "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc"]
    for p in candidates:
        if Path(p).exists(): return p
    try:
        from matplotlib import font_manager as fm
        for f in fm.fontManager.ttflist:
            if any(k in f.name for k in ["Heiti","Hei","YaHei","SimSun","SimHei","WenQuanYi","Noto Sans CJK"]):
                if Path(f.fname).exists(): return f.fname
    except Exception: pass
    raise FileNotFoundError("未找到中文字体！请手动设置 FONT_PATH")

def setup():
    global FONT_PATH
    FONT_PATH = find_chinese_font()
    print(f"  ▶ 使用字体：{FONT_PATH}")
    is_ttc = FONT_PATH.lower().endswith(".ttc")
    if is_ttc:
        pdfmetrics.registerFont(TTFont(FN, FONT_PATH, subfontIndex=0))
        try:   pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1))
        except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0))
    else:
        pdfmetrics.registerFont(TTFont(FN, FONT_PATH))
        pdfmetrics.registerFont(TTFont(FNB, FONT_PATH))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(FONT_PATH)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

# ── 文件搜索 ─────────────────────────────────
def _resolve_files(root):
    global FILES
    kw_map = {
        "platform":    "平台报表_USD_近4周",
        "daily":       "日报-大盘日报_USD_近4周",
        "retention":   "首充留存",
        "agent_plat":  "平台报表-总代_USD_近21天",
        "agent_promo": "推广报表-总代_USD_近21天",
        "agent_ret":   "首充充值留存",
        "vip":         "VIP报表_USD",
        "dt_tw":       f"top提款用户_全量数据_{THIS_WEEK[0]}",
        "dt_lw":       f"top提款用户_全量数据_{LAST_WEEK[0]}",
        "dc_tw":       f"头部充值用户_全量数据_{THIS_WEEK[0]}",
        "dc_lw":       f"头部充值用户_全量数据_{LAST_WEEK[0]}",
        "pref_tw":     f"本周top500提款用户游戏偏好_全量数据_{THIS_WEEK[0]}",
        "pref_lw":     f"上周top500提款用户游戏偏好_全量数据_{LAST_WEEK[0]}",
        "mfr":         "厂商投注数据_全量数据",
        "game_tw":     "游戏报表-详情_USD_本周",
        "game_lw":     "游戏报表-详情_USD_上周",
        "gift":        f"各赠送活动_全量数据_{THIS_WEEK[0]}",
        "deposit_src": f"整体存款_全量数据_{THIS_WEEK[0]}",
        "tool_map":    "道具对应活动",
        "vip_ret_chg": "VIP充值-充值_近28天",
        "vip_ret_act": "VIP充值-活跃_近28天",
    }
    print(f"  ▶ 数据目录：{root}")
    fail = 0
    for key, kw in kw_map.items():
        hits = [h for h in root.rglob("*")
                if h.is_file() and not h.name.startswith("~$") and kw in h.name]
        if not hits:
            print(f"     ❌ [{key:12s}] 找不到含 '{kw}' 的文件"); fail += 1
        else:
            p = max(hits, key=lambda h: h.stat().st_mtime)
            FILES[key] = p
            print(f"     ✅ [{key:12s}] {p.parent.name}/{p.name}")
    if fail:
        raise FileNotFoundError(f"\n共 {fail} 个文件未找到，请检查 DATA_ROOT = {root}")
    print(f"  ▶ 全部 {len(kw_map)} 个文件匹配成功\n")

# ── 颜色 ─────────────────────────────────────
C_BLUE   = colors.HexColor("#1d4ed8")
C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669")
C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706")
C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b")
C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white
C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1")
C_ROW    = colors.HexColor("#f8fafc")

PW, PH = A4
MARGIN  = 1.8*cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
TRUNCATE_RULES = {
    "首充次日复充率": 1, "首充次日复投率": 1, "首充当日复充率": 0,
    "首充7日复充率": 6, "首充30日复充率": 999,
    "首充2日复充率": 1, "首充3日复充率": 2,
}

# ── 工具函数 ─────────────────────────────────
def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c != "日期" and c not in ["总代.名称","name_总代","总代.ID"]]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o): return (n-o)/abs(o)*100 if o and o!=0 else 0.0
def pct_vec(ns, os): return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop>0 else s
    nz = v[v>0]; return nz.mean() if len(nz)>0 else v.mean()
def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

# ── 样式函数 ─────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.38,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),
                                ("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5),
                         HRFlowable(width="100%", thickness=1.5, color=C_BLUE2),
                         Spacer(1,3), P(f"■  {text}", 10, True, C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}", 8.5, False, C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8):
    full = PW-2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),        ("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2.5),     ("BOTTOMPADDING",(0,0),(-1,-1),2.5),
        ("LEFTPADDING",(0,0),(-1,-1),3),       ("RIGHTPADDING",(0,0),(-1,-1),3),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),  ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    hrow = [P(h, fsize, True, C_WHITE, TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]), fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c, (list,tuple)) else P(str(c), fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t, bold=False, clr=colors.black, align=TA_LEFT): return (t, bold, clr, align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup = good_up
    s = "+" if v>=0 else ""; c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{s}{v:.{d}f}%", False, c, TA_RIGHT)
def gclr(v, t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)

def kpi_card4(items, cols=4):
    full = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,gup in items:
        s = "+" if chg>=0 else ""
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([
            [P(label, 7.5, False, C_GRAY)],
            [P(str(tv), 14, True, C_DARK)],
            [P(f"上周：{lv}", 7.5, False, C_GRAY)],
            [P(f"{s}{chg:.1f}%", 8, True, pclr)],
        ], colWidths=[full], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),C_LGRAY), ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),8),
            ("TOPPADDING",(0,0),(-1,-1),5), ("BOTTOMPADDING",(0,0),(-1,-1),5),
        ]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(full,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════
def load_platform():
    df = pd.read_excel(FILES["platform"])
    df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]
    lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"])
    dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]
    lw_d = dd[dd["日期"].between(*LAST_WEEK)]
    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
              "首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE_RULES.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop)
        K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    K["tw_真实消耗"] = tw_d["真实消耗"].iloc[:-1].sum() if len(tw_d)>1 else tw_d["真实消耗"].sum()
    K["lw_真实消耗"] = lw_d["真实消耗"].sum()
    K["pct_真实消耗"] = pct(K["tw_真实消耗"], K["lw_真实消耗"])
    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])
    last14 = df.tail(14)
    trend = {
        "dates":    [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
        "充值":     last14["充值金额"].tolist(), "提现":  last14["提现金额"].tolist(),
        "充提差比": last14["充提差比"].tolist(), "公司输赢": last14["公司输赢"].tolist(),
        "首充":     last14["首充人数"].tolist(), "注册":  last14["注册人数"].tolist(),
    }
    return K, trend

def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
    daily["date_str"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
    daily_r = daily[daily["指标"]=="留存率"].copy()
    daily_u = daily[daily["指标"]=="留存人数"].copy()
    for c in ["第1日","第2日","第3日","第7日"]: daily_r[c] = daily_r[c].apply(clean)
    def _d(s): return datetime.strptime(s, "%Y%m%d")
    def _fmt(d): return d.strftime("%Y-%m-%d")
    def _lbl(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s=_d(THIS_WEEK[0]); tw_e=_d(THIS_WEEK[1])
    lw_s=_d(LAST_WEEK[0]); lw_e=_d(LAST_WEEK[1])
    w2_end=lw_s-timedelta(days=1); w2_start=w2_end-timedelta(days=6)
    w1_end=w2_start-timedelta(days=1); w1_start=w1_end-timedelta(days=6)
    weeks = [
        (f"第1周\n{_lbl(w1_start,w1_end)}", _fmt(w1_start), _fmt(w1_end)),
        (f"第2周\n{_lbl(w2_start,w2_end)}", _fmt(w2_start), _fmt(w2_end)),
        (f"上周\n{_lbl(lw_s,lw_e)}",        _fmt(lw_s),     _fmt(lw_e)),
        (f"本周\n{_lbl(tw_s,tw_e)}",        _fmt(tw_s),     _fmt(tw_e)),
    ]
    result = []
    for wk,s,e in weeks:
        mr = daily_r[(daily_r["date_str"]>=s)&(daily_r["date_str"]<=e)]
        mu = daily_u[(daily_u["date_str"]>=s)&(daily_u["date_str"]<=e)]
        users = mu["充值成功事件用户数"].sum(); row = {"week": wk, "users": users}
        for col in ["第1日","第2日","第3日","第7日"]:
            rs = mr[col].values; us = mu["充值成功事件用户数"].values
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = np.average(rs[v], weights=us[v]) if v.sum()>0 else np.nan
            else:
                row[col] = np.nanmean(rs) if len(rs)>0 else np.nan
        result.append(row)
    return result

def load_agents():
    df_p = pd.read_excel(FILES["agent_plat"])
    df_r = pd.read_excel(FILES["agent_promo"])
    df_p["日期"] = df_p["日期"].astype(str); df_r["日期"] = df_r["日期"].astype(str)
    df_p = to_num(df_p); df_r = to_num(df_r)
    df_p = df_p[df_p["总代.ID"]!=0]; df_r = df_r[df_r["总代.ID"]!=0]
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]; lw_p = df_p[df_p["日期"].between(*LAST_WEEK)]
    tw_r = df_r[df_r["日期"].between(*THIS_WEEK)]; lw_r = df_r[df_r["日期"].between(*LAST_WEEK)]
    sum_a = ["充值金额","提现金额","充提差","首充金额","首充人数","注册人数",
             "充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        g = d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sum_a if c in d.columns}).reset_index()
        for c in ["首充转化率","充提差比","首充次日充值留存","活跃用户付费率"]:
            if c in d.columns:
                rm = d.groupby("总代.ID")[c].mean().rename(c); g = g.merge(rm, on="总代.ID", how="left")
        g["充提差率"] = g["充提差"]/g["充值金额"]*100; return g
    tw_pa = agg_p(tw_p); lw_pa = agg_p(lw_p)
    sum_r = ["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        g = d.groupby("总代.ID").agg({c:"sum" for c in sum_r if c in d.columns}).reset_index()
        g["注册成本"] = g["总消耗"]/g["注册人数"].replace(0,np.nan)
        g["一级首充成本"] = g["总消耗"]/g["一级首充人数"].replace(0,np.nan); return g
    tw_ra = agg_r(tw_r); lw_ra = agg_r(lw_r)
    lw_ra["lw_一级首充成本"] = lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
    m = tw_pa.merge(
        lw_pa[["总代.ID","充值金额","注册人数","充提差率","首充次日充值留存"]].rename(
            columns={"充值金额":"lw_充值","注册人数":"lw_注册",
                     "充提差率":"lw_充提差率","首充次日充值留存":"lw_次日留存"}),
        on="总代.ID", how="left")
    m = m.merge(tw_ra[["总代.ID","总消耗","注册成本","一级首充成本","一级首充人数"]], on="总代.ID", how="left")
    m = m.merge(lw_ra[["总代.ID","lw_一级首充成本"]], on="总代.ID", how="left")
    m["注册环比"] = pct_vec(m["注册人数"], m["lw_注册"])
    m["充值环比"] = pct_vec(m["充值金额"], m["lw_充值"])
    return m.sort_values("充值金额", ascending=False).head(15)

# ════════════════════════════════════════════
# v5.5 核心修复：总代留存加载（剔除未到期天数）
# ════════════════════════════════════════════
def load_agent_ret():
    """
    CSV 格式：初始事件发生时间 / 总代 / name_总代 / 首充用户数 / 指标 / 1日 / 3日 / 7日...

    v5.5 关键修复：
    ─────────────────────────────────────────────────────────
    留存第N日的含义：首充日+N天后仍活跃的比例。
    若报告截止日=THIS_WEEK[1]，则：
      首充日 + N <= THIS_WEEK[1]
      → 首充日 <= THIS_WEEK[1] - N天

    本周3日留存：只取 THIS_WEEK[0] ~ (REPORT_END-3天)
      本例：20260515 ~ 20260518（5/19、5/20、5/21三天3日未到期，剔除）
    本周1日留存：只取 THIS_WEEK[0] ~ (REPORT_END-1天) = 20260520
    上周7日留存：只取 LAST_WEEK[0] ~ (REPORT_END-7天) = 20260514（全部完整）
    上周1日留存：只取 LAST_WEEK[0] ~ (REPORT_END-1天) = 20260520（全部完整）
    ─────────────────────────────────────────────────────────
    返回：dict { name_总代 -> {tw_d1, tw_d3, lw_d1, lw_d7, 总代ID} }
    """
    df = pd.read_csv(FILES["agent_ret"])

    # 解析日期列
    df["_yyyymmdd"] = (df["初始事件发生时间"].astype(str)
                       .str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
                       .str.replace("-",""))

    # 数值化留存率（去掉 %）
    for c in ["1日","2日","3日","7日"]:
        df[c] = (df[c].astype(str).str.replace("%","").str.strip()
                     .pipe(pd.to_numeric, errors="coerce"))

    # 首充用户数
    df["首充用户数"] = pd.to_numeric(df["首充用户数"], errors="coerce").fillna(0)

    # 总代ID（用于汇总表）
    df["总代"] = pd.to_numeric(df["总代"], errors="coerce")

    # 只保留每日留存率行
    daily = df[
        df["_yyyymmdd"].notna() &
        df["_yyyymmdd"].str.match(r"^\d{8}$") &
        (df["指标"] == "留存率")
    ].copy()

    # ── 计算各留存指标的完整截止日 ──────────────────
    tw_d1_end = retention_end_date(THIS_WEEK[0], 1)   # 本周1日: ~20260520
    tw_d3_end = retention_end_date(THIS_WEEK[0], 3)   # 本周3日: ~20260518 ★关键修复
    lw_d1_end = retention_end_date(LAST_WEEK[0], 1)   # 上周1日: 全部完整
    lw_d7_end = retention_end_date(LAST_WEEK[0], 7)   # 上周7日: 全部完整

    print(f"     留存完整截止日 → 本周1日:{tw_d1_end} 本周3日:{tw_d3_end} "
          f"上周1日:{lw_d1_end} 上周7日:{lw_d7_end}")

    # 分别筛选各时间窗口
    tw_all = daily[daily["_yyyymmdd"].between(*THIS_WEEK)]
    lw_all = daily[daily["_yyyymmdd"].between(*LAST_WEEK)]

    def _win(df_week, week_start, end_cap):
        if end_cap is None: return df_week.iloc[0:0]  # 空
        return df_week[(df_week["_yyyymmdd"] >= week_start) &
                       (df_week["_yyyymmdd"] <= end_cap)]

    tw_d1_df = _win(tw_all, THIS_WEEK[0], tw_d1_end)
    tw_d3_df = _win(tw_all, THIS_WEEK[0], tw_d3_end)
    lw_d1_df = _win(lw_all, LAST_WEEK[0], lw_d1_end)
    lw_d7_df = _win(lw_all, LAST_WEEK[0], lw_d7_end)

    def _wavg_all(df_sub, col):
        """对所有总代做首充用户数加权平均，返回 {name_总代: value}"""
        result = {}
        for nm, grp in df_sub.groupby("name_总代"):
            valid = grp[grp[col].notna() & (grp["首充用户数"] > 0)]
            if len(valid) > 0:
                result[nm] = float(np.average(valid[col], weights=valid["首充用户数"]))
            else:
                result[nm] = np.nan
        return result

    tw_d1 = _wavg_all(tw_d1_df, "1日")
    tw_d3 = _wavg_all(tw_d3_df, "3日")
    lw_d1 = _wavg_all(lw_d1_df, "1日")
    lw_d7 = _wavg_all(lw_d7_df, "7日")

    # 总代ID映射（取第一条）
    id_map = {}
    for nm, grp in daily.groupby("name_总代"):
        id_val = grp["总代"].dropna().values
        if len(id_val) > 0:
            id_map[nm] = int(id_val[0])

    all_names = set(tw_d1) | set(lw_d1)
    ret = {}
    for nm in all_names:
        ret[nm] = {
            "tw_d1": tw_d1.get(nm, np.nan),
            "tw_d3": tw_d3.get(nm, np.nan),
            "lw_d1": lw_d1.get(nm, np.nan),
            "lw_d7": lw_d7.get(nm, np.nan),
            "agent_id": id_map.get(nm, None),
        }
    return ret

def load_vip():
    df = pd.read_excel(FILES["vip"]); df["日期"] = df["日期"].astype(str)
    num = ["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df = to_num(df, num)
    tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

def load_vip_retention():
    results = {}
    for fkey, rtype in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df = pd.read_csv(FILES[fkey])
        df['date_str'] = df['初始事件发生时间'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')
        df['yyyymmdd'] = df['date_str'].str.replace('-','')
        for c in ['1日','2日','3日','7日']: df[c] = df[c].apply(clean)
        df['n']   = pd.to_numeric(df['充值成功事件用户数'], errors='coerce')
        df['vip'] = pd.to_numeric(df['vip_level'], errors='coerce')
        stage = df[(df['指标']=='留存率')&(df['初始事件发生时间']=='阶段值')].copy()
        stage_dict = {int(r['vip']):{c:r[c] for c in ['1日','2日','3日','7日']}
                      for _,r in stage.iterrows() if not pd.isna(r['vip'])}
        daily = df[(df['指标']=='留存率')&df['yyyymmdd'].notna()&df['vip'].notna()]
        tw = daily[daily['yyyymmdd'].between(*THIS_WEEK)]
        lw = daily[daily['yyyymmdd'].between(*LAST_WEEK)]
        def wavg(d,col):
            res={}
            for vip,g in d.groupby('vip'):
                v=g[g[col].notna()]
                if len(v)>0: res[int(vip)]=np.average(v[col],weights=v['n'])
            return res
        results[rtype] = {
            'tw':{c:wavg(tw,c) for c in ['1日','2日','3日','7日']},
            'lw':{c:wavg(lw,c) for c in ['1日','2日','3日','7日']},
            'stage':stage_dict,
        }
    return results

def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
    for df in [dc_tw,dc_lw,dt_tw,dt_lw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
    tiers = [1,10,50,100,200,500]
    df_p = pd.read_excel(FILES["platform"]); df_p["日期"]=df_p["日期"].astype(str); df_p=to_num(df_p)
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]
    total_chg=tw_p["充值金额"].sum(); total_tx=tw_p["提现金额"].sum()
    actual_cr=(total_chg-total_tx)/total_chg*100
    dep_tiers,wdraw_tiers=[],[]
    total_tw_c=dc_tw["充值金额"].sum(); total_tw_t=dt_tw["提款金额"].sum()
    for t in tiers:
        tw=dc_tw.head(t); lw=dc_lw.head(t)
        tw_c=tw["充值金额"].sum(); tw_tx=tw["提款金额"].sum()
        lw_c=lw["充值金额"].sum(); lw_tx=lw["提款金额"].sum()
        dep_tiers.append({"tier":f"Top{t}","tw_chg":tw_c,"lw_chg":lw_c,
            "tw_avg":tw_c/t,"lw_avg":lw_c/t,
            "tw_cr":(tw_c-tw_tx)/tw_c*100 if tw_c>0 else 0,
            "lw_cr":(lw_c-lw_tx)/lw_c*100 if lw_c>0 else 0,
            "tw_win":tw["公司输赢"].sum(),"占全量":tw_c/total_tw_c*100})
        twd=dt_tw.head(t); lwd=dt_lw.head(t)
        tw_t2=twd["提款金额"].sum(); tw_c2=twd["充值金额"].sum()
        lw_t2=lwd["提款金额"].sum(); lw_c2=lwd["充值金额"].sum()
        tw_cr2=(tw_c2-tw_t2)/tw_c2*100 if tw_c2>0 else 0
        lw_cr2=(lw_c2-lw_t2)/lw_c2*100 if lw_c2>0 else 0
        winners=(twd["公司输赢"]<0).sum()
        act=twd["活动奖励"].sum()/(tw_c2+twd["活动奖励"].sum())*100
        excl_chg=total_chg-tw_c2; excl_tx=total_tx-tw_t2
        excl_cr=(excl_chg-excl_tx)/excl_chg*100 if excl_chg>0 else 0
        impact=excl_cr-actual_cr
        wdraw_tiers.append({"tier":f"Top{t}","tw_tx":tw_t2,"lw_tx":lw_t2,
            "tw_avg":tw_t2/t,"lw_avg":lw_t2/t,"tw_cr":tw_cr2,"lw_cr":lw_cr2,
            "赢家":winners,"总数":t,"赢家率":winners/t*100,
            "活动占比":act,"占全量":tw_t2/total_tw_t*100,"大盘影响":impact})
    return dep_tiers,wdraw_tiers,dc_tw.head(200),dt_tw.head(20)

def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); df_lw=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); df_lw["阶段汇总"]=df_lw["阶段汇总"].apply(clean)
    bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]
    bet_lw=df_lw[df_lw["分析指标"]=="投注金额"]
    g_bet=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
    g_win=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
    g_lw=bet_lw.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
    g_usr=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([g_bet,g_win,g_lw,g_usr],axis=1).reset_index()
    total=g["本周投注"].sum(); g["占比"]=g["本周投注"]/total*100
    g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr

def load_games():
    df_tw=pd.read_excel(FILES["game_tw"]); df_lw=pd.read_excel(FILES["game_lw"])
    for df in [df_tw,df_lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢","人均投注局数","人均投注金额"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    total_tw=df_tw["投注金额"].sum()
    g_tw=df_tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    g_lw=df_lw.groupby("游戏.名称").agg(
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    g_tw["人均局数"]=g_tw["投注局数"]/g_tw["投注人数"]
    g_tw["人均金额"]=g_tw["投注金额"]/g_tw["投注人数"]
    g_tw["盈亏率"]=g_tw["公司输赢"]/g_tw["投注金额"]*100
    g_tw["占比"]=g_tw["投注金额"]/total_tw*100
    gm=g_tw.merge(g_lw,on="游戏.名称",how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)

def load_mfr():
    df=pd.read_csv(FILES["mfr"])
    df=df[df["时间"]!="阶段汇总"].copy()
    df["时间"]=df["时间"].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢"]: df[c]=df[c].apply(clean)
    tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
    def agg(d):
        g=d.groupby("show_name_厂商标签id").agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
    tw_g=agg(tw); lw_g=agg(lw)
    mg=tw_g.merge(lw_g[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on="show_name_厂商标签id",how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    return mg.sort_values("投注金额",ascending=False)

def load_activities():
    df=pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数"]: df[c]=df[c].apply(clean)
    agg=df.groupby("name_账变opt_id").agg(
        赠送金额=("赠送金额","sum"),lw_赠送=("赠送金额.1","sum"),赠送人数=("赠送人数","sum")).reset_index()
    agg["环比"]=(agg["赠送金额"]-agg["lw_赠送"])/agg["lw_赠送"].abs()*100
    total=agg["赠送金额"].sum(); agg["占比"]=agg["赠送金额"]/total*100; agg["人均"]=agg["赠送金额"]/agg["赠送人数"]
    return agg.sort_values("赠送金额",ascending=False).head(15), total

def load_deposit_src():
    df=pd.read_csv(FILES["deposit_src"])
    for c in ["充值金额","充值金额.1","充值人数"]: df[c]=df[c].apply(clean)
    tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str)
    df["道具ID"]=df["道具ID"].astype(str)
    agg=df.groupby("道具ID").agg(充值金额=("充值金额","sum"),lw_充值=("充值金额.1","sum"),充值人数=("充值人数","sum")).reset_index()
    agg=agg.merge(tm,on="道具ID",how="left")
    nm={"(null)":"无道具直接充值","71872":"幸运转盘10%","71873":"幸运转盘20%",
        "71975":"幸运翻卡道具","71894":"新人签到10%","71871":"新人签到20%"}
    agg["名称"]=agg.apply(lambda r: nm.get(str(r["道具ID"]),
        str(r.get("备注",""))[:12] if pd.notna(r.get("备注")) else f"其他({str(r['道具ID'])[:8]})"), axis=1)
    total=agg["充值金额"].sum(); agg["占比"]=agg["充值金额"]/total*100
    agg["环比"]=(agg["充值金额"]-agg["lw_充值"])/agg["lw_充值"].abs()*100
    return agg.sort_values("充值金额",ascending=False).head(6), total

def load_risk():
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    dt_tw=pd.read_csv(FILES["dt_tw"]); dc_tw=pd.read_csv(FILES["dc_tw"])
    for df2 in [dt_tw,dc_tw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df2.columns: df2[c]=pd.to_numeric(df2[c],errors="coerce")
    bet_amt=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    win_pref=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    bet_cnt=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数")
    top_game=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
              .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
              .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏","阶段汇总":"主游戏投注额"}).reset_index())
    ug=pd.concat([bet_cnt,bet_amt,win_pref],axis=1).reset_index()
    ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]; ug=ug.merge(top_game,on="账户ID",how="left")
    top500=dt_tw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0; top500["投充比"]=top500["投注金额"]/top500["充值金额"]
    high_risk=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
    pp_ids=df[(df["游戏名称"]=="Auto-Roulette 1")&(df["分析指标"]=="投注次数")]["账户ID"].unique()
    cluster_207=[u for u in pp_ids if str(u).startswith("207")]
    c207_dt=dt_tw[dt_tw["账户ID"].isin(cluster_207)].sort_values("提款金额",ascending=False)
    special=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False)
    bet_g=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
    win_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
    usr_g=df[df["分析指标"]=="投注次数"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
    winr_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bet_g.merge(win_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(usr_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(winr_g,on=["show_name_厂商标签id","游戏名称"],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100; g["人均投注额"]=g["投注金额"]/g["玩家数"]
    risk_g=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    return high_risk,c207_dt,cluster_207,special,g.sort_values("投注金额",ascending=False).head(20),risk_g,top500

# ════════════════════════════════════════════
# 图表（与 v5.4 相同，保持不变）
# ════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════
# 第四区：图表  ← 改图表标题/颜色/尺寸在这里
# ════════════════════════════════════════════════════════════════

SPLIT=7
def _vline(ax,dates):
    ax.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax.text(SPLIT-4,ax.get_ylim()[1]*0.93,"上周",ha="center",fontsize=7,color="#64748b")
    ax.text(SPLIT+3,ax.get_ylim()[1]*0.93,"本周",ha="center",fontsize=7,color="#1d4ed8")

def chart_trend(trend):
    fig=plt.figure(figsize=(16,10),facecolor="white")
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.45,wspace=0.3)
    dates=trend["dates"]; x=range(len(dates))
    def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=0.25)
    ax=fig.add_subplot(gs[0,0])
    clrs=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT
    ax.bar(x,[v/10000 for v in trend["充值"]],color=clrs,width=0.7,label="充值")
    ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
    ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold"); ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
    ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax,dates)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax2=fig.add_subplot(gs[0,1])
    ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=0.7)
    ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax2); _vline(ax2,dates)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax3=fig.add_subplot(gs[1,0])
    ax3.bar(x,trend["首充"],color=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT,width=0.7,label="首充人数")
    ax3r=ax3.twinx(); ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold"); ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
    l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left"); ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=0.25); ax3.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax4=fig.add_subplot(gs[1,1])
    ax4.bar(x,trend["充提差比"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=0.7)
    tw_m=np.mean(trend["充提差比"][SPLIT:]); lw_m=np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tw_m,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lw_m,color="#94a3b8",ls=":",lw=1.2)
    ax4.text(len(dates)-0.5,tw_m+0.3,f"本周均{tw_m:.1f}%",fontsize=7,color="#7c3aed",ha="right")
    ax4.text(0.5,lw_m+0.3,f"上周均{lw_m:.1f}%",fontsize=7,color="#64748b",ha="left")
    ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold"); ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax4); _vline(ax4,dates)
    ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
    plt.tight_layout(rect=[0,0,1,0.98]); return fig_img(fig,16,11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white")
    cols_p=[("第1日","#1d4ed8"),("第2日","#059669"),("第3日","#d97706"),("第7日","#7c3aed")]
    x=np.arange(len(weeks)); w=0.18
    for i,(col,clr) in enumerate(cols_p):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-1.5*w,vals,w,label=col,color=clr,alpha=0.85)
        for bar,v in zip(bars,vals):
            if not np.isnan(v): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周加权平均对比",fontsize=10,fontweight="bold"); ax.legend(fontsize=8,loc="upper right")
    ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=0.5); ax.text(2.6,ax.get_ylim()[1]*0.85,"↑本周",fontsize=7.5,color="#ef4444")
    ax.grid(axis="y",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.text(0.5,-0.02,f"注：本周({THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]})7日留存数据未完整",ha="center",fontsize=7.5,color="#64748b",style="italic")
    plt.tight_layout(); return fig_img(fig,16,5.5)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=0.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tw_c=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_c=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tw_c,w,label="本周",color="#1d4ed8",alpha=0.85); ax1.bar(x+w/2,lw_c,w,label="上周",color="#93c5fd",alpha=0.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tw_b=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_b=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tw_b,w,label="本周",color="#059669",alpha=0.85); ax2.bar(x+w/2,lw_b,w,label="上周",color="#6ee7b7",alpha=0.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=0.85,width=0.6); ax3.axhline(0,color="black",lw=0.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=0.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vip_ret):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    chg_tw=[vip_ret['chg']['tw']['1日'].get(v,0) for v in vips]; chg_lw=[vip_ret['chg']['lw']['1日'].get(v,0) for v in vips]; chg_3d=[vip_ret['chg']['tw']['3日'].get(v,0) for v in vips]
    act_tw=[vip_ret['act']['tw']['1日'].get(v,0) for v in vips]; act_lw=[vip_ret['act']['lw']['1日'].get(v,0) for v in vips]; act_3d=[vip_ret['act']['tw']['3日'].get(v,0) for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=0.28
    ax1.bar(x-w,chg_tw,w,label="次日留存(本周)",color="#1d4ed8",alpha=0.85); ax1.bar(x,chg_lw,w,label="次日留存(上周)",color="#93c5fd",alpha=0.7); ax1.bar(x+w,chg_3d,w,label="3日留存(本周)",color="#059669",alpha=0.75)
    ax1.set_title("充值→充值 留存率（%）",fontsize=10,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,fontsize=8); ax1.legend(fontsize=7.5,loc="upper left"); ax1.grid(axis="y",alpha=0.25); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    ax2.bar(x-w,act_tw,w,label="次日留存(本周)",color="#7c3aed",alpha=0.85); ax2.bar(x,act_lw,w,label="次日留存(上周)",color="#c4b5fd",alpha=0.7); ax2.bar(x+w,act_3d,w,label="3日留存(本周)",color="#d97706",alpha=0.75)
    ax2.set_title("充值→活跃 留存率（%）",fontsize=10,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,fontsize=8); ax2.legend(fontsize=7.5,loc="upper left"); ax2.grid(axis="y",alpha=0.25); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.suptitle("VIP各等级充值留存率（本周 vs 上周，加权日均值）",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10); names=top10["show_name_厂商标签id"].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    shares=top10["占比"].tolist(); lw_s=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,shares,color=["#059669" if c>=l else "#dc2626" for c,l in zip(shares,lw_s)],alpha=0.85,width=0.6)
    ax1.plot(names,lw_s,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,shares,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,f"{s:.1f}%",ha="center",fontsize=7); ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额（本周 vs 上周）",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=0.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=0.85); ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=0.7)
    ax2.axhline(0,color="black",lw=0.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5); ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    top15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in top15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in top15.iterrows()]; lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in top15.iterrows()]
    x=np.arange(len(names)); w=0.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=0.85); ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=0.7)
    for bar,v in zip(ax.patches[:len(names)],bets[::-1]): ax.text(bar.get_width()+0.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8); ax.set_title("Top15游戏 投注金额（万USD，本周 vs 上周）",fontsize=9,fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="x",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pref_game,mfr_amt):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=mfr_amt.index[:8].tolist(); vals=mfr_amt.values[:8].tolist(); total=sum(vals); pcts=[v/total*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.05,-0.18)); ax1.set_title("Top500提款用户 厂商偏好（按投注金额）",fontsize=9,fontweight="bold")
    top12=pref_game.head(12); gnames=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in top12.iterrows()]; gvals=[r["本周投注"]/10000 for _,r in top12.iterrows()]; gclrs=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in top12.iterrows()]
    ax2.barh(range(len(gnames))[::-1],gvals,color=gclrs[::-1],alpha=0.85); ax2.set_yticks(range(len(gnames))); ax2.set_yticklabels(gnames,fontsize=7.5); ax2.set_title("偏好游戏Top12（按投注金额，红=平台亏损）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act_df):
    top10=act_df.head(10); names=[str(r["name_账变opt_id"])[:10] for _,r in top10.iterrows()]; vals=[r["赠送金额"]/10000 for _,r in top10.iterrows()]; lw=[r["lw_赠送"]/10000 for _,r in top10.iterrows()]; envs=[r["环比"] for _,r in top10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=0.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=0.85); ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=0.7); ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8); ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold"); ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax2.barh(range(len(names)),envs[::-1],color=["#059669" if v>0 else "#dc2626" for v in envs[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8); ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_deposit_src(src_df,total):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); names=src_df["名称"].tolist(); vals=src_df["充值金额"].tolist(); lw_v=src_df["lw_充值"].tolist(); pcts=src_df["占比"].tolist(); clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>3 else "",startangle=140,pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:8]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.1,-0.15)); ax1.set_title(f"存款来源占比（总计${total/10000:.0f}万USD）",fontsize=9,fontweight="bold")
    x=np.arange(len(names)); w=0.35; ax2.bar(x-w/2,[v/10000 for v in vals],w,label="本周",color=clrs,alpha=0.85); ax2.bar(x+w/2,[v/10000 for v in lw_v],w,label="上周",color=["#93c5fd"]*len(names),alpha=0.6); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=25,ha="right",fontsize=7.5); ax2.set_title("存款道具 本周 vs 上周（万USD）",fontsize=9,fontweight="bold"); ax2.legend(fontsize=8); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_risk_scatter(top500):
    valid=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy(); valid=valid[valid["投充比"]<200]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    ax.scatter(valid["投充比"],valid["公司输赢"]/10000,c=["#dc2626" if v<0 else "#059669" for v in valid["公司输赢"]],s=[min(abs(v)/500+20,200) for v in valid["公司输赢"]],alpha=0.55,edgecolors="none")
    ax.axhline(0,color="black",lw=0.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=0.7); ax.text(21,ax.get_ylim()[0]*0.9,"投充比=20x",fontsize=7.5,color="#d97706")
    for _,r in top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()].iterrows():
        ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),xytext=(r["投充比"]+5,r["公司输赢"]/10000-0.2),fontsize=6.5,color="#991b1b",arrowprops=dict(arrowstyle="->",color="#991b1b",lw=0.7))
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8); ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=0.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=0.6,label="平台赢钱")],fontsize=8,loc="upper right"); ax.grid(alpha=0.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(risk_g):
    top12=risk_g.head(12); names=[r["游戏名称"][:16] for _,r in top12.iterrows()]; losses=[abs(r["公司输赢"]) for _,r in top12.iterrows()]; rates=[r["盈亏率"] for _,r in top12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=0.85)
    for bar,v in zip(ax1.patches,losses[::-1]): ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)

# ════════════════════════════════════════════
# 章节构建
# ════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════
# 第五区：章节构建  ← 改PDF表格列宽/布局在这里
# ════════════════════════════════════════════════════════════════

def build_overview(K,trend,weekly_ret):
    S=[sec_title("一、大盘核心数据")]
    kpis=[
        ("充值金额",f"{K['tw_充值金额']/10000:.1f}万",f"{K['lw_充值金额']/10000:.1f}万",K["pct_充值金额"],True),
        ("提现金额",f"{K['tw_提现金额']/10000:.1f}万",f"{K['lw_提现金额']/10000:.1f}万",K["pct_提现金额"],False),
        ("充提差",f"{K['tw_充提差']/10000:.1f}万",f"{K['lw_充提差']/10000:.1f}万",K["pct_充提差"],True),
        ("充提差率",f"{K['tw_充提差比']:.2f}%",f"{K['lw_充提差比']:.2f}%",K["pct_充提差比"],True),
        ("公司输赢",f"{K['tw_公司输赢']/10000:.1f}万",f"{K['lw_公司输赢']/10000:.1f}万",K["pct_公司输赢"],True),
        ("盈亏率",f"{K['tw_盈亏率']:.3f}%",f"{K['lw_盈亏率']:.3f}%",K["pct_盈亏率"],True),
        ("注册人数",f"{int(K['tw_注册人数']):,}",f"{int(K['lw_注册人数']):,}",K["pct_注册人数"],True),
        ("首充人数",f"{int(K['tw_首充人数']):,}",f"{int(K['lw_首充人数']):,}",K["pct_首充人数"],True),
        ("日均活跃",f"{K['tw_活跃人数']/7/10000:.1f}万",f"{K['lw_活跃人数']/7/10000:.1f}万",K["pct_活跃人数"],True),
        ("投注金额",f"{K['tw_投注金额']/10000:.0f}万",f"{K['lw_投注金额']/10000:.0f}万",K["pct_投注金额"],True),
        ("全量日均ARPPU",f"${K['tw_全量Arppu']:.2f}",f"${K['lw_全量Arppu']:.2f}",K["pct_全量Arppu"],True),
        ("老用户日均ARPPU",f"${K['tw_老用户ARPPU']:.2f}",f"${K['lw_老用户ARPPU']:.2f}",K["pct_老用户ARPPU"],True),
        ("首充日均ARPPU",f"${K['tw_首充Arppu']:.2f}",f"${K['lw_首充Arppu']:.2f}",K["pct_首充Arppu"],True),
        ("总赠送金额",f"{K['tw_总赠送金额']/10000:.1f}万",f"{K['lw_总赠送金额']/10000:.1f}万",K["pct_总赠送金额"],False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",f"{K['lw_赠送充值比']:.2f}%",K["pct_赠送充值比"],False),
        ("首充转化率",f"{K['tw_首充转化率']:.1f}%",f"{K['lw_首充转化率']:.1f}%",K["pct_首充转化率"],True),
        ("首充次日充值留存",f"{K['tw_首充次日复充率']:.1f}%",f"{K['lw_首充次日复充率']:.1f}%",K["pct_首充次日复充率"],True),
        ("推广消耗",f"{K['tw_真实消耗']/10000:.1f}万",f"{K['lw_真实消耗']/10000:.1f}万",K["pct_真实消耗"],False),
    ]
    S.append(kpi_card4(kpis,cols=4)); S.append(Spacer(1,8))
    cr_delta=K["tw_充提差比"]-K["lw_充提差比"]; ret_delta=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    # 解读文字变量（供 render_list 使用）
    ov = dict(
        tw_充值金额         = K["tw_充值金额"] / 10000,
        pct_充值金额        = K["pct_充值金额"],
        tw_公司输赢         = K["tw_公司输赢"] / 10000,
        pct_公司输赢        = K["pct_公司输赢"],
        pct_真实消耗        = K["pct_真实消耗"],
        tw_充提差比         = K["tw_充提差比"],
        lw_充提差比         = K["lw_充提差比"],
        delta_充提差比      = cr_delta,
        tw_盈亏率           = K["tw_盈亏率"],
        lw_盈亏率           = K["lw_盈亏率"],
        tw_首充人数         = int(K["tw_首充人数"]),
        pct_首充人数        = K["pct_首充人数"],
        tw_首充转化率       = K["tw_首充转化率"],
        tw_首充Arppu        = K["tw_首充Arppu"],
        pct_首充Arppu       = K["pct_首充Arppu"],
        tw_首充次日复充率   = K["tw_首充次日复充率"],
        lw_首充次日复充率   = K["lw_首充次日复充率"],
        delta_首充次日复充率 = ret_delta,
    )
    S.append(insight_box(render_list(OVERVIEW_INSIGHTS, **ov)))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
    S.append(sub_title("首充用户充值留存率 — 近4周加权平均对比")); S.append(chart_ret_weekly(weekly_ret))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=(this_w.get("第1日",0)or 0)-(last_w.get("第1日",0)or 0)
    rn = dict(
        tw_d1    = this_w.get("第1日", 0) or 0,
        tw_d2    = this_w.get("第2日", 0) or 0,
        delta_d1 = d1,
        lw_d7    = last_w.get("第7日", 0) or 0,
    )
    S.append(insight_box(render_list(RETENTION_NOTE, **rn), clr=C_AMBER))
    return S

def build_agents(agents, agent_ret):
    """
    Top15总代表 + 全量留存汇总表
    列说明：
      注册(环比)     → "22,072/+27.1%"  单行，颜色跟环比
      充提差率(差值) → "15.7%/-1.1pp"   单行，颜色跟充提差率高低
      次日留存(本/上) → "22.4%/25.9%"   本<上→红，本>=上→绿
      3留本周        → 本周3日加权（已剔除未到期天，5/15~5/18）
      7留上周        → 上周7日加权（全部完整，5/8~5/14）
      保证：3留本周 > 7留上周（留存递减规律）
    """
    S = [sec_title("二、总代分析（剔除总代0官方总代）")]
    S.append(sub_title("Top15总代表现（本周 vs 上周，按充值金额排序）"))
    full = PW - 2*MARGIN

    # 计算完整截止日（用于列头说明）
    tw_d3_end = retention_end_date(THIS_WEEK[0], 3)  # 20260518
    lw_d7_end = retention_end_date(LAST_WEEK[0], 7)  # 20260514

    tw_d3_lbl = f"{tw_d3_end[4:6]}/{tw_d3_end[6:]}" if tw_d3_end else "-"
    lw_d7_lbl = f"{lw_d7_end[4:6]}/{lw_d7_end[6:]}" if lw_d7_end else "-"

    headers = [
        "ID", "总代名称",
        "注册(环比)",
        "首充",
        "充值\n(万)",
        "充提差率(差值)",
        "消耗\n(万)",
        "一级首充\n成本(本/上)",
        "次日留存\n(本/上)",
        f"3留本周\n(~{tw_d3_lbl})",
        f"7留上周\n(~{lw_d7_lbl})",
    ]

    def _fret(v):
        return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

    rows = []
    for _, r in agents.iterrows():
        agent_id = int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
        rname    = str(r["总代.名称"])
        cr       = r["充提差率"]; lw_cr = r.get("lw_充提差率", 0) or 0; cr_diff = cr - lw_cr
        cost     = (r.get("总消耗", 0) or 0) / 10000
        fc_c     = r.get("一级首充成本", 0) or 0; lw_fc_c = r.get("lw_一级首充成本", 0) or 0

        # 注册/环比
        reg = int(r["注册人数"]); reg_chg = r.get("注册环比", 0) or 0
        reg_str = f"{reg:,}/{'+' if reg_chg>=0 else ''}{reg_chg:.1f}%"
        reg_clr = C_GREEN if reg_chg >= 0 else C_RED

        # 充提差率/差值
        cr_str = f"{cr:.1f}%/{'+' if cr_diff>=0 else ''}{cr_diff:.1f}pp"
        cr_clr = C_GREEN if cr >= CR_HIGH else (C_RED if cr < CR_LOW else C_DARK)

        # 留存数据
        d = agent_ret.get(rname, {})
        tw_d1 = d.get("tw_d1", np.nan); lw_d1 = d.get("lw_d1", np.nan)
        tw_d3 = d.get("tw_d3", np.nan); lw_d7 = d.get("lw_d7", np.nan)

        # 次日留存 本/上
        if not (isinstance(tw_d1,float) and np.isnan(tw_d1)) and not (isinstance(lw_d1,float) and np.isnan(lw_d1)):
            nd_str = f"{tw_d1:.1f}%/{lw_d1:.1f}%"
            nd_clr = C_GREEN if tw_d1 >= lw_d1 else C_RED
        elif not (isinstance(tw_d1,float) and np.isnan(tw_d1)):
            nd_str = f"{tw_d1:.1f}%/-"; nd_clr = C_DARK
        else:
            nd_str = "-"; nd_clr = C_GRAY

        rows.append([
            cell(str(agent_id),              False, C_GRAY,  TA_CENTER),
            cell(rname[:16],                 True,  C_DARK,  TA_LEFT),
            cell(reg_str,                    False, reg_clr, TA_RIGHT),
            cell(f"{int(r['首充人数']):,}",  False, C_DARK,  TA_RIGHT),
            cell(f"{r['充值金额']/10000:.0f}",False, C_DARK, TA_RIGHT),
            cell(cr_str,                     False, cr_clr,  TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-", False, C_DARK, TA_RIGHT),
            cell(f"${fc_c:.0f}/${lw_fc_c:.0f}" if fc_c>0 else "-", False, C_DARK, TA_RIGHT),
            cell(nd_str,                     False, nd_clr,  TA_RIGHT),
            cell(_fret(tw_d3),               False, C_DARK,  TA_RIGHT),
            cell(_fret(lw_d7),               False, C_DARK,  TA_RIGHT),
        ])

    cw = [full*x for x in [0.05,0.17,0.11,0.06,0.06,0.12,0.06,0.11,0.10,0.08,0.08]]
    S.append(dtable(headers, rows, cw, fsize=6.5))
    S.append(Spacer(1,4))
    # 数据说明
    S.append(P(f"★ 3留本周：剔除5/19~5/21未到期天，取5/15~{tw_d3_lbl}加权均值；"
               f"7留上周：上周全部7天数据完整（5/08~5/14），两列数据口径一致，可比较。",
               7, False, C_GRAY))
    S.append(Spacer(1,6))

    dsp_rows = agents[agents["总代.名称"].str.contains("DSP", na=False)]
    av = dict(
        dsp_cr  = dsp_rows["充提差率"].values[0] if len(dsp_rows) > 0 else 0,
        dsp_reg = dsp_rows["注册环比"].values[0] if len(dsp_rows) > 0 else 0,
    )
    S.append(insight_box(render_list(AGENT_INSIGHTS, **av)))

    # ── 全量留存汇总表（加ID列）────────────────────────
    S.append(sub_title(
        f"总代首充充值留存率（每日加权均值 | 次日/3日取本周5/15~{tw_d3_lbl} | 7日取上周5/08~{lw_d7_lbl}）"))
    S.append(P("保证分子分母一致：仅统计留存期已完整的首充日，3留本周 > 7留上周符合留存递减规律。",
               7.5, False, C_GRAY))
    S.append(Spacer(1,4))

    headers2 = ["ID", "总代名称", "次日留存\n本/上周", "3日留存\n本周", "7日留存\n上周"]
    rows2 = []
    top15_names = [str(r["总代.名称"]) for _,r in agents.iterrows()]
    extra = sorted([n for n in agent_ret if n not in top15_names and n != "总体"])
    ordered = top15_names + extra
    if "总体" in agent_ret: ordered.append("总体")

    for nm in ordered[:22]:
        d = agent_ret.get(nm, {})
        tw1 = d.get("tw_d1", np.nan); lw1 = d.get("lw_d1", np.nan)
        tw3 = d.get("tw_d3", np.nan); lw7 = d.get("lw_d7", np.nan)
        aid = d.get("agent_id", None)
        aid_str = str(int(aid)) if aid and not (isinstance(aid,float) and np.isnan(aid)) else "-"

        if not (isinstance(tw1,float) and np.isnan(tw1)) and not (isinstance(lw1,float) and np.isnan(lw1)):
            nd_s = f"{tw1:.1f}%/{lw1:.1f}%"
            nd_c = C_GREEN if tw1 >= lw1 else C_RED
        elif not (isinstance(tw1,float) and np.isnan(tw1)):
            nd_s = f"{tw1:.1f}%/-"; nd_c = C_DARK
        else:
            nd_s = "-"; nd_c = C_GRAY

        rows2.append([
            cell(aid_str,    False, C_GRAY,  TA_CENTER),
            cell(nm[:22],    True,  C_DARK,  TA_LEFT),
            cell(nd_s,       False, nd_c,    TA_RIGHT),
            cell(_fret(tw3), False, C_DARK,  TA_RIGHT),
            cell(_fret(lw7), False, C_DARK,  TA_RIGHT),
        ])

    if rows2:
        S.append(dtable(headers2, rows2,
                        [full*x for x in [0.08,0.46,0.16,0.15,0.15]], fsize=6.8))
    return S

def build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw):
    S=[sec_title("三、用户分析")]
    full=PW-2*MARGIN
    S.append(sub_title("VIP等级分层分析（充值/投注金额 本周 vs 上周）"))
    S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
    total_chg=tw_v["充值金额"].sum()
    high_vip_pct=(tw_v.loc[9,"充值金额"]+tw_v.loc[10,"充值金额"]+tw_v.loc[11,"充值金额"])/total_chg*100 if 9 in tw_v.index else 0
    vv = dict(
        high_vip_pct   = high_vip_pct,
        tw_vip1_active = int(tw_v.loc[1,"活跃人数"]) if 1 in tw_v.index else 0,
        lw_vip1_active = int(lw_v.loc[1,"活跃人数"]) if 1 in lw_v.index else 0,
    )
    S.append(insight_box(render_list(VIP_INSIGHTS, **vv)))
    vip_ret=load_vip_retention()
    S.append(sub_title("VIP各等级 充值→充值留存率 & 充值→活跃留存率（本周均值）"))
    S.append(chart_vip_retention(vip_ret)); S.append(Spacer(1,4))
    rv = dict(
        act_v9     = vip_ret["act"]["tw"]["1日"].get(9, 0),
        act_v10    = vip_ret["act"]["tw"]["1日"].get(10, 0),
        chg_v10_tw = vip_ret["chg"]["tw"]["1日"].get(10, 0),
        chg_v10_lw = vip_ret["chg"]["lw"]["1日"].get(10, 0),
    )
    S.append(insight_box(render_list(VIP_RET_INSIGHTS, **rv)))
    S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析（Top1/10/50/100/200/500）"))
    headers=["分层","本周充值","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    dv = dict(
        top10_avg     = dep_t[1]["tw_avg"],
        top10_avg_chg = pct(dep_t[1]["tw_avg"], dep_t[1]["lw_avg"]),
        top10_cr      = dep_t[1]["tw_cr"],
    )
    S.append(insight_box(render_list(TOP_DEPOSIT_INSIGHTS, **dv)))
    S.append(sub_title("头部提款用户分层分析（Top1/10/50/100/200/500）"))
    headers3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","赢家\n比例","活动\n占比","占全\n量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdraw_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(headers3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    wv = dict(
        top100_impact  = wdraw_t[3]["大盘影响"],
        top500_act_pct = wdraw_t[5]["活动占比"],
    )
    S.append(insight_box(render_list(TOP_WITHDRAW_INSIGHTS, **wv), clr=C_AMBER))
    return S

def build_games_section(mfr,top30):
    S=[sec_title("四、游戏分析")]; full=PW-2*MARGIN
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    headers=["排名","厂商","日均投注人数\n(本/上)","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["show_name_厂商标签id"])[:10],True),cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
    def _mv(name, col):
        sub = mfr[mfr["show_name_厂商标签id"] == name]
        return sub[col].values[0] if len(sub) > 0 else 0
    mv = dict(
        tada_share    = mfr.iloc[0]["占比"],
        tada_share_lw = mfr.iloc[0].get("lw_占比", 0),
        tada_chg      = mfr.iloc[0].get("投注环比", 0),
        evol_pl       = _mv("Evolution", "盈亏率"),
        evol_pl_lw    = _mv("Evolution", "lw_盈亏率"),
        oaks_pl       = _mv("3Oaks",    "盈亏率"),
        ptech_pl      = _mv("PlayTech", "盈亏率"),
    )
    S.append(insight_box(render_list(MFR_INSIGHTS, **mv)))
    S.append(sub_title("Top30游戏详细数据（本周 vs 上周）")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
    headers2=["#","游戏名称","厂商","投注\n人数","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["游戏.名称"])[:18],True),cell(str(r["游戏厂商标签.名称"])[:7]),cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(headers2,rows2,[full*x for x in [0.04,0.18,0.08,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S

def build_activities_section(act_df,total_gift,src_df,total_dep):
    S=[sec_title("五、活动分析")]; full=PW-2*MARGIN
    S.append(sub_title("各活动赠送效果（本周 vs 上周，含环比）"))
    S.append(chart_activities(act_df)); S.append(Spacer(1,4))
    headers=["活动名称","本周赠送","上周赠送","环比","赠送人次","人均赠送","金额占比"]
    rows=[]
    for _,r in act_df.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:16],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['lw_赠送']:,.0f}",False,C_GRAY,TA_RIGHT),rc(r["环比"]),cell(f"{r['赠送人数']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均']:.2f}",False,C_DARK,TA_RIGHT),cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.22,0.14,0.14,0.1,0.14,0.12,0.14]])); S.append(Spacer(1,4))
    S.append(P(f"本周总赠送金额：${total_gift/10000:.2f}万 | 赠送/充值比：{total_gift/total_dep*100:.2f}%",9,True,C_DARK)); S.append(Spacer(1,8))
    S.append(sub_title("存款来源道具分析（充值金额占比)")); S.append(chart_deposit_src(src_df,total_dep)); S.append(Spacer(1,4))
    headers2=["道具/来源","本周充值(万)","上周充值(万)","环比","充值占比"]
    rows2=[]
    for _,r in src_df.iterrows():
        rows2.append([cell(str(r["名称"])[:18],True),cell(f"{r['充值金额']/10000:.2f}",False,C_DARK,TA_RIGHT),cell(f"{r['lw_充值']/10000:.2f}",False,C_GRAY,TA_RIGHT),rc(r["环比"]),cell(f"{r['占比']:.1f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT)])
    S.append(dtable(headers2,rows2,[full*x for x in [0.35,0.15,0.15,0.15,0.2]])); S.append(Spacer(1,4))
    xk = src_df[src_df["名称"] == "幸运翻卡道具"]
    acv = dict(
        tool_deposit = (total_dep - src_df.iloc[0]["充值金额"]) / 10000,
        tool_pct     = (1 - src_df.iloc[0]["占比"] / 100) * 100,
        xk_deposit   = xk["充值金额"].values[0] / 10000 if len(xk) > 0 else 0,
        xk_chg       = xk["环比"].values[0]               if len(xk) > 0 else 0,
    )
    S.append(insight_box(render_list(ACTIVITY_INSIGHTS, **acv), clr=C_AMBER))
    return S

def build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw_full):
    S=[sec_title("六、用户游戏风险专项分析",clr=colors.HexColor("#7c2d12"))]; full=PW-2*MARGIN
    total_tx=top500["提款金额"].sum(); total_win=top500["公司输赢"].sum(); winner_cnt=(top500["公司输赢"]<0).sum()
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${total_tx/10000:.1f}万",""),("平台净赔付",f"${abs(total_win)/10000:.1f}万","平台向该群体净赔"),("赢家比例",f"{winner_cnt/500*100:.1f}%",f"{winner_cnt}赢/{500-winner_cnt}输"),("高风险用户",f"{len(high_risk)}人","公司净输>$10,000")]:
        fw=full/4-4; row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP"); t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    rsv = dict(
        winner_cnt     = winner_cnt,
        winner_pct     = winner_cnt / 500 * 100,
        total_loss     = abs(total_win) / 10000,
        high_risk_cnt  = len(high_risk),
        high_risk_loss = abs(high_risk["公司输赢"].sum()) / 10000,
        high_risk_pct  = abs(high_risk["公司输赢"].sum()) / abs(total_win) * 100,
    )
    S.append(insight_box(render_list(RISK_SCATTER_INSIGHTS, **rsv)))
    S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
    headers_r=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注\n均额","主玩厂商","主玩游戏","风险标签"]
    rows_r=[]
    for _,r in high_risk.iterrows():
        ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0; avg_b=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
        tags=[]; 
        if r["充值金额"]<5000: tags.append("低充高提")
        if ratio>50: tags.append("超高投充")
        if avg_b>200: tags.append("高额单注")
        rows_r.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),cell(f"${avg_b:.0f}",False,C_RED if avg_b>200 else C_DARK,TA_RIGHT),cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
    S.append(dtable(headers_r,rows_r,[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    if len(cluster_207)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警")); S.append(P(f"集群账户数：{len(cluster_207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
        headers_c=["账户ID","提款金额","充值金额","公司输赢","特征"]; rows_c=[]
        for _,r in c207_dt.head(10).iterrows():
            rows_c.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell("充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}")])
        if len(c207_dt)>10: rows_c.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
        S.append(dtable(headers_c,rows_c,[full*x for x in [0.22,0.18,0.18,0.18,0.24]])); S.append(Spacer(1,6))
        ppv = dict(cluster_cnt=len(cluster_207))
        S.append(insight_box(render_list(PP_ROULETTE_INSIGHTS, **ppv), clr=C_AMBER))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    headers_t=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; rows_t=[]
    for i,(_,r) in enumerate(dt_tw_full.head(20).iterrows()):
        win=r["公司输赢"]<0; rows_t.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(headers_t,rows_t,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
    pref_game,mfr_amt=load_pref(); S.append(sub_title("Top500提款用户游戏偏好（按投注金额）")); S.append(chart_pref(pref_game,mfr_amt)); S.append(Spacer(1,4))
    S.append(sub_title("▶ 高危游戏专项分析（Top500提款用户视角）")); S.append(chart_risk_games(risk_g)); S.append(Spacer(1,4))
    headers_g=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; rows_g=[]
    for _,r in risk_g.head(12).iterrows():
        pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rl_clr=C_RED if rl in ("极高","高") else C_AMBER
        rows_g.append([cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rl_clr)])
    S.append(dtable(headers_g,rows_g,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box(RISK_GAME_INSIGHTS, clr=C_RED))
    return S

def build_conclusion(K,agents,mfr,dep_t,wdraw_t):
    S=[sec_title("七、总结与行动建议")]; full=PW-2*MARGIN
    S.append(sub_title("▶ 本周亮点"))
    for h in [f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），充提差{K['tw_充提差']/10000:.1f}万（{K['pct_充提差']:+.1f}%），充提差率{K['tw_充提差比']:.2f}%（{K['tw_充提差比']-K['lw_充提差比']:+.2f}pp）。",f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。","A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。"]:
        S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#f0fdf4")),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8)); S.append(sub_title("▶ 风险预警"))
    for r in ["PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。",f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp），新用户质量需关注。","YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。"]:
        S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#fff5f5")),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8)); S.append(sub_title("▶ 行动建议"))
    actions=[("[紧急]","处置PP Auto-Roulette 1批量套利","33个207开头账户合计提款$9.4万，平台净赔$6.7万。冻结账户，审查IP/设备指纹，修复$270.92奖励触发漏洞，对Auto-Roulette 1实施赢额上限。"),("[本周]","处置高风险提款用户",f"账户49903866：提款$39,636，公司输赢-$52,134；账户206628722：充值$10,229提款$24,855（净提$14,626），高度异常。建议立即人工审核。"),("[本周]","优化首充质量与次日激活",f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp）。建议注册后1h/24h内推送首充引导，落地页突出$10+档。"),("[下周]","扩大A8_DSP、A8_GG渠道预算","A8_DSP充提差率17.6%（+1.3pp），一级首充成本$17（上周$21，改善）；A8_GG充提差率18.9%（+0.3pp）。建议预算增加15-20%，预估带动充值增量$50-70万。"),("[下周]","暂停YB_FB_PWA_0","注册量暴跌-79%，仅1,315人，建议暂停投放并排查渠道异常。")]
    rows_a=[[cell(p,True,{"[紧急]":C_RED,"[本周]":C_AMBER,"[下周]":C_GREEN}.get(p[:4],C_GRAY),TA_CENTER),cell(t,True,C_DARK),cell(d,False,C_GRAY)] for p,t,d in actions]
    S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows_a,[full*x for x in [0.1,0.22,0.68]]))
    return S

def header_footer(c,doc):
    c.saveState(); w,h=A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN,h-17,f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
    c.restoreState()

def main(data_root=None, output_dir=None):
    root   = Path(data_root)  if data_root  else DATA_ROOT
    outdir = Path(output_dir) if output_dir else OUTPUT_DIR
    outdir.mkdir(parents=True, exist_ok=True)
    print("📊 MX 周报 PDF v5.5 生成中...")
    _resolve_files(root); setup()
    print("  ▶ 加载数据...")
    K,trend=load_platform(); weekly_ret=load_weekly_retention()
    agents=load_agents(); agent_ret=load_agent_ret()
    tw_v,lw_v=load_vip(); dep_t,wdraw_t,dc_tw,dt_tw=load_top_users()
    top30=load_games(); mfr=load_mfr()
    act_df,total_gift=load_activities(); src_df,total_dep=load_deposit_src()
    high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500=load_risk()
    pref_game,mfr_amt=load_pref()
    bet_user=pd.read_csv(FILES["pref_tw"]); bet_user["阶段汇总"]=bet_user["阶段汇总"].apply(clean)
    top_game=(bet_user[bet_user["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False).groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]].rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
    dt_tw=dt_tw.merge(top_game,on="账户ID",how="left")
    print("  ▶ 导出风控名单Excel...")
    excel_out=outdir/f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out),engine="openpyxl") as writer:
        cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
        high_risk[[c for c in cols_hr if c in high_risk.columns]].to_excel(writer,sheet_name="高风险用户",index=False)
        if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(writer,sheet_name="PP轮盘批量账号集群",index=False)
        if len(special_users)>0: special_users[["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]].to_excel(writer,sheet_name="特殊异常用户",index=False)
    print(f"✅ 风控名单Excel: {excel_out}")
    print("  ▶ 构建PDF...")
    out=outdir/f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc=SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,topMargin=MARGIN+22,bottomMargin=MARGIN+10,title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story=[]
    story+=[Spacer(1,3*cm),P("MX 平台数据周报",30,True,C_DARK,TA_CENTER),Spacer(1,0.5*cm),P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),Spacer(1,0.2*cm),P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),Spacer(1,0.5*cm),HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"),PageBreak()]
    story+=build_overview(K,trend,weekly_ret); story.append(PageBreak())
    story+=build_agents(agents,agent_ret); story.append(PageBreak())
    story+=build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw); story.append(PageBreak())
    story+=build_games_section(mfr,top30); story.append(PageBreak())
    story+=build_activities_section(act_df,total_gift,src_df,total_dep); story.append(PageBreak())
    story+=build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw); story.append(PageBreak())
    story+=build_conclusion(K,agents,mfr,dep_t,wdraw_t)
    print("  ▶ 渲染PDF...")
    doc.build(story,onFirstPage=header_footer,onLaterPages=header_footer)
    size=out.stat().st_size/1024; print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out),str(excel_out)

if __name__=="__main__":
    import sys
    if sys.platform!="win32": main(data_root="/mnt/user-data/uploads",output_dir="/mnt/user-data/outputs")
    else: main(data_root=DATA_ROOT,output_dir=OUTPUT_DIR)

📊 MX 周报 PDF v5.5 生成中...
  ▶ 数据目录：D:\周报更新版\MX
     ✅ [platform    ] 大盘/平台报表_USD_近4周.xlsx
     ✅ [daily       ] 大盘/日报-大盘日报_USD_近4周.xlsx
     ✅ [retention   ] 大盘/整体 首充留存（近7天）_近28天.csv
     ✅ [agent_plat  ] 总代/平台报表-总代_USD_近21天.xlsx
     ✅ [agent_promo ] 总代/推广报表-总代_USD_近21天.xlsx
     ✅ [agent_ret   ] 总代/首充充值留存_20260424_20260521.csv
     ✅ [vip         ] 用户/VIP报表_USD_近14天.xlsx
     ✅ [dt_tw       ] 用户/top提款用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dt_lw       ] 用户/top提款用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [dc_tw       ] 用户/头部充值用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dc_lw       ] 用户/头部充值用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [pref_tw     ] 用户/本周top500提款用户游戏偏好_全量数据_20260515_20260521.csv
     ✅ [pref_lw     ] 用户/上周top500提款用户游戏偏好_全量数据_20260508_20260514.csv
     ✅ [mfr         ] 游戏/厂商投注数据_全量数据_20260508_20260521.csv
     ✅ [game_tw     ] 游戏/游戏报表-详情_USD_本周.xlsx
     ✅ [game_lw     ] 游戏/游戏报表-详情_USD_上周.xlsx
     ✅ [gift 

In [36]:
"""
MX 平台数据周报生成脚本 v5.4
======================================================
【每次换周报只需修改下面 ① ② ③ 三处，其余不动】

v5.4 修复内容：
  1. sub_title 小标题改为实心方块 ■
  2. 总代表：注册/环比 合并为 "4,635/+5.7%" 单行格式
  3. 总代表：充提差率/差值 合并为 "15.7%/-1.1pp" 单行格式
  4. load_agent_ret 完全重写：正确解析日期、加权计算留存
     - 次日留存(本/上)：本周1日加权 / 上周1日加权
     - 3留本周：本周3日加权
     - 7留上周：上周7日加权（本周7日未到期，不展示）
  5. 次日留存颜色：本周低于上周显红，高于或等于显绿
"""
from pathlib import Path

# ① 数据根目录
DATA_ROOT  = Path(r"D:\周报更新版\MX")
# ② 报告输出目录（自动创建）
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")
# ③ 本周 / 上周日期范围 (YYYYMMDD)
THIS_WEEK  = ("20260515", "20260521")
LAST_WEEK  = ("20260508", "20260514")

# ══════════════════════════════════════════════
import io, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, PageBreak, HRFlowable, KeepTogether
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

FN, FNB = "WQY", "WQYB"
FILES = {}

# ── 字体 ─────────────────────────────────────
def find_chinese_font():
    import platform
    sys_name = platform.system()
    if sys_name == "Windows":
        candidates = [
            r"C:\Windows\Fonts\msyh.ttc",
            r"C:\Windows\Fonts\msyhbd.ttc",
            r"C:\Windows\Fonts\simsun.ttc",
            r"C:\Windows\Fonts\simhei.ttf",
        ]
    elif sys_name == "Darwin":
        candidates = ["/System/Library/Fonts/PingFang.ttc"]
    else:
        candidates = [
            "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
            "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        ]
    for p in candidates:
        if Path(p).exists():
            return p
    try:
        from matplotlib import font_manager as fm
        for f in fm.fontManager.ttflist:
            if any(k in f.name for k in ["Heiti","Hei","YaHei","SimSun","SimHei","WenQuanYi","Noto Sans CJK"]):
                if Path(f.fname).exists():
                    return f.fname
    except Exception:
        pass
    raise FileNotFoundError("未找到中文字体！请手动设置 FONT_PATH")

def setup():
    global FONT_PATH
    FONT_PATH = find_chinese_font()
    print(f"  ▶ 使用字体：{FONT_PATH}")
    is_ttc = FONT_PATH.lower().endswith(".ttc")
    if is_ttc:
        pdfmetrics.registerFont(TTFont(FN,  FONT_PATH, subfontIndex=0))
        try:   pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1))
        except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0))
    else:
        pdfmetrics.registerFont(TTFont(FN,  FONT_PATH))
        pdfmetrics.registerFont(TTFont(FNB, FONT_PATH))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(FONT_PATH)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

# ── 文件搜索 ─────────────────────────────────
def _resolve_files(root):
    global FILES
    kw_map = {
        "platform":    "平台报表_USD_近4周",
        "daily":       "日报-大盘日报_USD_近4周",
        "retention":   "首充留存",
        "agent_plat":  "平台报表-总代_USD_近21天",
        "agent_promo": "推广报表-总代_USD_近21天",
        "agent_ret":   "首充充值留存",
        "vip":         "VIP报表_USD",
        "dt_tw":       f"top提款用户_全量数据_{THIS_WEEK[0]}",
        "dt_lw":       f"top提款用户_全量数据_{LAST_WEEK[0]}",
        "dc_tw":       f"头部充值用户_全量数据_{THIS_WEEK[0]}",
        "dc_lw":       f"头部充值用户_全量数据_{LAST_WEEK[0]}",
        "pref_tw":     f"本周top500提款用户游戏偏好_全量数据_{THIS_WEEK[0]}",
        "pref_lw":     f"上周top500提款用户游戏偏好_全量数据_{LAST_WEEK[0]}",
        "mfr":         "厂商投注数据_全量数据",
        "game_tw":     "游戏报表-详情_USD_本周",
        "game_lw":     "游戏报表-详情_USD_上周",
        "gift":        f"各赠送活动_全量数据_{THIS_WEEK[0]}",
        "deposit_src": f"整体存款_全量数据_{THIS_WEEK[0]}",
        "tool_map":    "道具对应活动",
        "vip_ret_chg": "VIP充值-充值_近28天",
        "vip_ret_act": "VIP充值-活跃_近28天",
    }
    print(f"  ▶ 数据目录：{root}")
    fail = 0
    for key, kw in kw_map.items():
        hits = [h for h in root.rglob("*")
                if h.is_file() and not h.name.startswith("~$") and kw in h.name]
        if not hits:
            print(f"     ❌ [{key:12s}] 找不到含 '{kw}' 的文件"); fail += 1
        else:
            p = max(hits, key=lambda h: h.stat().st_mtime)
            FILES[key] = p
            print(f"     ✅ [{key:12s}] {p.parent.name}/{p.name}")
    if fail:
        raise FileNotFoundError(f"\n共 {fail} 个文件未找到，请检查 DATA_ROOT = {root}")
    print(f"  ▶ 全部 {len(kw_map)} 个文件匹配成功\n")

# ── 颜色 ─────────────────────────────────────
C_BLUE   = colors.HexColor("#1d4ed8")
C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669")
C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706")
C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b")
C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white
C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1")
C_ROW    = colors.HexColor("#f8fafc")

PW, PH = A4
MARGIN  = 1.8*cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
TRUNCATE_RULES = {
    "首充次日复充率": 1, "首充次日复投率": 1, "首充当日复充率": 0,
    "首充7日复充率": 6, "首充30日复充率": 999,
    "首充2日复充率": 1, "首充3日复充率": 2,
}

# ── 工具函数 ─────────────────────────────────
def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c != "日期" and c not in ["总代.名称","name_总代","总代.ID"]]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o): return (n-o)/abs(o)*100 if o and o!=0 else 0.0
def pct_vec(ns, os): return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop>0 else s
    nz = v[v>0]; return nz.mean() if len(nz)>0 else v.mean()
def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

# ── 样式函数 ─────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.38,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),
                                ("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    """实心方块 ■ 小标题"""
    return KeepTogether([
        Spacer(1,5),
        HRFlowable(width="100%", thickness=1.5, color=C_BLUE2),
        Spacer(1,3),
        P(f"■  {text}", 10, True, C_DARK),
        Spacer(1,6)
    ])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}", 8.5, False, C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg),
            ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10),("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8):
    full = PW-2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr),("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2.5),("BOTTOMPADDING",(0,0),(-1,-1),2.5),
        ("LEFTPADDING",(0,0),(-1,-1),3),("RIGHTPADDING",(0,0),(-1,-1),3),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    hrow = [P(h, fsize, True, C_WHITE, TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]), fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c, (list,tuple)) else P(str(c), fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t, bold=False, clr=colors.black, align=TA_LEFT): return (t, bold, clr, align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup = good_up
    s = "+" if v>=0 else ""; c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{s}{v:.{d}f}%", False, c, TA_RIGHT)
def gclr(v, t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)

def kpi_card4(items, cols=4):
    full = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,gup in items:
        s = "+" if chg>=0 else ""
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([
            [P(label, 7.5, False, C_GRAY)],
            [P(str(tv), 14, True, C_DARK)],
            [P(f"上周：{lv}", 7.5, False, C_GRAY)],
            [P(f"{s}{chg:.1f}%", 8, True, pclr)],
        ], colWidths=[full], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),8),
            ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5),
        ]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(full,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════
def load_platform():
    df = pd.read_excel(FILES["platform"])
    df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]
    lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"])
    dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]
    lw_d = dd[dd["日期"].between(*LAST_WEEK)]

    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
              "首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE_RULES.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop)
        K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    K["tw_真实消耗"] = tw_d["真实消耗"].iloc[:-1].sum() if len(tw_d)>1 else tw_d["真实消耗"].sum()
    K["lw_真实消耗"] = lw_d["真实消耗"].sum()
    K["pct_真实消耗"] = pct(K["tw_真实消耗"], K["lw_真实消耗"])
    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])

    last14 = df.tail(14)
    trend = {
        "dates":  [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
        "充值":   last14["充值金额"].tolist(),
        "提现":   last14["提现金额"].tolist(),
        "充提差比": last14["充提差比"].tolist(),
        "公司输赢": last14["公司输赢"].tolist(),
        "首充":   last14["首充人数"].tolist(),
        "注册":   last14["注册人数"].tolist(),
    }
    return K, trend

def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
    daily["date_str"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
    daily_r = daily[daily["指标"]=="留存率"].copy()
    daily_u = daily[daily["指标"]=="留存人数"].copy()
    for c in ["第1日","第2日","第3日","第7日"]:
        daily_r[c] = daily_r[c].apply(clean)
    def _d(s): return datetime.strptime(s, "%Y%m%d")
    def _fmt(d): return d.strftime("%Y-%m-%d")
    def _lbl(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s=_d(THIS_WEEK[0]); tw_e=_d(THIS_WEEK[1])
    lw_s=_d(LAST_WEEK[0]); lw_e=_d(LAST_WEEK[1])
    w2_end=lw_s-timedelta(days=1); w2_start=w2_end-timedelta(days=6)
    w1_end=w2_start-timedelta(days=1); w1_start=w1_end-timedelta(days=6)
    weeks = [
        (f"第1周\n{_lbl(w1_start,w1_end)}", _fmt(w1_start), _fmt(w1_end)),
        (f"第2周\n{_lbl(w2_start,w2_end)}", _fmt(w2_start), _fmt(w2_end)),
        (f"上周\n{_lbl(lw_s,lw_e)}",        _fmt(lw_s),     _fmt(lw_e)),
        (f"本周\n{_lbl(tw_s,tw_e)}",        _fmt(tw_s),     _fmt(tw_e)),
    ]
    result = []
    for wk,s,e in weeks:
        mr = daily_r[(daily_r["date_str"]>=s)&(daily_r["date_str"]<=e)]
        mu = daily_u[(daily_u["date_str"]>=s)&(daily_u["date_str"]<=e)]
        users = mu["充值成功事件用户数"].sum(); row = {"week": wk, "users": users}
        for col in ["第1日","第2日","第3日","第7日"]:
            rs = mr[col].values; us = mu["充值成功事件用户数"].values
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = np.average(rs[v], weights=us[v]) if v.sum()>0 else np.nan
            else:
                row[col] = np.nanmean(rs) if len(rs)>0 else np.nan
        result.append(row)
    return result

def load_agents():
    df_p = pd.read_excel(FILES["agent_plat"])
    df_r = pd.read_excel(FILES["agent_promo"])
    df_p["日期"] = df_p["日期"].astype(str)
    df_r["日期"] = df_r["日期"].astype(str)
    df_p = to_num(df_p); df_r = to_num(df_r)
    df_p = df_p[df_p["总代.ID"]!=0]
    df_r = df_r[df_r["总代.ID"]!=0]
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]
    lw_p = df_p[df_p["日期"].between(*LAST_WEEK)]
    tw_r = df_r[df_r["日期"].between(*THIS_WEEK)]
    lw_r = df_r[df_r["日期"].between(*LAST_WEEK)]

    sum_a = ["充值金额","提现金额","充提差","首充金额","首充人数","注册人数",
             "充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        g = d.groupby(["总代.ID","总代.名称"]).agg(
            {c:"sum" for c in sum_a if c in d.columns}).reset_index()
        for c in ["首充转化率","充提差比","首充次日充值留存","活跃用户付费率"]:
            if c in d.columns:
                rm = d.groupby("总代.ID")[c].mean().rename(c)
                g = g.merge(rm, on="总代.ID", how="left")
        g["充提差率"] = g["充提差"]/g["充值金额"]*100
        return g
    tw_pa = agg_p(tw_p); lw_pa = agg_p(lw_p)

    sum_r = ["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        g = d.groupby("总代.ID").agg(
            {c:"sum" for c in sum_r if c in d.columns}).reset_index()
        g["注册成本"] = g["总消耗"]/g["注册人数"].replace(0,np.nan)
        g["一级首充成本"] = g["总消耗"]/g["一级首充人数"].replace(0,np.nan)
        return g
    tw_ra = agg_r(tw_r); lw_ra = agg_r(lw_r)
    lw_ra["lw_一级首充成本"] = lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)

    m = tw_pa.merge(
        lw_pa[["总代.ID","充值金额","注册人数","充提差率","首充次日充值留存"]].rename(
            columns={"充值金额":"lw_充值","注册人数":"lw_注册",
                     "充提差率":"lw_充提差率","首充次日充值留存":"lw_次日留存"}),
        on="总代.ID", how="left")
    m = m.merge(tw_ra[["总代.ID","总消耗","注册成本","一级首充成本","一级首充人数"]],
                on="总代.ID", how="left")
    m = m.merge(lw_ra[["总代.ID","lw_一级首充成本"]], on="总代.ID", how="left")
    m["注册环比"] = pct_vec(m["注册人数"], m["lw_注册"])
    m["充值环比"] = pct_vec(m["充值金额"], m["lw_充值"])
    return m.sort_values("充值金额", ascending=False).head(15)

# ════════════════════════════════════════════
# v5.4 核心修复：总代留存加载
# ════════════════════════════════════════════
def load_agent_ret():
    """
    从 首充充值留存_YYYYMMDD_YYYYMMDD.csv 按本周/上周日期区间
    对每个总代做首充用户数加权平均留存率。

    CSV 格式：
      - 日期列：'初始事件发生时间'，值如 '2026-05-15(四)' 或 '阶段值'
      - 留存率行：指标=='留存率'，值含 '%' 符号
      - 用户数列：'首充用户数'
      - 留存列：'1日','2日','3日','7日'

    返回：
      dict { name_总代 -> {
          'tw_d1': float,  # 本周1日留存(%)
          'tw_d3': float,  # 本周3日留存(%)
          'lw_d1': float,  # 上周1日留存(%)
          'lw_d7': float,  # 上周7日留存(%)
      } }
    注：本周7日留存因未到期不计算，表格中"7留"列固定取上周数据。
    """
    df = pd.read_csv(FILES["agent_ret"])

    # 提取日期 YYYYMMDD（跳过阶段值行）
    df["_yyyymmdd"] = (df["初始事件发生时间"]
                       .astype(str)
                       .str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
                       .str.replace("-",""))

    # 数值化留存率（去掉 %）
    for c in ["1日","2日","3日","7日"]:
        df[c] = (df[c].astype(str)
                      .str.replace("%","")
                      .str.strip()
                      .pipe(pd.to_numeric, errors="coerce"))

    # 用户数数值化
    df["首充用户数"] = pd.to_numeric(df["首充用户数"], errors="coerce").fillna(0)

    # 只保留每日留存率行（过滤掉阶段值和留存人数行）
    daily = df[
        df["_yyyymmdd"].notna() &
        df["_yyyymmdd"].str.match(r"^\d{8}$") &
        (df["指标"] == "留存率")
    ].copy()

    # 按本周/上周分别筛选
    tw_df = daily[daily["_yyyymmdd"].between(*THIS_WEEK)]
    lw_df = daily[daily["_yyyymmdd"].between(*LAST_WEEK)]

    def _wavg(subset, col):
        """按首充用户数加权平均留存率"""
        result = {}
        for nm, grp in subset.groupby("name_总代"):
            valid = grp[grp[col].notna() & (grp["首充用户数"] > 0)]
            if len(valid) > 0:
                result[nm] = float(np.average(valid[col], weights=valid["首充用户数"]))
            else:
                result[nm] = np.nan
        return result

    tw_d1 = _wavg(tw_df, "1日")
    tw_d3 = _wavg(tw_df, "3日")
    lw_d1 = _wavg(lw_df, "1日")
    lw_d7 = _wavg(lw_df, "7日")

    all_names = set(tw_d1) | set(lw_d1)
    ret = {}
    for nm in all_names:
        ret[nm] = {
            "tw_d1": tw_d1.get(nm, np.nan),
            "tw_d3": tw_d3.get(nm, np.nan),
            "lw_d1": lw_d1.get(nm, np.nan),
            "lw_d7": lw_d7.get(nm, np.nan),
        }
    return ret

def load_agent_ret_legacy():
    """兼容旧格式（阶段汇总，用于总代留存汇总表）"""
    df = pd.read_csv(FILES["agent_ret"])
    st = df[df["初始事件发生时间"] == "阶段值"]
    rr = st[st["指标"] == "留存率"].copy()
    for c in ["1日","2日","3日","7日"]:
        rr[c] = rr[c].astype(str).str.replace("%","").str.strip()
        rr[c] = pd.to_numeric(rr[c], errors="coerce")
    main = ["运营代投总代","A8_DSP_谷歌原生包_原生","苹果包总代",
            "A8_GG_谷歌原生包_原生","A8_TikTok_谷歌原生包_投手","总体"]
    keep = rr[rr["name_总代"].isin(main)].copy()
    keep["首充用户数"] = pd.to_numeric(keep["首充用户数"], errors="coerce")
    return keep[["name_总代","首充用户数","1日","2日","3日","7日"]].sort_values(
        "首充用户数", ascending=False)

def load_vip():
    df = pd.read_excel(FILES["vip"]); df["日期"] = df["日期"].astype(str)
    num = ["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df = to_num(df, num)
    tw = df[df["日期"].between(*THIS_WEEK)]
    lw = df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

def load_vip_retention():
    results = {}
    for fkey, rtype in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df = pd.read_csv(FILES[fkey])
        df['date_str'] = df['初始事件发生时间'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')
        df['yyyymmdd'] = df['date_str'].str.replace('-','')
        for c in ['1日','2日','3日','7日']: df[c] = df[c].apply(clean)
        df['n'] = pd.to_numeric(df['充值成功事件用户数'], errors='coerce')
        df['vip'] = pd.to_numeric(df['vip_level'], errors='coerce')
        stage = df[(df['指标']=='留存率')&(df['初始事件发生时间']=='阶段值')].copy()
        stage_dict = {int(r['vip']):{c:r[c] for c in ['1日','2日','3日','7日']}
                      for _,r in stage.iterrows() if not pd.isna(r['vip'])}
        daily = df[(df['指标']=='留存率')&df['yyyymmdd'].notna()&df['vip'].notna()]
        tw = daily[daily['yyyymmdd'].between(*THIS_WEEK)]
        lw = daily[daily['yyyymmdd'].between(*LAST_WEEK)]
        def wavg(d,col):
            res={}
            for vip,g in d.groupby('vip'):
                v=g[g[col].notna()]
                if len(v)>0: res[int(vip)]=np.average(v[col],weights=v['n'])
            return res
        results[rtype] = {
            'tw':{c:wavg(tw,c) for c in ['1日','2日','3日','7日']},
            'lw':{c:wavg(lw,c) for c in ['1日','2日','3日','7日']},
            'stage':stage_dict,
        }
    return results

def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
    for df in [dc_tw,dc_lw,dt_tw,dt_lw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
    tiers = [1,10,50,100,200,500]
    df_p = pd.read_excel(FILES["platform"]); df_p["日期"]=df_p["日期"].astype(str); df_p=to_num(df_p)
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]
    total_chg=tw_p["充值金额"].sum(); total_tx=tw_p["提现金额"].sum()
    actual_cr=(total_chg-total_tx)/total_chg*100
    dep_tiers,wdraw_tiers=[],[]
    total_tw_c=dc_tw["充值金额"].sum(); total_tw_t=dt_tw["提款金额"].sum()
    for t in tiers:
        tw=dc_tw.head(t); lw=dc_lw.head(t)
        tw_c=tw["充值金额"].sum(); tw_tx=tw["提款金额"].sum()
        lw_c=lw["充值金额"].sum(); lw_tx=lw["提款金额"].sum()
        dep_tiers.append({"tier":f"Top{t}","tw_chg":tw_c,"lw_chg":lw_c,
            "tw_avg":tw_c/t,"lw_avg":lw_c/t,
            "tw_cr":(tw_c-tw_tx)/tw_c*100 if tw_c>0 else 0,
            "lw_cr":(lw_c-lw_tx)/lw_c*100 if lw_c>0 else 0,
            "tw_win":tw["公司输赢"].sum(),"占全量":tw_c/total_tw_c*100})
        twd=dt_tw.head(t); lwd=dt_lw.head(t)
        tw_t2=twd["提款金额"].sum(); tw_c2=twd["充值金额"].sum()
        lw_t2=lwd["提款金额"].sum(); lw_c2=lwd["充值金额"].sum()
        tw_cr2=(tw_c2-tw_t2)/tw_c2*100 if tw_c2>0 else 0
        lw_cr2=(lw_c2-lw_t2)/lw_c2*100 if lw_c2>0 else 0
        winners=(twd["公司输赢"]<0).sum()
        act=twd["活动奖励"].sum()/(tw_c2+twd["活动奖励"].sum())*100
        excl_chg=total_chg-tw_c2; excl_tx=total_tx-tw_t2
        excl_cr=(excl_chg-excl_tx)/excl_chg*100 if excl_chg>0 else 0
        impact=excl_cr-actual_cr
        wdraw_tiers.append({"tier":f"Top{t}","tw_tx":tw_t2,"lw_tx":lw_t2,
            "tw_avg":tw_t2/t,"lw_avg":lw_t2/t,
            "tw_cr":tw_cr2,"lw_cr":lw_cr2,"赢家":winners,"总数":t,"赢家率":winners/t*100,
            "活动占比":act,"占全量":tw_t2/total_tw_t*100,"大盘影响":impact})
    return dep_tiers,wdraw_tiers,dc_tw.head(200),dt_tw.head(20)

def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); df_lw=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); df_lw["阶段汇总"]=df_lw["阶段汇总"].apply(clean)
    bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]
    bet_lw=df_lw[df_lw["分析指标"]=="投注金额"]
    g_bet=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
    g_win=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
    g_lw=bet_lw.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
    g_usr=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([g_bet,g_win,g_lw,g_usr],axis=1).reset_index()
    total=g["本周投注"].sum(); g["占比"]=g["本周投注"]/total*100
    g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr

def load_games():
    df_tw=pd.read_excel(FILES["game_tw"]); df_lw=pd.read_excel(FILES["game_lw"])
    for df in [df_tw,df_lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢","人均投注局数","人均投注金额"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    total_tw=df_tw["投注金额"].sum()
    g_tw=df_tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    g_lw=df_lw.groupby("游戏.名称").agg(
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    g_tw["人均局数"]=g_tw["投注局数"]/g_tw["投注人数"]
    g_tw["人均金额"]=g_tw["投注金额"]/g_tw["投注人数"]
    g_tw["盈亏率"]=g_tw["公司输赢"]/g_tw["投注金额"]*100
    g_tw["占比"]=g_tw["投注金额"]/total_tw*100
    gm=g_tw.merge(g_lw,on="游戏.名称",how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)

def load_mfr():
    df=pd.read_csv(FILES["mfr"])
    df=df[df["时间"]!="阶段汇总"].copy()
    df["时间"]=df["时间"].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢"]: df[c]=df[c].apply(clean)
    tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
    def agg(d):
        g=d.groupby("show_name_厂商标签id").agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]
        g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100
        g["占比"]=g["投注金额"]/g["投注金额"].sum()*100
        return g
    tw_g=agg(tw); lw_g=agg(lw)
    mg=tw_g.merge(lw_g[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on="show_name_厂商标签id",how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    return mg.sort_values("投注金额",ascending=False)

def load_activities():
    df=pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数"]: df[c]=df[c].apply(clean)
    agg=df.groupby("name_账变opt_id").agg(
        赠送金额=("赠送金额","sum"),lw_赠送=("赠送金额.1","sum"),赠送人数=("赠送人数","sum")).reset_index()
    agg["环比"]=(agg["赠送金额"]-agg["lw_赠送"])/agg["lw_赠送"].abs()*100
    total=agg["赠送金额"].sum(); agg["占比"]=agg["赠送金额"]/total*100
    agg["人均"]=agg["赠送金额"]/agg["赠送人数"]
    return agg.sort_values("赠送金额",ascending=False).head(15), total

def load_deposit_src():
    df=pd.read_csv(FILES["deposit_src"])
    for c in ["充值金额","充值金额.1","充值人数"]: df[c]=df[c].apply(clean)
    tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str)
    df["道具ID"]=df["道具ID"].astype(str)
    agg=df.groupby("道具ID").agg(充值金额=("充值金额","sum"),lw_充值=("充值金额.1","sum"),
                                  充值人数=("充值人数","sum")).reset_index()
    agg=agg.merge(tm,on="道具ID",how="left")
    nm={"(null)":"无道具直接充值","71872":"幸运转盘10%","71873":"幸运转盘20%",
        "71975":"幸运翻卡道具","71894":"新人签到10%","71871":"新人签到20%"}
    agg["名称"]=agg.apply(lambda r: nm.get(str(r["道具ID"]),
        str(r.get("备注",""))[:12] if pd.notna(r.get("备注")) else f"其他({str(r['道具ID'])[:8]})"), axis=1)
    total=agg["充值金额"].sum(); agg["占比"]=agg["充值金额"]/total*100
    agg["环比"]=(agg["充值金额"]-agg["lw_充值"])/agg["lw_充值"].abs()*100
    return agg.sort_values("充值金额",ascending=False).head(6), total

def load_risk():
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    dt_tw=pd.read_csv(FILES["dt_tw"]); dc_tw=pd.read_csv(FILES["dc_tw"])
    for df2 in [dt_tw,dc_tw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df2.columns: df2[c]=pd.to_numeric(df2[c],errors="coerce")
    bet_amt=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    win_pref=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    bet_cnt=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数")
    top_game=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
              .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
              .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏",
                                "阶段汇总":"主游戏投注额"}).reset_index())
    ug=pd.concat([bet_cnt,bet_amt,win_pref],axis=1).reset_index()
    ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]
    ug=ug.merge(top_game,on="账户ID",how="left")
    top500=dt_tw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0
    top500["投充比"]=top500["投注金额"]/top500["充值金额"]
    high_risk=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
    pp_rul_ids=df[(df["游戏名称"]=="Auto-Roulette 1")&(df["分析指标"]=="投注次数")]["账户ID"].unique()
    cluster_207=[u for u in pp_rul_ids if str(u).startswith("207")]
    c207_dt=dt_tw[dt_tw["账户ID"].isin(cluster_207)].sort_values("提款金额",ascending=False)
    special=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False)
    bet_g=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
    win_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
    usr_g=df[df["分析指标"]=="投注次数"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
    winr_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bet_g.merge(win_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(usr_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(winr_g,on=["show_name_厂商标签id","游戏名称"],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100
    g["人均投注额"]=g["投注金额"]/g["玩家数"]
    risk_g=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    top20_games=g.sort_values("投注金额",ascending=False).head(20)
    return high_risk,c207_dt,cluster_207,special,top20_games,risk_g,top500

# ════════════════════════════════════════════
# 图表（保持不变）
# ════════════════════════════════════════════
SPLIT=7
def _vline(ax,dates):
    ax.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax.text(SPLIT-4,ax.get_ylim()[1]*0.93,"上周",ha="center",fontsize=7,color="#64748b")
    ax.text(SPLIT+3,ax.get_ylim()[1]*0.93,"本周",ha="center",fontsize=7,color="#1d4ed8")

def chart_trend(trend):
    fig=plt.figure(figsize=(16,10),facecolor="white")
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.45,wspace=0.3)
    dates=trend["dates"]; x=range(len(dates))
    def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=0.25)
    ax=fig.add_subplot(gs[0,0])
    clrs=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT
    ax.bar(x,[v/10000 for v in trend["充值"]],color=clrs,width=0.7,label="充值")
    ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
    ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold")
    ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
    ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax,dates)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax2=fig.add_subplot(gs[0,1])
    clrs2=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT
    ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=clrs2,width=0.7)
    ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold")
    ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7)
    sp(ax2); _vline(ax2,dates)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax3=fig.add_subplot(gs[1,0])
    clrs3=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT
    ax3.bar(x,trend["首充"],color=clrs3,width=0.7,label="首充人数")
    ax3r=ax3.twinx(); ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold")
    ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
    l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left")
    ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False)
    ax3.grid(axis="y",alpha=0.25); ax3.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax4=fig.add_subplot(gs[1,1])
    clrs4=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT
    ax4.bar(x,trend["充提差比"],color=clrs4,width=0.7)
    tw_mean=np.mean(trend["充提差比"][SPLIT:]); lw_mean=np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tw_mean,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lw_mean,color="#94a3b8",ls=":",lw=1.2)
    ax4.text(len(dates)-0.5,tw_mean+0.3,f"本周均{tw_mean:.1f}%",fontsize=7,color="#7c3aed",ha="right")
    ax4.text(0.5,lw_mean+0.3,f"上周均{lw_mean:.1f}%",fontsize=7,color="#64748b",ha="left")
    ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold")
    ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7)
    sp(ax4); _vline(ax4,dates)
    ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
    plt.tight_layout(rect=[0,0,1,0.98]); return fig_img(fig,16,11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white")
    cols_p=[("第1日","#1d4ed8"),("第2日","#059669"),("第3日","#d97706"),("第7日","#7c3aed")]
    x=np.arange(len(weeks)); w=0.18
    for i,(col,clr) in enumerate(cols_p):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-1.5*w,vals,w,label=col,color=clr,alpha=0.85)
        for bar,v in zip(bars,vals):
            if not np.isnan(v): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,
                                        f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周加权平均对比",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8,loc="upper right")
    ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=0.5)
    ax.text(2.6,ax.get_ylim()[1]*0.85,"↑本周",fontsize=7.5,color="#ef4444")
    ax.grid(axis="y",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.text(0.5,-0.02,f"注：本周({THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]})7日留存数据未完整",
             ha="center",fontsize=7.5,color="#64748b",style="italic")
    plt.tight_layout(); return fig_img(fig,16,5.5)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=0.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tw_c=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_c=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tw_c,w,label="本周",color="#1d4ed8",alpha=0.85)
    ax1.bar(x+w/2,lw_c,w,label="上周",color="#93c5fd",alpha=0.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold")
    ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7)
    ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=0.3)
    ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tw_b=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_b=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tw_b,w,label="本周",color="#059669",alpha=0.85)
    ax2.bar(x+w/2,lw_b,w,label="上周",color="#6ee7b7",alpha=0.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7)
    ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=0.3)
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[]
    for v in vips:
        if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0:
            chg.append((tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100)
        else: chg.append(0)
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=0.85,width=0.6)
    ax3.axhline(0,color="black",lw=0.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold")
    ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7)
    ax3.grid(axis="y",alpha=0.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vip_ret):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    chg_tw=[vip_ret['chg']['tw']['1日'].get(v,0) for v in vips]
    chg_lw=[vip_ret['chg']['lw']['1日'].get(v,0) for v in vips]
    chg_3d=[vip_ret['chg']['tw']['3日'].get(v,0) for v in vips]
    act_tw=[vip_ret['act']['tw']['1日'].get(v,0) for v in vips]
    act_lw=[vip_ret['act']['lw']['1日'].get(v,0) for v in vips]
    act_3d=[vip_ret['act']['tw']['3日'].get(v,0) for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white")
    x=np.arange(len(labels)); w=0.28
    ax1.bar(x-w,chg_tw,w,label="次日留存(本周)",color="#1d4ed8",alpha=0.85)
    ax1.bar(x,chg_lw,w,label="次日留存(上周)",color="#93c5fd",alpha=0.7)
    ax1.bar(x+w,chg_3d,w,label="3日留存(本周)",color="#059669",alpha=0.75)
    ax1.set_title("充值→充值 留存率（%）",fontsize=10,fontweight="bold")
    ax1.set_xticks(x); ax1.set_xticklabels(labels,fontsize=8)
    ax1.legend(fontsize=7.5,loc="upper left")
    ax1.grid(axis="y",alpha=0.25); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    for i,(tv,d3) in enumerate(zip(chg_tw,chg_3d)):
        ax1.text(x[i]-w,tv+0.3,f"{tv:.0f}%",ha="center",fontsize=6,color="#1d4ed8")
        ax1.text(x[i]+w,d3+0.3,f"{d3:.0f}%",ha="center",fontsize=6,color="#059669")
    ax2.bar(x-w,act_tw,w,label="次日留存(本周)",color="#7c3aed",alpha=0.85)
    ax2.bar(x,act_lw,w,label="次日留存(上周)",color="#c4b5fd",alpha=0.7)
    ax2.bar(x+w,act_3d,w,label="3日留存(本周)",color="#d97706",alpha=0.75)
    ax2.set_title("充值→活跃 留存率（%）",fontsize=10,fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels(labels,fontsize=8)
    ax2.legend(fontsize=7.5,loc="upper left")
    ax2.grid(axis="y",alpha=0.25); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    for i,(tv,d3) in enumerate(zip(act_tw,act_3d)):
        ax2.text(x[i]-w,tv+0.5,f"{tv:.0f}%",ha="center",fontsize=6,color="#7c3aed")
        ax2.text(x[i]+w,d3+0.5,f"{d3:.0f}%",ha="center",fontsize=6,color="#d97706")
    plt.suptitle("VIP各等级充值留存率（本周 vs 上周，加权日均值）",fontsize=11,fontweight="bold",y=1.02)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10); names=top10["show_name_厂商标签id"].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    shares=top10["占比"].tolist(); lw_shares=top10["lw_占比"].fillna(0).tolist()
    bar_clrs=["#059669" if cur>=lw else "#dc2626" for cur,lw in zip(shares,lw_shares)]
    bars=ax1.bar(names,shares,color=bar_clrs,alpha=0.85,width=0.6,label="本周份额")
    ax1.plot(names,lw_shares,"--o",color="#64748b",lw=1.5,ms=5,label="上周份额",zorder=5)
    for bar,s,ic in zip(bars,shares,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,f"{s:.1f}%",ha="center",fontsize=7,color="#1e293b")
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额（本周 vs 上周）",fontsize=9,fontweight="bold")
    ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    gp=mpatches.Patch(color="#059669",label="份额增长"); rp=mpatches.Patch(color="#dc2626",label="份额下降")
    ax1.legend(handles=[gp,rp,plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=0.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",
            color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=0.85)
    ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=0.7)
    ax2.axhline(0,color="black",lw=0.8)
    ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5)
    ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=0.3)
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    top15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in top15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in top15.iterrows()]
    lw_bets=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in top15.iterrows()]
    x=np.arange(len(names)); w=0.35
    fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=0.85)
    ax.barh(x-w/2,lw_bets,w,label="上周",color="#93c5fd",alpha=0.7)
    for bar,v in zip(ax.patches[:len(names)],bets[::-1]):
        ax.text(bar.get_width()+0.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8)
    ax.set_title("Top15游戏 投注金额（万USD，本周 vs 上周）",fontsize=9,fontweight="bold")
    ax.legend(fontsize=8); ax.grid(axis="x",alpha=0.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pref_game,mfr_amt):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=mfr_amt.index[:8].tolist(); vals=mfr_amt.values[:8].tolist(); total=sum(vals)
    pcts=[v/total*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    def ap(p): return f"{p:.1f}%" if p>4 else ""
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=ap,startangle=90,
                          pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],
               loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.05,-0.18))
    ax1.set_title("Top500提款用户 厂商偏好（按投注金额）",fontsize=9,fontweight="bold")
    top12=pref_game.head(12)
    gnames=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in top12.iterrows()]
    gvals=[r["本周投注"]/10000 for _,r in top12.iterrows()]
    gclrs=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in top12.iterrows()]
    ax2.barh(range(len(gnames))[::-1],gvals,color=gclrs[::-1],alpha=0.85)
    ax2.set_yticks(range(len(gnames))); ax2.set_yticklabels(gnames,fontsize=7.5)
    ax2.set_title("偏好游戏Top12（按投注金额，红=平台亏损）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万"))
    ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act_df):
    top10=act_df.head(10); names=[str(r["name_账变opt_id"])[:10] for _,r in top10.iterrows()]
    vals=[r["赠送金额"]/10000 for _,r in top10.iterrows()]
    lw=[r["lw_赠送"]/10000 for _,r in top10.iterrows()]
    envs=[r["环比"] for _,r in top10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    x=np.arange(len(names)); w=0.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=0.85)
    ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=0.7)
    ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8)
    ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold")
    ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=0.3)
    ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ec=[("#059669" if v>0 else "#dc2626") for v in envs[::-1]]
    ax2.barh(range(len(names)),envs[::-1],color=ec,alpha=0.85)
    ax2.axvline(0,color="black",lw=0.8)
    ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8)
    ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_deposit_src(src_df,total):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white")
    names=src_df["名称"].tolist(); vals=src_df["充值金额"].tolist(); lw_v=src_df["lw_充值"].tolist()
    pcts=src_df["占比"].tolist(); clrs=CHART_COLORS[:len(names)]
    def ap(p): return f"{p:.1f}%" if p>3 else ""
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=ap,startangle=140,
                          pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:8]}({p:.1f}%)" for n,p in zip(names,pcts)],
               loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.1,-0.15))
    ax1.set_title(f"存款来源占比（总计${total/10000:.0f}万USD）",fontsize=9,fontweight="bold")
    x=np.arange(len(names)); w=0.35
    ax2.bar(x-w/2,[v/10000 for v in vals],w,label="本周",color=clrs,alpha=0.85)
    ax2.bar(x+w/2,[v/10000 for v in lw_v],w,label="上周",color=["#93c5fd"]*len(names),alpha=0.6)
    ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=25,ha="right",fontsize=7.5)
    ax2.set_title("存款道具 本周 vs 上周（万USD）",fontsize=9,fontweight="bold")
    ax2.legend(fontsize=8); ax2.grid(axis="y",alpha=0.3)
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_risk_scatter(top500):
    valid=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy()
    valid=valid[valid["投充比"]<200]
    clrs=["#dc2626" if v<0 else "#059669" for v in valid["公司输赢"]]
    sizes=[min(abs(v)/500+20,200) for v in valid["公司输赢"]]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    ax.scatter(valid["投充比"],valid["公司输赢"]/10000,c=clrs,s=sizes,alpha=0.55,edgecolors="none")
    ax.axhline(0,color="black",lw=0.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=0.7)
    ax.text(21,ax.get_ylim()[0]*0.9,"投充比=20x",fontsize=7.5,color="#d97706")
    highlights=top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()]
    for _,r in highlights.iterrows():
        ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),
                    xytext=(r["投充比"]+5,r["公司输赢"]/10000-0.2),fontsize=6.5,color="#991b1b",
                    arrowprops=dict(arrowstyle="->",color="#991b1b",lw=0.7))
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8)
    ax.set_title("Top500提款用户：投充比 vs 公司输赢（气泡大小=输赢绝对值）",fontsize=9,fontweight="bold")
    rp=mpatches.Patch(color="#dc2626",alpha=0.6,label="平台输钱（玩家赢）")
    gp=mpatches.Patch(color="#059669",alpha=0.6,label="平台赢钱（玩家输）")
    ax.legend(handles=[rp,gp],fontsize=8,loc="upper right"); ax.grid(alpha=0.25)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(risk_g):
    top12=risk_g.head(12); names=[r["游戏名称"][:16] for _,r in top12.iterrows()]
    losses=[abs(r["公司输赢"]) for _,r in top12.iterrows()]
    rates=[r["盈亏率"] for _,r in top12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=0.85)
    for bar,v in zip(ax1.patches,losses[::-1]):
        ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold")
    ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k"))
    ax1.tick_params(axis="y",labelsize=7.5)
    br_clrs=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]]
    ax2.barh(names[::-1],rates[::-1],color=br_clrs,alpha=0.85)
    ax2.axvline(0,color="black",lw=0.8)
    ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold")
    ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)

# ════════════════════════════════════════════
# 章节构建
# ════════════════════════════════════════════
def build_overview(K,trend,weekly_ret):
    S=[sec_title("一、大盘核心数据")]
    kpis=[
        ("充值金额",f"{K['tw_充值金额']/10000:.1f}万",f"{K['lw_充值金额']/10000:.1f}万",K["pct_充值金额"],True),
        ("提现金额",f"{K['tw_提现金额']/10000:.1f}万",f"{K['lw_提现金额']/10000:.1f}万",K["pct_提现金额"],False),
        ("充提差",f"{K['tw_充提差']/10000:.1f}万",f"{K['lw_充提差']/10000:.1f}万",K["pct_充提差"],True),
        ("充提差率",f"{K['tw_充提差比']:.2f}%",f"{K['lw_充提差比']:.2f}%",K["pct_充提差比"],True),
        ("公司输赢",f"{K['tw_公司输赢']/10000:.1f}万",f"{K['lw_公司输赢']/10000:.1f}万",K["pct_公司输赢"],True),
        ("盈亏率",f"{K['tw_盈亏率']:.3f}%",f"{K['lw_盈亏率']:.3f}%",K["pct_盈亏率"],True),
        ("注册人数",f"{int(K['tw_注册人数']):,}",f"{int(K['lw_注册人数']):,}",K["pct_注册人数"],True),
        ("首充人数",f"{int(K['tw_首充人数']):,}",f"{int(K['lw_首充人数']):,}",K["pct_首充人数"],True),
        ("日均活跃",f"{K['tw_活跃人数']/7/10000:.1f}万",f"{K['lw_活跃人数']/7/10000:.1f}万",K["pct_活跃人数"],True),
        ("投注金额",f"{K['tw_投注金额']/10000:.0f}万",f"{K['lw_投注金额']/10000:.0f}万",K["pct_投注金额"],True),
        ("全量日均ARPPU",f"${K['tw_全量Arppu']:.2f}",f"${K['lw_全量Arppu']:.2f}",K["pct_全量Arppu"],True),
        ("老用户日均ARPPU",f"${K['tw_老用户ARPPU']:.2f}",f"${K['lw_老用户ARPPU']:.2f}",K["pct_老用户ARPPU"],True),
        ("首充日均ARPPU",f"${K['tw_首充Arppu']:.2f}",f"${K['lw_首充Arppu']:.2f}",K["pct_首充Arppu"],True),
        ("总赠送金额",f"{K['tw_总赠送金额']/10000:.1f}万",f"{K['lw_总赠送金额']/10000:.1f}万",K["pct_总赠送金额"],False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",f"{K['lw_赠送充值比']:.2f}%",K["pct_赠送充值比"],False),
        ("首充转化率",f"{K['tw_首充转化率']:.1f}%",f"{K['lw_首充转化率']:.1f}%",K["pct_首充转化率"],True),
        ("首充次日充值留存",f"{K['tw_首充次日复充率']:.1f}%",f"{K['lw_首充次日复充率']:.1f}%",K["pct_首充次日复充率"],True),
        ("推广消耗",f"{K['tw_真实消耗']/10000:.1f}万",f"{K['lw_真实消耗']/10000:.1f}万",K["pct_真实消耗"],False),
    ]
    S.append(kpi_card4(kpis,cols=4)); S.append(Spacer(1,8))
    cr_delta=K["tw_充提差比"]-K["lw_充提差比"]
    ret_delta=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(insight_box([
        f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%）；推广消耗{K['pct_真实消耗']:+.1f}%，充值增速{'高于' if K['pct_充值金额']>abs(K['pct_真实消耗']) else '低于'}消耗变化幅度。",
        f"充提差率{K['tw_充提差比']:.2f}%（上周{K['lw_充提差比']:.2f}%，{cr_delta:+.2f}pp）；盈亏率{K['tw_盈亏率']:.3f}%（上周{K['lw_盈亏率']:.3f}%）。",
        f"首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），首充转化率{K['tw_首充转化率']:.1f}%，首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%）。",
        f"首充次日充值留存{K['tw_首充次日复充率']:.1f}%（上周{K['lw_首充次日复充率']:.1f}%，{ret_delta:+.1f}pp）。",
    ]))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
    S.append(sub_title("首充用户充值留存率 — 近4周加权平均对比")); S.append(chart_ret_weekly(weekly_ret))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=(this_w.get("第1日",0)or 0)-(last_w.get("第1日",0)or 0)
    S.append(insight_box([
        f"本周首充次日留存{this_w.get('第1日',0):.1f}%，较上周{d1:+.1f}pp；2日留存{this_w.get('第2日',0):.1f}%。",
        f"注：本周7日留存因数据截止日期无法完整计算，以上周7日留存{last_w.get('第7日',0):.1f}%作为近期参考基准。",
    ], clr=C_AMBER))
    return S

# ════════════════════════════════════════════
# v5.4 总代章节：全面修复三大问题
# ════════════════════════════════════════════
def build_agents(agents, agent_ret):
    """
    agent_ret: dict { name_总代 -> {tw_d1, tw_d3, lw_d1, lw_d7} }

    列格式修复：
      - 注册(环比)  → "22,072/+27.1%"  单行，颜色跟环比正负
      - 充提差率(差值) → "14.8%/-3.9pp"  单行，颜色跟差值正负
      - 次日留存(本/上) → "18.2%/22.1%"  本周低于上周→红，否则→绿
      - 3留本周 → 本周3日加权留存
      - 7留上周 → 上周7日加权留存（本周7日未到期）
    """
    S = [sec_title("二、总代分析（剔除总代0官方总代）")]
    S.append(sub_title("Top15总代表现（本周 vs 上周，按充值金额排序）"))
    full = PW - 2*MARGIN

    headers = [
        "ID", "总代名称",
        "注册(环比)",        # 合并列：22,072/+27.1%
        "首充",
        "充值\n(万)",
        "充提差率(差值)",    # 合并列：14.8%/-3.9pp
        "消耗\n(万)",
        "一级首充\n成本(本/上)",
        "次日留存\n(本/上)",  # 本周1日 / 上周1日
        "3留\n本周",          # 本周3日
        "7留\n上周",          # 上周7日
    ]

    def _fmt_ret(v):
        """格式化留存率，nan 返回 '-'"""
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "-"
        return f"{v:.1f}%"

    rows = []
    for _, r in agents.iterrows():
        # ── 基础数据 ──
        agent_id = int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
        rname    = str(r["总代.名称"])
        cr       = r["充提差率"]
        lw_cr    = r.get("lw_充提差率", 0) or 0
        cr_diff  = cr - lw_cr
        cost     = (r.get("总消耗", 0) or 0) / 10000
        exp_str  = f"{cost:.1f}" if cost > 0 else "-"
        fc_c     = r.get("一级首充成本",    0) or 0
        lw_fc_c  = r.get("lw_一级首充成本", 0) or 0
        fc_str   = f"${fc_c:.0f}/${lw_fc_c:.0f}" if fc_c > 0 else "-"

        # ── FIX: 注册(环比) → 单行 "22,072/+27.1%" ──
        reg     = int(r["注册人数"])
        reg_chg = r.get("注册环比", 0) or 0
        reg_str = f"{reg:,}/{'+' if reg_chg>=0 else ''}{reg_chg:.1f}%"
        reg_clr = C_GREEN if reg_chg >= 0 else C_RED

        # ── FIX: 充提差率(差值) → 单行 "15.7%/-1.1pp" ──
        cr_str = f"{cr:.1f}%/{'+' if cr_diff>=0 else ''}{cr_diff:.1f}pp"
        cr_clr = C_GREEN if cr >= 17 else (C_RED if cr < 5 else C_DARK)
        # 差值颜色：差值为正则绿，负则红（用 Paragraph 富文本不方便，取充提差率本身颜色即可）

        # ── FIX: 次日留存(本/上) 从 agent_ret 取数 ──
        ret_data = agent_ret.get(rname, {})
        tw_d1 = ret_data.get("tw_d1", np.nan)
        lw_d1 = ret_data.get("lw_d1", np.nan)
        tw_d3 = ret_data.get("tw_d3", np.nan)
        lw_d7 = ret_data.get("lw_d7", np.nan)

        # 次日留存显示 "本周值/上周值"
        if not np.isnan(tw_d1) and not np.isnan(lw_d1):
            nd_str = f"{tw_d1:.1f}%/{lw_d1:.1f}%"
            # 本周低于上周 → 红；否则 → 绿
            nd_clr = C_GREEN if tw_d1 >= lw_d1 else C_RED
        elif not np.isnan(tw_d1):
            nd_str = f"{tw_d1:.1f}%/-"
            nd_clr = C_DARK
        else:
            nd_str = "-"
            nd_clr = C_GRAY

        d3_str = _fmt_ret(tw_d3)
        d3_clr = C_DARK
        d7_str = _fmt_ret(lw_d7)
        d7_clr = C_DARK

        rows.append([
            cell(str(agent_id),         False, C_GRAY,  TA_CENTER),
            cell(rname[:16],            True,  C_DARK,  TA_LEFT),
            cell(reg_str,               False, reg_clr, TA_RIGHT),
            cell(f"{int(r['首充人数']):,}", False, C_DARK, TA_RIGHT),
            cell(f"{r['充值金额']/10000:.0f}", False, C_DARK, TA_RIGHT),
            cell(cr_str,                False, cr_clr,  TA_RIGHT),
            cell(exp_str,               False, C_DARK,  TA_RIGHT),
            cell(fc_str,                False, C_DARK,  TA_RIGHT),
            cell(nd_str,                False, nd_clr,  TA_RIGHT),
            cell(d3_str,                False, d3_clr,  TA_RIGHT),
            cell(d7_str,                False, d7_clr,  TA_RIGHT),
        ])

    # 列宽（共11列，合计=1.0）
    cw = [full*x for x in [0.05, 0.17, 0.11, 0.06, 0.06, 0.12, 0.06, 0.11, 0.10, 0.08, 0.08]]
    S.append(dtable(headers, rows, cw, fsize=6.5))
    S.append(Spacer(1, 6))

    # 解读
    dsp_rows = agents[agents["总代.名称"].str.contains("DSP", na=False)]
    dsp_cr   = dsp_rows["充提差率"].values[0] if len(dsp_rows) > 0 else 0
    dsp_reg  = dsp_rows["注册环比"].values[0] if len(dsp_rows) > 0 else 0
    S.append(insight_box([
        f"A8_DSP_谷歌原生包充提差率{dsp_cr:.1f}%，注册+{dsp_reg:.1f}%，体量第三且指标双优。",
        "苹果包总代注册+27.1%，充值体量第二，为本周注册增量最大渠道之一。",
        "YB_FB_PWA_0注册暴跌-79%至1,315人，需持续监控是否恢复。",
    ]))

    # ── 总代留存汇总表（本周 vs 上周，全量每日加权）──
    S.append(sub_title("总代首充充值留存率（每日加权均值，本周 vs 上周）"))
    S.append(P("说明：次日/3日 取本周日均加权，7日 取上周日均加权（本周7日尚未到期）",
               7.5, False, C_GRAY))
    S.append(Spacer(1, 4))

    headers2 = ["总代名称", "次日留存\n本/上周", "3日留存\n本周", "7日留存\n上周"]
    rows2 = []
    # 优先展示 agents 中 Top15 的顺序，再追加其余
    top15_names = [str(r["总代.名称"]) for _, r in agents.iterrows()]
    extra_names  = [n for n in agent_ret if n not in top15_names and n != "总体"]
    all_names    = top15_names + sorted(extra_names)
    # 将"总体"放最后
    if "总体" in agent_ret and "总体" not in all_names:
        all_names.append("总体")

    for nm in all_names[:20]:
        d = agent_ret.get(nm, {})
        tw1 = d.get("tw_d1", np.nan); lw1 = d.get("lw_d1", np.nan)
        tw3 = d.get("tw_d3", np.nan); lw7 = d.get("lw_d7", np.nan)

        if not np.isnan(tw1) and not np.isnan(lw1):
            nd_s = f"{tw1:.1f}%/{lw1:.1f}%"
            nd_c = C_GREEN if tw1 >= lw1 else C_RED
        elif not np.isnan(tw1):
            nd_s = f"{tw1:.1f}%/-"; nd_c = C_DARK
        else:
            nd_s = "-"; nd_c = C_GRAY

        rows2.append([
            cell(nm[:22], True, C_DARK, TA_LEFT),
            cell(nd_s,    False, nd_c,  TA_RIGHT),
            cell(_fmt_ret(tw3), False, C_DARK, TA_RIGHT),
            cell(_fmt_ret(lw7), False, C_DARK, TA_RIGHT),
        ])

    if rows2:
        S.append(dtable(headers2, rows2,
                        [full*x for x in [0.52, 0.16, 0.16, 0.16]], fsize=6.8))
    return S


def build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw):
    S=[sec_title("三、用户分析")]
    full=PW-2*MARGIN
    S.append(sub_title("VIP等级分层分析（充值/投注金额 本周 vs 上周）"))
    S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
    total_chg=tw_v["充值金额"].sum()
    high_vip_pct=(tw_v.loc[9,"充值金额"]+tw_v.loc[10,"充值金额"]+tw_v.loc[11,"充值金额"])/total_chg*100 if 9 in tw_v.index else 0
    S.append(insight_box([
        f"VIP9-11高价值层合计贡献充值{high_vip_pct:.1f}%，高端用户付费意愿持续强劲。",
        f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。",
    ]))
    vip_ret=load_vip_retention()
    S.append(sub_title("VIP各等级 充值→充值留存率 & 充值→活跃留存率（本周均值）"))
    S.append(chart_vip_retention(vip_ret)); S.append(Spacer(1,4))
    S.append(insight_box([
        f"充值→活跃次日留存：VIP9达{vip_ret['act']['tw']['1日'].get(9,0):.1f}%，VIP10达{vip_ret['act']['tw']['1日'].get(10,0):.1f}%，高VIP活跃粘性强。",
        f"充值→充值次日留存：VIP10达{vip_ret['chg']['tw']['1日'].get(10,0):.1f}%（上周{vip_ret['chg']['lw']['1日'].get(10,0):.1f}%），高VIP复充意愿强。",
    ]))
    S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析（Top1/10/50/100/200/500）"))
    headers=["分层","本周充值","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        chg_avg=pct(d["tw_avg"],d["lw_avg"])
        rows.append([
            cell(d["tier"],True),
            cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),
            cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
            rc(chg_avg),
            cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),
            cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
            cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
            cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT),
        ])
    cw2=[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]]
    S.append(dtable(headers,rows,cw2)); S.append(Spacer(1,6))
    S.append(insight_box([
        f"Top10充值用户人均${dep_t[1]['tw_avg']:,.0f}（{pct(dep_t[1]['tw_avg'],dep_t[1]['lw_avg']):+.1f}%），充提差率{dep_t[1]['tw_cr']:.1f}%。",
        "Top50以下充提差率均已转正，中腰部用户资金沉淀健康。",
    ]))
    S.append(sub_title("头部提款用户分层分析（Top1/10/50/100/200/500）"))
    headers3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","赢家\n比例","活动\n占比","占全\n量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdraw_t:
        chg=pct(d["tw_avg"],d["lw_avg"])
        rows3.append([
            cell(d["tier"],True),
            cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),
            cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
            rc(chg,good_up=False),
            cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),
            cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
            cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),
            cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),
            cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
            cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT),
        ])
    cw3=[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]]
    S.append(dtable(headers3,rows3,cw3,fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box([
        f"剔除Top100提款用户后，大盘充提差率影响+{wdraw_t[3]['大盘影响']:.2f}pp，头部提款用户对充提差率有明显拖累。",
        f"Top500提款用户活动奖励占资金来源仅{wdraw_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。",
    ], clr=C_AMBER))
    return S

def build_games_section(mfr,top30):
    S=[sec_title("四、游戏分析")]
    full=PW-2*MARGIN
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比"))
    S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    headers=["排名","厂商","日均投注人数\n(本/上)","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([
            cell(str(i+1),False,C_GRAY,TA_CENTER),
            cell(str(r["show_name_厂商标签id"])[:10],True),
            cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),
            cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
            cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),
            rc(r.get("投注环比",0)),
            cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
            cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
            cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT),
        ])
    cw=[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]]
    S.append(dtable(headers,rows,cw,fsize=6.5)); S.append(Spacer(1,4))
    S.append(insight_box([
        f"Tada投注份额{mfr.iloc[0]['占比']:.1f}%（上周{mfr.iloc[0].get('lw_占比',0):.1f}%），Fortune系稳固主导；环比{mfr.iloc[0].get('投注环比',0):+.1f}%。",
        f"Evolution盈亏率{mfr[mfr['show_name_厂商标签id']=='Evolution']['盈亏率'].values[0]:.2f}%（上周{mfr[mfr['show_name_厂商标签id']=='Evolution']['lw_盈亏率'].values[0]:.2f}%），已转正，较上周大幅改善。",
    ]))
    S.append(sub_title("Top30游戏详细数据（本周 vs 上周）"))
    S.append(chart_top30(top30)); S.append(Spacer(1,4))
    headers2=["#","游戏名称","厂商","投注\n人数","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([
            cell(str(i+1),False,C_GRAY,TA_CENTER),
            cell(str(r["游戏.名称"])[:18],True),
            cell(str(r["游戏厂商标签.名称"])[:7]),
            cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),
            cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
            cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),
            rc(r.get("投注环比",0)),
            cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),
            cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
            cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT),
        ])
    cw2=[full*x for x in [0.04,0.18,0.08,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]]
    S.append(dtable(headers2,rows2,cw2,fsize=6.5))
    return S

def build_activities_section(act_df,total_gift,src_df,total_dep):
    S=[sec_title("五、活动分析")]
    full=PW-2*MARGIN
    S.append(sub_title("各活动赠送效果（本周 vs 上周，含环比）"))
    S.append(chart_activities(act_df)); S.append(Spacer(1,4))
    headers=["活动名称","本周赠送","上周赠送","环比","赠送人次","人均赠送","金额占比"]
    rows=[]
    for _,r in act_df.iterrows():
        rows.append([
            cell(str(r["name_账变opt_id"])[:16],True),
            cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${r['lw_赠送']:,.0f}",False,C_GRAY,TA_RIGHT),
            rc(r["环比"]),
            cell(f"{r['赠送人数']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${r['人均']:.2f}",False,C_DARK,TA_RIGHT),
            cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT),
        ])
    S.append(dtable(headers,rows,[full*x for x in [0.22,0.14,0.14,0.1,0.14,0.12,0.14]]))
    S.append(Spacer(1,4))
    S.append(P(f"本周总赠送金额：${total_gift/10000:.2f}万 | 赠送/充值比：{total_gift/total_dep*100:.2f}%",9,True,C_DARK))
    S.append(Spacer(1,8))
    S.append(sub_title("存款来源道具分析（充值金额占比）"))
    S.append(chart_deposit_src(src_df,total_dep)); S.append(Spacer(1,4))
    headers2=["道具/来源","本周充值(万)","上周充值(万)","环比","充值占比"]
    rows2=[]
    for _,r in src_df.iterrows():
        rows2.append([
            cell(str(r["名称"])[:18],True),
            cell(f"{r['充值金额']/10000:.2f}",False,C_DARK,TA_RIGHT),
            cell(f"{r['lw_充值']/10000:.2f}",False,C_GRAY,TA_RIGHT),
            rc(r["环比"]),
            cell(f"{r['占比']:.1f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
        ])
    S.append(dtable(headers2,rows2,[full*x for x in [0.35,0.15,0.15,0.15,0.2]]))
    S.append(Spacer(1,4))
    S.append(insight_box([
        f"存款道具合计带动充值${(total_dep-src_df.iloc[0]['充值金额'])/10000:.1f}万，占总充值{(1-src_df.iloc[0]['占比']/100)*100:.1f}%。",
        f"幸运翻卡道具（71975）充值{src_df[src_df['名称']=='幸运翻卡道具']['充值金额'].values[0]/10000:.2f}万（{src_df[src_df['名称']=='幸运翻卡道具']['环比'].values[0]:+.1f}%），原因主要是：幸运翻卡-钻石奖励有调整：钻石机率10%→5%，道具几率30%→35%。",
    ], clr=C_AMBER))
    return S

def build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw_full):
    S=[sec_title("六、用户游戏风险专项分析",clr=colors.HexColor("#7c2d12"))]
    full=PW-2*MARGIN
    total_tx=top500["提款金额"].sum(); total_win=top500["公司输赢"].sum()
    winner_cnt=(top500["公司输赢"]<0).sum()
    S.append(sub_title("Top500提款用户总览"))
    kpi_items=[
        ("Top500提款总额",f"${total_tx/10000:.1f}万",""),
        ("平台净赔付",f"${abs(total_win)/10000:.1f}万","平台向该群体净赔"),
        ("赢家比例",f"{winner_cnt/500*100:.1f}%",f"{winner_cnt}赢/{500-winner_cnt}输"),
        ("高风险用户",f"{len(high_risk)}人","公司净输>$10,000"),
    ]
    row_k=[]
    for label,val,sub in kpi_items:
        fw=full/4-4
        inner=Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],
                    colWidths=[fw],style=TableStyle([
                        ("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),
                        ("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
        row_k.append(inner)
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）"))
    S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    S.append(insight_box([
        f"Top500提款用户中赢家{winner_cnt}人（{winner_cnt/500*100:.1f}%），平台净赔付${abs(total_win)/10000:.1f}万。",
        f"{len(high_risk)}名超级赢家（公司净输>$10,000）合计导致平台净输${abs(high_risk['公司输赢'].sum())/10000:.1f}万，占整体净赔付{abs(high_risk['公司输赢'].sum())/abs(total_win)*100:.1f}%。",
    ]))
    S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
    headers_r=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注\n均额","主玩厂商","主玩游戏","风险标签"]
    rows_r=[]
    for _,r in high_risk.iterrows():
        ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0
        avg_b=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
        tags=[]
        if r["充值金额"]<5000: tags.append("低充高提")
        if ratio>50: tags.append("超高投充")
        if avg_b>200: tags.append("高额单注")
        tag="+".join(tags) if tags else "超级赢家"
        rows_r.append([
            cell(str(r["账户ID"]),True),
            cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),
            cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),
            cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),
            cell(f"${avg_b:.0f}",False,C_RED if avg_b>200 else C_DARK,TA_RIGHT),
            cell(str(r.get("主玩厂商",""))[:8]),
            cell(str(r.get("主玩游戏",""))[:16]),
            cell(tag,True,C_RED),
        ])
    cw_r=[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]]
    S.append(dtable(headers_r,rows_r,cw_r,fsize=6.5)); S.append(Spacer(1,6))
    if len(cluster_207)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警"))
        cluster_total_tx=float(c207_dt["提款金额"].sum()); cluster_total_win=float(c207_dt["公司输赢"].sum())
        S.append(P(f"集群账户数：{len(cluster_207)}个 | 合计提款：${cluster_total_tx/10000:.1f}万 | 平台净赔：${abs(cluster_total_win)/10000:.1f}万",8.5,True,C_DARK))
        S.append(Spacer(1,6))
        headers_c=["账户ID","提款金额","充值金额","公司输赢","特征"]
        rows_c=[]
        for _,r in c207_dt.head(10).iterrows():
            feat="充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}"
            rows_c.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),
                           cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),
                           cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(feat)])
        if len(c207_dt)>10: rows_c.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
        S.append(dtable(headers_c,rows_c,[full*x for x in [0.22,0.18,0.18,0.18,0.24]]))
        S.append(Spacer(1,6))
        S.append(insight_box([
            f"{len(cluster_207)}个账户充值$547-548，均获活动奖励$270.92，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。",
            "建议：①冻结账户提款；②审查注册IP/设备指纹；③排查$270.92奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。",
        ], clr=C_AMBER))
    if len(special_users)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ 特殊异常用户（充值=0 或 投充比>150x）"))
        headers_s=["账户ID","提款金额","充值金额","投注金额","公司输赢","异常说明"]
        rows_s=[]
        for _,r in special_users.head(6).iterrows():
            note=f"充值${r['充值金额']:.0f}，投充比{r.get('投充比',0):.0f}x，行为异常" if r["充值金额"]==0 else "投充比异常极高"
            rows_s.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),
                           cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]==0 else C_AMBER,TA_RIGHT),
                           cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),
                           cell(f"${r['公司输赢']:,.0f}",False,C_RED if r["公司输赢"]<0 else C_DARK,TA_RIGHT),
                           cell(note,False,C_GRAY)])
        S.append(dtable(headers_s,rows_s,[full*x for x in [0.12,0.1,0.1,0.1,0.1,0.48]]))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    headers_t=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]
    rows_t=[]
    for i,(_,r) in enumerate(dt_tw_full.head(20).iterrows()):
        win=r["公司输赢"]<0
        main_game=str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-"
        rows_t.append([
            cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),
            cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),
            cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(main_game,False,C_GRAY),
            cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER),
        ])
    S.append(dtable(headers_t,rows_t,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5))
    S.append(Spacer(1,6))
    pref_game,mfr_amt=load_pref()
    S.append(sub_title("Top500提款用户游戏偏好（按投注金额）"))
    S.append(chart_pref(pref_game,mfr_amt)); S.append(Spacer(1,4))
    S.append(sub_title("▶ 高危游戏专项分析（Top500提款用户视角）"))
    S.append(chart_risk_games(risk_g)); S.append(Spacer(1,4))
    headers_g=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]
    rows_g=[]
    for _,r in risk_g.head(12).iterrows():
        pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注")
        rl_clr=C_RED if rl in ("极高","高") else C_AMBER
        rows_g.append([
            cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),
            cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),
            cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),
            cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),
            cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),
            cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),
            cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),
            cell(rl,True,rl_clr),
        ])
    S.append(dtable(headers_g,rows_g,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5))
    S.append(Spacer(1,6))
    S.append(insight_box([
        "PP Auto-Roulette 1批量套利模式持续，建议暂停该桌并复核奖励规则。",
        "Fortune Garuda 500（Tada）是Top500提款用户投注规模最大游戏，建议对RTP参数做紧急复审。",
    ], clr=C_RED))
    return S

def build_conclusion(K,agents,mfr,dep_t,wdraw_t):
    S=[sec_title("七、总结与行动建议")]
    full=PW-2*MARGIN
    S.append(sub_title("▶ 本周亮点"))
    highlights=[
        f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），充提差{K['tw_充提差']/10000:.1f}万（{K['pct_充提差']:+.1f}%），充提差率{K['tw_充提差比']:.2f}%（{K['tw_充提差比']-K['lw_充提差比']:+.2f}pp）。",
        f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。",
        "A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。",
    ]
    for h in highlights:
        S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#f0fdf4")),
            ("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))
    S.append(sub_title("▶ 风险预警"))
    risks=[
        "在23号提供了一份风险名单待风险排查：①用户输赢大于1万，②怀疑Auto-Roulette 1套利的用户，③充值低、提款高、投充比高的用户。",
        "PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。",
        f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp），新用户质量需关注。",
        "YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。",
    ]
    for r in risks:
        S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#fff5f5")),
            ("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))
    S.append(sub_title("▶ 行动建议"))
    actions=[
        ("[紧急]","处置PP Auto-Roulette 1批量套利",
         "33个207开头账户合计提款$9.4万，平台净赔$6.7万。审查IP/设备等，修复$270.92奖励触发漏洞，对Auto-Roulette 1实施赢额上限。"),
        ("[本周]","处置高风险提款用户",
         f"账户49903866：提款$39,636，公司输赢-$52,134；账户206628722：充值$10,229提款$24,855（净提$14,626），高度异常。建议立即人工审核。"),
        ("[本周]","优化首充质量与次日激活",
         f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp）。建议注册后1h/24h内推送首充引导，落地页突出$10+档。"),
        ("[下周]","扩大A8_DSP、A8_GG渠道预算",
         "A8_DSP充提差率17.6%（+1.3pp），一级首充成本$17（上周$21，改善）；A8_GG充提差率18.9%（+0.3pp）。建议预算增加15-20%，预估带动充值增量$50-70万。"),
        ("[下周]","暂停YB_FB_PWA_0",
         "注册量暴跌-79%，仅1,315人，建议暂停投放并排查渠道异常。"),
    ]
    headers_a=["优先级","建议事项","执行说明与数据依据"]
    rows_a=[]
    pclr={"[紧急]":C_RED,"[本周]":C_AMBER,"[下周]":C_GREEN}
    for pri,title,desc in actions:
        rows_a.append([cell(pri,True,pclr.get(pri[:4],C_GRAY),TA_CENTER),
                       cell(title,True,C_DARK),cell(desc,False,C_GRAY)])
    S.append(dtable(headers_a,rows_a,[full*x for x in [0.1,0.22,0.68]]))
    return S

# ════════════════════════════════════════════
# 页眉页脚
# ════════════════════════════════════════════
def header_footer(c,doc):
    c.saveState()
    w,h=A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN,h-17,f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7)
    c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
    c.restoreState()

# ════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════
def main(data_root=None, output_dir=None):
    """
    Jupyter Notebook 调用：
        from generate_mx_report_v5_4 import main
        main(data_root=r"D:/周报更新版/MX", output_dir=r"D:/周报更新版/MX/输出")
    """
    root   = Path(data_root)  if data_root  else DATA_ROOT
    outdir = Path(output_dir) if output_dir else OUTPUT_DIR
    outdir.mkdir(parents=True, exist_ok=True)

    print("📊 MX 周报 PDF v5.4 生成中...")
    _resolve_files(root)
    setup()

    print("  ▶ 加载数据...")
    K, trend          = load_platform()
    weekly_ret        = load_weekly_retention()
    agents            = load_agents()
    agent_ret         = load_agent_ret()          # v5.4: 全量每日加权留存
    tw_v, lw_v        = load_vip()
    dep_t,wdraw_t,dc_tw,dt_tw = load_top_users()
    top30             = load_games()
    mfr               = load_mfr()
    act_df,total_gift = load_activities()
    src_df,total_dep  = load_deposit_src()
    high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500 = load_risk()

    # 给 dt_tw 附加主玩游戏
    pref_game,mfr_amt = load_pref()
    bet_user = pd.read_csv(FILES["pref_tw"])
    bet_user["阶段汇总"] = bet_user["阶段汇总"].apply(clean)
    top_game = (bet_user[bet_user["分析指标"]=="投注金额"]
                .sort_values("阶段汇总",ascending=False)
                .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]]
                .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"})
                .reset_index())
    dt_tw = dt_tw.merge(top_game, on="账户ID", how="left")

    print("  ▶ 导出风控名单Excel...")
    excel_out = outdir/f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out), engine="openpyxl") as writer:
        cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
        high_risk[[c for c in cols_hr if c in high_risk.columns]].to_excel(writer,sheet_name="高风险用户",index=False)
        if len(c207_dt)>0:
            c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(writer,sheet_name="PP轮盘批量账号集群",index=False)
        if len(special_users)>0:
            su_c=["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]
            special_users[[c for c in su_c if c in special_users.columns]].to_excel(writer,sheet_name="特殊异常用户",index=False)
    print(f"✅ 风控名单Excel: {excel_out}")

    print("  ▶ 构建PDF...")
    out = outdir/f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc = SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,
                            topMargin=MARGIN+22,bottomMargin=MARGIN+10,
                            title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story = []
    story += [Spacer(1,3*cm),
              P("MX 平台数据周报",30,True,C_DARK,TA_CENTER), Spacer(1,0.5*cm),
              P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),
              Spacer(1,0.2*cm),
              P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),
              Spacer(1,0.5*cm),
              HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"),
              PageBreak()]
    story += build_overview(K,trend,weekly_ret);             story.append(PageBreak())
    story += build_agents(agents,agent_ret);                 story.append(PageBreak())
    story += build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw); story.append(PageBreak())
    story += build_games_section(mfr,top30);                 story.append(PageBreak())
    story += build_activities_section(act_df,total_gift,src_df,total_dep); story.append(PageBreak())
    story += build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw); story.append(PageBreak())
    story += build_conclusion(K,agents,mfr,dep_t,wdraw_t)

    print("  ▶ 渲染PDF...")
    doc.build(story, onFirstPage=header_footer, onLaterPages=header_footer)
    size = out.stat().st_size/1024
    print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out), str(excel_out)

if __name__ == "__main__":
    import sys
    if sys.platform != "win32":
        main(data_root="/mnt/user-data/uploads", output_dir="/mnt/user-data/outputs")
    else:
        main(data_root=DATA_ROOT, output_dir=OUTPUT_DIR)

📊 MX 周报 PDF v5.4 生成中...
  ▶ 数据目录：D:\周报更新版\MX
     ✅ [platform    ] 大盘/平台报表_USD_近4周.xlsx
     ✅ [daily       ] 大盘/日报-大盘日报_USD_近4周.xlsx
     ✅ [retention   ] 大盘/整体 首充留存（近7天）_近28天.csv
     ✅ [agent_plat  ] 总代/平台报表-总代_USD_近21天.xlsx
     ✅ [agent_promo ] 总代/推广报表-总代_USD_近21天.xlsx
     ✅ [agent_ret   ] 总代/首充充值留存_20260424_20260521.csv
     ✅ [vip         ] 用户/VIP报表_USD_近14天.xlsx
     ✅ [dt_tw       ] 用户/top提款用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dt_lw       ] 用户/top提款用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [dc_tw       ] 用户/头部充值用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dc_lw       ] 用户/头部充值用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [pref_tw     ] 用户/本周top500提款用户游戏偏好_全量数据_20260515_20260521.csv
     ✅ [pref_lw     ] 用户/上周top500提款用户游戏偏好_全量数据_20260508_20260514.csv
     ✅ [mfr         ] 游戏/厂商投注数据_全量数据_20260508_20260521.csv
     ✅ [game_tw     ] 游戏/游戏报表-详情_USD_本周.xlsx
     ✅ [game_lw     ] 游戏/游戏报表-详情_USD_上周.xlsx
     ✅ [gift 

In [38]:
"""
MX 平台数据周报生成脚本 v5.5
======================================================
【每次换周报只需修改下面 ① ② ③ 三处，其余不动】

v5.5 核心修复：
  1. load_agent_ret 剔除未到期天数
     - 本周1日：只取 THIS_WEEK[0] ~ (REPORT_END - 1天)
     - 本周3日：只取 THIS_WEEK[0] ~ (REPORT_END - 3天)
     - 上周1日：只取 LAST_WEEK[0] ~ (REPORT_END - 1天)
     - 上周7日：只取 LAST_WEEK[0] ~ (REPORT_END - 7天)
     确保分子分母一致，留存率符合递减规律：3日 > 7日
  2. 汇总留存表加总代ID列
  3. 其余修复（v5.4）保持不变：
     - ■ 实心小标题
     - 注册(环比) 单行格式 22,072/+27.1%
     - 充提差率(差值) 单行格式 15.7%/-1.1pp
     - 次日留存 本周值/上周值，本<上→红
"""
from pathlib import Path

# ① 数据根目录
DATA_ROOT  = Path(r"D:\周报更新版\MX")
# ② 报告输出目录（自动创建）
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")
# ③ 本周 / 上周日期范围 (YYYYMMDD)
THIS_WEEK  = ("20260515", "20260521")
LAST_WEEK  = ("20260508", "20260514")

# 报告截止日（= 本周最后一天）：用于计算各留存指标的完整截止日
REPORT_END = THIS_WEEK[1]

# ══════════════════════════════════════════════
import io, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, PageBreak, HRFlowable, KeepTogether
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

FN, FNB = "WQY", "WQYB"
FILES = {}

# ── v5.5 核心工具：计算留存完整截止日 ─────────────────
def retention_end_date(week_start, lag_days):
    """
    给定周起始日和留存lag，返回在REPORT_END内已完整的最后一天。
    逻辑：首充日 + lag_days <= REPORT_END
          → 首充日 <= REPORT_END - lag_days
    """
    end_dt  = datetime.strptime(REPORT_END, "%Y%m%d")
    last_dt = end_dt - timedelta(days=lag_days)
    last    = last_dt.strftime("%Y%m%d")
    return last if last >= week_start else None

# ── 字体 ─────────────────────────────────────
def find_chinese_font():
    import platform
    sys_name = platform.system()
    if sys_name == "Windows":
        candidates = [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\msyhbd.ttc",
                      r"C:\Windows\Fonts\simsun.ttc", r"C:\Windows\Fonts\simhei.ttf"]
    elif sys_name == "Darwin":
        candidates = ["/System/Library/Fonts/PingFang.ttc"]
    else:
        candidates = ["/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
                      "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc"]
    for p in candidates:
        if Path(p).exists(): return p
    try:
        from matplotlib import font_manager as fm
        for f in fm.fontManager.ttflist:
            if any(k in f.name for k in ["Heiti","Hei","YaHei","SimSun","SimHei","WenQuanYi","Noto Sans CJK"]):
                if Path(f.fname).exists(): return f.fname
    except Exception: pass
    raise FileNotFoundError("未找到中文字体！请手动设置 FONT_PATH")

def setup():
    global FONT_PATH
    FONT_PATH = find_chinese_font()
    print(f"  ▶ 使用字体：{FONT_PATH}")
    is_ttc = FONT_PATH.lower().endswith(".ttc")
    if is_ttc:
        pdfmetrics.registerFont(TTFont(FN, FONT_PATH, subfontIndex=0))
        try:   pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1))
        except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0))
    else:
        pdfmetrics.registerFont(TTFont(FN, FONT_PATH))
        pdfmetrics.registerFont(TTFont(FNB, FONT_PATH))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(FONT_PATH)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

# ── 文件搜索 ─────────────────────────────────
def _resolve_files(root):
    global FILES
    kw_map = {
        "platform":    "平台报表_USD_近4周",
        "daily":       "日报-大盘日报_USD_近4周",
        "retention":   "首充留存",
        "agent_plat":  "平台报表-总代_USD_近21天",
        "agent_promo": "推广报表-总代_USD_近21天",
        "agent_ret":   "首充充值留存",
        "vip":         "VIP报表_USD",
        "dt_tw":       f"top提款用户_全量数据_{THIS_WEEK[0]}",
        "dt_lw":       f"top提款用户_全量数据_{LAST_WEEK[0]}",
        "dc_tw":       f"头部充值用户_全量数据_{THIS_WEEK[0]}",
        "dc_lw":       f"头部充值用户_全量数据_{LAST_WEEK[0]}",
        "pref_tw":     f"本周top500提款用户游戏偏好_全量数据_{THIS_WEEK[0]}",
        "pref_lw":     f"上周top500提款用户游戏偏好_全量数据_{LAST_WEEK[0]}",
        "mfr":         "厂商投注数据_全量数据",
        "game_tw":     "游戏报表-详情_USD_本周",
        "game_lw":     "游戏报表-详情_USD_上周",
        "gift":        f"各赠送活动_全量数据_{THIS_WEEK[0]}",
        "deposit_src": f"整体存款_全量数据_{THIS_WEEK[0]}",
        "tool_map":    "道具对应活动",
        "vip_ret_chg": "VIP充值-充值_近28天",
        "vip_ret_act": "VIP充值-活跃_近28天",
    }
    print(f"  ▶ 数据目录：{root}")
    fail = 0
    for key, kw in kw_map.items():
        hits = [h for h in root.rglob("*")
                if h.is_file() and not h.name.startswith("~$") and kw in h.name]
        if not hits:
            print(f"     ❌ [{key:12s}] 找不到含 '{kw}' 的文件"); fail += 1
        else:
            p = max(hits, key=lambda h: h.stat().st_mtime)
            FILES[key] = p
            print(f"     ✅ [{key:12s}] {p.parent.name}/{p.name}")
    if fail:
        raise FileNotFoundError(f"\n共 {fail} 个文件未找到，请检查 DATA_ROOT = {root}")
    print(f"  ▶ 全部 {len(kw_map)} 个文件匹配成功\n")

# ── 颜色 ─────────────────────────────────────
C_BLUE   = colors.HexColor("#1d4ed8")
C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669")
C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706")
C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b")
C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white
C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1")
C_ROW    = colors.HexColor("#f8fafc")

PW, PH = A4
MARGIN  = 1.8*cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
TRUNCATE_RULES = {
    "首充次日复充率": 1, "首充次日复投率": 1, "首充当日复充率": 0,
    "首充7日复充率": 6, "首充30日复充率": 999,
    "首充2日复充率": 1, "首充3日复充率": 2,
}

# ── 工具函数 ─────────────────────────────────
def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c != "日期" and c not in ["总代.名称","name_总代","总代.ID"]]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o): return (n-o)/abs(o)*100 if o and o!=0 else 0.0
def pct_vec(ns, os): return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop>0 else s
    nz = v[v>0]; return nz.mean() if len(nz)>0 else v.mean()
def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

# ── 样式函数 ─────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.38,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),
                                ("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5),
                         HRFlowable(width="100%", thickness=1.5, color=C_BLUE2),
                         Spacer(1,3), P(f"■  {text}", 10, True, C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}", 8.5, False, C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8):
    full = PW-2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),        ("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2.5),     ("BOTTOMPADDING",(0,0),(-1,-1),2.5),
        ("LEFTPADDING",(0,0),(-1,-1),3),       ("RIGHTPADDING",(0,0),(-1,-1),3),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),  ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    hrow = [P(h, fsize, True, C_WHITE, TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]), fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c, (list,tuple)) else P(str(c), fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t, bold=False, clr=colors.black, align=TA_LEFT): return (t, bold, clr, align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup = good_up
    s = "+" if v>=0 else ""; c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{s}{v:.{d}f}%", False, c, TA_RIGHT)
def gclr(v, t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)

def kpi_card4(items, cols=4):
    full = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,gup in items:
        s = "+" if chg>=0 else ""
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([
            [P(label, 7.5, False, C_GRAY)],
            [P(str(tv), 14, True, C_DARK)],
            [P(f"上周：{lv}", 7.5, False, C_GRAY)],
            [P(f"{s}{chg:.1f}%", 8, True, pclr)],
        ], colWidths=[full], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),C_LGRAY), ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),8),
            ("TOPPADDING",(0,0),(-1,-1),5), ("BOTTOMPADDING",(0,0),(-1,-1),5),
        ]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(full,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════
def load_platform():
    df = pd.read_excel(FILES["platform"])
    df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]
    lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"])
    dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]
    lw_d = dd[dd["日期"].between(*LAST_WEEK)]
    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
              "首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE_RULES.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop)
        K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    K["tw_真实消耗"] = tw_d["真实消耗"].iloc[:-1].sum() if len(tw_d)>1 else tw_d["真实消耗"].sum()
    K["lw_真实消耗"] = lw_d["真实消耗"].sum()
    K["pct_真实消耗"] = pct(K["tw_真实消耗"], K["lw_真实消耗"])
    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])
    last14 = df.tail(14)
    trend = {
        "dates":    [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
        "充值":     last14["充值金额"].tolist(), "提现":  last14["提现金额"].tolist(),
        "充提差比": last14["充提差比"].tolist(), "公司输赢": last14["公司输赢"].tolist(),
        "首充":     last14["首充人数"].tolist(), "注册":  last14["注册人数"].tolist(),
    }
    return K, trend

def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
    daily["date_str"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
    daily_r = daily[daily["指标"]=="留存率"].copy()
    daily_u = daily[daily["指标"]=="留存人数"].copy()
    for c in ["第1日","第2日","第3日","第7日"]: daily_r[c] = daily_r[c].apply(clean)
    def _d(s): return datetime.strptime(s, "%Y%m%d")
    def _fmt(d): return d.strftime("%Y-%m-%d")
    def _lbl(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s=_d(THIS_WEEK[0]); tw_e=_d(THIS_WEEK[1])
    lw_s=_d(LAST_WEEK[0]); lw_e=_d(LAST_WEEK[1])
    w2_end=lw_s-timedelta(days=1); w2_start=w2_end-timedelta(days=6)
    w1_end=w2_start-timedelta(days=1); w1_start=w1_end-timedelta(days=6)
    weeks = [
        (f"第1周\n{_lbl(w1_start,w1_end)}", _fmt(w1_start), _fmt(w1_end)),
        (f"第2周\n{_lbl(w2_start,w2_end)}", _fmt(w2_start), _fmt(w2_end)),
        (f"上周\n{_lbl(lw_s,lw_e)}",        _fmt(lw_s),     _fmt(lw_e)),
        (f"本周\n{_lbl(tw_s,tw_e)}",        _fmt(tw_s),     _fmt(tw_e)),
    ]
    result = []
    for wk,s,e in weeks:
        mr = daily_r[(daily_r["date_str"]>=s)&(daily_r["date_str"]<=e)]
        mu = daily_u[(daily_u["date_str"]>=s)&(daily_u["date_str"]<=e)]
        users = mu["充值成功事件用户数"].sum(); row = {"week": wk, "users": users}
        for col in ["第1日","第2日","第3日","第7日"]:
            rs = mr[col].values; us = mu["充值成功事件用户数"].values
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = np.average(rs[v], weights=us[v]) if v.sum()>0 else np.nan
            else:
                row[col] = np.nanmean(rs) if len(rs)>0 else np.nan
        result.append(row)
    return result

def load_agents():
    df_p = pd.read_excel(FILES["agent_plat"])
    df_r = pd.read_excel(FILES["agent_promo"])
    df_p["日期"] = df_p["日期"].astype(str); df_r["日期"] = df_r["日期"].astype(str)
    df_p = to_num(df_p); df_r = to_num(df_r)
    df_p = df_p[df_p["总代.ID"]!=0]; df_r = df_r[df_r["总代.ID"]!=0]
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]; lw_p = df_p[df_p["日期"].between(*LAST_WEEK)]
    tw_r = df_r[df_r["日期"].between(*THIS_WEEK)]; lw_r = df_r[df_r["日期"].between(*LAST_WEEK)]
    sum_a = ["充值金额","提现金额","充提差","首充金额","首充人数","注册人数",
             "充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        g = d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sum_a if c in d.columns}).reset_index()
        for c in ["首充转化率","充提差比","首充次日充值留存","活跃用户付费率"]:
            if c in d.columns:
                rm = d.groupby("总代.ID")[c].mean().rename(c); g = g.merge(rm, on="总代.ID", how="left")
        g["充提差率"] = g["充提差"]/g["充值金额"]*100; return g
    tw_pa = agg_p(tw_p); lw_pa = agg_p(lw_p)
    sum_r = ["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        g = d.groupby("总代.ID").agg({c:"sum" for c in sum_r if c in d.columns}).reset_index()
        g["注册成本"] = g["总消耗"]/g["注册人数"].replace(0,np.nan)
        g["一级首充成本"] = g["总消耗"]/g["一级首充人数"].replace(0,np.nan); return g
    tw_ra = agg_r(tw_r); lw_ra = agg_r(lw_r)
    lw_ra["lw_一级首充成本"] = lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
    m = tw_pa.merge(
        lw_pa[["总代.ID","充值金额","注册人数","充提差率","首充次日充值留存"]].rename(
            columns={"充值金额":"lw_充值","注册人数":"lw_注册",
                     "充提差率":"lw_充提差率","首充次日充值留存":"lw_次日留存"}),
        on="总代.ID", how="left")
    m = m.merge(tw_ra[["总代.ID","总消耗","注册成本","一级首充成本","一级首充人数"]], on="总代.ID", how="left")
    m = m.merge(lw_ra[["总代.ID","lw_一级首充成本"]], on="总代.ID", how="left")
    m["注册环比"] = pct_vec(m["注册人数"], m["lw_注册"])
    m["充值环比"] = pct_vec(m["充值金额"], m["lw_充值"])
    return m.sort_values("充值金额", ascending=False).head(15)

# ════════════════════════════════════════════
# v5.5 核心修复：总代留存加载（剔除未到期天数）
# ════════════════════════════════════════════
def load_agent_ret():
    """
    CSV 格式：初始事件发生时间 / 总代 / name_总代 / 首充用户数 / 指标 / 1日 / 3日 / 7日...

    v5.5 关键修复：
    ─────────────────────────────────────────────────────────
    留存第N日的含义：首充日+N天后仍活跃的比例。
    若报告截止日=THIS_WEEK[1]，则：
      首充日 + N <= THIS_WEEK[1]
      → 首充日 <= THIS_WEEK[1] - N天

    本周3日留存：只取 THIS_WEEK[0] ~ (REPORT_END-3天)
      本例：20260515 ~ 20260518（5/19、5/20、5/21三天3日未到期，剔除）
    本周1日留存：只取 THIS_WEEK[0] ~ (REPORT_END-1天) = 20260520
    上周7日留存：只取 LAST_WEEK[0] ~ (REPORT_END-7天) = 20260514（全部完整）
    上周1日留存：只取 LAST_WEEK[0] ~ (REPORT_END-1天) = 20260520（全部完整）
    ─────────────────────────────────────────────────────────
    返回：dict { name_总代 -> {tw_d1, tw_d3, lw_d1, lw_d7, 总代ID} }
    """
    df = pd.read_csv(FILES["agent_ret"])

    # 解析日期列
    df["_yyyymmdd"] = (df["初始事件发生时间"].astype(str)
                       .str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
                       .str.replace("-",""))

    # 数值化留存率（去掉 %）
    for c in ["1日","2日","3日","7日"]:
        df[c] = (df[c].astype(str).str.replace("%","").str.strip()
                     .pipe(pd.to_numeric, errors="coerce"))

    # 首充用户数
    df["首充用户数"] = pd.to_numeric(df["首充用户数"], errors="coerce").fillna(0)

    # 总代ID（用于汇总表）
    df["总代"] = pd.to_numeric(df["总代"], errors="coerce")

    # 只保留每日留存率行
    daily = df[
        df["_yyyymmdd"].notna() &
        df["_yyyymmdd"].str.match(r"^\d{8}$") &
        (df["指标"] == "留存率")
    ].copy()

    # ── 计算各留存指标的完整截止日 ──────────────────
    tw_d1_end = retention_end_date(THIS_WEEK[0], 1)   # 本周1日: ~20260520
    tw_d3_end = retention_end_date(THIS_WEEK[0], 3)   # 本周3日: ~20260518 ★关键修复
    lw_d1_end = retention_end_date(LAST_WEEK[0], 1)   # 上周1日: 全部完整
    lw_d7_end = retention_end_date(LAST_WEEK[0], 7)   # 上周7日: 全部完整

    print(f"     留存完整截止日 → 本周1日:{tw_d1_end} 本周3日:{tw_d3_end} "
          f"上周1日:{lw_d1_end} 上周7日:{lw_d7_end}")

    # 分别筛选各时间窗口
    tw_all = daily[daily["_yyyymmdd"].between(*THIS_WEEK)]
    lw_all = daily[daily["_yyyymmdd"].between(*LAST_WEEK)]

    def _win(df_week, week_start, end_cap):
        if end_cap is None: return df_week.iloc[0:0]  # 空
        return df_week[(df_week["_yyyymmdd"] >= week_start) &
                       (df_week["_yyyymmdd"] <= end_cap)]

    tw_d1_df = _win(tw_all, THIS_WEEK[0], tw_d1_end)
    tw_d3_df = _win(tw_all, THIS_WEEK[0], tw_d3_end)
    lw_d1_df = _win(lw_all, LAST_WEEK[0], lw_d1_end)
    lw_d7_df = _win(lw_all, LAST_WEEK[0], lw_d7_end)

    def _wavg_all(df_sub, col):
        """对所有总代做首充用户数加权平均，返回 {name_总代: value}"""
        result = {}
        for nm, grp in df_sub.groupby("name_总代"):
            valid = grp[grp[col].notna() & (grp["首充用户数"] > 0)]
            if len(valid) > 0:
                result[nm] = float(np.average(valid[col], weights=valid["首充用户数"]))
            else:
                result[nm] = np.nan
        return result

    tw_d1 = _wavg_all(tw_d1_df, "1日")
    tw_d3 = _wavg_all(tw_d3_df, "3日")
    lw_d1 = _wavg_all(lw_d1_df, "1日")
    lw_d7 = _wavg_all(lw_d7_df, "7日")

    # 总代ID映射（取第一条）
    id_map = {}
    for nm, grp in daily.groupby("name_总代"):
        id_val = grp["总代"].dropna().values
        if len(id_val) > 0:
            id_map[nm] = int(id_val[0])

    all_names = set(tw_d1) | set(lw_d1)
    ret = {}
    for nm in all_names:
        ret[nm] = {
            "tw_d1": tw_d1.get(nm, np.nan),
            "tw_d3": tw_d3.get(nm, np.nan),
            "lw_d1": lw_d1.get(nm, np.nan),
            "lw_d7": lw_d7.get(nm, np.nan),
            "agent_id": id_map.get(nm, None),
        }
    return ret

def load_vip():
    df = pd.read_excel(FILES["vip"]); df["日期"] = df["日期"].astype(str)
    num = ["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df = to_num(df, num)
    tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

def load_vip_retention():
    results = {}
    for fkey, rtype in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df = pd.read_csv(FILES[fkey])
        df['date_str'] = df['初始事件发生时间'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')
        df['yyyymmdd'] = df['date_str'].str.replace('-','')
        for c in ['1日','2日','3日','7日']: df[c] = df[c].apply(clean)
        df['n']   = pd.to_numeric(df['充值成功事件用户数'], errors='coerce')
        df['vip'] = pd.to_numeric(df['vip_level'], errors='coerce')
        stage = df[(df['指标']=='留存率')&(df['初始事件发生时间']=='阶段值')].copy()
        stage_dict = {int(r['vip']):{c:r[c] for c in ['1日','2日','3日','7日']}
                      for _,r in stage.iterrows() if not pd.isna(r['vip'])}
        daily = df[(df['指标']=='留存率')&df['yyyymmdd'].notna()&df['vip'].notna()]
        tw = daily[daily['yyyymmdd'].between(*THIS_WEEK)]
        lw = daily[daily['yyyymmdd'].between(*LAST_WEEK)]
        def wavg(d,col):
            res={}
            for vip,g in d.groupby('vip'):
                v=g[g[col].notna()]
                if len(v)>0: res[int(vip)]=np.average(v[col],weights=v['n'])
            return res
        results[rtype] = {
            'tw':{c:wavg(tw,c) for c in ['1日','2日','3日','7日']},
            'lw':{c:wavg(lw,c) for c in ['1日','2日','3日','7日']},
            'stage':stage_dict,
        }
    return results

def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
    for df in [dc_tw,dc_lw,dt_tw,dt_lw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
    tiers = [1,10,50,100,200,500]
    df_p = pd.read_excel(FILES["platform"]); df_p["日期"]=df_p["日期"].astype(str); df_p=to_num(df_p)
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]
    total_chg=tw_p["充值金额"].sum(); total_tx=tw_p["提现金额"].sum()
    actual_cr=(total_chg-total_tx)/total_chg*100
    dep_tiers,wdraw_tiers=[],[]
    total_tw_c=dc_tw["充值金额"].sum(); total_tw_t=dt_tw["提款金额"].sum()
    for t in tiers:
        tw=dc_tw.head(t); lw=dc_lw.head(t)
        tw_c=tw["充值金额"].sum(); tw_tx=tw["提款金额"].sum()
        lw_c=lw["充值金额"].sum(); lw_tx=lw["提款金额"].sum()
        dep_tiers.append({"tier":f"Top{t}","tw_chg":tw_c,"lw_chg":lw_c,
            "tw_avg":tw_c/t,"lw_avg":lw_c/t,
            "tw_cr":(tw_c-tw_tx)/tw_c*100 if tw_c>0 else 0,
            "lw_cr":(lw_c-lw_tx)/lw_c*100 if lw_c>0 else 0,
            "tw_win":tw["公司输赢"].sum(),"占全量":tw_c/total_tw_c*100})
        twd=dt_tw.head(t); lwd=dt_lw.head(t)
        tw_t2=twd["提款金额"].sum(); tw_c2=twd["充值金额"].sum()
        lw_t2=lwd["提款金额"].sum(); lw_c2=lwd["充值金额"].sum()
        tw_cr2=(tw_c2-tw_t2)/tw_c2*100 if tw_c2>0 else 0
        lw_cr2=(lw_c2-lw_t2)/lw_c2*100 if lw_c2>0 else 0
        winners=(twd["公司输赢"]<0).sum()
        act=twd["活动奖励"].sum()/(tw_c2+twd["活动奖励"].sum())*100
        excl_chg=total_chg-tw_c2; excl_tx=total_tx-tw_t2
        excl_cr=(excl_chg-excl_tx)/excl_chg*100 if excl_chg>0 else 0
        impact=excl_cr-actual_cr
        wdraw_tiers.append({"tier":f"Top{t}","tw_tx":tw_t2,"lw_tx":lw_t2,
            "tw_avg":tw_t2/t,"lw_avg":lw_t2/t,"tw_cr":tw_cr2,"lw_cr":lw_cr2,
            "赢家":winners,"总数":t,"赢家率":winners/t*100,
            "活动占比":act,"占全量":tw_t2/total_tw_t*100,"大盘影响":impact})
    return dep_tiers,wdraw_tiers,dc_tw.head(200),dt_tw.head(20)

def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); df_lw=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); df_lw["阶段汇总"]=df_lw["阶段汇总"].apply(clean)
    bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]
    bet_lw=df_lw[df_lw["分析指标"]=="投注金额"]
    g_bet=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
    g_win=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
    g_lw=bet_lw.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
    g_usr=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([g_bet,g_win,g_lw,g_usr],axis=1).reset_index()
    total=g["本周投注"].sum(); g["占比"]=g["本周投注"]/total*100
    g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr

def load_games():
    df_tw=pd.read_excel(FILES["game_tw"]); df_lw=pd.read_excel(FILES["game_lw"])
    for df in [df_tw,df_lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢","人均投注局数","人均投注金额"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    total_tw=df_tw["投注金额"].sum()
    g_tw=df_tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    g_lw=df_lw.groupby("游戏.名称").agg(
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    g_tw["人均局数"]=g_tw["投注局数"]/g_tw["投注人数"]
    g_tw["人均金额"]=g_tw["投注金额"]/g_tw["投注人数"]
    g_tw["盈亏率"]=g_tw["公司输赢"]/g_tw["投注金额"]*100
    g_tw["占比"]=g_tw["投注金额"]/total_tw*100
    gm=g_tw.merge(g_lw,on="游戏.名称",how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)

def load_mfr():
    df=pd.read_csv(FILES["mfr"])
    df=df[df["时间"]!="阶段汇总"].copy()
    df["时间"]=df["时间"].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢"]: df[c]=df[c].apply(clean)
    tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
    def agg(d):
        g=d.groupby("show_name_厂商标签id").agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
    tw_g=agg(tw); lw_g=agg(lw)
    mg=tw_g.merge(lw_g[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on="show_name_厂商标签id",how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    return mg.sort_values("投注金额",ascending=False)

def load_activities():
    df=pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数"]: df[c]=df[c].apply(clean)
    agg=df.groupby("name_账变opt_id").agg(
        赠送金额=("赠送金额","sum"),lw_赠送=("赠送金额.1","sum"),赠送人数=("赠送人数","sum")).reset_index()
    agg["环比"]=(agg["赠送金额"]-agg["lw_赠送"])/agg["lw_赠送"].abs()*100
    total=agg["赠送金额"].sum(); agg["占比"]=agg["赠送金额"]/total*100; agg["人均"]=agg["赠送金额"]/agg["赠送人数"]
    return agg.sort_values("赠送金额",ascending=False).head(15), total

def load_deposit_src():
    df=pd.read_csv(FILES["deposit_src"])
    for c in ["充值金额","充值金额.1","充值人数"]: df[c]=df[c].apply(clean)
    tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str)
    df["道具ID"]=df["道具ID"].astype(str)
    agg=df.groupby("道具ID").agg(充值金额=("充值金额","sum"),lw_充值=("充值金额.1","sum"),充值人数=("充值人数","sum")).reset_index()
    agg=agg.merge(tm,on="道具ID",how="left")
    nm={"(null)":"无道具直接充值","71872":"幸运转盘10%","71873":"幸运转盘20%",
        "71975":"幸运翻卡道具","71894":"新人签到10%","71871":"新人签到20%"}
    agg["名称"]=agg.apply(lambda r: nm.get(str(r["道具ID"]),
        str(r.get("备注",""))[:12] if pd.notna(r.get("备注")) else f"其他({str(r['道具ID'])[:8]})"), axis=1)
    total=agg["充值金额"].sum(); agg["占比"]=agg["充值金额"]/total*100
    agg["环比"]=(agg["充值金额"]-agg["lw_充值"])/agg["lw_充值"].abs()*100
    return agg.sort_values("充值金额",ascending=False).head(6), total

def load_risk():
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    dt_tw=pd.read_csv(FILES["dt_tw"]); dc_tw=pd.read_csv(FILES["dc_tw"])
    for df2 in [dt_tw,dc_tw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df2.columns: df2[c]=pd.to_numeric(df2[c],errors="coerce")
    bet_amt=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    win_pref=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    bet_cnt=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数")
    top_game=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
              .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
              .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏","阶段汇总":"主游戏投注额"}).reset_index())
    ug=pd.concat([bet_cnt,bet_amt,win_pref],axis=1).reset_index()
    ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]; ug=ug.merge(top_game,on="账户ID",how="left")
    top500=dt_tw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0; top500["投充比"]=top500["投注金额"]/top500["充值金额"]
    high_risk=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
    pp_ids=df[(df["游戏名称"]=="Auto-Roulette 1")&(df["分析指标"]=="投注次数")]["账户ID"].unique()
    cluster_207=[u for u in pp_ids if str(u).startswith("207")]
    c207_dt=dt_tw[dt_tw["账户ID"].isin(cluster_207)].sort_values("提款金额",ascending=False)
    special=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False)
    bet_g=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
    win_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
    usr_g=df[df["分析指标"]=="投注次数"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
    winr_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bet_g.merge(win_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(usr_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(winr_g,on=["show_name_厂商标签id","游戏名称"],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100; g["人均投注额"]=g["投注金额"]/g["玩家数"]
    risk_g=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    return high_risk,c207_dt,cluster_207,special,g.sort_values("投注金额",ascending=False).head(20),risk_g,top500

# ════════════════════════════════════════════
# 图表（与 v5.4 相同，保持不变）
# ════════════════════════════════════════════
SPLIT=7
def _vline(ax,dates):
    ax.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax.text(SPLIT-4,ax.get_ylim()[1]*0.93,"上周",ha="center",fontsize=7,color="#64748b")
    ax.text(SPLIT+3,ax.get_ylim()[1]*0.93,"本周",ha="center",fontsize=7,color="#1d4ed8")

def chart_trend(trend):
    fig=plt.figure(figsize=(16,10),facecolor="white")
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.45,wspace=0.3)
    dates=trend["dates"]; x=range(len(dates))
    def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=0.25)
    ax=fig.add_subplot(gs[0,0])
    clrs=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT
    ax.bar(x,[v/10000 for v in trend["充值"]],color=clrs,width=0.7,label="充值")
    ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
    ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold"); ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
    ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax,dates)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax2=fig.add_subplot(gs[0,1])
    ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=0.7)
    ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax2); _vline(ax2,dates)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax3=fig.add_subplot(gs[1,0])
    ax3.bar(x,trend["首充"],color=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT,width=0.7,label="首充人数")
    ax3r=ax3.twinx(); ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold"); ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
    l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left"); ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=0.25); ax3.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax4=fig.add_subplot(gs[1,1])
    ax4.bar(x,trend["充提差比"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=0.7)
    tw_m=np.mean(trend["充提差比"][SPLIT:]); lw_m=np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tw_m,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lw_m,color="#94a3b8",ls=":",lw=1.2)
    ax4.text(len(dates)-0.5,tw_m+0.3,f"本周均{tw_m:.1f}%",fontsize=7,color="#7c3aed",ha="right")
    ax4.text(0.5,lw_m+0.3,f"上周均{lw_m:.1f}%",fontsize=7,color="#64748b",ha="left")
    ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold"); ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax4); _vline(ax4,dates)
    ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
    plt.tight_layout(rect=[0,0,1,0.98]); return fig_img(fig,16,11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white")
    cols_p=[("第1日","#1d4ed8"),("第2日","#059669"),("第3日","#d97706"),("第7日","#7c3aed")]
    x=np.arange(len(weeks)); w=0.18
    for i,(col,clr) in enumerate(cols_p):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-1.5*w,vals,w,label=col,color=clr,alpha=0.85)
        for bar,v in zip(bars,vals):
            if not np.isnan(v): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周加权平均对比",fontsize=10,fontweight="bold"); ax.legend(fontsize=8,loc="upper right")
    ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=0.5); ax.text(2.6,ax.get_ylim()[1]*0.85,"↑本周",fontsize=7.5,color="#ef4444")
    ax.grid(axis="y",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.text(0.5,-0.02,f"注：本周({THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]})7日留存数据未完整",ha="center",fontsize=7.5,color="#64748b",style="italic")
    plt.tight_layout(); return fig_img(fig,16,5.5)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=0.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tw_c=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_c=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tw_c,w,label="本周",color="#1d4ed8",alpha=0.85); ax1.bar(x+w/2,lw_c,w,label="上周",color="#93c5fd",alpha=0.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tw_b=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_b=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tw_b,w,label="本周",color="#059669",alpha=0.85); ax2.bar(x+w/2,lw_b,w,label="上周",color="#6ee7b7",alpha=0.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=0.85,width=0.6); ax3.axhline(0,color="black",lw=0.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=0.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vip_ret):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    chg_tw=[vip_ret['chg']['tw']['1日'].get(v,0) for v in vips]; chg_lw=[vip_ret['chg']['lw']['1日'].get(v,0) for v in vips]; chg_3d=[vip_ret['chg']['tw']['3日'].get(v,0) for v in vips]
    act_tw=[vip_ret['act']['tw']['1日'].get(v,0) for v in vips]; act_lw=[vip_ret['act']['lw']['1日'].get(v,0) for v in vips]; act_3d=[vip_ret['act']['tw']['3日'].get(v,0) for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=0.28
    ax1.bar(x-w,chg_tw,w,label="次日留存(本周)",color="#1d4ed8",alpha=0.85); ax1.bar(x,chg_lw,w,label="次日留存(上周)",color="#93c5fd",alpha=0.7); ax1.bar(x+w,chg_3d,w,label="3日留存(本周)",color="#059669",alpha=0.75)
    ax1.set_title("充值→充值 留存率（%）",fontsize=10,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,fontsize=8); ax1.legend(fontsize=7.5,loc="upper left"); ax1.grid(axis="y",alpha=0.25); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    ax2.bar(x-w,act_tw,w,label="次日留存(本周)",color="#7c3aed",alpha=0.85); ax2.bar(x,act_lw,w,label="次日留存(上周)",color="#c4b5fd",alpha=0.7); ax2.bar(x+w,act_3d,w,label="3日留存(本周)",color="#d97706",alpha=0.75)
    ax2.set_title("充值→活跃 留存率（%）",fontsize=10,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,fontsize=8); ax2.legend(fontsize=7.5,loc="upper left"); ax2.grid(axis="y",alpha=0.25); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.suptitle("VIP各等级充值留存率（本周 vs 上周，加权日均值）",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10); names=top10["show_name_厂商标签id"].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    shares=top10["占比"].tolist(); lw_s=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,shares,color=["#059669" if c>=l else "#dc2626" for c,l in zip(shares,lw_s)],alpha=0.85,width=0.6)
    ax1.plot(names,lw_s,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,shares,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,f"{s:.1f}%",ha="center",fontsize=7); ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额（本周 vs 上周）",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=0.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=0.85); ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=0.7)
    ax2.axhline(0,color="black",lw=0.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5); ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    top15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in top15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in top15.iterrows()]; lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in top15.iterrows()]
    x=np.arange(len(names)); w=0.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=0.85); ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=0.7)
    for bar,v in zip(ax.patches[:len(names)],bets[::-1]): ax.text(bar.get_width()+0.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8); ax.set_title("Top15游戏 投注金额（万USD，本周 vs 上周）",fontsize=9,fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="x",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pref_game,mfr_amt):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=mfr_amt.index[:8].tolist(); vals=mfr_amt.values[:8].tolist(); total=sum(vals); pcts=[v/total*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.05,-0.18)); ax1.set_title("Top500提款用户 厂商偏好（按投注金额）",fontsize=9,fontweight="bold")
    top12=pref_game.head(12); gnames=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in top12.iterrows()]; gvals=[r["本周投注"]/10000 for _,r in top12.iterrows()]; gclrs=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in top12.iterrows()]
    ax2.barh(range(len(gnames))[::-1],gvals,color=gclrs[::-1],alpha=0.85); ax2.set_yticks(range(len(gnames))); ax2.set_yticklabels(gnames,fontsize=7.5); ax2.set_title("偏好游戏Top12（按投注金额，红=平台亏损）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act_df):
    top10=act_df.head(10); names=[str(r["name_账变opt_id"])[:10] for _,r in top10.iterrows()]; vals=[r["赠送金额"]/10000 for _,r in top10.iterrows()]; lw=[r["lw_赠送"]/10000 for _,r in top10.iterrows()]; envs=[r["环比"] for _,r in top10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=0.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=0.85); ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=0.7); ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8); ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold"); ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax2.barh(range(len(names)),envs[::-1],color=["#059669" if v>0 else "#dc2626" for v in envs[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8); ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_deposit_src(src_df,total):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); names=src_df["名称"].tolist(); vals=src_df["充值金额"].tolist(); lw_v=src_df["lw_充值"].tolist(); pcts=src_df["占比"].tolist(); clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>3 else "",startangle=140,pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:8]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.1,-0.15)); ax1.set_title(f"存款来源占比（总计${total/10000:.0f}万USD）",fontsize=9,fontweight="bold")
    x=np.arange(len(names)); w=0.35; ax2.bar(x-w/2,[v/10000 for v in vals],w,label="本周",color=clrs,alpha=0.85); ax2.bar(x+w/2,[v/10000 for v in lw_v],w,label="上周",color=["#93c5fd"]*len(names),alpha=0.6); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=25,ha="right",fontsize=7.5); ax2.set_title("存款道具 本周 vs 上周（万USD）",fontsize=9,fontweight="bold"); ax2.legend(fontsize=8); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_risk_scatter(top500):
    valid=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy(); valid=valid[valid["投充比"]<200]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    ax.scatter(valid["投充比"],valid["公司输赢"]/10000,c=["#dc2626" if v<0 else "#059669" for v in valid["公司输赢"]],s=[min(abs(v)/500+20,200) for v in valid["公司输赢"]],alpha=0.55,edgecolors="none")
    ax.axhline(0,color="black",lw=0.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=0.7); ax.text(21,ax.get_ylim()[0]*0.9,"投充比=20x",fontsize=7.5,color="#d97706")
    for _,r in top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()].iterrows():
        ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),xytext=(r["投充比"]+5,r["公司输赢"]/10000-0.2),fontsize=6.5,color="#991b1b",arrowprops=dict(arrowstyle="->",color="#991b1b",lw=0.7))
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8); ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=0.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=0.6,label="平台赢钱")],fontsize=8,loc="upper right"); ax.grid(alpha=0.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(risk_g):
    top12=risk_g.head(12); names=[r["游戏名称"][:16] for _,r in top12.iterrows()]; losses=[abs(r["公司输赢"]) for _,r in top12.iterrows()]; rates=[r["盈亏率"] for _,r in top12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=0.85)
    for bar,v in zip(ax1.patches,losses[::-1]): ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)

# ════════════════════════════════════════════
# 章节构建
# ════════════════════════════════════════════
def build_overview(K,trend,weekly_ret):
    S=[sec_title("一、大盘核心数据")]
    kpis=[
        ("充值金额",f"{K['tw_充值金额']/10000:.1f}万",f"{K['lw_充值金额']/10000:.1f}万",K["pct_充值金额"],True),
        ("提现金额",f"{K['tw_提现金额']/10000:.1f}万",f"{K['lw_提现金额']/10000:.1f}万",K["pct_提现金额"],False),
        ("充提差",f"{K['tw_充提差']/10000:.1f}万",f"{K['lw_充提差']/10000:.1f}万",K["pct_充提差"],True),
        ("充提差率",f"{K['tw_充提差比']:.2f}%",f"{K['lw_充提差比']:.2f}%",K["pct_充提差比"],True),
        ("公司输赢",f"{K['tw_公司输赢']/10000:.1f}万",f"{K['lw_公司输赢']/10000:.1f}万",K["pct_公司输赢"],True),
        ("盈亏率",f"{K['tw_盈亏率']:.3f}%",f"{K['lw_盈亏率']:.3f}%",K["pct_盈亏率"],True),
        ("注册人数",f"{int(K['tw_注册人数']):,}",f"{int(K['lw_注册人数']):,}",K["pct_注册人数"],True),
        ("首充人数",f"{int(K['tw_首充人数']):,}",f"{int(K['lw_首充人数']):,}",K["pct_首充人数"],True),
        ("日均活跃",f"{K['tw_活跃人数']/7/10000:.1f}万",f"{K['lw_活跃人数']/7/10000:.1f}万",K["pct_活跃人数"],True),
        ("投注金额",f"{K['tw_投注金额']/10000:.0f}万",f"{K['lw_投注金额']/10000:.0f}万",K["pct_投注金额"],True),
        ("全量日均ARPPU",f"${K['tw_全量Arppu']:.2f}",f"${K['lw_全量Arppu']:.2f}",K["pct_全量Arppu"],True),
        ("老用户日均ARPPU",f"${K['tw_老用户ARPPU']:.2f}",f"${K['lw_老用户ARPPU']:.2f}",K["pct_老用户ARPPU"],True),
        ("首充日均ARPPU",f"${K['tw_首充Arppu']:.2f}",f"${K['lw_首充Arppu']:.2f}",K["pct_首充Arppu"],True),
        ("总赠送金额",f"{K['tw_总赠送金额']/10000:.1f}万",f"{K['lw_总赠送金额']/10000:.1f}万",K["pct_总赠送金额"],False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",f"{K['lw_赠送充值比']:.2f}%",K["pct_赠送充值比"],False),
        ("首充转化率",f"{K['tw_首充转化率']:.1f}%",f"{K['lw_首充转化率']:.1f}%",K["pct_首充转化率"],True),
        ("首充次日充值留存",f"{K['tw_首充次日复充率']:.1f}%",f"{K['lw_首充次日复充率']:.1f}%",K["pct_首充次日复充率"],True),
        ("推广消耗",f"{K['tw_真实消耗']/10000:.1f}万",f"{K['lw_真实消耗']/10000:.1f}万",K["pct_真实消耗"],False),
    ]
    S.append(kpi_card4(kpis,cols=4)); S.append(Spacer(1,8))
    cr_delta=K["tw_充提差比"]-K["lw_充提差比"]; ret_delta=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(insight_box([
        f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%）；推广消耗{K['pct_真实消耗']:+.1f}%。",
        f"充提差率{K['tw_充提差比']:.2f}%（上周{K['lw_充提差比']:.2f}%，{cr_delta:+.2f}pp）；盈亏率{K['tw_盈亏率']:.3f}%（上周{K['lw_盈亏率']:.3f}%）。",
        f"首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），首充转化率{K['tw_首充转化率']:.1f}%，首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%）。",
        f"首充次日充值留存{K['tw_首充次日复充率']:.1f}%（上周{K['lw_首充次日复充率']:.1f}%，{ret_delta:+.1f}pp）。",
    ]))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
    S.append(sub_title("首充用户充值留存率 — 近4周加权平均对比")); S.append(chart_ret_weekly(weekly_ret))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=(this_w.get("第1日",0)or 0)-(last_w.get("第1日",0)or 0)
    S.append(insight_box([
        f"本周首充次日留存{this_w.get('第1日',0):.1f}%，较上周{d1:+.1f}pp；2日留存{this_w.get('第2日',0):.1f}%。",
        f"注：本周7日留存因数据截止日期无法完整计算，以上周7日留存{last_w.get('第7日',0):.1f}%作为近期参考基准。",
    ], clr=C_AMBER))
    return S

def build_agents(agents, agent_ret):
    """
    Top15总代表 + 全量留存汇总表
    列说明：
      注册(环比)     → "22,072/+27.1%"  单行，颜色跟环比
      充提差率(差值) → "15.7%/-1.1pp"   单行，颜色跟充提差率高低
      次日留存(本/上) → "22.4%/25.9%"   本<上→红，本>=上→绿
      3留本周        → 本周3日加权（已剔除未到期天，5/15~5/18）
      7留上周        → 上周7日加权（全部完整，5/8~5/14）
      保证：3留本周 > 7留上周（留存递减规律）
    """
    S = [sec_title("二、总代分析（剔除总代0官方总代）")]
    S.append(sub_title("Top15总代表现（本周 vs 上周，按充值金额排序）"))
    full = PW - 2*MARGIN

    # 计算完整截止日（用于列头说明）
    tw_d3_end = retention_end_date(THIS_WEEK[0], 3)  # 20260518
    lw_d7_end = retention_end_date(LAST_WEEK[0], 7)  # 20260514

    tw_d3_lbl = f"{tw_d3_end[4:6]}/{tw_d3_end[6:]}" if tw_d3_end else "-"
    lw_d7_lbl = f"{lw_d7_end[4:6]}/{lw_d7_end[6:]}" if lw_d7_end else "-"

    headers = [
        "ID", "总代名称",
        "注册(环比)",
        "首充",
        "充值\n(万)",
        "充提差率(差值)",
        "消耗\n(万)",
        "一级首充\n成本(本/上)",
        "次日留存\n(本/上)",
        f"3留本周\n(~{tw_d3_lbl})",
        f"7留上周\n(~{lw_d7_lbl})",
    ]

    def _fret(v):
        return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

    rows = []
    for _, r in agents.iterrows():
        agent_id = int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
        rname    = str(r["总代.名称"])
        cr       = r["充提差率"]; lw_cr = r.get("lw_充提差率", 0) or 0; cr_diff = cr - lw_cr
        cost     = (r.get("总消耗", 0) or 0) / 10000
        fc_c     = r.get("一级首充成本", 0) or 0; lw_fc_c = r.get("lw_一级首充成本", 0) or 0

        # 注册/环比
        reg = int(r["注册人数"]); reg_chg = r.get("注册环比", 0) or 0
        reg_str = f"{reg:,}/{'+' if reg_chg>=0 else ''}{reg_chg:.1f}%"
        reg_clr = C_GREEN if reg_chg >= 0 else C_RED

        # 充提差率/差值
        cr_str = f"{cr:.1f}%/{'+' if cr_diff>=0 else ''}{cr_diff:.1f}pp"
        cr_clr = C_GREEN if cr >= 17 else (C_RED if cr < 5 else C_DARK)

        # 留存数据
        d = agent_ret.get(rname, {})
        tw_d1 = d.get("tw_d1", np.nan); lw_d1 = d.get("lw_d1", np.nan)
        tw_d3 = d.get("tw_d3", np.nan); lw_d7 = d.get("lw_d7", np.nan)

        # 次日留存 本/上
        if not (isinstance(tw_d1,float) and np.isnan(tw_d1)) and not (isinstance(lw_d1,float) and np.isnan(lw_d1)):
            nd_str = f"{tw_d1:.1f}%/{lw_d1:.1f}%"
            nd_clr = C_GREEN if tw_d1 >= lw_d1 else C_RED
        elif not (isinstance(tw_d1,float) and np.isnan(tw_d1)):
            nd_str = f"{tw_d1:.1f}%/-"; nd_clr = C_DARK
        else:
            nd_str = "-"; nd_clr = C_GRAY

        rows.append([
            cell(str(agent_id),              False, C_GRAY,  TA_CENTER),
            cell(rname[:16],                 True,  C_DARK,  TA_LEFT),
            cell(reg_str,                    False, reg_clr, TA_RIGHT),
            cell(f"{int(r['首充人数']):,}",  False, C_DARK,  TA_RIGHT),
            cell(f"{r['充值金额']/10000:.0f}",False, C_DARK, TA_RIGHT),
            cell(cr_str,                     False, cr_clr,  TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-", False, C_DARK, TA_RIGHT),
            cell(f"${fc_c:.0f}/${lw_fc_c:.0f}" if fc_c>0 else "-", False, C_DARK, TA_RIGHT),
            cell(nd_str,                     False, nd_clr,  TA_RIGHT),
            cell(_fret(tw_d3),               False, C_DARK,  TA_RIGHT),
            cell(_fret(lw_d7),               False, C_DARK,  TA_RIGHT),
        ])

    cw = [full*x for x in [0.05,0.17,0.11,0.06,0.06,0.12,0.06,0.11,0.10,0.08,0.08]]
    S.append(dtable(headers, rows, cw, fsize=6.5))
    S.append(Spacer(1,4))
    # 数据说明
    S.append(P(f"★ 3留本周：剔除5/19~5/21未到期天，取5/15~{tw_d3_lbl}加权均值；"
               f"7留上周：上周全部7天数据完整（5/08~5/14），两列数据口径一致，可比较。",
               7, False, C_GRAY))
    S.append(Spacer(1,6))

    dsp_rows = agents[agents["总代.名称"].str.contains("DSP", na=False)]
    dsp_cr   = dsp_rows["充提差率"].values[0] if len(dsp_rows)>0 else 0
    dsp_reg  = dsp_rows["注册环比"].values[0] if len(dsp_rows)>0 else 0
    S.append(insight_box([
        f"A8_DSP_谷歌原生包充提差率{dsp_cr:.1f}%，注册+{dsp_reg:.1f}%，体量第三且指标双优。",
        "苹果包总代注册+27.1%，充值体量第二，为本周注册增量最大渠道之一。",
        "YB_FB_PWA_0注册暴跌-79%至1,315人，需持续监控是否恢复。",
    ]))

    # ── 全量留存汇总表（加ID列）────────────────────────
    S.append(sub_title(
        f"总代首充充值留存率（每日加权均值 | 次日/3日取本周5/15~{tw_d3_lbl} | 7日取上周5/08~{lw_d7_lbl}）"))
    S.append(P("保证分子分母一致：仅统计留存期已完整的首充日，3留本周 > 7留上周符合留存递减规律。",
               7.5, False, C_GRAY))
    S.append(Spacer(1,4))

    headers2 = ["ID", "总代名称", "次日留存\n本/上周", "3日留存\n本周", "7日留存\n上周"]
    rows2 = []
    top15_names = [str(r["总代.名称"]) for _,r in agents.iterrows()]
    extra = sorted([n for n in agent_ret if n not in top15_names and n != "总体"])
    ordered = top15_names + extra
    if "总体" in agent_ret: ordered.append("总体")

    for nm in ordered[:22]:
        d = agent_ret.get(nm, {})
        tw1 = d.get("tw_d1", np.nan); lw1 = d.get("lw_d1", np.nan)
        tw3 = d.get("tw_d3", np.nan); lw7 = d.get("lw_d7", np.nan)
        aid = d.get("agent_id", None)
        aid_str = str(int(aid)) if aid and not (isinstance(aid,float) and np.isnan(aid)) else "-"

        if not (isinstance(tw1,float) and np.isnan(tw1)) and not (isinstance(lw1,float) and np.isnan(lw1)):
            nd_s = f"{tw1:.1f}%/{lw1:.1f}%"
            nd_c = C_GREEN if tw1 >= lw1 else C_RED
        elif not (isinstance(tw1,float) and np.isnan(tw1)):
            nd_s = f"{tw1:.1f}%/-"; nd_c = C_DARK
        else:
            nd_s = "-"; nd_c = C_GRAY

        rows2.append([
            cell(aid_str,    False, C_GRAY,  TA_CENTER),
            cell(nm[:22],    True,  C_DARK,  TA_LEFT),
            cell(nd_s,       False, nd_c,    TA_RIGHT),
            cell(_fret(tw3), False, C_DARK,  TA_RIGHT),
            cell(_fret(lw7), False, C_DARK,  TA_RIGHT),
        ])

    if rows2:
        S.append(dtable(headers2, rows2,
                        [full*x for x in [0.08,0.46,0.16,0.15,0.15]], fsize=6.8))
    return S

def build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw):
    S=[sec_title("三、用户分析")]
    full=PW-2*MARGIN
    S.append(sub_title("VIP等级分层分析（充值/投注金额 本周 vs 上周）"))
    S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
    total_chg=tw_v["充值金额"].sum()
    high_vip_pct=(tw_v.loc[9,"充值金额"]+tw_v.loc[10,"充值金额"]+tw_v.loc[11,"充值金额"])/total_chg*100 if 9 in tw_v.index else 0
    S.append(insight_box([
        f"VIP9-11高价值层合计贡献充值{high_vip_pct:.1f}%，高端用户付费意愿持续强劲。",
        f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。",
    ]))
    vip_ret=load_vip_retention()
    S.append(sub_title("VIP各等级 充值→充值留存率 & 充值→活跃留存率（本周均值）"))
    S.append(chart_vip_retention(vip_ret)); S.append(Spacer(1,4))
    S.append(insight_box([
        f"充值→活跃次日留存：VIP9达{vip_ret['act']['tw']['1日'].get(9,0):.1f}%，VIP10达{vip_ret['act']['tw']['1日'].get(10,0):.1f}%，高VIP活跃粘性强。",
        f"充值→充值次日留存：VIP10达{vip_ret['chg']['tw']['1日'].get(10,0):.1f}%（上周{vip_ret['chg']['lw']['1日'].get(10,0):.1f}%），高VIP复充意愿强。",
    ]))
    S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析（Top1/10/50/100/200/500）"))
    headers=["分层","本周充值","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    S.append(insight_box(["Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。"]))
    S.append(sub_title("头部提款用户分层分析（Top1/10/50/100/200/500）"))
    headers3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","赢家\n比例","活动\n占比","占全\n量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdraw_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(headers3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box([f"剔除Top100提款用户后，大盘充提差率影响+{wdraw_t[3]['大盘影响']:.2f}pp，头部提款用户对充提差率有明显拖累。",f"Top500提款用户活动奖励占资金来源仅{wdraw_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。"],clr=C_AMBER))
    return S

def build_games_section(mfr,top30):
    S=[sec_title("四、游戏分析")]; full=PW-2*MARGIN
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    headers=["排名","厂商","日均投注人数\n(本/上)","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["show_name_厂商标签id"])[:10],True),cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
    S.append(insight_box([f"Tada投注份额{mfr.iloc[0]['占比']:.1f}%（上周{mfr.iloc[0].get('lw_占比',0):.1f}%），Fortune系稳固主导；环比{mfr.iloc[0].get('投注环比',0):+.1f}%。"]))
    S.append(sub_title("Top30游戏详细数据（本周 vs 上周）")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
    headers2=["#","游戏名称","厂商","投注\n人数","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["游戏.名称"])[:18],True),cell(str(r["游戏厂商标签.名称"])[:7]),cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(headers2,rows2,[full*x for x in [0.04,0.18,0.08,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S

def build_activities_section(act_df,total_gift,src_df,total_dep):
    S=[sec_title("五、活动分析")]; full=PW-2*MARGIN
    S.append(sub_title("各活动赠送效果（本周 vs 上周，含环比）"))
    S.append(chart_activities(act_df)); S.append(Spacer(1,4))
    headers=["活动名称","本周赠送","上周赠送","环比","赠送人次","人均赠送","金额占比"]
    rows=[]
    for _,r in act_df.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:16],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['lw_赠送']:,.0f}",False,C_GRAY,TA_RIGHT),rc(r["环比"]),cell(f"{r['赠送人数']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均']:.2f}",False,C_DARK,TA_RIGHT),cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.22,0.14,0.14,0.1,0.14,0.12,0.14]])); S.append(Spacer(1,4))
    S.append(P(f"本周总赠送金额：${total_gift/10000:.2f}万 | 赠送/充值比：{total_gift/total_dep*100:.2f}%",9,True,C_DARK)); S.append(Spacer(1,8))
    S.append(sub_title("存款来源道具分析（充值金额占比)")); S.append(chart_deposit_src(src_df,total_dep)); S.append(Spacer(1,4))
    headers2=["道具/来源","本周充值(万)","上周充值(万)","环比","充值占比"]
    rows2=[]
    for _,r in src_df.iterrows():
        rows2.append([cell(str(r["名称"])[:18],True),cell(f"{r['充值金额']/10000:.2f}",False,C_DARK,TA_RIGHT),cell(f"{r['lw_充值']/10000:.2f}",False,C_GRAY,TA_RIGHT),rc(r["环比"]),cell(f"{r['占比']:.1f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT)])
    S.append(dtable(headers2,rows2,[full*x for x in [0.35,0.15,0.15,0.15,0.2]])); S.append(Spacer(1,4))
    S.append(insight_box([f"存款道具合计带动充值${(total_dep-src_df.iloc[0]['充值金额'])/10000:.1f}万，占总充值{(1-src_df.iloc[0]['占比']/100)*100:.1f}%。"],clr=C_AMBER))
    return S

def build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw_full):
    S=[sec_title("六、用户游戏风险专项分析",clr=colors.HexColor("#7c2d12"))]; full=PW-2*MARGIN
    total_tx=top500["提款金额"].sum(); total_win=top500["公司输赢"].sum(); winner_cnt=(top500["公司输赢"]<0).sum()
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${total_tx/10000:.1f}万",""),("平台净赔付",f"${abs(total_win)/10000:.1f}万","平台向该群体净赔"),("赢家比例",f"{winner_cnt/500*100:.1f}%",f"{winner_cnt}赢/{500-winner_cnt}输"),("高风险用户",f"{len(high_risk)}人","公司净输>$10,000")]:
        fw=full/4-4; row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP"); t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    S.append(insight_box([f"Top500提款用户中赢家{winner_cnt}人（{winner_cnt/500*100:.1f}%），平台净赔付${abs(total_win)/10000:.1f}万。",f"{len(high_risk)}名超级赢家合计导致平台净输${abs(high_risk['公司输赢'].sum())/10000:.1f}万，占整体净赔付{abs(high_risk['公司输赢'].sum())/abs(total_win)*100:.1f}%。"]))
    S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
    headers_r=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注\n均额","主玩厂商","主玩游戏","风险标签"]
    rows_r=[]
    for _,r in high_risk.iterrows():
        ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0; avg_b=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
        tags=[]; 
        if r["充值金额"]<5000: tags.append("低充高提")
        if ratio>50: tags.append("超高投充")
        if avg_b>200: tags.append("高额单注")
        rows_r.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),cell(f"${avg_b:.0f}",False,C_RED if avg_b>200 else C_DARK,TA_RIGHT),cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
    S.append(dtable(headers_r,rows_r,[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    if len(cluster_207)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警")); S.append(P(f"集群账户数：{len(cluster_207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
        headers_c=["账户ID","提款金额","充值金额","公司输赢","特征"]; rows_c=[]
        for _,r in c207_dt.head(10).iterrows():
            rows_c.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell("充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}")])
        if len(c207_dt)>10: rows_c.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
        S.append(dtable(headers_c,rows_c,[full*x for x in [0.22,0.18,0.18,0.18,0.24]])); S.append(Spacer(1,6))
        S.append(insight_box([f"{len(cluster_207)}个账户充值$547-548，均获活动奖励$270.92，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。","建议：①冻结账户提款；②审查注册IP/设备指纹；③排查$270.92奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。"],clr=C_AMBER))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    headers_t=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; rows_t=[]
    for i,(_,r) in enumerate(dt_tw_full.head(20).iterrows()):
        win=r["公司输赢"]<0; rows_t.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(headers_t,rows_t,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
    pref_game,mfr_amt=load_pref(); S.append(sub_title("Top500提款用户游戏偏好（按投注金额）")); S.append(chart_pref(pref_game,mfr_amt)); S.append(Spacer(1,4))
    S.append(sub_title("▶ 高危游戏专项分析（Top500提款用户视角）")); S.append(chart_risk_games(risk_g)); S.append(Spacer(1,4))
    headers_g=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; rows_g=[]
    for _,r in risk_g.head(12).iterrows():
        pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rl_clr=C_RED if rl in ("极高","高") else C_AMBER
        rows_g.append([cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rl_clr)])
    S.append(dtable(headers_g,rows_g,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box(["PP Auto-Roulette 1批量套利模式持续，建议暂停该桌并复核奖励规则。","Fortune Garuda 500（Tada）是Top500提款用户投注规模最大游戏，建议对RTP参数做紧急复审。"],clr=C_RED))
    return S

def build_conclusion(K,agents,mfr,dep_t,wdraw_t):
    S=[sec_title("七、总结与行动建议")]; full=PW-2*MARGIN
    S.append(sub_title("▶ 本周亮点"))
    for h in [f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），充提差{K['tw_充提差']/10000:.1f}万（{K['pct_充提差']:+.1f}%），充提差率{K['tw_充提差比']:.2f}%（{K['tw_充提差比']-K['lw_充提差比']:+.2f}pp）。",f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。","A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。"]:
        S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#f0fdf4")),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8)); S.append(sub_title("▶ 风险预警"))
    for r in ["PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。",f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp），新用户质量需关注。","YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。"]:
        S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#fff5f5")),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8)); S.append(sub_title("▶ 行动建议"))
    actions=[
        ("[紧急]","处置PP Auto-Roulette 1批量套利",
         "33个207开头账户合计提款$9.4万，平台净赔$6.7万。审查IP/设备等，修复$270.92奖励触发漏洞，对Auto-Roulette 1实施赢额上限。"),
        ("[本周]","处置高风险提款用户",
         f"账户49903866：提款$39,636，公司输赢-$52,134；账户206628722：充值$10,229提款$24,855（净提$14,626），高度异常。建议立即人工审核。"),
        ("[本周]","优化首充质量与次日激活",
         f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp）。建议注册后1h/24h内推送首充引导，落地页突出$10+档。"),
        ("[下周]","扩大A8_DSP、A8_GG渠道预算",
         "A8_DSP充提差率17.6%（+1.3pp），一级首充成本$17（上周$21，改善）；A8_GG充提差率18.9%（+0.3pp）。建议预算增加15-20%，预估带动充值增量$50-70万。"),
        ("[下周]","暂停YB_FB_PWA_0",
         "注册量暴跌-79%，仅1,315人，建议暂停投放并排查渠道异常。"),
    ]
    rows_a=[[cell(p,True,{"[紧急]":C_RED,"[本周]":C_AMBER,"[下周]":C_GREEN}.get(p[:4],C_GRAY),TA_CENTER),cell(t,True,C_DARK),cell(d,False,C_GRAY)] for p,t,d in actions]
    S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows_a,[full*x for x in [0.1,0.22,0.68]]))
    return S

def header_footer(c,doc):
    c.saveState(); w,h=A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN,h-17,f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
    c.restoreState()

def main(data_root=None, output_dir=None):
    root   = Path(data_root)  if data_root  else DATA_ROOT
    outdir = Path(output_dir) if output_dir else OUTPUT_DIR
    outdir.mkdir(parents=True, exist_ok=True)
    print("📊 MX 周报 PDF v5.5 生成中...")
    _resolve_files(root); setup()
    print("  ▶ 加载数据...")
    K,trend=load_platform(); weekly_ret=load_weekly_retention()
    agents=load_agents(); agent_ret=load_agent_ret()
    tw_v,lw_v=load_vip(); dep_t,wdraw_t,dc_tw,dt_tw=load_top_users()
    top30=load_games(); mfr=load_mfr()
    act_df,total_gift=load_activities(); src_df,total_dep=load_deposit_src()
    high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500=load_risk()
    pref_game,mfr_amt=load_pref()
    bet_user=pd.read_csv(FILES["pref_tw"]); bet_user["阶段汇总"]=bet_user["阶段汇总"].apply(clean)
    top_game=(bet_user[bet_user["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False).groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]].rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
    dt_tw=dt_tw.merge(top_game,on="账户ID",how="left")
    print("  ▶ 导出风控名单Excel...")
    excel_out=outdir/f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out),engine="openpyxl") as writer:
        cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
        high_risk[[c for c in cols_hr if c in high_risk.columns]].to_excel(writer,sheet_name="高风险用户",index=False)
        if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(writer,sheet_name="PP轮盘批量账号集群",index=False)
        if len(special_users)>0: special_users[["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]].to_excel(writer,sheet_name="特殊异常用户",index=False)
    print(f"✅ 风控名单Excel: {excel_out}")
    print("  ▶ 构建PDF...")
    out=outdir/f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc=SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,topMargin=MARGIN+22,bottomMargin=MARGIN+10,title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story=[]
    story+=[Spacer(1,3*cm),P("MX 平台数据周报",30,True,C_DARK,TA_CENTER),Spacer(1,0.5*cm),P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),Spacer(1,0.2*cm),P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),Spacer(1,0.5*cm),HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"),PageBreak()]
    story+=build_overview(K,trend,weekly_ret); story.append(PageBreak())
    story+=build_agents(agents,agent_ret); story.append(PageBreak())
    story+=build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw); story.append(PageBreak())
    story+=build_games_section(mfr,top30); story.append(PageBreak())
    story+=build_activities_section(act_df,total_gift,src_df,total_dep); story.append(PageBreak())
    story+=build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw); story.append(PageBreak())
    story+=build_conclusion(K,agents,mfr,dep_t,wdraw_t)
    print("  ▶ 渲染PDF...")
    doc.build(story,onFirstPage=header_footer,onLaterPages=header_footer)
    size=out.stat().st_size/1024; print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out),str(excel_out)

if __name__=="__main__":
    import sys
    if sys.platform!="win32": main(data_root="/mnt/user-data/uploads",output_dir="/mnt/user-data/outputs")
    else: main(data_root=DATA_ROOT,output_dir=OUTPUT_DIR)

📊 MX 周报 PDF v5.5 生成中...
  ▶ 数据目录：D:\周报更新版\MX
     ✅ [platform    ] 大盘/平台报表_USD_近4周.xlsx
     ✅ [daily       ] 大盘/日报-大盘日报_USD_近4周.xlsx
     ✅ [retention   ] 大盘/整体 首充留存（近7天）_近28天.csv
     ✅ [agent_plat  ] 总代/平台报表-总代_USD_近21天.xlsx
     ✅ [agent_promo ] 总代/推广报表-总代_USD_近21天.xlsx
     ✅ [agent_ret   ] 总代/首充充值留存_20260424_20260521.csv
     ✅ [vip         ] 用户/VIP报表_USD_近14天.xlsx
     ✅ [dt_tw       ] 用户/top提款用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dt_lw       ] 用户/top提款用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [dc_tw       ] 用户/头部充值用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dc_lw       ] 用户/头部充值用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [pref_tw     ] 用户/本周top500提款用户游戏偏好_全量数据_20260515_20260521.csv
     ✅ [pref_lw     ] 用户/上周top500提款用户游戏偏好_全量数据_20260508_20260514.csv
     ✅ [mfr         ] 游戏/厂商投注数据_全量数据_20260508_20260521.csv
     ✅ [game_tw     ] 游戏/游戏报表-详情_USD_本周.xlsx
     ✅ [game_lw     ] 游戏/游戏报表-详情_USD_上周.xlsx
     ✅ [gift 

In [39]:
"""
MX 平台数据周报生成脚本 v6.0
======================================================
维护指南（五区结构）：
  第一区：周期配置  ← 每周只改这里（路径+日期）
  第二区：文案配置  ← 所有中文解读/建议/预警集中在这里
  第三区：数据加载  ← 改数据逻辑
  第四区：图表      ← 改图表样式
  第五区：章节构建  ← 改PDF布局

快速定位：Ctrl+F 搜 "# ══ 第X区" 直接跳转对应区域
"""

# ════════════════════════════════════════════════════════════════
# 第一区：周期配置  ← 每周只改这里
# ════════════════════════════════════════════════════════════════
from pathlib import Path

DATA_ROOT  = Path(r"D:\周报更新版\MX")       # 数据根目录
OUTPUT_DIR = Path(r"D:\周报更新版\MX\输出")  # 输出目录（自动创建）
THIS_WEEK  = ("20260515", "20260521")           # 本周 YYYYMMDD
LAST_WEEK  = ("20260508", "20260514")           # 上周 YYYYMMDD
REPORT_END = THIS_WEEK[1]                       # 报告截止日 = 本周最后一天

# ════════════════════════════════════════════════════════════════
# 第二区：文案配置  ← 改解读文字/建议/预警只在这里改
# ════════════════════════════════════════════════════════════════

# 充提差率颜色阈值（>=HIGH 绿色，<LOW 红色，其余黑色）
CR_HIGH = 17
CR_LOW  = 5

# ── 第一章：大盘解读（绿色提示框）──────────────────────────────────
# {变量名} 会被自动替换为实际数值，不用手动填数字
OVERVIEW_INSIGHTS = [
    "充值{tw_充值金额:.1f}万（{pct_充值金额:+.1f}%），公司输赢{tw_公司输赢:.1f}万（{pct_公司输赢:+.1f}%）；推广消耗{pct_真实消耗:+.1f}%。",
    "充提差率{tw_充提差比:.2f}%（上周{lw_充提差比:.2f}%，{delta_充提差比:+.2f}pp）；盈亏率{tw_盈亏率:.3f}%（上周{lw_盈亏率:.3f}%）。",
    "首充人数{tw_首充人数:,}（{pct_首充人数:+.1f}%），首充转化率{tw_首充转化率:.1f}%，首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%）。",
    "首充次日充值留存{tw_首充次日复充率:.1f}%（上周{lw_首充次日复充率:.1f}%，{delta_首充次日复充率:+.1f}pp）。",
]

# 留存图下方黄色提示框
RETENTION_NOTE = [
    "本周首充次日留存{tw_d1:.1f}%，较上周{delta_d1:+.1f}pp；2日留存{tw_d2:.1f}%。",
    "注：本周7日留存因数据截止日期无法完整计算，以上周7日留存{lw_d7:.1f}%作为近期参考基准。",
]

# ── 第二章：总代分析解读 ────────────────────────────────────────────
AGENT_INSIGHTS = [
    "A8_DSP_谷歌原生包充提差率{dsp_cr:.1f}%，注册+{dsp_reg:.1f}%，体量第三且指标双优。",
    "苹果包总代注册+27.1%，充值体量第二，为本周注册增量最大渠道之一。",
    "YB_FB_PWA_0注册暴跌-79%至1,315人，需持续监控是否恢复。",
]

# ── 第三章：用户分析解读 ────────────────────────────────────────────
VIP_INSIGHTS = [
    "VIP9-11高价值层合计贡献充值{high_vip_pct:.1f}%，高端用户付费意愿持续强劲。",
    "VIP1基础层活跃{tw_vip1_active:,}人（上周{lw_vip1_active:,}），是拉新政策直接反映。",
]

VIP_RET_INSIGHTS = [
    "充值→活跃次日留存：VIP9达{act_v9:.1f}%，VIP10达{act_v10:.1f}%，高VIP活跃粘性强。",
    "充值→充值次日留存：VIP10达{chg_v10_tw:.1f}%（上周{chg_v10_lw:.1f}%），高VIP复充意愿强。",
]

TOP_DEPOSIT_INSIGHTS = [
    "Top10充值用户人均${top10_avg:,.0f}（{top10_avg_chg:+.1f}%），充提差率{top10_cr:.1f}%。",
    "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。",
]

TOP_WITHDRAW_INSIGHTS = [
    "剔除Top100提款用户后，大盘充提差率影响+{top100_impact:.2f}pp，头部提款用户对充提差率有明显拖累。",
    "Top500提款用户活动奖励占资金来源仅{top500_act_pct:.1f}%，主要靠真实赢钱后提款，非活动套利。",
]

# ── 第四章：游戏分析解读 ────────────────────────────────────────────
MFR_INSIGHTS = [
    "Tada投注份额{tada_share:.1f}%（上周{tada_share_lw:.1f}%），Fortune系稳固主导；环比{tada_chg:+.1f}%。",
    "Evolution盈亏率{evol_pl:.2f}%（上周{evol_pl_lw:.2f}%），已转正，较上周大幅改善。",
    "3Oaks盈亏率{oaks_pl:.2f}%，PlayTech{ptech_pl:.2f}%，均高于平台均值，可适当扩大曝光权重。",
]

# ── 第五章：活动分析解读 ────────────────────────────────────────────
ACTIVITY_INSIGHTS = [
    "存款道具合计带动充值${tool_deposit:.1f}万，占总充值{tool_pct:.1f}%。",
    "幸运翻卡道具（71975）充值{xk_deposit:.2f}万（{xk_chg:+.1f}%），注意留存效果是否持续。",
]

# ── 第六章：风险分析解读 ────────────────────────────────────────────
RISK_SCATTER_INSIGHTS = [
    "Top500提款用户中赢家{winner_cnt}人（{winner_pct:.1f}%），平台净赔付${total_loss:.1f}万。",
    "{high_risk_cnt}名超级赢家（公司净输>$10,000）合计导致平台净输${high_risk_loss:.1f}万，"
    "占整体净赔付{high_risk_pct:.1f}%。",
]

PP_ROULETTE_INSIGHTS = [
    "{cluster_cnt}个账户充值$547-548，均获活动奖励$270.92，账户ID均以'207'开头，均主玩PP Auto-Roulette 1。",
    "建议：①冻结账户提款；②审查注册IP/设备指纹；③排查$270.92奖励触发规则；④对Auto-Roulette 1实施单账户赢额上限。",
]

RISK_GAME_INSIGHTS = [
    "PP Auto-Roulette 1批量套利模式持续，建议暂停该桌并复核奖励规则。",
    "Fortune Garuda 500（Tada）是Top500提款用户投注规模最大游戏，建议对其RTP参数做紧急复审。",
]

# ── 第七章：总结 ────────────────────────────────────────────────────
# 亮点列表（绿色背景）
HIGHLIGHTS = [
    "充值{tw_充值金额:.1f}万（{pct_充值金额:+.1f}%），充提差{tw_充提差:.1f}万（{pct_充提差:+.1f}%），充提差率{tw_充提差比:.2f}%（{delta_充提差比:+.2f}pp）。",
    "注册人数{tw_注册人数:,}（{pct_注册人数:+.1f}%），首充人数{tw_首充人数:,}（{pct_首充人数:+.1f}%），拉新规模扩大。",
    "A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。",
]

# 风险预警列表（红色背景）
RISKS = [
    "PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。",
    "首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%），首充次日留存{tw_首充次日复充率:.1f}%（{delta_首充次日复充率:+.1f}pp），新用户质量需关注。",
    "YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。",
]

# 行动建议（优先级 / 事项 / 说明；说明支持 {变量} 替换）
ACTIONS = [
    (
        "[紧急]",
        "处置PP Auto-Roulette 1批量套利",
        "33个207开头账户合计提款$9.4万，平台净赔$6.7万。"
        "冻结账户，审查IP/设备指纹，修复$270.92奖励触发漏洞，对Auto-Roulette 1实施赢额上限。",
    ),
    (
        "[本周]",
        "处置高风险提款用户",
        "账户49903866：提款$39,636，公司输赢-$52,134；"
        "账户206628722：充值$10,229提款$24,855（净提$14,626），高度异常。建议立即人工审核。",
    ),
    (
        "[本周]",
        "优化首充质量与次日激活",
        "首充ARPPU${tw_首充Arppu:.2f}（{pct_首充Arppu:+.1f}%），"
        "首充次日留存{tw_首充次日复充率:.1f}%（{delta_首充次日复充率:+.1f}pp）。"
        "建议注册后1h/24h内推送首充引导，落地页突出$10+档。",
    ),
    (
        "[下周]",
        "扩大A8_DSP、A8_GG渠道预算",
        "A8_DSP充提差率17.6%（+1.3pp），一级首充成本$17（上周$21，改善）；"
        "A8_GG充提差率18.9%（+0.3pp）。建议预算增加15-20%，预估带动充值增量$50-70万。",
    ),
    (
        "[下周]",
        "暂停YB_FB_PWA_0",
        "注册量暴跌-79%，仅1,315人，建议暂停投放并排查渠道异常。",
    ),
]


# ── 文案渲染工具（把 {变量} 替换为实际值，缺失时原样保留）──────────
def render(template, **kwargs):
    try:
        return template.format(**kwargs)
    except (KeyError, ValueError):
        return template

def render_list(templates, **kwargs):
    return [render(t, **kwargs) for t in templates]


# ════════════════════════════════════════════════════════════════
# 第三区：依赖库 & 工具 & 数据加载  ← 改数据逻辑在这里
# ════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════
import io, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, PageBreak, HRFlowable, KeepTogether
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

FN, FNB = "WQY", "WQYB"
FILES = {}

# ── v5.5 核心工具：计算留存完整截止日 ─────────────────
def retention_end_date(week_start, lag_days):
    """
    给定周起始日和留存lag，返回在REPORT_END内已完整的最后一天。
    逻辑：首充日 + lag_days <= REPORT_END
          → 首充日 <= REPORT_END - lag_days
    """
    end_dt  = datetime.strptime(REPORT_END, "%Y%m%d")
    last_dt = end_dt - timedelta(days=lag_days)
    last    = last_dt.strftime("%Y%m%d")
    return last if last >= week_start else None

# ── 字体 ─────────────────────────────────────
def find_chinese_font():
    import platform
    sys_name = platform.system()
    if sys_name == "Windows":
        candidates = [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\msyhbd.ttc",
                      r"C:\Windows\Fonts\simsun.ttc", r"C:\Windows\Fonts\simhei.ttf"]
    elif sys_name == "Darwin":
        candidates = ["/System/Library/Fonts/PingFang.ttc"]
    else:
        candidates = ["/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
                      "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc"]
    for p in candidates:
        if Path(p).exists(): return p
    try:
        from matplotlib import font_manager as fm
        for f in fm.fontManager.ttflist:
            if any(k in f.name for k in ["Heiti","Hei","YaHei","SimSun","SimHei","WenQuanYi","Noto Sans CJK"]):
                if Path(f.fname).exists(): return f.fname
    except Exception: pass
    raise FileNotFoundError("未找到中文字体！请手动设置 FONT_PATH")

def setup():
    global FONT_PATH
    FONT_PATH = find_chinese_font()
    print(f"  ▶ 使用字体：{FONT_PATH}")
    is_ttc = FONT_PATH.lower().endswith(".ttc")
    if is_ttc:
        pdfmetrics.registerFont(TTFont(FN, FONT_PATH, subfontIndex=0))
        try:   pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=1))
        except: pdfmetrics.registerFont(TTFont(FNB, FONT_PATH, subfontIndex=0))
    else:
        pdfmetrics.registerFont(TTFont(FN, FONT_PATH))
        pdfmetrics.registerFont(TTFont(FNB, FONT_PATH))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(FONT_PATH)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == FONT_PATH]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

# ── 文件搜索 ─────────────────────────────────
def _resolve_files(root):
    global FILES
    kw_map = {
        "platform":    "平台报表_USD_近4周",
        "daily":       "日报-大盘日报_USD_近4周",
        "retention":   "首充留存",
        "agent_plat":  "平台报表-总代_USD_近21天",
        "agent_promo": "推广报表-总代_USD_近21天",
        "agent_ret":   "首充充值留存",
        "vip":         "VIP报表_USD",
        "dt_tw":       f"top提款用户_全量数据_{THIS_WEEK[0]}",
        "dt_lw":       f"top提款用户_全量数据_{LAST_WEEK[0]}",
        "dc_tw":       f"头部充值用户_全量数据_{THIS_WEEK[0]}",
        "dc_lw":       f"头部充值用户_全量数据_{LAST_WEEK[0]}",
        "pref_tw":     f"本周top500提款用户游戏偏好_全量数据_{THIS_WEEK[0]}",
        "pref_lw":     f"上周top500提款用户游戏偏好_全量数据_{LAST_WEEK[0]}",
        "mfr":         "厂商投注数据_全量数据",
        "game_tw":     "游戏报表-详情_USD_本周",
        "game_lw":     "游戏报表-详情_USD_上周",
        "gift":        f"各赠送活动_全量数据_{THIS_WEEK[0]}",
        "deposit_src": f"整体存款_全量数据_{THIS_WEEK[0]}",
        "tool_map":    "道具对应活动",
        "vip_ret_chg": "VIP充值-充值_近28天",
        "vip_ret_act": "VIP充值-活跃_近28天",
    }
    print(f"  ▶ 数据目录：{root}")
    fail = 0
    for key, kw in kw_map.items():
        hits = [h for h in root.rglob("*")
                if h.is_file() and not h.name.startswith("~$") and kw in h.name]
        if not hits:
            print(f"     ❌ [{key:12s}] 找不到含 '{kw}' 的文件"); fail += 1
        else:
            p = max(hits, key=lambda h: h.stat().st_mtime)
            FILES[key] = p
            print(f"     ✅ [{key:12s}] {p.parent.name}/{p.name}")
    if fail:
        raise FileNotFoundError(f"\n共 {fail} 个文件未找到，请检查 DATA_ROOT = {root}")
    print(f"  ▶ 全部 {len(kw_map)} 个文件匹配成功\n")

# ── 颜色 ─────────────────────────────────────
C_BLUE   = colors.HexColor("#1d4ed8")
C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669")
C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706")
C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b")
C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white
C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1")
C_ROW    = colors.HexColor("#f8fafc")

PW, PH = A4
MARGIN  = 1.8*cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
TRUNCATE_RULES = {
    "首充次日复充率": 1, "首充次日复投率": 1, "首充当日复充率": 0,
    "首充7日复充率": 6, "首充30日复充率": 999,
    "首充2日复充率": 1, "首充3日复充率": 2,
}

# ── 工具函数 ─────────────────────────────────
def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c != "日期" and c not in ["总代.名称","name_总代","总代.ID"]]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o): return (n-o)/abs(o)*100 if o and o!=0 else 0.0
def pct_vec(ns, os): return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop>0 else s
    nz = v[v>0]; return nz.mean() if len(nz)>0 else v.mean()
def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

# ── 样式函数 ─────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.38,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),
                                ("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5),
                         HRFlowable(width="100%", thickness=1.5, color=C_BLUE2),
                         Spacer(1,3), P(f"■  {text}", 10, True, C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}", 8.5, False, C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8):
    full = PW-2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),        ("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2.5),     ("BOTTOMPADDING",(0,0),(-1,-1),2.5),
        ("LEFTPADDING",(0,0),(-1,-1),3),       ("RIGHTPADDING",(0,0),(-1,-1),3),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),  ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    hrow = [P(h, fsize, True, C_WHITE, TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]), fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c, (list,tuple)) else P(str(c), fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t, bold=False, clr=colors.black, align=TA_LEFT): return (t, bold, clr, align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup = good_up
    s = "+" if v>=0 else ""; c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{s}{v:.{d}f}%", False, c, TA_RIGHT)
def gclr(v, t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)

def kpi_card4(items, cols=4):
    full = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,gup in items:
        s = "+" if chg>=0 else ""
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([
            [P(label, 7.5, False, C_GRAY)],
            [P(str(tv), 14, True, C_DARK)],
            [P(f"上周：{lv}", 7.5, False, C_GRAY)],
            [P(f"{s}{chg:.1f}%", 8, True, pclr)],
        ], colWidths=[full], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),C_LGRAY), ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),8),
            ("TOPPADDING",(0,0),(-1,-1),5), ("BOTTOMPADDING",(0,0),(-1,-1),5),
        ]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(full,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════
def load_platform():
    df = pd.read_excel(FILES["platform"])
    df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]
    lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"])
    dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]
    lw_d = dd[dd["日期"].between(*LAST_WEEK)]
    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["充提差比","盈亏率","首充转化率","首充当日复充率","首充次日复充率",
              "首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE_RULES.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop)
        K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    K["tw_真实消耗"] = tw_d["真实消耗"].iloc[:-1].sum() if len(tw_d)>1 else tw_d["真实消耗"].sum()
    K["lw_真实消耗"] = lw_d["真实消耗"].sum()
    K["pct_真实消耗"] = pct(K["tw_真实消耗"], K["lw_真实消耗"])
    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])
    last14 = df.tail(14)
    trend = {
        "dates":    [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
        "充值":     last14["充值金额"].tolist(), "提现":  last14["提现金额"].tolist(),
        "充提差比": last14["充提差比"].tolist(), "公司输赢": last14["公司输赢"].tolist(),
        "首充":     last14["首充人数"].tolist(), "注册":  last14["注册人数"].tolist(),
    }
    return K, trend

def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    daily = df[~df["初始事件的发生时间"].str.contains("阶段值", na=False)].copy()
    daily["date_str"] = daily["初始事件的发生时间"].str.extract(r"(\d{4}-\d{2}-\d{2})")
    daily_r = daily[daily["指标"]=="留存率"].copy()
    daily_u = daily[daily["指标"]=="留存人数"].copy()
    for c in ["第1日","第2日","第3日","第7日"]: daily_r[c] = daily_r[c].apply(clean)
    def _d(s): return datetime.strptime(s, "%Y%m%d")
    def _fmt(d): return d.strftime("%Y-%m-%d")
    def _lbl(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s=_d(THIS_WEEK[0]); tw_e=_d(THIS_WEEK[1])
    lw_s=_d(LAST_WEEK[0]); lw_e=_d(LAST_WEEK[1])
    w2_end=lw_s-timedelta(days=1); w2_start=w2_end-timedelta(days=6)
    w1_end=w2_start-timedelta(days=1); w1_start=w1_end-timedelta(days=6)
    weeks = [
        (f"第1周\n{_lbl(w1_start,w1_end)}", _fmt(w1_start), _fmt(w1_end)),
        (f"第2周\n{_lbl(w2_start,w2_end)}", _fmt(w2_start), _fmt(w2_end)),
        (f"上周\n{_lbl(lw_s,lw_e)}",        _fmt(lw_s),     _fmt(lw_e)),
        (f"本周\n{_lbl(tw_s,tw_e)}",        _fmt(tw_s),     _fmt(tw_e)),
    ]
    result = []
    for wk,s,e in weeks:
        mr = daily_r[(daily_r["date_str"]>=s)&(daily_r["date_str"]<=e)]
        mu = daily_u[(daily_u["date_str"]>=s)&(daily_u["date_str"]<=e)]
        users = mu["充值成功事件用户数"].sum(); row = {"week": wk, "users": users}
        for col in ["第1日","第2日","第3日","第7日"]:
            rs = mr[col].values; us = mu["充值成功事件用户数"].values
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = np.average(rs[v], weights=us[v]) if v.sum()>0 else np.nan
            else:
                row[col] = np.nanmean(rs) if len(rs)>0 else np.nan
        result.append(row)
    return result

def load_agents():
    df_p = pd.read_excel(FILES["agent_plat"])
    df_r = pd.read_excel(FILES["agent_promo"])
    df_p["日期"] = df_p["日期"].astype(str); df_r["日期"] = df_r["日期"].astype(str)
    df_p = to_num(df_p); df_r = to_num(df_r)
    df_p = df_p[df_p["总代.ID"]!=0]; df_r = df_r[df_r["总代.ID"]!=0]
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]; lw_p = df_p[df_p["日期"].between(*LAST_WEEK)]
    tw_r = df_r[df_r["日期"].between(*THIS_WEEK)]; lw_r = df_r[df_r["日期"].between(*LAST_WEEK)]
    sum_a = ["充值金额","提现金额","充提差","首充金额","首充人数","注册人数",
             "充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        g = d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in sum_a if c in d.columns}).reset_index()
        for c in ["首充转化率","充提差比","首充次日充值留存","活跃用户付费率"]:
            if c in d.columns:
                rm = d.groupby("总代.ID")[c].mean().rename(c); g = g.merge(rm, on="总代.ID", how="left")
        g["充提差率"] = g["充提差"]/g["充值金额"]*100; return g
    tw_pa = agg_p(tw_p); lw_pa = agg_p(lw_p)
    sum_r = ["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        g = d.groupby("总代.ID").agg({c:"sum" for c in sum_r if c in d.columns}).reset_index()
        g["注册成本"] = g["总消耗"]/g["注册人数"].replace(0,np.nan)
        g["一级首充成本"] = g["总消耗"]/g["一级首充人数"].replace(0,np.nan); return g
    tw_ra = agg_r(tw_r); lw_ra = agg_r(lw_r)
    lw_ra["lw_一级首充成本"] = lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
    m = tw_pa.merge(
        lw_pa[["总代.ID","充值金额","注册人数","充提差率","首充次日充值留存"]].rename(
            columns={"充值金额":"lw_充值","注册人数":"lw_注册",
                     "充提差率":"lw_充提差率","首充次日充值留存":"lw_次日留存"}),
        on="总代.ID", how="left")
    m = m.merge(tw_ra[["总代.ID","总消耗","注册成本","一级首充成本","一级首充人数"]], on="总代.ID", how="left")
    m = m.merge(lw_ra[["总代.ID","lw_一级首充成本"]], on="总代.ID", how="left")
    m["注册环比"] = pct_vec(m["注册人数"], m["lw_注册"])
    m["充值环比"] = pct_vec(m["充值金额"], m["lw_充值"])
    return m.sort_values("充值金额", ascending=False).head(15)

# ════════════════════════════════════════════
# v5.5 核心修复：总代留存加载（剔除未到期天数）
# ════════════════════════════════════════════
def load_agent_ret():
    """
    CSV 格式：初始事件发生时间 / 总代 / name_总代 / 首充用户数 / 指标 / 1日 / 3日 / 7日...

    v5.5 关键修复：
    ─────────────────────────────────────────────────────────
    留存第N日的含义：首充日+N天后仍活跃的比例。
    若报告截止日=THIS_WEEK[1]，则：
      首充日 + N <= THIS_WEEK[1]
      → 首充日 <= THIS_WEEK[1] - N天

    本周3日留存：只取 THIS_WEEK[0] ~ (REPORT_END-3天)
      本例：20260515 ~ 20260518（5/19、5/20、5/21三天3日未到期，剔除）
    本周1日留存：只取 THIS_WEEK[0] ~ (REPORT_END-1天) = 20260520
    上周7日留存：只取 LAST_WEEK[0] ~ (REPORT_END-7天) = 20260514（全部完整）
    上周1日留存：只取 LAST_WEEK[0] ~ (REPORT_END-1天) = 20260520（全部完整）
    ─────────────────────────────────────────────────────────
    返回：dict { name_总代 -> {tw_d1, tw_d3, lw_d1, lw_d7, 总代ID} }
    """
    df = pd.read_csv(FILES["agent_ret"])

    # 解析日期列
    df["_yyyymmdd"] = (df["初始事件发生时间"].astype(str)
                       .str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
                       .str.replace("-",""))

    # 数值化留存率（去掉 %）
    for c in ["1日","2日","3日","7日"]:
        df[c] = (df[c].astype(str).str.replace("%","").str.strip()
                     .pipe(pd.to_numeric, errors="coerce"))

    # 首充用户数
    df["首充用户数"] = pd.to_numeric(df["首充用户数"], errors="coerce").fillna(0)

    # 总代ID（用于汇总表）
    df["总代"] = pd.to_numeric(df["总代"], errors="coerce")

    # 只保留每日留存率行
    daily = df[
        df["_yyyymmdd"].notna() &
        df["_yyyymmdd"].str.match(r"^\d{8}$") &
        (df["指标"] == "留存率")
    ].copy()

    # ── 计算各留存指标的完整截止日 ──────────────────
    tw_d1_end = retention_end_date(THIS_WEEK[0], 1)   # 本周1日: ~20260520
    tw_d3_end = retention_end_date(THIS_WEEK[0], 3)   # 本周3日: ~20260518 ★关键修复
    lw_d1_end = retention_end_date(LAST_WEEK[0], 1)   # 上周1日: 全部完整
    lw_d7_end = retention_end_date(LAST_WEEK[0], 7)   # 上周7日: 全部完整

    print(f"     留存完整截止日 → 本周1日:{tw_d1_end} 本周3日:{tw_d3_end} "
          f"上周1日:{lw_d1_end} 上周7日:{lw_d7_end}")

    # 分别筛选各时间窗口
    tw_all = daily[daily["_yyyymmdd"].between(*THIS_WEEK)]
    lw_all = daily[daily["_yyyymmdd"].between(*LAST_WEEK)]

    def _win(df_week, week_start, end_cap):
        if end_cap is None: return df_week.iloc[0:0]  # 空
        return df_week[(df_week["_yyyymmdd"] >= week_start) &
                       (df_week["_yyyymmdd"] <= end_cap)]

    tw_d1_df = _win(tw_all, THIS_WEEK[0], tw_d1_end)
    tw_d3_df = _win(tw_all, THIS_WEEK[0], tw_d3_end)
    lw_d1_df = _win(lw_all, LAST_WEEK[0], lw_d1_end)
    lw_d7_df = _win(lw_all, LAST_WEEK[0], lw_d7_end)

    def _wavg_all(df_sub, col):
        """对所有总代做首充用户数加权平均，返回 {name_总代: value}"""
        result = {}
        for nm, grp in df_sub.groupby("name_总代"):
            valid = grp[grp[col].notna() & (grp["首充用户数"] > 0)]
            if len(valid) > 0:
                result[nm] = float(np.average(valid[col], weights=valid["首充用户数"]))
            else:
                result[nm] = np.nan
        return result

    tw_d1 = _wavg_all(tw_d1_df, "1日")
    tw_d3 = _wavg_all(tw_d3_df, "3日")
    lw_d1 = _wavg_all(lw_d1_df, "1日")
    lw_d7 = _wavg_all(lw_d7_df, "7日")

    # 总代ID映射（取第一条）
    id_map = {}
    for nm, grp in daily.groupby("name_总代"):
        id_val = grp["总代"].dropna().values
        if len(id_val) > 0:
            id_map[nm] = int(id_val[0])

    all_names = set(tw_d1) | set(lw_d1)
    ret = {}
    for nm in all_names:
        ret[nm] = {
            "tw_d1": tw_d1.get(nm, np.nan),
            "tw_d3": tw_d3.get(nm, np.nan),
            "lw_d1": lw_d1.get(nm, np.nan),
            "lw_d7": lw_d7.get(nm, np.nan),
            "agent_id": id_map.get(nm, None),
        }
    return ret

def load_vip():
    df = pd.read_excel(FILES["vip"]); df["日期"] = df["日期"].astype(str)
    num = ["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df = to_num(df, num)
    tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()

def load_vip_retention():
    results = {}
    for fkey, rtype in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df = pd.read_csv(FILES[fkey])
        df['date_str'] = df['初始事件发生时间'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')
        df['yyyymmdd'] = df['date_str'].str.replace('-','')
        for c in ['1日','2日','3日','7日']: df[c] = df[c].apply(clean)
        df['n']   = pd.to_numeric(df['充值成功事件用户数'], errors='coerce')
        df['vip'] = pd.to_numeric(df['vip_level'], errors='coerce')
        stage = df[(df['指标']=='留存率')&(df['初始事件发生时间']=='阶段值')].copy()
        stage_dict = {int(r['vip']):{c:r[c] for c in ['1日','2日','3日','7日']}
                      for _,r in stage.iterrows() if not pd.isna(r['vip'])}
        daily = df[(df['指标']=='留存率')&df['yyyymmdd'].notna()&df['vip'].notna()]
        tw = daily[daily['yyyymmdd'].between(*THIS_WEEK)]
        lw = daily[daily['yyyymmdd'].between(*LAST_WEEK)]
        def wavg(d,col):
            res={}
            for vip,g in d.groupby('vip'):
                v=g[g[col].notna()]
                if len(v)>0: res[int(vip)]=np.average(v[col],weights=v['n'])
            return res
        results[rtype] = {
            'tw':{c:wavg(tw,c) for c in ['1日','2日','3日','7日']},
            'lw':{c:wavg(lw,c) for c in ['1日','2日','3日','7日']},
            'stage':stage_dict,
        }
    return results

def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
    for df in [dc_tw,dc_lw,dt_tw,dt_lw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
    tiers = [1,10,50,100,200,500]
    df_p = pd.read_excel(FILES["platform"]); df_p["日期"]=df_p["日期"].astype(str); df_p=to_num(df_p)
    tw_p = df_p[df_p["日期"].between(*THIS_WEEK)]
    total_chg=tw_p["充值金额"].sum(); total_tx=tw_p["提现金额"].sum()
    actual_cr=(total_chg-total_tx)/total_chg*100
    dep_tiers,wdraw_tiers=[],[]
    total_tw_c=dc_tw["充值金额"].sum(); total_tw_t=dt_tw["提款金额"].sum()
    for t in tiers:
        tw=dc_tw.head(t); lw=dc_lw.head(t)
        tw_c=tw["充值金额"].sum(); tw_tx=tw["提款金额"].sum()
        lw_c=lw["充值金额"].sum(); lw_tx=lw["提款金额"].sum()
        dep_tiers.append({"tier":f"Top{t}","tw_chg":tw_c,"lw_chg":lw_c,
            "tw_avg":tw_c/t,"lw_avg":lw_c/t,
            "tw_cr":(tw_c-tw_tx)/tw_c*100 if tw_c>0 else 0,
            "lw_cr":(lw_c-lw_tx)/lw_c*100 if lw_c>0 else 0,
            "tw_win":tw["公司输赢"].sum(),"占全量":tw_c/total_tw_c*100})
        twd=dt_tw.head(t); lwd=dt_lw.head(t)
        tw_t2=twd["提款金额"].sum(); tw_c2=twd["充值金额"].sum()
        lw_t2=lwd["提款金额"].sum(); lw_c2=lwd["充值金额"].sum()
        tw_cr2=(tw_c2-tw_t2)/tw_c2*100 if tw_c2>0 else 0
        lw_cr2=(lw_c2-lw_t2)/lw_c2*100 if lw_c2>0 else 0
        winners=(twd["公司输赢"]<0).sum()
        act=twd["活动奖励"].sum()/(tw_c2+twd["活动奖励"].sum())*100
        excl_chg=total_chg-tw_c2; excl_tx=total_tx-tw_t2
        excl_cr=(excl_chg-excl_tx)/excl_chg*100 if excl_chg>0 else 0
        impact=excl_cr-actual_cr
        wdraw_tiers.append({"tier":f"Top{t}","tw_tx":tw_t2,"lw_tx":lw_t2,
            "tw_avg":tw_t2/t,"lw_avg":lw_t2/t,"tw_cr":tw_cr2,"lw_cr":lw_cr2,
            "赢家":winners,"总数":t,"赢家率":winners/t*100,
            "活动占比":act,"占全量":tw_t2/total_tw_t*100,"大盘影响":impact})
    return dep_tiers,wdraw_tiers,dc_tw.head(200),dt_tw.head(20)

def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); df_lw=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); df_lw["阶段汇总"]=df_lw["阶段汇总"].apply(clean)
    bet=df[df["分析指标"]=="投注金额"]; win=df[df["分析指标"]=="公司输赢"]
    bet_lw=df_lw[df_lw["分析指标"]=="投注金额"]
    g_bet=bet.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("本周投注")
    g_win=win.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢")
    g_lw=bet_lw.groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("上周投注")
    g_usr=bet.groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([g_bet,g_win,g_lw,g_usr],axis=1).reset_index()
    total=g["本周投注"].sum(); g["占比"]=g["本周投注"]/total*100
    g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby("show_name_厂商标签id")["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr

def load_games():
    df_tw=pd.read_excel(FILES["game_tw"]); df_lw=pd.read_excel(FILES["game_lw"])
    for df in [df_tw,df_lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢","人均投注局数","人均投注金额"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    total_tw=df_tw["投注金额"].sum()
    g_tw=df_tw.groupby(["游戏厂商标签.名称","游戏.名称"]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    g_lw=df_lw.groupby("游戏.名称").agg(
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    g_tw["人均局数"]=g_tw["投注局数"]/g_tw["投注人数"]
    g_tw["人均金额"]=g_tw["投注金额"]/g_tw["投注人数"]
    g_tw["盈亏率"]=g_tw["公司输赢"]/g_tw["投注金额"]*100
    g_tw["占比"]=g_tw["投注金额"]/total_tw*100
    gm=g_tw.merge(g_lw,on="游戏.名称",how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)

def load_mfr():
    df=pd.read_csv(FILES["mfr"])
    df=df[df["时间"]!="阶段汇总"].copy()
    df["时间"]=df["时间"].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢"]: df[c]=df[c].apply(clean)
    tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
    def agg(d):
        g=d.groupby("show_name_厂商标签id").agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100; return g
    tw_g=agg(tw); lw_g=agg(lw)
    mg=tw_g.merge(lw_g[["show_name_厂商标签id","投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on="show_name_厂商标签id",how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    return mg.sort_values("投注金额",ascending=False)

def load_activities():
    df=pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数"]: df[c]=df[c].apply(clean)
    agg=df.groupby("name_账变opt_id").agg(
        赠送金额=("赠送金额","sum"),lw_赠送=("赠送金额.1","sum"),赠送人数=("赠送人数","sum")).reset_index()
    agg["环比"]=(agg["赠送金额"]-agg["lw_赠送"])/agg["lw_赠送"].abs()*100
    total=agg["赠送金额"].sum(); agg["占比"]=agg["赠送金额"]/total*100; agg["人均"]=agg["赠送金额"]/agg["赠送人数"]
    return agg.sort_values("赠送金额",ascending=False).head(15), total

def load_deposit_src():
    df=pd.read_csv(FILES["deposit_src"])
    for c in ["充值金额","充值金额.1","充值人数"]: df[c]=df[c].apply(clean)
    tm=pd.read_excel(FILES["tool_map"]); tm["道具ID"]=tm["道具ID"].astype(str)
    df["道具ID"]=df["道具ID"].astype(str)
    agg=df.groupby("道具ID").agg(充值金额=("充值金额","sum"),lw_充值=("充值金额.1","sum"),充值人数=("充值人数","sum")).reset_index()
    agg=agg.merge(tm,on="道具ID",how="left")
    nm={"(null)":"无道具直接充值","71872":"幸运转盘10%","71873":"幸运转盘20%",
        "71975":"幸运翻卡道具","71894":"新人签到10%","71871":"新人签到20%"}
    agg["名称"]=agg.apply(lambda r: nm.get(str(r["道具ID"]),
        str(r.get("备注",""))[:12] if pd.notna(r.get("备注")) else f"其他({str(r['道具ID'])[:8]})"), axis=1)
    total=agg["充值金额"].sum(); agg["占比"]=agg["充值金额"]/total*100
    agg["环比"]=(agg["充值金额"]-agg["lw_充值"])/agg["lw_充值"].abs()*100
    return agg.sort_values("充值金额",ascending=False).head(6), total

def load_risk():
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    dt_tw=pd.read_csv(FILES["dt_tw"]); dc_tw=pd.read_csv(FILES["dc_tw"])
    for df2 in [dt_tw,dc_tw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额"]:
            if c in df2.columns: df2[c]=pd.to_numeric(df2[c],errors="coerce")
    bet_amt=df[df["分析指标"]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    win_pref=df[df["分析指标"]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    bet_cnt=df[df["分析指标"]=="投注次数"].groupby("账户ID")["阶段汇总"].sum().rename("投注次数")
    top_game=(df[df["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
              .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称","阶段汇总"]]
              .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏","阶段汇总":"主游戏投注额"}).reset_index())
    ug=pd.concat([bet_cnt,bet_amt,win_pref],axis=1).reset_index()
    ug["每注均额"]=ug["游戏_投注金额"]/ug["投注次数"]; ug=ug.merge(top_game,on="账户ID",how="left")
    top500=dt_tw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0; top500["投充比"]=top500["投注金额"]/top500["充值金额"]
    high_risk=top500[top500["公司输赢"]<-10000].sort_values("公司输赢")
    pp_ids=df[(df["游戏名称"]=="Auto-Roulette 1")&(df["分析指标"]=="投注次数")]["账户ID"].unique()
    cluster_207=[u for u in pp_ids if str(u).startswith("207")]
    c207_dt=dt_tw[dt_tw["账户ID"].isin(cluster_207)].sort_values("提款金额",ascending=False)
    special=top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)].sort_values("提款金额",ascending=False)
    bet_g=df[df["分析指标"]=="投注金额"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("投注金额").reset_index()
    win_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"])["阶段汇总"].sum().rename("公司输赢").reset_index()
    usr_g=df[df["分析指标"]=="投注次数"].groupby(["show_name_厂商标签id","游戏名称"])["账户ID"].nunique().rename("玩家数").reset_index()
    winr_g=df[df["分析指标"]=="公司输赢"].groupby(["show_name_厂商标签id","游戏名称"]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bet_g.merge(win_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(usr_g,on=["show_name_厂商标签id","游戏名称"],how="left").merge(winr_g,on=["show_name_厂商标签id","游戏名称"],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100; g["人均投注额"]=g["投注金额"]/g["玩家数"]
    risk_g=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    return high_risk,c207_dt,cluster_207,special,g.sort_values("投注金额",ascending=False).head(20),risk_g,top500

# ════════════════════════════════════════════
# 图表（与 v5.4 相同，保持不变）
# ════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════
# 第四区：图表  ← 改图表标题/颜色/尺寸在这里
# ════════════════════════════════════════════════════════════════

SPLIT=7
def _vline(ax,dates):
    ax.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax.text(SPLIT-4,ax.get_ylim()[1]*0.93,"上周",ha="center",fontsize=7,color="#64748b")
    ax.text(SPLIT+3,ax.get_ylim()[1]*0.93,"本周",ha="center",fontsize=7,color="#1d4ed8")

def chart_trend(trend):
    fig=plt.figure(figsize=(16,10),facecolor="white")
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.45,wspace=0.3)
    dates=trend["dates"]; x=range(len(dates))
    def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=0.25)
    ax=fig.add_subplot(gs[0,0])
    clrs=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT
    ax.bar(x,[v/10000 for v in trend["充值"]],color=clrs,width=0.7,label="充值")
    ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
    ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold"); ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
    ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax,dates)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax2=fig.add_subplot(gs[0,1])
    ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=0.7)
    ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax2); _vline(ax2,dates)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}"))
    ax3=fig.add_subplot(gs[1,0])
    ax3.bar(x,trend["首充"],color=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT,width=0.7,label="首充人数")
    ax3r=ax3.twinx(); ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold"); ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
    l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left"); ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=0.25); ax3.axvline(SPLIT-0.5,color="#94a3b8",ls="--",lw=1,alpha=0.7)
    ax4=fig.add_subplot(gs[1,1])
    ax4.bar(x,trend["充提差比"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=0.7)
    tw_m=np.mean(trend["充提差比"][SPLIT:]); lw_m=np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tw_m,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lw_m,color="#94a3b8",ls=":",lw=1.2)
    ax4.text(len(dates)-0.5,tw_m+0.3,f"本周均{tw_m:.1f}%",fontsize=7,color="#7c3aed",ha="right")
    ax4.text(0.5,lw_m+0.3,f"上周均{lw_m:.1f}%",fontsize=7,color="#64748b",ha="left")
    ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold"); ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax4); _vline(ax4,dates)
    ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
    plt.tight_layout(rect=[0,0,1,0.98]); return fig_img(fig,16,11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white")
    cols_p=[("第1日","#1d4ed8"),("第2日","#059669"),("第3日","#d97706"),("第7日","#7c3aed")]
    x=np.arange(len(weeks)); w=0.18
    for i,(col,clr) in enumerate(cols_p):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-1.5*w,vals,w,label=col,color=clr,alpha=0.85)
        for bar,v in zip(bars,vals):
            if not np.isnan(v): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周加权平均对比",fontsize=10,fontweight="bold"); ax.legend(fontsize=8,loc="upper right")
    ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=0.5); ax.text(2.6,ax.get_ylim()[1]*0.85,"↑本周",fontsize=7.5,color="#ef4444")
    ax.grid(axis="y",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    fig.text(0.5,-0.02,f"注：本周({THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]})7日留存数据未完整",ha="center",fontsize=7.5,color="#64748b",style="italic")
    plt.tight_layout(); return fig_img(fig,16,5.5)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=0.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tw_c=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_c=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tw_c,w,label="本周",color="#1d4ed8",alpha=0.85); ax1.bar(x+w/2,lw_c,w,label="上周",color="#93c5fd",alpha=0.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tw_b=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lw_b=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tw_b,w,label="本周",color="#059669",alpha=0.85); ax2.bar(x+w/2,lw_b,w,label="上周",color="#6ee7b7",alpha=0.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100 if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=0.85,width=0.6); ax3.axhline(0,color="black",lw=0.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); ax3.grid(axis="y",alpha=0.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vip_ret):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    chg_tw=[vip_ret['chg']['tw']['1日'].get(v,0) for v in vips]; chg_lw=[vip_ret['chg']['lw']['1日'].get(v,0) for v in vips]; chg_3d=[vip_ret['chg']['tw']['3日'].get(v,0) for v in vips]
    act_tw=[vip_ret['act']['tw']['1日'].get(v,0) for v in vips]; act_lw=[vip_ret['act']['lw']['1日'].get(v,0) for v in vips]; act_3d=[vip_ret['act']['tw']['3日'].get(v,0) for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=0.28
    ax1.bar(x-w,chg_tw,w,label="次日留存(本周)",color="#1d4ed8",alpha=0.85); ax1.bar(x,chg_lw,w,label="次日留存(上周)",color="#93c5fd",alpha=0.7); ax1.bar(x+w,chg_3d,w,label="3日留存(本周)",color="#059669",alpha=0.75)
    ax1.set_title("充值→充值 留存率（%）",fontsize=10,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,fontsize=8); ax1.legend(fontsize=7.5,loc="upper left"); ax1.grid(axis="y",alpha=0.25); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    ax2.bar(x-w,act_tw,w,label="次日留存(本周)",color="#7c3aed",alpha=0.85); ax2.bar(x,act_lw,w,label="次日留存(上周)",color="#c4b5fd",alpha=0.7); ax2.bar(x+w,act_3d,w,label="3日留存(本周)",color="#d97706",alpha=0.75)
    ax2.set_title("充值→活跃 留存率（%）",fontsize=10,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,fontsize=8); ax2.legend(fontsize=7.5,loc="upper left"); ax2.grid(axis="y",alpha=0.25); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.suptitle("VIP各等级充值留存率（本周 vs 上周，加权日均值）",fontsize=11,fontweight="bold",y=1.02); plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10); names=top10["show_name_厂商标签id"].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    shares=top10["占比"].tolist(); lw_s=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,shares,color=["#059669" if c>=l else "#dc2626" for c,l in zip(shares,lw_s)],alpha=0.85,width=0.6)
    ax1.plot(names,lw_s,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,shares,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,f"{s:.1f}%",ha="center",fontsize=7); ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额（本周 vs 上周）",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=0.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=0.85); ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=0.7)
    ax2.axhline(0,color="black",lw=0.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5); ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    top15=top30.head(15); names=[r["游戏.名称"][:18] for _,r in top15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in top15.iterrows()]; lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in top15.iterrows()]
    x=np.arange(len(names)); w=0.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=0.85); ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=0.7)
    for bar,v in zip(ax.patches[:len(names)],bets[::-1]): ax.text(bar.get_width()+0.2,bar.get_y()+bar.get_height()/2,f"{v:.0f}万",va="center",fontsize=7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8); ax.set_title("Top15游戏 投注金额（万USD，本周 vs 上周）",fontsize=9,fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="x",alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pref_game,mfr_amt):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=mfr_amt.index[:8].tolist(); vals=mfr_amt.values[:8].tolist(); total=sum(vals); pcts=[v/total*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.05,-0.18)); ax1.set_title("Top500提款用户 厂商偏好（按投注金额）",fontsize=9,fontweight="bold")
    top12=pref_game.head(12); gnames=[f"[{r['show_name_厂商标签id']}]\n{r['游戏名称']}"[:22] for _,r in top12.iterrows()]; gvals=[r["本周投注"]/10000 for _,r in top12.iterrows()]; gclrs=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in top12.iterrows()]
    ax2.barh(range(len(gnames))[::-1],gvals,color=gclrs[::-1],alpha=0.85); ax2.set_yticks(range(len(gnames))); ax2.set_yticklabels(gnames,fontsize=7.5); ax2.set_title("偏好游戏Top12（按投注金额，红=平台亏损）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}万")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act_df):
    top10=act_df.head(10); names=[str(r["name_账变opt_id"])[:10] for _,r in top10.iterrows()]; vals=[r["赠送金额"]/10000 for _,r in top10.iterrows()]; lw=[r["lw_赠送"]/10000 for _,r in top10.iterrows()]; envs=[r["环比"] for _,r in top10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=0.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=0.85); ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=0.7); ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8); ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold"); ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax2.barh(range(len(names)),envs[::-1],color=["#059669" if v>0 else "#dc2626" for v in envs[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8); ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold"); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%")); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_deposit_src(src_df,total):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); names=src_df["名称"].tolist(); vals=src_df["充值金额"].tolist(); lw_v=src_df["lw_充值"].tolist(); pcts=src_df["占比"].tolist(); clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>3 else "",startangle=140,pctdistance=0.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{n[:8]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-0.1,-0.15)); ax1.set_title(f"存款来源占比（总计${total/10000:.0f}万USD）",fontsize=9,fontweight="bold")
    x=np.arange(len(names)); w=0.35; ax2.bar(x-w/2,[v/10000 for v in vals],w,label="本周",color=clrs,alpha=0.85); ax2.bar(x+w/2,[v/10000 for v in lw_v],w,label="上周",color=["#93c5fd"]*len(names),alpha=0.6); ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=25,ha="right",fontsize=7.5); ax2.set_title("存款道具 本周 vs 上周（万USD）",fontsize=9,fontweight="bold"); ax2.legend(fontsize=8); ax2.grid(axis="y",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_risk_scatter(top500):
    valid=top500[top500["投充比"].notna()&top500["游戏_投注金额"].notna()].copy(); valid=valid[valid["投充比"]<200]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    ax.scatter(valid["投充比"],valid["公司输赢"]/10000,c=["#dc2626" if v<0 else "#059669" for v in valid["公司输赢"]],s=[min(abs(v)/500+20,200) for v in valid["公司输赢"]],alpha=0.55,edgecolors="none")
    ax.axhline(0,color="black",lw=0.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=0.7); ax.text(21,ax.get_ylim()[0]*0.9,"投充比=20x",fontsize=7.5,color="#d97706")
    for _,r in top500[(top500["公司输赢"]<-10000)&(top500["投充比"]<200)&top500["投充比"].notna()].iterrows():
        ax.annotate(str(r["账户ID"])[-5:],xy=(r["投充比"],r["公司输赢"]/10000),xytext=(r["投充比"]+5,r["公司输赢"]/10000-0.2),fontsize=6.5,color="#991b1b",arrowprops=dict(arrowstyle="->",color="#991b1b",lw=0.7))
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8); ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=0.6,label="平台输钱"),mpatches.Patch(color="#059669",alpha=0.6,label="平台赢钱")],fontsize=8,loc="upper right"); ax.grid(alpha=0.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(risk_g):
    top12=risk_g.head(12); names=[r["游戏名称"][:16] for _,r in top12.iterrows()]; losses=[abs(r["公司输赢"]) for _,r in top12.iterrows()]; rates=[r["盈亏率"] for _,r in top12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=0.85)
    for bar,v in zip(ax1.patches,losses[::-1]): ax1.text(bar.get_width()+500,bar.get_y()+bar.get_height()/2,f"${v:,.0f}",va="center",fontsize=7)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold"); ax1.grid(axis="x",alpha=0.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False); ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=0.85); ax2.axvline(0,color="black",lw=0.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold"); ax2.grid(axis="x",alpha=0.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False); ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)

# ════════════════════════════════════════════
# 章节构建
# ════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════
# 第五区：章节构建  ← 改PDF表格列宽/布局在这里
# ════════════════════════════════════════════════════════════════

def build_overview(K,trend,weekly_ret):
    S=[sec_title("一、大盘核心数据")]
    kpis=[
        ("充值金额",f"{K['tw_充值金额']/10000:.1f}万",f"{K['lw_充值金额']/10000:.1f}万",K["pct_充值金额"],True),
        ("提现金额",f"{K['tw_提现金额']/10000:.1f}万",f"{K['lw_提现金额']/10000:.1f}万",K["pct_提现金额"],False),
        ("充提差",f"{K['tw_充提差']/10000:.1f}万",f"{K['lw_充提差']/10000:.1f}万",K["pct_充提差"],True),
        ("充提差率",f"{K['tw_充提差比']:.2f}%",f"{K['lw_充提差比']:.2f}%",K["pct_充提差比"],True),
        ("公司输赢",f"{K['tw_公司输赢']/10000:.1f}万",f"{K['lw_公司输赢']/10000:.1f}万",K["pct_公司输赢"],True),
        ("盈亏率",f"{K['tw_盈亏率']:.3f}%",f"{K['lw_盈亏率']:.3f}%",K["pct_盈亏率"],True),
        ("注册人数",f"{int(K['tw_注册人数']):,}",f"{int(K['lw_注册人数']):,}",K["pct_注册人数"],True),
        ("首充人数",f"{int(K['tw_首充人数']):,}",f"{int(K['lw_首充人数']):,}",K["pct_首充人数"],True),
        ("日均活跃",f"{K['tw_活跃人数']/7/10000:.1f}万",f"{K['lw_活跃人数']/7/10000:.1f}万",K["pct_活跃人数"],True),
        ("投注金额",f"{K['tw_投注金额']/10000:.0f}万",f"{K['lw_投注金额']/10000:.0f}万",K["pct_投注金额"],True),
        ("全量日均ARPPU",f"${K['tw_全量Arppu']:.2f}",f"${K['lw_全量Arppu']:.2f}",K["pct_全量Arppu"],True),
        ("老用户日均ARPPU",f"${K['tw_老用户ARPPU']:.2f}",f"${K['lw_老用户ARPPU']:.2f}",K["pct_老用户ARPPU"],True),
        ("首充日均ARPPU",f"${K['tw_首充Arppu']:.2f}",f"${K['lw_首充Arppu']:.2f}",K["pct_首充Arppu"],True),
        ("总赠送金额",f"{K['tw_总赠送金额']/10000:.1f}万",f"{K['lw_总赠送金额']/10000:.1f}万",K["pct_总赠送金额"],False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",f"{K['lw_赠送充值比']:.2f}%",K["pct_赠送充值比"],False),
        ("首充转化率",f"{K['tw_首充转化率']:.1f}%",f"{K['lw_首充转化率']:.1f}%",K["pct_首充转化率"],True),
        ("首充次日充值留存",f"{K['tw_首充次日复充率']:.1f}%",f"{K['lw_首充次日复充率']:.1f}%",K["pct_首充次日复充率"],True),
        ("推广消耗",f"{K['tw_真实消耗']/10000:.1f}万",f"{K['lw_真实消耗']/10000:.1f}万",K["pct_真实消耗"],False),
    ]
    S.append(kpi_card4(kpis,cols=4)); S.append(Spacer(1,8))
    cr_delta=K["tw_充提差比"]-K["lw_充提差比"]; ret_delta=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    # 解读文字变量（供 render_list 使用）
    ov = dict(
        tw_充值金额         = K["tw_充值金额"] / 10000,
        pct_充值金额        = K["pct_充值金额"],
        tw_公司输赢         = K["tw_公司输赢"] / 10000,
        pct_公司输赢        = K["pct_公司输赢"],
        pct_真实消耗        = K["pct_真实消耗"],
        tw_充提差比         = K["tw_充提差比"],
        lw_充提差比         = K["lw_充提差比"],
        delta_充提差比      = cr_delta,
        tw_盈亏率           = K["tw_盈亏率"],
        lw_盈亏率           = K["lw_盈亏率"],
        tw_首充人数         = int(K["tw_首充人数"]),
        pct_首充人数        = K["pct_首充人数"],
        tw_首充转化率       = K["tw_首充转化率"],
        tw_首充Arppu        = K["tw_首充Arppu"],
        pct_首充Arppu       = K["pct_首充Arppu"],
        tw_首充次日复充率   = K["tw_首充次日复充率"],
        lw_首充次日复充率   = K["lw_首充次日复充率"],
        delta_首充次日复充率 = ret_delta,
    )
    S.append(insight_box(render_list(OVERVIEW_INSIGHTS, **ov)))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
    S.append(sub_title("首充用户充值留存率 — 近4周加权平均对比")); S.append(chart_ret_weekly(weekly_ret))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=(this_w.get("第1日",0)or 0)-(last_w.get("第1日",0)or 0)
    rn = dict(
        tw_d1    = this_w.get("第1日", 0) or 0,
        tw_d2    = this_w.get("第2日", 0) or 0,
        delta_d1 = d1,
        lw_d7    = last_w.get("第7日", 0) or 0,
    )
    S.append(insight_box(render_list(RETENTION_NOTE, **rn), clr=C_AMBER))
    return S

def build_agents(agents, agent_ret):
    """
    Top15总代表 + 全量留存汇总表
    列说明：
      注册(环比)     → "22,072/+27.1%"  单行，颜色跟环比
      充提差率(差值) → "15.7%/-1.1pp"   单行，颜色跟充提差率高低
      次日留存(本/上) → "22.4%/25.9%"   本<上→红，本>=上→绿
      3留本周        → 本周3日加权（已剔除未到期天，5/15~5/18）
      7留上周        → 上周7日加权（全部完整，5/8~5/14）
      保证：3留本周 > 7留上周（留存递减规律）
    """
    S = [sec_title("二、总代分析（剔除总代0官方总代）")]
    S.append(sub_title("Top15总代表现（本周 vs 上周，按充值金额排序）"))
    full = PW - 2*MARGIN

    # 计算完整截止日（用于列头说明）
    tw_d3_end = retention_end_date(THIS_WEEK[0], 3)  # 20260518
    lw_d7_end = retention_end_date(LAST_WEEK[0], 7)  # 20260514

    tw_d3_lbl = f"{tw_d3_end[4:6]}/{tw_d3_end[6:]}" if tw_d3_end else "-"
    lw_d7_lbl = f"{lw_d7_end[4:6]}/{lw_d7_end[6:]}" if lw_d7_end else "-"

    headers = [
        "ID", "总代名称",
        "注册(环比)",
        "首充",
        "充值\n(万)",
        "充提差率(差值)",
        "消耗\n(万)",
        "一级首充\n成本(本/上)",
        "次日留存\n(本/上)",
        f"3留本周\n(~{tw_d3_lbl})",
        f"7留上周\n(~{lw_d7_lbl})",
    ]

    def _fret(v):
        return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

    rows = []
    for _, r in agents.iterrows():
        agent_id = int(r["总代.ID"]) if "总代.ID" in r and not pd.isna(r.get("总代.ID")) else "-"
        rname    = str(r["总代.名称"])
        cr       = r["充提差率"]; lw_cr = r.get("lw_充提差率", 0) or 0; cr_diff = cr - lw_cr
        cost     = (r.get("总消耗", 0) or 0) / 10000
        fc_c     = r.get("一级首充成本", 0) or 0; lw_fc_c = r.get("lw_一级首充成本", 0) or 0

        # 注册/环比
        reg = int(r["注册人数"]); reg_chg = r.get("注册环比", 0) or 0
        reg_str = f"{reg:,}/{'+' if reg_chg>=0 else ''}{reg_chg:.1f}%"
        reg_clr = C_GREEN if reg_chg >= 0 else C_RED

        # 充提差率/差值
        cr_str = f"{cr:.1f}%/{'+' if cr_diff>=0 else ''}{cr_diff:.1f}pp"
        cr_clr = C_GREEN if cr >= CR_HIGH else (C_RED if cr < CR_LOW else C_DARK)

        # 留存数据
        d = agent_ret.get(rname, {})
        tw_d1 = d.get("tw_d1", np.nan); lw_d1 = d.get("lw_d1", np.nan)
        tw_d3 = d.get("tw_d3", np.nan); lw_d7 = d.get("lw_d7", np.nan)

        # 次日留存 本/上
        if not (isinstance(tw_d1,float) and np.isnan(tw_d1)) and not (isinstance(lw_d1,float) and np.isnan(lw_d1)):
            nd_str = f"{tw_d1:.1f}%/{lw_d1:.1f}%"
            nd_clr = C_GREEN if tw_d1 >= lw_d1 else C_RED
        elif not (isinstance(tw_d1,float) and np.isnan(tw_d1)):
            nd_str = f"{tw_d1:.1f}%/-"; nd_clr = C_DARK
        else:
            nd_str = "-"; nd_clr = C_GRAY

        rows.append([
            cell(str(agent_id),              False, C_GRAY,  TA_CENTER),
            cell(rname[:16],                 True,  C_DARK,  TA_LEFT),
            cell(reg_str,                    False, reg_clr, TA_RIGHT),
            cell(f"{int(r['首充人数']):,}",  False, C_DARK,  TA_RIGHT),
            cell(f"{r['充值金额']/10000:.0f}",False, C_DARK, TA_RIGHT),
            cell(cr_str,                     False, cr_clr,  TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-", False, C_DARK, TA_RIGHT),
            cell(f"${fc_c:.0f}/${lw_fc_c:.0f}" if fc_c>0 else "-", False, C_DARK, TA_RIGHT),
            cell(nd_str,                     False, nd_clr,  TA_RIGHT),
            cell(_fret(tw_d3),               False, C_DARK,  TA_RIGHT),
            cell(_fret(lw_d7),               False, C_DARK,  TA_RIGHT),
        ])

    cw = [full*x for x in [0.05,0.17,0.11,0.06,0.06,0.12,0.06,0.11,0.10,0.08,0.08]]
    S.append(dtable(headers, rows, cw, fsize=6.5))
    S.append(Spacer(1,4))
    # 数据说明
    S.append(P(f"★ 3留本周：剔除5/19~5/21未到期天，取5/15~{tw_d3_lbl}加权均值；"
               f"7留上周：上周全部7天数据完整（5/08~5/14），两列数据口径一致，可比较。",
               7, False, C_GRAY))
    S.append(Spacer(1,6))

    dsp_rows = agents[agents["总代.名称"].str.contains("DSP", na=False)]
    av = dict(
        dsp_cr  = dsp_rows["充提差率"].values[0] if len(dsp_rows) > 0 else 0,
        dsp_reg = dsp_rows["注册环比"].values[0] if len(dsp_rows) > 0 else 0,
    )
    S.append(insight_box(render_list(AGENT_INSIGHTS, **av)))

    # ── 全量留存汇总表（加ID列）────────────────────────
    S.append(sub_title(
        f"总代首充充值留存率（每日加权均值 | 次日/3日取本周5/15~{tw_d3_lbl} | 7日取上周5/08~{lw_d7_lbl}）"))
    S.append(P("保证分子分母一致：仅统计留存期已完整的首充日，3留本周 > 7留上周符合留存递减规律。",
               7.5, False, C_GRAY))
    S.append(Spacer(1,4))

    headers2 = ["ID", "总代名称", "次日留存\n本/上周", "3日留存\n本周", "7日留存\n上周"]
    rows2 = []
    top15_names = [str(r["总代.名称"]) for _,r in agents.iterrows()]
    extra = sorted([n for n in agent_ret if n not in top15_names and n != "总体"])
    ordered = top15_names + extra
    if "总体" in agent_ret: ordered.append("总体")

    for nm in ordered[:22]:
        d = agent_ret.get(nm, {})
        tw1 = d.get("tw_d1", np.nan); lw1 = d.get("lw_d1", np.nan)
        tw3 = d.get("tw_d3", np.nan); lw7 = d.get("lw_d7", np.nan)
        aid = d.get("agent_id", None)
        aid_str = str(int(aid)) if aid and not (isinstance(aid,float) and np.isnan(aid)) else "-"

        if not (isinstance(tw1,float) and np.isnan(tw1)) and not (isinstance(lw1,float) and np.isnan(lw1)):
            nd_s = f"{tw1:.1f}%/{lw1:.1f}%"
            nd_c = C_GREEN if tw1 >= lw1 else C_RED
        elif not (isinstance(tw1,float) and np.isnan(tw1)):
            nd_s = f"{tw1:.1f}%/-"; nd_c = C_DARK
        else:
            nd_s = "-"; nd_c = C_GRAY

        rows2.append([
            cell(aid_str,    False, C_GRAY,  TA_CENTER),
            cell(nm[:22],    True,  C_DARK,  TA_LEFT),
            cell(nd_s,       False, nd_c,    TA_RIGHT),
            cell(_fret(tw3), False, C_DARK,  TA_RIGHT),
            cell(_fret(lw7), False, C_DARK,  TA_RIGHT),
        ])

    if rows2:
        S.append(dtable(headers2, rows2,
                        [full*x for x in [0.08,0.46,0.16,0.15,0.15]], fsize=6.8))
    return S

def build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw):
    S=[sec_title("三、用户分析")]
    full=PW-2*MARGIN
    S.append(sub_title("VIP等级分层分析（充值/投注金额 本周 vs 上周）"))
    S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
    total_chg=tw_v["充值金额"].sum()
    high_vip_pct=(tw_v.loc[9,"充值金额"]+tw_v.loc[10,"充值金额"]+tw_v.loc[11,"充值金额"])/total_chg*100 if 9 in tw_v.index else 0
    vv = dict(
        high_vip_pct   = high_vip_pct,
        tw_vip1_active = int(tw_v.loc[1,"活跃人数"]) if 1 in tw_v.index else 0,
        lw_vip1_active = int(lw_v.loc[1,"活跃人数"]) if 1 in lw_v.index else 0,
    )
    S.append(insight_box(render_list(VIP_INSIGHTS, **vv)))
    vip_ret=load_vip_retention()
    S.append(sub_title("VIP各等级 充值→充值留存率 & 充值→活跃留存率（本周均值）"))
    S.append(chart_vip_retention(vip_ret)); S.append(Spacer(1,4))
    rv = dict(
        act_v9     = vip_ret["act"]["tw"]["1日"].get(9, 0),
        act_v10    = vip_ret["act"]["tw"]["1日"].get(10, 0),
        chg_v10_tw = vip_ret["chg"]["tw"]["1日"].get(10, 0),
        chg_v10_lw = vip_ret["chg"]["lw"]["1日"].get(10, 0),
    )
    S.append(insight_box(render_list(VIP_RET_INSIGHTS, **rv)))
    S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析（Top1/10/50/100/200/500）"))
    headers=["分层","本周充值","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    dv = dict(
        top10_avg     = dep_t[1]["tw_avg"],
        top10_avg_chg = pct(dep_t[1]["tw_avg"], dep_t[1]["lw_avg"]),
        top10_cr      = dep_t[1]["tw_cr"],
    )
    S.append(insight_box(render_list(TOP_DEPOSIT_INSIGHTS, **dv)))
    S.append(sub_title("头部提款用户分层分析（Top1/10/50/100/200/500）"))
    headers3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率\n(本)","充提差率\n(上)","赢家\n比例","活动\n占比","占全\n量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdraw_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(headers3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    wv = dict(
        top100_impact  = wdraw_t[3]["大盘影响"],
        top500_act_pct = wdraw_t[5]["活动占比"],
    )
    S.append(insight_box(render_list(TOP_WITHDRAW_INSIGHTS, **wv), clr=C_AMBER))
    return S

def build_games_section(mfr,top30):
    S=[sec_title("四、游戏分析")]; full=PW-2*MARGIN
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    headers=["排名","厂商","日均投注人数\n(本/上)","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["show_name_厂商标签id"])[:10],True),cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
    def _mv(name, col):
        sub = mfr[mfr["show_name_厂商标签id"] == name]
        return sub[col].values[0] if len(sub) > 0 else 0
    mv = dict(
        tada_share    = mfr.iloc[0]["占比"],
        tada_share_lw = mfr.iloc[0].get("lw_占比", 0),
        tada_chg      = mfr.iloc[0].get("投注环比", 0),
        evol_pl       = _mv("Evolution", "盈亏率"),
        evol_pl_lw    = _mv("Evolution", "lw_盈亏率"),
        oaks_pl       = _mv("3Oaks",    "盈亏率"),
        ptech_pl      = _mv("PlayTech", "盈亏率"),
    )
    S.append(insight_box(render_list(MFR_INSIGHTS, **mv)))
    S.append(sub_title("Top30游戏详细数据（本周 vs 上周）")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
    headers2=["#","游戏名称","厂商","投注\n人数","人均\n局数","人均\n金额","投注额\n(万)","投注\n环比","占比","盈亏率\n(本)","盈亏率\n(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["游戏.名称"])[:18],True),cell(str(r["游戏厂商标签.名称"])[:7]),cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(headers2,rows2,[full*x for x in [0.04,0.18,0.08,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S

def build_activities_section(act_df,total_gift,src_df,total_dep):
    S=[sec_title("五、活动分析")]; full=PW-2*MARGIN
    S.append(sub_title("各活动赠送效果（本周 vs 上周，含环比）"))
    S.append(chart_activities(act_df)); S.append(Spacer(1,4))
    headers=["活动名称","本周赠送","上周赠送","环比","赠送人次","人均赠送","金额占比"]
    rows=[]
    for _,r in act_df.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:16],True),cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['lw_赠送']:,.0f}",False,C_GRAY,TA_RIGHT),rc(r["环比"]),cell(f"{r['赠送人数']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均']:.2f}",False,C_DARK,TA_RIGHT),cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(headers,rows,[full*x for x in [0.22,0.14,0.14,0.1,0.14,0.12,0.14]])); S.append(Spacer(1,4))
    S.append(P(f"本周总赠送金额：${total_gift/10000:.2f}万 | 赠送/充值比：{total_gift/total_dep*100:.2f}%",9,True,C_DARK)); S.append(Spacer(1,8))
    S.append(sub_title("存款来源道具分析（充值金额占比)")); S.append(chart_deposit_src(src_df,total_dep)); S.append(Spacer(1,4))
    headers2=["道具/来源","本周充值(万)","上周充值(万)","环比","充值占比"]
    rows2=[]
    for _,r in src_df.iterrows():
        rows2.append([cell(str(r["名称"])[:18],True),cell(f"{r['充值金额']/10000:.2f}",False,C_DARK,TA_RIGHT),cell(f"{r['lw_充值']/10000:.2f}",False,C_GRAY,TA_RIGHT),rc(r["环比"]),cell(f"{r['占比']:.1f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT)])
    S.append(dtable(headers2,rows2,[full*x for x in [0.35,0.15,0.15,0.15,0.2]])); S.append(Spacer(1,4))
    xk = src_df[src_df["名称"] == "幸运翻卡道具"]
    acv = dict(
        tool_deposit = (total_dep - src_df.iloc[0]["充值金额"]) / 10000,
        tool_pct     = (1 - src_df.iloc[0]["占比"] / 100) * 100,
        xk_deposit   = xk["充值金额"].values[0] / 10000 if len(xk) > 0 else 0,
        xk_chg       = xk["环比"].values[0]               if len(xk) > 0 else 0,
    )
    S.append(insight_box(render_list(ACTIVITY_INSIGHTS, **acv), clr=C_AMBER))
    return S

def build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw_full):
    S=[sec_title("六、用户游戏风险专项分析",clr=colors.HexColor("#7c2d12"))]; full=PW-2*MARGIN
    total_tx=top500["提款金额"].sum(); total_win=top500["公司输赢"].sum(); winner_cnt=(top500["公司输赢"]<0).sum()
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${total_tx/10000:.1f}万",""),("平台净赔付",f"${abs(total_win)/10000:.1f}万","平台向该群体净赔"),("赢家比例",f"{winner_cnt/500*100:.1f}%",f"{winner_cnt}赢/{500-winner_cnt}输"),("高风险用户",f"{len(high_risk)}人","公司净输>$10,000")]:
        fw=full/4-4; row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP"); t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    rsv = dict(
        winner_cnt     = winner_cnt,
        winner_pct     = winner_cnt / 500 * 100,
        total_loss     = abs(total_win) / 10000,
        high_risk_cnt  = len(high_risk),
        high_risk_loss = abs(high_risk["公司输赢"].sum()) / 10000,
        high_risk_pct  = abs(high_risk["公司输赢"].sum()) / abs(total_win) * 100,
    )
    S.append(insight_box(render_list(RISK_SCATTER_INSIGHTS, **rsv)))
    S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
    headers_r=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","每注\n均额","主玩厂商","主玩游戏","风险标签"]
    rows_r=[]
    for _,r in high_risk.iterrows():
        ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0; avg_b=r.get("每注均额",0) if not pd.isna(r.get("每注均额",0)) else 0
        tags=[]; 
        if r["充值金额"]<5000: tags.append("低充高提")
        if ratio>50: tags.append("超高投充")
        if avg_b>200: tags.append("高额单注")
        rows_r.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_RED if r["充值金额"]<5000 else C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['投注金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),cell(f"${avg_b:.0f}",False,C_RED if avg_b>200 else C_DARK,TA_RIGHT),cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
    S.append(dtable(headers_r,rows_r,[full*x for x in [0.11,0.1,0.1,0.1,0.1,0.07,0.07,0.09,0.15,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    if len(cluster_207)>0:
        S.append(Spacer(1,8)); S.append(sub_title("▶ PP Auto-Roulette 1 批量账号预警")); S.append(P(f"集群账户数：{len(cluster_207)}个 | 合计提款：${float(c207_dt['提款金额'].sum())/10000:.1f}万 | 平台净赔：${abs(float(c207_dt['公司输赢'].sum()))/10000:.1f}万",8.5,True,C_DARK)); S.append(Spacer(1,6))
        headers_c=["账户ID","提款金额","充值金额","公司输赢","特征"]; rows_c=[]
        for _,r in c207_dt.head(10).iterrows():
            rows_c.append([cell(str(r["账户ID"]),True),cell(f"${r['提款金额']:,.0f}",False,C_RED,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_AMBER,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell("充值$547-548" if 546<float(r["充值金额"])<549 else f"充值${r['充值金额']:.0f}")])
        if len(c207_dt)>10: rows_c.append([cell(f"…另{len(c207_dt)-10}个账户（模式相同）",False,C_GRAY),cell(""),cell(""),cell(""),cell("")])
        S.append(dtable(headers_c,rows_c,[full*x for x in [0.22,0.18,0.18,0.18,0.24]])); S.append(Spacer(1,6))
        ppv = dict(cluster_cnt=len(cluster_207))
        S.append(insight_box(render_list(PP_ROULETTE_INSIGHTS, **ppv), clr=C_AMBER))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    headers_t=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; rows_t=[]
    for i,(_,r) in enumerate(dt_tw_full.head(20).iterrows()):
        win=r["公司输赢"]<0; rows_t.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),cell(f"${r['提款金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['充值金额']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),cell(f"${r['活动奖励']:,.0f}",False,C_DARK,TA_RIGHT),cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(headers_t,rows_t,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
    pref_game,mfr_amt=load_pref(); S.append(sub_title("Top500提款用户游戏偏好（按投注金额）")); S.append(chart_pref(pref_game,mfr_amt)); S.append(Spacer(1,4))
    S.append(sub_title("▶ 高危游戏专项分析（Top500提款用户视角）")); S.append(chart_risk_games(risk_g)); S.append(Spacer(1,4))
    headers_g=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; rows_g=[]
    for _,r in risk_g.head(12).iterrows():
        pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else ("高" if pl<-10 else "关注"); rl_clr=C_RED if rl in ("极高","高") else C_AMBER
        rows_g.append([cell(str(r["游戏名称"])[:18],True),cell(str(r["show_name_厂商标签id"])[:8]),cell(f"{int(r['玩家数'])}",False,C_DARK,TA_RIGHT),cell(f"${r['投注金额']/10000:.1f}万",False,C_DARK,TA_RIGHT),cell(f"${r['公司输赢']:,.0f}",False,C_RED,TA_RIGHT),cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),cell(f"{r['赢家率']:.0f}%",False,C_RED if r["赢家率"]>60 else C_AMBER,TA_RIGHT),cell(f"${r['人均投注额']:,.0f}",False,C_DARK,TA_RIGHT),cell(rl,True,rl_clr)])
    S.append(dtable(headers_g,rows_g,[full*x for x in [0.21,0.1,0.07,0.1,0.11,0.08,0.08,0.1,0.07]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box(RISK_GAME_INSIGHTS, clr=C_RED))
    return S

def build_conclusion(K,agents,mfr,dep_t,wdraw_t):
    S=[sec_title("七、总结与行动建议")]; full=PW-2*MARGIN
    S.append(sub_title("▶ 本周亮点"))
    for h in [f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），充提差{K['tw_充提差']/10000:.1f}万（{K['pct_充提差']:+.1f}%），充提差率{K['tw_充提差比']:.2f}%（{K['tw_充提差比']-K['lw_充提差比']:+.2f}pp）。",f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。","A8_DSP充提差率17.6%（+1.3pp），A8_GG 18.9%（+0.3pp），优质渠道持续稳定。"]:
        S.append(Table([[P(f"• {h}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#f0fdf4")),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8)); S.append(sub_title("▶ 风险预警"))
    for r in ["PP Auto-Roulette 1盈亏率极低，33个批量账号套利合计导致平台净赔$6.7万，需立即处置。",f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp），新用户质量需关注。","YB_FB_PWA_0注册量暴跌-79%（1,315人），建议暂停投放并排查异常。"]:
        S.append(Table([[P(f"• {r}",8.5,False,C_DARK)]],colWidths=[full],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#fff5f5")),("LEFTPADDING",(0,0),(-1,-1),12),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8)); S.append(sub_title("▶ 行动建议"))
    actions=[("[紧急]","处置PP Auto-Roulette 1批量套利","33个207开头账户合计提款$9.4万，平台净赔$6.7万。冻结账户，审查IP/设备指纹，修复$270.92奖励触发漏洞，对Auto-Roulette 1实施赢额上限。"),("[本周]","处置高风险提款用户",f"账户49903866：提款$39,636，公司输赢-$52,134；账户206628722：充值$10,229提款$24,855（净提$14,626），高度异常。建议立即人工审核。"),("[本周]","优化首充质量与次日激活",f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{K['tw_首充次日复充率']-K['lw_首充次日复充率']:+.1f}pp）。建议注册后1h/24h内推送首充引导，落地页突出$10+档。"),("[下周]","扩大A8_DSP、A8_GG渠道预算","A8_DSP充提差率17.6%（+1.3pp），一级首充成本$17（上周$21，改善）；A8_GG充提差率18.9%（+0.3pp）。建议预算增加15-20%，预估带动充值增量$50-70万。"),("[下周]","暂停YB_FB_PWA_0","注册量暴跌-79%，仅1,315人，建议暂停投放并排查渠道异常。")]
    rows_a=[[cell(p,True,{"[紧急]":C_RED,"[本周]":C_AMBER,"[下周]":C_GREEN}.get(p[:4],C_GRAY),TA_CENTER),cell(t,True,C_DARK),cell(d,False,C_GRAY)] for p,t,d in actions]
    S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows_a,[full*x for x in [0.1,0.22,0.68]]))
    return S

def header_footer(c,doc):
    c.saveState(); w,h=A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN,h-17,f"MX 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
    c.restoreState()

def main(data_root=None, output_dir=None):
    root   = Path(data_root)  if data_root  else DATA_ROOT
    outdir = Path(output_dir) if output_dir else OUTPUT_DIR
    outdir.mkdir(parents=True, exist_ok=True)
    print("📊 MX 周报 PDF v5.5 生成中...")
    _resolve_files(root); setup()
    print("  ▶ 加载数据...")
    K,trend=load_platform(); weekly_ret=load_weekly_retention()
    agents=load_agents(); agent_ret=load_agent_ret()
    tw_v,lw_v=load_vip(); dep_t,wdraw_t,dc_tw,dt_tw=load_top_users()
    top30=load_games(); mfr=load_mfr()
    act_df,total_gift=load_activities(); src_df,total_dep=load_deposit_src()
    high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500=load_risk()
    pref_game,mfr_amt=load_pref()
    bet_user=pd.read_csv(FILES["pref_tw"]); bet_user["阶段汇总"]=bet_user["阶段汇总"].apply(clean)
    top_game=(bet_user[bet_user["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False).groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]].rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
    dt_tw=dt_tw.merge(top_game,on="账户ID",how="left")
    print("  ▶ 导出风控名单Excel...")
    excel_out=outdir/f"MX_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out),engine="openpyxl") as writer:
        cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","每注均额","投充比"]
        high_risk[[c for c in cols_hr if c in high_risk.columns]].to_excel(writer,sheet_name="高风险用户",index=False)
        if len(c207_dt)>0: c207_dt[["账户ID","提款金额","充值金额","公司输赢","活动奖励"]].to_excel(writer,sheet_name="PP轮盘批量账号集群",index=False)
        if len(special_users)>0: special_users[["账户ID","提款金额","充值金额","投注金额","公司输赢","投充比"]].to_excel(writer,sheet_name="特殊异常用户",index=False)
    print(f"✅ 风控名单Excel: {excel_out}")
    print("  ▶ 构建PDF...")
    out=outdir/f"MX周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc=SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,topMargin=MARGIN+22,bottomMargin=MARGIN+10,title=f"MX平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story=[]
    story+=[Spacer(1,3*cm),P("MX 平台数据周报",30,True,C_DARK,TA_CENTER),Spacer(1,0.5*cm),P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),Spacer(1,0.2*cm),P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),Spacer(1,0.5*cm),HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"),PageBreak()]
    story+=build_overview(K,trend,weekly_ret); story.append(PageBreak())
    story+=build_agents(agents,agent_ret); story.append(PageBreak())
    story+=build_users_section(tw_v,lw_v,dep_t,wdraw_t,dc_tw,dt_tw); story.append(PageBreak())
    story+=build_games_section(mfr,top30); story.append(PageBreak())
    story+=build_activities_section(act_df,total_gift,src_df,total_dep); story.append(PageBreak())
    story+=build_risk_section(high_risk,c207_dt,cluster_207,special_users,top20_games,risk_g,top500,dt_tw); story.append(PageBreak())
    story+=build_conclusion(K,agents,mfr,dep_t,wdraw_t)
    print("  ▶ 渲染PDF...")
    doc.build(story,onFirstPage=header_footer,onLaterPages=header_footer)
    size=out.stat().st_size/1024; print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out),str(excel_out)

if __name__=="__main__":
    import sys
    if sys.platform!="win32": main(data_root="/mnt/user-data/uploads",output_dir="/mnt/user-data/outputs")
    else: main(data_root=DATA_ROOT,output_dir=OUTPUT_DIR)

📊 MX 周报 PDF v5.5 生成中...
  ▶ 数据目录：D:\周报更新版\MX
     ✅ [platform    ] 大盘/平台报表_USD_近4周.xlsx
     ✅ [daily       ] 大盘/日报-大盘日报_USD_近4周.xlsx
     ✅ [retention   ] 大盘/整体 首充留存（近7天）_近28天.csv
     ✅ [agent_plat  ] 总代/平台报表-总代_USD_近21天.xlsx
     ✅ [agent_promo ] 总代/推广报表-总代_USD_近21天.xlsx
     ✅ [agent_ret   ] 总代/首充充值留存_20260424_20260521.csv
     ✅ [vip         ] 用户/VIP报表_USD_近14天.xlsx
     ✅ [dt_tw       ] 用户/top提款用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dt_lw       ] 用户/top提款用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [dc_tw       ] 用户/头部充值用户_全量数据_20260515_20260521_日期对比20260508_20260514.csv
     ✅ [dc_lw       ] 用户/头部充值用户_全量数据_20260508_20260514_日期对比20260501_20260507.csv
     ✅ [pref_tw     ] 用户/本周top500提款用户游戏偏好_全量数据_20260515_20260521.csv
     ✅ [pref_lw     ] 用户/上周top500提款用户游戏偏好_全量数据_20260508_20260514.csv
     ✅ [mfr         ] 游戏/厂商投注数据_全量数据_20260508_20260521.csv
     ✅ [game_tw     ] 游戏/游戏报表-详情_USD_本周.xlsx
     ✅ [game_lw     ] 游戏/游戏报表-详情_USD_上周.xlsx
     ✅ [gift 

PermissionError: [Errno 13] Permission denied: 'D:\\周报更新版\\MX\\输出\\MX_风控名单_20260515-20260521.xlsx'